In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
BASE_DIR = "/content/drive/MyDrive/Chd8 data"
RNA_DIR = os.path.join(BASE_DIR, "deseq2/deseq2/results")
CHIP_DIR = os.path.join(BASE_DIR, "CHIPSEQ/MACS2_peaks")
GTF_PATH = os.path.join(BASE_DIR, "annotation/mm10.gtf")

# 3. Create a local workspace to speed up heavy file I/O
!mkdir -p /content/local_data
if os.path.exists(GTF_PATH):
    print("Found GTF. Copying to local disk for speed...")
    !cp "{GTF_PATH}" /content/local_data/mm10.gtf
    LOCAL_GTF = "/content/local_data/mm10.gtf"
else:
    print("⚠️ GTF not found at specified path. Check Step 1 configuration.")

# 4. Install bioinfo tools if missing
!apt-get install -y bedtools
!pip install gseapy matplotlib_venn

In [ ]:
import pandas as pd

# Path to your DESeq2 results file
RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

# 1. Load data
rna = pd.read_csv(RNA_KO_PATH, sep="\t")

# 2. Filter baseMean > 10 and drop NaNs in key columns
rna_filtered = rna[rna["baseMean"] > 10].dropna(
    subset=["log2FoldChange", "padj"]
)

# 3. Filter for downregulated genes (padj < 0.05 and log2FoldChange < -0.5)
downregulated_df = rna_filtered[
    (rna_filtered["padj"] < 0.05) & (rna_filtered["log2FoldChange"] < -0.5)
]

# 4. Extract unique, clean gene symbols
downregulated_genes = sorted(
    downregulated_df["GENESYMBOL"]
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
    .tolist()
)

print(f"Extracted downregulated gene count: {len(downregulated_genes)}")

In [ ]:
downregulated_genes = sorted(
    downregulated_df["GENESYMBOL"]
    .astype(str)
    .str.strip()
    .str.capitalize()
    .unique()
    .tolist()
)

In [ ]:
import os
import pandas as pd
import gseapy as gp

# ------------------------------------------------------------------------------
# 1. SETUP PATHS
# ------------------------------------------------------------------------------
RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
OUTPUT_DIR = "/content/drive/MyDrive/CHD8_GO_Analysis/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. FILTERING PARAMETERS — per -:
#    "adjusted P < 0.05, |log2FC| > 0.5; n = 1,618"
#    Since the list is described as "significantly downregulated," direction
#    is isolated (log2FoldChange < -0.5), not just |log2FC| > 0.5 which would
#    include upregulated genes too.
# ------------------------------------------------------------------------------
PADJ_CUT = 0.05
LFC_CUT = 0.5

# ------------------------------------------------------------------------------
# 3. BUILD DOWNREGULATED-ONLY DEG GENE LIST
# ------------------------------------------------------------------------------
print("1. Filtering CHD8-KO DEGs for significantly DOWNREGULATED genes...")

rna = pd.read_csv(RNA_KO_PATH, sep="\t")
rna_filtered = rna[rna["baseMean"] > 10].dropna(subset=["log2FoldChange", "padj"])

downregulated_df = rna_filtered[
    (rna_filtered["padj"] < PADJ_CUT) & (rna_filtered["log2FoldChange"] < -LFC_CUT)
]

down_genes = sorted(
    downregulated_df["GENESYMBOL"]
    .astype(str)
    .str.strip()
    .str.capitalize()  # mouse convention (Rpl32, not RPL32)
    .unique()
    .tolist()
)
print(f"   Downregulated DEG count: {len(down_genes)}")

background_genes = sorted(
    rna_filtered["GENESYMBOL"]
    .astype(str)
    .str.strip()
    .str.capitalize()
    .unique()
    .tolist()
)
print(f"   Background gene list length: {len(background_genes)}")

# ------------------------------------------------------------------------------
# 4. RUN GO ENRICHMENT (Enrichr, GO_Biological_Process_2023)
# ------------------------------------------------------------------------------
print("\n2. Executing GO Biological Process 2023 Enrichment Analysis...")

try:
    enr = gp.enrichr(
        gene_list=down_genes,
        gene_sets=["GO_Biological_Process_2023"],
        organism="mouse",
        background=background_genes,
        outdir=None,
        cutoff=1.0,
        verbose=False,
    )
    go_results = enr.results
except Exception as e:
    print(f"Background-enrichment call failed ({e}).")
    print("Falling back to Enrichr's default whole-genome background...")
    enr = gp.enrichr(
        gene_list=down_genes,
        gene_sets=["GO_Biological_Process_2023"],
        organism="mouse",
        outdir=None,
        cutoff=1.0,
        verbose=False,
    )
    go_results = enr.results

# ------------------------------------------------------------------------------
# 5. FILTER TO SIGNIFICANT TERMS
#    nominal P < 0.05 was used as the significance threshold."
#    -> filter on nominal P-value, NOT Adjusted P-value.
# ------------------------------------------------------------------------------
sig_go_results = go_results[go_results["P-value"] < 0.05].copy()
sig_go_results["input_n_genes"] = len(down_genes)

# ------------------------------------------------------------------------------
# 6. SAVE TABLE
# ------------------------------------------------------------------------------
output_file = os.path.join(OUTPUT_DIR, "Table_S5_GO_KO_Downregulated_REGENERATED.csv")
sig_go_results.to_csv(output_file, index=False)

print(f"\n✅ Analysis complete. {len(sig_go_results)} terms met nominal P < 0.05.")
print(f"   Input gene count: {len(down_genes)} (- states n = 1,618)")
print(f"   Results saved to: {output_file}")

In [ ]:
!apt-get install -y bedtools -qq > /dev/null
!pip install pybedtools -q

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pybedtools
import gseapy as gp
import warnings
warnings.filterwarnings('ignore')

# ── - rcParams ───────────────────────────────────────────────────
matplotlib.rcParams['font.family']       = 'sans-serif'
matplotlib.rcParams['font.sans-serif']   = ['Arial', 'Liberation Sans', 'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']      = 42
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5
matplotlib.rcParams['xtick.major.size']  = 2.5
matplotlib.rcParams['ytick.major.size']  = 2.5

# ── Paths ─────────────────────────────────────────────────────────────────────
PEAK_PATH   = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH    = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"   # NEW: needed for NPC-specific subtraction
TSS_PATH    = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
SAVE_DIR    = "/content/drive/MyDrive/Chd8 data/figures/"
MOTIF_DIR   = "/content/drive/MyDrive/CHD8_TF_Motif_Analysis/"
os.makedirs(MOTIF_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

DOWNLOADS = os.path.join(os.path.expanduser('~'), 'Downloads')
os.makedirs(DOWNLOADS, exist_ok=True)
TIFF_PATH = os.path.join(DOWNLOADS, 'SuppFig4_TF_Enrichment_GenomeBiology_Tier2_254.tiff')
PDF_PATH  = os.path.join(DOWNLOADS, 'SuppFig4_TF_Enrichment_GenomeBiology_Tier2_254.pdf')
GENE_LIST_OUT = os.path.join(MOTIF_DIR, 'Tier2_RNAxChIP_254genes.txt')

# ── Colours ───────────────────────────────────────────────────────────────────
PANEL_COLORS = ['#2166ac', '#4dac26', '#d6604d', '#f1a340', '#74add1']
C_NEUT       = '#333333'
C_REF        = '#bbbbbb'
C_SIG_LINE   = '#d6604d'

def shorten_db(db):
    return (db
            .replace('TF_Perturbations_Followed_by_Expression', 'TF Perturbations')
            .replace('ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X', 'ENCODE+ChEA')
            .replace('ENCODE_TF_ChIP-seq_2015', 'ENCODE ChIP-seq 2015')
            .replace('TRANSFAC_and_JASPAR_PWMs', 'TRANSFAC+JASPAR')
            .replace('ChEA_2022', 'ChEA 2022'))

def clean_term_label(term):
    """Extract just the TF symbol — first space-delimited token, uppercased."""
    t = term.split('(')[0].strip()
    label = t.split(' ')[0].split('_')[0].strip()
    return label.upper()[:20]

PADJ_CUT = 0.05
LFC_CUT  = 0.5

STANDARD_CHROMS = set([f'chr{i}' for i in range(1, 20)] + ['chrX', 'chrY'])

def is_standard(f):
    return str(f.chrom) in STANDARD_CHROMS

def fix_chr(f):
    if not str(f.chrom).startswith('chr'):
        f.chrom = 'chr' + str(f.chrom)
    return f

def format_mouse_gene(g):
    return g.strip().upper()

def enrichr_with_retry(gene_list, db, max_retries=4, backoff=15):
    for attempt in range(max_retries):
        try:
            enr = gp.enrichr(
                gene_list=gene_list, gene_sets=[db], organism='mouse',
                outdir=None, cutoff=1.0, verbose=False,
            )
            if enr.results is not None and not enr.results.empty:
                return enr.results.copy()
            return None
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'rate' in err_str.lower():
                wait = backoff * (2 ** attempt)
                print(f"   Rate-limited on {db}, waiting {wait}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                print(f"   {db} failed: {e}")
                return None
    print(f"   {db} exhausted retries.")
    return None

# ══════════════════════════════════════════════════════════════════════════
# STEP 1: Build the Tier 2 gene set — CHD8 NPC-SPECIFIC-bound ∩ significant DEG
# ══════════════════════════════════════════════════════════════════════════
print("1. Building Tier 2 gene set (RNA ∩ NPC-specific ChIP, no ATAC filter)...")

npc = pybedtools.BedTool(PEAK_PATH).each(fix_chr).filter(is_standard).sort().saveas()
esc = pybedtools.BedTool(ESC_PATH).each(fix_chr).filter(is_standard).sort().saveas()
tss = pybedtools.BedTool(TSS_PATH).each(fix_chr).filter(is_standard).sort().saveas()

npc_specific = npc.subtract(esc, A=True).saveas()

# Gene symbol confirmed at field index 3 (TSS bed: chrom,start,end,gene,score,strand)
bound = tss.intersect(npc_specific, u=True, wa=True)
bound_genes = set()
for f in bound:
    gene_col = f.fields[3] if len(f.fields) > 3 else None
    if gene_col and len(str(gene_col)) > 1:
        bound_genes.add(format_mouse_gene(gene_col))
print(f"   CHD8 NPC-specific bound genes (TSS-proximal): {len(bound_genes):,}")
# Expected ≈ 3,754 to match the confirmed diagnostic run

# 1b. RNA-seq DEGs at padj<0.05 & |log2FC|>0.5
rna = pd.read_csv(RNA_KO_PATH, sep='\t')
rna = rna[rna['baseMean'] > 10].dropna(subset=['log2FoldChange', 'padj'])


gene_col_name = 'GENESYMBOL'
padj_col_name = 'padj'
lfc_col_name  = 'log2FoldChange'

for _label, _col in [('gene', gene_col_name), ('padj', padj_col_name), ('log2FC', lfc_col_name)]:
    if _col not in rna.columns:
        raise KeyError(f"Expected {_label} column '{_col}' not found. Actual columns: {rna.columns.tolist()}")

rna_sig = rna[(rna[padj_col_name] < PADJ_CUT) & (rna[lfc_col_name].abs() > LFC_CUT)].copy()
rna_sig['gene_fmt'] = rna_sig[gene_col_name].astype(str).apply(format_mouse_gene)
deg_genes = set(rna_sig['gene_fmt'])
print(f"   Significant DEGs (padj<{PADJ_CUT}, |log2FC|>{LFC_CUT}): {len(deg_genes):,}")

tier2_genes = sorted(bound_genes & deg_genes)
print(f"   Tier 2 (NPC-specific CHD8-bound ∩ DEG) genes: {len(tier2_genes):,}")

if len(tier2_genes) == 0:
    raise RuntimeError("Tier 2 gene set came back empty — check inputs before proceeding.")

with open(GENE_LIST_OUT, 'w') as fh:
    fh.write('\n'.join(tier2_genes))
print(f"   ✅ Tier 2 gene list saved: {GENE_LIST_OUT}")

triple_genes = tier2_genes
# ══════════════════════════════════════════════════════════════════════════
# STEP 2: Run Enrichr fresh on the Tier 2 gene set
# ══════════════════════════════════════════════════════════════════════════
print("\n2. Running Enrichr on Tier 2 gene set...")

PLOT_DB_ORDER = [
    'TF_Perturbations_Followed_by_Expression',
    'ChEA_2022',
    'ENCODE_TF_ChIP-seq_2015',
    'ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X',
    'TRANSFAC_and_JASPAR_PWMs',
]

enrichr_results = {}
for db in PLOT_DB_ORDER:
    print(f"   Querying {db}...")
    res = enrichr_with_retry(triple_genes, db)
    if res is not None:
        res['-log10P']   = -np.log10(res['P-value'].clip(lower=1e-300))
        res['-log10FDR'] = -np.log10(res['Adjusted P-value'].clip(lower=1e-300))
        enrichr_results[db] = res
    time.sleep(2)

if not enrichr_results:
    print("   ⚠️ No Enrichr results returned for any database — check internet access / API status.")

# ══════════════════════════════════════════════════════════════════════════
# STEP 3: Build panel list — dedup by TF symbol, keep most significant row
# ══════════════════════════════════════════════════════════════════════════
print("\n3. Building figure panels...")

panels = []
for db in PLOT_DB_ORDER:
    if db not in enrichr_results:
        continue
    res = enrichr_results[db]
    sig     = res[res['Adjusted P-value'] < 0.05].copy()
    use_fdr = not sig.empty
    if use_fdr:
        sig = sig.nlargest(min(12, len(sig)), '-log10FDR').copy()
    else:
        sig = res.nsmallest(12, 'P-value').copy()

    sig['Label'] = sig['Term'].apply(clean_term_label)

    sig = (sig
           .sort_values('P-value', ascending=True)
           .drop_duplicates(subset='Label', keep='first')
           .sort_values('-log10P', ascending=True))

    panels.append({
        'db_short': shorten_db(db),
        'df':       sig,
        'use_fdr':  use_fdr,
        'color':    PANEL_COLORS[len(panels) % len(PANEL_COLORS)]
    })

# ══════════════════════════════════════════════════════════════════════════
# STEP 4: Figure
# ══════════════════════════════════════════════════════════════════════════
print("\n4. Generating figure...")

n_panels = max(len(panels), 1)

FIG_W = 6.693
FIG_H = max(3.0, n_panels * 1.8)
DPI   = 600

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=DPI)
fig.patch.set_facecolor('white')

if not panels:
    ax = fig.add_subplot(1, 1, 1)
    ax.text(0.5, 0.5, 'No Enrichr results available\nCheck internet connection',
            ha='center', va='center', fontsize=6,
            transform=ax.transAxes, color=C_NEUT)
    ax.set_axis_off()
else:
    gs = gridspec.GridSpec(
        n_panels, 1,
        figure=fig,
        left=0.28,
        right=0.97,
        top=0.94,
        bottom=0.04,
        hspace=0.90
    )

    panel_letters = 'abcde'

    for idx, panel in enumerate(panels):
        ax      = fig.add_subplot(gs[idx])
        df      = panel['df']
        color   = panel['color']
        use_fdr = panel['use_fdr']
        db_label = panel['db_short']

        ax.set_facecolor('white')

        bars = ax.barh(
            df['Label'],
            df['-log10P'],
            color=color,
            edgecolor='white',
            linewidth=0.4,
            height=0.55,
            alpha=0.88
        )

        x_max = df['-log10P'].max()
        ax.set_xlim(0, x_max * 1.55)
        x_right = ax.get_xlim()[1]

        if 'Overlap' in df.columns:
            label_positions = []
            for bar, (_, row) in zip(bars, df.iterrows()):
                raw_x = bar.get_width() + x_max * 0.025
                raw_y = bar.get_y() + bar.get_height() / 2
                label_positions.append((raw_x, raw_y, str(row['Overlap'])))

            MIN_GAP = 0.38
            adjusted_y = [pos[1] for pos in label_positions]
            for i in range(1, len(adjusted_y)):
                if abs(adjusted_y[i] - adjusted_y[i-1]) < MIN_GAP:
                    mid = (adjusted_y[i] + adjusted_y[i-1]) / 2
                    adjusted_y[i-1] = mid - MIN_GAP / 2
                    adjusted_y[i]   = mid + MIN_GAP / 2

            for (raw_x, _, txt), adj_y in zip(label_positions, adjusted_y):
                if raw_x < x_right * 0.97:
                    ax.text(
                        raw_x, adj_y, txt,
                        va='center', ha='left',
                        fontsize=4.5, color=C_NEUT,
                        clip_on=True
                    )

        ax.axvline(-np.log10(0.05), color=C_SIG_LINE,
                   linestyle='--', lw=0.75, label='p = 0.05', zorder=0)

        if not use_fdr:
            ax.text(0.99, 0.98,
                    'Top 12 nominal p (FDR not reached)',
                    transform=ax.transAxes, fontsize=3.5,
                    ha='right', va='top',
                    color=C_REF, style='italic')

        ax.set_xlabel(r'$-\log_{10}(p)$', fontsize=5.5, labelpad=2)
        ax.set_title(db_label, fontsize=6, fontweight='bold',
                     pad=3, color=color, loc='left')
        ax.tick_params(axis='y', labelsize=5, length=0, pad=4)
        ax.tick_params(axis='x', labelsize=5, length=2.5,
                       width=0.75, pad=2)
        ax.margins(y=0.10)

        ax.legend(fontsize=4, loc='lower right', frameon=True,
                  framealpha=0.9, edgecolor=C_REF,
                  handlelength=0.8, handletextpad=0.3,
                  borderpad=0.3)

        ax.text(-0.26, 1.05, panel_letters[idx],
                transform=ax.transAxes,
                fontsize=8, fontweight='bold', va='top', ha='left')

        sns.despine(ax=ax, left=True, offset=2, trim=True)
        ax.spines['left'].set_visible(False)

fig.text(0.5, 0.995,
         f'Supplementary Figure 4: TF co-binding enrichment at Tier 2 '
         f'CHD8-bound/DEG loci (NPC, n={len(triple_genes)})',
         ha='center', va='top',
         fontsize=6.5, fontweight='bold', color=C_NEUT)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig(TIFF_PATH, dpi=DPI, bbox_inches='tight',
            format='tiff', facecolor='white',
            pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PDF_PATH,  dpi=DPI, bbox_inches='tight',
            format='pdf', backend='pdf', facecolor='white')
plt.show()

for path, lbl in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb     = os.path.getsize(path) / 1e6
    status = '✅ within 10 MB' if mb < 10 else '⚠️ EXCEEDS 10 MB limit'
    print(f"✅ {lbl} : {path}  ({mb:.1f} MB  {status})")

for db, res in enrichr_results.items():
    out_path = os.path.join(
        MOTIF_DIR,
        f"TF_Enrichr_Tier2_254_{db.replace('-','_').replace(' ','_')}.csv"
    )
    res.to_csv(out_path, index=False)
    print(f"✅ Table saved: {out_path}")

try:
    from google.colab import files as colab_files
    colab_files.download(TIFF_PATH)
    colab_files.download(PDF_PATH)
    print("\n📥 Downloads triggered.")
except ImportError:
    print("\n📁 Files written to ~/Downloads.")

print("\n" + "=" * 65)
print("SUPPLEMENTARY FIGURE 4 (TIER 2, n=254) COMPLETE")
print("=" * 65)
print(f"   Tier 2 genes analysed : {len(triple_genes):,}")
for db, res in enrichr_results.items():
    n_fdr = (res['Adjusted P-value'] < 0.05).sum()
    n_nom = (res['P-value'] < 0.05).sum()
    print(f"   {shorten_db(db):<40} "
          f"{n_fdr} FDR<0.05 | {n_nom} nominal p<0.05")

In [ ]:
import os

table_paths = []
for db, res in enrichr_results.items():
    out_path = os.path.join(
        MOTIF_DIR,
        f"TF_Enrichr_Tier2_254_{db.replace('-','_').replace(' ','_')}.csv"
    )
    table_paths.append(out_path)

# Also grab the Tier 2 gene list itself
table_paths.append(GENE_LIST_OUT)

try:
    from google.colab import files as colab_files
    for path in table_paths:
        if os.path.exists(path):
            colab_files.download(path)
            print(f"📥 Downloaded: {path}")
        else:
            print(f"⚠️ Not found, skipping: {path}")
except ImportError:
    print("Not running in Colab — files are already saved to:")
    for path in table_paths:
        print(f"  {path}")

In [ ]:
import pybedtools

PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"

def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

chip_peaks = pybedtools.BedTool(PEAK_PATH).each(fix_naming).saveas()
tss        = pybedtools.BedTool(TSS_PATH).each(fix_naming).saveas()

print("=== Raw TSS file, first 3 lines ===")
for i, f in enumerate(tss):
    print(f.fields)
    if i >= 2:
        break

print("\n=== Raw peak file, first 3 lines ===")
for i, f in enumerate(chip_peaks):
    print(f.fields)
    if i >= 2:
        break

print("\n=== TSS ∩ peaks, wb=True, first 5 matches ===")
bound = tss.intersect(chip_peaks, wb=True)
n_records = 0
for i, f in enumerate(bound):
    print(f"record {i}: {f.fields}")
    n_records += 1
    if i >= 4:
        break

print(f"\nTotal intersect records (may include duplicates per TSS): "
      f"{len(bound)}")
print(f"TSS file has {len(tss)} entries; peak file has {len(chip_peaks)} entries.")

# Also try the reverse orientation, in case peaks-as-A / TSS-as-B gives a
# cleaner field layout for your annotation file
print("\n=== peaks ∩ TSS (reversed), wb=True, first 5 matches ===")
bound_rev = chip_peaks.intersect(tss, wb=True)
for i, f in enumerate(bound_rev):
    print(f"record {i}: {f.fields}")
    if i >= 4:
        break

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import pybedtools

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────────
DESEQ2_DIR = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
DESEQ2_FILES = {
    "KO":  DESEQ2_DIR + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    "KD1": DESEQ2_DIR + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv",
    "KD2": DESEQ2_DIR + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv",
}

PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
# same KO DESeq2 table is reused for the target-overlap step

output_dir = "/content/Figure2_output"
os.makedirs(output_dir, exist_ok=True)

DPI = 300
plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

# ══════════════════════════════════════════════════════════════════════════
# PART 1 — FIGURE 2A: three-panel volcano (KO / KD1 / KD2), corrected cutoff
# ══════════════════════════════════════════════════════════════════════════

def clean_df(path):
    df = pd.read_csv(path, sep='\t')
    gene_col = next((c for c in df.columns
                      if c.upper() in ['GENESYMBOL', 'SYMBOL', 'GENE', 'MGI_SYMBOL']),
                     'GENESYMBOL')
    df = df.rename(columns={gene_col: 'symbol'})
    df = df.dropna(subset=['symbol', 'log2FoldChange', 'padj'])
    if 'baseMean' in df.columns:
        n_before = len(df)
        df = df[df['baseMean'] > 10].copy()
        print(f"   [{os.path.basename(path)}] baseMean > 10 filter: "
              f"{n_before:,} -> {len(df):,} genes")
    df['padj'] = df['padj'].replace(0, 1e-300)
    return df

print("=" * 70)
print("PART 1 — FIGURE 2A: DEG counts (corrected LFC threshold)")
print("=" * 70)

df_ko  = clean_df(DESEQ2_FILES["KO"])
df_kd1 = clean_df(DESEQ2_FILES["KD1"])
df_kd2 = clean_df(DESEQ2_FILES["KD2"])

fdr_cutoff = 0.05
lfc_cutoff = 0.5

def categorize(row):
    if row['padj'] >= fdr_cutoff:            return 'NS'
    if row['log2FoldChange'] < -lfc_cutoff:  return 'Down'
    if row['log2FoldChange'] >  lfc_cutoff:  return 'Up'
    return 'NS'

for df in [df_ko, df_kd1, df_kd2]:
    df['category'] = df.apply(categorize, axis=1)

print("\nAUDIT — copy these into the - (Abstract / Results / Methods):")
for name, df in [("KO", df_ko), ("KD1", df_kd1), ("KD2", df_kd2)]:
    n_down = (df['category'] == 'Down').sum()
    n_up   = (df['category'] == 'Up').sum()
    print(f"   {name}: {n_down:,} down / {n_up:,} up "
          f"(total DEGs = {n_down + n_up:,})")

# Flag the ±1 duplicate-symbol issue explicitly (4933434E20Rik case)
dupe_check = df_ko[df_ko.duplicated('symbol', keep=False)].sort_values('symbol')
if not dupe_check.empty:
    print(f"\n   NOTE: {dupe_check['symbol'].nunique()} gene symbol(s) map to "
          f"multiple rows in KO table (e.g. duplicate Ensembl IDs).")
    print(f"   This is the source of the 1,134 vs 1,135 / 2,752 vs 2,753 "
          f"discrepancy — decide once whether the - reports row-level")
    print(f"   counts (2,753) or symbol-deduplicated counts (2,752), then use "
          f"the SAME convention everywhere (Abstract, Results, Fig 2, Fisher's test).")
    print(dupe_check[['symbol', 'log2FoldChange', 'padj']].to_string(index=False))

# ── Figure 2A plot ────────────────────────────────────────────────────────
colors = {'NS': '#d3d3d3', 'Down': '#2E86AB', 'Up': '#A23B72'}
titles = ['CHD8 KO vs WT', 'CHD8 KD 8.1 vs Scramble', 'CHD8 KD 8.2 vs Scramble']

FIG2A_WIDTH_IN  = 170 / 25.4
FIG2A_HEIGHT_IN = 70  / 25.4

fig_a, axes = plt.subplots(1, 3, figsize=(FIG2A_WIDTH_IN, FIG2A_HEIGHT_IN))

for i, (df, title) in enumerate(zip([df_ko, df_kd1, df_kd2], titles)):
    ax = axes[i]
    for cat in ['NS', 'Up', 'Down']:
        subset = df[df['category'] == cat]
        ax.scatter(
            subset['log2FoldChange'], -np.log10(subset['padj']),
            c=colors[cat], s=2, alpha=0.6,
            label=f"{cat} (n={len(subset):,})", rasterized=True,
        )
    ax.axhline(-np.log10(fdr_cutoff), color='black', linestyle='--', lw=0.5)
    ax.axvline(-lfc_cutoff, color='black', linestyle='--', lw=0.5)
    ax.axvline( lfc_cutoff, color='black', linestyle='--', lw=0.5)
    ax.set_title(title, fontsize=6, fontweight='bold', loc='left', pad=3)
    ax.set_xlabel('log$_2$ Fold Change', fontsize=6)
    ax.tick_params(labelsize=5)
    if i == 0:
        ax.set_ylabel('-log$_{10}$ Adjusted P-value', fontsize=6)
    ax.legend(frameon=False, loc='upper right', fontsize=5,
              markerscale=1.5, handletextpad=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout(pad=0.4, w_pad=0.8)

fig2a_pdf  = os.path.join(output_dir, "Figure2A.pdf")
fig2a_tiff = os.path.join(output_dir, "Figure2A.tiff")
fig_a.savefig(fig2a_pdf,  format='pdf',  bbox_inches='tight', pad_inches=0.02)
fig_a.savefig(fig2a_tiff, format='tiff', dpi=DPI, bbox_inches='tight',
              pad_inches=0.02, pil_kwargs={"compression": "tiff_lzw"})
plt.close(fig_a)
print(f"\n✅ Figure 2A saved: {fig2a_pdf}")

# ══════════════════════════════════════════════════════════════════════════
# PART 2 — FIGURE 2B: CHD8 direct-target overlay + ECDF, with a diagnostic
#           block
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("PART 2 — FIGURE 2B: direct-target overlap + diagnostics")
print("=" * 70)

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

print("\n1. Identifying NPC-specific ChIP targets (peak-level, ±2kb TSS)...")
npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()

npc_specific = npc_bt.subtract(esc_bt, A=True)
hits = tss_bt.intersect(npc_specific, u=True, wa=True)

# Raw gene names straight off the TSS bed intersection, BEFORE any
# normalization — this is the variable most likely to differ between scripts
raw_target_names = [str(f[3]).strip() for f in hits if len(str(f[3])) > 1]
print(f"   Raw TSS-intersect hits (before any symbol normalization): "
      f"{len(raw_target_names):,} rows, {len(set(raw_target_names)):,} unique raw strings")

# ── Reuse the already-cleaned KO table from Part 1 (baseMean>10 already applied) ──
de_df = df_ko.copy()  # 'symbol' column, baseMean>10 already filtered, from Part 1

print(f"\n2. RNA-seq universe for matching: {len(de_df):,} genes "
      f"(baseMean > 10, from Part 1's df_ko)")

# ── DIAGNOSTIC: run the target-list construction 4 different ways ─────────
print("\n3. Diagnostic — target-overlap count under different symbol conventions:")

def match_count(target_names, de_df, label, dedup_keep=None):
    """Match target_names against de_df['symbol'] and report counts."""
    de = de_df.copy()
    de['gene_key'] = de['symbol'].astype(str).str.upper().str.strip()
    targets_upper = set(str(t).upper().strip() for t in target_names)

    if dedup_keep == 'first':
        de = de.drop_duplicates('gene_key', keep='first')
    elif dedup_keep == 'highest_baseMean' and 'baseMean' in de.columns:
        de = de.sort_values('baseMean', ascending=False).drop_duplicates('gene_key', keep='first')

    de['is_target'] = de['gene_key'].isin(targets_upper)
    n = de['is_target'].sum()
    print(f"   [{label:35s}] target genes matched in RNA-seq table: {n:,}")
    return de, n

# (a) raw target list, no symbol dedup on RNA-seq side
de_a, n_a = match_count(raw_target_names, de_df, "raw names, no RNA-seq dedup")

# (b) unique target set (set() applied before matching, as target-ID script did)
de_b, n_b = match_count(list(set(raw_target_names)), de_df, "unique target names, no dedup")

# (c) unique target set, RNA-seq symbols deduplicated (keep first row)
de_c, n_c = match_count(list(set(raw_target_names)), de_df, "unique targets, RNA-seq dedup=first", dedup_keep='first')

# (d) unique target set, RNA-seq symbols deduplicated by highest baseMean
#     (keeps the "-"/dominant transcript when one symbol maps to 2+ Ensembl IDs
de_d, n_d = match_count(list(set(raw_target_names)), de_df, "unique targets, RNA-seq dedup=highest baseMean", dedup_keep='highest_baseMean')

print(f"\n   >>> Compare these four numbers to 1,496 and 1,478.")
print(f"   >>> Whichever convention reproduces 1,496 is what the original target-ID")
print(f"       script used; whichever reproduces 1,478 is what the Supp Fig 5 /")
print(f"       rescue-fraction script used. Standardize the - on ONE.")

# ── Use the pipeline-standard convention (c: unique targets, symbol-deduped
#    RNA-seq universe)
de_df = de_c
target_list = list(set(raw_target_names))

all_targets = de_df[de_df['is_target']]
sig_targets = all_targets[all_targets['padj'] < 0.05]
others      = de_df[~de_df['is_target']]
targets_lfc = all_targets['log2FoldChange']
others_lfc  = others['log2FoldChange']

_, pval = mannwhitneyu(targets_lfc, others_lfc, alternative='two-sided')
effect  = targets_lfc.median() - others_lfc.median()

print("\n" + "-" * 70)
print("AUDIT — copy into - (using de-duplicated convention, variant c):")
print(f"   Targets in RNA-seq         : {len(all_targets):,}")
print(f"   FDR-significant targets    : {len(sig_targets):,}")
print(f"   MWU p-value                : {pval:.3e}")
print(f"   Δmedian LFC                : {effect:.4f}")
print("-" * 70)

if pval < 0.05:
    mwu_label, target_col = f"MWU p = {pval:.2e}", '#D35400'
else:
    mwu_label, target_col = f"MWU p = {pval:.2e} (ns)", '#7F8C8D'

# ── Figure 2B plot (volcano + ECDF) ─────────────────────────────────────
FIG2B_WIDTH_IN  = 170 / 25.4
FIG2B_HEIGHT_IN = 80  / 25.4

fig_b, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(FIG2B_WIDTH_IN, FIG2B_HEIGHT_IN))

ax_a.scatter(others['log2FoldChange'], -np.log10(others['padj']),
             s=1.5, color='#CACFD2', alpha=0.25, rasterized=True,
             label=f'Non-targets (n={len(others):,})')
ax_a.scatter(all_targets['log2FoldChange'], -np.log10(all_targets['padj']),
             s=5, color='#E67E22', alpha=0.5, rasterized=True,
             label=f'CHD8 targets (n={len(all_targets):,})')
ax_a.scatter(sig_targets['log2FoldChange'], -np.log10(sig_targets['padj']),
             s=10, color='#D35400', alpha=0.95, edgecolors='white',
             linewidths=0.3, rasterized=True,
             label=f'FDR sig. targets (n={len(sig_targets):,})')

lim = np.ceil(max(abs(de_df['log2FoldChange'].quantile(0.005)),
                   abs(de_df['log2FoldChange'].quantile(0.995))) * 1.1)
ax_a.set_xlim(-lim, lim)
ax_a.set_ylim(0, None)
ax_a.axvline(0, color='black', lw=0.5)
ax_a.axhline(-np.log10(0.05), color='#7F8C8D', lw=0.5, ls='--', label='FDR = 0.05')
ax_a.set_title('CHD8 direct targets, KO vs WT', fontsize=6, fontweight='bold', loc='left', pad=4)
ax_a.set_xlabel('log$_2$ Fold Change', fontsize=6)
ax_a.set_ylabel('-log$_{10}$ Adjusted P-value', fontsize=6)
ax_a.tick_params(labelsize=5, width=0.5, length=2)
ax_a.legend(frameon=False, loc='upper right', fontsize=5, markerscale=2, handletextpad=0.3)
ax_a.spines[['top', 'right']].set_visible(False)
ax_a.spines[['left', 'bottom']].set_linewidth(0.5)

sns.ecdfplot(data=others_lfc.values, color='#95A5A6', lw=1.2, linestyle='--',
             label=f'Non-targets (n={len(others_lfc):,})', ax=ax_b)
sns.ecdfplot(data=targets_lfc.values, color=target_col, lw=1.8, linestyle='-',
             label=f'CHD8 targets (n={len(targets_lfc):,})', ax=ax_b)
ax_b.axvline(0, color='black', ls='--', lw=0.5, alpha=0.5)

stat_text = f"{mwu_label}\nΔmedian = {effect:.3f}"
ax_b.text(0.04, 0.96, stat_text, transform=ax_b.transAxes, va='top',
          fontsize=5, linespacing=1.4,
          bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                     edgecolor='#CACFD2', linewidth=0.5, alpha=0.9))
ax_b.set_title('LFC distribution: targets vs non-targets', fontsize=6,
               fontweight='bold', loc='left', pad=4)
ax_b.set_xlabel('log$_2$ Fold Change', fontsize=6)
ax_b.set_ylabel('Cumulative proportion', fontsize=6)
ax_b.tick_params(labelsize=5, width=0.5, length=2)
ax_b.legend(frameon=False, loc='lower right', fontsize=5, handletextpad=0.3)
ax_b.spines[['top', 'right']].set_visible(False)
ax_b.spines[['left', 'bottom']].set_linewidth(0.5)

plt.tight_layout(pad=0.5, w_pad=1.2, h_pad=0)

fig2b_pdf  = os.path.join(output_dir, "Figure2B.pdf")
fig2b_tiff = os.path.join(output_dir, "Figure2B.tiff")
fig_b.savefig(fig2b_pdf,  format='pdf', pad_inches=0.02)
fig_b.savefig(fig2b_tiff, format='tiff', dpi=DPI, pad_inches=0.02,
              pil_kwargs={"compression": "tiff_lzw"})
plt.close(fig_b)
print(f"\n✅ Figure 2B saved: {fig2b_pdf}")

# ══════════════════════════════════════════════════════════════════════════
# PART 3 — Rescue-fraction master universe (Figure 7 pipeline)
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("PART 3 — Rescue-fraction master universe (KO ∩ FL ∩ ΔC ∩ ΔH)")
print("=" * 70)

RESCUE_FILES = {
    "KO": DESEQ2_FILES["KO"],  # reuse same path from Part 1
    "FL": DESEQ2_DIR + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    "dC": DESEQ2_DIR + "Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    "dH": DESEQ2_DIR + "Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}

def find_ensembl_col(df):
    """Locate the Ensembl Gene ID column by name pattern or by content
    (ENSMUSG-style IDs), without assuming a fixed column name."""
    name_candidates = [c for c in df.columns
                        if c.upper() in ('GENEID', 'GENE_ID', 'ENSEMBL_ID',
                                          'ENSEMBLID', 'ENSEMBL_GENE_ID',
                                          'ID', 'ROW.NAMES', 'ROWNAMES')]
    if name_candidates:
        return name_candidates[0]
    # fall back: scan columns for ENSMUSG-pattern values
    for c in df.columns:
        sample = df[c].astype(str).head(20)
        if sample.str.match(r'^ENSMUSG\d+').any():
            return c
    return None

def find_symbol_col(df):
    return next((c for c in df.columns
                 if c.upper() in ['GENESYMBOL', 'SYMBOL', 'GENE', 'MGI_SYMBOL']),
                None)

def load_rescue_table(path, tag):
    df = pd.read_csv(path, sep='\t')
    print(f"\n   [{tag}] loaded {path.split('/')[-1]} — {len(df):,} rows, "
          f"columns: {list(df.columns)}")
    ens_col = find_ensembl_col(df)
    sym_col = find_symbol_col(df)
    if ens_col is None:
        print(f"   ⚠ [{tag}] Could not auto-detect an Ensembl ID column. "
              f"Inspect the column list above and hard-code it manually "
              f"(edit `find_ensembl_col` or pass ens_col explicitly).")
    else:
        print(f"   [{tag}] using '{ens_col}' as Ensembl Gene ID column"
              + (f", '{sym_col}' as gene symbol" if sym_col else ""))
    keep_cols = [c for c in [ens_col, sym_col, 'baseMean', 'log2FoldChange', 'padj'] if c]
    out = df[keep_cols].copy()
    rename_map = {'log2FoldChange': f'LFC_{tag}', 'baseMean': f'baseMean_{tag}',
                  'padj': f'padj_{tag}'}
    if ens_col: rename_map[ens_col] = 'gene_id'
    if sym_col: rename_map[sym_col] = 'symbol'
    out = out.rename(columns=rename_map)
    return out

tbl_ko = load_rescue_table(RESCUE_FILES["KO"], "KO")
tbl_fl = load_rescue_table(RESCUE_FILES["FL"], "FL")
tbl_dc = load_rescue_table(RESCUE_FILES["dC"], "dC")
tbl_dh = load_rescue_table(RESCUE_FILES["dH"], "dH")

if 'gene_id' not in tbl_ko.columns:
    print("\n   ⚠ STOPPING PART 3: no Ensembl ID column was found in the KO table.")
    print("     Paste the printed column list above back to Claude so the join key")
    print("     can be hard-coded correctly, then rerun this section.")
else:
    # ── Build master gene universe: inner-join all 4 tables on Ensembl ID ──
    master = tbl_ko.merge(tbl_fl[['gene_id', 'LFC_FL', 'baseMean_FL']], on='gene_id', how='left') \
                    .merge(tbl_dc[['gene_id', 'LFC_dC', 'baseMean_dC']], on='gene_id', how='left') \
                    .merge(tbl_dh[['gene_id', 'LFC_dH', 'baseMean_dH']], on='gene_id', how='left') \
                    .rename(columns={'LFC_KO': 'LFC_KO'})
    # LFC_KO comes from tbl_ko's own column rename (log2FoldChange -> LFC_KO)
    if 'LFC_KO' not in master.columns and 'log2FoldChange' in tbl_ko.columns:
        master = master.rename(columns={'log2FoldChange': 'LFC_KO'})

    print(f"\n   Master table before dropna: {len(master):,} rows")
    master_clean = master.dropna(subset=['LFC_KO', 'LFC_FL', 'LFC_dC', 'LFC_dH']).copy()
    print(f"   Master table after dropna on LFC_KO/FL/dC/dH: {len(master_clean):,} rows")

    # ── Intersect master universe with the 3,754 NPC-specific ChIP targets ──
    master_clean['gene_key'] = master_clean['symbol'].astype(str).str.upper().str.strip()
    targets_upper = set(str(t).upper().strip() for t in target_list)  # from Part 2, 3,754 unique raw names
    master_clean['is_target'] = master_clean['gene_key'].isin(targets_upper)

    n_master_targets = master_clean['is_target'].sum()
    print(f"\n   >>> NPC-specific CHD8 targets present in the 4-way master universe: "
          f"{n_master_targets:,}")
    print(f"   >>> Compare this to 1,478 (Supp Fig 5 / rescue-section number).")

    # ── Figure 7A/B universe: additionally require |LFC_KO| > 0.5 ──────────
    rescue_universe = master_clean[master_clean['is_target'] & (master_clean['LFC_KO'].abs() > 0.5)].copy()
    print(f"\n   Figure 7A/B rescue-eligible universe "
          f"(target genes with |LFC_KO| > 0.5 in 4-way master): {len(rescue_universe):,} genes")

    # ── Compute rescue fractions, clipped to [-1, 2] per Methods ────────────
    for construct in ['FL', 'dC', 'dH']:
        rescue_universe[f'RF_{construct}'] = (
            (rescue_universe[f'LFC_{construct}'] - rescue_universe['LFC_KO'])
            / (0 - rescue_universe['LFC_KO'])
        ).clip(-1, 2)

    print("\n   Rescue fraction summary (median, on the master-universe-filtered set):")
    for construct, label in [('FL', 'Full-length'), ('dC', 'ΔChromo'), ('dH', 'ΔHelicase')]:
        med = rescue_universe[f'RF_{construct}'].median()
        pct_rescued = (rescue_universe[f'RF_{construct}'] >= 0.5).mean() * 100
        print(f"      {label:12s}: median RF = {med:.3f}  |  "
              f"{pct_rescued:.1f}% of genes rescued (RF ≥ 0.5)")

    print("\n   Compare these median RF / %rescued values against the Abstract claims")
    print("   (~1.0% ΔChromo-rescued, ~41.1% ΔHelicase, ~70% full-length) and against")
    print("   Figure 7A/B — if they don't match, the -'s Fig 7 numbers were")
    print("   likely computed on a different universe (e.g. all 3,754 targets with")
    print("   |LFC_KO|>0.5, WITHOUT the 4-way master-table intersection) — worth testing")
    print("   as a follow-up variant if this one doesn't reconcile.")

    # ── PART 3b — retry with baseMean>10 actually applied (bug fix) ────────
    print("\n" + "-" * 70)
    print("PART 3b — same master universe, with baseMean>10 correctly applied")
    print("-" * 70)

    # Variant 1: baseMean > 10 in KO only (matches Part 1/2's filter exactly)
    master_bm_ko = master_clean[master_clean['baseMean_KO'] > 10].copy() \
        if 'baseMean_KO' in master_clean.columns else None
    if master_bm_ko is not None:
        n1 = master_bm_ko['is_target'].sum()
        print(f"   Variant 1 — baseMean_KO > 10 only: "
              f"{len(master_bm_ko):,} genes in universe, "
              f"{n1:,} are NPC-specific CHD8 targets")
    else:
        print("   Variant 1 skipped: 'baseMean_KO' column not found after merge "
              "(check load_rescue_table's baseMean_{tag} naming).")

    # Variant 2: baseMean > 10 required in ALL FOUR tables (stricter)
    bm_cols = [c for c in ['baseMean_KO', 'baseMean_FL', 'baseMean_dC', 'baseMean_dH']
               if c in master_clean.columns]
    if len(bm_cols) == 4:
        mask_all4 = (master_clean[bm_cols] > 10).all(axis=1)
        master_bm_all4 = master_clean[mask_all4].copy()
        n2 = master_bm_all4['is_target'].sum()
        print(f"   Variant 2 — baseMean > 10 in ALL FOUR tables: "
              f"{len(master_bm_all4):,} genes in universe, "
              f"{n2:,} are NPC-specific CHD8 targets")
    else:
        print(f"   Variant 2 skipped: only found baseMean columns {bm_cols} "
              f"(need all 4 — check merge step included baseMean_FL/dC/dH).")

    print("\n   >>> Whichever of Variant 1 / Variant 2 lands on 1,478 is the correct")
    print("       universe definition for the Supp Fig 5 / Figure 7 direct-target count.")

    # ── PART 3c — test boundary condition: baseMean >= 10 (inclusive) ──────
    print("\n" + "-" * 70)
    print("PART 3c — boundary check: baseMean >= 10 (inclusive) instead of > 10")
    print("-" * 70)

    master_bm_ge = master_clean[master_clean['baseMean_KO'] >= 10].copy() \
        if 'baseMean_KO' in master_clean.columns else None
    if master_bm_ge is not None:
        n3 = master_bm_ge['is_target'].sum()
        print(f"   baseMean_KO >= 10 (inclusive): "
              f"{len(master_bm_ge):,} genes in universe, "
              f"{n3:,} are NPC-specific CHD8 targets")
        boundary_genes = master_clean[
            (master_clean['baseMean_KO'] >= 9.5) & (master_clean['baseMean_KO'] <= 10.5)
        ][['symbol', 'baseMean_KO', 'is_target']].sort_values('baseMean_KO')
        if not boundary_genes.empty:
            print(f"\n   Genes with baseMean_KO near the 10.0 boundary "
                  f"(9.5–10.5 range) — these are what >= vs > would flip:")
            print(boundary_genes.to_string(index=False))
    else:
        print("   Skipped: 'baseMean_KO' column not found.")

    print("\n   If 1,478 still isn't hit exactly, the 2-gene residual is most likely")
    print("   an Ensembl-ID-to-symbol mapping edge case (e.g. a gene present under")
    print("   two symbol spellings, or a target-list gene matched under a synonym")
    print("   in the original pipeline but not here) — not worth chasing further;")
    print("   1,476 is close enough to confidently identify the correct methodology")
    print("   (4-way baseMean>10 intersection), and the - can be updated to")
    print("   report the reproducible value (1,476) with this Methods description.")

    # ══════════════════════════════════════════════════════════════════════
    # PART 4 — Recompute
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("PART 4 — Recompute Supp Fig 5 rescue-rate claim (857/1,478 = 58.0%)")
    print("=" * 70)

    universe = master_bm_ko.copy()  # 13,386 genes, baseMean>10 in all 4 tables
    universe['RF_FL'] = (
        (universe['LFC_FL'] - universe['LFC_KO']) / (0 - universe['LFC_KO'])
    ).replace([np.inf, -np.inf], np.nan).clip(-1, 2)

    direct_targets = universe[universe['is_target']].copy()
    non_targets    = universe[~universe['is_target']].copy()

    direct_rescued = (direct_targets['RF_FL'] >= 0.5).sum()
    direct_total   = direct_targets['RF_FL'].notna().sum()
    nontarget_rescued = (non_targets['RF_FL'] >= 0.5).sum()
    nontarget_total   = non_targets['RF_FL'].notna().sum()

    print(f"\n   Direct targets (corrected universe): {direct_total:,} genes with valid RF, "
          f"{direct_rescued:,} rescued (RF≥0.5) = {100*direct_rescued/direct_total:.1f}%")
    print(f"   Non-target genes (same universe):     {nontarget_total:,} genes with valid RF, "
          f"{nontarget_rescued:,} rescued (RF≥0.5) = {100*nontarget_rescued/nontarget_total:.1f}%")
    print(f"\n   - claims: 857/1,478 = 58.0% direct; 57.1% indirect (n=11,951)")
    print(f"   >>> Compare {direct_total:,} to 1,478/1,476, {nontarget_total:,} to 11,951,")
    print(f"       and {direct_rescued:,} to 857.")

    from scipy.stats import fisher_exact
    table_2x2 = [[direct_rescued, direct_total - direct_rescued],
                 [nontarget_rescued, nontarget_total - nontarget_rescued]]
    odds_ratio, p_val = fisher_exact(table_2x2)
    print(f"\n   Recomputed Fisher's exact test: OR = {odds_ratio:.3f}, p = {p_val:.3f}")
    print(f"   - claims: OR = 1.04, p = 0.272 — compare directly.")

    # ══════════════════════════════════════════════════════════════════════
    # PART 5 — KO/KD directional DEG count
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("PART 5 — KO/KD directional DEG count anomaly (raw vs shrunk LFC test)")
    print("=" * 70)

    def load_full_table(path, tag):
        df = pd.read_csv(path, sep='\t')
        gene_col = next((c for c in df.columns
                          if c.upper() in ['GENESYMBOL', 'SYMBOL', 'GENE', 'MGI_SYMBOL']),
                         'GENESYMBOL')
        df = df.rename(columns={gene_col: 'symbol'})
        has_raw = 'rawlog2FoldChange' in df.columns
        print(f"   [{tag}] {len(df):,} raw rows; has 'rawlog2FoldChange' column: {has_raw}")
        return df, has_raw

    df_ko_full, ko_has_raw   = load_full_table(DESEQ2_FILES["KO"],  "KO")
    df_kd2_full, kd2_has_raw = load_full_table(DESEQ2_FILES["KD2"], "KD2")

    def count_variants(df, tag, has_raw, target_down, target_up):
        df = df.dropna(subset=['symbol', 'log2FoldChange', 'padj']).copy()
        if 'baseMean' in df.columns:
            df = df[df['baseMean'] > 10].copy()
        df['padj'] = df['padj'].replace(0, 1e-300)

        print(f"\n   [{tag}] target: {target_down:,} down / {target_up:,} up")

        # Variant A: shrunk LFC, |LFC|>0.5, padj<0.05
        sig = df[df['padj'] < 0.05]
        a_down = (sig['log2FoldChange'] < -0.5).sum()
        a_up   = (sig['log2FoldChange'] >  0.5).sum()
        print(f"   [{tag}] Variant A — shrunk LFC, |LFC|>0.5, padj<0.05: "
              f"{a_down:,} down / {a_up:,} up")

        # Variant B: shrunk LFC, padj<0.05
        b_down = (sig['log2FoldChange'] < 0).sum()
        b_up   = (sig['log2FoldChange'] > 0).sum()
        print(f"   [{tag}] Variant B — shrunk LFC, padj<0.05 only (no |LFC| cutoff): "
              f"{b_down:,} down / {b_up:,} up")

        if has_raw:
            df['rawlog2FoldChange'] = pd.to_numeric(df['rawlog2FoldChange'], errors='coerce')
            sig_raw = df.dropna(subset=['rawlog2FoldChange'])
            sig_raw = sig_raw[sig_raw['padj'] < 0.05]

            # Variant C: RAW (unshrunk) LFC, |rawLFC|>0.5, padj<0.05
            c_down = (sig_raw['rawlog2FoldChange'] < -0.5).sum()
            c_up   = (sig_raw['rawlog2FoldChange'] >  0.5).sum()
            print(f"   [{tag}] Variant C — RAW LFC, |rawLFC|>0.5, padj<0.05: "
                  f"{c_down:,} down / {c_up:,} up  <-- test this against target")

            # Variant D: RAW LFC, padj<0.05 only, no magnitude cutoff
            d_down = (sig_raw['rawlog2FoldChange'] < 0).sum()
            d_up   = (sig_raw['rawlog2FoldChange'] > 0).sum()
            print(f"   [{tag}] Variant D — RAW LFC, padj<0.05 only (no cutoff): "
                  f"{d_down:,} down / {d_up:,} up")
        else:
            print(f"   [{tag}] Variants C/D skipped — no 'rawlog2FoldChange' column found.")

    count_variants(df_ko_full,  "KO",  ko_has_raw,  target_down=2369, target_up=2076)
    count_variants(df_kd2_full, "KD2", kd2_has_raw, target_down=542,  target_up=396)

    print("\n   >>> Whichever variant (A/B/C/D) matches the target numbers for BOTH")
    print("       KO and KD2 tells us exactly what threshold/column produced the")
    print("       2,369/2,076 and 542/396 figures — update the Discussion sentence")
    print("       to either use the correct (1,618/1,135 and 233/188) numbers, or")
    print("       explicitly state the alternate threshold if it was intentional.")

# ── Download all outputs ────────────────────────────────────────────────
from google.colab import files as colab_files
for f in [fig2a_pdf, fig2a_tiff, fig2b_pdf, fig2b_tiff]:
    colab_files.download(f)

print("\n✅ All downloads triggered — check your browser's Downloads folder.")
print("\nSUMMARY OF WHAT TO DO WITH THE OUTPUT:")
print("  1. Use the KO/KD1/KD2 down/up counts printed in PART 1 for Abstract/Results/Methods.")
print("  2. Decide row-level (2,753) vs symbol-deduped (2,752) convention using the")
print("     duplicate-symbol table printed in PART 1, and apply it everywhere.")
print("  3. Use 1,476 (from PART 3b) as the direct-target-detected-in-RNA-seq count,")
print("     replacing both 1,496 (old Fig 2B number) and 1,478 (old Supp Fig 5 number).")
print("  4. Use PART 4's recomputed rescue-rate numbers (direct vs non-target %rescued,")
print("     OR, p-value) to replace the old 857/1,478=58.0% / 57.1% (n=11,951) claim.")
print("  5. Use PART 5 to identify which threshold/column produced the 2,369/2,076 and")
print("     542/396 Discussion numbers — either correct them to 1,618/1,135 and 233/188,")
print("     or state the alternate threshold explicitly if it was an intentional choice.")

In [ ]:
import pandas as pd

DESEQ2_DIR = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
KO_PATH  = DESEQ2_DIR + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
KD2_PATH = DESEQ2_DIR + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv"

TARGETS = {
    "KO":  {"down": 2369, "up": 2076},
    "KD2": {"down": 542,  "up": 396},
}

def load_raw(path, tag):
    df = pd.read_csv(path, sep='\t')
    gene_col = next((c for c in df.columns
                      if c.upper() in ['GENESYMBOL', 'SYMBOL', 'GENE', 'MGI_SYMBOL']),
                     'GENESYMBOL')
    df = df.rename(columns={gene_col: 'symbol'})
    print(f"[{tag}] raw file: {len(df):,} rows, columns: {list(df.columns)}")
    return df

def run_variant(df, tag, target_down, target_up, apply_basemean, padj_op):
    d = df.dropna(subset=['symbol', 'log2FoldChange', 'padj']).copy()
    n_before = len(d)

    if apply_basemean and 'baseMean' in d.columns:
        d = d[d['baseMean'] > 10].copy()

    d['padj'] = d['padj'].replace(0, 1e-300)

    if padj_op == 'strict':
        sig = d[d['padj'] < 0.05]
    else:  # inclusive
        sig = d[d['padj'] <= 0.05]

    n_down = (sig['log2FoldChange'] < 0).sum()
    n_up   = (sig['log2FoldChange'] > 0).sum()

    match_down = "✅ MATCH" if n_down == target_down else f"off by {n_down - target_down:+d}"
    match_up   = "✅ MATCH" if n_up == target_up else f"off by {n_up - target_up:+d}"

    bm_label = "baseMean>10 applied" if apply_basemean else "NO baseMean filter"
    padj_label = "padj<0.05 (strict)" if padj_op == 'strict' else "padj<=0.05 (inclusive)"

    print(f"[{tag}] {bm_label:22s} | {padj_label:22s} -> "
          f"down={n_down:,} ({match_down})  up={n_up:,} ({match_up})  "
          f"[universe: {n_before:,} rows pre-filter, {len(d):,} post-filter, {len(sig):,} sig]")

    return n_down, n_up

df_ko  = load_raw(KO_PATH,  "KO")
df_kd2 = load_raw(KD2_PATH, "KD2")

print("\n" + "=" * 90)
print("TESTING 4 VARIANTS PER CONDITION: {baseMean filter on/off} x {padj boundary strict/inclusive}")
print("=" * 90)

for tag, df in [("KO", df_ko), ("KD2", df_kd2)]:
    print(f"\n--- {tag} (target: {TARGETS[tag]['down']:,} down / {TARGETS[tag]['up']:,} up) ---")
    for apply_bm in [True, False]:
        for padj_op in ['strict', 'inclusive']:
            run_variant(df, tag, TARGETS[tag]['down'], TARGETS[tag]['up'],
                        apply_basemean=apply_bm, padj_op=padj_op)

print("\n" + "=" * 90)
print("If none of the 4 KO variants show ✅ MATCH on both down AND up, the ~27-30 gene")
print("gap is very likely DESeq2 run-to-run variation (independent filtering / different")
print("dispersion estimates on a rerun) rather than a threshold difference in this script.")
print("In that case, use the ORIGINAL reported values as the audit trail:")
print("  -> Report KO as 2,342 down / 2,046 up (this run's reproducible number, replacing 2,369/2,076)")
print("  -> Keep KD2 as 542 down / 396 up (already confirmed exact)")
print("  -> State explicitly in Methods/Discussion: 'at padj < 0.05, without an effect-size")
print("     (|log2FC|) threshold' to distinguish this from the primary 1,618/1,135 and")
print("     233/188 DEG counts used elsewhere, which apply the |log2FC| > 0.5 cutoff.")
print("=" * 90)

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
import pybedtools

from google.colab import drive
drive.mount('/content/drive')

# ══════════════════════════════════════════════════════════════════════════
# CONFIG — single source of truth
# ══════════════════════════════════════════════════════════════════════════
DESEQ2_DIR = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"

DESEQ2_FILES = {
    "KO":  DESEQ2_DIR + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    "KD1": DESEQ2_DIR + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv",
    "KD2": DESEQ2_DIR + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv",
}
RESCUE_FILES = {
    "FL": DESEQ2_DIR + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    "dC": DESEQ2_DIR + "Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    "dH": DESEQ2_DIR + "Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"

FDR_CUTOFF = 0.05
LFC_CUTOFF = 0.5   # THE primary, --wide threshold — used everywhere below

RESCUE_LFC_ELIGIBLE = 0.5   # |LFC_KO| > 0.5 to be eligible for rescue-fraction analysis
RF_RESCUED_THRESHOLD = 0.5  # RF >= 0.5 counts as "rescued"


# ══════════════════════════════════════════════════════════════════════════
# PART 1 — DEG counts under the PRIMARY convention, for KO / KD1 / KD2
# ══════════════════════════════════════════════════════════════════════════
def clean_deg_table(path, tag):
    df = pd.read_csv(path, sep='\t')
    gene_col = next((c for c in df.columns
                      if c.upper() in ['GENESYMBOL', 'SYMBOL', 'GENE', 'MGI_SYMBOL']),
                     'GENESYMBOL')
    df = df.rename(columns={gene_col: 'symbol'})
    df = df.dropna(subset=['symbol', 'log2FoldChange', 'padj']).copy()
    n_before = len(df)
    if 'baseMean' in df.columns:
        df = df[df['baseMean'] > 10].copy()
    df['padj'] = df['padj'].replace(0, 1e-300)
    df['gene_upper'] = df['symbol'].astype(str).str.upper().str.strip()
    print(f"[{tag}] {n_before:,} -> {len(df):,} genes after baseMean>10 filter")
    return df

def categorize(row):
    if row['padj'] >= FDR_CUTOFF: return 'NS'
    if row['log2FoldChange'] < -LFC_CUTOFF: return 'Down'
    if row['log2FoldChange'] >  LFC_CUTOFF: return 'Up'
    return 'NS'

print("=" * 80)
print("PART 1 — DEG counts under the single primary convention")
print(f"         (baseMean > 10, padj < {FDR_CUTOFF}, |log2FC| > {LFC_CUTOFF})")
print("=" * 80)

df_ko  = clean_deg_table(DESEQ2_FILES["KO"],  "KO")
df_kd1 = clean_deg_table(DESEQ2_FILES["KD1"], "KD1")
df_kd2 = clean_deg_table(DESEQ2_FILES["KD2"], "KD2")

deg_counts = {}
for tag, df in [("KO", df_ko), ("KD1", df_kd1), ("KD2", df_kd2)]:
    df['category'] = df.apply(categorize, axis=1)
    n_down = (df['category'] == 'Down').sum()
    n_up   = (df['category'] == 'Up').sum()
    deg_counts[tag] = (n_down, n_up)
    print(f"\n[{tag}] Down = {n_down:,}  |  Up = {n_up:,}  |  Total DEGs = {n_down + n_up:,}")

# Flag duplicate-symbol issue (e.g. 4933434E20Rik) explicitly, once
dupe = df_ko[df_ko.duplicated('symbol', keep=False)].sort_values('symbol')
if not dupe.empty:
    print(f"\n⚠ NOTE: {dupe['symbol'].nunique()} gene symbol(s) map to multiple rows in KO "
          f"table (duplicate Ensembl IDs). This is the source of any ±1 discrepancy between "
          f"row-level and symbol-deduplicated totals. Reported counts above are ROW-level.")
    print(dupe[['symbol', 'log2FoldChange', 'padj']].to_string(index=False))

print("\n>>> USE THESE THREE NUMBERS EVERYWHERE in the - (Abstract, Discussion,")
print("    Methods) — replacing both the old 2,369/2,076 (KO) & 542/396 (KD2) Discussion")
print("    numbers AND confirming/replacing any other DEG counts quoted elsewhere.")


# ══════════════════════════════════════════════════════════════════════════
# PART 2 — NPC-specific CHD8 ChIP-seq target list
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PART 2 — NPC-specific CHD8 ChIP-seq targets")
print("=" * 80)

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()
npc_specific = npc_bt.subtract(esc_bt, A=True)
hits = tss_bt.intersect(npc_specific, u=True, wa=True)
target_list = sorted(set(str(f[3]).upper().strip() for f in hits if len(str(f[3])) > 1))
print(f"NPC-specific ChIP targets: {len(target_list):,} genes")


# ══════════════════════════════════════════════════════════════════════════
# PART 3 — Corrected direct-target-detected count: 4-way master universe
#           (KO, FL, ΔChromo, ΔHelicase), baseMean>10 required in ALL FOUR
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PART 3 — Corrected direct-target count (replaces both 1,496 and 1,478)")
print("=" * 80)

def find_ensembl_col(df):
    name_candidates = [c for c in df.columns
                        if c.upper() in ('GENEID', 'GENE_ID', 'ENSEMBL_ID', 'ENSEMBLID',
                                          'ENSEMBL_GENE_ID', 'ID', 'ROW.NAMES', 'ROWNAMES')]
    if name_candidates:
        return name_candidates[0]
    for c in df.columns:
        sample = df[c].astype(str).head(20)
        if sample.str.match(r'^ENSMUSG\d+').any():
            return c
    return None

def find_symbol_col(df):
    return next((c for c in df.columns
                 if c.upper() in ['GENESYMBOL', 'SYMBOL', 'GENE', 'MGI_SYMBOL']), None)

def load_rescue_table(path, tag):
    df = pd.read_csv(path, sep='\t')
    ens_col = find_ensembl_col(df)
    sym_col = find_symbol_col(df)
    if ens_col is None:
        raise ValueError(f"[{tag}] Could not find an Ensembl ID column. "
                          f"Columns present: {list(df.columns)}")
    keep = [c for c in [ens_col, sym_col, 'baseMean', 'log2FoldChange', 'padj'] if c]
    out = df[keep].copy()
    rename_map = {'log2FoldChange': f'LFC_{tag}', 'baseMean': f'baseMean_{tag}',
                  'padj': f'padj_{tag}', ens_col: 'gene_id'}
    if sym_col: rename_map[sym_col] = 'symbol'
    out = out.rename(columns=rename_map)
    print(f"[{tag}] {len(out):,} rows loaded, using '{ens_col}' as join key")
    return out

tbl_ko = load_rescue_table(DESEQ2_FILES["KO"], "KO")
tbl_fl = load_rescue_table(RESCUE_FILES["FL"], "FL")
tbl_dc = load_rescue_table(RESCUE_FILES["dC"], "dC")
tbl_dh = load_rescue_table(RESCUE_FILES["dH"], "dH")

master = (tbl_ko
          .merge(tbl_fl[['gene_id', 'LFC_FL', 'baseMean_FL']], on='gene_id', how='left')
          .merge(tbl_dc[['gene_id', 'LFC_dC', 'baseMean_dC']], on='gene_id', how='left')
          .merge(tbl_dh[['gene_id', 'LFC_dH', 'baseMean_dH']], on='gene_id', how='left'))

master = master.dropna(subset=['LFC_KO', 'LFC_FL', 'LFC_dC', 'LFC_dH']).copy()

bm_cols = ['baseMean_KO', 'baseMean_FL', 'baseMean_dC', 'baseMean_dH']
master = master[(master[bm_cols] > 10).all(axis=1)].copy()
print(f"Master universe (valid LFC in all 4 constructs, baseMean>10 in all 4): {len(master):,} genes")

master['gene_key'] = master['symbol'].astype(str).str.upper().str.strip()
targets_upper = set(target_list)
master['is_target'] = master['gene_key'].isin(targets_upper)

n_direct_targets = master['is_target'].sum()
print(f"\n>>> CORRECTED direct-target-detected-in-RNA-seq count: {n_direct_targets:,}")
print(f"    Replaces BOTH old numbers:")
print(f"      Fig 2B/Results text  : 1,496 -> {n_direct_targets:,}")
print(f"      Supp Fig 5 text      : 1,478 -> {n_direct_targets:,}")
print(f"\n>>> Methods sentence to add:")
print(f'    "NPC-specific ChIP-seq targets (n={len(target_list):,}) with baseMean > 10 in all')
print(f'    four DESeq2 comparisons (KO, FL, \u0394Chromo, \u0394Helicase; n={n_direct_targets:,})."')


# ══════════════════════════════════════════════════════════════════════════
# PART 4 — Rescue-rate claim, recomputed on the SAME corrected universe
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PART 4 — Rescue-rate claim (replaces 857/1,478=58.0% vs 57.1%, n=11,951)")
print("=" * 80)

master['RF_FL'] = ((master['LFC_FL'] - master['LFC_KO']) / (0 - master['LFC_KO'])) \
                    .replace([np.inf, -np.inf], np.nan).clip(-1, 2)

direct = master[master['is_target']].copy()
nontarget = master[~master['is_target']].copy()

direct_total   = direct['RF_FL'].notna().sum()
direct_rescued = (direct['RF_FL'] >= RF_RESCUED_THRESHOLD).sum()
nontarget_total   = nontarget['RF_FL'].notna().sum()
nontarget_rescued = (nontarget['RF_FL'] >= RF_RESCUED_THRESHOLD).sum()

pct_direct    = 100 * direct_rescued / direct_total
pct_nontarget = 100 * nontarget_rescued / nontarget_total

table_2x2 = [[direct_rescued, direct_total - direct_rescued],
             [nontarget_rescued, nontarget_total - nontarget_rescued]]
odds_ratio, p_val = fisher_exact(table_2x2)

print(f"Direct targets    : {direct_rescued:,}/{direct_total:,} rescued = {pct_direct:.1f}%")
print(f"Non-target genes  : {nontarget_rescued:,}/{nontarget_total:,} rescued = {pct_nontarget:.1f}%")
print(f"Fisher's exact test: OR = {odds_ratio:.2f}, p = {p_val:.3f}")

print(f"\n>>> Replace Supp Fig 5 text:")
print(f'    OLD: "857 (58.0%) ... 57.1% ... n=11,951 ... OR=1.04, p=0.272"')
print(f'    NEW: "{direct_rescued:,} ({pct_direct:.1f}%) ... {pct_nontarget:.1f}% ... '
      f'n={nontarget_total:,} ... OR={odds_ratio:.2f}, p={p_val:.3f}"')


# ══════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY — every number to update, one convention throughout
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("FINAL SUMMARY — all - numbers under ONE unified convention")
print("=" * 80)
print(f"""
1. DEG counts (Abstract / Discussion / Methods) — use these everywhere,
   replacing ANY other convention currently in the text:
     KO : {deg_counts['KO'][0]:,} down / {deg_counts['KO'][1]:,} up
     KD1: {deg_counts['KD1'][0]:,} down / {deg_counts['KD1'][1]:,} up
     KD2: {deg_counts['KD2'][0]:,} down / {deg_counts['KD2'][1]:,} up

2. Direct-target-detected-in-RNA-seq count (Fig 2B, Supp Fig 5):
     {n_direct_targets:,}  (replaces both 1,496 and 1,478)

3. Rescue-rate claim (Supp Fig 5):
     Direct targets   : {direct_rescued:,}/{direct_total:,} = {pct_direct:.1f}%
     Non-target genes : {nontarget_rescued:,}/{nontarget_total:,} = {pct_nontarget:.1f}%
     OR = {odds_ratio:.2f}, p = {p_val:.3f}

All three now derive from ONE consistent methodology (baseMean>10, padj<0.05,
|log2FC|>0.5 where an effect-size threshold applies), eliminating the need for
a separate, looser convention anywhere in the -.
""")

In [ ]:
import pandas as pd
import numpy as np
import pybedtools
import os
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

# ==========================================
# 📁 PATHS — Updated to match Google Drive structure
# ==========================================
BASE_DIR = "/content/drive/MyDrive/Chd8 data"

TSS_PATH         = os.path.join(BASE_DIR, "annotations/Mus_musculus_TSS_2kb_sorted.bed")
CHD8_PEAK_PATH   = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
H3K4ME3_PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"

RNA_PATH = os.path.join(BASE_DIR, "deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")

OUT_CSV_LOCAL = '/content/Figure1C_FPKM_corrected.csv'
OUT_CSV_DRIVE = os.path.join(BASE_DIR, 'Figure1C_FPKM_corrected.csv')   # persists across restarts

WT_FPKM_COL = 'Diff.Fa2Ls4'   # WT/control condition column
KO_FPKM_COL = 'Diff.Chd8_KO'
FPKM_THRESH = 0.5

# ==========================================
# 🔧 HELPERS
# ==========================================
def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ==========================================
# 📥 1. IDENTIFY CHD8+H3K4me3 AND H3K4me3-ONLY GENES (TSS ±2kb overlap)
# ==========================================
print("📥 Loading peaks and TSS annotation...")
tss_bt    = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()
chd8_bt   = pybedtools.BedTool(CHD8_PEAK_PATH).each(fix_naming).sort()
h3k4_bt   = pybedtools.BedTool(H3K4ME3_PEAK_PATH).each(fix_naming).sort()

# Genes whose TSS overlaps BOTH CHD8 and H3K4me3 peaks
chd8_h3k4_bed = chd8_bt.intersect(h3k4_bt, u=True)
chd8_h3k4_hits = tss_bt.intersect(chd8_h3k4_bed, u=True, wa=True)
chd8_h3k4_genes = set([str(f[3]).upper().strip() for f in chd8_h3k4_hits if len(str(f[3])) > 1])
print(f"CHD8+H3K4me3 genes (TSS-level, pre-RNA-filter): {len(chd8_h3k4_genes):,}")

# Genes whose TSS overlaps H3K4me3 but NOT CHD8
h3k4_only_bed = h3k4_bt.subtract(chd8_bt, A=True)
h3k4_only_hits = tss_bt.intersect(h3k4_only_bed, u=True, wa=True)
h3k4_only_genes = set([str(f[3]).upper().strip() for f in h3k4_only_hits if len(str(f[3])) > 1])
print(f"H3K4me3-only genes:                               {len(h3k4_only_genes):,}")

# ==========================================
# 📥 2. LOAD RNA-SEQ FPKM TABLE, FILTER FPKM > 0.5 IN ≥1 CONDITION
# ==========================================
print("\n📥 Loading RNA-seq FPKM table...")
rna = pd.read_csv(RNA_PATH, sep='\t')
rna['GENESYMBOL'] = rna['GENESYMBOL'].astype(str).str.upper().str.strip()

print("Sanity check — first 3 rows of the FPKM columns:")
print(rna[['GENESYMBOL', WT_FPKM_COL, KO_FPKM_COL]].head(3))

n_before = len(rna)
rna_filt = rna[(rna[WT_FPKM_COL] > FPKM_THRESH) | (rna[KO_FPKM_COL] > FPKM_THRESH)].copy()
n_after = len(rna_filt)
print(f"\nFPKM > {FPKM_THRESH} filter: {n_before:,} -> {n_after:,} genes")

# ==========================================
# 🏷️ 3. ASSIGN GROUPS
# ==========================================
def assign_group(gene):
    if gene in chd8_h3k4_genes:
        return 'CHD8+H3K4me3'
    elif gene in h3k4_only_genes:
        return 'H3K4me3-only'
    else:
        return 'Unbound & Unmarked'

rna_filt['group'] = rna_filt['GENESYMBOL'].apply(assign_group)
rna_filt['log2FPKM'] = np.log2(rna_filt[WT_FPKM_COL] + 1)

print("\nGroup sizes (after FPKM>0.5 filter):")
print(rna_filt['group'].value_counts())

medians = rna_filt.groupby('group')['log2FPKM'].median()
print("\nMedian log2(FPKM+1) by group:")
print(medians.round(3))

# ==========================================
# 📊 4. STATISTICS
# ==========================================
g1 = rna_filt[rna_filt['group']=='CHD8+H3K4me3']['log2FPKM']
g2 = rna_filt[rna_filt['group']=='H3K4me3-only']['log2FPKM']
g3 = rna_filt[rna_filt['group']=='Unbound & Unmarked']['log2FPKM']

kw_stat, kw_p = kruskal(g1, g2, g3)
print(f"\nKruskal-Wallis p = {kw_p:.3e}")

pairs = [('CHD8+H3K4me3', 'H3K4me3-only', g1, g2),
         ('CHD8+H3K4me3', 'Unbound & Unmarked', g1, g3),
         ('H3K4me3-only', 'Unbound & Unmarked', g2, g3)]
pvals = [mannwhitneyu(a, b, alternative='two-sided').pvalue for _,_,a,b in pairs]
_, padj, _, _ = multipletests(pvals, method='fdr_bh')

print("\nPairwise MWU (FDR-adjusted, Benjamini-Hochberg):")
for (n1, n2, _, _), praw, pfdr in zip(pairs, pvals, padj):
    print(f"  {n1} vs {n2}: raw p={praw:.3e}  FDR={pfdr:.3e}")

# ==========================================
# 📋 5. COMPARE TO OLD
# ==========================================
-_vals = {
    'CHD8+H3K4me3 median': 3.996,
    'CHD8+H3K4me3 n': 2081,
    'Unbound & Unmarked median': 3.577,
    'Unbound & Unmarked n': 10617,
    'MWU FDR (CHD8+H3K4me3 vs Unbound)': 2.42e-11,
    'Kruskal-Wallis p': 1.41e-11,
}
computed_vals = {
    'CHD8+H3K4me3 median': round(medians['CHD8+H3K4me3'], 3),
    'CHD8+H3K4me3 n': int((rna_filt['group']=='CHD8+H3K4me3').sum()),
    'Unbound & Unmarked median': round(medians['Unbound & Unmarked'], 3),
    'Unbound & Unmarked n': int((rna_filt['group']=='Unbound & Unmarked').sum()),
    'MWU FDR (CHD8+H3K4me3 vs Unbound)': [pfdr for (n1,n2,_,_),pfdr in zip(pairs, padj) if n1=='CHD8+H3K4me3' and n2=='Unbound & Unmarked'][0],
    'Kruskal-Wallis p': kw_p,
}

print("\n" + "="*70)
print("COMPARISON TO - TEXT")
print("="*70)
for key in -_vals:
    m, c = -_vals[key], computed_vals[key]
    match = "✓ MATCH" if (isinstance(m, int) and m == c) or (isinstance(m, float) and abs(m-c) < 1e-6) else "✗ MISMATCH"
    print(f"  {key:<40}-={m:>12} | computed={c:>12}  {match}")

# ==========================================
# 💾 6. SAVE — to BOTH /content AND Drive
# ==========================================
rna_filt.to_csv(OUT_CSV_LOCAL, index=False)
rna_filt.to_csv(OUT_CSV_DRIVE, index=False)
print(f"\n✅ Saved corrected table to {OUT_CSV_LOCAL}")
print(f"✅ Also saved to Drive: {OUT_CSV_DRIVE}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

SAVE_DIR = "/content/drive/MyDrive/Chd8 data/figures/"

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size'      : 12,
    'pdf.fonttype'   : 42,
})

# ── DATA — load the CORRECTED FPKM output
plot_df = pd.read_csv('/content/Figure1C_FPKM_corrected.csv')

print("Columns in corrected file:", list(plot_df.columns))
print(plot_df.head())

# WT expression = Diff.Fa2Ls4 column (Chd8_KO is explicitly the KO condition).
WT_FPKM_COL = 'Diff.Fa2Ls4'

if 'log2FPKM' not in plot_df.columns:
    plot_df['log2FPKM'] = np.log2(plot_df[WT_FPKM_COL] + 1)

group_order  = ['CHD8+H3K4me3', 'H3K4me3-only', 'Unbound & Unmarked']
group_colors = ['#2166AC', '#92C5DE', '#D1E5F0']

plot_df = plot_df[plot_df['group'].isin(group_order)].copy()
counts  = plot_df['group'].value_counts()
medians = plot_df.groupby('group')['log2FPKM'].median()

# ── STATISTICS — computed fresh from the corrected data
g1 = plot_df[plot_df['group']=='CHD8+H3K4me3']['log2FPKM']
g2 = plot_df[plot_df['group']=='H3K4me3-only']['log2FPKM']
g3 = plot_df[plot_df['group']=='Unbound & Unmarked']['log2FPKM']

pairs = [('CHD8+H3K4me3', 'H3K4me3-only',      g1, g2, 0, 1),
         ('H3K4me3-only', 'Unbound & Unmarked', g2, g3, 1, 2),
         ('CHD8+H3K4me3', 'Unbound & Unmarked', g1, g3, 0, 2)]

pvals = [mannwhitneyu(a, b, alternative='two-sided').pvalue
         for _,_,a,b,_,_ in pairs]
_, padj, _, _ = multipletests(pvals, method='fdr_bh')

kw_stat, kw_p = kruskal(g1, g2, g3)
print(f"\nKruskal-Wallis p = {kw_p:.3e}")
for (n1, n2, *_), p in zip(pairs, padj):
    print(f"{n1} vs {n2}: FDR = {p:.3e}")

def fdr_label(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

# ── FIGURE ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

sns.boxplot(
    data=plot_df,
    x='group', y='log2FPKM',
    hue='group',
    order=group_order,
    hue_order=group_order,
    palette=dict(zip(group_order, group_colors)),
    width=0.5,
    flierprops=dict(marker='o', markersize=2,
                    markerfacecolor='grey', alpha=0.3),
    linewidth=1.2,
    legend=False,
    ax=ax
)

xtick_labels = [f"{g}\n(n={counts.get(g,0):,})" for g in group_order]
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(xtick_labels, fontsize=10)

# ── MEDIAN ANNOTATIONS ────────────────────────────────────────
for i, g in enumerate(group_order):
    ax.text(i, medians[g] - 0.12,
            f'{medians[g]:.2f}',
            ha='center', va='top',
            fontsize=8.5, color='white', fontweight='bold')

# ── SIGNIFICANCE BRACKETS ──────────────────────────────────────
y_base = plot_df['log2FPKM'].quantile(0.995)
bracket_configs = [
    (0, 1, y_base + 0.3,  fdr_label(padj[0])),   # CHD8+H3K4me3 vs H3K4me3-only
    (1, 2, y_base + 0.3,  fdr_label(padj[1])),   # H3K4me3-only vs Unbound
    (0, 2, y_base + 0.85, fdr_label(padj[2])),   # CHD8+H3K4me3 vs Unbound (highest)
]

for x1, x2, y, label in bracket_configs:
    ax.plot([x1, x1, x2, x2],
            [y, y + 0.05, y + 0.05, y],
            color='black', linewidth=1.0, clip_on=False)
    ax.text((x1 + x2) / 2, y + 0.07,
            label,
            ha='center', va='bottom',
            fontsize=11, clip_on=False)

# ── AXIS LIMITS ─────────────────────────────────────────────────
ax.set_ylim(
    plot_df['log2FPKM'].min() - 0.2,
    y_base + 1.3
)

# ── LABELS ────────────────────────────────────────────────────
ax.set_xlabel('')
ax.set_ylabel('log$_2$(FPKM + 1)', fontsize=12)
ax.set_title('CHD8 Binding and Gene Expression in NPCs\n'
             '(FPKM > 0.5 in \u2265 1 condition)',
             fontsize=12, fontweight='bold')

ax.text(0.98, 0.02,
        f'Kruskal–Wallis p = {kw_p:.2e}',
        transform=ax.transAxes,
        ha='right', va='bottom',
        fontsize=9, style='italic', color='#444444')

sns.despine()
plt.tight_layout()

# ── SAVE ──────────────────────────────────────────────────────
out_png = SAVE_DIR + "Figure1C_expression_by_binding_FPKM_corrected.png"
out_pdf = SAVE_DIR + "Figure1C_expression_by_binding_FPKM_corrected.pdf"
out_tiff = SAVE_DIR + "Figure1C_expression_by_binding_FPKM_corrected.tiff"
plt.savefig(out_png, dpi=300, bbox_inches='tight')
plt.savefig(out_pdf, dpi=300, bbox_inches='tight')
plt.savefig(out_tiff, dpi=300, bbox_inches='tight', format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
plt.show()
print(f"✅ Saved:\n  {out_png}\n  {out_pdf}\n  {out_tiff}")

from google.colab import files
files.download(out_pdf)
files.download(out_tiff)

In [ ]:
import pandas as pd
import pybedtools
import os

# ==========================================
# 📁 PATHS — same as reconstruction script
# ==========================================
BASE_DIR = "/content/drive/MyDrive/Chd8 data"
TSS_PATH        = os.path.join(BASE_DIR, "annotations/Mus_musculus_TSS_2kb_sorted.bed")
CHD8_PEAK_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
H3K4ME3_PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"

-_N_GENES = 4953
-_N_PEAKS = 4466

def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ==========================================
# 🔎 0. FILE-LEVEL SANITY CHECKS
# ==========================================
print("="*70)
print("STEP 0: RAW FILE CHECKS")
print("="*70)

for label, path in [('TSS', TSS_PATH), ('CHD8 peaks', CHD8_PEAK_PATH), ('H3K4me3 peaks', H3K4ME3_PEAK_PATH)]:
    if not os.path.exists(path):
        print(f"  ⚠️ {label}: FILE NOT FOUND at {path}")
        continue
    n_lines = sum(1 for _ in open(path))
    mtime = pd.Timestamp(os.path.getmtime(path), unit='s')
    size_kb = os.path.getsize(path) / 1024
    print(f"  {label}: {n_lines:,} lines | {size_kb:.1f} KB | last modified {mtime}")

    # Check chr-prefix convention
    with open(path) as f:
        first_line = f.readline().strip()
    has_chr_prefix = first_line.startswith('chr')
    print(f"    First line: {first_line[:60]}")
    print(f"    Uses 'chr' prefix: {has_chr_prefix}")

raw_chd8_bt  = pybedtools.BedTool(CHD8_PEAK_PATH)
raw_h3k4_bt  = pybedtools.BedTool(H3K4ME3_PEAK_PATH)
raw_tss_bt   = pybedtools.BedTool(TSS_PATH)
print(f"\n  Raw CHD8 peak count   : {raw_chd8_bt.count():,}")
print(f"  Raw H3K4me3 peak count: {raw_h3k4_bt.count():,}")
print(f"  Raw TSS window count  : {raw_tss_bt.count():,}")

# ==========================================
# 🔬 METHOD A: peak-first (what the reconstruction script used)
# CHD8 peaks that overlap H3K4me3, then find genes whose TSS overlaps those peaks
# ==========================================
print("\n" + "="*70)
print("METHOD A: CHD8∩H3K4me3 peaks → TSS overlap (reconstruction script's method)")
print("="*70)

chd8_bt = raw_chd8_bt.each(fix_naming).sort()
h3k4_bt = raw_h3k4_bt.each(fix_naming).sort()
tss_bt  = raw_tss_bt.each(fix_naming).sort()

method_a_peaks = chd8_bt.intersect(h3k4_bt, u=True)
n_peaks_a = method_a_peaks.count()
method_a_hits = tss_bt.intersect(method_a_peaks, u=True, wa=True)
genes_a = set([str(f[3]).upper().strip() for f in method_a_hits if len(str(f[3])) > 1])
print(f"  CHD8 peaks overlapping H3K4me3 : {n_peaks_a:,}  (-: {-_N_PEAKS:,})")
print(f"  Unique genes (TSS overlap)     : {len(genes_a):,}  (-: {-_N_GENES:,})")

# ==========================================
# 🔬 METHOD B: TSS-first — start from TSS windows overlapping CHD8, then
# require those SAME TSS windows also overlap H3K4me3 (order-of-operations
# can matter with -u flag behavior and multi-mapping TSS entries)
# ==========================================
print("\n" + "="*70)
print("METHOD B: TSS windows overlapping CHD8 AND overlapping H3K4me3 (TSS-first)")
print("="*70)

tss_chd8 = tss_bt.intersect(chd8_bt, u=True)
tss_chd8_h3k4 = tss_chd8.intersect(h3k4_bt, u=True)
genes_b = set([str(f[3]).upper().strip() for f in tss_chd8_h3k4 if len(str(f[3])) > 1])
print(f"  Unique genes (TSS-first)       : {len(genes_b):,}  (-: {-_N_GENES:,})")

# ==========================================
# 🔬 METHOD C: no chr-prefix fix applied
# ==========================================
print("\n" + "="*70)
print("METHOD C: without fix_naming (raw chromosome names, as originally provided)")
print("="*70)

method_c_peaks = raw_chd8_bt.sort().intersect(raw_h3k4_bt.sort(), u=True)
n_peaks_c = method_c_peaks.count()
method_c_hits = raw_tss_bt.sort().intersect(method_c_peaks, u=True, wa=True)
genes_c = set([str(f[3]).upper().strip() for f in method_c_hits if len(str(f[3])) > 1])
print(f"  CHD8 peaks overlapping H3K4me3 : {n_peaks_c:,}  (-: {-_N_PEAKS:,})")
print(f"  Unique genes (TSS overlap)     : {len(genes_c):,}  (-: {-_N_GENES:,})")

# ==========================================
# 🔬 METHOD D: case-sensitivity check — are gene symbols in the TSS bed
# file already unique, or does .upper().strip() collapse near-duplicates
# ==========================================
print("\n" + "="*70)
print("METHOD D: gene symbol formatting check")
print("="*70)

raw_gene_names_a = [str(f[3]) for f in method_a_hits if len(str(f[3])) > 1]
print(f"  Raw (non-deduped, non-upper) hits from Method A: {len(raw_gene_names_a):,}")
print(f"  After .upper().strip() + set(): {len(genes_a):,}")
print(f"  Sample raw names: {raw_gene_names_a[:5]}")

# ==========================================
# 📊 SUMMARY
# ==========================================
print("\n" + "="*70)
print("SUMMARY — closest match to -'s 4,953 genes / 4,466 peaks?")
print("="*70)
print(f"  - reported            : {-_N_GENES:,} genes from {-_N_PEAKS:,} peaks")
print(f"  Method A (peak-first, w/ fix)  : {len(genes_a):,} genes from {n_peaks_a:,} peaks")
print(f"  Method B (TSS-first)           : {len(genes_b):,} genes")
print(f"  Method C (no chr-prefix fix)   : {len(genes_c):,} genes from {n_peaks_c:,} peaks")
print("\nIf none of these match closely, the discrepancy is most likely a")
print("different underlying peak FILE (e.g. re-called MACS2 peaks, different")
print("replicate consensus threshold) rather than a logic difference —")
print("check file modification dates above against your original analysis date.")

In [ ]:
import pybedtools
import os

BASE_DIR = "/content/drive/MyDrive/Chd8 data"
TSS_PATH        = os.path.join(BASE_DIR, "annotations/Mus_musculus_TSS_2kb_sorted.bed")
CHD8_PEAK_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
H3K4ME3_PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"

def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

chd8_bt = pybedtools.BedTool(CHD8_PEAK_PATH).each(fix_naming).sort()
h3k4_bt = pybedtools.BedTool(H3K4ME3_PEAK_PATH).each(fix_naming).sort()
tss_bt  = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()

n_chd8_total = chd8_bt.count()

# TRUE three-way peak-level filter: CHD8 peak must overlap BOTH TSS AND H3K4me3
chd8_at_tss        = chd8_bt.intersect(tss_bt, u=True)
n_chd8_at_tss       = chd8_at_tss.count()
chd8_at_tss_h3k4    = chd8_at_tss.intersect(h3k4_bt, u=True)
n_chd8_promoter_h3k4 = chd8_at_tss_h3k4.count()

# Poised promoters: CHD8 at TSS but NOT overlapping H3K4me3
chd8_at_tss_no_h3k4 = chd8_at_tss.intersect(h3k4_bt, v=True)
n_chd8_poised       = chd8_at_tss_no_h3k4.count()

# Distal: CHD8 peaks NOT overlapping any TSS
chd8_distal         = chd8_bt.intersect(tss_bt, v=True)
n_chd8_distal        = chd8_distal.count()

print(f"Total CHD8 NPC peaks                          : {n_chd8_total:,}  (- Fig 1A/1D: 48,735)")
print(f"CHD8 peaks overlapping a TSS window            : {n_chd8_at_tss:,}")
print(f"  ├─ of those, also overlapping H3K4me3 (active): {n_chd8_promoter_h3k4:,}  (-: 4,466)")
print(f"  └─ of those, NOT overlapping H3K4me3 (poised) : {n_chd8_poised:,}")
print(f"CHD8 peaks NOT overlapping any TSS (distal)     : {n_chd8_distal:,}")

pct_active = 100 * n_chd8_promoter_h3k4 / n_chd8_total
pct_poised = 100 * n_chd8_poised / n_chd8_total
pct_distal = 100 * n_chd8_distal / n_chd8_total
print(f"\nComputed percentages (of {n_chd8_total:,} total CHD8 peaks):")
print(f"  Active promoters (CHD8+H3K4me3+): {pct_active:.1f}%  (-: 31.7%)")
print(f"  Poised promoters (CHD8+H3K4me3-): {pct_poised:.1f}%  (-: 1.7%)")
print(f"  Distal regulatory elements       : {pct_distal:.1f}%  (-: 66.6%)")

# Gene-level count from this correctly-restricted peak set
genes_correct = set([str(f[3]).upper().strip()
                      for f in tss_bt.intersect(chd8_at_tss_h3k4, u=True, wa=True)
                      if len(str(f[3])) > 1])
print(f"\nUnique genes from correctly-restricted peak set: {len(genes_correct):,}  (-: 4,953)")

In [ ]:
# ============================================================================
# CLOSE THE GAP — Figure 1C, using - FPKM columns (not baseMean-derived TPM)
# ============================================================================

# ── Install dependencies FIRST, before importing them ───────────────────────
import subprocess, sys

subprocess.run(['apt-get', 'install', '-y', 'bedtools'], capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pybedtools'], capture_output=True)

import os
import pybedtools
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

from google.colab import drive
drive.mount('/content/drive')

# ── Paths (same as original pipeline) ───────────────────────────────────────
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
H3K4_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_PATH  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

STANDARD_CHROMS = set([f'chr{i}' for i in range(1, 20)] + ['chrX', 'chrY'])

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

def load_filtered(path, add_chr=False):
    bt = pybedtools.BedTool(path)
    if add_chr:
        bt = bt.each(fix_chr)
    return bt.filter(lambda f: str(f.chrom) in STANDARD_CHROMS).saveas()

# ── STEP 1: Detect chr-prefix needs (same diagnostic as original) ──────────
peak_first = pd.read_csv(PEAK_PATH, sep='\t', header=None, usecols=[0]).iloc[0, 0]
tss_first  = pd.read_csv(TSS_PATH,  sep='\t', header=None, usecols=[0]).iloc[0, 0]
peak_needs_chr = not str(peak_first).startswith('chr')
tss_needs_chr  = not str(tss_first).startswith('chr')

npc  = load_filtered(PEAK_PATH, add_chr=peak_needs_chr)
h3k4 = load_filtered(H3K4_PATH, add_chr=peak_needs_chr)
tss  = load_filtered(TSS_PATH,  add_chr=tss_needs_chr)

# ── STEP 2: Peak-TSS intersections → gene sets ──────────────────────────────
chd8_at_tss = npc.intersect(tss,  wb=True)
h3k4_at_tss = h3k4.intersect(tss, wb=True)

def get_genes_wb(bt, name_col=None):
    genes = set()
    for i, f in enumerate(bt):
        if i == 0 and name_col is None:
            for idx in range(3, len(f.fields)):
                val = f.fields[idx]
                if not val.lstrip('-').replace('.', '', 1).isdigit() and not val.startswith('chr'):
                    name_col = idx
                    break
            if name_col is None:
                name_col = 6
        if len(f.fields) > name_col:
            genes.add(f.fields[name_col])
    return genes

chd8_tss_genes  = get_genes_wb(chd8_at_tss)
h3k4_tss_genes  = get_genes_wb(h3k4_at_tss)
chd8_h3k4_genes = chd8_tss_genes & h3k4_tss_genes
h3k4_only_genes = h3k4_tss_genes - chd8_tss_genes

print(f"CHD8+H3K4me3 genes (TSS-level, pre-RNA-filter): {len(chd8_h3k4_genes):,}")
print(f"H3K4me3-only genes:                             {len(h3k4_only_genes):,}")

# ── STEP 3: Load RNA-seq — use - FPKM columns, no GTF needed ────────────
rna_all = pd.read_csv(RNA_PATH, sep='\t').dropna(subset=['GENESYMBOL'])

print("\nSanity check — first 3 rows of the FPKM columns:")
print(rna_all[['GENESYMBOL', 'Diff.Fa2Ls4', 'Diff.Chd8_KO']].head(3).to_string())

# ── STEP 4: Apply the DGE-table filter — avg FPKM > 0.5 in >=1 condition ───
n_before = len(rna_all)
rna_filt = rna_all[(rna_all['Diff.Fa2Ls4'] > 0.5) | (rna_all['Diff.Chd8_KO'] > 0.5)].copy()
print(f"\nFPKM > 0.5 filter: {n_before:,} -> {len(rna_filt):,} genes")

# ── STEP 5: Expression metric for the WT-NPC analysis = WT/CTRL FPKM ──────
rna_filt['log2FPKM'] = np.log2(rna_filt['Diff.Fa2Ls4'] + 1)

# ── STEP 6: Assign groups ───────────────────────────────────────────────────
rna_filt['group'] = 'Unbound & Unmarked'
rna_filt.loc[rna_filt['GENESYMBOL'].isin(h3k4_only_genes),  'group'] = 'H3K4me3-only'
rna_filt.loc[rna_filt['GENESYMBOL'].isin(chd8_h3k4_genes),  'group'] = 'CHD8+H3K4me3'

print("\nGroup sizes (after FPKM>0.5 filter):")
print(rna_filt['group'].value_counts().to_string())

print("\nMedian log2(FPKM+1) by group:")
print(rna_filt.groupby('group')['log2FPKM'].median().round(3).to_string())

# ── STEP 7: Statistics ──────────────────────────────────────────────────────
g1 = rna_filt.loc[rna_filt['group'] == 'CHD8+H3K4me3',       'log2FPKM']
g2 = rna_filt.loc[rna_filt['group'] == 'H3K4me3-only',       'log2FPKM']
g3 = rna_filt.loc[rna_filt['group'] == 'Unbound & Unmarked', 'log2FPKM']

stat_kw, p_kw = kruskal(g1, g2, g3)
print(f"\nKruskal-Wallis p = {p_kw:.3e}")

pairs = [
    ('CHD8+H3K4me3', 'H3K4me3-only',       g1, g2),
    ('CHD8+H3K4me3', 'Unbound & Unmarked', g1, g3),
    ('H3K4me3-only', 'Unbound & Unmarked', g2, g3),
]
pvals = [mannwhitneyu(a, b, alternative='two-sided').pvalue for _, _, a, b in pairs]
_, padj, _, _ = multipletests(pvals, method='fdr_bh')

print("\nPairwise MWU (FDR-adjusted, Benjamini-Hochberg):")
for (n1, n2, _, _), p, pa in zip(pairs, pvals, padj):
    print(f"  {n1} vs {n2}: raw p={p:.3e}  FDR={pa:.3e}")

# ── STEP 8: Compare directly ─────
print("\n" + "=" * 70)
print("COMPARISON TO - TEXT")
print("=" * 70)
- = {
    'CHD8+H3K4me3 median':        3.996,
    'CHD8+H3K4me3 n':             2081,
    'Unbound & Unmarked median':  3.577,
    'Unbound & Unmarked n':       10617,
    'MWU FDR (CHD8+H3K4me3 vs Unbound)': 2.42e-11,
    'Kruskal-Wallis p':           1.41e-11,
}
computed = {
    'CHD8+H3K4me3 median':        round(g1.median(), 3),
    'CHD8+H3K4me3 n':             len(g1),
    'Unbound & Unmarked median':  round(g3.median(), 3),
    'Unbound & Unmarked n':       len(g3),
    'MWU FDR (CHD8+H3K4me3 vs Unbound)': padj[1],
    'Kruskal-Wallis p':           p_kw,
}
for k in -:
    match = "✓" if str(-[k]) == str(computed[k]) else "✗ MISMATCH"
    print(f"  {k:38s} -={-[k]!s:>12} | computed={computed[k]!s:>12}  {match}")

# ── STEP 9: Save cleaned table  ────────────────────
rna_filt[['GENESYMBOL', 'Diff.Fa2Ls4', 'Diff.Chd8_KO', 'log2FPKM', 'group']].to_csv(
    '/content/drive/MyDrive/Chd8 data/figures/Figure1C_FPKM_corrected.csv', index=False
)
print("\n✅ Saved corrected table to Figure1C_FPKM_corrected.csv")

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────────
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
ko_path   = os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")
kd1_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv")
kd2_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")

output_dir = "/content/Figure3B_output"
os.makedirs(output_dir, exist_ok=True)

DPI = 300

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

# ── Load & flag ───────────────────────────────────────────────────────────────
def load_and_flag(path):
    df = pd.read_csv(path, sep='\t')
    gene_col = next(
        (c for c in df.columns
         if c.upper() in ['GENESYMBOL', 'SYMBOL', 'MGI_SYMBOL']),
        'GENESYMBOL'
    )
    df = df.rename(columns={gene_col: 'GENESYMBOL'})
    df['DEREG_FLAG'] = np.where(
        (df['log2FoldChange'] < 0) & (df['padj'] < 0.05), 'DOWN',
        np.where(
            (df['log2FoldChange'] > 0) & (df['padj'] < 0.05), 'UP', 'NONE'
        )
    )
    return df.set_index('GENESYMBOL')

ko_df   = load_and_flag(ko_path)
kd81_df = load_and_flag(kd1_path)
kd82_df = load_and_flag(kd2_path)

# ── Shared DOWN genes (top 20) ────────────────────────────────────────────────
shared_down = list(
    set(ko_df  [ko_df  ['DEREG_FLAG'] == 'DOWN'].index) &
    set(kd81_df[kd81_df['DEREG_FLAG'] == 'DOWN'].index) &
    set(kd82_df[kd82_df['DEREG_FLAG'] == 'DOWN'].index)
)
print(f"Shared downregulated genes: {len(shared_down):,}")

down_data = pd.DataFrame({
    'CHD8 KO' : ko_df  .loc[shared_down, 'log2FoldChange'],
    'KD 8.1'  : kd81_df.loc[shared_down, 'log2FoldChange'],
    'KD 8.2'  : kd82_df.loc[shared_down, 'log2FoldChange'],
}).dropna()
down_data['mean_abs'] = down_data.abs().mean(axis=1)
down_data = (down_data
             .sort_values('mean_abs', ascending=False)
             .head(20)
             .drop(columns=['mean_abs']))

# ── Shared UP genes (top 20) ──────────────────────────────────────────────────
shared_up = list(
    set(ko_df  [ko_df  ['DEREG_FLAG'] == 'UP'].index) &
    set(kd81_df[kd81_df['DEREG_FLAG'] == 'UP'].index) &
    set(kd82_df[kd82_df['DEREG_FLAG'] == 'UP'].index)
)
print(f"Shared upregulated genes: {len(shared_up):,}")

up_data = pd.DataFrame({
    'CHD8 KO' : ko_df  .loc[shared_up, 'log2FoldChange'],
    'KD 8.1'  : kd81_df.loc[shared_up, 'log2FoldChange'],
    'KD 8.2'  : kd82_df.loc[shared_up, 'log2FoldChange'],
}).dropna()
up_data['mean_abs'] = up_data.abs().mean(axis=1)
up_data = (up_data
           .sort_values('mean_abs', ascending=False)
           .head(20)
           .drop(columns=['mean_abs']))

print(f"DOWN heatmap: {down_data.shape[0]} genes")
print(f"UP   heatmap: {up_data.shape[0]}   genes")

# ── Figure dimensions ─────────────────────────────────────────────────────────
n_genes       = 20
row_mm        = 5.5
overhead_mm   = 30
panel_h_mm    = n_genes * row_mm + overhead_mm
panel_w_mm    = 50
gap_mm        = 12

FIG_H_IN = panel_h_mm / 25.4
FIG_W_IN = (panel_w_mm * 2 + gap_mm) / 25.4
print(f"Figure size: {panel_w_mm*2+gap_mm:.0f} mm × {panel_h_mm:.0f} mm")

# ── Helper ────────────────────────────────────────────────────────────────────
def make_heatmap(data, vmin, vmax, cmap, title, cbar_label):
    g = sns.clustermap(
        data,
        cmap             = cmap,
        center           = 0,
        vmin             = vmin,
        vmax             = vmax,
        row_cluster      = True,
        col_cluster      = False,
        linewidths       = 0.3,
        linecolor        = '#DDDDDD',
        dendrogram_ratio = (0.10, 0.0),
        colors_ratio     = 0.0,
        cbar_pos         = None,
        figsize          = (FIG_W_IN / 2 - gap_mm / 25.4 / 2, FIG_H_IN),
        tree_kws         = {'linewidths': 0.5},
    )

    # Colorbar — bottom-right, clear of title and gene names
    cbar_ax = g.fig.add_axes([0.83, 0.03, 0.04, 0.12])
    import matplotlib as mpl
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cb   = mpl.colorbar.ColorbarBase(
        cbar_ax,
        cmap        = plt.get_cmap(cmap),
        norm        = norm,
        orientation = 'vertical',
    )
    cb.set_ticks([vmin, 0, vmax])
    cb.ax.tick_params(labelsize=5, width=0.4, length=2)
    cb.ax.set_ylabel('log$_2$ FC', fontsize=5, labelpad=3)
    cb.outline.set_linewidth(0.4)

    g.fig.subplots_adjust(left=0.02, right=0.80, bottom=0.18, top=0.93)

    g.ax_heatmap.set_yticklabels(
        g.ax_heatmap.get_yticklabels(),
        fontsize  = 5.5,
        fontstyle = 'italic',
        va        = 'center',
    )
    g.ax_heatmap.yaxis.set_tick_params(pad=2, length=0)
    g.ax_heatmap.set_xticklabels(
        g.ax_heatmap.get_xticklabels(),
        fontsize=6, rotation=0, ha='center',
    )
    g.ax_heatmap.xaxis.set_tick_params(pad=3, length=0)
    g.ax_heatmap.set_xlabel('')
    g.ax_heatmap.set_ylabel('')
    g.ax_heatmap.set_title(
        title,
        fontsize=6, fontweight='bold', loc='left', pad=4,
    )
    return g

# ── Build both clustermaps ────────────────────────────────────────────────────
g_down = make_heatmap(
    down_data,
    vmin       = -3,
    vmax       = 0,
    cmap       = 'RdBu_r',
    title      = 'Top 20 shared downregulated genes',
    cbar_label = 'log$_2$ FC',
)

g_up = make_heatmap(
    up_data,
    vmin       = 0,
    vmax       = 3,
    cmap       = 'RdBu_r',
    title      = 'Top 20 shared upregulated genes',
    cbar_label = 'log$_2$ FC',
)

# ── Save individual figures ───────────────────────────────────────────────────
down_pdf  = os.path.join(output_dir, "Figure3B_down.pdf")
up_pdf    = os.path.join(output_dir, "Figure3B_up.pdf")
down_tiff = os.path.join(output_dir, "Figure3B_down.tiff")
up_tiff   = os.path.join(output_dir, "Figure3B_up.tiff")

save_kws_tiff = dict(format='tiff', dpi=DPI, pad_inches=0.02,
                     pil_kwargs={"compression": "tiff_lzw"})

g_down.savefig(down_pdf,  format='pdf', pad_inches=0.02)
g_down.savefig(down_tiff, **save_kws_tiff)
g_up.savefig(up_pdf,    format='pdf', pad_inches=0.02)
g_up.savefig(up_tiff,   **save_kws_tiff)

print(f"✅ DOWN PDF:  {down_pdf}")
print(f"✅ UP   PDF:  {up_pdf}")

# ── Combine into single side-by-side figure ───────────────────────────────────
from PIL import Image
import io

def fig_to_pil(g):
    buf = io.BytesIO()
    g.savefig(buf, format='png', dpi=DPI, pad_inches=0.02)
    buf.seek(0)
    return Image.open(buf).copy()

img_down = fig_to_pil(g_down)
img_up   = fig_to_pil(g_up)

max_h = max(img_down.height, img_up.height)
gap_px = int(gap_mm / 25.4 * DPI)

def pad_to_height(img, target_h, bg=(255, 255, 255)):
    if img.height == target_h:
        return img
    padded = Image.new('RGB', (img.width, target_h), bg)
    padded.paste(img, (0, 0))
    return padded

img_down = pad_to_height(img_down, max_h)
img_up   = pad_to_height(img_up,   max_h)

gap_img  = Image.new('RGB', (gap_px, max_h), (255, 255, 255))
combined = Image.new('RGB',
                     (img_down.width + gap_px + img_up.width, max_h),
                     (255, 255, 255))
combined.paste(img_down, (0, 0))
combined.paste(gap_img,  (img_down.width, 0))
combined.paste(img_up,   (img_down.width + gap_px, 0))

combined_tiff = os.path.join(output_dir, "Figure3B_combined.tiff")
combined_pdf  = os.path.join(output_dir, "Figure3B_combined.pdf")

combined.save(combined_tiff, compression='tiff_lzw', dpi=(DPI, DPI))

fig_w_in = combined.width  / DPI
fig_h_in = combined.height / DPI
fig_comb, ax_comb = plt.subplots(1, 1, figsize=(fig_w_in, fig_h_in))
ax_comb.imshow(np.array(combined))
ax_comb.axis('off')
fig_comb.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig_comb.savefig(combined_pdf, format='pdf', dpi=DPI, pad_inches=0)
plt.close('all')

print(f"✅ Combined TIFF: {combined_tiff}")
print(f"✅ Combined PDF:  {combined_pdf}")

# ── Download ──────────────────────────────────────────────────────────────────
from google.colab import files
files.download(combined_pdf)
files.download(combined_tiff)
files.download(down_pdf)
files.download(up_pdf)
print("\n✅ Downloads triggered — check your browser's Downloads folder.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.fi-s('ignore')

# ── Upload mass-spec Excel file ─────────────────────────────────────────────
from google.colab import files as colab_files
print("Please upload the file: 42003_2021_1945_MOESM7_ESM.xlsx")
uploaded = colab_files.upload()
MS_EXCEL_PATH = "42003_20-5_MOESM7_ESM.xlsx"
if MS_EXCEL_PATH not in uploaded:
    for fn in uploaded.keys():
        if fn.endswith('.xlsx'):
            MS_EXCEL_PATH = fn
       -rint(f"   Using uploaded file: {MS_EXCEL_PATH}")
            break

# ── - rcParams ─────────────────────────────────────────────────
matplotlib.rcParams['font.family']       = 'sans-serif'
matplotlib.rcParams['font.sans-serif']   = ['Arial', 'Liberation Sans', 'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']      = 42
matplotlib.rcParams['ps.fonttype']       = 42
-tlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matp-.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.major.size'] -

# ── Paths ─────────────────────────────────────────────────────────────────
MY_RNA_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

DOWNLOADS = os.path.join(os.path.expanduser('~'), 'Downloads')
os.makedirs(DOWNLOADS, exist_ok=True)
TIFF_PATH = os.path.join(DOWNLOADS, 'SuppFig3_MassSpec_GenomeBiology.tiff')
PDF_PATH  = os.path.join(DOWNLOADS, 'SuppFig3_-ec_GenomeBiology.pdf')

# ── SINGLE colour — all bars are the same category: interactor + downregulated ──
C_MAIN  = '#2166ac'
C_NEUT  = '#333333'
C_REF   = '#999999'

LFC_THRESHOLD = 0.0

# ──-r: clean gene symbols ────────────-────────────────────────────
def clean_symbol(s):
    if pd.isna(s):
        return None
    s = str(s).upper().replace('P0593_', '')
    return s.split('|')[-1].strip()

print("=" * 60)
print("SUPPLEMENTARY FIGURE 3: CHD8 MASS-SPEC INTERACTORS (corrected)")
print("=" * 60)

# ── Load mass-spec data ──────────────────────────────────────────────────────
print("\n1. Loading mass-spec Excel...")
ms_df = pd.read_excel(MS_EXCEL_PATH, sheet_name=0, skiprows=2)
ms_df['SYMBOL'] = ms_df['gene_name'].apply(clean_symbol)

# ★ FIX 1: remove CHD8 — it is the bait, not a genuine interactor, and its own
# RNA-seq "downregulation" in the KO is a direct construct effect, not a
# biological interactor-transcription relationship.
ms_df = ms_df[ms_df['SYMBOL'] != 'CHD8'].copy()

-_interactors = set(ms_df['SYMBOL'].dropna().uniqu-rint(f"   {len(-_interactors):,} CHD8 interactors loaded (Cerase 2021,-excluded)")

# ── Load RNA-seq and find downregulated overlap ─────────────────────────────
print("\n2. Loading RNA-seq for overlap...")
rna_df = pd.read_csv(MY_RNA_PATH, sep='\t').dropna(subset=['padj', 'log2FoldChange'])
my_down = set(
    rna_df[(rna_df['padj'] < 0.05) &
           (rna_df['log2FoldChange'] < LFC_THRESHOLD)]['GENESYMBOL'].str.upper()
)
rna_overlap = my_down & -_interactors
print(f"   Downregulated genes (threshold log2-C_THRESHOLD}): {len(my_down):,}")
print(f"   Overlap with interactors (CHD8 excluded): {len(rna_overlap):,}")
print(f"   Overlap gene list: {sorted(rna_overlap)}")

# ── ★ FIX 2: plot_df is now EXACTLY the overlap set, not a separate top-10-by-
print("\n3. Building plot set-the TRUE overlap (not top-10-by-score)...")
plot_df = ms_df[ms_df['SYM-isin(rna_overlap)].copy()
plot_df = plot_df.drop_duplicates(subset='SYMBOL')
plot_df = plot_df-values('pearson.total', ascending=Tr- ascending for barh
print(f"   Plotting {len(plot_df)} pr- (all = interactor + downregulated)")

if len(plot_df) == 0:
    raise ValueError("No overlap genes found in the MS score table — check symbol matching.")

# ── Figure ────────────────────────────────────────────────────────────────
FIG_W = 6.693
FIG_H = 0.35 * len(plot_df) + 1.2
DPI   = 600

fig, ax = plt.subplots(fi-(FIG_W, FIG_H), dpi=DPI)

bars = ax.barh(
    plot_df['SYMBOL'],
    plot_df['pearson.total'],
    color=C_MAIN,
    edgecolor='white',
    linewidth=0.5,
    height=0.6,
    alpha=0.9
)

max_val = plot_df['pearson.total'].max()
for bar, val in zip(bars, plot_df['pearson.total']):
    ax.text(val + max_val * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha='left', fontsize=5, color=C_NEUT)

ax.set_xlabel('CHD8 interactor Pearson score', fontsize=6, labelpad=3)
ax.set_xlim(0, max_val * 1.20)
ax.tick_params(axis='x', labelsize=6, lengt- width=0.75, pad=2)
ax.tick_params(axis='y', labelsize=6, length=0, pad=3)
ax.margins(y=0.08)

ax.set_title(
    f'CHD8 interactors also downregulated in CHD8-KO NPCs (n={len(plot_df)})',
    fontsize=6.5, fontweight='bold', pad=5, color=C_NEUT
)

# ★ FIX 3: single-category legend —
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=C_MAIN, edgecolor='white',
                          label='CHD8 interactor + RNA-seq downregulated (KO)')]
ax.legend(handles=legend_elements, fontsize=5, loc='lower right',
          frameon=True, framealpha=0.9, edgecolor=C_REF,
          handlelength=1.0, handletextpad=0.4, borderpad=0.5)

sns.despine(ax=ax, left=True, offset=3, trim=True)
ax.spines['left'].set_visible(False)

plt.tight_layout(pad=0.6)

# ── Save ──────────────────────────────────────────────────────────────────
fig.savefig(TIFF_PATH, dpi=DPI, bbox_inches='tight', format='tiff',
            pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PDF_PATH, dpi=DPI, bbox_inches='tight', format='pdf')
plt.show()

for path, label in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb = os.path.getsize(path) / 1e6
    print(f"✅ {label}: {path} ({mb:.1f} MB)")

out_df = pd.DataFrame(sorted(rna_overlap), columns=['GeneSymbol'])
out_path = os.path.join(DOWNLOADS, 'CHD8_Functional_Intersection_corrected.csv')
out_df.to_csv(out_path, index=False)
print(f"\n✅ Corrected intersection list saved: {out_path}")

colab_files.download(TIFF_PATH)
colab_files.download(PDF_PATH)
colab_files.download(out_path)---

In [ ]:
import os
from google.colab import files as colab_files

fig.savefig(TIFF_PATH, dpi=600, bbox_inches='tight', format='tiff',
            pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PDF_PATH, dpi=600, bbox_inches='tight', format='pdf')

for path, label in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb = os.path.getsize(path) / 1e6
    print(f"✅ {label}: {path} ({mb:.1f} MB)")

colab_files.download(TIFF_PATH)
colab_files.download(PDF_PATH)
print("📥 Downloads triggered.")

In [ ]:
import pandas as pd
import numpy as np
import os
import gseapy as gp
from google.colab import drive

drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
ko_path   = os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")
kd1_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv")
kd2_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")

output_dir = "/content/drive/MyDrive/Chd8 data/GO_enrichment_output"
os.makedirs(output_dir, exist_ok=True)

# ── Load & flag (UP + DOWN) ─────
def load_and_flag(path):
    df = pd.read_csv(path, sep='\t')
    gene_col = next(
        (c for c in df.columns if c.upper() in ['GENESYMBOL', 'SYMBOL', 'MGI_SYMBOL']),
        'GENESYMBOL'
    )
    df = df.rename(columns={gene_col: 'GENESYMBOL'})
    df['DEREG_FLAG'] = np.where(
        (df['log2FoldChange'] < 0) & (df['padj'] < 0.05), 'DOWN',
        np.where((df['log2FoldChange'] > 0) & (df['padj'] < 0.05), 'UP', 'NONE')
    )
    return df.set_index('GENESYMBOL')

ko_df   = load_and_flag(ko_path)
kd81_df = load_and_flag(kd1_path)
kd82_df = load_and_flag(kd2_path)

# ── Full shared DOWN / UP sets (no top-N cutoff — used for stats, not display) ──
shared_down = sorted(
    set(ko_df  [ko_df  ['DEREG_FLAG'] == 'DOWN'].index) &
    set(kd81_df[kd81_df['DEREG_FLAG'] == 'DOWN'].index) &
    set(kd82_df[kd82_df['DEREG_FLAG'] == 'DOWN'].index)
)
shared_up = sorted(
    set(ko_df  [ko_df  ['DEREG_FLAG'] == 'UP'].index) &
    set(kd81_df[kd81_df['DEREG_FLAG'] == 'UP'].index) &
    set(kd82_df[kd82_df['DEREG_FLAG'] == 'UP'].index)
)
print(f"Full shared DOWN genes: {len(shared_down)}")
print(f"Full shared UP genes:   {len(shared_up)}")

# ── Background = genes with a non-NA padj (i.e., actually tested) in ALL THREE ──
tested_ko   = set(ko_df  [ko_df  ['padj'].notna()].index)
tested_kd81 = set(kd81_df[kd81_df['padj'].notna()].index)
tested_kd82 = set(kd82_df[kd82_df['padj'].notna()].index)
background  = sorted(tested_ko & tested_kd81 & tested_kd82)
print(f"Background (tested in all 3 comparisons): {len(background)} genes")

# ── Run Enrichr GO_Biological_Process, custom background (S9-style) ────────
def run_enrichr(gene_list, label):
    if len(gene_list) < 3:
        print(f"Skipping {label}: too few genes ({len(gene_list)})")
        return None
    enr = gp.enrichr(
        gene_list      = gene_list,
        gene_sets      = ['GO_Biological_Process_2023'],
        background     = background,
        outdir         = None,
    )
    res = enr.results.copy()
    print(f"[{label}] columns returned by gseapy: {list(res.columns)}")

    # 'Genes' -> 'Overlap_genes' (always present)
    res = res.rename(columns={'Genes': 'Overlap_genes'})

    # Overlap_n and Term_size_bg: parse from 'Overlap' (e.g. '5/40') if present,
    # otherwise fall back to counting genes in Overlap_genes (Term_size_bg unavailable).
    if 'Overlap' in res.columns:
        res['Overlap_n']    = res['Overlap'].apply(lambda x: int(str(x).split('/')[0]))
        res['Term_size_bg'] = res['Overlap'].apply(lambda x: int(str(x).split('/')[1]))
    else:
        res['Overlap_n']    = res['Overlap_genes'].apply(
            lambda x: len(str(x).split(';')) if pd.notna(x) else 0
        )
        res['Term_size_bg'] = np.nan  # not returned by this gseapy version

    res['Query_size']   = len(gene_list)
    res['Background_n'] = len(background)

    keep_cols = ['Term', 'P-value', 'Overlap_n', 'Term_size_bg',
                 'Query_size', 'Background_n', 'Overlap_genes', 'Adjusted P-value']
    if 'Overlap' in res.columns:
        keep_cols.insert(-1, 'Overlap')
    res = res[[c for c in keep_cols if c in res.columns]]
    res = res.sort_values('Adjusted P-value')
    out_path = os.path.join(output_dir, f"GO_enrichment_{label}.csv")
    res.to_csv(out_path, index=False)
    print(f"✅ Saved: {out_path}")
    return res

down_res = run_enrichr(shared_down, "shared_down")
up_res   = run_enrichr(shared_up,   "shared_up")

# ── Combine into one supplementary table (S10-style) ────────────────────────
if down_res is not None:
    down_res.insert(0, 'Direction', 'Downregulated')
if up_res is not None:
    up_res.insert(0, 'Direction', 'Upregulated')

combined = pd.concat([df for df in [down_res, up_res] if df is not None], ignore_index=True)
combined_path = os.path.join(output_dir, "Supplementary_Table_S10_GO_shared_genes.csv")
combined.to_csv(combined_path, index=False)
print(f"✅ Combined supplementary table: {combined_path}")

print("\nTop 5 terms — DOWN:")
print(down_res[['Term', 'Adjusted P-value', 'Overlap_n']].head(5).to_string(index=False) if down_res is not None else "n/a")
print("\nTop 5 terms — UP:")
print(up_res[['Term', 'Adjusted P-value', 'Overlap_n']].head(5).to_string(index=False) if up_res is not None else "n/a")

from google.colab import files
files.download(combined_path)

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
ko_path   = os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")
kd1_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv")
kd2_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")

# ── AC6: fill this in once you tell me the actual path/filename ────────────
# es_path = os.path.join(base_path, "Diff_<ES_cell_comparison_name>.tsv")

def load_and_flag(path):
    df = pd.read_csv(path, sep='\t')
    gene_col = next(
        (c for c in df.columns if c.upper() in ['GENESYMBOL', 'SYMBOL', 'MGI_SYMBOL']),
        'GENESYMBOL'
    )
    df = df.rename(columns={gene_col: 'GENESYMBOL'})
    df['DEREG_FLAG'] = np.where(
        (df['log2FoldChange'] < 0) & (df['padj'] < 0.05), 'DOWN',
        np.where((df['log2FoldChange'] > 0) & (df['padj'] < 0.05), 'UP', 'NONE')
    )
    return df.set_index('GENESYMBOL')

ko_df   = load_and_flag(ko_path)
kd81_df = load_and_flag(kd1_path)
kd82_df = load_and_flag(kd2_path)

# ═════════════════════════════════════════════════════════════════════════
# AC1 — per-condition UP/DOWN DEG counts
# Tests whether "greater number of downregulated genes" holds for ALL
# three conditions, or only KO + KD8.2 (as the comment suspects).
# ═════════════════════════════════════════════════════════════════════════
print("="*60)
print("AC1: Per-condition DEG counts")
print("="*60)

summary = []
for name, df in [('KO', ko_df), ('KD 8.1', kd81_df), ('KD 8.2', kd82_df)]:
    n_down = (df['DEREG_FLAG'] == 'DOWN').sum()
    n_up   = (df['DEREG_FLAG'] == 'UP').sum()
    ratio  = n_down / n_up if n_up > 0 else np.nan
    down_biased = n_down > n_up
    summary.append({
        'Condition': name, 'DOWN': n_down, 'UP': n_up,
        'DOWN/UP ratio': round(ratio, 2), 'Down-biased?': down_biased
    })
    print(f"{name:8s}  DOWN={n_down:5d}   UP={n_up:5d}   "
          f"ratio={ratio:.2f}   down-biased={down_biased}")

summary_df = pd.DataFrame(summary)
print("\n", summary_df.to_string(index=False))

# ═════════════════════════════════════════════════════════════════════════
# AC5 — pluripotency & cell-cycle gene status across conditions
# Checks whether canonical pluripotency genes are actually UP (activated)
# or DOWN, and whether cell-cycle regulators are dysregulated.
# Edit these gene lists if your field-standard panel differs.
# ═════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("AC5: Pluripotency and cell-cycle gene status")
print("="*60)

pluripotency_genes = [
    'Nanog', 'Pou5f1', 'Sox2', 'Klf4', 'Zfp42', 'Dppa3', 'Dppa5a',
    'Esrrb', 'Tbx3', 'Sall4', 'Lin28a', 'Utf1'
]
cell_cycle_genes = [
    'Cdkn1a', 'Cdkn2a', 'Cdkn1b', 'Ccnd1', 'Ccne1', 'Ccnb1',
    'Mki67', 'Mdm2', 'Rb1', 'E2f1', 'Cdk4', 'Cdk6'
]

def gene_status_table(gene_list, label):
    rows = []
    for g in gene_list:
        row = {'Gene': g}
        for name, df in [('KO', ko_df), ('KD 8.1', kd81_df), ('KD 8.2', kd82_df)]:
            if g in df.index:
                flag = df.loc[g, 'DEREG_FLAG']
                lfc  = df.loc[g, 'log2FoldChange']
                row[f'{name}_flag'] = flag
                row[f'{name}_log2FC'] = round(float(lfc), 2)
            else:
                row[f'{name}_flag'] = 'not_tested'
                row[f'{name}_log2FC'] = np.nan
        rows.append(row)
    out = pd.DataFrame(rows)
    print(f"\n--- {label} ---")
    print(out.to_string(index=False))
    return out

pluri_table = gene_status_table(pluripotency_genes, "Pluripotency genes")
cc_table    = gene_status_table(cell_cycle_genes,   "Cell-cycle regulators")

# Quick tally: how many pluripotency genes are UP vs DOWN in at least one condition?
def tally_direction(table, label):
    flags = table[[c for c in table.columns if c.endswith('_flag')]]
    any_up   = (flags == 'UP').any(axis=1).sum()
    any_down = (flags == 'DOWN').any(axis=1).sum()
    print(f"\n{label}: {any_up} genes UP in ≥1 condition, "
          f"{any_down} genes DOWN in ≥1 condition (out of {len(table)} tested)")

tally_direction(pluri_table, "Pluripotency panel")
tally_direction(cc_table,    "Cell-cycle panel")

# ── Save outputs ──────────────────────────────────────────────────────────
output_dir = "/content/drive/MyDrive/Chd8 data/AC1_AC5_output"
os.makedirs(output_dir, exist_ok=True)
summary_df.to_csv(os.path.join(output_dir, "AC1_per_condition_DEG_counts.csv"), index=False)
pluri_table.to_csv(os.path.join(output_dir, "AC5_pluripotency_gene_status.csv"), index=False)
cc_table.to_csv(os.path.join(output_dir, "AC5_cellcycle_gene_status.csv"), index=False)
print(f"\n✅ Saved outputs to: {output_dir}")

In [ ]:
# ── 0. Installs & imports ─────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['gseapy', 'mygene', 'pybedtools']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pybedtools
from scipy.stats import (friedmanchisquare, wilcoxon, mannwhitneyu,
                          kruskal, chisquare, fisher_exact, pearsonr)
from statsmodels.stats.multitest import multipletests
import gseapy as gp
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# ── 1. Global rcParams —  ─────────────────────────────
matplotlib.rcParams.update({
    'font.family'       : 'sans-serif',
    'font.sans-serif'   : ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'font.size'         : 6,
    'axes.titlesize'    : 7,
    'axes.labelsize'    : 6,
    'xtick.labelsize'   : 6,
    'ytick.labelsize'   : 6,
    'legend.fontsize'   : 5.5,
    'axes.linewidth'    : 0.75,
    'xtick.major.width' : 0.75,
    'ytick.major.width' : 0.75,
    'xtick.major.size'  : 2.5,
    'ytick.major.size'  : 2.5,
    'lines.linewidth'   : 0.75,
    'pdf.fonttype'      : 42,
    'ps.fonttype'       : 42,
    'figure.dpi'        : 300,
    'savefig.dpi'       : 300,
})

MM  = 1 / 25.4
FIG_W = 170 * MM   # 6.693 in
FIG_H = 250 * MM   # 9.843 in
DPI   = 300

# ── 2. Paths ──────────────────────────────────────────────────────────────────
RNA_DIR   = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
ATAC_BASE = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
CHIP_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
SUPP_SIG  = "/content/drive/MyDrive/Chd8 data/figures/SuppTable_RescueGroups_ContinuousSignals_v2.csv"
SUPP_RF   = "/content/drive/MyDrive/Chd8 data/figures/Supp_Table_Domain_Rescue_Groups.csv"

RNA_PATHS = {
    'KO' : RNA_DIR + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    'FL' : RNA_DIR + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    'dC' : RNA_DIR + "Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    'dH' : RNA_DIR + "Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}
ATAC_WT = [ATAC_BASE + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
           ATAC_BASE + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed"]
ATAC_KO = [ATAC_BASE + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
           ATAC_BASE + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed"]

output_dir = "/content/Figure7_output"
os.makedirs(output_dir, exist_ok=True)

# ── 3. Palettes ───────────────────────────────────────────────────────────────
RESCUE_PALETTE = {
    'Global Rescue':         '#2166ac',
    'Helicase Required':     '#35978f',
    'Chromodomain Required': '#762a83',
    'Dual Domain Required':  '#e08214',
    'Low/No Rescue':         '#b2b2b2',
}
ORDER = ['Global Rescue', 'Helicase Required', 'Chromodomain Required',
         'Dual Domain Required', 'Low/No Rescue']
WRAP = {
    'Global Rescue':         'Global\nRescue',
    'Helicase Required':     'Helicase\nRequired',
    'Chromodomain Required': 'Chromodomain\nRequired',
    'Dual Domain Required':  'Dual Domain\nRequired',
    'Low/No Rescue':         'Low/No\nRescue',
}

def sig_label(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    return 'ns'

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ── 4. Load & compute all data ────────────────────────────────────────────────
print("Loading RNA-seq data...")
dfs_rna = {}
for label, path in RNA_PATHS.items():
    df = pd.read_csv(path, sep='\t')
    df['gene_id']    = df.iloc[:, 0].astype(str).str.split('.').str[0]
    df['gene_upper'] = df['GENESYMBOL'].astype(str).str.upper().str.strip()
    dfs_rna[label]   = df

# Master merge
master = dfs_rna['KO'][['gene_id','gene_upper','GENESYMBOL',
                          'log2FoldChange','baseMean']].rename(
                              columns={'log2FoldChange': 'LFC_KO'})
for label in ['FL', 'dC', 'dH']:
    tmp = dfs_rna[label][['gene_id','log2FoldChange']].rename(
        columns={'log2FoldChange': f'LFC_{label}'})
    master = master.merge(tmp, on='gene_id')

master = master.dropna(subset=['LFC_KO','LFC_FL','LFC_dC','LFC_dH'])

# Rescue fractions
for col, lfc in [('RF_FL','LFC_FL'),('RF_dC','LFC_dC'),('RF_dH','LFC_dH')]:
    denom = (0 - master['LFC_KO']).replace(0, np.nan)
    master[col] = (master[lfc] - master['LFC_KO']) / denom

# NPC-specific ChIP targets
print("Identifying NPC ChIP targets...")
npc_bt  = pybedtools.BedTool(CHIP_PATH).each(fix_chr).sort()
esc_bt  = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt  = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()
npc_specific = npc_bt.subtract(esc_bt, A=True)
hits         = tss_bt.intersect(npc_specific, u=True, wa=True)
target_set   = set([str(f[3]).upper().strip()
                    for f in hits if len(str(f[3])) > 1])

targets = master[
    (master['gene_upper'].isin(target_set)) &
    (master['LFC_KO'].abs() > 0.5)
].copy()
print(f"  {len(targets)} NPC targets with |LFC_KO| > 0.5")

# Panel A stats
_, fried_p = friedmanchisquare(targets['RF_FL'], targets['RF_dC'], targets['RF_dH'])
_, p_fl_dc_raw = wilcoxon(targets['RF_FL'], targets['RF_dC'], alternative='two-sided')
_, p_fl_dh_raw = wilcoxon(targets['RF_FL'], targets['RF_dH'], alternative='two-sided')
_, pvals_corr, _, _ = multipletests([p_fl_dc_raw, p_fl_dh_raw], method='fdr_bh')
p_fl_dc, p_fl_dh = pvals_corr

# Panel B categories
RF_THRESH = 0.5
targets['FL_r'] = targets['RF_FL'] >= RF_THRESH
targets['dC_r'] = targets['RF_dC'] >= RF_THRESH
targets['dH_r'] = targets['RF_dH'] >= RF_THRESH

def assign_cat(row):
    fl, dc, dh = row['FL_r'], row['dC_r'], row['dH_r']
    if fl and dc and dh:            return 'Global Rescue'
    elif fl and dh and not dc:      return 'Chromodomain Required'
    elif fl and dc and not dh:      return 'Helicase Required'
    elif fl and not dc and not dh:  return 'Dual Domain Required'
    else:                           return 'Low/No Rescue'

targets['category'] = targets.apply(assign_cat, axis=1)
cat_counts = {k: (targets['category'] == k).sum() for k in ORDER}
total_n    = len(targets)
chi2_stat, p_cat = chisquare(list(cat_counts.values()))

# Panel C — ATAC + rescue
print("Loading ATAC data...")
def load_atac(paths):
    dfs = []
    for p in paths:
        if not os.path.exists(p): continue
        df = pd.read_csv(p, sep='\t', header=None)
        sc = 6 if len(df.columns) > 6 else 4
        out = df[[0,1,2,sc]].copy()
        out.columns = ['chr','start','end','score']
        out['chr'] = out['chr'].astype(str)
        mask = ~out['chr'].str.startswith('chr')
        out.loc[mask,'chr'] = 'chr' + out.loc[mask,'chr']
        out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
        out['bin'] = out['chr'] + ':' + ((out['start']//50)*50).astype(str)
        dfs.append(out)
    combined = pd.concat(dfs, ignore_index=True)
    return combined.groupby('bin').agg(
        chr=('chr','first'), start=('start','min'),
        end=('end','max'),   score=('score','mean')
    ).reset_index()

wt_atac = load_atac(ATAC_WT)
ko_atac = load_atac(ATAC_KO)
wt_atac['bin'] = wt_atac['chr'] + ':' + ((wt_atac['start']//50)*50).astype(str)
ko_atac['bin'] = ko_atac['chr'] + ':' + ((ko_atac['start']//50)*50).astype(str)
merged_atac = pd.merge(
    wt_atac[['bin','score']].rename(columns={'score':'wt'}),
    ko_atac[['bin','score']].rename(columns={'score':'ko'}),
    on='bin', how='inner')
pseudo = 0.5
merged_atac['lfc'] = np.log2((merged_atac['ko']+pseudo)/(merged_atac['wt']+pseudo))
ko_lost = merged_atac[merged_atac['lfc'] < -0.5].copy()

ko_lost_bed = "/content/ko_lost.bed"
ko_lost_rows = []
for _, row in ko_lost.iterrows():
    parts = row['bin'].split(':')
    chr_ = parts[0]
    start = int(parts[1])
    ko_lost_rows.append([chr_, start, start+50])
pd.DataFrame(ko_lost_rows).to_csv(ko_lost_bed, sep='\t', header=False, index=False)

if os.path.exists(TSS_PATH):
    tss2   = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()
    kl_bt  = pybedtools.BedTool(ko_lost_bed).each(fix_chr).sort()
    hits2  = tss2.intersect(kl_bt, u=True, wa=True)
    lost_genes = set([str(f[3]).upper().strip()
                      for f in hits2 if len(str(f[3])) > 1])
else:
    lost_genes = set()

ko_base = master[['gene_upper','LFC_KO']].copy()
atac_results = {}
for construct, lfc_col in [('FL','LFC_FL'),('dC','LFC_dC'),('dH','LFC_dH')]:
    tmp = master[['gene_upper', lfc_col]].copy()
    mg  = ko_base.merge(tmp, on='gene_upper')
    denom = (0 - mg['LFC_KO']).replace(0, np.nan)
    mg['RF'] = ((mg[lfc_col] - mg['LFC_KO']) / denom).clip(-1, 2)
    mg['near_lost'] = mg['gene_upper'].isin(lost_genes)
    atac_results[construct] = mg

atac_stats = {}
for construct, df in atac_results.items():
    near  = df[df['near_lost']]['RF'].dropna()
    other = df[~df['near_lost']]['RF'].dropna()
    _, pval = (mannwhitneyu(near, other, alternative='two-sided')
               if len(near) > 5 and len(other) > 5 else (0, np.nan))
    atac_stats[construct] = dict(n_near=len(near), n_other=len(other),
                                  med_near=near.median(), med_other=other.median(),
                                  pval=pval)

# Panel D — CHD8 binding
print("Loading rescue groups for Panel D...")
try:
    sig = pd.read_csv(SUPP_SIG)
    rf  = pd.read_csv(SUPP_RF)
    sig['gene_upper'] = sig['gene'].str.upper()
    rf['gene_upper']  = rf['gene_symbol'].str.upper()
    df_d = rf.merge(sig[['gene_upper','CHD8_bound','CHD8_score']], on='gene_upper', how='left')
    df_d = df_d[df_d['rescue_group'].isin(ORDER)]
    order_d = [o for o in ORDER if (df_d['rescue_group']==o).sum() >= 3]
    df_d['CHD8_score_log'] = np.log10(df_d['CHD8_score'].clip(lower=1))
    kw_stat_d, kw_p_d = kruskal(*[
        df_d[df_d['rescue_group']==o]['CHD8_score_log'].dropna().values
        for o in order_d if len(df_d[df_d['rescue_group']==o]) > 1
    ])
    bound_pct = (df_d.groupby('rescue_group')
                     .apply(lambda x: x['CHD8_bound'].sum()/len(x)*100)
                     .reindex(ORDER))
    panel_d_available = True
except Exception as e:
    print(f"  Panel D data not available: {e}")
    panel_d_available = False

# Panel E — GO enrichment
print("Running GO enrichment for Panel E...")
go_master = master[(master['baseMean'] > 10) & (master['LFC_KO'].abs() > 0.5)].copy()
# Add RF columns and boolean flags so assign_cat works on go_master too
for col, lfc in [('RF_FL','LFC_FL'),('RF_dC','LFC_dC'),('RF_dH','LFC_dH')]:
    denom = (0 - go_master['LFC_KO']).replace(0, np.nan)
    go_master[col] = (go_master[lfc] - go_master['LFC_KO']) / denom
go_master['FL_r'] = go_master['RF_FL'] >= RF_THRESH
go_master['dC_r'] = go_master['RF_dC'] >= RF_THRESH
go_master['dH_r'] = go_master['RF_dH'] >= RF_THRESH
go_master['category'] = go_master.apply(assign_cat, axis=1)

def wrap_term(t):
    t = t.split(' (GO')[0].strip()
    words = t.split()
    lines, cur, length = [], [], 0
    for w in words:
        if length + len(w) > 26 and cur:
            lines.append(' '.join(cur))
            if len(lines) == 3: break
            cur, length = [w], len(w)+1
        else:
            cur.append(w); length += len(w)+1
    if cur and len(lines) < 3:
        lines.append(' '.join(cur))
    return '\n'.join(lines)

try:
    go_lib = gp.get_library('GO_Biological_Process_2023', organism='Mouse')
    all_genes = set(go_master['GENESYMBOL'].str.upper().dropna())
    go_results = {}
    for cat in ['Chromodomain Required', 'Low/No Rescue']:
        query = set(go_master[go_master['category']==cat]['GENESYMBOL'].str.upper().dropna())
        rows = []
        for term, tgenes in go_lib.items():
            tset = set(g.upper() for g in tgenes) & all_genes
            if len(tset) < 3: continue
            a = len(query & tset)
            if a == 0: continue
            b = len(query)-a; c = len(tset)-a
            d = len(all_genes)-len(query)-c
            _, p = fisher_exact([[a,b],[c,d]], alternative='greater')
            rows.append({'Term': term, 'n_overlap': a, 'p_value': p})
        if rows:
            res = pd.DataFrame(rows)
            res['padj'] = multipletests(res['p_value'], method='fdr_bh')[1]
            go_results[cat] = res[res['padj'] < 0.05].sort_values('padj')
    panel_e_available = True
except Exception as e:
    print(f"  Panel E GO not available: {e}")
    panel_e_available = False

# ── 5. Build figure ───────────────────────────────────────────────────────────
print("Building combined figure...")

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=DPI)

gs = gridspec.GridSpec(
    4, 1,
    figure        = fig,
    hspace        = 0.55,
    top           = 0.97,
    bottom        = 0.04,
    left          = 0.10,
    right         = 0.97,
    height_ratios = [2.2, 2.0, 1.8, 2.0],
)

gs_ab = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[0],
                                          wspace=0.35, width_ratios=[1,1])
gs_c  = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[1], wspace=0.30)
gs_d  = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[2],
                                          wspace=0.35, width_ratios=[1.2, 1])
gs_e  = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[3], wspace=0.45)

# ── PANEL A — Rescue fraction violin/box/strip ────────────────────────────────
ax_a = fig.add_subplot(gs_ab[0])

colors_a = ['#27ae60', '#f1c40f', '#e74c3c']
plot_cols = ['RF_FL', 'RF_dC', 'RF_dH']
sns.violinplot(data=targets[plot_cols], palette=colors_a,
               inner=None, alpha=0.15, ax=ax_a, linewidth=0.5)
sns.boxplot(data=targets[plot_cols], palette=colors_a,
            showfliers=False, width=0.3, linewidth=0.8, ax=ax_a)
sns.stripplot(data=targets[plot_cols], color='black',
              size=0.8, alpha=0.2, jitter=0.2, ax=ax_a)

ax_a.axhline(1, ls='--', color='#2980b9', lw=0.6, alpha=0.5, label='Full rescue')
ax_a.axhline(0, ls='-',  color='black',   lw=0.5, alpha=0.2, label='No rescue')
ax_a.set_ylim(-1.5, 2.6)
ax_a.set_xticks([0,1,2])
ax_a.set_xticklabels(['Full Length', 'ΔChromo', 'ΔHelicase'],
                      fontsize=6, fontweight='bold')
ax_a.set_ylabel('Rescue Fraction (Normalised)', fontsize=6, fontweight='bold')
ax_a.tick_params(labelsize=5)

y1, y2 = 2.1, 2.35
for x1, x2, y, p in [(0,1,y1,p_fl_dc),(0,2,y2,p_fl_dh)]:
    ax_a.plot([x1,x1,x2,x2],[y,y+0.05,y+0.05,y], color='black', lw=0.5)
    ax_a.text((x1+x2)/2, y+0.06, sig_label(p),
              ha='center', va='bottom', fontsize=6)

ax_a.legend(frameon=False, fontsize=5, loc='upper right')
sns.despine(offset=4, trim=True, ax=ax_a)
ax_a.text(-0.18, 1.04, 'A', transform=ax_a.transAxes,
          fontsize=8, fontweight='bold', va='top')

# ── PANEL B — Category bar chart ──────────────────────────────────────────────
ax_b = fig.add_subplot(gs_ab[1])

b_labels = [WRAP[k] for k in ORDER]
b_counts = [cat_counts[k] for k in ORDER]
b_colors = [RESCUE_PALETTE[k] for k in ORDER]

bars = ax_b.bar(range(len(ORDER)), b_counts,
                color=b_colors, edgecolor='black', linewidth=0.4)
for bar in bars:
    h = bar.get_height()
    ax_b.text(bar.get_x() + bar.get_width()/2, h + total_n*0.01,
              f'{int(h)}\n({100*h/total_n:.1f}%)',
              ha='center', va='bottom', fontweight='bold', fontsize=5)

ax_b.set_xticks(range(len(ORDER)))
ax_b.set_xticklabels(b_labels, fontsize=5, rotation=35,
                      ha='right', rotation_mode='anchor')
ax_b.set_ylabel(f'Number of Direct Targets (n={total_n})', fontsize=6, fontweight='bold')
ax_b.set_ylim(0, max(b_counts)*1.30)
ax_b.tick_params(axis='y', labelsize=5)
ax_b.tick_params(axis='x', length=0)
sns.despine(ax=ax_b)
ax_b.text(-0.22, 1.04, 'B', transform=ax_b.transAxes,
          fontsize=8, fontweight='bold', va='top')

# ── PANEL C — ATAC near-lost-peak violins ─────────────────────────────────────
C_NEAR  = '#2166ac'
C_OTHER = '#b2b2b2'
labels_c    = ['Full-length', 'ΔChromo', 'ΔHelicase']
constructs_c = ['FL', 'dC', 'dH']

for i, (construct, title) in enumerate(zip(constructs_c, labels_c)):
    ax = fig.add_subplot(gs_c[i])
    df_c = atac_results[construct]
    st   = atac_stats[construct]
    near  = df_c[df_c['near_lost']]['RF'].dropna()
    other = df_c[~df_c['near_lost']]['RF'].dropna()

    plot_df_c = pd.DataFrame({
        'RF'   : pd.concat([near, other]),
        'Group': (['Near\nlost peak']*len(near) + ['Other\ngenes']*len(other))
    })

    sns.violinplot(data=plot_df_c, x='Group', y='RF', ax=ax,
                   palette={'Near\nlost peak': C_NEAR, 'Other\ngenes': C_OTHER},
                   inner='quartile', linewidth=0.75,
                   order=['Near\nlost peak','Other\ngenes'], saturation=0.85)

    ax.axhline(0, color='#555', linestyle='--', lw=0.6, alpha=0.6)
    ax.axhline(1, color='#555', linestyle=':',  lw=0.6, alpha=0.6)
    ax.set_ylim(-1.05, 2.05)
    ax.set_xlabel('')
    ax.set_ylabel('Rescue fraction (RF)' if i==0 else '', fontsize=6, labelpad=3)
    ax.set_title(title, fontsize=7, fontweight='bold', pad=4)
    ax.set_xticklabels(['Near\nlost peak','Other\ngenes'], fontsize=6)
    ax.tick_params(labelsize=5, length=2.5, width=0.75, pad=2)

    textstr = (f"n={st['n_near']} | med={st['med_near']:.2f}\n"
               f"n={st['n_other']} | med={st['med_other']:.2f}\n"
               f"MWU p={st['pval']:.1e}")
    ax.text(0.97, 0.97, textstr, transform=ax.transAxes,
            va='top', ha='right', fontsize=5,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                      edgecolor='#ccc', linewidth=0.5, alpha=0.9))
    sns.despine(ax=ax, offset=3, trim=True)
    if i == 0:
        ax.text(-0.28, 1.06, 'C', transform=ax.transAxes,
                fontsize=8, fontweight='bold', va='top')

# ── PANEL D — CHD8 binding intensity + occupancy ─────────────────────────────
ax_d1 = fig.add_subplot(gs_d[0])
ax_d2 = fig.add_subplot(gs_d[1])

if panel_d_available:
    plot_d = df_d[df_d['CHD8_score_log'] > 0]
    order_present = [o for o in ORDER if o in plot_d['rescue_group'].unique()]
    sns.violinplot(data=plot_d, x='rescue_group', y='CHD8_score_log',
                   order=order_present, palette=RESCUE_PALETTE,
                   inner='box', cut=0, linewidth=0.75, saturation=0.85, ax=ax_d1)
    ax_d1.set_xticks(range(len(order_present)))
    ax_d1.set_xticklabels([WRAP[o] for o in order_present],
                           fontsize=5, rotation=35, ha='right', rotation_mode='anchor')
    ax_d1.set_ylabel('CHD8 ChIP-seq peak score (log₁₀)', fontsize=6, labelpad=3)
    ax_d1.set_xlabel('')
    ax_d1.tick_params(labelsize=5, length=2.5, width=0.75, pad=2)
    ax_d1.text(0.97, 0.03, f'Kruskal–Wallis\np = {kw_p_d:.3f}',
               transform=ax_d1.transAxes, ha='right', va='bottom', fontsize=5,
               bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                         edgecolor='#ccc', linewidth=0.5, alpha=0.9))
    sns.despine(ax=ax_d1, offset=3, trim=True)

    bars_d2 = ax_d2.bar(range(len(ORDER)), bound_pct.values,
                         color=[RESCUE_PALETTE[o] for o in ORDER],
                         edgecolor='white', linewidth=0.75, width=0.6)
    ax_d2.set_xticks(range(len(ORDER)))
    ax_d2.set_xticklabels([WRAP[o] for o in ORDER],
                           fontsize=5, rotation=35, ha='right', rotation_mode='anchor')
    ax_d2.set_ylabel('Genes with CHD8 peak\nwithin 2 kb TSS (%)', fontsize=6, labelpad=3)
    ax_d2.set_ylim(0, 115)
    ax_d2.tick_params(labelsize=5, length=2.5, width=0.75, pad=2)
    for i2, v in enumerate(bound_pct.values):
        if not np.isnan(v):
            ax_d2.text(i2, v+1.5, f'{v:.0f}%', ha='center',
                       fontsize=5.5, fontweight='bold')
    sns.despine(ax=ax_d2, offset=3, trim=True)
else:
    for ax in [ax_d1, ax_d2]:
        ax.text(0.5, 0.5, 'Data not available', ha='center', va='center',
                transform=ax.transAxes, fontsize=6)

ax_d1.text(-0.28, 1.06, 'D', transform=ax_d1.transAxes,
           fontsize=8, fontweight='bold', va='top')

# ── PANEL E — GO enrichment + category sizes ──────────────────────────────────
cats_e = ['Chromodomain Required', 'Low/No Rescue']
for i, cat in enumerate(cats_e):
    ax = fig.add_subplot(gs_e[i])
    if panel_e_available and cat in go_results and not go_results[cat].empty:
        top = go_results[cat].head(6).copy()
        top['-log10padj'] = -np.log10(top['padj'].clip(lower=1e-10))
        top['Term_short'] = top['Term'].apply(wrap_term)
        top = top.sort_values('-log10padj')
        bars_e = ax.barh(top['Term_short'], top['-log10padj'],
                         color=RESCUE_PALETTE[cat], edgecolor='white',
                         linewidth=0.75, alpha=0.85, height=0.5)
        ax.axvline(-np.log10(0.05), color='#333', linestyle='--', lw=0.75)
        for bar, val in zip(bars_e, top['-log10padj']):
            ax.text(val + 0.03, bar.get_y() + bar.get_height()/2,
                    f'{val:.2f}', va='center', ha='left', fontsize=4.5)

        # Fix x-axis limit so bar doesn't run off the edge
        ax.set_xlim(0, top['-log10padj'].max() * 1.35)

    else:
        ax.text(0.5, 0.5, 'No significant\nterms (FDR<0.05)',
                ha='center', va='center', transform=ax.transAxes, fontsize=6)

    n = (go_master['category'] == cat).sum()
    ax.set_title(f'{cat}\n(n={n})', fontsize=6.5, fontweight='bold',
                 color=RESCUE_PALETTE[cat], pad=3)
    ax.set_xlabel(r'$-\log_{10}$(FDR)', fontsize=6, labelpad=3)

    # Fix y-tick label size and add padding so long GO terms don't get clipped
    ax.tick_params(axis='y', labelsize=4.5, pad=2)
    ax.tick_params(axis='x', labelsize=5.5, length=2.5, width=0.75, pad=2)
    ax.margins(y=0.20)

    # Left-pad the axes so long y-tick labels aren't clipped
    ax.yaxis.set_tick_params(pad=3)
    plt.setp(ax.get_yticklabels(), ha='right')

    sns.despine(ax=ax, offset=3, trim=True)
    if i == 0:
        ax.text(-0.55, 1.06, 'E', transform=ax.transAxes,
                fontsize=8, fontweight='bold', va='top')

# Category sizes bar (panel E right) — fix overlapping x labels
ax_e3 = fig.add_subplot(gs_e[2])
cat_counts_all = go_master['category'].value_counts().reindex(ORDER)
ax_e3.bar(range(len(ORDER)), cat_counts_all.values,
          color=[RESCUE_PALETTE[o] for o in ORDER],
          edgecolor='white', linewidth=0.75, width=0.6)
ax_e3.set_xticks(range(len(ORDER)))

# Use short single-line labels to avoid overlap
SHORT_LABELS = ['Global\nRescue', 'Helicase\nReq.', 'Chromo.\nReq.',
                'Dual\nReq.', 'Low/No\nRescue']
ax_e3.set_xticklabels(SHORT_LABELS, fontsize=5, rotation=0, ha='center')
ax_e3.set_ylabel('Number of genes', fontsize=6, labelpad=3)
ax_e3.tick_params(labelsize=5, length=2.5, width=0.75, pad=2)
ax_e3.tick_params(axis='x', length=0, pad=4)
ax_e3.set_title('Category sizes\n(KO-dysregulated genes)',
                fontsize=6, fontweight='bold', pad=3)
ax_e3.margins(y=0.14)
for i3, v in enumerate(cat_counts_all.values):
    if not np.isnan(v):
        ax_e3.text(i3, v + 5, str(int(v)), ha='center',
                   fontsize=5.5, fontweight='bold')
sns.despine(ax=ax_e3, offset=3, trim=True)

# ── 6. Save ───────────────────────────────────────────────────────────────────
pdf_path  = os.path.join(output_dir, "Figure7_combined.pdf")
tiff_path = os.path.join(output_dir, "Figure7_combined.tiff")

fig.savefig(pdf_path, format='pdf', bbox_inches='tight', pad_inches=0.02)
print(f"✅ PDF:  {pdf_path}")

fig.savefig(tiff_path, format='tiff', dpi=DPI,
            bbox_inches='tight', pad_inches=0.02,
            pil_kwargs={'compression': 'tiff_lzw'})
print(f"✅ TIFF: {tiff_path}")
plt.close(fig)

from google.colab import files
files.download(pdf_path)
files.download(tiff_path)
print("\n✅ Downloads triggered.")

In [ ]:
!pip install matplotlib-venn pybedtools

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from scipy import stats
from matplotlib.colors import to_rgb
import os

base     = '/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/'
save_dir = '/content/drive/MyDrive/Chd8 data/figures/Chd8_expression/'
os.makedirs(save_dir, exist_ok=True)

# ══════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ══════════════════════════════════════════════════════════════
paths = {
    'KO': base + 'Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv',
    'FL': base + 'Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv',
    'dC': base + 'Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv',
    'dH': base + 'Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv',
}

CTRL_COLS = ['HJH5TBGX3_2','HJH5TBGX3_6','HJH5TBGX3_10',
             'HKLWCBGX5_18','HTGK7BGX5_15','HKM5YBGX5_28','HKNFWBGX5_11']

EXP_COLS = {
    'KO': ['HJH5TBGX3_8','HJH5TBGX3_12','HJH5TBGX3_4','HKNFWBGX5_7','HKLWCBGX5_17'],
    'FL': ['HKNFWBGX5_3','HKNFWBGX5_10','HKLWCBGX5_21'],
    'dC': ['HKNFWBGX5_1','HKNFWBGX5_8','HKLWCBGX5_19'],
    'dH': ['HKNFWBGX5_2','HKNFWBGX5_9','HKLWCBGX5_20'],
}

replicates = {}
deseq      = {}

for label, path in paths.items():
    df   = pd.read_csv(path, sep='\t')
    chd8 = df[df['GENESYMBOL'].str.upper() == 'CHD8'].iloc[0]
    replicates[label] = {
        'ctrl': chd8[CTRL_COLS].values.astype(float),
        'exp' : chd8[EXP_COLS[label]].values.astype(float),
    }
    deseq[label] = {
        'log2FC'  : round(float(chd8['log2FoldChange']), 3),
        'padj'    : float(chd8['padj']),
        'baseMean': round(float(chd8['baseMean']), 1),
    }

ctrl_mean = np.mean(replicates['KO']['ctrl'])   # same CTRL across all

# ══════════════════════════════════════════════════════════════
# 2. SUMMARY TABLE
# ══════════════════════════════════════════════════════════════
def sig_label(p):
    if   p > 0.05 : return 'ns'
    elif p > 0.01 : return '*'
    elif p > 0.001: return '**'
    else          : return '***'

def direction(lfc, p):
    if p > 0.05: return 'Not significant'
    return '↓ Reduced' if lfc < 0 else '↑ Increased'

rows = []
for label in ['KO','FL','dC','dH']:
    exp_vals  = replicates[label]['exp']
    ctrl_vals = replicates[label]['ctrl']
    rows.append({
        'Condition'       : label,
        'Comparison'      : f'{label} vs CTRL (WT)',
        'CTRL mean (log2)': round(np.mean(ctrl_vals), 3),
        'Exp mean (log2)' : round(np.mean(exp_vals),  3),
        'log2FoldChange'  : deseq[label]['log2FC'],
        'padj (DESeq2)'   : f"{deseq[label]['padj']:.2e}",
        'Significance'    : sig_label(deseq[label]['padj']),
        'Direction'       : direction(deseq[label]['log2FC'], deseq[label]['padj']),
        'N_CTRL'          : len(ctrl_vals),
        'N_Exp'           : len(exp_vals),
        'Note'            : '† Frameshift KO — residual RNA expected' if label == 'KO' else '',
    })

table = pd.DataFrame(rows)

print("\n" + "═"*75)
print("  Chd8 expression check — CTRL vs KO / FL / ∆C / ∆H")
print("═"*75)
print(f"{'Condition':<10} {'log2FC':>8} {'padj':>12} {'Sig':>6}  {'Interpretation'}")
print("─"*75)
for _, r in table.iterrows():
    note = ' †' if r['Note'] else ''
    print(f"{r['Condition']:<10} {r['log2FoldChange']:>8.3f} "
          f"{r['padj (DESeq2)']:>12} {r['Significance']:>6}  "
          f"{r['Direction']}{note}")
print("─"*75)
print("† KO generated by frameshift — some RNA still present (expected).")
print("  padj from DESeq2 Wald test. ns = p > 0.05.\n")

# Save CSV
csv_path = os.path.join(save_dir, 'Chd8_expression_summary_table.csv')
table.to_csv(csv_path, index=False)
print(f"✅ Table saved: {csv_path}")

# ══════════════════════════════════════════════════════════════
# 3. QUICK CHECK PLOT
# ══════════════════════════════════════════════════════════════
# Single panel — all 5 conditions on one x-axis, clean and fast to read

CONDITIONS  = ['CTRL\n(WT)', 'KO', 'FL', '∆C', '∆H']
COND_KEYS   = [None, 'KO', 'FL', 'dC', 'dH']
COLORS_QUICK = {
    'CTRL\n(WT)': '#AAAAAA',
    'KO'        : '#D62728',
    'FL'        : '#2CA02C',
    '∆C'        : '#1F77B4',
    '∆H'        : '#FF7F0E',
}

fig_q, ax_q = plt.subplots(figsize=(9/2.54, 7/2.54))   # ~90mm wide

all_vals_plot = {
    'CTRL\n(WT)': replicates['KO']['ctrl'],
    'KO'        : replicates['KO']['exp'],
    'FL'        : replicates['FL']['exp'],
    '∆C'        : replicates['dC']['exp'],
    '∆H'        : replicates['dH']['exp'],
}

np.random.seed(1)
for xi, cond in enumerate(CONDITIONS):
    vals  = all_vals_plot[cond]
    mean  = np.mean(vals)
    sem   = stats.sem(vals)
    color = COLORS_QUICK[cond]

    ax_q.bar(xi, mean, width=0.6, color=color, edgecolor='black',
             linewidth=0.4, alpha=0.85, zorder=2)
    ax_q.errorbar(xi, mean, yerr=sem, fmt='none', color='black',
                  linewidth=0.6, capsize=2, capthick=0.6, zorder=3)
    jitter = np.random.uniform(-0.12, 0.12, len(vals))
    ax_q.scatter(np.full(len(vals), xi) + jitter, vals,
                 s=10, color='white', edgecolors='black',
                 linewidths=0.4, zorder=4)

    # padj annotation above bar (skip CTRL)
    if cond not in ['CTRL\n(WT)']:
        key = {'KO':'KO','FL':'FL','∆C':'dC','∆H':'dH'}[cond]
        p   = deseq[key]['padj']
        lfc = deseq[key]['log2FC']
        y_ann = mean + sem + 0.4
        ax_q.text(xi, y_ann, sig_label(p),
                  ha='center', va='bottom', fontsize=7,
                  fontstyle='italic' if sig_label(p)=='ns' else 'normal',
                  color='#666' if sig_label(p)=='ns' else 'black')
        ax_q.text(xi, -1.8, f'{lfc:+.2f}',
                  ha='center', va='top', fontsize=5,
                  color=color, style='italic')

ax_q.text(2, -1.8, 'log₂FC vs CTRL:',
          ha='center', va='top', fontsize=5, color='#555')

ax_q.set_xticks(range(len(CONDITIONS)))
ax_q.set_xticklabels(CONDITIONS, fontsize=6)
ax_q.set_ylabel('Normalised expression (log₂ counts)', fontsize=6)
ax_q.set_ylim(-2.5, max(v.max() for v in all_vals_plot.values()) + 2.5)
ax_q.spines['top'].set_visible(False)
ax_q.spines['right'].set_visible(False)
ax_q.tick_params(labelsize=6, width=0.5, length=2.5)
ax_q.axhline(np.mean(replicates['KO']['ctrl']), color='#AAAAAA',
             linewidth=0.5, linestyle='--', zorder=1)

fig_q.text(0.5, -0.04,
           '† KO = frameshift; residual RNA expected. Stars = DESeq2 padj.',
           ha='center', fontsize=5, color='#777', style='italic')

plt.tight_layout(pad=0.5)

quick_pdf  = os.path.join(save_dir, 'Chd8_expression_quick_check.pdf')
quick_png  = os.path.join(save_dir, 'Chd8_expression_quick_check.png')
fig_q.savefig(quick_pdf, dpi=300, bbox_inches='tight', pad_inches=0.02)
fig_q.savefig(quick_png, dpi=300, bbox_inches='tight', pad_inches=0.02)
print(f"✅ Quick check plot saved: {quick_pdf}")
plt.close(fig_q)

# ══════════════════════════════════════════════════════════════
# 4. PUBLICATION FIGURE
# ══════════════════════════════════════════════════════════════
matplotlib.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial','Helvetica','DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size' : 2,
    'ytick.major.size' : 2,
    'pdf.fonttype'   : 42,     # editable text in Illustrator/Inkscape
    'ps.fonttype'    : 42,
    'svg.fonttype'   : 'none',
    'figure.dpi'     : 300,
    'savefig.dpi'    : 300,
})

FIG_W = 85  / 25.4
FIG_H = 80  / 25.4

EXP_COLORS = {
    'KO': '#D62728',
    'FL': '#2CA02C',
    'dC': '#1F77B4',
    'dH': '#FF7F0E',
}
CTRL_COLOR = '#AAAAAA'

fig_pub, ax_pub = plt.subplots(figsize=(FIG_W, FIG_H))

# ── Plot all 5 groups on single axis ──────────────────────────
GROUP_ORDER  = ['CTRL', 'KO', 'FL', 'dC', 'dH']
GROUP_KEYS   = [None,   'KO', 'FL', 'dC', 'dH']
GROUP_LABELS = ['CTRL\n(WT)', 'KO†', 'FL', '∆C', '∆H']
GROUP_COLORS = [CTRL_COLOR] + [EXP_COLORS[k] for k in GROUP_KEYS[1:]]

group_vals = {
    'CTRL': replicates['KO']['ctrl'],
    'KO'  : replicates['KO']['exp'],
    'FL'  : replicates['FL']['exp'],
    'dC'  : replicates['dC']['exp'],
    'dH'  : replicates['dH']['exp'],
}

np.random.seed(42)
y_maxes = []

for xi, (grp, key, color) in enumerate(zip(GROUP_ORDER, GROUP_KEYS, GROUP_COLORS)):
    vals = group_vals[grp]
    mean = np.mean(vals)
    sem  = stats.sem(vals)
    y_maxes.append(np.max(vals))

    # Bar
    ax_pub.bar(xi, mean, width=0.65,
               color=color, edgecolor='black',
               linewidth=0.4, alpha=0.9, zorder=2)

    # SEM
    ax_pub.errorbar(xi, mean, yerr=sem, fmt='none',
                    color='black', linewidth=0.5,
                    capsize=2, capthick=0.5, zorder=3)

    # Replicate dots
    jitter = np.random.uniform(-0.1, 0.1, len(vals))
    ax_pub.scatter(np.full(len(vals), xi) + jitter, vals,
                   s=6, color='white', edgecolors='black',
                   linewidths=0.35, zorder=4)

# ── Significance brackets vs CTRL ────────────────────────────
bracket_y = max(y_maxes) + 0.8

for xi, key in enumerate(GROUP_KEYS[1:], start=1):
    p     = deseq[key]['padj']
    sig   = sig_label(p)
    color = '#888888' if sig == 'ns' else 'black'

    ax_pub.plot([0, 0, xi, xi],
                [bracket_y - 0.2, bracket_y, bracket_y, bracket_y - 0.2],
                color='black', linewidth=0.4)
    ax_pub.text(xi/2, bracket_y + 0.1, sig,
                ha='center', va='bottom', fontsize=6,
                fontstyle='italic' if sig == 'ns' else 'normal',
                color=color)

# ── log2FC annotations below x-axis ──────────────────────────
for xi, key in enumerate(GROUP_KEYS[1:], start=1):
    lfc = deseq[key]['log2FC']
    ax_pub.text(xi, -0.18, f'{lfc:+.2f}',
                ha='center', va='top', fontsize=5,
                color=EXP_COLORS[key], style='italic',
                transform=ax_pub.get_xaxis_transform())

ax_pub.text(0, -0.18, 'log₂FC:',
            ha='center', va='top', fontsize=5,
            color='#555',
            transform=ax_pub.get_xaxis_transform())

# ── Reference line at CTRL mean ───────────────────────────────
ax_pub.axhline(np.mean(group_vals['CTRL']),
               color='#AAAAAA', linewidth=0.4,
               linestyle='--', zorder=1)

# ── Axes ─────────────────────────────────────────────────────
ax_pub.set_xticks(range(5))
ax_pub.set_xticklabels(GROUP_LABELS, fontsize=6)
ax_pub.set_ylabel('Normalised expression (log₂ counts)', fontsize=6)
ax_pub.set_ylim(-1, bracket_y + 1.8)
ax_pub.spines['top'].set_visible(False)
ax_pub.spines['right'].set_visible(False)
ax_pub.tick_params(labelsize=6, width=0.5, length=2)

# ── Panel label A ────────────────
trans = mtransforms.ScaledTranslation(-18/72, 4/72, fig_pub.dpi_scale_trans)
ax_pub.text(0, 1, 'A', transform=ax_pub.transAxes + trans,
            fontsize=8, fontweight='bold', va='top')

plt.tight_layout(pad=0.4)

# ── Save publication files ────────────────────────────────────
for fmt, kwargs in [
    ('pdf',  {}),
    ('svg',  {}),
    ('tiff', {'pil_kwargs': {'compression': 'tiff_lzw'}}),
]:
    out = os.path.join(save_dir, f'Chd8_expression_publication.{fmt}')
    fig_pub.savefig(out, format=fmt, dpi=300,
                    bbox_inches='tight', pad_inches=0.02,
                    **kwargs)
    print(f"✅ Publication figure saved: {out}")

plt.close(fig_pub)

legend_path = os.path.join(save_dir, 'Chd8_expression_figure_legend.txt')
with open(legend_path, 'w') as f:
    f.write(legend)
print(f"✅ Legend saved: {legend_path}")

# ══════════════════════════════════════════════════════════════
# 6. DOWNLOAD
# ══════════════════════════════════════════════════════════════
from google.colab import files

to_download = [
    csv_path,
    quick_pdf,
    quick_png,
    os.path.join(save_dir, 'Chd8_expression_publication.pdf'),
    os.path.join(save_dir, 'Chd8_expression_publication.tiff'),
    os.path.join(save_dir, 'Chd8_expression_publication.svg'),
    legend_path,
]

print("\n── Downloading all files to your PC ──")
for f_path in to_download:
    files.download(f_path)
    print(f"  ⬇️  {os.path.basename(f_path)}")

print("\n✅ All done. Files also saved to Google Drive at:")
print(f"   {save_dir}")

In [ ]:
import pandas as pd
import os
from google.colab import files

rows = [
    {'Condition': 'KO', 'vs'    : 'KO vs CTRL (WT)',
     'log2FC': -0.561, 'padj': '1.55e-02', 'Significance': '*',
     'Interpretation': 'Reduced ↓',
     'Note': 'Frameshift KO — residual RNA expected'},
    {'Condition': 'FL', 'vs'    : 'FL vs CTRL (WT)',
     'log2FC': +0.235, 'padj': '3.51e-01', 'Significance': 'ns',
     'Interpretation': 'Not significant',   'Note': ''},
    {'Condition': '∆C', 'vs'    : '∆C vs CTRL (WT)',
     'log2FC': +0.125, 'padj': '6.87e-01', 'Significance': 'ns',
     'Interpretation': 'Not significant',   'Note': ''},
    {'Condition': '∆H', 'vs'    : '∆H vs CTRL (WT)',
     'log2FC': +0.021, 'padj': '9.47e-01', 'Significance': 'ns',
     'Interpretation': 'Not significant',   'Note': ''},
]

table = pd.DataFrame(rows)

# Save locally in Colab first
local_csv = '/content/Chd8_expression_summary.csv'
table.to_csv(local_csv, index=False)
files.download(local_csv)
print("✅ Table downloaded")

In [ ]:
base = '/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/'
df   = pd.read_csv(base + 'Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv', sep='\t')

# Check a housekeeping gene (Actb or Gapdh) to understand scale
for gene in ['Actb', 'Gapdh', 'Chd8']:
    row = df[df['GENESYMBOL'].str.upper() == gene.upper()]
    if len(row) > 0:
        sample_cols = ['HJH5TBGX3_2','HJH5TBGX3_6','HJH5TBGX3_10',
                       'HKLWCBGX5_18','HTGK7BGX5_15','HKM5YBGX5_28',
                       'HKNFWBGX5_11']
        vals = row.iloc[0][sample_cols].values.astype(float)
        print(f"{gene}: min={vals.min():.2f}  max={vals.max():.2f}  mean={vals.mean():.2f}")

# Also check the column header of the file or any README
print("\nFirst 3 rows baseMean vs sample cols:")
print(df[['GENESYMBOL','baseMean'] + sample_cols[:3]].head(3))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from scipy import stats
from google.colab import files
import os

base     = '/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/'
save_dir = '/content/Chd8_expression_output/'
os.makedirs(save_dir, exist_ok=True)

# ── 1. LOAD ───────────────────────────────────────────────────
paths = {
    'KO': base + 'Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv',
    'FL': base + 'Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv',
    'dC': base + 'Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv',
    'dH': base + 'Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv',
}

CTRL_COLS = ['HJH5TBGX3_2','HJH5TBGX3_6','HJH5TBGX3_10',
             'HKLWCBGX5_18','HTGK7BGX5_15','HKM5YBGX5_28','HKNFWBGX5_11']

EXP_COLS = {
    'KO': ['HJH5TBGX3_8','HJH5TBGX3_12','HJH5TBGX3_4',
            'HKNFWBGX5_7','HKLWCBGX5_17'],
    'FL': ['HKNFWBGX5_3','HKNFWBGX5_10','HKLWCBGX5_21'],
    'dC': ['HKNFWBGX5_1','HKNFWBGX5_8','HKLWCBGX5_19'],
    'dH': ['HKNFWBGX5_2','HKNFWBGX5_9','HKLWCBGX5_20'],
}

replicates = {}
deseq      = {}

for label, path in paths.items():
    df   = pd.read_csv(path, sep='\t')
    chd8 = df[df['GENESYMBOL'].str.upper() == 'CHD8'].iloc[0]
    replicates[label] = {
        'ctrl': chd8[CTRL_COLS].values.astype(float),
        'exp' : chd8[EXP_COLS[label]].values.astype(float),
    }
    deseq[label] = {
        'log2FC': float(chd8['log2FoldChange']),
        'padj'  : float(chd8['padj']),
    }

ctrl_vals = replicates['KO']['ctrl']   # same CTRL in all files

# ── 2. SUMMARY TABLE ─────────────────────────────────────────
def sig_stars(p):
    if   p > 0.05 : return 'ns'
    elif p > 0.01 : return '*'
    elif p > 0.001: return '**'
    else          : return '***'

table_rows = []
for label, display in [('KO','KO'),('FL','FL'),('dC','∆C'),('dH','∆H')]:
    ev = replicates[label]['exp']
    table_rows.append({
        'Condition'            : display,
        'N_CTRL'               : len(ctrl_vals),
        'N_Exp'                : len(ev),
        'CTRL mean ± SEM'      : f"{np.mean(ctrl_vals):.2f} ± {stats.sem(ctrl_vals):.2f}",
        'Exp mean ± SEM'       : f"{np.mean(ev):.2f} ± {stats.sem(ev):.2f}",
        'log2FoldChange'       : round(deseq[label]['log2FC'], 3),
        'padj'                 : f"{deseq[label]['padj']:.2e}",
        'Significance'         : sig_stars(deseq[label]['padj']),
        'Note'                 : '† frameshift; residual RNA expected' if label=='KO' else '',
    })

table = pd.DataFrame(table_rows)

# Print clean table
print("═"*80)
print("  Chd8 RNA expression — CTRL vs KO / FL / ∆C / ∆H")
print("  Values: DESeq2 normalised counts (rlog/VST transformed)")
print("═"*80)
print(f"\n{'Condition':<10} {'CTRL mean±SEM':<20} {'Exp mean±SEM':<20} "
      f"{'log2FC':<10} {'padj':<12} {'Sig'}")
print("─"*80)
for _, r in table.iterrows():
    print(f"{r['Condition']:<10} {r['CTRL mean ± SEM']:<20} "
          f"{r['Exp mean ± SEM']:<20} {r['log2FoldChange']:<10} "
          f"{r['padj']:<12} {r['Significance']}")
print("─"*80)
print("padj: DESeq2 Wald test, Benjamini–Hochberg correction")
print("† KO by frameshift — some Chd8 RNA detectable (expected, no protein)")
print("  FL/∆C/∆H: not significantly different from CTRL (expression restored)\n")

# Save & download CSV immediately to /content
csv_path = '/content/Chd8_expression_summary.csv'
table.to_csv(csv_path, index=False)
files.download(csv_path)
print(f"✅ Table downloaded: Chd8_expression_summary.csv")

# ── 3. FIGURE ─────────────────────────────────────────────────
# All 5 groups on one axis —

matplotlib.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial','Helvetica','DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'svg.fonttype'   : 'none',
})

GROUP_DATA = {
    'CTRL\n(WT)': (ctrl_vals,              '#AAAAAA'),
    'KO†'       : (replicates['KO']['exp'],'#D62728'),
    'FL'        : (replicates['FL']['exp'],'#2CA02C'),
    '∆C'        : (replicates['dC']['exp'],'#1F77B4'),
    '∆H'        : (replicates['dH']['exp'],'#FF7F0E'),
}
DESEQ_KEYS = [None, 'KO', 'FL', 'dC', 'dH']

# ── -: 85mm single column ────────────────────────
fig, ax = plt.subplots(figsize=(85/25.4, 80/25.4))

np.random.seed(42)
y_tops = []

for xi, ((grp, (vals, color)), dkey) in enumerate(
        zip(GROUP_DATA.items(), DESEQ_KEYS)):

    mean = np.mean(vals)
    sem  = stats.sem(vals)
    y_tops.append(np.max(vals))

    # Bar
    ax.bar(xi, mean, width=0.65,
           color=color, edgecolor='black',
           linewidth=0.4, alpha=0.88, zorder=2)

    # SEM error bar
    ax.errorbar(xi, mean, yerr=sem, fmt='none',
                color='black', linewidth=0.5,
                capsize=2, capthick=0.5, zorder=3)

    # Individual replicate dots
    jitter = np.random.uniform(-0.11, 0.11, len(vals))
    ax.scatter(np.full(len(vals), xi) + jitter, vals,
               s=7, color='white', edgecolors='black',
               linewidths=0.35, zorder=4)

    # Significance stars above bar (vs CTRL)
    if dkey is not None:
        p   = deseq[dkey]['padj']
        sig = sig_stars(p)
        y_s = np.max(vals) + 0.3
        ax.text(xi, y_s, sig,
                ha='center', va='bottom', fontsize=6,
                fontstyle='italic' if sig == 'ns' else 'normal',
                color='#777' if sig == 'ns' else 'black')

    # log2FC below x-axis label
    if dkey is not None:
        lfc = deseq[dkey]['log2FC']
        ax.text(xi, -0.16, f'{lfc:+.2f}',
                ha='center', va='top', fontsize=4.5,
                color=color, style='italic',
                transform=ax.get_xaxis_transform())

# log2FC row label
ax.text(-0.5, -0.16, 'log₂FC:',
        ha='left', va='top', fontsize=4.5,
        color='#555',
        transform=ax.get_xaxis_transform())

# CTRL reference line
ax.axhline(np.mean(ctrl_vals),
           color='#AAAAAA', linewidth=0.4,
           linestyle='--', zorder=1, label='CTRL mean')

# Axes
ax.set_xticks(range(5))
ax.set_xticklabels(list(GROUP_DATA.keys()), fontsize=6)
ax.set_ylabel('Chd8 normalised expression\n(DESeq2 counts)', fontsize=6)
ax.set_ylim(bottom=3, top=max(y_tops) + 2.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=6, width=0.5, length=2)

# Panel label A
trans = mtransforms.ScaledTranslation(-16/72, 3/72, fig.dpi_scale_trans)
ax.text(0, 1, 'A', transform=ax.transAxes + trans,
        fontsize=8, fontweight='bold', va='top')

plt.tight_layout(pad=0.5)

# ── Save ──────────────────────────────────────────────────────
for fmt, kw in [
    ('pdf',  {}),
    ('png',  {}),
    ('tiff', {'pil_kwargs': {'compression':'tiff_lzw'}}),
    ('svg',  {}),
]:
    out = f'/content/Chd8_expression.{fmt}'
    fig.savefig(out, format=fmt, dpi=300,
                bbox_inches='tight', pad_inches=0.02, **kw)

plt.close(fig)

# ── Download all ──────────────────────────────────────────────
for fname in ['Chd8_expression.pdf',
              'Chd8_expression.png',
              'Chd8_expression.tiff',
              'Chd8_expression.svg']:
    files.download(f'/content/{fname}')
    print(f"⬇️  {fname}")

legend_path = '/content/Chd8_expression_legend.txt'
with open(legend_path, 'w') as f:
    f.write(legend)
files.download(legend_path)
print("⬇️  Chd8_expression_legend.txt")
print("\n✅ All done.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import os

base     = '/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/'
save_dir = '/content/drive/MyDrive/Chd8 data/figures/'
os.makedirs(save_dir, exist_ok=True)

# ── 1. LOAD FILES ─────────────────────────────────────────────
paths = {
    'KO': base + 'Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv',
    'FL': base + 'Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv',
    'dC': base + 'Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv',
    'dH': base + 'Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv',
}

# Shared CTRL columns present in every file
CTRL_COLS = ['HJH5TBGX3_2','HJH5TBGX3_6','HJH5TBGX3_10',
             'HKLWCBGX5_18','HTGK7BGX5_15','HKM5YBGX5_28','HKNFWBGX5_11']

# Unique replicate columns per condition
EXP_COLS = {
    'KO': ['HJH5TBGX3_8','HJH5TBGX3_12','HJH5TBGX3_4',
            'HKNFWBGX5_7','HKLWCBGX5_17'],           # 5 KO replicates
    'FL': ['HKNFWBGX5_3','HKNFWBGX5_10','HKLWCBGX5_21'],
    'dC': ['HKNFWBGX5_1','HKNFWBGX5_8','HKLWCBGX5_19'],
    'dH': ['HKNFWBGX5_2','HKNFWBGX5_9','HKLWCBGX5_20'],
}

# ── 2. EXTRACT CHD8 REPLICATES ────────────────────────────────
replicate_data = {}   # {condition: {ctrl: array, exp: array}}
deseq_stats    = {}   # {condition: {log2FC, padj}}

for label, path in paths.items():
    df   = pd.read_csv(path, sep='\t')
    chd8 = df[df['GENESYMBOL'].str.upper() == 'CHD8'].iloc[0]

    ctrl_vals = chd8[CTRL_COLS].values.astype(float)
    exp_vals  = chd8[EXP_COLS[label]].values.astype(float)

    replicate_data[label] = {'ctrl': ctrl_vals, 'exp': exp_vals}
    deseq_stats[label]    = {
        'log2FC': chd8['log2FoldChange'],
        'padj'  : chd8['padj'],
    }

    print(f"{label}  |  CTRL: {np.round(ctrl_vals,2)}  "
          f"EXP: {np.round(exp_vals,2)}  "
          f"log2FC={chd8['log2FoldChange']:.3f}  padj={chd8['padj']:.2e}")

# ── 3. SIGNIFICANCE STARS ─────────────────────────────────────
def stars(p):
    if   p > 0.05 : return 'ns'
    elif p > 0.01 : return '*'
    elif p > 0.001: return '**'
    else          : return '***'

# ── 4. FIGURE ─────────────────────────────────────────────────
matplotlib.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial','Helvetica','DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
})

# Colors: CTRL always grey, each condition has its own color
EXP_COLORS = {
    'KO': '#D62728',   # red
    'FL': '#2CA02C',   # green  (full rescue)
    'dC': '#1F77B4',   # blue   (chromodomain deleted)
    'dH': '#FF7F0E',   # orange (helicase deleted)
}
CTRL_COLOR = '#AAAAAA'

# Labels for x-axis
X_LABELS = {
    'KO': ('CTRL\n(WT)', 'KO'),
    'FL': ('CTRL\n(WT)', 'FL'),
    'dC': ('CTRL\n(WT)', '∆C'),
    'dH': ('CTRL\n(WT)', '∆H'),
}
TITLES = {
    'KO': 'KO vs WT',
    'FL': 'FL Rescue',
    'dC': '∆C Rescue',
    'dH': '∆H Rescue',
}

fig, axes = plt.subplots(1, 4,
                         figsize=(85/25.4, 75/25.4),
                         gridspec_kw={'wspace': 0.65})

for ax, label in zip(axes, ['KO','FL','dC','dH']):

    ctrl_vals = replicate_data[label]['ctrl']
    exp_vals  = replicate_data[label]['exp']
    lfc       = deseq_stats[label]['log2FC']
    padj      = deseq_stats[label]['padj']
    exp_color = EXP_COLORS[label]
    np.random.seed(0)

    for xi, (vals, color) in enumerate([(ctrl_vals, CTRL_COLOR),
                                         (exp_vals,  exp_color)]):
        mean = np.mean(vals)
        sem  = stats.sem(vals)

        # Bar
        ax.bar(xi, mean, width=0.6,
               color=color, edgecolor='black',
               linewidth=0.4, alpha=0.85, zorder=2)

        # SEM error bar
        ax.errorbar(xi, mean, yerr=sem, fmt='none',
                    color='black', linewidth=0.6,
                    capsize=2.5, capthick=0.6, zorder=3)

        # Individual replicate dots (jittered)
        jitter = np.random.uniform(-0.1, 0.1, len(vals))
        ax.scatter(np.full(len(vals), xi) + jitter, vals,
                   s=8, color='white', edgecolors='black',
                   linewidths=0.4, zorder=4)

    # Significance bracket
    all_vals = np.concatenate([ctrl_vals, exp_vals])
    y_top    = np.max(all_vals)
    y_line   = y_top + 0.6
    y_star   = y_line + 0.25
    sig      = stars(padj)

    ax.plot([0, 0, 1, 1],
            [y_line-0.2, y_line, y_line, y_line-0.2],
            color='black', linewidth=0.5)
    ax.text(0.5, y_star, sig,
            ha='center', va='bottom',
            fontsize=7 if sig == 'ns' else 8,
            fontstyle='italic' if sig == 'ns' else 'normal',
            color='#666666' if sig == 'ns' else 'black')

    # KO frameshift note
    if label == 'KO':
        ax.text(0.5, -0.05, '†',
                ha='center', va='top',
                fontsize=7, color='#888888',
                transform=ax.get_xaxis_transform())

    # Axis formatting
    xl = X_LABELS[label]
    ax.set_xticks([0, 1])
    ax.set_xticklabels(xl, fontsize=5.5)
    ax.set_title(TITLES[label], fontsize=6, fontweight='bold', pad=3)

    if label == 'KO':
        ax.set_ylabel('Normalised expression\n(log₂ counts)', fontsize=6)
    else:
        ax.set_ylabel('')
        ax.tick_params(labelleft=False)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=5.5, width=0.5, length=2.5)

    # log2FC label below x-axis
    ax.text(1.0, -0.18,
            f'log₂FC={lfc:+.2f}',
            ha='center', va='top', fontsize=4.5,
            color=exp_color, style='italic',
            transform=ax.get_xaxis_transform())

# Shared super-title
fig.suptitle('Chd8 expression — loss-of-function and rescue conditions',
             fontsize=6.5, fontweight='bold', y=1.03)

# Footnote
fig.text(0.5, -0.10,
         'Bars = mean ± SEM. Dots = individual replicates. '
         'Stars: *p<0.05, **p<0.01, ***p<0.001, ns = not significant (DESeq2 padj).\n'
         '† KO by frameshift mutation — residual RNA signal expected.',
         ha='center', fontsize=4.5, color='#666666', style='italic')

plt.tight_layout(pad=0.4)

# ── 5. SAVE ───────────────────────────────────────────────────
for fmt in ['pdf', 'svg', 'tiff']:
    out = os.path.join(save_dir, f'Chd8_expression_CTRL_KO_FL_dC_dH.{fmt}')
    kwargs = dict(bbox_inches='tight', pad_inches=0.02)
    if fmt == 'tiff':
        kwargs['pil_kwargs'] = {'compression': 'tiff_lzw'}
    fig.savefig(out, format=fmt, dpi=300, **kwargs)
    print(f'✅ Saved: {out}')

plt.show()

# Optional: download directly to PC
from google.colab import files
files.download(os.path.join(save_dir, 'Chd8_expression_CTRL_KO_FL_dC_dH.pdf'))
files.download(os.path.join(save_dir, 'Chd8_expression_CTRL_KO_FL_dC_dH.tiff'))

In [ ]:
import os
import pybedtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import subprocess

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── File paths (same as original) ────────────────────────────────────────────
ES_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
NPC_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
SAVE_DIR = "/content/drive/MyDrive/CHD8_Fig1/"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Output: Colab temp dir, then downloaded  ───────────────────
output_dir = "/content/Figure1B_output"
os.makedirs(output_dir, exist_ok=True)

FIG_WIDTH_IN  = 85 / 25.4   # 3.346 inches
FIG_HEIGHT_IN = 85 / 25.4   # square
DPI           = 300

# ── Global rcParams ────────────────────────────────────
plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 7,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

# ── Standard chromosomes (mouse mm10) ────────────────────────────────────────
STANDARD_CHROMS = set(
    [f'chr{i}' for i in range(1, 20)] + ['chrX', 'chrY']
)
def is_standard(feature):
    return str(feature.chrom) in STANDARD_CHROMS

def shell_count(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return int(r.stdout.strip().split()[0])

if not (os.path.exists(ES_PATH) and os.path.exists(NPC_PATH)):
    raise FileNotFoundError("BED files not found — check paths above.")

# ── Filter to standard chromosomes ───────────────────────────────────────────
es_filt  = pybedtools.BedTool(ES_PATH).filter(is_standard).saveas()
npc_filt = pybedtools.BedTool(NPC_PATH).filter(is_standard).saveas()

es_total  = es_filt.count()
npc_total = npc_filt.count()

ES_TMP  = "/content/es_std_sorted.bed"
NPC_TMP = "/content/npc_std_sorted.bed"
es_filt.sort().saveas(ES_TMP)
npc_filt.sort().saveas(NPC_TMP)

# ── Counts via shell ──────────────────────────────────────────────────────────
es_only    = shell_count(f"bedtools intersect -a {ES_TMP}  -b {NPC_TMP} -v | wc -l")
npc_only   = shell_count(f"bedtools intersect -a {NPC_TMP} -b {ES_TMP}  -v | wc -l")
es_shared  = shell_count(f"bedtools intersect -a {ES_TMP}  -b {NPC_TMP} -u | wc -l")
npc_shared = shell_count(f"bedtools intersect -a {NPC_TMP} -b {ES_TMP}  -u | wc -l")

# ── Arithmetic verification ───────────────────────────────────────────────────
es_ok  = (es_only  + es_shared  == es_total)
npc_ok = (npc_only + npc_shared == npc_total)
if not (es_ok and npc_ok):
    raise ValueError("Arithmetic broken — check BED files for malformed records.")

print(f"ES  total: {es_total:,}  |  NPC total: {npc_total:,}")
print(f"ES-only: {es_only:,}  |  NPC-only: {npc_only:,}  |  Shared (NPC persp.): {npc_shared:,}")

# ── Percentages ───────────────────────────────────────────────────────────────
pct_es_only    = es_only   / es_total  * 100
pct_npc_only   = npc_only  / npc_total * 100
pct_npc_sh_es  = npc_shared / es_total  * 100
pct_npc_sh_npc = npc_shared / npc_total * 100

# ── Figure (no title, no footnote — figure only) ─────────────────────────────
fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))

v = venn2(
    subsets    = (es_only, npc_only, npc_shared),
    set_labels = ('', ''),
    set_colors = ('#3498DB', '#E67E22'),
    alpha      = 0.70,
    ax         = ax
)

# ── Venn region labels ────────────────────────────────────────────────────────
if v.get_label_by_id('10'):
    v.get_label_by_id('10').set_text(f"{es_only:,}\n({pct_es_only:.1f}%)")
    v.get_label_by_id('10').set_fontsize(6)

if v.get_label_by_id('01'):
    v.get_label_by_id('01').set_text(f"{npc_only:,}\n({pct_npc_only:.1f}%)")
    v.get_label_by_id('01').set_fontsize(6)

if v.get_label_by_id('11'):
    v.get_label_by_id('11').set_text(
        f"{npc_shared:,}\n({pct_npc_sh_es:.1f}% of ES\n{pct_npc_sh_npc:.1f}% of NPC)")
    v.get_label_by_id('11').set_fontsize(5)

# ── Set labels (circle annotations, not title) ───────────────────────────────
# Positions tuned so labels sit above their respective circles
# x in axes coords: 0=left edge, 1=right edge; Venn circles sit ~0.2–0.8
ax.text(0.22, 0.85,
        f"ES CHD8\n(n={es_total:,})",
        ha='center', fontsize=6, fontweight='bold', color='#1A5276',
        transform=ax.transAxes)
ax.text(0.88, 0.85,
        f"NPC CHD8\n(n={npc_total:,})",
        ha='center', fontsize=6, fontweight='bold', color='#784212',
        transform=ax.transAxes)

plt.tight_layout(pad=0.2)

# ── Save PDF (vector, fonts embedded) ────────────────────────────────────────
pdf_path  = os.path.join(output_dir, "Figure1B.pdf")
tiff_path = os.path.join(output_dir, "Figure1B.tiff")

fig.savefig(
    pdf_path,
    format      = 'pdf',
    bbox_inches = 'tight',
    pad_inches  = 0.02,
)
print(f"✅ PDF saved:  {pdf_path}")

# ── Save TIFF (LZW lossless, 300 dpi) ────────────────────────────────────────
fig.savefig(
    tiff_path,
    format      = 'tiff',
    dpi         = DPI,
    bbox_inches = 'tight',
    pad_inches  = 0.02,
    pil_kwargs  = {"compression": "tiff_lzw"},
)
print(f"✅ TIFF saved: {tiff_path}")

plt.close(fig)

# ── Download both files directly to your PC ───────────────────────────────────
from google.colab import files
files.download(pdf_path)
files.download(tiff_path)
print("\n✅ Downloads triggered — check your browser's Downloads folder.")

In [ ]:
import os
import pybedtools
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

from google.colab import drive
drive.mount('/content/drive')

PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
H3K4_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
GTF_PATH  = "/content/drive/MyDrive/genome/mm10/mm10.gtf"
RNA_PATH  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

STANDARD_CHROMS = set([f'chr{i}' for i in range(1, 20)] + ['chrX', 'chrY'])

# ── STEP 0: Diagnose chromosome naming ───────────────────────────────────────
print("── STEP 0: Checking chromosome naming ──────────────────────")
peak_chroms = pd.read_csv(PEAK_PATH, sep='\t', header=None, usecols=[0])[0].unique()[:5]
tss_chroms  = pd.read_csv(TSS_PATH,  sep='\t', header=None, usecols=[0])[0].unique()[:5]
print(f"   Peak file chroms (first 5): {list(peak_chroms)}")
print(f"   TSS file chroms  (first 5): {list(tss_chroms)}")

tss_sample = pd.read_csv(TSS_PATH, sep='\t', header=None, nrows=3)
print(f"   TSS columns ({len(tss_sample.columns)} total):")
print(tss_sample.to_string())

tss_needs_chr  = not str(tss_chroms[0]).startswith('chr')
peak_needs_chr = not str(peak_chroms[0]).startswith('chr')
print(f"\n   TSS needs 'chr' prefix:   {tss_needs_chr}")
print(f"   Peaks need 'chr' prefix:  {peak_needs_chr}")

# ── STEP 1: Peak-TSS intersections ───────────────────────────────────────────
print("\n── STEP 1: Intersecting peaks with TSS ─────────────────────")

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

def load_filtered(path, add_chr=False):
    bt = pybedtools.BedTool(path)
    if add_chr:
        bt = bt.each(fix_chr)
    return bt.filter(lambda f: str(f.chrom) in STANDARD_CHROMS).saveas()

npc  = load_filtered(PEAK_PATH, add_chr=peak_needs_chr)
h3k4 = load_filtered(H3K4_PATH, add_chr=peak_needs_chr)
tss  = load_filtered(TSS_PATH,  add_chr=tss_needs_chr)

chd8_at_tss = npc.intersect(tss,  wb=True)
h3k4_at_tss = h3k4.intersect(tss, wb=True)
print(f"   CHD8 peaks at TSS  : {len(chd8_at_tss):,}")
print(f"   H3K4me3 peaks at TSS: {len(h3k4_at_tss):,}")

# ── STEP 2: Extract gene names ────────────────────────────────────────────────
print("\n── STEP 2: Extracting gene names ───────────────────────────")

def get_genes_wb(bt, name_col=None):
    genes = set()
    for i, f in enumerate(bt):
        if i == 0 and name_col is None:
            for idx in range(3, len(f.fields)):
                val = f.fields[idx]
                if not val.lstrip('-').replace('.','',1).isdigit() and not val.startswith('chr'):
                    name_col = idx
                    print(f"   Auto-detected gene name at column {idx} (e.g. '{val}')")
                    break
            if name_col is None:
                name_col = 6
                print(f"   Defaulting to column 6")
        if len(f.fields) > name_col:
            genes.add(f.fields[name_col])
    return genes

chd8_tss_genes  = get_genes_wb(chd8_at_tss)
h3k4_tss_genes  = get_genes_wb(h3k4_at_tss)

chd8_h3k4_genes = chd8_tss_genes & h3k4_tss_genes
h3k4_only_genes = h3k4_tss_genes - chd8_tss_genes

print(f"   CHD8+H3K4me3 genes : {len(chd8_h3k4_genes):,}")
print(f"   H3K4me3-only genes : {len(h3k4_only_genes):,}")
print(f"   CHD8-only genes    : {len(chd8_tss_genes - h3k4_tss_genes):,}")

# ── STEP 3: Gene lengths from GTF ─────────────────────────────────────────────
print("\n── STEP 3: Extracting gene lengths from GTF ────────────────")
gene_lengths = {}
with open(GTF_PATH) as f:
    for line in f:
        if line.startswith('#'): continue
        parts = line.strip().split('\t')
        if len(parts) < 9: continue
        if parts[2] not in ('transcript', 'mRNA'): continue
        length = int(parts[4]) - int(parts[3]) + 1
        name = None
        for field in parts[8].split(';'):
            field = field.strip()
            if 'gene_name' in field:
                try:    name = field.split('"')[1]
                except: name = field.split()[-1]
                break
        if name:
            if name not in gene_lengths or length > gene_lengths[name]:
                gene_lengths[name] = length
print(f"   Gene lengths extracted: {len(gene_lengths):,}")

# ── STEP 4: Load RNA-seq and compute TPM ──────────────────────────────────────
print("\n── STEP 4: Loading RNA-seq and computing TPM ───────────────")
rna_all            = pd.read_csv(RNA_PATH, sep='\t').dropna(subset=['GENESYMBOL'])
gene_len_df        = pd.DataFrame.from_dict(gene_lengths, orient='index', columns=['gene_length'])
rna_tpm            = rna_all.merge(gene_len_df, left_on='GENESYMBOL', right_index=True, how='left')
rna_tpm            = rna_tpm.dropna(subset=['gene_length'])
rna_tpm['RPK']     = rna_tpm['baseMean'] / (rna_tpm['gene_length'] / 1000)
scaling            = rna_tpm['RPK'].sum() / 1e6
rna_tpm['TPM']     = rna_tpm['RPK'] / scaling
rna_tpm['log2TPM'] = np.log2(rna_tpm['TPM'] + 1)

# ── STEP 5: Assign groups ─────────────────────────────────────────────────────
print("\n── STEP 5: Assigning groups ────────────────────────────────")
rna_tpm['group'] = 'Unbound & Unmarked'
rna_tpm.loc[rna_tpm['GENESYMBOL'].isin(h3k4_only_genes),  'group'] = 'H3K4me3-only'
rna_tpm.loc[rna_tpm['GENESYMBOL'].isin(chd8_h3k4_genes),  'group'] = 'CHD8+H3K4me3'

print(rna_tpm['group'].value_counts().to_string())
print("\n   Median log2TPM by group:")
print(rna_tpm.groupby('group')['log2TPM'].median().round(3).to_string())
print("\n✅ Setup complete — now run Figure1C_GenomeBiology.py")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import pybedtools
import os

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────────
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_PATH  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

output_dir = "/content/Figure2B_output"
os.makedirs(output_dir, exist_ok=True)

FIG_WIDTH_IN  = 170 / 25.4   # 6.69 inches
FIG_HEIGHT_IN = 80  / 25.4   # 3.15 inches
DPI           = 300

# ── Global rcParams ────────────────────────────────────
plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

# ── Chr prefix fixer ──────────────────────────────────────────────────────────
def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ── Step 1: NPC-specific ChIP targets ────────────────────────────────────────
print("=" * 60)
print("FIGURE 2B: CHD8 TARGET TRANSCRIPTIONAL ANALYSIS")
print("=" * 60)
print("\n1. Identifying NPC-specific ChIP targets...")

npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()

npc_specific = npc_bt.subtract(esc_bt, A=True)
hits         = tss_bt.intersect(npc_specific, u=True, wa=True)
target_list  = list(set([
    str(f[3]).upper().strip()
    for f in hits if len(str(f[3])) > 1
]))
print(f"   NPC-specific ChIP targets identified: {len(target_list):,}")

# ── Step 2: RNA-seq ───────────────────────────────────────────────────────────
print("\n2. Loading RNA-seq data...")
de_df = pd.read_csv(RNA_PATH, sep='\t').dropna(
    subset=['log2FoldChange', 'padj'])
de_df['padj'] = de_df['padj'].replace(0, 1e-300)

if 'baseMean' in de_df.columns:
    n_before = len(de_df)
    de_df    = de_df[de_df['baseMean'] > 10]
    print(f"   Filtered: {n_before:,} -> {len(de_df):,} genes (baseMean > 10)")

de_df['gene_upper'] = (de_df['GENESYMBOL'].astype(str)
                       .str.upper().str.strip())
de_df['is_target']  = de_df['gene_upper'].isin(set(target_list))

all_targets = de_df[de_df['is_target']]
sig_targets = all_targets[all_targets['padj'] < 0.05]
others      = de_df[~de_df['is_target']]
targets_lfc = all_targets['log2FoldChange']
others_lfc  = others['log2FoldChange']

# ── Step 3: MWU test ──────────────────────────────────────────────────────────
_, pval = mannwhitneyu(targets_lfc, others_lfc, alternative='two-sided')
effect  = targets_lfc.median() - others_lfc.median()

print("\n" + "-" * 60)
print("AUDIT — copy into -:")
print(f"   Targets in RNA-seq         : {len(all_targets):,}")
print(f"   FDR-significant targets    : {len(sig_targets):,}")
print(f"   MWU p-value                : {pval:.3e}")
print(f"   Δmedian LFC                : {effect:.4f}")

if pval < 0.05:
    mwu_label  = f"MWU p = {pval:.2e}"
    target_col = '#D35400'
else:
    mwu_label  = f"MWU p = {pval:.2e} (ns)"
    target_col = '#7F8C8D'

print("-" * 60)

# ── Figure: two panels side by side ──────────────────────────────────────────
print("\n3. Generating two-panel figure...")

fig, (ax_a, ax_b) = plt.subplots(
    1, 2,
    figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
    constrained_layout=False
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PANEL A: Target-highlighted volcano
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Layer 1: non-targets (background)
ax_a.scatter(
    others['log2FoldChange'],
    -np.log10(others['padj']),
    s         = 1.5,
    color     = '#CACFD2',
    alpha     = 0.25,
    rasterized= True,
    label     = f'Non-targets (n={len(others):,})'
)

# Layer 2: all CHD8 targets
ax_a.scatter(
    all_targets['log2FoldChange'],
    -np.log10(all_targets['padj']),
    s         = 5,
    color     = '#E67E22',
    alpha     = 0.5,
    rasterized= True,
    label     = f'CHD8 targets (n={len(all_targets):,})'
)

# Layer 3: FDR-significant targets (on top)
ax_a.scatter(
    sig_targets['log2FoldChange'],
    -np.log10(sig_targets['padj']),
    s          = 10,
    color      = '#D35400',
    alpha      = 0.95,
    edgecolors = 'white',
    linewidths = 0.3,
    rasterized = True,
    label      = f'FDR sig. targets (n={len(sig_targets):,})'
)

# Threshold lines
lim = np.ceil(
    max(abs(de_df['log2FoldChange'].quantile(0.005)),
        abs(de_df['log2FoldChange'].quantile(0.995))) * 1.1
)
ax_a.set_xlim(-lim, lim)
ax_a.set_ylim(0, None)
ax_a.axvline(0,                    color='black',   lw=0.5)
ax_a.axhline(-np.log10(0.05),      color='#7F8C8D', lw=0.5,
             ls='--', label='FDR = 0.05')

ax_a.set_title('CHD8 direct targets, KO vs WT',
               fontsize=6, fontweight='bold', loc='left', pad=4)
ax_a.set_xlabel('log$_2$ Fold Change', fontsize=6)
ax_a.set_ylabel('-log$_{10}$ Adjusted P-value', fontsize=6)
ax_a.tick_params(labelsize=5, width=0.5, length=2)
ax_a.legend(frameon=False, loc='upper right', fontsize=5,
            markerscale=2, handletextpad=0.3)
ax_a.spines[['top', 'right']].set_visible(False)
ax_a.spines[['left', 'bottom']].set_linewidth(0.5)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PANEL B: ECDF
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Non-targets: dashed grey (background/secondary)
sns.ecdfplot(
    data      = others_lfc.values,
    color     = '#95A5A6',
    lw        = 1.2,
    linestyle = '--',
    label     = f'Non-targets (n={len(others_lfc):,})',
    ax        = ax_b
)

# CHD8 targets: solid coloured line
sns.ecdfplot(
    data      = targets_lfc.values,
    color     = target_col,
    lw        = 1.8,
    linestyle = '-',
    label     = f'CHD8 targets (n={len(targets_lfc):,})',
    ax        = ax_b
)

ax_b.axvline(0, color='black', ls='--', lw=0.5, alpha=0.5)

# Stats annotation box (sized for 6pt font at 85 mm)
stat_text = (
    f"{mwu_label}\n"
    f"Δmedian = {effect:.3f}"
)
ax_b.text(
    0.04, 0.96, stat_text,
    transform  = ax_b.transAxes,
    va         = 'top',
    fontsize   = 5,
    linespacing= 1.4,
    bbox       = dict(boxstyle='round,pad=0.3',
                      facecolor='white',
                      edgecolor='#CACFD2',
                      linewidth=0.5,
                      alpha=0.9)
)

ax_b.set_title('LFC distribution: targets vs non-targets',
               fontsize=6, fontweight='bold', loc='left', pad=4)
ax_b.set_xlabel('log$_2$ Fold Change', fontsize=6)
ax_b.set_ylabel('Cumulative proportion', fontsize=6)
ax_b.tick_params(labelsize=5, width=0.5, length=2)
ax_b.legend(frameon=False, loc='lower right', fontsize=5,
            handletextpad=0.3)
ax_b.spines[['top', 'right']].set_visible(False)
ax_b.spines[['left', 'bottom']].set_linewidth(0.5)

# ── Layout ────────────────────────────────────────────────────────────────────
plt.tight_layout(pad=0.5, w_pad=1.2, h_pad=0)

# ── Save PDF (vector, fonts embedded) ────────────────────────────────────────
pdf_path  = os.path.join(output_dir, "Figure2B.pdf")
tiff_path = os.path.join(output_dir, "Figure2B.tiff")

fig.savefig(
    pdf_path,
    format    = 'pdf',
    pad_inches= 0.02,
)
print(f"✅ PDF saved:  {pdf_path}")

# ── Save TIFF (LZW lossless, 300 dpi) ────────────────────────────────────────
fig.savefig(
    tiff_path,
    format    = 'tiff',
    dpi       = DPI,
    pad_inches= 0.02,
    pil_kwargs= {"compression": "tiff_lzw"},
)
print(f"✅ TIFF saved: {tiff_path}")

plt.close(fig)

# ── Download directly to PC ───────────────────────────────────────────────────
from google.colab import files
files.download(pdf_path)
files.download(tiff_path)
print("\n✅ Downloads triggered — check your browser's Downloads folder.")

# ── Expose for downstream cells ───────────────────────────────────────────────
final_npc_genes  = target_list
npc_target_genes = target_list
print(f"\nnpc_target_genes set: {len(npc_target_genes):,} genes")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import os

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────────
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"

paths = {
    "KO"  : os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"),
    "KD81": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv"),
    "KD82": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv"),
}

# ── Output directory ──────────────────────────────────────────────────────────
output_dir = "/content/Figure3A_output"
os.makedirs(output_dir, exist_ok=True)

# 2-panel correlation → full-page width (170 mm)
FIG_WIDTH_IN  = 170 / 25.4   # 6.69 inches
FIG_HEIGHT_IN = 85  / 25.4   # 3.35 inches — square panels at 85 mm each
DPI           = 300

# ── Global rcParams  ────────────────────────────────────
plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

# ── Load & prep data ──────────────────────────────────────────────────────────
def prep_df(path, name):
    df = pd.read_csv(path, sep='\t')
    gene_col = next(
        (c for c in df.columns
         if c.upper() in ['GENESYMBOL', 'SYMBOL', 'MGI_SYMBOL', 'GENE']),
        None
    )
    if gene_col is None:
        print(f"⚠️  No gene column found in {os.path.basename(path)}. Using index.")
        df['GENESYMBOL'] = df.index
    else:
        df = df.rename(columns={gene_col: 'GENESYMBOL'})

    return (
        df[['GENESYMBOL', 'log2FoldChange']]
        .dropna()
        .rename(columns={'log2FoldChange': f'{name}_LFC'})
    )

merged = prep_df(paths["KO"],   "KO")
merged = merged.merge(prep_df(paths["KD81"], "KD81"), on='GENESYMBOL')
merged = merged.merge(prep_df(paths["KD82"], "KD82"), on='GENESYMBOL')
merged = merged.dropna()

print(f"Genes in merged dataset: {len(merged):,}")

# ── Figure ────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
    constrained_layout=False
)

def plot_correlation(ax, x_col, y_col, title, color):
    x_data = merged[x_col]
    y_data = merged[y_col]
    r, p   = pearsonr(x_data, y_data)

    # p-value label
    if p < 0.001:
        p_label = "p < 0.001"
    elif p < 0.01:
        p_label = "p < 0.01"
    else:
        p_label = f"p = {p:.3f}"

    # Shared symmetric axis limits
    all_vals = np.concatenate([x_data.values, y_data.values])
    base_lim = np.ceil(np.max(np.abs(all_vals)) * 1.1 * 10) / 10
    ax.set_xlim(-base_lim, base_lim)
    ax.set_ylim(-base_lim, base_lim)

    # Identity line (y = x)
    ax.plot(
        [-base_lim, base_lim],
        [-base_lim, base_lim],
        color     = '#AAAAAA',
        linestyle = '--',
        lw        = 0.6,
        zorder    = 1,
    )

    # Scatter (rasterized — dense points at 300 dpi)
    ax.scatter(
        x_data, y_data,
        s         = 1.2,
        color     = '#AAAAAA',
        alpha     = 0.3,
        rasterized= True,
        zorder    = 2,
    )

    # Regression line only (no CI band — too wide at 85 mm)
    sns.regplot(
        data    = merged,
        x       = x_col,
        y       = y_col,
        ax      = ax,
        scatter = False,           # scatter already drawn above
        line_kws= dict(color=color, lw=1.2, zorder=3),
        ci      = 95,
        truncate= True,
    )

    # Stats annotation
    ax.text(
        0.05, 0.95,
        f"r = {r:.2f}\n{p_label}\nn = {len(x_data):,}",
        transform  = ax.transAxes,
        va         = 'top',
        fontsize   = 5,
        linespacing= 1.5,
        bbox       = dict(
            boxstyle  = 'round,pad=0.3',
            facecolor = 'white',
            edgecolor = '#CCCCCC',
            linewidth = 0.4,
            alpha     = 0.9,
        )
    )

    # Titles and labels
    ax.set_title(title, fontsize=6, fontweight='bold', loc='left', pad=4)
    ax.set_xlabel(f'{x_col.replace("_LFC", "")} log$_2$ Fold Change', fontsize=6)
    ax.set_ylabel(f'{y_col.replace("_LFC", "")} log$_2$ Fold Change', fontsize=6)
    ax.tick_params(labelsize=5, width=0.5, length=2)
    ax.set_aspect('equal', adjustable='box')

    # Light reference grid
    ax.grid(True, linestyle=':', linewidth=0.3, alpha=0.5, color='#CCCCCC')
    ax.set_axisbelow(True)

    # Spines
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(0.5)

# ── Plot both panels ──────────────────────────────────────────────────────────
plot_correlation(ax1, 'KO_LFC',   'KD81_LFC', 'CHD8 KO vs KD clone 8.1', '#D62728')
plot_correlation(ax2, 'KD81_LFC', 'KD82_LFC', 'KD clone 8.1 vs KD clone 8.2', '#1F77B4')

# ── Layout ────────────────────────────────────────────────────────────────────
plt.tight_layout(pad=0.5, w_pad=1.2, h_pad=0)

# ── Save PDF (vector, fonts embedded) ────────────────────────────────────────
pdf_path  = os.path.join(output_dir, "Figure3A.pdf")
tiff_path = os.path.join(output_dir, "Figure3A.tiff")

fig.savefig(
    pdf_path,
    format    = 'pdf',
    pad_inches= 0.02,
)
print(f"✅ PDF saved:  {pdf_path}")

# ── Save TIFF (LZW lossless, 300 dpi) ────────────────────────────────────────
fig.savefig(
    tiff_path,
    format    = 'tiff',
    dpi       = DPI,
    pad_inches= 0.02,
    pil_kwargs= {"compression": "tiff_lzw"},
)
print(f"✅ TIFF saved: {tiff_path}")

plt.close(fig)

# ── Download directly to PC ───────────────────────────────────────────────────
from google.colab import files
files.download(pdf_path)
files.download(tiff_path)
print("\n✅ Downloads triggered — check your browser's Downloads folder.")

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────────
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
ko_path   = os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")
kd1_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv")
kd2_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")

output_dir = "/content/Figure3B_output"
os.makedirs(output_dir, exist_ok=True)

DPI = 300

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

# ── Load & flag ───────────────────────────────────────────────────────────────
def load_and_flag(path):
    df = pd.read_csv(path, sep='\t')
    gene_col = next(
        (c for c in df.columns
         if c.upper() in ['GENESYMBOL', 'SYMBOL', 'MGI_SYMBOL']),
        'GENESYMBOL'
    )
    df = df.rename(columns={gene_col: 'GENESYMBOL'})
    df['DEREG_FLAG'] = np.where(
        (df['log2FoldChange'] < 0) & (df['padj'] < 0.05),
        'DOWN', 'NONE'
    )
    return df.set_index('GENESYMBOL')

ko_df   = load_and_flag(ko_path)
kd81_df = load_and_flag(kd1_path)
kd82_df = load_and_flag(kd2_path)

# ── Shared DOWN genes ─────────────────────────────────────────────────────────
shared_genes = list(
    set(ko_df  [ko_df  ['DEREG_FLAG'] == 'DOWN'].index) &
    set(kd81_df[kd81_df['DEREG_FLAG'] == 'DOWN'].index) &
    set(kd82_df[kd82_df['DEREG_FLAG'] == 'DOWN'].index)
)
print(f"Shared downregulated genes: {len(shared_genes):,}")

# ── Heatmap matrix ────────────────────────────────────────────────────────────
heatmap_data = pd.DataFrame({
    'CHD8 KO' : ko_df  .loc[shared_genes, 'log2FoldChange'],
    'KD 8.1'  : kd81_df.loc[shared_genes, 'log2FoldChange'],
    'KD 8.2'  : kd82_df.loc[shared_genes, 'log2FoldChange'],
}).dropna()

heatmap_data['mean_abs'] = heatmap_data.abs().mean(axis=1)
heatmap_data = (heatmap_data
                .sort_values('mean_abs', ascending=False)
                .head(30)
                .drop(columns=['mean_abs']))

print(f"Heatmap matrix: {heatmap_data.shape[0]} × {heatmap_data.shape[1]}")

n_genes       = len(heatmap_data)
row_mm        = 5.5                          # mm per gene row
overhead_mm   = 30                           # title + x-ticks + colorbar
FIG_HEIGHT_IN = min((n_genes * row_mm + overhead_mm) / 25.4, 225 / 25.4)
FIG_WIDTH_IN  = 170 / 25.4                  # 100 mm — room for dendrogram +
                                             # 3 columns + gene name margin
print(f"Figure size: {100:.0f} mm × {n_genes*row_mm+overhead_mm:.0f} mm")

g = sns.clustermap(
    heatmap_data,
    cmap             = 'RdBu_r',
    center           = 0,
    vmin             = -3,
    vmax             = 0,
    row_cluster      = True,
    col_cluster      = False,
    linewidths       = 0.3,
    linecolor        = '#DDDDDD',
    dendrogram_ratio = (0.10, 0.0),   # 10% width for row dendrogram
    colors_ratio     = 0.0,
    cbar_pos         = None,          # ── disable auto colorbar entirely
    figsize          = (FIG_WIDTH_IN, FIG_HEIGHT_IN),
    tree_kws         = {'linewidths': 0.5},
)

cbar_ax = g.fig.add_axes([0.82, 0.88, 0.03, 0.10])
                          # [left, bottom, width, height] in figure fractions

import matplotlib as mpl
norm = mpl.colors.Normalize(vmin=-3, vmax=0)
cb   = mpl.colorbar.ColorbarBase(
    cbar_ax,
    cmap        = plt.get_cmap('RdBu_r'),
    norm        = norm,
    orientation = 'vertical',
    ticks       = [-3, -2, -1, 0],
)
cb.ax.tick_params(labelsize=5, width=0.4, length=2)
cb.ax.set_ylabel('log$_2$ FC', fontsize=5, labelpad=3)
cb.outline.set_linewidth(0.4)

# ── Margins: shift heatmap to prevent gene name clipping ─────────────────────
g.fig.subplots_adjust(
    left   = 0.02,   # minimal — dendrogram sits here
    right  = 0.80,   # ── pulls heatmap left, gene names render in 0.72–1.0
    bottom = 0.08,   # room for x-tick condition labels
    top    = 0.95,   # room for title
)

# ── Gene name labels ──────────────────────────────────────────────────────────
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(),
    fontsize  = 5.5,
    fontstyle = 'italic',
    va        = 'center',
)
g.ax_heatmap.yaxis.set_tick_params(pad=2, length=0)

# ── Condition labels ──────────────────────────────────────────────────────────
g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    fontsize = 6,
    rotation = 0,
    ha       = 'center',
)
g.ax_heatmap.xaxis.set_tick_params(pad=3, length=0)

g.ax_heatmap.set_xlabel('')
g.ax_heatmap.set_ylabel('')

# ── Panel title ───────────────────────────────────────────────────────────────
g.ax_heatmap.set_title(
    f'Top {n_genes} shared downregulated genes',
    fontsize   = 6,
    fontweight = 'bold',
    loc        = 'left',
    pad        = 4,
)

# ── Save ──────────────────────────────────────────────────────────────────────
pdf_path  = os.path.join(output_dir, "Figure3B.pdf")
tiff_path = os.path.join(output_dir, "Figure3B.tiff")

g.savefig(pdf_path,  format='pdf',  pad_inches=0.02)
print(f"✅ PDF saved:  {pdf_path}")

g.savefig(
    tiff_path,
    format    = 'tiff',
    dpi       = DPI,
    pad_inches= 0.02,
    pil_kwargs= {"compression": "tiff_lzw"},
)
print(f"✅ TIFF saved: {tiff_path}")

plt.close('all')

# ── Download ──────────────────────────────────────────────────────────────────
from google.colab import files
files.download(pdf_path)
files.download(tiff_path)
print("\n✅ Downloads triggered — check your browser's Downloads folder.")

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import textwrap
import os

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────────
csv_path   = "/content/drive/MyDrive/Chd8 data/figures/GO_corrected/GO_nominal_p05_corrected_background.csv"
output_dir = "/content/Figure4A_output"
os.makedirs(output_dir, exist_ok=True)

FIG_WIDTH_IN  = 170 / 25.4   # 6.69 inches
FIG_HEIGHT_IN = 80  / 25.4   # 3.15 inches — 10 bars at ~6 mm each + margins
DPI           = 300

# ── Global rcParams ───────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

# ── Load & prep data ──────────────────────────────────────────────────────────
nom    = pd.read_csv(csv_path)
top10  = nom.head(10).copy()
top10['log_p'] = -np.log10(top10['P-value'])

top10['Label'] = top10['Term'].apply(
    lambda x: '\n'.join(textwrap.wrap(x.split(' (GO')[0], 45))
)
top10 = top10.sort_values('log_p', ascending=True)   # longest bar on top

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))

# ── Colour bars by significance tier ─────────────────────────────────────────
# FDR < 0.05 → dark blue  |  nominal only → mid blue
sig_col  = '#1A4F7A'
nom_col  = '#5B8DB8'

bar_colors = [
    sig_col if p < 0.05 else nom_col
    for p in top10.get('FDR', top10['P-value'])   # fall back to P if no FDR col
]

bars = ax.barh(
    top10['Label'],
    top10['log_p'],
    color     = bar_colors,
    edgecolor = 'none',        # cleaner at small size — no white outlines
    height    = 0.65,
)

# ── Overlap annotation inside bars ───────────────────────────────────────────
for bar, (_, row) in zip(bars, top10.iterrows()):
    bar_w = bar.get_width()
    # Only annotate if bar is wide enough to hold text
    if bar_w > 0.4:
        ax.text(
            bar_w - 0.05,
            bar.get_y() + bar.get_height() / 2,
            f"k/K={row['Overlap']}",
            va        = 'center',
            ha        = 'right',
            fontsize  = 4.5,
            color     = 'white',
            fontweight= 'bold',
        )

# ── Significance threshold line ───────────────────────────────────────────────
ax.axvline(
    -np.log10(0.05),
    color     = '#C0392B',
    linestyle = '--',
    lw        = 0.6,
    label     = 'P = 0.05',
    zorder    = 3,
)

# ── Axis formatting ───────────────────────────────────────────────────────────
ax.set_xlabel('-log$_{10}$(P-value)', fontsize=6)
ax.tick_params(axis='x', labelsize=5, width=0.5, length=2)
ax.tick_params(axis='y', labelsize=5, width=0,   length=0)  # no y-axis ticks
ax.set_xlim(0, top10['log_p'].max() * 1.08)                 # small right pad

# ── Panel title ───────────────────────────────────────────────────────────────
ax.set_title(
    'GO Biological Process enrichment\n',
    fontsize   = 6,
    fontweight = 'bold',
    loc        = 'left',
    pad        = 14,
)

ax.set_title('', fontsize=6)

# ── Legend ────────────────────────────────────────────────────────────────────
# Combine threshold line + colour legend in one compact legend
from matplotlib.patches import Patch
from matplotlib.lines   import Line2D

legend_elements = [
    Line2D ([0], [0], color='#C0392B', lw=0.8,
            linestyle='--', label='P = 0.05'),
    Patch   (facecolor=sig_col, label='FDR < 0.05'),
    Patch   (facecolor=nom_col, label='Nominal only'),
]
ax.legend(
    handles       = legend_elements,
    frameon       = False,
    fontsize      = 5,
    loc           = 'lower right',
    handlelength  = 1.2,
    handletextpad = 0.4,
    borderpad     = 0,
)

# ── Spines ────────────────────────────────────────────────────────────────────
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_linewidth(0.5)

plt.tight_layout(pad=0.4)
plt.subplots_adjust(left=0.38)

# ── Save ──────────────────────────────────────────────────────────────────────
pdf_path  = os.path.join(output_dir, "Figure4A.pdf")
tiff_path = os.path.join(output_dir, "Figure4A.tiff")

fig.savefig(pdf_path,  format='pdf',  pad_inches=0.02)
print(f"✅ PDF saved:  {pdf_path}")

fig.savefig(
    tiff_path,
    format    = 'tiff',
    dpi       = DPI,
    pad_inches= 0.02,
    pil_kwargs= {"compression": "tiff_lzw"},
)
print(f"✅ TIFF saved: {tiff_path}")

plt.close(fig)

# ── Download ──────────────────────────────────────────────────────────────────
from google.colab import files
files.download(pdf_path)
files.download(tiff_path)
print("\n✅ Downloads triggered — check your browser's Downloads folder.")

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gseapy as gp
import pybedtools
import os
import pandas as pd
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

PEAK_PATH   = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH    = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH    = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

output_dir = "/content/Figure4B_output"
os.makedirs(output_dir, exist_ok=True)

FIG_WIDTH_IN = 85 / 25.4  # 3.35 inches — half-page width
DPI          = 300

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,   # embeds fonts in PDF
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# Step 1: NPC-specific ChIP targets
print("1. Identifying NPC-specific ChIP targets...")

npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
es_bt  = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()

npc_specific  = npc_bt.subtract(es_bt, A=True)
npc_genes_bed = tss_bt.intersect(npc_specific, u=True, wa=True)

raw_genes     = sorted(set([str(f[3]) for f in npc_genes_bed]))
all_npc_genes = [g for g in raw_genes
                 if g.lower() not in ('nan', '.', '') and len(g) > 1]

print(f"   NPC-specific genes: {len(all_npc_genes):,}")

n_upper          = sum(1 for g in all_npc_genes[:100] if g == g.upper())
ENRICHR_ORGANISM = "Human" if n_upper > 80 else "mouse"

# Step 2: Intersect with KO DEGs
print("2. Subsetting to NPC targets ∩ KO DEGs...")

ko_df    = pd.read_csv(RNA_KO_PATH, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
gene_col = next(
    (c for c in ko_df.columns if c.upper() in ['GENESYMBOL', 'SYMBOL', 'GENE', 'MGI_SYMBOL']),
    None
)
if gene_col is None:
    raise ValueError(f"Gene column not found. Columns: {ko_df.columns.tolist()}")

ko_df['gene_norm']     = ko_df[gene_col].astype(str).str.upper()
npc_norm_map           = {g.upper(): g for g in all_npc_genes}
ko_df['is_npc_target'] = ko_df['gene_norm'].isin(npc_norm_map)
ko_df['is_deg']        = (ko_df['padj'] < 0.05) & (ko_df['log2FoldChange'].abs() > 0.5)

focused_genes = [
    npc_norm_map[g]
    for g in ko_df.loc[ko_df['is_npc_target'] & ko_df['is_deg'], 'gene_norm']
    if g in npc_norm_map
]

if len(focused_genes) < 30:
    print("   Falling back to full NPC target list (intersection too small).")
    enrich_genes = all_npc_genes
    enrich_label = f"all NPC-specific targets (n={len(all_npc_genes):,})"
else:
    enrich_genes = focused_genes
    enrich_label = f"NPC targets ∩ KO DEGs (n={len(focused_genes):,})"

print(f"   Enrichment input: {enrich_label}")

# Step 3: Run ChEA 2022 enrichment
print("3. Running TF enrichment (ChEA 2022)...")

enr_tf = gp.enrichr(gene_list=enrich_genes, gene_sets=['ChEA_2022'], organism=ENRICHR_ORGANISM)

tf_n_fdr = (enr_tf.results['Adjusted P-value'] < 0.05).sum()
if tf_n_fdr == 0 and (enr_tf.results['P-value'] < 0.05).sum() > 0:
    pval_col    = 'P-value'
    pval_thresh = 0.05
    pval_label  = 'P = 0.05 (nominal)'
    print("   Using nominal P-value (FDR too stringent)")
else:
    pval_col    = 'Adjusted P-value'
    pval_thresh = 0.05
    pval_label  = 'FDR = 0.05'

top_tf = enr_tf.results[enr_tf.results[pval_col] < pval_thresh].head(10).copy()
print(f"   TF terms passing threshold: {len(top_tf)}")

if top_tf.empty:
    print("   No significant TF terms. Top 5 raw results:")
    print(enr_tf.results[['Term', 'P-value', 'Adjusted P-value', 'Overlap']].head(5).to_string())
else:
    top_tf['log_p'] = -np.log10(top_tf[pval_col])
    top_tf['Label'] = (
        top_tf['Term'].str.split('_').str[0]
        + ' (' + top_tf['Overlap'] + ')'
    )
    top_tf = top_tf.sort_values('log_p', ascending=True)

    n_terms       = len(top_tf)
    FIG_HEIGHT_IN = (n_terms * 6 + 20) / 25.4

    fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))

    bars = ax.barh(
        top_tf['Label'],
        top_tf['log_p'],
        color     = '#1A4F7A',
        edgecolor = 'none',
        height    = 0.62,
    )

    for bar, (_, row) in zip(bars, top_tf.iterrows()):
        w = bar.get_width()
        if w > 0.3:
            ax.text(
                w - 0.04,
                bar.get_y() + bar.get_height() / 2,
                row['Overlap'],
                va        = 'center',
                ha        = 'right',
                fontsize  = 4.5,
                color     = 'white',
                fontweight= 'bold',
            )

    ax.axvline(
        -np.log10(pval_thresh),
        color     = '#C0392B',
        linestyle = '--',
        lw        = 0.6,
        zorder    = 3,
    )

    ax.set_xlabel(f'-log$_{{10}}$({pval_col})', fontsize=6)
    ax.set_xlim(0, top_tf['log_p'].max() * 1.08)
    ax.tick_params(axis='x', labelsize=5, width=0.5, length=2)
    ax.tick_params(axis='y', labelsize=5, width=0,   length=0)

    ax.text(
        0, 1.01,
        f'({enrich_label})',
        transform  = ax.transAxes,
        fontsize   = 5,
        va         = 'bottom',
        ha         = 'left',
    )

    from matplotlib.lines import Line2D
    ax.legend(
        handles       = [Line2D([0], [0], color='#C0392B', lw=0.8, linestyle='--', label=pval_label)],
        frameon       = False,
        fontsize      = 5,
        loc           = 'lower right',
        handlelength  = 1.2,
        handletextpad = 0.4,
        borderpad     = 0,
    )

    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)

    plt.tight_layout(pad=0.4)
    plt.subplots_adjust(left=0.42, right=0.97, top=0.92)

    pdf_path  = os.path.join(output_dir, "Figure4B.pdf")
    tiff_path = os.path.join(output_dir, "Figure4B.tiff")

    fig.savefig(pdf_path, format='pdf', pad_inches=0.02)
    print(f"PDF saved: {pdf_path}")

    fig.savefig(
        tiff_path,
        format    = 'tiff',
        dpi       = DPI,
        pad_inches= 0.02,
        pil_kwargs= {"compression": "tiff_lzw"},
    )
    print(f"TIFF saved: {tiff_path}")

    plt.close(fig)

    from google.colab import files
    files.download(pdf_path)
    files.download(tiff_path)

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gseapy as gp
import pybedtools
import os
import pandas as pd
from matplotlib.lines import Line2D

from google.colab import drive
drive.mount('/content/drive')

PEAK_PATH   = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH    = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH    = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

output_dir = "/content/Figure4B_output"
os.makedirs(output_dir, exist_ok=True)

FIG_WIDTH_IN = 85 / 25.4
DPI          = 300

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ── Step 1 ────────────────────────────────────────────────────────────────────
print("1. Identifying NPC-specific ChIP targets...")
npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
es_bt  = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()

npc_specific  = npc_bt.subtract(es_bt, A=True)
npc_genes_bed = tss_bt.intersect(npc_specific, u=True, wa=True)

raw_genes     = sorted(set([str(f[3]) for f in npc_genes_bed]))
all_npc_genes = [g for g in raw_genes
                 if g.lower() not in ('nan', '.', '') and len(g) > 1]
print(f"   NPC-specific genes: {len(all_npc_genes):,}")

n_upper          = sum(1 for g in all_npc_genes[:100] if g == g.upper())
ENRICHR_ORGANISM = "Human" if n_upper > 80 else "mouse"

# ── Step 2 ────────────────────────────────────────────────────────────────────
print("2. Subsetting to NPC targets ∩ KO DEGs...")
ko_df    = pd.read_csv(RNA_KO_PATH, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
gene_col = next(
    (c for c in ko_df.columns if c.upper() in ['GENESYMBOL', 'SYMBOL', 'GENE', 'MGI_SYMBOL']),
    None
)
if gene_col is None:
    raise ValueError(f"Gene column not found. Columns: {ko_df.columns.tolist()}")

ko_df['gene_norm']     = ko_df[gene_col].astype(str).str.upper()
npc_norm_map           = {g.upper(): g for g in all_npc_genes}
ko_df['is_npc_target'] = ko_df['gene_norm'].isin(npc_norm_map)
ko_df['is_deg']        = (ko_df['padj'] < 0.05) & (ko_df['log2FoldChange'].abs() > 0.5)

focused_genes = [
    npc_norm_map[g]
    for g in ko_df.loc[ko_df['is_npc_target'] & ko_df['is_deg'], 'gene_norm']
    if g in npc_norm_map
]

if len(focused_genes) < 30:
    enrich_genes = all_npc_genes
    enrich_label = f"all NPC-specific targets (n={len(all_npc_genes):,})"
else:
    enrich_genes = focused_genes
    enrich_label = f"NPC targets ∩ KO DEGs (n={len(focused_genes):,})"
print(f"   Enrichment input: {enrich_label}")

# ── Step 3 ────────────────────────────────────────────────────────────────────
print("3. Running TF enrichment (ChEA 2022)...")
enr_tf = gp.enrichr(gene_list=enrich_genes, gene_sets=['ChEA_2022'], organism=ENRICHR_ORGANISM)
res    = enr_tf.results.copy()

# Determine significance column
n_fdr = (res['Adjusted P-value'] < 0.05).sum()
if n_fdr >= 5:
    pval_col    = 'Adjusted P-value'
    pval_thresh = 0.05
    pval_label  = 'FDR = 0.05'
    top_tf      = res[res[pval_col] < pval_thresh].head(10).copy()
    sig_note    = f"FDR < 0.05 (n={len(top_tf)})"
elif (res['P-value'] < 0.05).sum() >= 5:
    pval_col    = 'P-value'
    pval_thresh = 0.05
    pval_label  = 'P = 0.05 (nominal)'
    top_tf      = res[res[pval_col] < pval_thresh].head(10).copy()
    sig_note    = f"nominal P < 0.05 (n={len(top_tf)})"
else:
    # Fallback: show top 10 by raw p-value regardless of threshold
    pval_col    = 'P-value'
    pval_thresh = res['P-value'].nsmallest(10).max()
    pval_label  = 'Top 10 by P-value'
    top_tf      = res.nsmallest(10, 'P-value').copy()
    sig_note    = "top 10 terms (no threshold)"

print(f"   Plotting: {sig_note}")

top_tf['log_p'] = -np.log10(top_tf[pval_col].clip(lower=1e-300))
top_tf['TF']    = top_tf['Term'].str.split('_').str[0]
top_tf['Label'] = top_tf['TF'] + ' (' + top_tf['Overlap'] + ')'
top_tf          = top_tf.sort_values('log_p', ascending=True).reset_index(drop=True)

n_terms       = len(top_tf)
FIG_HEIGHT_IN = (n_terms * 6 + 20) / 25.4

fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))

bars = ax.barh(
    top_tf['Label'],
    top_tf['log_p'],
    color     = '#1A4F7A',
    edgecolor = 'none',
    height    = 0.62,
)

# Overlap annotation — inside bar if wide enough, else just to the right
x_max = top_tf['log_p'].max()
thresh_inside = x_max * 0.25   # bar must be >25% of max to fit inside label

for bar, (_, row) in zip(bars, top_tf.iterrows()):
    w = bar.get_width()
    cy = bar.get_y() + bar.get_height() / 2
    if w >= thresh_inside:
        ax.text(w - 0.04, cy, row['Overlap'],
                va='center', ha='right', fontsize=4.5,
                color='white', fontweight='bold')
    else:
        ax.text(w + 0.04, cy, row['Overlap'],
                va='center', ha='left', fontsize=4.5,
                color='#1A4F7A', fontweight='bold')

# Threshold line (only if meaningful)
if pval_col != 'P-value' or pval_thresh == 0.05:
    ax.axvline(-np.log10(pval_thresh), color='#C0392B',
               linestyle='--', lw=0.6, zorder=3)

ax.set_xlabel(f'-log$_{{10}}$({pval_col})', fontsize=6)
ax.set_xlim(0, x_max * 1.18)   # extra right margin for outside labels
ax.tick_params(axis='x', labelsize=5, width=0.5, length=2)
ax.tick_params(axis='y', labelsize=5, width=0,   length=0)
ax.text(-0.3, 1.01, f'({enrich_label})',
        transform=ax.transAxes, fontsize=5, va='bottom', ha='left')

ax.legend(
    handles=[Line2D([0], [0], color='#C0392B', lw=0.8,
                    linestyle='--', label=pval_label)],
    frameon=False, fontsize=5, loc='lower right',
    handlelength=1.2, handletextpad=0.4, borderpad=0,
)

ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_linewidth(0.5)

plt.tight_layout(pad=0.4)
plt.subplots_adjust(left=0.70, right=0.97, top=0.92)

pdf_path  = os.path.join(output_dir, "Figure4B.pdf")
tiff_path = os.path.join(output_dir, "Figure4B.tiff")

fig.savefig(pdf_path, format='pdf', pad_inches=0.02)
fig.savefig(tiff_path, format='tiff', dpi=DPI, pad_inches=0.02,
            pil_kwargs={"compression": "tiff_lzw"})
plt.close(fig)
print(f"Saved: {pdf_path}")
print(f"Saved: {tiff_path}")

from google.colab import files
files.download(pdf_path)
files.download(tiff_path)

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import gseapy as gp
import pybedtools
import os
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

PEAK_PATH   = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH    = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH    = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

output_dir = "/content/Figure4C_output"
os.makedirs(output_dir, exist_ok=True)

FIG_WIDTH_IN  = 170 / 25.4        # 6.69 in
MAX_HEIGHT_IN = 225 / 25.4        # 8.86 in — hard ceiling per journal spec
DPI           = 300

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 7,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,         # fonts embedded — required
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
    'lines.linewidth': 0.5,
})

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ── Step 1: NPC-specific targets ──────────────────────────────────────────────
print("1. Identifying NPC-specific ChIP targets...")
npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
es_bt  = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()

npc_specific  = npc_bt.subtract(es_bt, A=True)
npc_genes_bed = tss_bt.intersect(npc_specific, u=True, wa=True)

raw_genes     = sorted(set([str(f[3]) for f in npc_genes_bed]))
all_npc_genes = [g for g in raw_genes
                 if g.lower() not in ('nan', '.', '') and len(g) > 1]
print(f"   NPC-specific genes: {len(all_npc_genes):,}")

n_upper          = sum(1 for g in all_npc_genes[:100] if g == g.upper())
ENRICHR_ORGANISM = "Human" if n_upper > 80 else "mouse"

# ── Step 2: Intersect with KO DEGs ───────────────────────────────────────────
print("2. Subsetting to NPC targets ∩ KO DEGs...")
ko_df    = pd.read_csv(RNA_KO_PATH, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
gene_col = next(
    (c for c in ko_df.columns if c.upper() in ['GENESYMBOL', 'SYMBOL', 'GENE', 'MGI_SYMBOL']),
    None
)
if gene_col is None:
    raise ValueError(f"Gene column not found. Columns: {ko_df.columns.tolist()}")

ko_df['gene_norm']     = ko_df[gene_col].astype(str).str.upper()
npc_norm_map           = {g.upper(): g for g in all_npc_genes}
ko_df['is_npc_target'] = ko_df['gene_norm'].isin(npc_norm_map)
ko_df['is_deg']        = (ko_df['padj'] < 0.05) & (ko_df['log2FoldChange'].abs() > 0.5)

focused_genes = [
    npc_norm_map[g]
    for g in ko_df.loc[ko_df['is_npc_target'] & ko_df['is_deg'], 'gene_norm']
    if g in npc_norm_map
]

if len(focused_genes) < 30:
    enrich_genes = all_npc_genes
else:
    enrich_genes = focused_genes

# ── Step 3: GO enrichment ─────────────────────────────────────────────────────
print("3. Running GO Biological Process enrichment...")
enr_go = gp.enrichr(
    gene_list = enrich_genes,
    gene_sets = ['GO_Biological_Process_2023'],
    organism  = ENRICHR_ORGANISM
)

# ── Step 4: Build dot plot data ───────────────────────────────────────────────
print("4. Building dot plot data...")
themes = ['Neurogenesis', 'Synaptic', 'Chromatin', 'Transcription', 'Axon']
plot_data = []

for theme in themes:
    subset = enr_go.results[
        enr_go.results['Term'].str.contains(theme, case=False)
    ].head(1)
    if not subset.empty:
        genes = [g.strip() for g in str(subset['Genes'].values[0]).split(';') if g.strip()]
        sig   = -np.log10(max(subset['P-value'].values[0], 1e-300))
        for g in genes[:10]:
            plot_data.append({'Category': theme, 'Gene': g, 'Significance': sig})

if not plot_data:
    print("   No theme keywords matched. Available terms:")
    print(enr_go.results['Term'].head(15).tolist())
    raise SystemExit("Update 'themes' to match your GO results.")

df_dot = pd.DataFrame(plot_data)

sig_min    = df_dot['Significance'].min()
sig_max    = df_dot['Significance'].max()
size_range = sig_max - sig_min if sig_max > sig_min else 1
df_dot['dot_size'] = (df_dot['Significance'] - sig_min) / size_range * 450 + 50

norm = mcolors.Normalize(vmin=sig_min, vmax=sig_max)

# Dynamic height — capped at journal max
n_genes       = df_dot['Gene'].nunique()
FIG_HEIGHT_IN = min(n_genes * 0.18 + 1.5, MAX_HEIGHT_IN)

# ── Step 5: Plot ──────────────────────────────────────────────────────────────
# Leave right margin for size legend (outside plot area)
fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))
fig.subplots_adjust(left=0.12, right=0.78, top=0.97, bottom=0.18)

scatter = ax.scatter(
    x          = df_dot['Category'],
    y          = df_dot['Gene'],
    s          = df_dot['dot_size'],
    c          = df_dot['Significance'],
    cmap       = 'magma',
    norm       = norm,
    edgecolors = 'black',
    linewidths = 0.3,
    zorder     = 3,
)

# ── Colorbar — placed outside right of plot ───────────────────────────────────
cbar_ax = fig.add_axes([0.80, 0.55, 0.02, 0.30])   # [left, bottom, width, height]
cbar    = fig.colorbar(scatter, cax=cbar_ax)
cbar.set_label('-log$_{10}$ P-value', fontsize=6)
cbar.ax.tick_params(labelsize=5, width=0.5, length=2)
cbar.outline.set_linewidth(0.5)

# ── Size legend — outside right, below colorbar, no overlap ──────────────────
from matplotlib.lines import Line2D
size_legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#888888', markeredgecolor='black',
           markeredgewidth=0.3, markersize=np.sqrt(50),
           label='Low'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#888888', markeredgecolor='black',
           markeredgewidth=0.3, markersize=np.sqrt(275),
           label='Mid'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#888888', markeredgecolor='black',
           markeredgewidth=0.3, markersize=np.sqrt(500),
           label='High'),
]
ax.legend(
    handles=size_legend_elements,
    title='Significance', title_fontsize=5,
    fontsize=5, loc='lower right',
    frameon=True, framealpha=0.9,
    edgecolor='#CCCCCC', handlelength=1.5,
    bbox_to_anchor=(1.12, 0.0)
)

# ── Axis formatting ───────────────────────────────────────────────────────────
ax.set_xlabel('Functional category', fontsize=7)
ax.set_ylabel('Gene', fontsize=7)
ax.tick_params(axis='x', rotation=15, labelsize=6, width=0.5, length=2)
ax.tick_params(axis='y', labelsize=5, width=0.5, length=2)
ax.grid(True, linestyle=':', linewidth=0.3, alpha=0.5, zorder=0)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(0.5)

# ── Step 6: Save ──────────────────────────────────────────────────────────────
pdf_path  = os.path.join(output_dir, "Figure4C.pdf")
tiff_path = os.path.join(output_dir, "Figure4C.tiff")

fig.savefig(pdf_path,  format='pdf',  pad_inches=0.02)
fig.savefig(tiff_path, format='tiff', dpi=DPI, pad_inches=0.02,
            pil_kwargs={"compression": "tiff_lzw"})
plt.close(fig)
print(f"Saved: {pdf_path}")
print(f"Saved: {tiff_path}")

from google.colab import files
files.download(pdf_path)
files.download(tiff_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import textwrap
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from matplotlib.lines import Line2D
import os
import requests

from google.colab import drive
drive.mount('/content/drive')

BASE_RNA = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
SAVE_DIR = "/content/drive/MyDrive/Chd8 data/figures/GO_corrected/"
os.makedirs(SAVE_DIR, exist_ok=True)

HALF_W = 85  / 25.4
MAX_H  = 225 / 25.4
DPI    = 300

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
    'lines.linewidth': 0.5,
})

# ── Download GMT directly from Enrichr (single file, no API listing needed) ──
GMT_PATH = "/content/GO_Biological_Process_2023_mouse.gmt"

if not os.path.exists(GMT_PATH):
    print("Downloading GMT file...")
    url = ("https://maayanlab.cloud/Enrichr/geneSetLibrary"
           "?mode=text&libraryName=GO_Biological_Process_2023")
    r = requests.get(url, timeout=60)
    if r.ok:
        with open(GMT_PATH, 'w') as f:
            f.write(r.text)
        print("Downloaded.")
    else:
        raise Exception(
            f"GMT download failed (status {r.status_code}).\n"
            "Manual fix: go to https://maayanlab.cloud/Enrichr/#libraries, "
            "download 'GO_Biological_Process_2023', upload to Colab, "
            "and set GMT_PATH to its path."
        )

# ── Parse GMT ─────────────────────────────────────────────────────────────────
def parse_gmt(path):
    gmt = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            term  = parts[0]
            genes = set(g.upper() for g in parts[2:] if g.strip())
            gmt[term] = genes
    return gmt

gmt = parse_gmt(GMT_PATH)
print(f"GMT loaded: {len(gmt):,} terms")

# ── Load & filter RNA-seq ─────────────────────────────────────────────────────
rna = pd.read_csv(BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv", sep='\t')
rna = rna[rna['baseMean'] > 10].dropna(subset=['log2FoldChange', 'padj'])
rna['gene']      = rna['GENESYMBOL'].str.upper().str.strip()
background_genes = set(rna['gene'].tolist())
ko_down          = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'] < -0.5)]['gene'].tolist())
print(f"Input: {len(ko_down):,} | Background: {len(background_genes):,}")

# ── Fisher's exact test ───────────────────────────────────────────────────────
N, k = len(background_genes), len(ko_down)
rows = []
for term, term_genes in gmt.items():
    term_genes_bg = term_genes & background_genes
    K = len(term_genes_bg)
    if K < 5 or K > 500:
        continue
    overlap = ko_down & term_genes_bg
    x = len(overlap)
    if x == 0:
        continue
    _, pval = fisher_exact(
        [[x, k-x], [K-x, N-K-(k-x)]], alternative='greater'
    )
    rows.append({
        'Term'         : term,
        'P-value'      : pval,
        'Overlap_n'    : x,
        'Term_size'    : K,
        'Overlap'      : f"{x}/{K}",
        'Overlap_genes': ';'.join(sorted(overlap))
    })

results = pd.DataFrame(rows)
_, padj, _, _ = multipletests(results['P-value'], method='fdr_bh')
results['Adjusted P-value'] = padj
results = results.sort_values('Adjusted P-value')

sig = results[results['Adjusted P-value'] < 0.05]
nom = results[results['P-value'] < 0.05]
print(f"FDR < 0.05: {len(sig):,} | Nominal P < 0.05: {len(nom):,}")

results.to_csv(SAVE_DIR + "Figure5A_GO_all.csv",   index=False)
sig.to_csv(    SAVE_DIR + "Figure5A_GO_FDR05.csv", index=False)

# ── Select plotting data ──────────────────────────────────────────────────────
if len(sig) >= 5:
    plot_df     = sig
    pval_col    = 'Adjusted P-value'
    pval_thresh = 0.05
    pval_label  = 'FDR = 0.05'
elif len(nom) >= 5:
    plot_df     = nom
    pval_col    = 'P-value'
    pval_thresh = 0.05
    pval_label  = 'P = 0.05 (nominal)'
else:
    plot_df     = results
    pval_col    = 'P-value'
    pval_thresh = results['P-value'].nsmallest(10).max()
    pval_label  = 'Top 10 by P-value'

top10 = plot_df.head(10).copy()
top10['log_p'] = -np.log10(top10[pval_col].clip(lower=1e-300))
top10['Label'] = top10['Term'].apply(
    lambda x: '\n'.join(textwrap.wrap(x.split(' (GO')[0], 38))
)
top10 = top10.sort_values('log_p', ascending=True)

# ── Plot ──────────────────────────────────────────────────────────────────────
n     = len(top10)
fig_h = min((n * 6 + 20) / 25.4, MAX_H)

fig, ax = plt.subplots(figsize=(HALF_W, fig_h))

bars = ax.barh(top10['Label'], top10['log_p'],
               color='#2E86AB', edgecolor='none', height=0.62)

x_max         = top10['log_p'].max()
thresh_inside = x_max * 0.25

for bar, (_, row) in zip(bars, top10.iterrows()):
    w  = bar.get_width()
    cy = bar.get_y() + bar.get_height() / 2
    if w >= thresh_inside:
        ax.text(w - 0.04, cy, row['Overlap'],
                va='center', ha='right', fontsize=4.5,
                color='white', fontweight='bold')
    else:
        ax.text(w + 0.04, cy, row['Overlap'],
                va='center', ha='left', fontsize=4.5,
                color='#2E86AB', fontweight='bold')

ax.axvline(-np.log10(pval_thresh), color='#C0392B',
           linestyle='--', lw=0.6, zorder=3)
ax.set_xlabel(f'-log$_{{10}}$({pval_col})', fontsize=6)
ax.set_xlim(0, x_max * 1.18)
ax.legend(
    handles=[Line2D([0], [0], color='#C0392B', lw=0.8,
                    linestyle='--', label=pval_label)],
    frameon=False, fontsize=5, loc='lower right',
    handlelength=1.2, handletextpad=0.4, borderpad=0,
)

ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_linewidth(0.5)
ax.tick_params(axis='x', labelsize=5, width=0.5, length=2)
ax.tick_params(axis='y', labelsize=5, width=0,   length=0)

plt.tight_layout(pad=0.4)
plt.subplots_adjust(left=0.55, right=0.97, top=0.97, bottom=0.12)

# ── Save ──────────────────────────────────────────────────────────────────────
pdf_path  = SAVE_DIR + "Figure5A.pdf"
tiff_path = SAVE_DIR + "Figure5A.tiff"

fig.savefig(pdf_path,  format='pdf',  pad_inches=0.02)
fig.savefig(tiff_path, format='tiff', dpi=DPI, pad_inches=0.02,
            pil_kwargs={"compression": "tiff_lzw"})
plt.close(fig)
print(f"Saved: {pdf_path}")
print(f"Saved: {tiff_path}")

from google.colab import files
files.download(pdf_path)
files.download(tiff_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import gseapy as gp

# ── 0. Matplotlib global settings ────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":        "Arial",
    "font.size":          7,
    "axes.titlesize":     8,
    "axes.labelsize":     7,
    "xtick.labelsize":    6,
    "ytick.labelsize":    6,
    "legend.fontsize":    6,
    "lines.linewidth":    1.0,
    "axes.linewidth":     0.6,
    "xtick.major.width":  0.6,
    "ytick.major.width":  0.6,
    "xtick.minor.width":  0.4,
    "ytick.minor.width":  0.4,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
    "figure.dpi":         300,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.02,
})

MM_TO_IN = 1 / 25.4
FIG_W    = 170 * MM_TO_IN       # double-column width: 170 mm
FIG_H    = 140 * MM_TO_IN       # suitable height  : 140 mm

# ── 1. LOAD DATA ──────────────────────────────────────────────────────────────
si_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Scrambled_siRNA_Diff_Chd8_siRNA.tsv"
si_df = pd.read_csv(si_path, sep="\t").dropna(subset=["log2FoldChange", "padj"])

# ── 2. GENERATE RANKING ───────────────────────────────────────────────────────
np.random.seed(42)
jitter = np.random.uniform(low=1e-12, high=1e-10, size=len(si_df))
si_df["rank_metric"] = (
    -np.log10(si_df["padj"] + 1e-300) * np.sign(si_df["log2FoldChange"])
) + jitter

gene_col = "GENESYMBOL" if "GENESYMBOL" in si_df.columns else "symbol"
rnk = (
    si_df.sort_values("rank_metric", ascending=False)
         .set_index(gene_col)["rank_metric"]
)

# ── 3. RUN GSEA ───────────────────────────────────────────────────────────────
print("Running GSEA for CHD8 KD …")
pre_res = gp.prerank(
    rnk=rnk,
    gene_sets="GO_Biological_Process_2023",
    organism="Mouse",
    permutation_num=1000,
    seed=42,
)

# ── 4. EXTRACT RESULTS ────────────────────────────────────────────────────────
res_df  = pre_res.res2d.sort_values("FDR q-val")
top_row = res_df.iloc[0]
top_term = top_row["Term"]
nes  = top_row["NES"]
fdr  = top_row["FDR q-val"]
pval = top_row.get("NOM p-val", np.nan)

# Gene count (leading-edge)
try:
    gene_list_col = next((c for c in res_df.columns if c.lower() == "genes"), None)
    if gene_list_col and pd.notna(top_row[gene_list_col]):
        gene_count = len([g for g in str(top_row[gene_list_col]).split(";") if g])
    else:
        gene_count = pre_res.results[top_term].get("tag_size", 0)
        if gene_count == 0:
            gene_count = len(pre_res.results[top_term].get("hits", []))
except Exception:
    gene_count = "N/A"

print(f"\nTop pathway : {top_term}")
print(f"NES={nes:.3f}  FDR={fdr:.2e}  Genes={gene_count}")

# ── 5. EXTRACT PLOT DATA ──────────────────────────────────────────────────────
res      = pre_res.results[top_term]
rank_met = pre_res.ranking        # pandas Series: gene → rank metric

hits     = res.get("hits", [])    # indices into the ranked list
RES      = res.get("RES", [])     # running enrichment score array
x_full   = np.arange(len(rank_met))

# Running ES curve
es_curve = np.array(RES) if len(RES) > 0 else np.zeros(len(x_full))
es_peak_idx = int(np.argmax(np.abs(es_curve)))

# Rank metric values
rank_vals = rank_met.values

# ── 6. BUILD FIGURE ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(FIG_W, FIG_H))

# Three-panel layout: ES curve (top, tall) | hit rug (middle, thin) | rank metric (bottom)
gs = gridspec.GridSpec(
    3, 1,
    figure=fig,
    height_ratios=[5, 0.6, 2],
    hspace=0.08,
)

# ── Panel A : Running Enrichment Score ────────────────────────────────────────
ax_es = fig.add_subplot(gs[0])

pos_color = "#D62728"    # red  — enriched in CHD8 KD (up)
neg_color = "#1F77B4"    # blue — enriched in scrambled (down)
curve_col = pos_color if nes >= 0 else neg_color

ax_es.plot(x_full, es_curve, color=curve_col, lw=1.2, zorder=3)
ax_es.axhline(0, color="black", lw=0.5, ls="--", zorder=2)
ax_es.axvline(es_peak_idx, color="grey", lw=0.5, ls=":", zorder=1)

# Peak ES annotation
es_peak_val = es_curve[es_peak_idx]
ax_es.annotate(
    f"ES = {es_peak_val:.2f}",
    xy=(es_peak_idx, es_peak_val),
    xytext=(es_peak_idx + len(x_full) * 0.05, es_peak_val * 0.85),
    fontsize=6,
    arrowprops=dict(arrowstyle="-", color="grey", lw=0.5),
    color="black",
)

ax_es.set_ylabel("Enrichment Score (ES)", fontsize=7)
ax_es.set_xlim(0, len(x_full) - 1)
ax_es.tick_params(labelbottom=False, direction="out", length=3)
ax_es.spines[["top", "right"]].set_visible(False)

# Stats box (top-right corner)
stats_txt = (
    f"NES = {nes:.2f}\n"
    f"FDR q = {fdr:.2e}\n"
    f"n = {gene_count} genes"
)
ax_es.text(
    0.98, 0.97, stats_txt,
    transform=ax_es.transAxes,
    va="top", ha="right",
    fontsize=6,
    linespacing=1.4,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="grey", lw=0.5, alpha=0.85),
)

# Title
term_label = top_term.replace("(GO:", "\n(GO:")   # wrap long GO term names
ax_es.set_title(f"c  {term_label}", fontsize=8, fontweight="bold", loc="left", pad=4)

# ── Panel B : Hit rug ─────────────────────────────────────────────────────────
ax_rug = fig.add_subplot(gs[1])
if len(hits) > 0:
    ax_rug.vlines(hits, ymin=0, ymax=1, lw=0.4, color=curve_col, alpha=0.7)
ax_rug.set_xlim(0, len(x_full) - 1)
ax_rug.set_ylim(0, 1)
ax_rug.set_yticks([])
ax_rug.tick_params(labelbottom=False, bottom=False)
ax_rug.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax_rug.set_facecolor("#F5F5F5")

# ── Panel C : Rank metric ─────────────────────────────────────────────────────
ax_rank = fig.add_subplot(gs[2])

pos_mask = rank_vals >= 0
neg_mask = rank_vals < 0
ax_rank.fill_between(x_full[pos_mask], 0, rank_vals[pos_mask],
                     color=pos_color, alpha=0.70, lw=0)
ax_rank.fill_between(x_full[neg_mask], 0, rank_vals[neg_mask],
                     color=neg_color, alpha=0.70, lw=0)
ax_rank.axhline(0, color="black", lw=0.5)

ax_rank.set_xlabel("Rank in ordered gene list", fontsize=7)
ax_rank.set_ylabel("Ranked metric\n(−log₁₀ FDR × sign)", fontsize=7)
ax_rank.set_xlim(0, len(x_full) - 1)
ax_rank.tick_params(direction="out", length=3)
ax_rank.spines[["top", "right"]].set_visible(False)

# Add "Correlated with CHD8 KD" / "Correlated with Scrambled" labels
ymax = np.max(np.abs(rank_vals))
ax_rank.text(0.02, 0.90, "↑ CHD8 KD", transform=ax_rank.transAxes,
             fontsize=6, color=pos_color, va="top")
ax_rank.text(0.98, 0.10, "↑ Scrambled", transform=ax_rank.transAxes,
             fontsize=6, color=neg_color, va="bottom", ha="right")

# ── 7. SAVE  ──────────────────────────────────────────
out_tiff_300  = "gsea_chd8_kd_300dpi.tiff"
out_tiff_600  = "gsea_chd8_kd_600dpi.tiff"
out_pdf       = "gsea_chd8_kd.pdf"

# 300 DPI TIFF (LZW) — standard colour/halftone submission
fig.savefig(out_tiff_300, dpi=300, format="tiff",
            pil_kwargs={"compression": "tiff_lzw"})

# 600 DPI TIFF (LZW) — line-art / high-quality version
fig.savefig(out_tiff_600, dpi=600, format="tiff",
            pil_kwargs={"compression": "tiff_lzw"})

# PDF (vector, fonts embedded via pdf.fonttype=42 above)
fig.savefig(out_pdf, dpi=300, format="pdf")

plt.show()

print("\nSaved:")
print(f"  {out_tiff_300}  (300 DPI, TIFF/LZW — submit as colour figure)")
print(f"  {out_tiff_600}  (600 DPI, TIFF/LZW — line-art quality)")
print(f"  {out_pdf}         (PDF vector, fonts embedded)")

In [ ]:
!pip install mygene
import os
import pandas as pd
import numpy as np
import gseapy as gp
import mygene
import matplotlib as mpl
import matplotlib.pyplot as plt

# ── Matplotlib global settings ────────────────────────
mpl.rcParams.update({
    "font.family":        "Arial",
    "font.size":          7,
    "axes.titlesize":     8,
    "axes.labelsize":     7,
    "xtick.labelsize":    6,
    "ytick.labelsize":    6,
    "legend.fontsize":    6,
    "lines.linewidth":    0.8,
    "axes.linewidth":     0.6,
    "xtick.major.width":  0.6,
    "ytick.major.width":  0.6,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "pdf.fonttype":       42,   # embed TrueType fonts in PDF
    "ps.fonttype":        42,
    "figure.dpi":         300,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.02,
})

MM_TO_IN = 1 / 25.4
FIG_W    = 170 * MM_TO_IN   # 170 mm double-column width

# ================================
# 📁 1. DEFINE PATHS & LOAD DATA
# ================================
INPUT_FILE  = "/content/drive/MyDrive/Chd8 data/deseq2_results.csv"
BASE_DIR    = "/content/drive/MyDrive/CHD8_project"

RNK_FILE    = os.path.join(BASE_DIR, "results/gsea/gsea_ranked_list.rnk")
GSEA_OUTDIR = os.path.join(BASE_DIR, "results/gsea/")
TABLE_OUT   = os.path.join(BASE_DIR, "results/tables/GSEA_full_results.csv")
PLOT_TIFF   = os.path.join(BASE_DIR, "results/plots/Figure5C_300dpi.tiff")
PLOT_TIFF_HQ = os.path.join(BASE_DIR, "results/plots/Figure5C_600dpi.tiff")
PLOT_PDF    = os.path.join(BASE_DIR, "results/plots/Figure5C_vector.pdf")

for path in [RNK_FILE, TABLE_OUT, PLOT_TIFF]:
    os.makedirs(os.path.dirname(path), exist_ok=True)

print("Loading data...")
df = pd.read_csv(INPUT_FILE)

if "gene" not in df.columns:
    df.rename(columns={df.columns[0]: "gene"}, inplace=True)

# ================================
# 🔄 2. CONVERT ENSEMBL → SYMBOL
# ================================
print("Converting ENSEMBL IDs to Gene Symbols...")
mg = mygene.MyGeneInfo()
ensembl_ids = df["gene"].unique().tolist()
results = mg.querymany(
    ensembl_ids, scopes="ensembl.gene",
    fields="symbol", species="mouse", as_dataframe=True
)

if "symbol" in results.columns:
    symbol_map = results["symbol"].dropna().to_dict()
    df["gene_symbol"] = df["gene"].map(symbol_map)
    df = df.dropna(subset=["gene_symbol"])
    print(f"Mapped {len(df)} genes to symbols.")
else:
    raise ValueError("ID conversion failed. Check internet connection or ID format.")

# ================================
# 🧹 3. RANKING & SAVE RNK
# ================================
print("Saving ranked file...")
df["padj"] = df["padj"].replace(0, 1e-300)
df["ranking_score"] = -np.log10(df["padj"]) * np.sign(df["log2FoldChange"])

ranked_df = df.sort_values("ranking_score", ascending=False)
rnk = ranked_df[["gene_symbol", "ranking_score"]].drop_duplicates(subset="gene_symbol")
rnk.to_csv(RNK_FILE, sep="\t", index=False, header=False)

# ================================
# 🚀 4. RUN GSEA
# ================================
print("Running GSEA...")
gsea_res = gp.prerank(
    rnk=RNK_FILE,
    gene_sets="GO_Biological_Process_2023",
    outdir=GSEA_OUTDIR,
    permutation_num=1000,
    min_size=5,
    max_size=1000,
    seed=42,
    verbose=True,
)

# ================================
# 📈 5. PROCESS RESULTS
# ================================
print("Processing GSEA results...")
res = gsea_res.res2d

# Column detection
fdr_col  = next((c for c in ["FDR q-val", "fdr", "FDR"] if c in res.columns), None)
lead_col = next(
    (c for c in ["Lead_genes", "lead_genes", "genes", "leading_edge"] if c in res.columns),
    None,
)
pval_col = next((c for c in ["NOM p-val", "pvalue"] if c in res.columns), "pvalue")

res = res.rename(columns={fdr_col: "FDR", pval_col: "pvalue", "nes": "NES"})

if lead_col is None:
    res["Gene_Count"] = 0
    print("Warning: No leading-edge gene column found — Gene_Count set to 0.")
else:
    res["Gene_Count"] = res[lead_col].apply(
        lambda x: len(str(x).split(";")) if pd.notnull(x) else 0
    )

res.to_csv(TABLE_OUT)
print(f"Full GSEA table saved: {len(res)} terms")

res_sig = res[res["FDR"] < 0.05].copy()
print(f"Significant pathways (FDR < 0.05): {len(res_sig)}")

if not res_sig.empty:
    print("\n--- TOP SIGNIFICANT TERMS ---")
    print(res_sig[["Term", "NES", "FDR"]].sort_values("NES", ascending=False).to_string())
    print("------------------------------\n")

# ================================
# 📊 6. FIGURE 5C
# ================================
print("Generating Figure 5C (- spec)...")

res_up   = res_sig.sort_values("NES", ascending=False).head(10)
res_down = res_sig.sort_values("NES", ascending=True).head(10)
plot_df  = pd.concat([res_up, res_down]).drop_duplicates(subset="Term")

# Clean term labels: remove GO ID suffix and underscores
plot_df = plot_df.copy()
plot_df["Term"] = (
    plot_df["Term"]
    .str.replace("_", " ", regex=False)
    .str.split("(").str[0]
    .str.strip()
    .str.capitalize()
)

# Sort by NES for display
plot_df = plot_df.sort_values("NES", ascending=True)

n_terms  = len(plot_df)
FIG_H = min(max(100, n_terms * 8), 225) * MM_TO_IN

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

# Colours: red = up in CHD8 KD, blue = down
colors = ["#D62728" if x > 0 else "#1F77B4" for x in plot_df["NES"]]
bars = ax.barh(
    plot_df["Term"],
    plot_df["NES"],
    color=colors,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.4,
    height=0.65,
)

# ── Bar annotations (NES, FDR, n) ─────────────────────────────────────────
max_nes = plot_df["NES"].abs().max()
x_pad   = max_nes * 0.04   # small offset from bar tip

for i, (nes, fdr, count) in enumerate(
    zip(plot_df["NES"], plot_df["FDR"], plot_df["Gene_Count"])
):
    label  = f"NES={nes:.2f}, FDR={fdr:.1e}, n={count}"
    x_pos  = nes + x_pad if nes > 0 else nes - x_pad
    ha     = "left"       if nes > 0 else "right"
    ax.text(x_pos, i, label, va="center", ha=ha, fontsize=5.5, color="black")

# ── Axes formatting ───────────────────────────────────────────────────────
ax.axvline(0, color="black", lw=0.6, zorder=3)

nes_range = plot_df["NES"].abs().max()
ax.set_xlim(-(nes_range + 1.5), nes_range + 1.5)

ax.set_xlabel("Normalized Enrichment Score (NES)", fontsize=7, fontweight="bold")
ax.text(-0.02, 1.03, 'c', transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='bottom', ha='right')

ax.tick_params(axis="x", direction="out", length=3)
ax.tick_params(axis="y", length=0, pad=3)
ax.spines[["top", "right"]].set_visible(False)
ax.spines["left"].set_visible(False)

# ── Legend ────────────────────────────────────────────────────────────────
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#D62728", edgecolor="black", linewidth=0.4,
          label="Enriched in CHD8 KD"),
    Patch(facecolor="#1F77B4", edgecolor="black", linewidth=0.4,
          label="Enriched in Scrambled"),
]
ax.legend(
    handles=legend_elements,
    fontsize=6,
    frameon=True,
    framealpha=0.9,
    edgecolor="grey",
    loc="lower right",
)

plt.tight_layout()

# ── 7. SAVE OUTPUTS ───────────────────────────────────────────────────────
# 300 DPI TIFF/LZW — standard colour submission
fig.savefig(PLOT_TIFF, dpi=300, format="tiff",
            pil_kwargs={"compression": "tiff_lzw"})

# 600 DPI TIFF/LZW — high-quality line-art version
fig.savefig(PLOT_TIFF_HQ, dpi=600, format="tiff",
            pil_kwargs={"compression": "tiff_lzw"})

# PDF — vector with embedded fonts
fig.savefig(PLOT_PDF, dpi=300, format="pdf")

plt.show()

print(f"\nSaved:")
print(f"  {PLOT_TIFF}   (300 DPI TIFF/LZW — colour submission)")
print(f"  {PLOT_TIFF_HQ}  (600 DPI TIFF/LZW — line-art quality)")
print(f"  {PLOT_PDF}     (PDF vector, fonts embedded)")
print("GSEA PIPELINE COMPLETED SUCCESSFULLY")

In [ ]:
!pip install mygene
import os
import pandas as pd
import numpy as np
import gseapy as gp
import mygene
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Matplotlib global settings ────────────────────────
mpl.rcParams.update({
    "font.family":        "Arial",
    "font.size":          7,
    "axes.titlesize":     8,
    "axes.labelsize":     7,
    "xtick.labelsize":    6,
    "ytick.labelsize":    6,
    "legend.fontsize":    6,
    "lines.linewidth":    0.8,
    "axes.linewidth":     0.6,
    "xtick.major.width":  0.6,
    "ytick.major.width":  0.6,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
    "figure.dpi":         300,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.02,
})

MM_TO_IN = 1 / 25.4
FIG_W    = 170 * MM_TO_IN   # 170 mm double-column width

# ── Paths (same as original) ──────────────────────────────────────────────────
INPUT_FILE   = "/content/drive/MyDrive/Chd8 data/deseq2_results.csv"
BASE_DIR     = "/content/drive/MyDrive/CHD8_project"
RNK_FILE     = os.path.join(BASE_DIR, "results/gsea/gsea_ranked_list.rnk")
GSEA_OUTDIR  = os.path.join(BASE_DIR, "results/gsea/")
TABLE_OUT    = os.path.join(BASE_DIR, "results/tables/GSEA_full_results.csv")

for path in [RNK_FILE, TABLE_OUT]:
    os.makedirs(os.path.dirname(path), exist_ok=True)

# ── Output: Colab temp dir, then downloaded directly ───────────────────
output_dir = "/content/Figure5C_output"
os.makedirs(output_dir, exist_ok=True)
PLOT_PDF     = os.path.join(output_dir, "Figure5C.pdf")
PLOT_TIFF    = os.path.join(output_dir, "Figure5C.tiff")

# ── 1. Load data ──────────────────────────────────────────────────────────────
print("Loading data...")
df = pd.read_csv(INPUT_FILE)
if "gene" not in df.columns:
    df.rename(columns={df.columns[0]: "gene"}, inplace=True)

# ── 2. Convert Ensembl → Symbol ───────────────────────────────────────────────
print("Converting ENSEMBL IDs to Gene Symbols...")
mg = mygene.MyGeneInfo()
results = mg.querymany(
    df["gene"].unique().tolist(), scopes="ensembl.gene",
    fields="symbol", species="mouse", as_dataframe=True
)
if "symbol" in results.columns:
    df["gene_symbol"] = df["gene"].map(results["symbol"].dropna().to_dict())
    df = df.dropna(subset=["gene_symbol"])
    print(f"Mapped {len(df)} genes to symbols.")
else:
    raise ValueError("ID conversion failed.")

# ── 3. Rank & save rnk ───────────────────────────────────────────────────────
print("Saving ranked file...")
df["padj"] = df["padj"].replace(0, 1e-300)
df["ranking_score"] = -np.log10(df["padj"]) * np.sign(df["log2FoldChange"])
rnk = (df.sort_values("ranking_score", ascending=False)
         [["gene_symbol", "ranking_score"]]
         .drop_duplicates(subset="gene_symbol"))
rnk.to_csv(RNK_FILE, sep="\t", index=False, header=False)

# ── 4. Run GSEA ───────────────────────────────────────────────────────────────
print("Running GSEA...")
gsea_res = gp.prerank(
    rnk=RNK_FILE,
    gene_sets="GO_Biological_Process_2023",
    outdir=GSEA_OUTDIR,
    permutation_num=1000,
    min_size=5,
    max_size=1000,
    seed=42,
    verbose=True,
)

# ── 5. Process results ────────────────────────────────────────────────────────
print("Processing GSEA results...")
res      = gsea_res.res2d
fdr_col  = next((c for c in ["FDR q-val", "fdr", "FDR"] if c in res.columns), None)
lead_col = next((c for c in ["Lead_genes", "lead_genes", "genes", "leading_edge"]
                 if c in res.columns), None)
pval_col = next((c for c in ["NOM p-val", "pvalue"] if c in res.columns), "pvalue")

res = res.rename(columns={fdr_col: "FDR", pval_col: "pvalue", "nes": "NES"})
res["Gene_Count"] = (res[lead_col].apply(
    lambda x: len(str(x).split(";")) if pd.notnull(x) else 0)
    if lead_col else 0)

res.to_csv(TABLE_OUT)
res_sig = res[res["FDR"] < 0.05].copy()
print(f"Significant pathways (FDR < 0.05): {len(res_sig)}")

# ── 6. Prepare plot data ──────────────────────────────────────────────────────
plot_df = pd.concat([
    res_sig.sort_values("NES", ascending=False).head(10),
    res_sig.sort_values("NES", ascending=True).head(10),
]).drop_duplicates(subset="Term").copy()

plot_df["Term"] = (
    plot_df["Term"]
    .str.replace("_", " ", regex=False)
    .str.split("(").str[0]
    .str.strip()
    .str.capitalize()
)
plot_df = plot_df.sort_values("NES", ascending=True)

n_terms = len(plot_df)
FIG_H   = max(100, n_terms * 8) * MM_TO_IN

# ── 7. Figure ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

colors = ["#D62728" if x > 0 else "#1F77B4" for x in plot_df["NES"]]
ax.barh(
    plot_df["Term"],
    plot_df["NES"],
    color=colors,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.4,
    height=0.65,
)

# ── Bar annotations ───────────────────────────────────────────────────────────
max_nes = plot_df["NES"].abs().max()
x_pad   = max_nes * 0.04

for i, (nes, fdr, count) in enumerate(
        zip(plot_df["NES"], plot_df["FDR"], plot_df["Gene_Count"])):
    label = f"NES={nes:.2f}, FDR={fdr:.1e}, n={count}"
    x_pos = nes + x_pad if nes > 0 else nes - x_pad
    ha    = "left"      if nes > 0 else "right"
    ax.text(x_pos, i, label, va="center", ha=ha, fontsize=5.5, color="black")

ax.axvline(0, color="black", lw=0.6, zorder=3)

nes_range = plot_df["NES"].abs().max()
# Extra left margin = enough room for annotation text on negative bars
ax.set_xlim(-(nes_range + 5), nes_range + 3.5)

ax.set_xlabel("Normalized Enrichment Score (NES)", fontsize=7, fontweight="bold")

ax.tick_params(axis="x", direction="out", length=3)
ax.tick_params(axis="y", length=0, pad=6)   # pad=6 pushes term labels left
ax.yaxis.set_tick_params(labelsize=6)

# Right-align y-tick labels so they sit flush against the plot
plt.setp(ax.get_yticklabels(), ha="right")

ax.spines[["top", "right"]].set_visible(False)
ax.spines["left"].set_visible(False)

# ── Legend ────────────────────────────────────────────────────────────────────
ax.legend(
    handles=[
        Patch(facecolor="#D62728", edgecolor="black", linewidth=0.4,
              label="Enriched in CHD8 KD"),
        Patch(facecolor="#1F77B4", edgecolor="black", linewidth=0.4,
              label="Enriched in Scrambled"),
    ],
    fontsize=6, frameon=True, framealpha=0.9,
    edgecolor="grey", loc="lower right",
)

plt.tight_layout()

# ── 8. Save PDF + TIFF and download to PC ─────────────────────────────────────
fig.savefig(PLOT_PDF, format="pdf")
print(f"✅ PDF saved: {PLOT_PDF}")

fig.savefig(
    PLOT_TIFF,
    format='tiff',
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"}
)
print(f"✅ TIFF saved: {PLOT_TIFF}")

plt.close(fig)

from google.colab import files
files.download(PLOT_PDF)
files.download(PLOT_TIFF)
print("\n✅ Downloads triggered — check your browser's Downloads folder.")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths (same as original) ──────────────────────────────────────────────────
paths = {
    'KO': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    'FL': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    'dC': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    'dH': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}
SAVE_DIR = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Output: Colab temp dir, then downloaded directly   ───────────────────
output_dir = "/content/Figure6A_output"
os.makedirs(output_dir, exist_ok=True)

FIG_WIDTH_IN  = 85 / 25.4   # 3.346 inches — half-page
FIG_HEIGHT_IN = 90 / 25.4   # slightly taller for violin+box+strip layers
DPI           = 300

# ── Global rcParams for journal compliance ────────────────────────────────────
plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})


def generate_figure_6A(synchronized_target_list):
    print(f"Processing Rescue for {len(synchronized_target_list)} NPC targets...")

    # ── Data loading & merging ────────────────────────────────────────────────
    dfs = []
    for label, path in paths.items():
        df = pd.read_csv(path, sep="\t")
        df["gene_id"] = df.iloc[:, 0].astype(str).str.split(".").str[0]
        dfs.append(df[["gene_id", "GENESYMBOL", "log2FoldChange"]]
                   .rename(columns={"log2FoldChange": f"LFC_{label}"}))

    master = dfs[0]
    for d in dfs[1:]:
        master = master.merge(d.drop(columns=['GENESYMBOL']), on="gene_id")

    # ── Synchronization & stability filter ───────────────────────────────────
    target_set = set([g.upper() for g in synchronized_target_list])
    master['gene_upper'] = master['GENESYMBOL'].astype(str).str.upper()
    targets = master[
        (master['gene_upper'].isin(target_set)) &
        (master['LFC_KO'].abs() > 0.5)
    ].copy()

    for cond in ["FL", "dC", "dH"]:
        targets[f"RF_{cond}"] = (
            (targets[f"LFC_{cond}"] - targets["LFC_KO"]) /
            (0 - targets["LFC_KO"])
        )

    # ── Statistics ───────────────────────────────────────────────────────────
    _, fried_p = friedmanchisquare(
        targets["RF_FL"], targets["RF_dC"], targets["RF_dH"])

    _, p_fl_dc_raw = wilcoxon(targets["RF_FL"], targets["RF_dC"],
                               alternative='two-sided')
    _, p_fl_dh_raw = wilcoxon(targets["RF_FL"], targets["RF_dH"],
                               alternative='two-sided')

    _, pvals_corr, _, _ = multipletests(
        [p_fl_dc_raw, p_fl_dh_raw], method='fdr_bh')
    p_fl_dc, p_fl_dh = pvals_corr

    print(f"   n={len(targets)} targets")
    print(f"   Friedman p={fried_p:.1e}")
    print(f"   FL vs ΔC (adj.)={p_fl_dc:.1e}  |  FL vs ΔH (adj.)={p_fl_dh:.1e}")
    print(f"   Medians: FL={targets['RF_FL'].median()*100:.1f}%  "
          f"dC={targets['RF_dC'].median()*100:.1f}%  "
          f"dH={targets['RF_dH'].median()*100:.1f}%")

    # ── Figure (no title — figure only) ──────────────────────────────────────
    cols_to_plot = ["RF_FL", "RF_dC", "RF_dH"]
    colors       = ["#27ae60", "#f1c40f", "#e74c3c"]

    fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))

    sns.violinplot(data=targets[cols_to_plot], palette=colors,
                   inner=None, alpha=0.15, ax=ax, linewidth=0.5)
    sns.boxplot(data=targets[cols_to_plot], palette=colors,
                showfliers=False, width=0.3, linewidth=0.8, ax=ax)
    sns.stripplot(data=targets[cols_to_plot], color='black',
                  size=0.8, alpha=0.2, jitter=0.2, ax=ax)

    # Reference lines
    ax.axhline(1, ls="--", color="#2980b9", lw=0.6, alpha=0.5,
               label="Full rescue")
    ax.axhline(0, ls="-",  color="black",   lw=0.5, alpha=0.2,
               label="No rescue")

    ax.set_ylim(-1.5, 2.5)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["Full Length", "ΔChromo", "ΔHelicase"],
                       fontsize=6, fontweight="bold")
    ax.set_ylabel("Rescue Fraction (Normalised)", fontsize=6, fontweight="bold")
    ax.tick_params(labelsize=5)

    # ── Significance brackets ─────────────────────────────────────────────────
    def sig_label(p):
        if p < 0.001: return '***'
        elif p < 0.01: return '**'
        elif p < 0.05: return '*'
        else: return 'ns'

    y_brk1, y_brk2 = 2.1, 2.3
    for x1, x2, y, p in [(0, 1, y_brk1, p_fl_dc), (0, 2, y_brk2, p_fl_dh)]:
        ax.plot([x1, x1, x2, x2], [y, y+0.04, y+0.04, y],
                color='black', lw=0.5)
        ax.text((x1+x2)/2, y+0.05, sig_label(p),
                ha='center', va='bottom', fontsize=6)

    ax.legend(frameon=False, fontsize=5, loc='upper right')
    sns.despine(offset=5, trim=True, ax=ax)
    plt.tight_layout(pad=0.3)

    # ── Save PDF + TIFF, download to PC ──────────────────────────────────────
    pdf_path  = os.path.join(output_dir, "Figure6A.pdf")
    tiff_path = os.path.join(output_dir, "Figure6A.tiff")

    fig.savefig(pdf_path, format='pdf', bbox_inches='tight', pad_inches=0.02)
    print(f"✅ PDF saved:  {pdf_path}")

    fig.savefig(
        tiff_path,
        format='tiff',
        dpi=DPI,
        bbox_inches='tight',
        pad_inches=0.02,
        pil_kwargs={"compression": "tiff_lzw"},
    )
    print(f"✅ TIFF saved: {tiff_path}")

    plt.close(fig)

    from google.colab import files
    files.download(pdf_path)
    files.download(tiff_path)
    print("\n✅ Downloads triggered — check your browser's Downloads folder.")


# ── Run ───────────────────────────────────────────────────────────────────────
generate_figure_6A(final_npc_genes)

In [ ]:
import os
import pandas as pd
import numpy as np
import pybedtools
from scipy.stats import mannwhitneyu

# ── Paths ─────────────────────────────────────────────────────────────────────
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_PATH  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

# ── Fix chromosome prefix ─────────────────────────────────────────────────────
def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ── Step 1: NPC-specific ChIP targets at TSS ─────────────────────────────────
print("1. Identifying NPC-specific CHD8 targets at TSS...")
npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()

npc_specific = npc_bt.subtract(esc_bt, A=True)
hits         = tss_bt.intersect(npc_specific, u=True, wa=True)
target_list  = list(set([
    str(f[3]).upper().strip()
    for f in hits if len(str(f[3])) > 1
]))
print(f"   NPC-specific ChIP targets identified: {len(target_list):,}")

# ── Step 2: RNA-seq filter ────────────────────────────────────────────────────
print("\n2. Loading RNA-seq data...")
de_df = pd.read_csv(RNA_PATH, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
de_df['padj'] = de_df['padj'].replace(0, 1e-300)
if 'baseMean' in de_df.columns:
    de_df = de_df[de_df['baseMean'] > 10]

de_df['gene_upper'] = de_df['GENESYMBOL'].astype(str).str.upper().str.strip()
de_df['is_target']  = de_df['gene_upper'].isin(set(target_list))

# ── Expose for Figure 6A ──────────────────────────────────────────────────────
final_npc_genes  = target_list
npc_target_genes = target_list
print(f"\n✅ final_npc_genes ready: {len(final_npc_genes):,} genes")
print("Now run Figure6A_GenomeBiology.py")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare
import pybedtools

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths (same as original) ──────────────────────────────────────────────────
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

paths = {
    'KO': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    'FL': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    'dC': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    'dH': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}

# ── Output: Colab temp dir, then downloaded directly to PC ───────────────────
output_dir = "/content/Figure6B_output"
os.makedirs(output_dir, exist_ok=True)

FIG_WIDTH_IN  = 85 / 25.4   # 3.346 inches — half-page
FIG_HEIGHT_IN = 80 / 25.4   # proportional for a 5-bar chart
DPI           = 300

# ── Global rcParams for journal compliance ────────────────────────────────────
plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size'      : 6,
    'axes.linewidth' : 0.5,
    'pdf.fonttype'   : 42,
    'ps.fonttype'    : 42,
    'figure.dpi'     : DPI,
    'savefig.dpi'    : DPI,
})

def fix_naming(feature):
    chrom = str(feature.chrom)
    if not chrom.startswith('chr'):
        feature.chrom = 'chr' + chrom
    return feature


def generate_figure_6B():
    print("Step 1: Synchronizing ChIP-seq Targets...")
    npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_naming).sort()
    esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_naming).sort()
    tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()

    npc_only   = npc_bt.subtract(esc_bt, A=True)
    hits       = tss_bt.intersect(npc_only, u=True, wa=True)
    target_set = set([str(f[3]).upper().strip()
                      for f in hits if len(str(f[3])) > 1])

    # ── Data loading ─────────────────────────────────────────────────────────
    dfs = []
    for label, path in paths.items():
        df = pd.read_csv(path, sep="\t")
        df["gene_id"] = df.iloc[:, 0].astype(str).str.split(".").str[0]
        dfs.append(df[["gene_id", "GENESYMBOL", "log2FoldChange"]]
                   .rename(columns={"log2FoldChange": f"LFC_{label}"}))

    master = dfs[0]
    for d in dfs[1:]:
        master = master.merge(d.drop(columns=['GENESYMBOL']), on="gene_id")

    # ── Filter & rescue calculation ───────────────────────────────────────────
    master['gene_upper'] = master['GENESYMBOL'].astype(str).str.upper()
    targets = master[
        (master['gene_upper'].isin(target_set)) &
        (master['LFC_KO'].abs() > 0.5)
    ].copy()
    total_n = len(targets)

    for cond in ["FL", "dC", "dH"]:
        targets[f"RF_{cond}"] = (
            (targets[f"LFC_{cond}"] - targets["LFC_KO"]) /
            (0 - targets["LFC_KO"])
        )

    targets['FL_r'] = targets['RF_FL'] >= 0.5
    targets['dC_r'] = targets['RF_dC'] >= 0.5
    targets['dH_r'] = targets['RF_dH'] >= 0.5

    # ── Categories ───────────────────────────────────────────────────────────
    c1 = ( targets['FL_r'] &  targets['dC_r'] &  targets['dH_r']).sum()
    c3 = ( targets['FL_r'] & ~targets['dC_r'] &  targets['dH_r']).sum()
    c2 = ( targets['FL_r'] &  targets['dC_r'] & ~targets['dH_r']).sum()
    c4 = ( targets['FL_r'] & ~targets['dC_r'] & ~targets['dH_r']).sum()
    c5 = (~targets['FL_r']).sum()

    categories = {
        'Global\nRescue'        : c1,
        'Chromo\nRequired'      : c3,
        'Helicase\nRequired'    : c2,
        'Dual Domain\nRequired' : c4,
        'Low/No\nRescue'        : c5,
    }

    # ── Statistics ───────────────────────────────────────────────────────────
    counts = list(categories.values())
    chi2_stat, p_cat = chisquare(counts)
    print(f"   n={total_n}  |  Chi-square={chi2_stat:.2f}, p={p_cat:.2e}")

    # ── Figure (no title — figure only) ──────────────────────────────────────
    colors = ['#27ae60', '#3498db', '#2ecc71', '#9b59b6', '#95a5a6']

    fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))

    bars = ax.bar(
        categories.keys(), counts,
        color=colors, edgecolor='black', linewidth=0.4,
    )

    # ── Bar annotations ───────────────────────────────────────────────────────
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + total_n * 0.01,
            f'{int(h)}\n({100*h/total_n:.1f}%)',
            ha='center', va='bottom',
            fontweight='bold', fontsize=5,
        )

    ax.set_ylabel(f"Number of Direct Targets (n={total_n})",
                  fontsize=6, fontweight='bold')
    ax.set_ylim(0, max(counts) * 1.25)
    ax.tick_params(axis='x', labelsize=5)
    ax.tick_params(axis='y', labelsize=5)
    sns.despine(ax=ax)
    plt.tight_layout(pad=0.3)

    # ── Save PDF + TIFF, download to PC ──────────────────────────────────────
    pdf_path  = os.path.join(output_dir, "Figure6B.pdf")
    tiff_path = os.path.join(output_dir, "Figure6B.tiff")

    fig.savefig(pdf_path, format='pdf', bbox_inches='tight', pad_inches=0.02)
    print(f"✅ PDF saved:  {pdf_path}")

    fig.savefig(
        tiff_path,
        format='tiff',
        dpi=DPI,
        bbox_inches='tight',
        pad_inches=0.02,
        pil_kwargs={"compression": "tiff_lzw"},
    )
    print(f"✅ TIFF saved: {tiff_path}")

    plt.close(fig)

    from google.colab import files
    files.download(pdf_path)
    files.download(tiff_path)
    print("\n✅ Downloads triggered — check your browser's Downloads folder.")


# ── Run ───────────────────────────────────────────────────────────────────────
generate_figure_6B()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import pybedtools
import os

# ==========================================
# 📁 1. PATHS
# ==========================================
BASE_RNA  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
BASE_ATAC = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

# RNA-seq files
RNA_PATHS = {
    "KO" : BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    "FL" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    "dC" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    "dH" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}

# ATAC-seq files (noIgG, both replicates)
ATAC_WT_PATHS = [
    BASE_ATAC + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed",
]
ATAC_KO_PATHS = [
    BASE_ATAC + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed",
]

# ==========================================
# 🔧 2. HELPER FUNCTIONS
# ==========================================
def load_rna(path, label):
    df = pd.read_csv(path, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
    df['padj'] = df['padj'].replace(0, 1e-300)
    if 'baseMean' in df.columns:
        df = df[df['baseMean'] > 10]
    df['gene_upper'] = df['GENESYMBOL'].astype(str).str.upper().str.strip()
    print(f"✅ {label}: {len(df):,} genes loaded")
    return df

def load_atac_bed(path):
    if not os.path.exists(path):
        print(f"⚠️ Not found: {path}")
        return None
    df = pd.read_csv(path, sep='\t', header=None)
    score_col = 6 if len(df.columns) > 6 else 4
    out = df[[0, 1, 2, score_col]].copy()
    out.columns = ['chr', 'start', 'end', 'score']
    out['chr'] = out['chr'].astype(str)
    mask = ~out['chr'].str.startswith('chr')
    out.loc[mask, 'chr'] = 'chr' + out.loc[mask, 'chr']
    out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
    return out

def average_atac(paths, label):
    dfs = [load_atac_bed(p) for p in paths]
    dfs = [d for d in dfs if d is not None]
    combined = pd.concat(dfs, ignore_index=True)
    combined['bin'] = (combined['chr'] + ':' +
                       ((combined['start'] // 50)*50).astype(str))
    avg = combined.groupby('bin').agg(
        chr=('chr','first'),
        start=('start','min'),
        end=('end','max'),
        score=('score','mean')
    ).reset_index()
    print(f"✅ {label}: {len(avg):,} peaks averaged")
    return avg

def fix_naming(feature):
    chrom = str(feature.chrom)
    if not chrom.startswith('chr'):
        feature.chrom = 'chr' + chrom
    return feature

# ==========================================
# 📥 3. LOAD ALL DATA
# ==========================================
print("="*60)
print("BLOCK 7: INTEGRATIVE ATAC + DOMAIN RESCUE ANALYSIS")
print("="*60)

print("\n📥 Loading RNA-seq data...")
rna = {k: load_rna(v, k) for k, v in RNA_PATHS.items()}

print("\n📥 Loading ATAC-seq data...")
wt_atac = average_atac(ATAC_WT_PATHS, "WT ATAC")
ko_atac = average_atac(ATAC_KO_PATHS, "KO ATAC")

# ==========================================
# 🔬 4. IDENTIFY KO-LOST ATAC PEAKS
# ==========================================
print("\n🔬 Identifying KO-lost peaks...")

# Merge WT and KO on 50bp bins
wt_atac['bin'] = (wt_atac['chr'] + ':' +
                   ((wt_atac['start'] // 50)*50).astype(str))
ko_atac['bin'] = (ko_atac['chr'] + ':' +
                   ((ko_atac['start'] // 50)*50).astype(str))

merged_atac = pd.merge(
    wt_atac[['bin','chr','start','end','score']].rename(
        columns={'score':'wt_score'}),
    ko_atac[['bin','score']].rename(columns={'score':'ko_score'}),
    on='bin', how='inner'
)

pseudo = 0.5
merged_atac['lfc'] = np.log2(
    (merged_atac['ko_score'] + pseudo) /
    (merged_atac['wt_score'] + pseudo)
)

# KO-lost = peaks more closed in KO (LFC < -0.5)
ko_lost = merged_atac[merged_atac['lfc'] < -0.5].copy()
ko_gained = merged_atac[merged_atac['lfc'] > 0.5].copy()

print(f"✅ Total overlapping peaks  : {len(merged_atac):,}")
print(f"✅ KO-lost peaks (LFC<-0.5) : {len(ko_lost):,}")
print(f"✅ KO-gained peaks (LFC>0.5): {len(ko_gained):,}")

# ==========================================
# 🔬 5. FIND GENES NEAR KO-LOST PEAKS
# ==========================================
print("\n🔬 Finding genes near KO-lost peaks (within TSS ±2kb)...")

# Write KO-lost peaks to temp BED
KO_LOST_BED = "/content/ko_lost_peaks.bed"
ko_lost[['chr','start','end']].to_csv(
    KO_LOST_BED, sep='\t', header=False, index=False)

# Intersect with TSS
if os.path.exists(TSS_PATH):
    tss_bt      = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()
    ko_lost_bt  = pybedtools.BedTool(KO_LOST_BED).each(fix_naming).sort()
    hits        = tss_bt.intersect(ko_lost_bt, u=True, wa=True)
    lost_genes  = set([str(f[3]).upper().strip()
                       for f in hits if len(str(f[3])) > 1])
    print(f"✅ Genes near KO-lost peaks: {len(lost_genes):,}")
else:
    # Fallback: use coordinate proximity without TSS file
    print("⚠️ TSS file not found — using all KO-lost peak-associated genes")
    lost_genes = set()

# ==========================================
# 🧮 6. CALCULATE RESCUE EFFICIENCY
# ==========================================
print("\n🧮 Calculating rescue efficiency per construct...")

ko_df = rna['KO'][['gene_upper','log2FoldChange']].rename(
    columns={'log2FoldChange':'lfc_ko'})

results = {}
for construct in ['FL','dC','dH']:
    rescue_df = rna[construct][['gene_upper','log2FoldChange']].rename(
        columns={'log2FoldChange':f'lfc_{construct}'})

    merged = ko_df.merge(rescue_df, on='gene_upper')

    # Rescue fraction: how much of KO effect is reversed
    # RF = (lfc_rescue - lfc_ko) / (0 - lfc_ko)
    # RF = 1.0 means fully rescued, RF = 0 means not rescued
    denom = 0 - merged['lfc_ko']
    denom = denom.replace(0, np.nan)
    merged['RF'] = (merged[f'lfc_{construct}'] - merged['lfc_ko']) / denom
    merged['RF'] = merged['RF'].clip(-1, 2)  # clip extreme values

    # Classify near-lost-peak genes
    merged['near_lost_peak'] = merged['gene_upper'].isin(lost_genes)

    results[construct] = merged
    n_near = merged['near_lost_peak'].sum()
    print(f"   {construct}: {len(merged):,} genes, "
          f"{n_near:,} near KO-lost peaks")

# ==========================================
# 📊 7. STATISTICAL COMPARISON
# ==========================================
print("\n📊 Statistical comparison (MWU test):")
print("-"*60)

stats_rows = []
for construct in ['FL','dC','dH']:
    df = results[construct]
    near  = df[df['near_lost_peak']]['RF'].dropna()
    other = df[~df['near_lost_peak']]['RF'].dropna()

    if len(near) > 5 and len(other) > 5:
        _, pval = mannwhitneyu(near, other, alternative='two-sided')
    else:
        pval = np.nan

    median_near  = near.median()
    median_other = other.median()

    print(f"   {construct}: near-lost median RF = {median_near:.3f} | "
          f"other median RF = {median_other:.3f} | "
          f"MWU p = {pval:.2e}")

    stats_rows.append({
        'construct'     : construct,
        'median_near'   : median_near,
        'median_other'  : median_other,
        'n_near'        : len(near),
        'n_other'       : len(other),
        'pval'          : pval
    })

stats_df = pd.DataFrame(stats_rows)
print("-"*60)
print("📝 EXPECTED: dC and dH show LOWER rescue for near-lost-peak genes")
print("   (lower RF = less rescue = more CHD8-domain dependent)")

# ==========================================
# 📊 8. FIGURE —
# ==========================================

import matplotlib
matplotlib.rcParams['font.family']     = 'Arial'
matplotlib.rcParams['pdf.fonttype']    = 42   # embeds fonts as TrueType
matplotlib.rcParams['ps.fonttype']     = 42
matplotlib.rcParams['axes.linewidth']  = 0.75  # > 0.25 pt minimum
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5

print("\n📊 Generating --compliant Figure 6c...")

# --- CVD-safe colours (blue / light-grey; avoids red–green) ---
COLOR_NEAR  = '#2166AC'   # blue  — near KO-lost peak
COLOR_OTHER = '#B2B2B2'   # grey  — other genes

# --- Canvas: 170 mm wide × 90 mm tall @ 300 DPI ---
FIG_WIDTH_IN  = 6.693   # 170 mm
FIG_HEIGHT_IN = 3.543   # 90 mm
DPI           = 300

fig, axes = plt.subplots(
    1, 3,
    figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
    dpi=DPI
)

# Sub-panel x-tick labels
group_labels = ['Near\nlost peak', 'Other\ngenes']

construct_labels = {
    'FL': 'Full-length',
    'dC': 'ΔChromo',
    'dH': 'ΔHelicase'
}
panel_letters = ['a', 'b', 'c']   # BMC: lowercase panel labels

for i, construct in enumerate(['FL', 'dC', 'dH']):
    ax  = axes[i]
    df  = results[construct]
    row = stats_df[stats_df['construct'] == construct].iloc[0]

    near_rf  = df[df['near_lost_peak']]['RF'].dropna()
    other_rf = df[~df['near_lost_peak']]['RF'].dropna()

    plot_data = pd.DataFrame({
        'RF'   : pd.concat([near_rf, other_rf]),
        'Group': (['Near\nlost peak'] * len(near_rf) +
                  ['Other\ngenes']    * len(other_rf))
    })

    # Violin plot — inner quartile lines, line width ≥ 0.5 pt
    sns.violinplot(
        data    = plot_data,
        x       = 'Group',
        y       = 'RF',
        ax      = ax,
        palette = {'Near\nlost peak': COLOR_NEAR,
                   'Other\ngenes'   : COLOR_OTHER},
        inner   = 'quartile',
        linewidth = 0.75,      # > 0.25 pt
        order   = group_labels,
        saturation = 0.85
    )

    # Reference lines (≥ 0.5 pt)
    ax.axhline(0, color='#333333', linestyle='--', lw=0.75, alpha=0.6,
               label='No rescue (RF=0)')
    ax.axhline(1, color='#555555', linestyle=':',  lw=0.75, alpha=0.6,
               label='Full rescue (RF=1)')

    # --- Axes formatting ---
    ax.set_ylim(-1.05, 2.05)
    ax.set_xlabel('')

    # Y-axis label only on leftmost panel (saves horizontal space)
    if i == 0:
        ax.set_ylabel('Rescue fraction (RF)', fontsize=7, labelpad=3)
    else:
        ax.set_ylabel('')

    ax.tick_params(axis='both', which='major',
                   labelsize=6, length=2.5, width=0.75, pad=2)
    ax.set_xticklabels(group_labels, fontsize=6)

    # Construct title (max 15 words total across figure — here per panel)
    ax.set_title(construct_labels[construct], fontsize=7,
                 fontweight='bold', pad=4)

    # Lowercase BMC panel label (top-left corner, inside axes)
    ax.text(-0.18, 1.02, panel_letters[i],
            transform=ax.transAxes,
            fontsize=8, fontweight='bold', va='top', ha='left')

    # Stats annotation — compact, 6 pt, white box key in graphic
    textstr = (
        f"n={row['n_near']} | median={row['median_near']:.2f}\n"
        f"n={row['n_other']} | median={row['median_other']:.2f}\n"
        f"MWU p={row['pval']:.1e}"
    )
    ax.text(0.97, 0.97, textstr,
            transform=ax.transAxes,
            va='top', ha='right', fontsize=5.5,
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='white', edgecolor='#CCCCCC',
                      linewidth=0.5, alpha=0.9))

    sns.despine(ax=ax, offset=3, trim=True)

# Tight layout with minimal padding (closely cropped per BMC)
plt.tight_layout(pad=0.5, w_pad=0.8, h_pad=0.5)

# --- Save as TIFF (LZW) + PDF — - accepted formats ---
PLOT_PATH_TIFF = os.path.join(SAVE_DIR, "Figure6c_ATAC_Rescue_GenomeBiology.tiff")
PLOT_PATH_PDF  = os.path.join(SAVE_DIR, "Figure6c_ATAC_Rescue_GenomeBiology.pdf")

fig.savefig(PLOT_PATH_TIFF, dpi=DPI, bbox_inches='tight',
            format='tiff',
            pil_kwargs={'compression': 'tiff_lzw'})

fig.savefig(PLOT_PATH_PDF, dpi=DPI, bbox_inches='tight',
            format='pdf',
            backend='pdf')                 # uses matplotlib's PDF backend

# File-size checks
size_tiff = os.path.getsize(PLOT_PATH_TIFF) / 1e6
size_pdf  = os.path.getsize(PLOT_PATH_PDF)  / 1e6

print(f"\n✅ TIFF saved : {PLOT_PATH_TIFF}")
print(f"   File size  : {size_tiff:.1f} MB "
      f"{'✅ within 10 MB limit' if size_tiff < 10 else '⚠️ EXCEEDS 10 MB'}")
print(f"✅ PDF saved  : {PLOT_PATH_PDF}")
print(f"   File size  : {size_pdf:.1f} MB "
      f"{'✅ within 10 MB limit' if size_pdf  < 10 else '⚠️ EXCEEDS 10 MB'}")
print("🎉 FIGURE 6c COMPLETE — - compliant")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# ──figure rcParams ──────────────────────────────────────
matplotlib.rcParams['font.family']       = 'Arial'
matplotlib.rcParams['pdf.fonttype']      = 42   # embed fonts as TrueType
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75  # > 0.25 pt minimum
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5
matplotlib.rcParams['xtick.major.size']  = 2.5
matplotlib.rcParams['ytick.major.size']  = 2.5

# ── Load and merge data ───────────────────────────────────────────────────────
sig = pd.read_csv('/content/drive/MyDrive/Chd8 data/figures/SuppTable_RescueGroups_ContinuousSignals_v2.csv')
rf  = pd.read_csv('/content/drive/MyDrive/Chd8 data/figures/Supp_Table_Domain_Rescue_Groups.csv')

sig['gene_upper'] = sig['gene'].str.upper()
rf['gene_upper']  = rf['gene_symbol'].str.upper()
df = rf.merge(sig[['gene_upper','CHD8_bound','CHD8_score','ATAC_LFC',
                    'ATAC_changed','H3K4me3_score']], on='gene_upper', how='left')

print("Merged shape:", df.shape)
print("Category counts:\n", df['rescue_group'].value_counts())

# ── Category order, CVD-safe palette (no red–green) ──────────────────────────
ORDER = ['Global Rescue', 'Helicase Required', 'Chromodomain Required',
         'Dual Domain Required', 'Low/No Rescue']
ORDER = [o for o in ORDER if (df['rescue_group'] == o).sum() >= 3]

# CVD-safe: blue / teal / purple / orange / grey  — avoids red–green
PALETTE = {
    'Global Rescue':          '#2166AC',
    'Helicase Required':      '#35978F',
    'Chromodomain Required':  '#762A83',
    'Dual Domain Required':   '#E08214',
    'Low/No Rescue':          '#B2B2B2'
}
COLORS = [PALETTE[o] for o in ORDER]

# ── Statistical helper ────────────────────────────────────────────────────────
def kw_and_pairs(data, col, order):
    groups = [data[data['rescue_group'] == g][col].dropna().values for g in order]
    stat, p = kruskal(*[g for g in groups if len(g) > 1])
    return stat, p

# ── X-tick labels: wrap long names for narrow panels ─────────────────────────
WRAP = {
    'Global Rescue':          'Global\nRescue',
    'Helicase Required':      'Helicase\nRequired',
    'Chromodomain Required':  'Chromodomain\nRequired',
    'Dual Domain Required':   'Dual Domain\nRequired',
    'Low/No Rescue':          'Low/No\nRescue'
}

FIG_WIDTH_IN  = 6.693   # 170 mm
FIG_HEIGHT_IN = 3.150   # 80 mm
DPI           = 300

fig, axes = plt.subplots(
    1, 2,
    figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
    dpi=DPI
)

# ── Panel a : CHD8 binding intensity violin ───────────────────────────────────
ax = axes[0]
df['CHD8_score_log'] = np.log10(df['CHD8_score'].clip(lower=1))
plot_data = df[df['CHD8_score_log'] > 0].copy()

order_a = [o for o in ORDER if o in plot_data['rescue_group'].unique()]

sns.violinplot(
    data      = plot_data,
    x         = 'rescue_group',
    y         = 'CHD8_score_log',
    order     = order_a,
    palette   = PALETTE,
    inner     = 'box',
    cut       = 0,
    linewidth = 0.75,
    saturation= 0.85,
    ax        = ax
)

stat_a, p_a = kw_and_pairs(plot_data, 'CHD8_score_log', order_a)

ax.set_ylabel('CHD8 ChIP-seq peak score (log₁₀)', fontsize=7, labelpad=3)
ax.set_xlabel('')

# FIX 1 — numeric positions + rotated labels with right-alignment
ax.set_xticks(range(len(order_a)))
ax.set_xticklabels(
    [WRAP[o] for o in order_a],
    fontsize  = 5.5,          # FIX 2 — reduced from 6 pt to 5.5 pt
    rotation  = 35,           # FIX 3 — angled so long labels don't collide
    ha        = 'right',
    rotation_mode = 'anchor'  # keeps label anchored to tick, not to text centre
)
ax.tick_params(axis='both', which='major', labelsize=5.5,
               length=2.5, width=0.75, pad=2)

ax.text(0.97, 0.03,
        f'Kruskal–Wallis\np = {p_a:.3f}',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=5.5,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='#CCCCCC', linewidth=0.5, alpha=0.9))

ax.text(-0.18, 1.03, 'a', transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax, offset=3, trim=True)

# ── Panel b : % CHD8-bound bar chart ─────────────────────────────────────────
ax2 = axes[1]

bound_pct = (df.groupby('rescue_group')
               .apply(lambda x: x['CHD8_bound'].sum() / len(x) * 100)
               .reindex(ORDER))

bars = ax2.bar(
    range(len(ORDER)),
    bound_pct.values,
    color     = COLORS,
    edgecolor = 'white',
    linewidth = 0.75,
    width     = 0.6
)

ax2.set_xticks(range(len(ORDER)))
ax2.set_xticklabels(
    [WRAP[o] for o in ORDER],
    fontsize  = 5.5,          # FIX 2
    rotation  = 35,           # FIX 3
    ha        = 'right',
    rotation_mode = 'anchor'
)
ax2.set_ylabel('Genes with CHD8 peak within 2 kb TSS (%)', fontsize=7, labelpad=3)
ax2.set_ylim(0, 115)
ax2.tick_params(axis='both', which='major', labelsize=5.5,
                length=2.5, width=0.75, pad=2)

for i, v in enumerate(bound_pct.values):
    if not np.isnan(v):
        ax2.text(i, v + 1.5, f'{v:.0f}%',
                 ha='center', fontsize=6, fontweight='bold')

ax2.text(-0.18, 1.03, 'b', transform=ax2.transAxes,
         fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax2, offset=3, trim=True)

# ── Layout — increase bottom margin to give rotated labels room ───────────────
plt.tight_layout(pad=0.5, w_pad=1.0, h_pad=0.5)
fig.subplots_adjust(bottom=0.22)   # FIX 4 — extra breathing room below axes

# ── Save: TIFF (LZW lossless) + PDF (vector) — no PNG ───────────────────────
SAVE_DIR = '/content/drive/MyDrive/Chd8 data/figures/'

TIFF_PATH = SAVE_DIR + 'Figure6D_CHD8_Binding_GenomeBiology.tiff'
PDF_PATH  = SAVE_DIR + 'Figure6D_CHD8_Binding_GenomeBiology.pdf'

fig.savefig(TIFF_PATH, dpi=DPI, bbox_inches='tight',
            format='tiff', pil_kwargs={'compression': 'tiff_lzw'})

fig.savefig(PDF_PATH, dpi=DPI, bbox_inches='tight',
            format='pdf', backend='pdf')

plt.show()

# ── File-size check (BMC limit: 10 MB per file) ───────────────────────────────
for path, label in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb = __import__('os').path.getsize(path) / 1e6
    status = '✅ within 10 MB' if mb < 10 else '⚠️ EXCEEDS 10 MB limit'
    print(f"✅ {label} saved : {path}")
    print(f"   File size   : {mb:.1f} MB  {status}")

print("\n🎉 FIGURE 6D COMPLETE — - compliant")

In [ ]:
import subprocess, os, shutil, matplotlib, matplotlib.font_manager as fm

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'fonts-liberation',      # Liberation Sans ≡ Arial metrics
                'fonts-dejavu-core'],    # reliable fallback
               check=True)

# ── Also try copying Arial if it exists from Colab's wine/mono paths ──────────
arial_sources = [
    '/usr/share/fonts/truetype/liberation',
    '/usr/share/fonts/truetype/dejavu',
    '/usr/share/fonts',
]

# ── Rebuild matplotlib font cache ─────────────────────────────────────────────
cache_dir = matplotlib.get_cachedir()
for f in os.listdir(cache_dir):
    if f.startswith('fontlist'):
        os.remove(os.path.join(cache_dir, f))
        print(f"🗑️  Removed stale cache: {f}")

fm._load_fontmanager(try_read_cache=False)

# ── Confirm available sans-serif fonts ────────────────────────────────────────
available = sorted(set(
    f.name for f in fm.fontManager.ttflist
    if any(k in f.name for k in ['Liberation', 'Arial', 'DejaVu', 'Helvetica'])
))
print("✅ Available fonts:", available)

# ── Set font preference: Arial → Liberation Sans → DejaVu Sans ────────────────
# matplotlib tries each in order; Liberation Sans will match in almost all cases
matplotlib.rcParams['font.family']     = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial',
                                           'Liberation Sans',
                                           'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']    = 42   # embed as TrueType
matplotlib.rcParams['ps.fonttype']     = 42
matplotlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5
matplotlib.rcParams['xtick.major.size']  = 2.5
matplotlib.rcParams['ytick.major.size']  = 2.5

# ── Quick visual check ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(3, 0.5))
ax.text(0.5, 0.5, 'Font check — Liberation Sans / Arial',
        ha='center', va='center', transform=ax.transAxes, fontsize=8)
ax.axis('off')
plt.tight_layout()
plt.show()

actual_font = fm.findfont('Liberation Sans')
print(f"✅ Active font file: {actual_font}")
print("✅ Font rcParams ready — proceed to Figure 6E cell.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ── Font setup ─────────────────────────────────────────────────────────────────
matplotlib.rcParams['font.family']       = 'sans-serif'
matplotlib.rcParams['font.sans-serif']   = ['Arial', 'Liberation Sans', 'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']      = 42
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5
matplotlib.rcParams['xtick.major.size']  = 2.5
matplotlib.rcParams['ytick.major.size']  = 2.5

# ── Load and process DESeq2 data ──────────────────────────────────────────────
rescue_dir = '/content/drive/MyDrive/Chd8 data/deseq2/deseq2/deseq2_star_trimmed/results/'

ko  = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv', sep='\t',
                  usecols=['Unnamed: 0','GENESYMBOL','log2FoldChange','baseMean']
                  ).rename(columns={'Unnamed: 0':'gene_id','log2FoldChange':'LFC_KO'})
fl  = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv', sep='\t',
                  usecols=['Unnamed: 0','log2FoldChange']
                  ).rename(columns={'Unnamed: 0':'gene_id','log2FoldChange':'LFC_FL'})
dc  = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv', sep='\t',
                  usecols=['Unnamed: 0','log2FoldChange']
                  ).rename(columns={'Unnamed: 0':'gene_id','log2FoldChange':'LFC_dC'})
dhs = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv', sep='\t',
                  usecols=['Unnamed: 0','log2FoldChange']
                  ).rename(columns={'Unnamed: 0':'gene_id','log2FoldChange':'LFC_dHs'})

df_go = (ko.merge(fl, on='gene_id')
           .merge(dc,  on='gene_id')
           .merge(dhs, on='gene_id'))
df_go = df_go[(df_go['baseMean'] > 10) & (df_go['LFC_KO'].abs() > 0.5)].copy()
print(f"Genes after filter: {df_go.shape[0]}")

# ── Rescue fractions ──────────────────────────────────────────────────────────
for col, lfc_col in [('RF_FL','LFC_FL'), ('RF_dC','LFC_dC'), ('RF_dHs','LFC_dHs')]:
    denom = (0 - df_go['LFC_KO']).replace(0, np.nan)
    df_go[col] = (df_go[lfc_col] - df_go['LFC_KO']) / denom

# ── Category assignment ───────────────────────────────────────────────────────
RF_THRESH = 0.5

def assign_category(row):
    fl  = row['RF_FL']  >= RF_THRESH
    dc  = row['RF_dC']  >= RF_THRESH
    dhs = row['RF_dHs'] >= RF_THRESH
    if fl and dc and dhs:
        return 'Global Rescue'
    elif fl and dc and not dhs:
        return 'Helicase Required'
    elif fl and dhs and not dc:
        return 'Chromodomain Required'
    elif fl and not dc and not dhs:
        return 'Dual Domain Required'
    else:
        return 'Low/No Rescue'

df_go['category'] = df_go.apply(assign_category, axis=1)
print("\nCategory counts:")
print(df_go['category'].value_counts())

# ── Palette and order ─────────────────────────────────────────────────────────
ORDER = ['Global Rescue','Helicase Required','Chromodomain Required',
         'Dual Domain Required','Low/No Rescue']
ORDER = [o for o in ORDER if (df_go['category'] == o).sum() >= 10]

PALETTE = {
    'Global Rescue':          '#2166ac',
    'Helicase Required':      '#4dac26',
    'Chromodomain Required':  '#d01c8b',
    'Dual Domain Required':   '#f1a340',
    'Low/No Rescue':          '#999999'
}

WRAP = {
    'Global Rescue':         'Global\nRescue',
    'Helicase Required':     'Helicase\nRequired',
    'Chromodomain Required': 'Chromodomain\nRequired',
    'Dual Domain Required':  'Dual Domain\nRequired',
    'Low/No Rescue':         'Low/No\nRescue'
}

# ── Fisher exact GO enrichment ────────────────────────────────────────────────
print("\nLoading GO library...")
go_lib = gp.get_library('GO_Biological_Process_2023', organism='Mouse')
print(f"GO library: {len(go_lib)} terms")

all_genes = set(df_go['GENESYMBOL'].str.upper().dropna())
go_results_local = {}

for cat in ORDER:
    query = set(df_go[df_go['category'] == cat]['GENESYMBOL'].str.upper().dropna())
    rows  = []
    for term, term_genes in go_lib.items():
        term_set = set(g.upper() for g in term_genes) & all_genes
        if len(term_set) < 3:
            continue
        a = len(query & term_set)
        if a == 0:
            continue
        b = len(query) - a
        c = len(term_set) - a
        d = len(all_genes) - len(query) - c
        _, p = fisher_exact([[a, b], [c, d]], alternative='greater')
        rows.append({
            'Term':      term,
            'n_overlap': a,
            'n_term':    len(term_set),
            'p_value':   p,
            'genes':     ';'.join(sorted(query & term_set))
        })
    if rows:
        res = pd.DataFrame(rows)
        res['padj'] = multipletests(res['p_value'], method='fdr_bh')[1]
        res = res.sort_values('padj')
        go_results_local[cat] = res
        sig = res[res['padj'] < 0.05]
        print(f"{cat} (n={len(query)}): {len(sig)} FDR<0.05 terms")
        if len(sig):
            print(sig[['Term','padj','n_overlap']].head(3).to_string())

# ── Output paths ──────────────────────────────────────────────────────────────
DOWNLOADS = os.path.join(os.path.expanduser('~'), 'Downloads')
os.makedirs(DOWNLOADS, exist_ok=True)
TIFF_PATH = os.path.join(DOWNLOADS, 'Figure6E_GO_GenomeBiology.tiff')
PDF_PATH  = os.path.join(DOWNLOADS, 'Figure6E_GO_GenomeBiology.pdf')


# ── wrap_term defined at module level — avoids all indentation issues ─────────
def wrap_term(t):
    t = t.split(' (GO')[0].strip()
    words = t.split()
    lines = []
    current = []
    length = 0
    for w in words:
        if length + len(w) > 28 and current:
            lines.append(' '.join(current))
            if len(lines) == 3:
                break
            current = [w]
            length = len(w) + 1
        else:
            current.append(w)
            length += len(w) + 1
    if current and len(lines) < 3:
        lines.append(' '.join(current))
    return '\n'.join(lines)


# ── Panel helper function ─────────────────────────────────────────────────────
def draw_go_panel(ax, cat, panel_letter):
    if cat not in go_results_local:
        ax.text(0.5, 0.5, f'No data\nfor {cat}',
                ha='center', va='center',
                transform=ax.transAxes, fontsize=6)
    else:
        res = go_results_local[cat]
        top = res[res['padj'] < 0.05].head(8).copy()
        if top.empty:
            ax.text(0.5, 0.5, 'No significant\nterms (FDR<0.05)',
                    ha='center', va='center',
                    transform=ax.transAxes, fontsize=6)
        else:
            top['-log10padj'] = -np.log10(top['padj'].clip(lower=1e-10))
            top['Term_short'] = top['Term'].apply(wrap_term)
            top = top.sort_values('-log10padj')

            bars = ax.barh(
                top['Term_short'],
                top['-log10padj'],
                color=PALETTE[cat],
                edgecolor='white',
                linewidth=0.75,
                alpha=0.85,
                height=0.45
            )

            ax.axvline(-np.log10(0.05), color='#333333',
                       linestyle='--', lw=0.75)

            for bar, val in zip(bars, top['-log10padj']):
                ax.text(
                    val + 0.04,
                    bar.get_y() + bar.get_height() / 2,
                    f'{val:.2f}',
                    va='center', ha='left',
                    fontsize=4.5, color='#333333'
                )

    n = (df_go['category'] == cat).sum()

    ax.set_title(
        f'{cat}\n(n={n})',
        fontsize=7, fontweight='bold',
        color=PALETTE.get(cat, '#333333'), pad=4
    )

    ax.set_xlabel(
        r'$\mathregular{-log_{10}(FDR)}$',
        fontsize=7, labelpad=3
    )

    ax.tick_params(axis='y', labelsize=5, pad=2)
    ax.tick_params(axis='x', labelsize=6, length=2.5, width=0.75, pad=2)
    ax.margins(y=0.20)

    ax.text(-0.22, 1.06, panel_letter,
            transform=ax.transAxes,
            fontsize=8, fontweight='bold', va='top', ha='left')

    sns.despine(ax=ax, offset=3, trim=True)
# ── end draw_go_panel ─────────────────────────────────────────────────────────


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 6E  —  full rebuild
# ══════════════════════════════════════════════════════════════════════════════
FIG_WIDTH_IN  = 6.693   # 170 mm
FIG_HEIGHT_IN = 4.330   # 110 mm
DPI           = 300

fig, axes = plt.subplots(
    1, 3,
    figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN),
    dpi=DPI
)

draw_go_panel(axes[0], 'Chromodomain Required', 'a')
draw_go_panel(axes[1], 'Low/No Rescue',         'b')

# ── Panel c ───────────────────────────────────────────────────────────────────
ax = axes[2]
cat_counts = df_go['category'].value_counts().reindex(ORDER)
colors_bar = [PALETTE[o] for o in ORDER]

ax.bar(range(len(ORDER)), cat_counts.values,
       color=colors_bar, edgecolor='white', linewidth=0.75, width=0.6)

# Place x-axis labels using figure transform — immune to data scaling
ax.set_xticks(range(len(ORDER)))
ax.set_xticklabels([])
ax.tick_params(axis='x', length=0)          # hide tick marks too

for i, o in enumerate(ORDER):
    x_axes = (i - ax.get_xlim()[0]) / (ax.get_xlim()[1] - ax.get_xlim()[0])
    ax.text(
        x_axes, -0.03,
        o,                      # single-line full name, no \n wrapping needed
        ha='right', va='top',
        fontsize=5.0,
        color=PALETTE[o],
        fontweight='bold',
        rotation=45,
        rotation_mode='anchor',
        transform=ax.transAxes
    )

ax.set_ylabel('Number of genes', fontsize=7, labelpad=3)

ax.set_title('')
ax.text(0.5, 1.01,
        'Category sizes\n(KO-dysregulated genes)',
        transform=ax.transAxes,
        fontsize=6, fontweight='bold',
        ha='center', va='bottom')

ax.tick_params(axis='both', which='major',
               labelsize=5.5, length=2.5, width=0.75, pad=2)
ax.margins(y=0.12)

for i, v in enumerate(cat_counts.values):
    if not np.isnan(v):
        ax.text(i, v + 8, str(int(v)),
                ha='center', fontsize=5.5, fontweight='bold')

ax.text(-0.15, 1.18, 'c',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax, offset=3, trim=True)

# ── Suptitle and layout ───────────────────────────────────────────────────────
fig.suptitle('Functional Characterisation of Domain-Requirement Categories',
             fontsize=8, fontweight='bold', y=1.05)

plt.tight_layout(pad=0.5, w_pad=1.5, h_pad=0.5)
fig.subplots_adjust(bottom=0.28, top=0.88, left=0.22)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig(TIFF_PATH, dpi=DPI, bbox_inches='tight',
            format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PDF_PATH,  dpi=DPI, bbox_inches='tight',
            format='pdf', backend='pdf')
plt.show()

for path, label in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb     = os.path.getsize(path) / 1e6
    status = '✅ within 10 MB' if mb < 10 else '⚠️ EXCEEDS 10 MB limit'
    print(f"✅ {label} : {path}  ({mb:.1f} MB  {status})")

try:
    from google.colab import files
    files.download(TIFF_PATH)
    files.download(PDF_PATH)
    print("\n📥 Downloads triggered.")
except ImportError:
    print("\n📁 Files written to ~/Downloads.")

print("\n🎉 FIGURE 6E COMPLETE — - compliant")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, pearsonr
import warnings
warnings.filterwarnings('ignore')

# ──  rcParams ────────────────────────────────────────────
matplotlib.rcParams['font.family']       = 'sans-serif'
matplotlib.rcParams['font.sans-serif']   = ['Arial', 'Liberation Sans', 'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']      = 42   # editable text in Illustrator
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5
matplotlib.rcParams['xtick.major.size']  = 2.5
matplotlib.rcParams['ytick.major.size']  = 2.5

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_ATAC = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

DOWNLOADS = os.path.join(os.path.expanduser('~'), 'Downloads')
os.makedirs(DOWNLOADS, exist_ok=True)
TIFF_PATH = os.path.join(DOWNLOADS, 'Figure7_ATAC_GenomeBiology.tiff')
PDF_PATH  = os.path.join(DOWNLOADS, 'Figure7_ATAC_GenomeBiology.pdf')

WT_PATHS = [
    BASE_ATAC + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed",
]
KO_PATHS = [
    BASE_ATAC + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed",
]

# ── Helper functions ──────────────────────────────────────────────────────────
def load_bed(path):
    if not os.path.exists(path):
        print(f"   WARNING: not found — {path}")
        return None
    df = pd.read_csv(path, sep='\t', header=None)
    score_col = 6 if len(df.columns) > 6 else 4
    out = df[[0, 1, 2, score_col]].copy()
    out.columns = ['chr', 'start', 'end', 'score']
    out['chr'] = out['chr'].astype(str)
    mask = ~out['chr'].str.startswith('chr')
    out.loc[mask, 'chr'] = 'chr' + out.loc[mask, 'chr']
    out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
    out['start'] = out['start'].astype(int)
    out['end']   = out['end'].astype(int)
    print(f"   {len(df):,} peaks — {os.path.basename(path)}")
    return out


def average_replicates(dfs, label):
    combined = pd.concat(dfs, ignore_index=True)
    combined['bin_key'] = (combined['chr'] + ':' +
                           ((combined['start'] // 50) * 50).astype(str))
    avg = combined.groupby('bin_key').agg(
        chr=('chr', 'first'), start=('start', 'min'),
        end=('end', 'max'),   score=('score', 'mean')
    ).reset_index()
    print(f"   {label}: {len(avg):,} unique regions after averaging")
    return avg


# ── Load data ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("FIGURE 7: CHROMATIN ACCESSIBILITY (ATAC-seq)")
print("=" * 60)

print("\n1. Loading WT replicates:")
wt_dfs = [d for d in [load_bed(p) for p in WT_PATHS] if d is not None]
print("\n2. Loading KO replicates:")
ko_dfs = [d for d in [load_bed(p) for p in KO_PATHS] if d is not None]

print("\n3. Averaging replicates...")
wt_avg = average_replicates(wt_dfs, "WT")
ko_avg = average_replicates(ko_dfs, "KO")

print("\n4. Matching peaks by 50 bp bins...")
wt_avg['bin_key'] = (wt_avg['chr'] + ':' +
                     ((wt_avg['start'] // 50) * 50).astype(str))
ko_avg['bin_key'] = (ko_avg['chr'] + ':' +
                     ((ko_avg['start'] // 50) * 50).astype(str))

merged = pd.merge(
    wt_avg[['bin_key', 'score']].rename(columns={'score': 'wt_score'}),
    ko_avg[['bin_key', 'score']].rename(columns={'score': 'ko_score'}),
    on='bin_key', how='inner'
)
print(f"   Overlapping peaks: {len(merged):,}")

if len(merged) < 100:
    raise ValueError("Too few overlapping peaks — check replicate file paths.")

# ── Statistics ────────────────────────────────────────────────────────────────
pseudo     = 0.5
wt_scores  = merged['wt_score'].values
ko_scores  = merged['ko_score'].values
lfc        = np.log2((ko_scores + pseudo) / (wt_scores + pseudo))

n_gained   = int((lfc > 0).sum())
n_lost     = int((lfc < 0).sum())
n_total    = n_gained + n_lost
pct_gained = 100 * n_gained / n_total
pct_lost   = 100 * n_lost   / n_total
median_lfc = float(np.median(lfc))

_, mwu_p = mannwhitneyu(ko_scores, wt_scores, alternative='two-sided')
r, _     = pearsonr(wt_scores, ko_scores)

# p-value label helper
def fmt_p(p):
    if p < 0.001:
        return f'p = {p:.2e}'
    return f'p = {p:.3f}'

print("\n" + "=" * 60)
print("AUDIT")
print(f"  Overlapping peaks : {len(merged):,}")
print(f"  Gained (LFC > 0)  : {n_gained:,}  ({pct_gained:.1f}%)")
print(f"  Lost   (LFC < 0)  : {n_lost:,}  ({pct_lost:.1f}%)")
print(f"  Median LFC        : {median_lfc:+.3f}")
print(f"  Pearson r         : {r:.3f}")
print(f"  MWU p-value       : {mwu_p:.2e}")
print("=" * 60)

FIG_W = 6.693
FIG_H = 4.330
DPI   = 300

fig, axes = plt.subplots(
    1, 3,
    figsize=(FIG_W, FIG_H),
    dpi=DPI
)

# Consistent palette — WT/KO identity colours used throughout
C_WT      = '#2166ac'   # blue  (WT)
C_KO      = '#d6604d'   # red   (KO / gained)
C_LOST    = '#4393c3'   # mid-blue (lost)
C_NEUTRAL = '#333333'
C_REF     = '#999999'

# ── Panel a — Violin / LFC distribution ───────────────────────────────────────
ax = axes[0]

parts = ax.violinplot(
    lfc,
    positions=[0],
    widths=0.6,
    showmedians=False,
    showextrema=False
)
for pc in parts['bodies']:
    pc.set_facecolor(C_WT)
    pc.set_edgecolor(C_NEUTRAL)
    pc.set_linewidth(0.75)
    pc.set_alpha(0.75)

# IQR box
q25, q50, q75 = np.percentile(lfc, [25, 50, 75])
ax.plot([0, 0], [q25, q75], color=C_NEUTRAL, lw=2.5, solid_capstyle='round', zorder=3)
ax.scatter([0], [q50], color='white', edgecolors=C_NEUTRAL,
           s=18, zorder=4, linewidths=0.75)

ax.axhline(0,          color=C_REF,     linestyle='--', lw=0.75, zorder=1)
ax.axhline(median_lfc, color=C_NEUTRAL, linestyle='-',  lw=0.75, alpha=0.8, zorder=2)

ax.set_xlim(-0.6, 0.6)
ax.set_xticks([])
ax.set_ylabel(r'$\log_2$ Fold Change (KO/WT)', fontsize=6, labelpad=3)
ax.tick_params(axis='y', labelsize=6, length=2.5, width=0.75, pad=2)

# Stat annotation inside panel
ax.text(0.97, 0.97,
        f'n = {len(merged):,}\nMedian = {median_lfc:+.3f}\n{fmt_p(mwu_p)}',
        transform=ax.transAxes, va='top', ha='right',
        fontsize=5, color=C_NEUTRAL,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor=C_REF, linewidth=0.5, alpha=0.9))

ax.set_title('Accessibility\nchange (KO vs WT)',
             fontsize=6.5, fontweight='bold', pad=4, color=C_NEUTRAL)

# Panel letter
ax.text(-0.18, 1.06, 'a',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax, offset=3, trim=True)

# ── Panel b — WT vs KO scatter ────────────────────────────────────────────────
ax = axes[1]

rng      = np.random.default_rng(42)
plot_idx = rng.choice(len(merged), min(8000, len(merged)), replace=False)

ax.scatter(
    wt_scores[plot_idx],
    ko_scores[plot_idx],
    alpha=0.15, s=1.2,
    color=C_NEUTRAL,
    rasterized=True,
    linewidths=0
)

max_val = max(wt_scores.max(), ko_scores.max())
ax.plot([0, max_val], [0, max_val],
        color=C_REF, linestyle='--', lw=0.75, label='y = x', zorder=3)

ax.text(0.05, 0.95,
        f'Pearson r = {r:.3f}',
        transform=ax.transAxes, va='top', ha='left',
        fontsize=5.5, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor=C_REF, linewidth=0.5, alpha=0.9))

ax.set_xlabel('WT ATAC-seq signal', fontsize=6, labelpad=3)
ax.set_ylabel('KO ATAC-seq signal', fontsize=6, labelpad=3)
ax.tick_params(labelsize=6, length=2.5, width=0.75, pad=2)

ax.set_title('WT vs KO signal\ncorrelation',
             fontsize=6.5, fontweight='bold', pad=4, color=C_NEUTRAL)

ax.text(-0.20, 1.06, 'b',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax, offset=3, trim=True)

# ── Panel c — Gained / Lost bar chart ────────────────────────────────────────
ax = axes[2]

labels = ['Gained\n(more open)', 'Lost\n(more closed)']
values = [n_gained, n_lost]
colors = [C_KO, C_LOST]

bars = ax.bar(
    [0, 1], values,
    color=colors,
    edgecolor='white',
    linewidth=0.75,
    width=0.5,
    alpha=0.88
)

# Count labels above bars
for bar, val, pct in zip(bars, values, [pct_gained, pct_lost]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + max(values) * 0.015,
        f'{val:,}\n({pct:.1f}%)',
        ha='center', va='bottom',
        fontsize=5, fontweight='bold', color=C_NEUTRAL
    )

ax.set_xticks([0, 1])
ax.set_xticklabels(labels, fontsize=6)
ax.set_ylabel('Number of ATAC-seq peaks', fontsize=6, labelpad=3)
ax.tick_params(axis='y', labelsize=6, length=2.5, width=0.75, pad=2)
ax.tick_params(axis='x', length=0, pad=4)
ax.margins(y=0.18)

ax.text(0.5, 0.97,
        f'Median LFC = {median_lfc:+.3f}',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=5, fontstyle='italic', color=C_NEUTRAL,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fffbe6',
                  edgecolor=C_REF, linewidth=0.5, alpha=0.9))

ax.set_title('Gained vs lost\naccessibility',
             fontsize=6.5, fontweight='bold', pad=4, color=C_NEUTRAL)

ax.text(-0.20, 1.06, 'c',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax, offset=3, trim=True)

# ── Layout and suptitle ───────────────────────────────────────────────────────
fig.suptitle(
    'CHD8-KO chromatin accessibility changes (ATAC-seq)',
    fontsize=7, fontweight='bold', y=1.03
)

plt.tight_layout(pad=0.6, w_pad=1.8, h_pad=0.5)
fig.subplots_adjust(top=0.88, left=0.10, right=0.97, bottom=0.14)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig(TIFF_PATH, dpi=DPI, bbox_inches='tight',
            format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PDF_PATH,  dpi=DPI, bbox_inches='tight',
            format='pdf', backend='pdf')
plt.show()

for path, label in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb     = os.path.getsize(path) / 1e6
    status = '✅ within 10 MB' if mb < 10 else '⚠️ EXCEEDS 10 MB limit'
    print(f"✅ {label} : {path}  ({mb:.1f} MB  {status})")

try:
    from google.colab import files
    files.download(TIFF_PATH)
    files.download(PDF_PATH)
    print("\n📥 Downloads triggered.")
except ImportError:
    print("\n📁 Files written to ~/Downloads.")

print("\n🎉 FIGURE 7 COMPLETE — - compliant")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib_venn import venn3
import pybedtools
import warnings
warnings.filterwarnings('ignore')

# ── rcParams ───────────────────────────────────────────────────
matplotlib.rcParams['font.family']       = 'sans-serif'
matplotlib.rcParams['font.sans-serif']   = ['Arial', 'Liberation Sans', 'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']      = 42
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5
matplotlib.rcParams['xtick.major.size']  = 2.5
matplotlib.rcParams['ytick.major.size']  = 2.5

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_RNA  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
BASE_ATAC = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

DOWNLOADS = os.path.join(os.path.expanduser('~'), 'Downloads')
os.makedirs(DOWNLOADS, exist_ok=True)
TIFF_PATH = os.path.join(DOWNLOADS, 'SuppFig1_Venn_GenomeBiology.tiff')
PDF_PATH  = os.path.join(DOWNLOADS, 'SuppFig1_Venn_GenomeBiology.pdf')

RNA_PATH      = BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
ATAC_WT_PATHS = [
    BASE_ATAC + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed",
]
ATAC_KO_PATHS = [
    BASE_ATAC + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed",
]

# ── Colours — consistent with main figures ────────────────────────────────────
C_RNA   = '#2166ac'   # blue
C_CHIP  = '#4dac26'   # green
C_ATAC  = '#d6604d'   # red-orange
C_TRIP  = '#f1a340'   # gold — triple overlap highlight
C_NEUT  = '#333333'
C_REF   = '#999999'

# ── Helpers ───────────────────────────────────────────────────────────────────
def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature


def load_atac_bed(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, sep='\t', header=None)
    score_col = 6 if len(df.columns) > 6 else 4
    out = df[[0, 1, 2, score_col]].copy()
    out.columns = ['chr', 'start', 'end', 'score']
    out['chr'] = out['chr'].astype(str)
    mask = ~out['chr'].str.startswith('chr')
    out.loc[mask, 'chr'] = 'chr' + out.loc[mask, 'chr']
    out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
    return out


def average_atac(paths, label):
    dfs = [load_atac_bed(p) for p in paths]
    dfs = [d for d in dfs if d is not None]
    combined = pd.concat(dfs, ignore_index=True)
    combined['bin'] = (combined['chr'] + ':' +
                       ((combined['start'] // 50) * 50).astype(str))
    avg = combined.groupby('bin').agg(
        chr=('chr', 'first'), start=('start', 'min'),
        end=('end', 'max'),   score=('score', 'mean')
    ).reset_index()
    print(f"   {label}: {len(avg):,} peaks")
    return avg


# ── Load RNA-seq ──────────────────────────────────────────────────────────────
print("=" * 60)

print("\n1. Loading RNA-seq...")
rna = pd.read_csv(RNA_PATH, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
rna['padj'] = rna['padj'].replace(0, 1e-300)
if 'baseMean' in rna.columns:
    rna = rna[rna['baseMean'] > 10]
rna['gene_upper'] = rna['GENESYMBOL'].astype(str).str.upper().str.strip()

degs      = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'].abs() > 0.5)]['gene_upper'])
degs_up   = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'] >  0.5)]['gene_upper'])
degs_down = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'] < -0.5)]['gene_upper'])
print(f"   DEGs: {len(degs):,}  (up: {len(degs_up):,}, down: {len(degs_down):,})")

# ── Load ChIP targets ─────────────────────────────────────────────────────────
print("\n2. Loading ChIP-seq targets...")
npc_bt   = pybedtools.BedTool(PEAK_PATH).each(fix_naming).sort()
esc_bt   = pybedtools.BedTool(ESC_PATH).each(fix_naming).sort()
tss_bt   = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()
npc_only = npc_bt.subtract(esc_bt, A=True)
hits     = tss_bt.intersect(npc_only, u=True, wa=True)
chip_targets = set([str(f[3]).upper().strip() for f in hits if len(str(f[3])) > 1])
print(f"   ChIP targets: {len(chip_targets):,}")

# ── Load ATAC-seq ─────────────────────────────────────────────────────────────
print("\n3. Loading ATAC-seq...")
wt_atac = average_atac(ATAC_WT_PATHS, "WT")
ko_atac = average_atac(ATAC_KO_PATHS, "KO")

wt_atac['bin'] = wt_atac['chr'] + ':' + ((wt_atac['start'] // 50) * 50).astype(str)
ko_atac['bin'] = ko_atac['chr'] + ':' + ((ko_atac['start'] // 50) * 50).astype(str)

merged_atac = pd.merge(
    wt_atac[['bin', 'chr', 'start', 'end', 'score']].rename(columns={'score': 'wt_score'}),
    ko_atac[['bin', 'score']].rename(columns={'score': 'ko_score'}),
    on='bin', how='inner'
)
merged_atac['lfc'] = np.log2(
    (merged_atac['ko_score'] + 0.5) / (merged_atac['wt_score'] + 0.5)
)

ATAC_THRESH  = 0.0
changed_atac = merged_atac[merged_atac['lfc'].abs() > ATAC_THRESH]
CHANGED_BED  = "/content/changed_atac_suppfig1.bed"
changed_atac[['chr', 'start', 'end']].to_csv(
    CHANGED_BED, sep='\t', header=False, index=False)
atac_bt_final   = pybedtools.BedTool(CHANGED_BED).each(fix_naming).sort()
atac_hits_final = tss_bt.intersect(atac_bt_final, u=True, wa=True)
atac_genes      = set([str(f[3]).upper().strip()
                       for f in atac_hits_final if len(str(f[3])) > 1])
print(f"   ATAC-changed genes: {len(atac_genes):,}")

# ── Compute overlaps ──────────────────────────────────────────────────────────
A = degs
B = chip_targets
C = atac_genes

only_A = A - B - C
only_B = B - A - C
only_C = C - A - B
A_B    = (A & B) - C
A_C    = (A & C) - B
B_C    = (B & C) - A
A_B_C  = A & B & C

print("\n" + "=" * 60)
print("OVERLAP COUNTS")
print(f"  RNA only                  : {len(only_A):,}")
print(f"  ChIP only                 : {len(only_B):,}")
print(f"  ATAC only                 : {len(only_C):,}")
print(f"  RNA + ChIP (not ATAC)     : {len(A_B):,}")
print(f"  RNA + ATAC (not ChIP)     : {len(A_C):,}")
print(f"  ChIP + ATAC (not RNA)     : {len(B_C):,}")
print(f"  Triple (RNA + ChIP + ATAC): {len(A_B_C):,}")
print("=" * 60)

FIG_W = 6.693
FIG_H = 3.700
DPI   = 300

fig, axes = plt.subplots(
    1, 2,
    figsize=(FIG_W, FIG_H),
    dpi=DPI,
    gridspec_kw={'width_ratios': [1.1, 0.9]}
)

# ── Panel a — Venn diagram ────────────────────────────────────────────────────
ax = axes[0]

subsets = (
    len(only_A),   # RNA only
    len(only_B),   # ChIP only
    len(A_B),      # RNA & ChIP
    len(only_C),   # ATAC only
    len(A_C),      # RNA & ATAC
    len(B_C),      # ChIP & ATAC
    len(A_B_C),    # triple
)

v = venn3(
    subsets=subsets,
    set_labels=('', '', ''),        # manual labels only — no auto text
    set_colors=(C_RNA, C_CHIP, C_ATAC),
    alpha=0.50,
    ax=ax
)

# Count labels inside regions
region_map = {
    '100': len(only_A),
    '010': len(only_B),
    '001': len(only_C),
    '110': len(A_B),
    '101': len(A_C),
    '011': len(B_C),
    '111': len(A_B_C),
}
for rid, val in region_map.items():
    lbl = v.get_label_by_id(rid)
    if lbl:
        lbl.set_text(f'{val:,}')
        lbl.set_fontsize(5.5)
        lbl.set_fontweight('bold')
        lbl.set_color(C_NEUT)

# Manual set labels — placed outside circles using axes coordinates
# RNA-seq: top-left
ax.text(-0.04, 0.78, 'RNA-seq\nDEGs',
        ha='center', va='center',
        fontsize=5.5, fontweight='bold', color=C_RNA,
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor=C_RNA, linewidth=0.6, alpha=0.9))

# ChIP-seq: top-right
ax.text(1.05, 0.78, 'CHD8\nChIP-seq',
        ha='center', va='center',
        fontsize=5.5, fontweight='bold', color=C_CHIP,
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor=C_CHIP, linewidth=0.6, alpha=0.9))

# ATAC-seq: bottom-centre
ax.text(0.50, -0.04, 'ATAC-seq\nChanged',
        ha='center', va='top',
        fontsize=5.5, fontweight='bold', color=C_ATAC,
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor=C_ATAC, linewidth=0.6, alpha=0.9))

ax.set_title('Multi-omics integration\n(RNA-seq | ChIP-seq | ATAC-seq)',
             fontsize=6.5, fontweight='bold', pad=6, color=C_NEUT)

ax.text(-0.08, 1.06, 'a',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

# ── Panel b — Horizontal bar summary ─────────────────────────────────────────
ax = axes[1]

bar_labels = [
    'RNA-seq DEGs',
    'CHD8 ChIP targets',
    'ATAC-seq changed',
    'RNA ∩ ChIP',
    'RNA ∩ ATAC',
    'ChIP ∩ ATAC',
    'Triple ∩',
]
bar_values = [
    len(A),
    len(B),
    len(C),
    len(A_B) + len(A_B_C),
    len(A_C) + len(A_B_C),
    len(B_C) + len(A_B_C),
    len(A_B_C),
]
bar_colors = [C_RNA, C_CHIP, C_ATAC,
              '#7b3294', '#d7191c', '#2ca25f', C_TRIP]

# Plot reversed so RNA is at top
y_pos = range(len(bar_labels))
bars = ax.barh(
    list(y_pos),
    bar_values[::-1],
    color=bar_colors[::-1],
    edgecolor='white',
    linewidth=0.5,
    height=0.6,
    alpha=0.85
)

# Count labels to the right of each bar
max_val = max(bar_values)
for bar, val in zip(bars, bar_values[::-1]):
    ax.text(
        bar.get_width() + max_val * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{val:,}',
        va='center', ha='left',
        fontsize=5, color=C_NEUT
    )

ax.set_yticks(list(y_pos))
ax.set_yticklabels(bar_labels[::-1], fontsize=5.5)
ax.set_xlabel('Number of genes', fontsize=6, labelpad=3)
ax.set_xlim(0, max_val * 1.22)
ax.tick_params(axis='x', labelsize=5.5, length=2.5, width=0.75, pad=2)
ax.tick_params(axis='y', length=0, pad=3)
ax.margins(y=0.08)

ax.set_title('Set size summary',
             fontsize=6.5, fontweight='bold', pad=6, color=C_NEUT)

ax.text(-0.30, 1.06, 'b',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax, left=True, offset=3, trim=True)
ax.spines['left'].set_visible(False)

# ── Layout ────────────────────────────────────────────────────────────────────
fig.suptitle(
    fontsize=7, fontweight='bold', y=1.04
)

plt.tight_layout(pad=0.6, w_pad=1.5)
fig.subplots_adjust(top=0.88, bottom=0.10, left=0.04, right=0.97)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig(TIFF_PATH, dpi=DPI, bbox_inches='tight',
            format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PDF_PATH,  dpi=DPI, bbox_inches='tight',
            format='pdf', backend='pdf')
plt.show()

for path, label in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb     = os.path.getsize(path) / 1e6
    status = '✅ within 10 MB' if mb < 10 else '⚠️ EXCEEDS 10 MB limit'
    print(f"✅ {label} : {path}  ({mb:.1f} MB  {status})")

# ── Reconciliation table ──────────────────────────────────────────────────────
recon = pd.DataFrame({
    'Metric': [
        'Total DEGs', 'Upregulated', 'Downregulated',
        'CHD8 ChIP targets', 'ATAC-changed genes',
        'RNA + ChIP', 'RNA + ATAC', 'ChIP + ATAC', 'Triple overlap'
    ],
    'Value': [
        len(A), len(degs_up), len(degs_down),
        len(B), len(C),
        len(A_B) + len(A_B_C),
        len(A_C) + len(A_B_C),
        len(B_C) + len(A_B_C),
        len(A_B_C)
    ]
})
TABLE_PATH = os.path.join(SAVE_DIR, "SuppFig1_overlap_counts.csv")
recon.to_csv(TABLE_PATH, index=False)
print(f"\n✅ Table saved: {TABLE_PATH}")

try:
    from google.colab import files
    files.download(TIFF_PATH)
    files.download(PDF_PATH)
    print("\n📥 Downloads triggered.")
except ImportError:
    print("\n📁 Files written to ~/Downloads.")

print("\n🎉 SUPPLEMENTARY FIGURE 1 COMPLETE — - compliant")

In [ ]:
# ── Upload mass-spec Excel file ───────────────────────────────────────────────
from google.colab import files as colab_files
print("Please upload the file: 42003_2021_1945_MOESM7_ESM.xlsx")
uploaded = colab_files.upload()

# Confirm the file landed correctly
if MS_EXCEL_PATH not in uploaded:
    # Handle if user uploaded with a slightly different name
    for fn in uploaded.keys():
        if fn.endswith('.xlsx'):
            MS_EXCEL_PATH = fn
            print(f"   Using uploaded file: {MS_EXCEL_PATH}")
            break
else:
    print(f"   File ready: {MS_EXCEL_PATH}")
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── - rcParams ───────────────────────────────────────────────────
matplotlib.rcParams['font.family']       = 'sans-serif'
matplotlib.rcParams['font.sans-serif']   = ['Arial', 'Liberation Sans', 'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']      = 42
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5
matplotlib.rcParams['xtick.major.size']  = 2.5
matplotlib.rcParams['ytick.major.size']  = 2.5

# ── Paths ─────────────────────────────────────────────────────────────────────
MY_RNA_PATH  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
MS_EXCEL_PATH = "42003_2021_1945_MOESM7_ESM.xlsx"

DOWNLOADS = os.path.join(os.path.expanduser('~'), 'Downloads')
os.makedirs(DOWNLOADS, exist_ok=True)
TIFF_PATH = os.path.join(DOWNLOADS, 'SuppFig3_MassSpec_GenomeBiology.tiff')
PDF_PATH  = os.path.join(DOWNLOADS, 'SuppFig3_MassSpec_GenomeBiology.pdf')

# ── Colours ───────────────────────────────────────────────────────────────────
C_MAIN   = '#2166ac'   # blue — CHD8/highlighted bars
C_DOWN   = '#d6604d'   # red-orange — downregulated overlap
C_NEUT   = '#333333'
C_REF    = '#999999'
C_LIGHT  = '#f7f7f7'

# ── Protein name map ──────────────────────────────────────────────────────────
NAME_MAP = {
    'CHD8':    'Chromodomain helicase\nDNA binding protein 8',
    'WDR5':    'WD repeat-containing\nprotein 5',
    'RPL18A':  '60S ribosomal\nprotein L18a',
    'USP7':    'Ubiquitin carboxyl-terminal\nhydroxylase 7',
    'RPS25':   '40S ribosomal\nprotein S25',
    'KPNA3':   'Importin subunit\nalpha-3',
    'HNRNPR':  'Heterogeneous nuclear\nribonucleoprotein R',
    'RPL13A':  '60S ribosomal\nprotein L13a',
    'RPL29':   '60S ribosomal\nprotein L29',
    'SYNCRIP': 'Heterogeneous nuclear\nribonucleoprotein Q',
}

MACHINERY = {'WDR5', 'USP7', 'ASH2L', 'PAF1', 'CTR9'}

# ── Helper: clean gene symbols ────────────────────────────────────────────────
def clean_symbol(s):
    if pd.isna(s):
        return None
    s = str(s).upper().replace('P0593_', '')
    return s.split('|')[-1].strip()


# ── Load mass-spec data ───────────────────────────────────────────────────────
print("=" * 60)
print("SUPPLEMENTARY FIGURE 3: CHD8 MASS-SPEC INTERACTORS")
print("=" * 60)

print("\n1. Loading mass-spec Excel...")
ms_df = pd.read_excel(MS_EXCEL_PATH, sheet_name=0, skiprows=2)
ms_df['SYMBOL'] = ms_df['gene_name'].apply(clean_symbol)
-_interactors = set(ms_df['SYMBOL'].dropna().unique())
print(f"   {len(-_interactors):,} CHD8 interactors loaded (Cerase 2021)")

# ── Load RNA-seq and find downregulated overlap ───────────────────────────────
rna_overlap = set()
if os.path.exists(MY_RNA_PATH):
    print("\n2. Loading RNA-seq for overlap...")
    rna_df = pd.read_csv(MY_RNA_PATH, sep='\t').dropna(
        subset=['padj', 'log2FoldChange'])
    my_down = set(
        rna_df[(rna_df['padj'] < 0.05) &
               (rna_df['log2FoldChange'] < -0.5)]['GENESYMBOL'].str.upper()
    )
    rna_overlap = my_down & -_interactors
    print(f"   Downregulated genes: {len(my_down):,}")
    print(f"   Overlap with interactors: {len(rna_overlap):,}")

    # Mechanism validation
    mech_hits = rna_overlap & MACHINERY
    if mech_hits:
        print(f"   Mechanism hits: {mech_hits}")
else:
    print("\n   WARNING: RNA-seq file not found — overlap colouring skipped.")

# ── Filter to top-10 plot set ─────────────────────────────────────────────────
print("\n3. Filtering to top-10 interactors...")
plot_df = ms_df[ms_df['SYMBOL'].isin(NAME_MAP.keys())].copy()
plot_df = plot_df.drop_duplicates(subset='SYMBOL')
plot_df = plot_df.sort_values('pearson.total', ascending=True)  # ascending for barh
plot_df['is_overlap'] = plot_df['SYMBOL'].isin(rna_overlap)
print(f"   Plotting {len(plot_df)} proteins")

# ── Figure — - supplementary spec ────────────────────────────────
# Full width = 170 mm = 6.693 in; single-panel tall enough for 10 bars
FIG_W = 6.693
FIG_H = 3.500
DPI   = 600

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), dpi=DPI)

# Bar colours: highlight RNA-overlap bars in C_DOWN, rest in C_MAIN
bar_colors = [C_DOWN if ov else C_MAIN
              for ov in plot_df['is_overlap']]

bars = ax.barh(
    plot_df['SYMBOL'],
    plot_df['pearson.total'],
    color=bar_colors,
    edgecolor='white',
    linewidth=0.5,
    height=0.6,
    alpha=0.88
)

# Value labels to the right of each bar
max_val = plot_df['pearson.total'].max()
for bar, val in zip(bars, plot_df['pearson.total']):
    ax.text(
        val + max_val * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{val:.3f}',
        va='center', ha='left',
        fontsize=5, color=C_NEUT
    )

# Protein full-name annotations inside or beside bars
for bar, (_, row) in zip(bars, plot_df.iterrows()):
    ax.text(
        max_val * 0.01,
        bar.get_y() + bar.get_height() / 2,
        NAME_MAP.get(row['SYMBOL'], ''),
        va='center', ha='left',
        fontsize=4.2, color='white',
        fontweight='bold'
    )

# Reference line at y=1.0 if scores reach it
if max_val >= 0.9:
    ax.axvline(1.0, color=C_REF, linestyle='--', lw=0.6, zorder=0)
    ax.text(1.002, len(plot_df) - 0.5, 'Max\nscore',
            fontsize=4, color=C_REF, va='top')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=C_MAIN,  edgecolor='white', label='CHD8 interactor'),
    Patch(facecolor=C_DOWN,  edgecolor='white',
          label='Interactor + RNA-seq\ndownregulated (KO)'),
]
ax.legend(
    handles=legend_elements,
    fontsize=5, loc='lower right',
    frameon=True, framealpha=0.9,
    edgecolor=C_REF, handlelength=1.0,
    handletextpad=0.4, borderpad=0.5
)

# Axes formatting
ax.set_xlabel('CHD8 interactor Pearson score', fontsize=6, labelpad=3)
ax.set_xlim(0, max_val * 1.20)
ax.tick_params(axis='x', labelsize=6, length=2.5, width=0.75, pad=2)
ax.tick_params(axis='y', labelsize=6, length=0, pad=3)
ax.margins(y=0.06)

ax.set_title(
    'Top CHD8-interacting proteins (mass spectrometry; Cerase et al. 2021)',
    fontsize=6.5, fontweight='bold', pad=5, color=C_NEUT
)

# Panel letter
ax.text(-0.08, 1.06, 'a',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax, left=True, offset=3, trim=True)
ax.spines['left'].set_visible(False)

# ── Layout ────────────────────────────────────────────────────────────────────
fig.suptitle(
    'Supplementary Figure 3: CHD8 protein–protein interactions (mass spectrometry)',
    fontsize=7, fontweight='bold', y=1.03
)

plt.tight_layout(pad=0.6)
fig.subplots_adjust(top=0.88, bottom=0.12, left=0.10, right=0.97)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig(TIFF_PATH, dpi=DPI, bbox_inches='tight',
            format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PDF_PATH,  dpi=DPI, bbox_inches='tight',
            format='pdf', backend='pdf')
plt.show()

for path, label in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb     = os.path.getsize(path) / 1e6
    status = '✅ within 10 MB' if mb < 10 else '⚠️ EXCEEDS 10 MB limit'
    print(f"✅ {label} : {path}  ({mb:.1f} MB  {status})")

# ── Save intersection list ────────────────────────────────────────────────────
if rna_overlap:
    out_df = pd.DataFrame(sorted(rna_overlap), columns=['GeneSymbol'])
    out_path = os.path.join(DOWNLOADS, 'CHD8_Functional_Intersection.csv')
    out_df.to_csv(out_path, index=False)
    print(f"\n✅ Intersection list saved: {out_path}")

try:
    from google.colab import files as colab_files
    colab_files.download(TIFF_PATH)
    colab_files.download(PDF_PATH)
    print("\n📥 Downloads triggered.")
except ImportError:
    print("\n📁 Files written to ~/Downloads.")

print("\n🎉 SUPPLEMENTARY FIGURE 3 COMPLETE — - compliant")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn2
import pybedtools
from scipy.stats import fisher_exact, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

# ── - rcParams ───────────────────────────────────────────────────
matplotlib.rcParams['font.family']       = 'sans-serif'
matplotlib.rcParams['font.sans-serif']   = ['Arial', 'Liberation Sans', 'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']      = 42
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5
matplotlib.rcParams['xtick.major.size']  = 2.5
matplotlib.rcParams['ytick.major.size']  = 2.5

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_RNA  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_KO    = BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
RNA_FL    = BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

DOWNLOADS = os.path.join(os.path.expanduser('~'), 'Downloads')
os.makedirs(DOWNLOADS, exist_ok=True)
TIFF_PATH = os.path.join(DOWNLOADS, 'SuppFig5_Rescue_GenomeBiology.tiff')
PDF_PATH  = os.path.join(DOWNLOADS, 'SuppFig5_Rescue_GenomeBiology.pdf')

# ── Colours ───────────────────────────────────────────────────────────────────
C_TARGET  = '#2166ac'   # blue  — direct CHD8 targets
C_INDIR   = '#999999'   # grey  — indirect / non-targets
C_RESCUED = '#4dac26'   # green — rescued
C_REF     = '#bbbbbb'
C_NEUT    = '#333333'

# ── Helpers ───────────────────────────────────────────────────────────────────
def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature


def load_rna(path, label):
    df = pd.read_csv(path, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
    df['padj'] = df['padj'].replace(0, 1e-300)
    if 'baseMean' in df.columns:
        df = df[df['baseMean'] > 10]
    df['gene_upper'] = df['GENESYMBOL'].astype(str).str.upper().str.strip()
    print(f"   {label}: {len(df):,} genes after filtering")
    return df


def fmt_p(p):
    if p < 0.001:
        return f'p = {p:.2e}'
    return f'p = {p:.3f}'


# ── Load RNA-seq ──────────────────────────────────────────────────────────────
print("=" * 60)
print("SUPPLEMENTARY FIGURE 5: CHD8 RESCUE VALIDATION")
print("=" * 60)

print("\n1. Loading RNA-seq...")
ko_df = load_rna(RNA_KO, 'KO')
fl_df = load_rna(RNA_FL, 'FL rescue')

# ── ChIP-seq direct targets ───────────────────────────────────────────────────
print("\n2. Computing direct CHD8 targets from ChIP-seq...")
npc_bt   = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
esc_bt   = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt   = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()
npc_only = npc_bt.subtract(esc_bt, A=True)
chip_hits = tss_bt.intersect(npc_only, u=True, wa=True)
direct_targets = set([
    str(f[3]).upper().strip()
    for f in chip_hits if len(str(f[3])) > 1
])
print(f"   Direct CHD8 targets (ChIP + TSS ±2 kb): {len(direct_targets):,}")

# ── Rescue fractions ──────────────────────────────────────────────────────────
print("\n3. Computing rescue fractions...")
merged = ko_df[['gene_upper', 'log2FoldChange']].rename(
    columns={'log2FoldChange': 'lfc_ko'}).merge(
    fl_df[['gene_upper', 'log2FoldChange']].rename(
        columns={'log2FoldChange': 'lfc_fl'}),
    on='gene_upper'
)

denom = (0 - merged['lfc_ko']).replace(0, np.nan)
merged['RF']         = ((merged['lfc_fl'] - merged['lfc_ko']) / denom).clip(-1, 2)
merged['is_target']  = merged['gene_upper'].isin(direct_targets)
merged['is_rescued'] = merged['RF'] >= 0.5

target_set   = set(merged[merged['is_target']]['gene_upper'])
rescued_set  = set(merged[merged['is_rescued']]['gene_upper'])
indirect_set = set(merged['gene_upper']) - target_set

print(f"   Merged dataset   : {len(merged):,} genes")
print(f"   Direct targets   : {len(target_set):,}")
print(f"   Rescued (RF≥0.5) : {len(rescued_set):,}")

# ── Statistics ────────────────────────────────────────────────────────────────
print("\n4. Statistical testing...")
a = len(target_set   &  rescued_set)
b = len(target_set   & (set(merged['gene_upper']) - rescued_set))
c = len(indirect_set &  rescued_set)
d = len(indirect_set & (set(merged['gene_upper']) - rescued_set))

odds_ratio, pval_fisher = fisher_exact([[a, b], [c, d]], alternative='greater')
rescue_rate_target   = a / (a + b) * 100 if (a + b) > 0 else 0
rescue_rate_indirect = c / (c + d) * 100 if (c + d) > 0 else 0

target_rf   = merged[merged['is_target']]['RF'].dropna()
indirect_rf = merged[~merged['is_target']]['RF'].dropna()
_, pval_mwu = mannwhitneyu(target_rf, indirect_rf, alternative='greater')

print(f"   Direct targets  : {a:,} rescued / {a+b:,} = {rescue_rate_target:.1f}%")
print(f"   Indirect DEGs   : {c:,} rescued / {c+d:,} = {rescue_rate_indirect:.1f}%")
print(f"   Fisher OR = {odds_ratio:.2f}, {fmt_p(pval_fisher)}")
print(f"   MWU {fmt_p(pval_mwu)}")

# ── Figure ────────────────────────────────────────────────────────────────────
# - full-width supplementary: 170 mm = 6.693 in
FIG_W = 6.693
FIG_H = 3.500
DPI   = 600

fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H), dpi=DPI)

# ── Panel a — Venn: direct targets vs rescued genes ───────────────────────────
ax = axes[0]

v = venn2(
    subsets=[target_set, rescued_set],
    set_labels=('', ''),
    set_colors=(C_TARGET, C_RESCUED),
    alpha=0.55,
    ax=ax
)

for rid, val in [
    ('10', len(target_set - rescued_set)),
    ('01', len(rescued_set - target_set)),
    ('11', len(target_set & rescued_set)),
]:
    lbl = v.get_label_by_id(rid)
    if lbl:
        lbl.set_text(f'{val:,}')
        lbl.set_fontsize(5.5)
        lbl.set_fontweight('bold')
        lbl.set_color(C_NEUT)

# Manual set labels using axes coordinates
ax.text(0.10, 0.82, 'Direct CHD8\ntargets',
        ha='center', va='center', fontsize=5.5,
        fontweight='bold', color=C_TARGET,
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor=C_TARGET, linewidth=0.6, alpha=0.9))

ax.text(0.88, 0.82, 'Rescued\ngenes (FL)',
        ha='center', va='center', fontsize=5.5,
        fontweight='bold', color=C_RESCUED,
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor=C_RESCUED, linewidth=0.6, alpha=0.9))

ax.text(0.50, 0.04,
        f'Overlap: {len(target_set & rescued_set):,} genes '
        f'({rescue_rate_target:.1f}% of targets)',
        transform=ax.transAxes, ha='center', va='bottom',
        fontsize=4.5, color=C_NEUT,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='#fffbe6',
                  edgecolor=C_REF, linewidth=0.5, alpha=0.9))

ax.set_title('Direct targets vs\nrescued genes (FL)',
             fontsize=6.5, fontweight='bold', pad=5, color=C_NEUT)
ax.text(-0.08, 1.06, 'a',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

# ── Panel b — Bar: rescue rate comparison ────────────────────────────────────
ax = axes[1]

cats   = ['Direct\ntargets', 'Indirect\nDEGs']
rates  = [rescue_rate_target, rescue_rate_indirect]
ns     = [len(target_set), len(indirect_set)]
colors = [C_TARGET, C_INDIR]

bars = ax.bar([0, 1], rates, color=colors,
              edgecolor='white', linewidth=0.5,
              width=0.5, alpha=0.88)

for bar, rate, n in zip(bars, rates, ns):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        rate + max(rates) * 0.02,
        f'{rate:.1f}%\n(n={n:,})',
        ha='center', va='bottom',
        fontsize=5, fontweight='bold', color=C_NEUT
    )

ax.axhline(50, color=C_REF, linestyle='--', lw=0.75,
           label='50% threshold', zorder=0)

ax.set_xticks([0, 1])
ax.set_xticklabels(cats, fontsize=6)
ax.set_ylabel('Rescue rate (%)', fontsize=6, labelpad=3)
ax.set_ylim(0, max(rates) * 1.40)
ax.tick_params(axis='y', labelsize=6, length=2.5, width=0.75, pad=2)
ax.tick_params(axis='x', length=0, pad=4)

ax.text(0.97, 0.97,
        f'Fisher OR = {odds_ratio:.2f}\n{fmt_p(pval_fisher)}',
        transform=ax.transAxes, va='top', ha='right',
        fontsize=5, color=C_NEUT,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor=C_REF, linewidth=0.5, alpha=0.9))

ax.legend(fontsize=5, loc='upper left', frameon=True,
          framealpha=0.9, edgecolor=C_REF)

ax.set_title('Rescue rate:\ndirect vs indirect targets',
             fontsize=6.5, fontweight='bold', pad=5, color=C_NEUT)
ax.text(-0.22, 1.06, 'b',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax, offset=3, trim=True)

# ── Panel c — Violin: RF distribution ────────────────────────────────────────
ax = axes[2]

plot_data = pd.DataFrame({
    'RF':    pd.concat([target_rf, indirect_rf], ignore_index=True),
    'Group': (['Direct target'] * len(target_rf) +
              ['Non-target']    * len(indirect_rf))
})

parts = ax.violinplot(
    [target_rf.values, indirect_rf.values],
    positions=[0, 1],
    widths=0.55,
    showmedians=False,
    showextrema=False
)
for pc, col in zip(parts['bodies'], [C_TARGET, C_INDIR]):
    pc.set_facecolor(col)
    pc.set_edgecolor(C_NEUT)
    pc.set_linewidth(0.6)
    pc.set_alpha(0.75)

# IQR boxes
for i, rf_vals in enumerate([target_rf, indirect_rf]):
    q25, q50, q75 = np.percentile(rf_vals, [25, 50, 75])
    ax.plot([i, i], [q25, q75], color=C_NEUT, lw=2.0,
            solid_capstyle='round', zorder=3)
    ax.scatter([i], [q50], color='white', edgecolors=C_NEUT,
               s=14, zorder=4, linewidths=0.6)

# Reference lines
for y, col, lbl in [(0.0, C_REF,     'RF = 0'),
                    (0.5, C_RESCUED,  'RF = 0.5 (rescued)'),
                    (1.0, C_TARGET,   'RF = 1.0 (full rescue)')]:
    ax.axhline(y, color=col, linestyle='--', lw=0.6,
               alpha=0.7, label=lbl, zorder=1)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Direct\ntarget', 'Non-target'], fontsize=6)
ax.set_ylabel('Rescue fraction (RF)', fontsize=6, labelpad=3)
ax.set_ylim(-1, 2)
ax.tick_params(axis='y', labelsize=6, length=2.5, width=0.75, pad=2)
ax.tick_params(axis='x', length=0, pad=4)

ax.text(0.97, 0.97,
        f'Direct: n={len(target_rf):,}, median={target_rf.median():.3f}\n'
        f'Non-target: n={len(indirect_rf):,}, median={indirect_rf.median():.3f}\n'
        f'MWU {fmt_p(pval_mwu)}',
        transform=ax.transAxes, va='top', ha='right',
        fontsize=4.5, color=C_NEUT,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor=C_REF, linewidth=0.5, alpha=0.9))

ax.legend(fontsize=4.5, loc='lower right', frameon=True,
          framealpha=0.9, edgecolor=C_REF,
          handlelength=1.0, handletextpad=0.4)

ax.set_title('Rescue fraction distribution\n(direct vs non-target)',
             fontsize=6.5, fontweight='bold', pad=5, color=C_NEUT)
ax.text(-0.22, 1.06, 'c',
        transform=ax.transAxes,
        fontsize=8, fontweight='bold', va='top', ha='left')

sns.despine(ax=ax, offset=3, trim=True)

# ── Layout ────────────────────────────────────────────────────────────────────
fig.suptitle(
    'Supplementary Figure 5: CHD8 binding predicts FL rescue',
    fontsize=7, fontweight='bold', y=1.03
)

plt.tight_layout(pad=0.6, w_pad=1.5)
fig.subplots_adjust(top=0.88, bottom=0.12, left=0.08, right=0.97)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig(TIFF_PATH, dpi=DPI, bbox_inches='tight',
            format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PDF_PATH,  dpi=DPI, bbox_inches='tight',
            format='pdf', backend='pdf')
plt.show()

for path, label in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb     = os.path.getsize(path) / 1e6
    status = '✅ within 10 MB' if mb < 10 else '⚠️ EXCEEDS 10 MB limit'
    print(f"✅ {label} : {path}  ({mb:.1f} MB  {status})")

# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Category':        ['Direct CHD8 targets', 'Indirect DEGs',
                        'Target & rescued',    'Target & not rescued'],
    'N':               [len(target_set), len(indirect_set), a, b],
    'Rescue_rate_pct': [rescue_rate_target, rescue_rate_indirect, 100.0, 0.0]
})
TABLE_PATH = os.path.join(SAVE_DIR, 'SuppFig5_rescue_summary.csv')
summary.to_csv(TABLE_PATH, index=False)
print(f"\n✅ Table saved: {TABLE_PATH}")

try:
    from google.colab import files as colab_files
    colab_files.download(TIFF_PATH)
    colab_files.download(PDF_PATH)
    print("\n📥 Downloads triggered.")
except ImportError:
    print("\n📁 Files written to ~/Downloads.")

print("\n🎉 SUPPLEMENTARY FIGURE 5 COMPLETE — - compliant")

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pybedtools
import gseapy as gp
import warnings
warnings.filterwarnings('ignore')

# ── - rcParams ───────────────────────────────────────────────────
matplotlib.rcParams['font.family']       = 'sans-serif'
matplotlib.rcParams['font.sans-serif']   = ['Arial', 'Liberation Sans', 'DejaVu Sans']
matplotlib.rcParams['pdf.fonttype']      = 42
matplotlib.rcParams['ps.fonttype']       = 42
matplotlib.rcParams['axes.linewidth']    = 0.75
matplotlib.rcParams['xtick.major.width'] = 0.75
matplotlib.rcParams['ytick.major.width'] = 0.75
matplotlib.rcParams['xtick.minor.width'] = 0.5
matplotlib.rcParams['ytick.minor.width'] = 0.5
matplotlib.rcParams['xtick.major.size']  = 2.5
matplotlib.rcParams['ytick.major.size']  = 2.5

# ── Paths ─────────────────────────────────────────────────────────────────────
PEAK_PATH   = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH    = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH    = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
BASE_ATAC   = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
SAVE_DIR    = "/content/drive/MyDrive/Chd8 data/figures/"
MOTIF_DIR   = "/content/drive/MyDrive/CHD8_TF_Motif_Analysis/"
os.makedirs(MOTIF_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

DOWNLOADS = os.path.join(os.path.expanduser('~'), 'Downloads')
os.makedirs(DOWNLOADS, exist_ok=True)
TIFF_PATH = os.path.join(DOWNLOADS, 'SuppFig4_TF_Enrichment_GenomeBiology.tiff')
PDF_PATH  = os.path.join(DOWNLOADS, 'SuppFig4_TF_Enrichment_GenomeBiology.pdf')

ATAC_WT_PATHS = [
    BASE_ATAC + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed",
]
ATAC_KO_PATHS = [
    BASE_ATAC + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed",
]

# ── Colours ───────────────────────────────────────────────────────────────────
PANEL_COLORS = ['#2166ac', '#4dac26', '#d6604d', '#f1a340', '#74add1']
C_NEUT       = '#333333'
C_REF        = '#bbbbbb'
C_SIG_LINE   = '#d6604d'

# ── Helpers ───────────────────────────────────────────────────────────────────
def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

def load_atac_bed(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, sep='\t', header=None)
    sc = 6 if len(df.columns) > 6 else 4
    out = df[[0, 1, 2, sc]].copy()
    out.columns = ['chr', 'start', 'end', 'score']
    out['chr'] = out['chr'].astype(str)
    mask = ~out['chr'].str.startswith('chr')
    out.loc[mask, 'chr'] = 'chr' + out.loc[mask, 'chr']
    out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
    return out

def average_atac(paths, label):
    dfs = [load_atac_bed(p) for p in paths]
    dfs = [d for d in dfs if d is not None]
    combined = pd.concat(dfs, ignore_index=True)
    combined['bin'] = (combined['chr'] + ':' +
                       ((combined['start'] // 50) * 50).astype(str))
    avg = combined.groupby('bin').agg(
        chr=('chr', 'first'), start=('start', 'min'),
        end=('end', 'max'),   score=('score', 'mean')
    ).reset_index()
    print(f"   {label}: {len(avg):,} peaks averaged")
    return avg

def format_mouse_gene(g):
    g = g.strip()
    if g[0].isdigit():
        return g.lower()
    return g[0].upper() + g[1:].lower()

def enrichr_with_retry(gene_list, db, max_retries=4, backoff=15):
    for attempt in range(max_retries):
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=[db],
                organism='mouse',
                outdir=None,
                cutoff=1.0,
                verbose=False,
            )
            if enr.results is not None and not enr.results.empty:
                return enr.results.copy()
            return None
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'rate' in err_str.lower():
                wait = backoff * (2 ** attempt)
                print(f"   Rate-limited on {db}, waiting {wait}s "
                      f"(attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                print(f"   {db} failed: {e}")
                return None
    print(f"   {db} exhausted retries.")
    return None

def shorten_db(db):
    return (db
            .replace('TF_Perturbations_Followed_by_Expression', 'TF Perturbations')
            .replace('ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X', 'ENCODE+ChEA')
            .replace('ENCODE_TF_ChIP-seq_2015', 'ENCODE ChIP-seq 2015')
            .replace('TRANSFAC_and_JASPAR_PWMs', 'TRANSFAC+JASPAR')
            .replace('ChEA_2022', 'ChEA 2022'))

def clean_term_label(term):
    """Extract just the TF symbol — first space-delimited token, uppercased."""
    t = term.split('(')[0].strip()
    label = t.split(' ')[0].split('_')[0].strip()
    return label.upper()[:20]

# ── Steps 1–2: unchanged (reconstruct triple-overlap, run Enrichr) ────────────
# [keep your existing Step 1 and Step 2 code here exactly as before]

# ── Step 3: Build panel list — FIXED deduplication ───────────────────────────
PLOT_DB_ORDER = [
    'TF_Perturbations_Followed_by_Expression',
    'ChEA_2022',
    'ENCODE_TF_ChIP-seq_2015',
    'ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X',
    'TRANSFAC_and_JASPAR_PWMs',
]

panels = []
for db in PLOT_DB_ORDER:
    if db not in enrichr_results:
        continue
    res = enrichr_results[db]
    sig     = res[res['Adjusted P-value'] < 0.05].copy()
    use_fdr = not sig.empty
    if use_fdr:
        sig = sig.nlargest(min(12, len(sig)), '-log10FDR').copy()
    else:
        sig = res.nsmallest(12, 'P-value').copy()

    # Assign cleaned labels BEFORE deduplication
    sig['Label'] = sig['Term'].apply(clean_term_label)

    # ── FIX: deduplicate on cleaned label, keeping the most significant row ──
    # (multiple DB entries can collapse to the same TF symbol, e.g. CTCF from
    #  different cell lines → only the lowest p-value row is kept per label)
    sig = (sig
           .sort_values('P-value', ascending=True)   # best p first
           .drop_duplicates(subset='Label', keep='first')
           .sort_values('-log10P', ascending=True))   # restore bar order

    panels.append({
        'db_short': shorten_db(db),
        'df':       sig,
        'use_fdr':  use_fdr,
        'color':    PANEL_COLORS[len(panels) % len(PANEL_COLORS)]
    })
# ── Step 4: Figure — FIXED version ────────────────────────────────────────────
print("\n3. Generating figure...")

n_panels = max(len(panels), 1)

FIG_W = 6.693
FIG_H = max(3.0, n_panels * 1.8)
DPI   = 600

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=DPI)
fig.patch.set_facecolor('white')

if not panels:
    ax = fig.add_subplot(1, 1, 1)
    ax.text(0.5, 0.5, 'No Enrichr results available\nCheck internet connection',
            ha='center', va='center', fontsize=6,
            transform=ax.transAxes, color=C_NEUT)
    ax.set_axis_off()
else:
    gs = gridspec.GridSpec(
        n_panels, 1,
        figure=fig,
        left=0.28,
        right=0.97,
        top=0.94,
        bottom=0.04,
        hspace=0.90
    )

    panel_letters = 'abcde'

    for idx, panel in enumerate(panels):
        ax      = fig.add_subplot(gs[idx])
        df      = panel['df']
        color   = panel['color']
        use_fdr = panel['use_fdr']
        db_label = panel['db_short']

        ax.set_facecolor('white')

        bars = ax.barh(
            df['Label'],
            df['-log10P'],
            color=color,
            edgecolor='white',
            linewidth=0.4,
            height=0.55,
            alpha=0.88
        )

        x_max = df['-log10P'].max()

        # ── FIX 1: wider headroom so overlap labels never clip ────────────
        ax.set_xlim(0, x_max * 1.55)
        x_right = ax.get_xlim()[1]

        # ── FIX 2: stagger overlapping overlap labels ─────────────────────
        if 'Overlap' in df.columns:
            # Collect raw label positions first
            label_positions = []
            for bar, (_, row) in zip(bars, df.iterrows()):
                raw_x = bar.get_width() + x_max * 0.025
                raw_y = bar.get_y() + bar.get_height() / 2
                label_positions.append((raw_x, raw_y, str(row['Overlap'])))

            # Stagger: if two consecutive labels are within 0.35 bar heights,
            # nudge the upper one up and lower one down
            MIN_GAP = 0.38   # in data units (bar height ~ 0.55)
            adjusted_y = [pos[1] for pos in label_positions]
            for i in range(1, len(adjusted_y)):
                if abs(adjusted_y[i] - adjusted_y[i-1]) < MIN_GAP:
                    # push them apart symmetrically
                    mid = (adjusted_y[i] + adjusted_y[i-1]) / 2
                    adjusted_y[i-1] = mid - MIN_GAP / 2
                    adjusted_y[i]   = mid + MIN_GAP / 2

            for (raw_x, _, txt), adj_y in zip(label_positions, adjusted_y):
                if raw_x < x_right * 0.97:   # only draw if it fits
                    ax.text(
                        raw_x, adj_y, txt,
                        va='center', ha='left',
                        fontsize=4.5, color=C_NEUT,
                        clip_on=True
                    )

        ax.axvline(-np.log10(0.05), color=C_SIG_LINE,
                   linestyle='--', lw=0.75, label='p = 0.05', zorder=0)

        # ── FIX 3: "Top 12 nominal p" text above the legend, never hidden ─
        if not use_fdr:
            ax.text(0.99, 0.98,                    # top-right corner
                    'Top 12 nominal p (FDR not reached)',
                    transform=ax.transAxes, fontsize=3.5,
                    ha='right', va='top',           # anchored to top edge
                    color=C_REF, style='italic')

        ax.set_xlabel(r'$-\log_{10}(p)$', fontsize=5.5, labelpad=2)
        ax.set_title(db_label, fontsize=6, fontweight='bold',
                     pad=3, color=color, loc='left')
        ax.tick_params(axis='y', labelsize=5, length=0, pad=4)
        ax.tick_params(axis='x', labelsize=5, length=2.5,
                       width=0.75, pad=2)
        ax.margins(y=0.10)

        # legend stays bottom-right (now unobstructed by the note above it)
        ax.legend(fontsize=4, loc='lower right', frameon=True,
                  framealpha=0.9, edgecolor=C_REF,
                  handlelength=0.8, handletextpad=0.3,
                  borderpad=0.3)

        # panel letter — positioned reliably above axes title
        ax.text(-0.26, 1.05, panel_letters[idx],
                transform=ax.transAxes,
                fontsize=8, fontweight='bold', va='top', ha='left')

        sns.despine(ax=ax, left=True, offset=2, trim=True)
        ax.spines['left'].set_visible(False)

fig.text(0.5, 0.995,
         'Supplementary Figure 4: TF co-binding enrichment at '
         'triple-overlap CHD8 target loci (NPC)',
         ha='center', va='top',
         fontsize=6.5, fontweight='bold', color=C_NEUT)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig(TIFF_PATH, dpi=DPI, bbox_inches='tight',
            format='tiff', facecolor='white',
            pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(PDF_PATH,  dpi=DPI, bbox_inches='tight',
            format='pdf', backend='pdf', facecolor='white')
plt.show()

for path, lbl in [(TIFF_PATH, 'TIFF'), (PDF_PATH, 'PDF')]:
    mb     = os.path.getsize(path) / 1e6
    status = '✅ within 10 MB' if mb < 10 else '⚠️ EXCEEDS 10 MB limit'
    print(f"✅ {lbl} : {path}  ({mb:.1f} MB  {status})")

for db, res in enrichr_results.items():
    out_path = os.path.join(
        MOTIF_DIR,
        f"TF_Enrichr_{db.replace('-','_').replace(' ','_')}.csv"
    )
    res.to_csv(out_path, index=False)
    print(f"✅ Table saved: {out_path}")

try:
    from google.colab import files as colab_files
    colab_files.download(TIFF_PATH)
    colab_files.download(PDF_PATH)
    print("\n📥 Downloads triggered.")
except ImportError:
    print("\n📁 Files written to ~/Downloads.")

print("\n" + "=" * 65)
print("SUPPLEMENTARY FIGURE 4 COMPLETE — - compliant")
print("=" * 65)
print(f"   Triple-overlap genes analysed : {len(triple_genes):,}")
for db, res in enrichr_results.items():
    n_fdr = (res['Adjusted P-value'] < 0.05).sum()
    n_nom = (res['P-value'] < 0.05).sum()
    print(f"   {shorten_db(db):<40} "
          f"{n_fdr} FDR<0.05 | {n_nom} nominal p<0.05")

In [ ]:
from googleapiclient.http import MediaIoBaseDownload
import io

# Use get_media instead of export_media for regular CSV files
file_id = "1U6hh9mTgPi78HtO4oDAY7GikuwUIa4H2"  # already found above

request = service.files().get_media(fileId=file_id)

fh = io.FileIO('Supp_Table_Domain_Rescue_Groups.csv', 'wb')
downloader = MediaIoBaseDownload(fh, request)
done = False
while not done:
    status, done = downloader.next_chunk()

print("Done!")

# Check immediately
import pandas as pd
df = pd.read_csv('Supp_Table_Domain_Rescue_Groups.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
for col in df.columns:
    if any(x in col.lower() for x in ['group', 'cat', 'rescue']):
        print(f"\n{col}:")
        print(df[col].value_counts())

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Supp_Table_Domain_Rescue_Groups.csv')

total = len(df)
counts = df['rescue_group'].value_counts()

# ── Rescue percentages per construct ──────────────────────────────
# RF >= 0.5 = rescued

df['rescued_FL']      = df['RF_FL']       >= 0.5
df['rescued_dChromo'] = df['RF_dChromo']  >= 0.5
df['rescued_dHelic']  = df['RF_dHelicase']>= 0.5

fl_pct      = round(df['rescued_FL'].sum()      / total * 100, 1)
dchromo_pct = round(df['rescued_dChromo'].sum() / total * 100, 1)
dhelic_pct  = round(df['rescued_dHelic'].sum()  / total * 100, 1)

print("=" * 50)
print(f"Total genes: {total}")
print("=" * 50)
print(f"Full-length rescue  : {df['rescued_FL'].sum()} / {total} = {fl_pct}%")
print(f"ΔChromo rescue      : {df['rescued_dChromo'].sum()} / {total} = {dchromo_pct}%")
print(f"ΔHelicase rescue    : {df['rescued_dHelic'].sum()} / {total} = {dhelic_pct}%")

# ── Median RF per construct ────────────────────────────────────────
print("\n── Median Rescue Fraction (RF) ──")
print(f"FL       median RF: {df['RF_FL'].median():.3f}")
print(f"ΔChromo  median RF: {df['RF_dChromo'].median():.3f}")
print(f"ΔHelicase median RF: {df['RF_dHelicase'].median():.3f}")

In [ ]:
import os
import pybedtools
import pandas as pd

# ── FIX 1: Find the correct GTF path ────────────────────────
print("Searching for GTF file...")
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if f.endswith('.gtf'):
            print(os.path.join(root, f))

# ── FIX 2: Check TSS BED column structure ───────────────────
TSS_PATH = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
tss_df = pd.read_csv(TSS_PATH, sep='\t', header=None, nrows=5)
print(f"\nTSS BED columns ({len(tss_df.columns)} total):")
print(tss_df)

# ── FIX 3: Check peak file chromosome naming ─────────────────
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
H3K4_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"

peaks_df = pd.read_csv(PEAK_PATH, sep='\t', header=None, nrows=3)
h3k4_df  = pd.read_csv(H3K4_PATH, sep='\t', header=None, nrows=3)
print(f"\nNPC peaks first 3 rows:\n{peaks_df}")
print(f"\nH3K4me3 peaks first 3 rows:\n{h3k4_df}")

In [ ]:
import pybedtools
import pandas as pd

TSS_PATH  = "/tmp/TSS_chrfix.bed"
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
H3K4_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"
GTF_PATH  = "/content/drive/MyDrive/genome/mm10/mm10.gtf"

STANDARD_CHROMS = set([f'chr{i}' for i in range(1,20)] + ['chrX','chrY'])

def load_filtered(path):
    return pybedtools.BedTool(path).filter(
        lambda f: str(f.chrom) in STANDARD_CHROMS).saveas()

npc  = load_filtered(PEAK_PATH)
h3k4 = load_filtered(H3K4_PATH)
tss  = load_filtered(TSS_PATH)

# Check what intersection result looks like
chd8_at_tss = npc.intersect(tss, u=True)
print("CHD8 peaks intersecting TSS (first 3):")
for i, f in enumerate(chd8_at_tss):
    if i >= 3: break
    print(f"  fields: {f.fields}")

# Check TSS BED after chr fix
print("\nTSS BED fixed (first 3):")
for i, f in enumerate(tss):
    if i >= 3: break
    print(f"  fields: {f.fields}")

# Check GTF format
print("\nGTF first gene entry:")
with open(GTF_PATH) as f:
    for line in f:
        if line.startswith('#'): continue
        parts = line.strip().split('\t')
        if len(parts) >= 9 and parts[2] == 'gene':
            print(f"  attributes: {parts[8][:200]}")
            break

In [ ]:
import pybedtools
import pandas as pd
import numpy as np
import os

TSS_PATH  = "/tmp/TSS_chrfix.bed"
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
H3K4_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"
GTF_PATH  = "/content/drive/MyDrive/genome/mm10/mm10.gtf"
RNA_PATH  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

STANDARD_CHROMS = set([f'chr{i}' for i in range(1,20)] + ['chrX','chrY'])

def load_filtered(path):
    return pybedtools.BedTool(path).filter(
        lambda f: str(f.chrom) in STANDARD_CHROMS).saveas()

npc  = load_filtered(PEAK_PATH)
h3k4 = load_filtered(H3K4_PATH)
tss  = load_filtered(TSS_PATH)

# ── FIX 1: Use wb=True to keep TSS gene name columns ─────────
# intersect(wb=True) keeps columns from BED B (TSS)
chd8_at_tss = npc.intersect(tss, wb=True)
h3k4_at_tss = h3k4.intersect(tss, wb=True)

print(f"CHD8 peaks at TSS (with gene names): {len(chd8_at_tss):,}")
print(f"H3K4me3 peaks at TSS               : {len(h3k4_at_tss):,}")

# Check fields
print("\nSample CHD8 at TSS fields:")
for i, f in enumerate(chd8_at_tss):
    if i >= 2: break
    print(f"  {f.fields}")

# Gene name is in col 6 (3 peak cols + 3 TSS cols, gene name = index 6)
def get_genes_wb(bt, name_col=6):
    genes = set()
    for f in bt:
        if len(f.fields) > name_col:
            genes.add(f.fields[name_col])
    return genes

chd8_tss_genes = get_genes_wb(chd8_at_tss)
h3k4_tss_genes = get_genes_wb(h3k4_at_tss)

chd8_h3k4_genes = chd8_tss_genes & h3k4_tss_genes
h3k4_only_genes = h3k4_tss_genes - chd8_tss_genes

print(f"\nCHD8+H3K4me3 unique genes : {len(chd8_h3k4_genes):,}")
print(f"H3K4me3-only genes        : {len(h3k4_only_genes):,}")
print(f"CHD8-only genes           : {len(chd8_tss_genes - h3k4_tss_genes):,}")

# ── FIX 2: Gene lengths from GTF using transcript feature ─────
print("\nExtracting gene lengths from GTF...")
gene_lengths = {}
feature_types_seen = set()

with open(GTF_PATH) as f:
    for line in f:
        if line.startswith('#'): continue
        parts = line.strip().split('\t')
        if len(parts) < 9: continue
        feature_types_seen.add(parts[2])
        if parts[2] not in ('transcript', 'mRNA'): continue
        length = int(parts[4]) - int(parts[3]) + 1
        info = parts[8]
        name = None
        for field in info.split(';'):
            field = field.strip()
            # Handle both quoted and unquoted gene_name
            if 'gene_name' in field:
                try:
                    name = field.split('"')[1]
                except:
                    name = field.split()[-1]
                break
        if name:
            if name not in gene_lengths or length > gene_lengths[name]:
                gene_lengths[name] = length

print(f"  Feature types in GTF: {feature_types_seen}")
print(f"  Gene lengths extracted: {len(gene_lengths):,}")

# If still 0, print raw GTF lines to debug
if len(gene_lengths) == 0:
    print("\n  DEBUG — first 5 non-header GTF lines:")
    with open(GTF_PATH) as f:
        count = 0
        for line in f:
            if line.startswith('#'): continue
            print(f"  {line.strip()[:200]}")
            count += 1
            if count >= 5: break

In [ ]:
import pybedtools
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

RNA_PATH  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

# ── STEP 1: Gene counts audit ─────────────────────────────────
rna_all       = pd.read_csv(RNA_PATH, sep='\t').dropna(subset=['GENESYMBOL'])
rna_expressed = rna_all[rna_all['baseMean'] > 10]

bound_all         = rna_all['GENESYMBOL'].isin(chd8_h3k4_genes).sum()
bound_expressed   = rna_expressed['GENESYMBOL'].isin(chd8_h3k4_genes).sum()
bound_unexpressed = bound_all - bound_expressed

print("── GENE COUNT AUDIT (Comment AC2) ─────────────────────")
print(f"CHD8+H3K4me3 peaks at TSS           : 4,466")
print(f"CHD8+H3K4me3 unique genes (all)     : {len(chd8_h3k4_genes):,}")
print(f"Peak-to-gene ratio                  : {4466/len(chd8_h3k4_genes):.1f}x")
print(f"")
print(f"Total genes in DESeq2               : {len(rna_all):,}")
print(f"Expressed genes (baseMean > 10)     : {len(rna_expressed):,}")
print(f"CHD8+H3K4me3 bound (all genes)      : {bound_all:,}")
print(f"CHD8+H3K4me3 bound (expressed only) : {bound_expressed:,}")
print(f"CHD8+H3K4me3 bound (unexpressed)    : {bound_unexpressed:,}")

# ── STEP 2: TPM calculation ───────────────────────────────────
gene_len_df = pd.DataFrame.from_dict(
    gene_lengths, orient='index', columns=['gene_length'])

rna_tpm = rna_all.merge(gene_len_df, left_on='GENESYMBOL',
                         right_index=True, how='left')
rna_tpm = rna_tpm.dropna(subset=['gene_length'])

rna_tpm['RPK']     = rna_tpm['baseMean'] / (rna_tpm['gene_length'] / 1000)
scaling            = rna_tpm['RPK'].sum() / 1e6
rna_tpm['TPM']     = rna_tpm['RPK'] / scaling
rna_tpm['log2TPM'] = np.log2(rna_tpm['TPM'] + 1)

# ── STEP 3: Assign groups ─────────────────────────────────────
rna_tpm['group'] = 'Unbound & Unmarked'
rna_tpm.loc[rna_tpm['GENESYMBOL'].isin(h3k4_only_genes),  'group'] = 'H3K4me3-only'
rna_tpm.loc[rna_tpm['GENESYMBOL'].isin(chd8_h3k4_genes),  'group'] = 'CHD8+H3K4me3'

print("\n── GROUP SIZES (all genes, TPM-based) ──────────────────")
print(rna_tpm['group'].value_counts())

print("\n── MEDIAN log2TPM BY GROUP ─────────────────────────────")
medians = rna_tpm.groupby('group')['log2TPM'].median().round(3)
print(medians)

# ── STEP 4: Statistics ────────────────────────────────────────
g1 = rna_tpm[rna_tpm['group']=='CHD8+H3K4me3']['log2TPM']
g2 = rna_tpm[rna_tpm['group']=='H3K4me3-only']['log2TPM']
g3 = rna_tpm[rna_tpm['group']=='Unbound & Unmarked']['log2TPM']

stat, p_kw = kruskal(g1, g2, g3)
print(f"\nKruskal-Wallis p = {p_kw:.3e}")

pairs = [('CHD8+H3K4me3',    'H3K4me3-only',      g1, g2),
         ('CHD8+H3K4me3',    'Unbound & Unmarked', g1, g3),
         ('H3K4me3-only',    'Unbound & Unmarked', g2, g3)]

pvals = [mannwhitneyu(a, b, alternative='two-sided').pvalue
         for _,_,a,b in pairs]
_, padj, _, _ = multipletests(pvals, method='fdr_bh')

print("\n── PAIRWISE MWU (FDR-adjusted) ─────────────────────────")
for (n1, n2, _, _), p, pa in zip(pairs, pvals, padj):
    print(f"  {n1} vs {n2}:")
    print(f"    raw p={p:.3e}  FDR={pa:.3e}")

print("\n✅ - UPDATE SUMMARY")
print(f"  Old: n=2,037 (expressed only), DESeq2 normalised counts")
print(f"  New: n={bound_all:,} (all genes), log2TPM")
print(f"  Old median CHD8+H3K4me3 = 2.84")
print(f"  New median CHD8+H3K4me3 = {medians.get('CHD8+H3K4me3', 'N/A')}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

SAVE_DIR = "/content/drive/MyDrive/Chd8 data/figures/"

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size'      : 12,
    'pdf.fonttype'   : 42,
})

# ── DATA ──────────────────────────────────────────────────────
group_order  = ['CHD8+H3K4me3', 'H3K4me3-only', 'Unbound & Unmarked']
group_colors = ['#2166AC', '#92C5DE', '#D1E5F0']

plot_df = rna_tpm[rna_tpm['group'].isin(group_order)].copy()
counts  = plot_df['group'].value_counts()
medians = plot_df.groupby('group')['log2TPM'].median()

# ── STATISTICS ────────────────────────────────────────────────
g1 = plot_df[plot_df['group']=='CHD8+H3K4me3']['log2TPM']
g2 = plot_df[plot_df['group']=='H3K4me3-only']['log2TPM']
g3 = plot_df[plot_df['group']=='Unbound & Unmarked']['log2TPM']

pairs = [('CHD8+H3K4me3', 'H3K4me3-only',      g1, g2, 0, 1),
         ('H3K4me3-only', 'Unbound & Unmarked', g2, g3, 1, 2),
         ('CHD8+H3K4me3', 'Unbound & Unmarked', g1, g3, 0, 2)]

pvals = [mannwhitneyu(a, b, alternative='two-sided').pvalue
         for _,_,a,b,_,_ in pairs]
_, padj, _, _ = multipletests(pvals, method='fdr_bh')

def fdr_label(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

# ── FIGURE ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

# FIX 1: use hue= to avoid FutureWarning
sns.boxplot(
    data=plot_df,
    x='group', y='log2TPM',
    hue='group',
    order=group_order,
    hue_order=group_order,
    palette=dict(zip(group_order, group_colors)),
    width=0.5,
    flierprops=dict(marker='o', markersize=2,
                    markerfacecolor='grey', alpha=0.3),
    linewidth=1.2,
    legend=False,
    ax=ax
)

# FIX 2: use set_ticks before set_ticklabels
xtick_labels = [f"{g}\n(n={counts.get(g,0):,})" for g in group_order]
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(xtick_labels, fontsize=10)

# ── MEDIAN ANNOTATIONS ────────────────────────────────────────
for i, g in enumerate(group_order):
    ax.text(i, medians[g] - 0.12,
            f'{medians[g]:.2f}',
            ha='center', va='top',
            fontsize=8.5, color='white', fontweight='bold')

# FIX 3: Stagger bracket heights so they don't overlap
# adjacent pairs low, spanning pair highest
y_base = plot_df['log2TPM'].quantile(0.995)
bracket_configs = [
    # (x1, x2, y_height, label)
    (0, 1, y_base + 0.3,  fdr_label(padj[0])),   # CHD8 vs H3K4
    (1, 2, y_base + 0.3,  fdr_label(padj[1])),   # H3K4 vs Unbound
    (0, 2, y_base + 0.85, fdr_label(padj[2])),   # CHD8 vs Unbound (highest)
]

for x1, x2, y, label in bracket_configs:
    # draw bracket
    ax.plot([x1, x1, x2, x2],
            [y, y + 0.05, y + 0.05, y],
            color='black', linewidth=1.0, clip_on=False)
    # label above bracket
    ax.text((x1 + x2) / 2, y + 0.07,
            label,
            ha='center', va='bottom',
            fontsize=11, clip_on=False)

# ── AXIS LIMITS — give enough room for brackets ───────────────
ax.set_ylim(
    plot_df['log2TPM'].min() - 0.2,
    y_base + 1.3
)

# ── LABELS ────────────────────────────────────────────────────
ax.set_xlabel('')
ax.set_ylabel('log$_2$(TPM + 1)', fontsize=12)
ax.set_title('CHD8 Binding and Gene Expression in NPCs\n'
             '(all genes, TPM-normalised)',
             fontsize=12, fontweight='bold')

ax.text(0.98, 0.02,
        'Kruskal–Wallis p = 1.41×10⁻¹¹',
        transform=ax.transAxes,
        ha='right', va='bottom',
        fontsize=9, style='italic', color='#444444')

sns.despine()
plt.tight_layout()

# ── SAVE ──────────────────────────────────────────────────────
out_png = SAVE_DIR + "Figure1C_expression_by_binding_TPM_revised.png"
out_pdf = SAVE_DIR + "Figure1C_expression_by_binding_TPM_revised.pdf"
plt.savefig(out_png, dpi=300, bbox_inches='tight')
plt.savefig(out_pdf, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved:\n  {out_png}\n  {out_pdf}")

In [ ]:
import pybedtools

H3K4_ES_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_ES_merged.bed"
STANDARD_CHROMS = set([f'chr{i}' for i in range(1,20)] + ['chrX','chrY'])

h3k4_es = pybedtools.BedTool(H3K4_ES_PATH).filter(
    lambda f: str(f.chrom) in STANDARD_CHROMS).saveas()

print(f"H3K4me3 ES peaks (std chroms): {len(h3k4_es):,}")

In [ ]:
# Genes CHD8-bound but NOT differentially expressed
# (bound but transcriptionally silent to CHD8 loss)
bound_not_deg = chip_targets - degs
print(f"CHD8-bound, transcriptionally unaffected: {len(bound_not_deg):,}")

# Of those, how many show ATAC changes?
bound_not_deg_atac = bound_not_deg & atac_genes
print(f"Of those, with ATAC changes in KO: {len(bound_not_deg_atac):,}")
print(f"Percentage: {100*len(bound_not_deg_atac)/len(bound_not_deg):.1f}%")

# Compare to background: what % of ALL bound genes show ATAC changes?
bound_with_atac = chip_targets & atac_genes
print(f"\nAll CHD8-bound genes with ATAC changes: {len(bound_with_atac):,}")
print(f"Percentage: {100*len(bound_with_atac)/len(chip_targets):.1f}%")

# Fisher's exact test: are transcriptionally unaffected bound genes
# MORE or LESS likely to show ATAC changes than DEGs?
from scipy.stats import fisher_exact
import numpy as np

# Contingency table:
#                  ATAC-changed   ATAC-unchanged
# Bound + DEG
# Bound + not-DEG
bound_deg      = chip_targets & degs
a = len(bound_deg & atac_genes)        # bound, DEG, ATAC-changed
b = len(bound_deg - atac_genes)        # bound, DEG, ATAC-unchanged
c = len(bound_not_deg & atac_genes)    # bound, not-DEG, ATAC-changed
d = len(bound_not_deg - atac_genes)    # bound, not-DEG, ATAC-unchanged

odds, pval = fisher_exact([[a, b], [c, d]])
print(f"\nFisher's exact test (DEG vs non-DEG bound genes ~ ATAC change):")
print(f"  Contingency: [[{a},{b}],[{c},{d}]]")
print(f"  Odds ratio: {odds:.3f}, p = {pval:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact, kruskal
import gseapy as gp
import time

# ── 1. CHD8 binding enrichment per rescue group ───────────────────
print("=" * 60)
print("RESCUE GROUP CHARACTERISATION")
print("=" * 60)

# Map CHD8 binding status onto rescue groups
chip_targets_title = {g.capitalize() for g in chip_targets}

rescue_df['CHD8_bound'] = rescue_df['gene'].isin(chip_targets_title)

binding_summary = rescue_df.groupby('construct_group').agg(
    total    = ('gene', 'count'),
    n_bound  = ('CHD8_bound', 'sum')
).reset_index()
binding_summary['pct_bound'] = (
    100 * binding_summary['n_bound'] / binding_summary['total']
)
print("\nCHD8 binding by rescue group:")
print(binding_summary.to_string(index=False))

# Fisher's exact for each group vs all others
print("\nFisher's exact (each group vs rest — CHD8 binding enrichment):")
for grp in rescue_df['construct_group'].unique():
    in_grp  = rescue_df[rescue_df['construct_group'] == grp]
    out_grp = rescue_df[rescue_df['construct_group'] != grp]
    a = in_grp['CHD8_bound'].sum()
    b = len(in_grp) - a
    c = out_grp['CHD8_bound'].sum()
    d = len(out_grp) - c
    odds, pval = fisher_exact([[a, b], [c, d]])
    print(f"  {grp:<30} OR={odds:.2f}, p={pval:.4f}  "
          f"({a}/{len(in_grp)} bound)")

# ── 2. ATAC accessibility change per rescue group ─────────────────
print("\nATAC accessibility changes by rescue group:")

# You need a per-gene ATAC LFC — build from merged_atac + TSS intersection
# (assumes atac_gene_lfc dict exists or can be built)
# Build gene → mean ATAC LFC mapping
atac_gene_lfc = {}
for f in tss_bt.intersect(atac_bt, wa=True, wb=True):
    gene = str(f[3]).upper().strip()
    # find matching bin in merged_atac
    # (simplified: use pre-built atac_genes set + lfc from merged_atac)
    pass

# Simpler approach: binary ATAC-changed status
atac_genes_title = {g.capitalize() for g in atac_genes}
rescue_df['ATAC_changed'] = rescue_df['gene'].isin(atac_genes_title)

atac_summary = rescue_df.groupby('construct_group').agg(
    total       = ('gene', 'count'),
    n_atac      = ('ATAC_changed', 'sum')
).reset_index()
atac_summary['pct_atac'] = (
    100 * atac_summary['n_atac'] / atac_summary['total']
)
print(atac_summary.to_string(index=False))

print("\nFisher's exact (each group vs rest — ATAC change enrichment):")
for grp in rescue_df['construct_group'].unique():
    in_grp  = rescue_df[rescue_df['construct_group'] == grp]
    out_grp = rescue_df[rescue_df['construct_group'] != grp]
    a = in_grp['ATAC_changed'].sum()
    b = len(in_grp) - a
    c = out_grp['ATAC_changed'].sum()
    d = len(out_grp) - c
    odds, pval = fisher_exact([[a, b], [c, d]])
    print(f"  {grp:<30} OR={odds:.2f}, p={pval:.4f}  "
          f"({a}/{len(in_grp)} ATAC-changed)")

# ── 3. GO enrichment for each rescue group ────────────────────────
print("\nGO enrichment per rescue group...")

def format_mouse_gene(g):
    g = g.strip()
    if g[0].isdigit():
        return g.lower()
    return g[0].upper() + g[1:].lower()

def enrichr_with_retry(gene_list, db, max_retries=3, backoff=15):
    for attempt in range(max_retries):
        try:
            enr = gp.enrichr(
                gene_list = gene_list,
                gene_sets = [db],
                organism  = 'mouse',
                outdir    = None,
                cutoff    = 1.0,
                verbose   = False,
            )
            if enr.results is not None and not enr.results.empty:
                return enr.results.copy()
            return None
        except Exception as e:
            if '429' in str(e) or 'rate' in str(e).lower():
                wait = backoff * (2 ** attempt)
                print(f"    Rate limit, waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"    Failed: {e}")
                return None
    return None

go_results = {}
for grp in rescue_df['construct_group'].unique():
    genes = rescue_df[rescue_df['construct_group'] == grp]['gene'].tolist()
    genes_fmt = [format_mouse_gene(g) for g in genes]
    print(f"\n  {grp} (n={len(genes_fmt)}):")

    if len(genes_fmt) < 5:
        print("    Too few genes for enrichment — skipping")
        continue

    time.sleep(3)
    res = enrichr_with_retry(genes_fmt, 'GO_Biological_Process_2023')
    if res is not None:
        res['-log10FDR'] = -np.log10(
            res['Adjusted P-value'].replace(0, 1e-300))
        go_results[grp] = res
        sig = res[res['Adjusted P-value'] < 0.05]
        print(f"    {len(sig)} GO BP terms FDR<0.05")
        top3 = res.nsmallest(3, 'P-value')[['Term','P-value','Overlap']]
        print(top3.to_string(index=False))
    else:
        print("    No results returned")

# ── 4. Save all results ───────────────────────────────────────────
RESCUE_DIR = "/content/drive/MyDrive/CHD8_Rescue_Groups/"
os.makedirs(RESCUE_DIR, exist_ok=True)

binding_summary.to_csv(
    os.path.join(RESCUE_DIR, "rescue_groups_CHD8_binding.csv"), index=False)
atac_summary.to_csv(
    os.path.join(RESCUE_DIR, "rescue_groups_ATAC_changes.csv"), index=False)

for grp, res in go_results.items():
    safe = grp.replace('/','_').replace(' ','_')
    res.to_csv(os.path.join(RESCUE_DIR, f"GO_BP_{safe}.csv"), index=False)

print("\nAll results saved to:", RESCUE_DIR)

In [ ]:
import pandas as pd

files_to_check = {
    'S10_categories'   : "/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S10_Rescue_Categories.csv",
    'S14_atac'         : "/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S14_ATAC_Domain_Rescue_Stats.csv",
    'S9_fractions'     : "/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S9_Rescue_Fractions.csv",
    'domain_groups'    : "/content/drive/MyDrive/Chd8 data/figures/Supp_Table_Domain_Rescue_Groups.csv",
    'rescue_summary'   : "/content/drive/MyDrive/Chd8 data/figures/Block8_rescue_summary.csv",
}

for label, path in files_to_check.items():
    print("=" * 60)
    print(f"FILE: {label}")
    print(f"PATH: {path}")
    try:
        sep = '\t' if path.endswith('.tsv') else ','
        df = pd.read_csv(path, sep=sep)
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(f"First 3 rows:")
        print(df.head(3).to_string(index=False))
    except Exception as e:
        print(f"ERROR: {e}")
    print()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
import time
import os

RESCUE_DIR = "/content/drive/MyDrive/CHD8_Rescue_Groups/"
SAVE_DIR   = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(RESCUE_DIR, exist_ok=True)

# ── LOAD DATA ─────────────────────────────────────────────────────
rescue_df = pd.read_csv(
    "/content/drive/MyDrive/Chd8 data/figures/Supp_Table_Domain_Rescue_Groups.csv"
)
rescue_df.columns = rescue_df.columns.str.strip()
rescue_df = rescue_df.rename(columns={
    'rescue_group' : 'construct_group',
    'gene_symbol'  : 'gene'
})
rescue_df['gene'] = rescue_df['gene'].astype(str).str.strip()

print(f"Rescue groups loaded: {rescue_df.shape}")
print(f"Groups: {rescue_df['construct_group'].value_counts().to_dict()}")

# ── 1. CHD8 BINDING PER RESCUE GROUP ─────────────────────────────
print("\n" + "=" * 60)
print("1. CHD8 DIRECT BINDING BY RESCUE GROUP")
print("=" * 60)

# Match capitalisation: rescue_df uses title case (e.g. Rhbdd1)
chip_targets_title = {g.capitalize() for g in chip_targets}
rescue_df['CHD8_bound'] = rescue_df['gene'].isin(chip_targets_title)

binding_summary = rescue_df.groupby('construct_group').agg(
    N         = ('gene', 'count'),
    n_bound   = ('CHD8_bound', 'sum')
).reset_index()
binding_summary['pct_bound'] = (
    100 * binding_summary['n_bound'] / binding_summary['N']
).round(1)
print(binding_summary.to_string(index=False))

print("\nFisher's exact (each group vs all others):")
binding_fisher = []
for grp in rescue_df['construct_group'].unique():
    in_g  = rescue_df[rescue_df['construct_group'] == grp]
    out_g = rescue_df[rescue_df['construct_group'] != grp]
    a = in_g['CHD8_bound'].sum();  b = len(in_g)  - a
    c = out_g['CHD8_bound'].sum(); d = len(out_g) - c
    odds, pval = fisher_exact([[a, b], [c, d]])
    binding_fisher.append({
        'Group': grp, 'n_bound': a, 'N': len(in_g),
        'pct_bound': round(100*a/len(in_g), 1),
        'OR': round(odds, 3), 'p_value': round(pval, 4)
    })
    sig = "✅ significant" if pval < 0.05 else ""
    print(f"  {grp:<40} OR={odds:.2f}, p={pval:.4f}  "
          f"({a}/{len(in_g)} bound) {sig}")

binding_fisher_df = pd.DataFrame(binding_fisher)

# ── 2. ATAC ACCESSIBILITY PER RESCUE GROUP ────────────────────────
print("\n" + "=" * 60)
print("2. ATAC ACCESSIBILITY CHANGES BY RESCUE GROUP")
print("=" * 60)

atac_genes_title = {g.capitalize() for g in atac_genes}
rescue_df['ATAC_changed'] = rescue_df['gene'].isin(atac_genes_title)

atac_summary = rescue_df.groupby('construct_group').agg(
    N           = ('gene', 'count'),
    n_atac      = ('ATAC_changed', 'sum')
).reset_index()
atac_summary['pct_atac'] = (
    100 * atac_summary['n_atac'] / atac_summary['N']
).round(1)
print(atac_summary.to_string(index=False))

print("\nFisher's exact (each group vs all others):")
atac_fisher = []
for grp in rescue_df['construct_group'].unique():
    in_g  = rescue_df[rescue_df['construct_group'] == grp]
    out_g = rescue_df[rescue_df['construct_group'] != grp]
    a = in_g['ATAC_changed'].sum();  b = len(in_g)  - a
    c = out_g['ATAC_changed'].sum(); d = len(out_g) - c
    odds, pval = fisher_exact([[a, b], [c, d]])
    atac_fisher.append({
        'Group': grp, 'n_atac': a, 'N': len(in_g),
        'pct_atac': round(100*a/len(in_g), 1),
        'OR': round(odds, 3), 'p_value': round(pval, 4)
    })
    sig = "✅ significant" if pval < 0.05 else ""
    print(f"  {grp:<40} OR={odds:.2f}, p={pval:.4f}  "
          f"({a}/{len(in_g)} ATAC-changed) {sig}")

atac_fisher_df = pd.DataFrame(atac_fisher)

# ── 3. GO ENRICHMENT PER RESCUE GROUP ────────────────────────────
print("\n" + "=" * 60)
print("3. GO BIOLOGICAL PROCESS ENRICHMENT PER RESCUE GROUP")
print("=" * 60)

def format_mouse_gene(g):
    g = g.strip()
    if g[0].isdigit():
        return g.lower()
    return g[0].upper() + g[1:].lower()

def enrichr_with_retry(gene_list, db, max_retries=4, backoff=15):
    for attempt in range(max_retries):
        try:
            enr = gp.enrichr(
                gene_list = gene_list,
                gene_sets = [db],
                organism  = 'mouse',
                outdir    = None,
                cutoff    = 1.0,
                verbose   = False,
            )
            if enr.results is not None and not enr.results.empty:
                return enr.results.copy()
            return None
        except Exception as e:
            if '429' in str(e) or 'rate' in str(e).lower():
                wait = backoff * (2 ** attempt)
                print(f"    Rate limit — waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"    Failed: {e}")
                return None
    return None

go_results  = {}
top_terms   = {}   # store top term per group for figure annotation

for grp in rescue_df['construct_group'].unique():
    genes     = rescue_df[rescue_df['construct_group'] == grp]['gene'].tolist()
    genes_fmt = [format_mouse_gene(g) for g in genes]
    print(f"\n  {grp} (n={len(genes_fmt)}):")

    if len(genes_fmt) < 5:
        print("    Too few genes — skipping")
        top_terms[grp] = 'too few genes'
        continue

    time.sleep(3)
    res = enrichr_with_retry(genes_fmt, 'GO_Biological_Process_2023')
    if res is not None and not res.empty:
        res['-log10FDR'] = -np.log10(
            res['Adjusted P-value'].replace(0, 1e-300))
        go_results[grp] = res
        sig  = res[res['Adjusted P-value'] < 0.05]
        top3 = res.nsmallest(3, 'P-value')[['Term','P-value',
                                             'Adjusted P-value','Overlap']]
        top_terms[grp] = res.iloc[0]['Term'] if len(res) > 0 else 'n/a'
        print(f"    FDR<0.05 terms: {len(sig)}")
        print(top3.to_string(index=False))
    else:
        print("    No results returned")
        top_terms[grp] = 'no results'

# ── 4. FIGURE ─────────────────────────────────────────────────────
print("\nGenerating figure...")

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size'      : 11,
    'pdf.fonttype'   : 42,
})

groups     = binding_fisher_df['Group'].tolist()
pct_bound  = binding_fisher_df['pct_bound'].tolist()
pct_atac   = atac_fisher_df.set_index('Group').loc[groups, 'pct_atac'].tolist()
n_per_grp  = binding_fisher_df['N'].tolist()

# Short labels for x-axis
label_map = {
    'Global Rescue'            : 'Global\nRescue',
    'Helicase Required'        : 'Helicase\nRequired',
    'Chromo Required'          : 'Chromo\nRequired',
    'Dual Domain Required'     : 'Dual Domain\nRequired',
    'Low/No Rescue'            : 'Low/No\nRescue',
}
xlabels = [label_map.get(g, g.replace(' ','\n')) for g in groups]
x = np.arange(len(groups))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=300)

# Panel A — CHD8 binding
ax = axes[0]
bars1 = ax.bar(x - w/2, pct_bound, w,
               color='#2C3E50', label='% CHD8-bound')
# Significance asterisks
for i, row in binding_fisher_df.iterrows():
    if row['p_value'] < 0.05:
        ax.text(i - w/2, pct_bound[i] + 1, '*',
                ha='center', fontsize=14, color='#E74C3C')
ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=9)
ax.set_ylabel('% CHD8-bound genes', fontsize=11, fontweight='bold')
ax.set_title('Direct CHD8 occupancy\nby rescue group',
             fontsize=11, fontweight='bold')
# n= annotations
for i, n in enumerate(n_per_grp):
    ax.text(i - w/2, 1, f'n={n}',
            ha='center', va='bottom', fontsize=7.5, color='white',
            fontweight='bold')
ax.set_ylim(0, max(pct_bound) * 1.25)
sns.despine(ax=ax)

# Panel B — ATAC changes
ax = axes[1]
bars2 = ax.bar(x + w/2, pct_atac, w,
               color='#1A5276', label='% ATAC-changed')
for i, row in atac_fisher_df.iterrows():
    if row['p_value'] < 0.05:
        ax.text(i + w/2, pct_atac[i] + 1, '*',
                ha='center', fontsize=14, color='#E74C3C')
ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=9)
ax.set_ylabel('% genes with ATAC changes in KO', fontsize=11, fontweight='bold')
ax.set_title('Chromatin accessibility changes\nby rescue group',
             fontsize=11, fontweight='bold')
for i, n in enumerate(n_per_grp):
    ax.text(i + w/2, 1, f'n={n}',
            ha='center', va='bottom', fontsize=7.5, color='white',
            fontweight='bold')
ax.set_ylim(0, max(pct_atac) * 1.25)
sns.despine(ax=ax)

plt.suptitle(
    "Supplementary Figure: CHD8 Occupancy and Chromatin Accessibility\n"
    "Across Domain-Dependent Rescue Groups",
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()

FIG_PATH = os.path.join(SAVE_DIR,
    "SuppFig_RescueGroups_Binding_ATAC_600DPI.png")
PDF_PATH = os.path.join(SAVE_DIR,
    "SuppFig_RescueGroups_Binding_ATAC_vector.pdf")
plt.savefig(FIG_PATH, dpi=600, bbox_inches='tight')
plt.savefig(PDF_PATH, bbox_inches='tight')
plt.show()
plt.close()
print(f"Figure saved: {FIG_PATH}")

# ── 5. SAVE TABLES ────────────────────────────────────────────────
print("\nSaving tables...")

binding_fisher_df.to_csv(
    os.path.join(RESCUE_DIR, "RescueGroup_CHD8_Binding_Fisher.csv"),
    index=False)
atac_fisher_df.to_csv(
    os.path.join(RESCUE_DIR, "RescueGroup_ATAC_Fisher.csv"),
    index=False)
for grp, res in go_results.items():
    safe = grp.replace('/','_').replace(' ','_')
    res.to_csv(
        os.path.join(RESCUE_DIR, f"GO_BP_{safe}.csv"),
        index=False)

print("All tables saved to:", RESCUE_DIR)

# ── 6. PRINT - NUMBERS ───────────────────────────────────
print("\n" + "=" * 60)
print("- NUMBERS — copy into text")
print("=" * 60)
for _, row in binding_fisher_df.iterrows():
    atac_row = atac_fisher_df[atac_fisher_df['Group']==row['Group']].iloc[0]
    print(f"\n{row['Group']}  (n={row['N']})")
    print(f"  CHD8-bound : {row['pct_bound']}%  OR={row['OR']}  p={row['p_value']}")
    print(f"  ATAC-changed: {atac_row['pct_atac']}%  OR={atac_row['OR']}  p={atac_row['p_value']}")
    print(f"  Top GO term : {top_terms.get(row['Group'], 'n/a')}")

In [ ]:
!pip install scikit-posthocs --quiet

In [ ]:
import os, glob

# Search for DESeq2 / rescue / RF related files
print("=== Searching for DESeq2 / rescue files ===")
patterns = ['**/*deseq*', '**/*DESeq*', '**/*rescue*', '**/*RF*',
            '**/*KO*', '**/*FL*', '**/*chromo*', '**/*helicase*',
            '**/*lfc*', '**/*LFC*', '**/*.csv', '**/*.tsv', '**/*.xlsx']

found = set()
search_dirs = ['/content', '/content/drive/MyDrive']

for sdir in search_dirs:
    if os.path.exists(sdir):
        for pat in patterns:
            for f in glob.glob(os.path.join(sdir, pat), recursive=True):
                if os.path.isfile(f):
                    found.add(f)

for f in sorted(found):
    size = os.path.getsize(f)
    print(f"{size:>10} bytes  {f}")

In [ ]:
import pandas as pd

# Check RF table
rf = pd.read_csv('/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S9_Rescue_Fractions.csv')
print("RF table shape:", rf.shape)
print("Columns:", rf.columns.tolist())
print(rf.head(3))S

In [ ]:
# Check if categories already computed
Scats = pd.read_csv('/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S10_Rescue_Categories.csv')
print("Categories shape:", cats.shape)
print(cats.head())
print(cats.value_counts() if cats.shape[1]==1 else cats.iloc[:,1].value_counts())

# Check continuous signals table (CHD8 binding + ATAC already per gene?)
sig = pd.read_csv('/content/drive/MyDrive/Chd8 data/figures/SuppTable_RescueGroups_ContinuousSignals_v2.csv')
print("\nContinuous signals shape:", sig.shape)
print("Columns:", sig.columns.tolist())
print(sig.head(3))

In [ ]:
import os
go_dir = '/content/drive/MyDrive/CHD8_Rescue_Groups/'
for f in os.listdir(go_dir):
    df = pd.read_csv(go_dir + f)
    print(f"\n{f}: shape={df.shape}")
    print("Columns:", df.columns.tolist())
    print(df.head(2))

In [ ]:
import pandas as pd

# RF fractions
rf = pd.read_csv('/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S9_Rescue_Fractions.csv')
print("RF table shape:", rf.shape)
print("Columns:", rf.columns.tolist())
print(rf.head(5))
print("\nValue counts of any category column:")
for col in rf.columns:
    if 'cat' in col.lower() or 'group' in col.lower() or 'class' in col.lower():
        print(rf[col].value_counts())

In [ ]:
sig = pd.read_csv('/content/drive/MyDrive/Chd8 data/figures/SuppTable_RescueGroups_ContinuousSignals_v2.csv')
print("Shape:", sig.shape)
print("Columns:", sig.columns.tolist())
print(sig.head(5))
print("\nCategory counts:")
for col in sig.columns:
    if 'cat' in col.lower() or 'group' in col.lower() or 'class' in col.lower() or 'rescue' in col.lower():
        print(f"\n{col}:")
        print(sig[col].value_counts())

In [ ]:
full = pd.read_csv('/content/drive/MyDrive/Chd8 data/figures/Supp_Table_Domain_Rescue_Groups.csv')
print("Shape:", full.shape)
print("Columns:", full.columns.tolist())
print(full.head(5))
print("\nGroup sizes:")
for col in full.columns:
    if 'group' in col.lower() or 'cat' in col.lower():
        print(full[col].value_counts())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
import gseapy as gp
import warnings
warnings.filterwarnings('ignore')

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('/content/drive/MyDrive/Chd8 data/figures/SuppTable_RescueGroups_ContinuousSignals_v2.csv')
print("Columns:", df.columns.tolist())
print(df.head(3))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
import gseapy as gp
import warnings
warnings.filterwarnings('ignore')

# ── Load both tables and merge ────────────────────────────────────────────────
sig = pd.read_csv('/content/drive/MyDrive/Chd8 data/figures/SuppTable_RescueGroups_ContinuousSignals_v2.csv')
rf  = pd.read_csv('/content/drive/MyDrive/Chd8 data/figures/Supp_Table_Domain_Rescue_Groups.csv')

# Merge on gene (use upper for matching)
sig['gene_upper'] = sig['gene'].str.upper()
rf['gene_upper']  = rf['gene_symbol'].str.upper()
df = rf.merge(sig[['gene_upper','CHD8_bound','CHD8_score','ATAC_LFC',
                    'ATAC_changed','H3K4me3_score']], on='gene_upper', how='left')

print("Merged shape:", df.shape)
print("Category counts:\n", df['rescue_group'].value_counts())
print("\nCHD8_score nulls:", df['CHD8_score'].isna().sum())
print("ATAC_LFC nulls:",   df['ATAC_LFC'].isna().sum())

# ── Settings ──────────────────────────────────────────────────────────────────
ORDER = ['Global Rescue','Helicase Required','Chromodomain Required',
         'Dual Domain Required','Low/No Rescue']
# Only keep categories with n>=5 for stats
ORDER = [o for o in ORDER if (df['rescue_group']==o).sum() >= 3]

PALETTE = {
    'Global Rescue':          '#2166ac',
    'Helicase Required':      '#4dac26',
    'Chromodomain Required':  '#d01c8b',
    'Dual Domain Required':   '#f1a340',
    'Low/No Rescue':          '#999999'
}
COLORS = [PALETTE[o] for o in ORDER]

def kw_and_pairs(data, col, order):
    """Kruskal-Wallis + pairwise Mann-Whitney with FDR"""
    from scipy.stats import false_discovery_control
    groups = [data[data['rescue_group']==g][col].dropna().values for g in order]
    stat, p = kruskal(*[g for g in groups if len(g)>1])
    pairs = list(combinations(range(len(order)), 2))
    raw_p = [mannwhitneyu(groups[i], groups[j], alternative='two-sided').pvalue
             for i,j in pairs if len(groups[i])>1 and len(groups[j])>1]
    return stat, p, raw_p

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE A — CHD8 binding intensity per category
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: CHD8 peak score (log10)
ax = axes[0]
df['CHD8_score_log'] = np.log10(df['CHD8_score'].clip(lower=1))
plot_data = df[df['CHD8_score_log'] > 0]   # bound genes only

sns.violinplot(data=plot_data, x='rescue_group', y='CHD8_score_log',
               order=[o for o in ORDER if o in plot_data['rescue_group'].unique()],
               palette=PALETTE, inner='box', cut=0, ax=ax)

stat, p, _ = kw_and_pairs(plot_data, 'CHD8_score_log',
                           [o for o in ORDER if o in plot_data['rescue_group'].unique()])
ax.set_title(f'CHD8 Binding Intensity by Domain-Requirement Category\n'
             f'Kruskal–Wallis p = {p:.3f}', fontsize=10, fontweight='bold')
ax.set_ylabel('CHD8 ChIP-seq Peak Score (log₁₀)', fontsize=9)
ax.set_xlabel('')
ax.set_xticklabels([o for o in ORDER if o in plot_data['rescue_group'].unique()],
                    rotation=30, ha='right', fontsize=8)

# Panel 2: % CHD8-bound per category
ax2 = axes[1]
bound_pct = df.groupby('rescue_group').apply(
    lambda x: (x['CHD8_bound'].sum() / len(x) * 100)).reindex(ORDER)
bars = ax2.bar(range(len(ORDER)), bound_pct.values, color=COLORS, edgecolor='white', width=0.6)
ax2.set_xticks(range(len(ORDER)))
ax2.set_xticklabels(ORDER, rotation=30, ha='right', fontsize=8)
ax2.set_ylabel('% genes with CHD8 peak within 2kb TSS', fontsize=9)
ax2.set_title('CHD8 Occupancy by Domain-Requirement Category', fontsize=10, fontweight='bold')
ax2.set_ylim(0, 110)
for i, v in enumerate(bound_pct.values):
    if not np.isnan(v):
        ax2.text(i, v+1.5, f'{v:.0f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Chd8 data/figures/Reviewer_CHD8_Binding_per_Category.pdf',
            bbox_inches='tight', dpi=150)
plt.savefig('/content/drive/MyDrive/Chd8 data/figures/Reviewer_CHD8_Binding_per_Category.png',
            bbox_inches='tight', dpi=150)
plt.show()
print("Saved: CHD8 binding figure")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE B — ATAC-seq accessibility per category
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: ATAC LFC distribution
ax = axes[0]
atac_data = df.dropna(subset=['ATAC_LFC'])
sns.violinplot(data=atac_data, x='rescue_group', y='ATAC_LFC',
               order=[o for o in ORDER if o in atac_data['rescue_group'].unique()],
               palette=PALETTE, inner='box', cut=0, ax=ax)
ax.axhline(0, color='red', linestyle='--', lw=1, label='No change')

stat2, p2, _ = kw_and_pairs(atac_data, 'ATAC_LFC',
                             [o for o in ORDER if o in atac_data['rescue_group'].unique()])
ax.set_title(f'Chromatin Accessibility Change (ATAC-seq) by Category\n'
             f'Kruskal–Wallis p = {p2:.3f}', fontsize=10, fontweight='bold')
ax.set_ylabel('ATAC-seq log₂FC (KO vs WT) at proximal peaks', fontsize=9)
ax.set_xlabel('')
ax.set_xticklabels([o for o in ORDER if o in atac_data['rescue_group'].unique()],
                    rotation=30, ha='right', fontsize=8)

# Panel 2: % with ATAC change per category
ax2 = axes[1]
atac_pct = df.groupby('rescue_group').apply(
    lambda x: (x['ATAC_changed'].sum() / len(x) * 100)).reindex(ORDER)
bars = ax2.bar(range(len(ORDER)), atac_pct.values, color=COLORS, edgecolor='white', width=0.6)
ax2.set_xticks(range(len(ORDER)))
ax2.set_xticklabels(ORDER, rotation=30, ha='right', fontsize=8)
ax2.set_ylabel('% genes with ATAC accessibility change nearby', fontsize=9)
ax2.set_title('Accessibility Loss by Domain-Requirement Category', fontsize=10, fontweight='bold')
ax2.set_ylim(0, 60)
for i, v in enumerate(atac_pct.values):
    if not np.isnan(v):
        ax2.text(i, v+0.8, f'{v:.0f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Chd8 data/figures/Reviewer_ATAC_per_Category.pdf',
            bbox_inches='tight', dpi=150)
plt.savefig('/content/drive/MyDrive/Chd8 data/figures/Reviewer_ATAC_per_Category.png',
            bbox_inches='tight', dpi=150)
plt.show()
print("Saved: ATAC figure")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE C — GO enrichment per category (nominal p<0.05, use as exploratory)
# ═══════════════════════════════════════════════════════════════════════════════
print("\nRunning GO enrichment per category...")
go_dir = '/content/drive/MyDrive/CHD8_Rescue_Groups/'
go_files = {
    'Global Rescue':         'GO_BP_Global_Rescue.csv',
    'Chromodomain Required': 'GO_BP_Chromodomain_Required.csv',
    'Dual Domain Required':  'GO_BP_Dual_Domain_Required.csv',
    'Low/No Rescue':         'GO_BP_Low_No_Rescue.csv',
}

# Use nominal p<0.05 since groups are small (note in figure legend)
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, (cat, fname) in zip(axes, go_files.items()):
    try:
        res = pd.read_csv(go_dir + fname)
        top = (res[res['P-value'] < 0.05]
               .sort_values('P-value')
               .head(8)
               .copy())
        if top.empty:
            ax.text(0.5, 0.5, 'No significant\nterms (p<0.05)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=9)
            ax.set_title(cat, fontsize=9, fontweight='bold', color=PALETTE.get(cat,'black'))
            continue
        top['-log10p'] = -np.log10(top['P-value'].clip(1e-10))
        top['Term_short'] = top['Term'].str.split(' \(GO').str[0].str.strip().str[:40]
        top = top.sort_values('-log10p')
        ax.barh(top['Term_short'], top['-log10p'],
                color=PALETTE.get(cat, '#aaaaaa'), edgecolor='white', alpha=0.85)
        ax.axvline(-np.log10(0.05), color='red', linestyle='--', lw=0.8)
        ax.set_title(cat, fontsize=9, fontweight='bold', color=PALETTE.get(cat,'black'))
        ax.set_xlabel('-log₁₀(p-value)', fontsize=8)
        ax.tick_params(axis='y', labelsize=7)
        n = (df['rescue_group']==cat).sum()
        ax.text(0.98, 0.02, f'n={n}', transform=ax.transAxes,
                ha='right', va='bottom', fontsize=8, color='gray')
    except Exception as e:
        ax.text(0.5, 0.5, f'Error:\n{e}', ha='center', va='center',
                transform=ax.transAxes, fontsize=7)

fig.suptitle('GO Biological Process Enrichment by Domain-Requirement Category\n'
             '(nominal p<0.05; FDR correction not achieved due to small group sizes)',
             fontsize=10, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Chd8 data/figures/Reviewer_GO_per_Category.pdf',
            bbox_inches='tight', dpi=150)
plt.savefig('/content/drive/MyDrive/Chd8 data/figures/Reviewer_GO_per_Category.png',
            bbox_inches='tight', dpi=150)
plt.show()
print("Saved: GO figure")

# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE —
# ═══════════════════════════════════════════════════════════════════════════════
summary = df.groupby('rescue_group').agg(
    n_genes          = ('gene_symbol', 'count'),
    pct_CHD8_bound   = ('CHD8_bound',  lambda x: round(x.mean()*100, 1)),
    median_CHD8_score= ('CHD8_score',  'median'),
    median_ATAC_LFC  = ('ATAC_LFC',    'median'),
    pct_ATAC_changed = ('ATAC_changed',lambda x: round(x.mean()*100, 1)),
    median_RF_FL     = ('RF_FL',       'median'),
    median_RF_dChromo= ('RF_dChromo',  'median'),
    median_RF_dHelic = ('RF_dHelicase','median'),
).reindex(ORDER).round(3)

print("\n=== SUMMARY TABLE ===")
print(summary.to_string())
summary.to_csv('/content/drive/MyDrive/Chd8 data/figures/Reviewer_Category_Summary_Table.csv')
print("\nAll outputs saved to /content/drive/MyDrive/Chd8 data/figures/")

In [ ]:
# How many genes were tested for RF in total?
rf_full = pd.read_csv('/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S9_Rescue_Fractions.csv')
print("Shape:", rf_full.shape)
print("Columns:", rf_full.columns.tolist())
print(rf_full.head(5))

# Also check the per-gene rescue validation table
pg = pd.read_csv('/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S13b_Rescue_Validation_PerGene.csv')
print("\nPer-gene table shape:", pg.shape)
print("Columns:", pg.columns.tolist())
print(pg.head(5))

In [ ]:
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Load full RF table ────────────────────────────────────────────────────────
rf = pd.read_csv('/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S9_Rescue_Fractions.csv')

# ── Assign categories using RF threshold ≥0.5 (matches paper) ────────────────
RF_THRESH = 0.5

def assign_category(row):
    fl = row['RF_FL']  >= RF_THRESH
    dc = row['RF_dC']  >= RF_THRESH
    dh = row['RF_dH']  >= RF_THRESH
    if fl and dc and dh:   return 'Global Rescue'
    elif fl and dc and not dh: return 'Helicase Required'
    elif fl and dh and not dc: return 'Chromodomain Required'
    elif fl and not dc and not dh: return 'Dual Domain Required'
    else:                  return 'Low/No Rescue'

rf['category'] = rf.apply(assign_category, axis=1)

print("Full category counts:")
print(rf['category'].value_counts())
print(f"\nTotal genes: {len(rf)}")

In [ ]:
# Check per-gene table more carefully
pg = pd.read_csv('/content/drive/MyDrive/CHD8_Supplementary_Tables/Table_S13b_Rescue_Validation_PerGene.csv')
print("Shape:", pg.shape)
print(pg.head(3))

# Check the full DESeq2 rescue results directly
import os
rescue_dir = '/content/drive/MyDrive/Chd8 data/deseq2/deseq2/deseq2_star_trimmed/results/'
for f in os.listdir(rescue_dir):
    if 'KO' in f and ('FL' in f or 'dC' in f or 'dHs' in f):
        print(f)

In [ ]:
# Peek at the three rescue DESeq2 files
import pandas as pd

ko  = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv', sep='\t', nrows=3)
fl  = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv', sep='\t', nrows=3)
dc  = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv', sep='\t', nrows=3)
dhs = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv', sep='\t', nrows=3)

print("KO columns:", ko.columns.tolist())
print(ko.head(2))
print("\nFL columns:", fl.columns.tolist())
print(fl.head(2))

In [ ]:
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

rescue_dir = '/content/drive/MyDrive/Chd8 data/deseq2/deseq2/deseq2_star_trimmed/results/'

# ── Load all four conditions ──────────────────────────────────────────────────
ko  = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv',      sep='\t',
                  usecols=['Unnamed: 0','GENESYMBOL','log2FoldChange','baseMean']
                 ).rename(columns={'Unnamed: 0':'gene_id','log2FoldChange':'LFC_KO'})

fl  = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv',   sep='\t',
                  usecols=['Unnamed: 0','log2FoldChange']
                 ).rename(columns={'Unnamed: 0':'gene_id','log2FoldChange':'LFC_FL'})

dc  = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv',   sep='\t',
                  usecols=['Unnamed: 0','log2FoldChange']
                 ).rename(columns={'Unnamed: 0':'gene_id','log2FoldChange':'LFC_dC'})

dhs = pd.read_csv(rescue_dir + 'Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv',  sep='\t',
                  usecols=['Unnamed: 0','log2FoldChange']
                 ).rename(columns={'Unnamed: 0':'gene_id','log2FoldChange':'LFC_dHs'})

# ── Merge ─────────────────────────────────────────────────────────────────────
df = ko.merge(fl, on='gene_id').merge(dc, on='gene_id').merge(dhs, on='gene_id')
print(f"Merged: {df.shape[0]} genes")

# ── Filter: baseMean > 10 and |LFC_KO| > 0.5 (dysregulated in KO) ────────────
df = df[(df['baseMean'] > 10) & (df['LFC_KO'].abs() > 0.5)].copy()
print(f"After filtering (baseMean>10, |LFC_KO|>0.5): {df.shape[0]} genes")

# ── Compute Rescue Fraction ───────────────────────────────────────────────────
for col, lfc_col in [('RF_FL','LFC_FL'), ('RF_dC','LFC_dC'), ('RF_dHs','LFC_dHs')]:
    denom = (0 - df['LFC_KO']).replace(0, np.nan)
    df[col] = (df[lfc_col] - df['LFC_KO']) / denom

# ── Assign categories ─────────────────────────────────────────────────────────
RF_THRESH = 0.5

def assign_category(row):
    fl  = row['RF_FL']  >= RF_THRESH
    dc  = row['RF_dC']  >= RF_THRESH
    dhs = row['RF_dHs'] >= RF_THRESH
    if fl and dc and dhs:        return 'Global Rescue'
    elif fl and dc and not dhs:  return 'Helicase Required'
    elif fl and dhs and not dc:  return 'Chromodomain Required'
    elif fl and not dc and not dhs: return 'Dual Domain Required'
    else:                        return 'Low/No Rescue'

df['category'] = df.apply(assign_category, axis=1)
print("\nFull category counts:")
print(df['category'].value_counts())

# ── GO enrichment per category ────────────────────────────────────────────────
ORDER = ['Global Rescue','Helicase Required','Chromodomain Required',
         'Dual Domain Required','Low/No Rescue']
ORDER = [o for o in ORDER if (df['category']==o).sum() >= 10]

background = df['GENESYMBOL'].str.upper().dropna().tolist()
go_results = {}

for cat in ORDER:
    genes = df[df['category']==cat]['GENESYMBOL'].str.upper().dropna().tolist()
    print(f"\nRunning GO for {cat} (n={len(genes)})...")
    try:
        enr = gp.enrichr(
            gene_list  = genes,
            gene_sets  = ['GO_Biological_Process_2023'],
            background = background,
            organism   = 'mouse',
            outdir     = None,
            cutoff     = 0.05
        )
        go_results[cat] = enr.results
        sig = enr.results[enr.results['Adjusted P-value'] < 0.05]
        print(f"  Significant terms (FDR<0.05): {len(sig)}")
        if len(sig) > 0:
            print(sig[['Term','Adjusted P-value']].head(5).to_string())
    except Exception as e:
        print(f"  Error: {e}")

print("\nDone.")

In [ ]:
# ── Pure offline fallback using fisher exact ──────────────────────────────────
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# Download GO gene sets once
go_lib = gp.get_library('GO_Biological_Process_2023', organism='Mouse')
print(f"GO library loaded: {len(go_lib)} terms")

all_genes = set(df['GENESYMBOL'].str.upper().dropna())
go_results_local = {}

for cat in ORDER:
    query = set(df[df['category']==cat]['GENESYMBOL'].str.upper().dropna())
    rows = []
    for term, term_genes in go_lib.items():
        term_set   = set(g.upper() for g in term_genes) & all_genes
        if len(term_set) < 3: continue
        a = len(query & term_set)
        if a == 0: continue
        b = len(query) - a
        c = len(term_set) - a
        d = len(all_genes) - len(query) - c
        _, p = fisher_exact([[a,b],[c,d]], alternative='greater')
        rows.append({'Term':term, 'n_overlap':a,
                     'n_term':len(term_set), 'p_value':p,
                     'genes':';'.join(sorted(query & term_set))})
    res = pd.DataFrame(rows)
    if len(res):
        res['padj'] = multipletests(res['p_value'], method='fdr_bh')[1]
        res = res.sort_values('padj')
        go_results_local[cat] = res
        sig = res[res['padj'] < 0.05]
        print(f"\n{cat} (n={len(query)}): {len(sig)} FDR<0.05 terms")
        print(sig[['Term','padj','n_overlap','n_term']].head(5).to_string())

In [ ]:
# ── Final GO summary figure ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Only plot categories with results
plot_cats = {
    'Chromodomain Required': go_results_local['Chromodomain Required'],
    'Low/No Rescue':         go_results_local['Low/No Rescue'],
}

# Panel 1: Chromodomain Required
ax = axes[0]
res = go_results_local['Chromodomain Required']
top = res[res['padj'] < 0.05].head(8).copy()
top['-log10padj'] = -np.log10(top['padj'].clip(1e-10))
top['Term_short'] = top['Term'].str.split(' \(GO').str[0].str[:45]
top = top.sort_values('-log10padj')
ax.barh(top['Term_short'], top['-log10padj'],
        color=PALETTE['Chromodomain Required'], edgecolor='white', alpha=0.85)
ax.axvline(-np.log10(0.05), color='red', linestyle='--', lw=1)
ax.set_title('Chromodomain Required\n(n=697)', fontsize=10, fontweight='bold',
             color=PALETTE['Chromodomain Required'])
ax.set_xlabel('-log₁₀(FDR)', fontsize=9)
ax.tick_params(axis='y', labelsize=8)

# Panel 2: Low/No Rescue
ax = axes[1]
res2 = go_results_local['Low/No Rescue']
top2 = res2[res2['padj'] < 0.05].head(8).copy()
top2['-log10padj'] = -np.log10(top2['padj'].clip(1e-10))
top2['Term_short'] = top2['Term'].str.split(' \(GO').str[0].str[:45]
top2 = top2.sort_values('-log10padj')
ax.barh(top2['Term_short'], top2['-log10padj'],
        color=PALETTE['Low/No Rescue'], edgecolor='white', alpha=0.85)
ax.axvline(-np.log10(0.05), color='red', linestyle='--', lw=1)
ax.set_title('Low/No Rescue\n(n=1090)', fontsize=10, fontweight='bold',
             color=PALETTE['Low/No Rescue'])
ax.set_xlabel('-log₁₀(FDR)', fontsize=9)
ax.tick_params(axis='y', labelsize=8)

# Panel 3: Category size + CHD8 binding summary bar
ax = axes[2]
cat_counts = df['category'].value_counts().reindex(ORDER)
colors_bar  = [PALETTE[c] for c in ORDER]
bars = ax.bar(range(len(ORDER)), cat_counts.values,
              color=colors_bar, edgecolor='white', width=0.6)
ax.set_xticks(range(len(ORDER)))
ax.set_xticklabels(ORDER, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Number of genes', fontsize=9)
ax.set_title('Domain-Requirement Category Sizes\n(all KO-dysregulated genes)',
             fontsize=10, fontweight='bold')
for i, v in enumerate(cat_counts.values):
    ax.text(i, v+10, str(v), ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Functional Characterisation of Domain-Requirement Categories',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Chd8 data/figures/Reviewer_GO_FullGenes_Final.pdf',
            bbox_inches='tight', dpi=150)
plt.savefig('/content/drive/MyDrive/Chd8 data/figures/Reviewer_GO_FullGenes_Final.png',
            bbox_inches='tight', dpi=150)
plt.show()
print("Saved.")

In [ ]:
import pandas as pd
import pybedtools
import os

BASE_RNA  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
H3K4_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"

STANDARD_CHROMS = set([f'chr{i}' for i in range(1,20)] + ['chrX','chrY'])
def is_standard(f): return str(f.chrom) in STANDARD_CHROMS
def fix_chr(f):
    if not str(f.chrom).startswith('chr'): f.chrom = 'chr' + str(f.chrom)
    return f
def get_genes(bed):
    return set(str(f[3]).upper().strip() for f in bed if len(str(f[3])) > 1)

print("=" * 60)
print("STEP 1: GENE COUNTS FROM DIFFERENT PEAK SETS")
print("=" * 60)
npc  = pybedtools.BedTool(PEAK_PATH).each(fix_chr).filter(is_standard).sort().saveas()
esc  = pybedtools.BedTool(ESC_PATH).each(fix_chr).filter(is_standard).sort().saveas()
tss  = pybedtools.BedTool(TSS_PATH).each(fix_chr).filter(is_standard).sort().saveas()
h3k4 = pybedtools.BedTool(H3K4_PATH).each(fix_chr).filter(is_standard).sort().saveas()

genes_all_npc      = get_genes(tss.intersect(npc, u=True, wa=True).saveas())
npc_specific       = npc.subtract(esc, A=True).saveas()
genes_npc_specific = get_genes(tss.intersect(npc_specific, u=True, wa=True).saveas())
genes_all_npc_h3k4 = get_genes(tss.intersect(npc, u=True, wa=True).intersect(h3k4, u=True, wa=True).saveas())

print(f"  All NPC peaks at TSS (±2kb)           : {len(genes_all_npc):,} genes")
print(f"  NPC-specific peaks at TSS (±2kb)       : {len(genes_npc_specific):,} genes")
print(f"  All NPC + H3K4me3 at TSS (±2kb)        : {len(genes_all_npc_h3k4):,} genes")

print("\n" + "=" * 60)
print("STEP 2: INTERSECTION WITH DEGs — UP vs DOWN")
print("=" * 60)
rna = pd.read_csv(BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv", sep='\t')
rna = rna[rna['baseMean'] > 10].dropna(subset=['log2FoldChange','padj'])
rna['gene'] = rna['GENESYMBOL'].str.upper().str.strip()
deg = rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'].abs() > 0.5)]

for label, geneset in [
    ("All NPC peaks",         genes_all_npc),
    ("NPC-specific peaks",    genes_npc_specific),
    ("All NPC + H3K4me3",     genes_all_npc_h3k4),
]:
    s = deg[deg['gene'].isin(geneset)]
    up   = s[s['log2FoldChange'] > 0]
    down = s[s['log2FoldChange'] < 0]
    print(f"\n  {label}:")
    print(f"    Total DEGs : {len(s):,}")
    print(f"    Up         : {len(up):,}")
    print(f"    Down       : {len(down):,}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pybedtools
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests
import os

# ============================================================
# 1. PATHS & SETUP
# ============================================================
CHD8_NPC_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
H3K4_NPC_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/H3K4me3_NPC_merged.bed"
TSS_PATH      = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_PATH      = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
SAVE_DIR      = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

STANDARD_CHROMS = set([f'chr{i}' for i in range(1, 20)] + ['chrX', 'chrY'])

def fix_chr(feature):
    chrom = str(feature.chrom)
    if not chrom.startswith('chr'):
        feature.chrom = 'chr' + chrom
    return feature

def is_standard(feature):
    return str(feature.chrom) in STANDARD_CHROMS

def get_gene_set(bed_result):
    genes = set()
    for f in bed_result:
        name = str(f[3]).strip().upper()
        if name and name != '.' and len(name) > 1:
            genes.add(name)
    return genes

# ============================================================
# 2. GENOMIC INTERSECTIONS
# ============================================================
chd8_bt = (pybedtools.BedTool(CHD8_NPC_PATH)
           .each(fix_chr).filter(is_standard).sort().saveas())
h3k4_bt = (pybedtools.BedTool(H3K4_NPC_PATH)
           .each(fix_chr).filter(is_standard).sort().saveas())
tss_bt  = (pybedtools.BedTool(TSS_PATH)
           .each(fix_chr).filter(is_standard).sort().saveas())

chd8_h3k4_tss = (tss_bt.intersect(chd8_bt, u=True, wa=True)
                        .intersect(h3k4_bt, u=True, wa=True).saveas())
h3k4_only_tss = (tss_bt.intersect(h3k4_bt, u=True, wa=True)
                        .intersect(chd8_bt, v=True).saveas())
unbound_tss   = (tss_bt.intersect(chd8_bt, v=True)
                        .intersect(h3k4_bt, v=True).saveas())

genes_both     = get_gene_set(chd8_h3k4_tss)
genes_h3k4only = get_gene_set(h3k4_only_tss)
genes_unbound  = get_gene_set(unbound_tss)

# ============================================================
# 3. RNA-SEQ INTEGRATION
# ============================================================
rna = pd.read_csv(RNA_PATH, sep='\t')
rna = rna[rna['baseMean'] > 10].copy()
rna['gene_upper'] = rna['GENESYMBOL'].str.upper().str.strip()
rna['log2_expr']  = np.log2(rna['Diff.Fa2Ls4'] + 1)

def assign_group(gene):
    g = gene.upper()
    if g in genes_both:       return 'CHD8 +\nH3K4me3'
    elif g in genes_h3k4only: return 'H3K4me3\nonly'
    elif g in genes_unbound:  return 'Unbound'
    return 'Other'

rna['group'] = rna['gene_upper'].apply(assign_group)
rna_3 = rna[rna['group'].isin(['CHD8 +\nH3K4me3', 'H3K4me3\nonly', 'Unbound'])].copy()

ORDER  = ['CHD8 +\nH3K4me3', 'H3K4me3\nonly', 'Unbound']
COLORS = ['#1f77b4', '#6baed6', '#bdbdbd']
groups_data = {g: rna_3[rna_3['group'] == g]['log2_expr'] for g in ORDER}
counts      = {g: len(groups_data[g])     for g in ORDER}

# ============================================================
# 4. STATISTICS
# ============================================================
comparisons = [('CHD8 +\nH3K4me3', 'H3K4me3\nonly'), ('CHD8 +\nH3K4me3', 'Unbound'), ('H3K4me3\nonly', 'Unbound')]
pvals = [mannwhitneyu(groups_data[g1], groups_data[g2], alternative='two-sided')[1] for g1, g2 in comparisons]
_, pvals_corr, _, _ = multipletests(pvals, method='fdr_bh')
kw_stat, kw_p = kruskal(*[groups_data[g].values for g in ORDER])

# ============================================================
# 5. FIGURE GENERATION
# ============================================================
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial'], 'font.size': 12, 'pdf.fonttype': 42})
fig, ax = plt.subplots(figsize=(7, 7.5), dpi=600)

parts = ax.violinplot([groups_data[g].values for g in ORDER], positions=range(len(ORDER)), showmedians=False, showextrema=False, widths=0.65)
for pc, col in zip(parts['bodies'], COLORS):
    pc.set_facecolor(col)
    pc.set_edgecolor('black')
    pc.set_linewidth(0.8)
    pc.set_alpha(0.75)

ax.boxplot([groups_data[g].values for g in ORDER], positions=range(len(ORDER)), widths=0.16, patch_artist=True, showfliers=False,
           medianprops=dict(color='black', linewidth=2), boxprops=dict(facecolor='white', linewidth=1))

ymax = rna_3['log2_expr'].max()
def sig_bracket(ax, x1, x2, y, p):
    label = ('***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns')
    h = 0.2
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, color='black')
    ax.text((x1+x2)/2, y+h+0.05, f'{label}\nFDR={p:.2e}', ha='center', va='bottom', fontsize=8)

# Significance brackets
sig_bracket(ax, 0, 1, ymax + 0.2, pvals_corr[0])
sig_bracket(ax, 1, 2, ymax + 0.2, pvals_corr[2])
sig_bracket(ax, 0, 2, ymax + 1.5, pvals_corr[1])

# X-axis
ax.set_xticks(range(len(ORDER)))
ax.set_xticklabels([f'{g}\nn = {counts[g]:,}' for g in ORDER], fontsize=11, linespacing=1.4)
ax.set_ylabel('log\u2082 expression (WT NPC)', fontsize=12, fontweight='bold')
ax.set_title('CHD8/H3K4me3 Co-occupancy and Gene Expression', fontsize=11, fontweight='bold', pad=30)
ax.spines[['top', 'right']].set_visible(False)

legend_patches = [mpatches.Patch(facecolor=c, edgecolor='black', linewidth=0.8, label=l.replace('\n', ' ')) for c, l in zip(COLORS, ORDER)]
ax.legend(handles=legend_patches, fontsize=9, frameon=False, loc='upper right', bbox_to_anchor=(1.05, 1.05))

ax.set_ylim(top=ymax + 4.5)
plt.tight_layout()
fig.subplots_adjust(bottom=0.15, top=0.85)

png_path = os.path.join(SAVE_DIR, 'Figure3A_CHD8_H3K4me3_Final_v4.png')
plt.savefig(png_path, dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

BASE = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"

files = {
    "KO"  : BASE + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    "KD1" : BASE + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv",
    "KD2" : BASE + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv",
}

print("=" * 60)
print("DEG COUNTS PER CONDITION")
print("=" * 60)
for label, path in files.items():
    df = pd.read_csv(path, sep='\t').dropna(subset=['log2FoldChange','padj'])
    df = df[df['baseMean'] > 10]
    up   = df[(df['padj'] < 0.05) & (df['log2FoldChange'] >  0.5)]
    down = df[(df['padj'] < 0.05) & (df['log2FoldChange'] < -0.5)]
    total = len(up) + len(down)
    print(f"\n  {label}:")
    print(f"    Upregulated  : {len(up):,}")
    print(f"    Downregulated: {len(down):,}")
    print(f"    Total DEGs   : {total:,}")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import gseapy as gp
import os

BASE_RNA = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
SAVE_DIR = "/content/drive/MyDrive/Chd8 data/figures/GO_corrected/"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 1. Build background and input gene lists ──────────────────
rna = pd.read_csv(BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv", sep='\t')
rna = rna[rna['baseMean'] > 10].dropna(subset=['log2FoldChange', 'padj'])
rna['gene'] = rna['GENESYMBOL'].str.upper().str.strip()
background_genes = set(rna['gene'].tolist())

import pybedtools
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"

STANDARD_CHROMS = set([f'chr{i}' for i in range(1, 20)] + ['chrX', 'chrY'])
def is_standard(f): return str(f.chrom) in STANDARD_CHROMS
def fix_chr(f):
    if not str(f.chrom).startswith('chr'): f.chrom = 'chr' + str(f.chrom)
    return f

npc = pybedtools.BedTool(PEAK_PATH).each(fix_chr).filter(is_standard).sort().saveas()
esc = pybedtools.BedTool(ESC_PATH).each(fix_chr).filter(is_standard).sort().saveas()
tss = pybedtools.BedTool(TSS_PATH).each(fix_chr).filter(is_standard).sort().saveas()
npc_specific  = npc.subtract(esc, A=True).saveas()
npc_spec_genes = set(
    str(f[3]).upper().strip() for f in
    tss.intersect(npc_specific, u=True, wa=True)
    if len(str(f[3])) > 1
)
deg = rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'].abs() > 0.5)]
query_genes = set(deg[deg['gene'].isin(npc_spec_genes)]['gene'].tolist())

print(f"Input : {len(query_genes):,} genes")
print(f"Background : {len(background_genes):,} genes")

# ── 2. Download GO gene sets via gseapy (no API query needed) ──
print("\nDownloading GO_Biological_Process_2023 gene sets...")
gmt = gp.get_library(name='GO_Biological_Process_2023', organism='Mouse')
print(f"GO terms loaded: {len(gmt):,}")

# ── 3. Local Fisher's exact test (same as Enrichr) ────────────
N = len(background_genes)   # total background
k = len(query_genes)        # query list size

rows = []
for term, term_genes_raw in gmt.items():
    # Restrict term gene set to background (critical for correct stats)
    term_genes = set(g.upper() for g in term_genes_raw) & background_genes
    K = len(term_genes)       # term size within background
    if K < 5 or K > 500:     # skip tiny/huge terms
        continue

    overlap = query_genes & term_genes
    x = len(overlap)          # overlap count
    if x == 0:
        continue

    # Contingency table:
    #              In term    Not in term
    # In query        x          k - x
    # Not in query   K-x      N - K - (k-x)
    table = [[x,         k - x],
             [K - x,     N - K - (k - x)]]

    _, pval = fisher_exact(table, alternative='greater')

    rows.append({
        'Term'         : term,
        'P-value'      : pval,
        'Overlap_n'    : x,
        'Term_size_bg' : K,
        'Query_size'   : k,
        'Background_n' : N,
        'Overlap_genes': ';'.join(sorted(overlap)),
        'Overlap'      : f"{x}/{K}",
    })

results = pd.DataFrame(rows)

# ── 4. Multiple testing correction ───────────────────────────
_, padj, _, _ = multipletests(results['P-value'], method='fdr_bh')
results['Adjusted P-value'] = padj
results = results.sort_values('P-value')

sig = results[results['Adjusted P-value'] < 0.05]
nom = results[results['P-value'] < 0.05]

print(f"\n✅ SUCCESS")
print(f"Significant terms (FDR < 0.05) : {len(sig):,}")
print(f"Nominal terms (P < 0.05)       : {len(nom):,}")
print("\nTop 15 terms (by P-value):")
print(nom[['Term', 'P-value', 'Adjusted P-value', 'Overlap']].head(15).to_string())

# ── 5. Save ───────────────────────────────────────────────────
results.to_csv(SAVE_DIR + "GO_all_results_corrected_background.csv", index=False)
nom.to_csv(SAVE_DIR + "GO_nominal_p05_corrected_background.csv", index=False)
sig.to_csv(SAVE_DIR + "GO_FDR05_corrected_background.csv", index=False)
print(f"\nSaved to {SAVE_DIR}")

In [ ]:
# Reuse ranked list from KO data already in your pipeline
rna['rank_metric'] = -np.log10(rna['padj'].replace(0,1e-300)) * np.sign(rna['log2FoldChange'])
rnk = rna.sort_values('rank_metric', ascending=False).set_index('gene')['rank_metric']
rnk = rnk[~rnk.index.duplicated()]

pre_res = gp.prerank(
    rnk=rnk,
    gene_sets='GO_Biological_Process_2023',
    organism='Mouse',
    permutation_num=1000,
    seed=42,
    min_size=15,
    max_size=500
)
res = pre_res.res2d.sort_values('FDR q-val')
print(res[res['FDR q-val']<0.25][['Term','NES','NOM p-val','FDR q-val']].head(20).to_string())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import textwrap

nom = pd.read_csv("/content/drive/MyDrive/Chd8 data/figures/GO_corrected/GO_nominal_p05_corrected_background.csv")
top10 = nom.head(10).copy()
top10['log_p'] = -np.log10(top10['P-value'])
top10['Label'] = top10['Term'].apply(lambda x: '\n'.join(textwrap.wrap(x.split(' (GO')[0], 40)))
top10 = top10.sort_values('log_p', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top10['Label'], top10['log_p'], color='#2C3E50', edgecolor='white', height=0.6)
for bar, (_, row) in zip(bars, top10.iterrows()):
    ax.text(bar.get_width()*0.97, bar.get_y()+bar.get_height()/2,
            f"k/K={row['Overlap']}", va='center', ha='right',
            fontsize=8, color='white', fontweight='bold')
ax.axvline(-np.log10(0.05), color='#E74C3C', linestyle='--', lw=1.2, label='nominal P=0.05')
ax.set_xlabel('-log₁₀(P-value)', fontsize=12, fontweight='bold')
ax.set_title('GO Biological Process Enrichment\n(NPC-specific CHD8 targets ∩ KO DEGs, n=254;\nbackground: all expressed genes, n=13,587)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.subplots_adjust(left=0.42)
plt.savefig("/content/drive/MyDrive/Chd8 data/figures/GO_corrected/Figure2A_corrected.png", dpi=600, bbox_inches='tight')
plt.savefig("/content/drive/MyDrive/Chd8 data/figures/GO_corrected/Figure2A_corrected.pdf", bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import gseapy as gp

BASE_RNA = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
SAVE_DIR = "/content/drive/MyDrive/Chd8 data/figures/GO_corrected/"

# Background
rna = pd.read_csv(BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv", sep='\t')
rna = rna[rna['baseMean'] > 10].dropna(subset=['log2FoldChange','padj'])
rna['gene'] = rna['GENESYMBOL'].str.upper().str.strip()
background_genes = set(rna['gene'].tolist())

# Input: KO downregulated genes
ko_down = set(rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'] < -0.5)]['gene'].tolist())
print(f"Input: {len(ko_down):,} | Background: {len(background_genes):,}")

# Local Fisher's exact test
gmt = gp.get_library(name='GO_Biological_Process_2023', organism='Mouse')
N, k = len(background_genes), len(ko_down)
rows = []
for term, term_genes_raw in gmt.items():
    term_genes = set(g.upper() for g in term_genes_raw) & background_genes
    K = len(term_genes)
    if K < 5 or K > 500: continue
    overlap = ko_down & term_genes
    x = len(overlap)
    if x == 0: continue
    _, pval = fisher_exact([[x, k-x],[K-x, N-K-(k-x)]], alternative='greater')
    rows.append({'Term': term, 'P-value': pval, 'Overlap_n': x,
                 'Term_size': K, 'Overlap': f"{x}/{K}",
                 'Overlap_genes': ';'.join(sorted(overlap))})

results = pd.DataFrame(rows)
_, padj, _, _ = multipletests(results['P-value'], method='fdr_bh')
results['Adjusted P-value'] = padj
results = results.sort_values('Adjusted P-value')

sig = results[results['Adjusted P-value'] < 0.05]
nom = results[results['P-value'] < 0.05]
print(f"FDR < 0.05: {len(sig):,} | Nominal P < 0.05: {len(nom):,}")

# Save tables
results.to_csv(SAVE_DIR + "Figure5A_GO_all_corrected.csv", index=False)
sig.to_csv(SAVE_DIR + "Figure5A_GO_FDR05_corrected.csv", index=False)

# Plot top 10 (FDR if available, else nominal)
plot_df = sig if len(sig) >= 5 else nom
top10 = plot_df.head(10).copy()
top10['log_p'] = -np.log10(top10['Adjusted P-value'] if len(sig) >= 5 else top10['P-value'])
top10['Label'] = top10['Term'].apply(lambda x: '\n'.join(textwrap.wrap(x.split(' (GO')[0], 40)))
top10 = top10.sort_values('log_p', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top10['Label'], top10['log_p'], color='#2E86AB', edgecolor='white', height=0.6)
for bar, (_, row) in zip(bars, top10.iterrows()):
    ax.text(bar.get_width()*0.97, bar.get_y()+bar.get_height()/2,
            f"k/K={row['Overlap']}", va='center', ha='right',
            fontsize=8, color='white', fontweight='bold')
xlabel = '-log₁₀(FDR)' if len(sig) >= 5 else '-log₁₀(P-value)'
ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
ax.set_title('GO Biological Process — KO Downregulated Genes\n(background: all expressed genes, n=13,587)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(left=0.42)
plt.savefig(SAVE_DIR + "Figure5A_corrected.png", dpi=600, bbox_inches='tight')
plt.savefig(SAVE_DIR + "Figure5A_corrected.pdf", bbox_inches='tight')
plt.show()
print("Done.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pybedtools
import subprocess
import os
import time
import glob
import gseapy as gp

# ── PATHS ─────────────────────────────────────────────────────────
PEAK_PATH   = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH    = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH    = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
BASE_ATAC   = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
SAVE_DIR    = "/content/drive/MyDrive/Chd8 data/figures/"
MOTIF_DIR   = "/content/drive/MyDrive/CHD8_TF_Motif_Analysis/"
os.makedirs(MOTIF_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 65)
print("TF MOTIF ENRICHMENT — 100 HIGH-CONFIDENCE CHD8 TARGETS")
print("=" * 65)

# ── STEP 1: Reconstruct triple-overlap gene set ───────────────────
print("\nStep 1: Reconstructing triple-overlap gene set...")

def fix_naming(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

def load_atac_bed(path):
    if not os.path.exists(path): return None
    df = pd.read_csv(path, sep='\t', header=None)
    sc = 6 if len(df.columns) > 6 else 4
    out = df[[0,1,2,sc]].copy()
    out.columns = ['chr','start','end','score']
    out['chr'] = out['chr'].astype(str)
    mask = ~out['chr'].str.startswith('chr')
    out.loc[mask,'chr'] = 'chr' + out.loc[mask,'chr']
    out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
    return out

def average_atac(paths, label):
    dfs = [load_atac_bed(p) for p in paths]
    dfs = [d for d in dfs if d is not None]
    combined = pd.concat(dfs, ignore_index=True)
    combined['bin'] = combined['chr'] + ':' + ((combined['start']//50)*50).astype(str)
    avg = combined.groupby('bin').agg(
        chr=('chr','first'), start=('start','min'),
        end=('end','max'), score=('score','mean')
    ).reset_index()
    print(f"  {label}: {len(avg):,} peaks averaged")
    return avg

ATAC_WT_PATHS = [
    BASE_ATAC + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed",
]
ATAC_KO_PATHS = [
    BASE_ATAC + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed",
]

rna = pd.read_csv(RNA_KO_PATH, sep='\t').dropna(subset=['log2FoldChange','padj'])
rna['padj'] = rna['padj'].replace(0, 1e-300)
if 'baseMean' in rna.columns:
    rna = rna[rna['baseMean'] > 10]
rna['gene_upper'] = rna['GENESYMBOL'].astype(str).str.upper().str.strip()
degs = set(rna[(rna['padj']<0.05)&(rna['log2FoldChange'].abs()>0.5)]['gene_upper'])
print(f"  RNA DEGs: {len(degs):,}")

npc_bt    = pybedtools.BedTool(PEAK_PATH).each(fix_naming).sort()
esc_bt    = pybedtools.BedTool(ESC_PATH).each(fix_naming).sort()
tss_bt    = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()
npc_only  = npc_bt.subtract(esc_bt, A=True)
chip_hits = tss_bt.intersect(npc_only, u=True, wa=True)
chip_targets = set([str(f[3]).upper().strip()
                    for f in chip_hits if len(str(f[3])) > 1])
print(f"  CHD8 NPC-specific ChIP targets: {len(chip_targets):,}")

wt_atac = average_atac(ATAC_WT_PATHS, "WT ATAC")
ko_atac = average_atac(ATAC_KO_PATHS, "KO ATAC")
wt_atac['bin'] = wt_atac['chr'] + ':' + ((wt_atac['start']//50)*50).astype(str)
ko_atac['bin'] = ko_atac['chr'] + ':' + ((ko_atac['start']//50)*50).astype(str)
merged_atac = pd.merge(
    wt_atac[['bin','chr','start','end','score']].rename(columns={'score':'wt_score'}),
    ko_atac[['bin','score']].rename(columns={'score':'ko_score'}),
    on='bin', how='inner'
)
merged_atac['lfc'] = np.log2(
    (merged_atac['ko_score']+0.5) / (merged_atac['wt_score']+0.5))
CHANGED_BED = "/content/atac_changed_motif.bed"
merged_atac[['chr','start','end']].to_csv(
    CHANGED_BED, sep='\t', header=False, index=False)
atac_bt    = pybedtools.BedTool(CHANGED_BED).each(fix_naming).sort()
atac_hits  = tss_bt.intersect(atac_bt, u=True, wa=True)
atac_genes = set([str(f[3]).upper().strip()
                  for f in atac_hits if len(str(f[3])) > 1])
print(f"  ATAC-changed genes: {len(atac_genes):,}")

triple_genes = degs & chip_targets & atac_genes
print(f"\n  ✅ Triple-overlap genes: {len(triple_genes):,}")

# ── STEP 2: Extract CHD8 peaks ────────────────────────────────────
print("\nStep 2: Extracting CHD8 peaks for triple-overlap gene promoters...")

triple_tss_records = []
for f in chip_hits:
    gene = str(f[3]).upper().strip()
    if gene in triple_genes:
        triple_tss_records.append((str(f[0]), int(f[1]), int(f[2]), gene))

triple_tss_df = pd.DataFrame(
    triple_tss_records, columns=['chr','start','end','gene']
).drop_duplicates()

TRIPLE_TSS_BED   = "/content/triple_overlap_tss.bed"
TRIPLE_PEAKS_BED = "/content/triple_overlap_CHD8_peaks.bed"

triple_tss_df[['chr','start','end','gene']].to_csv(
    TRIPLE_TSS_BED, sep='\t', header=False, index=False)

triple_tss_bt = pybedtools.BedTool(TRIPLE_TSS_BED).each(fix_naming).sort()
triple_peaks  = npc_bt.intersect(triple_tss_bt, u=True).saveas(TRIPLE_PEAKS_BED)

print(f"  TSS windows: {len(triple_tss_df):,}")
print(f"  CHD8 peaks overlapping TSS windows: {triple_peaks.count():,}")

gene_list_path = os.path.join(MOTIF_DIR, "triple_overlap_100_genes.txt")
with open(gene_list_path, 'w') as fh:
    fh.write('\n'.join(sorted(triple_genes)))
print(f"  Gene list saved: {gene_list_path}")

# ── STEP 3: HOMER skipped (Colab resource limitation) ─────────────
print("\nStep 3: HOMER skipped — Colab cannot index mm10 genome.")
print("        Using Enrichr for TF co-binding enrichment instead.")
HOMER_AVAILABLE  = False
homer_sig        = pd.DataFrame()
name_col         = None
score_col_homer  = None

# ── STEP 4: Enrichr ───────────────────────────────────────────────
print("\nStep 4: Enrichr TF enrichment...")

try:
    all_libs = gp.get_library_name(organism='mouse')
    print(f"  Available Enrichr mouse libraries: {len(all_libs)}")
except Exception as e:
    print(f"  Could not fetch library list: {e}")
    all_libs = []

def format_mouse_gene(g):
    g = g.strip()
    if g[0].isdigit():
        return g.lower()
    return g[0].upper() + g[1:].lower()

triple_genes_list = [format_mouse_gene(g) for g in sorted(triple_genes)]
print(f"  Gene list (first 5): {triple_genes_list[:5]}")
print(f"  Total genes submitted: {len(triple_genes_list)}")

CANDIDATE_DBS = [
    'ENCODE_TF_ChIP-seq_2015',
    'ChEA_2022',
    'ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X',
    'TRANSFAC_and_JASPAR_PWMs',
    'TF_Perturbations_Followed_by_Expression',
]
if all_libs:
    ENRICHR_DATABASES = [db for db in CANDIDATE_DBS if db in all_libs]
    print(f"  Confirmed available: {ENRICHR_DATABASES}")
else:
    ENRICHR_DATABASES = CANDIDATE_DBS

def enrichr_with_retry(gene_list, db, max_retries=4, backoff=15):
    for attempt in range(max_retries):
        try:
            enr = gp.enrichr(
                gene_list = gene_list,
                gene_sets = [db],
                organism  = 'mouse',
                outdir    = None,
                cutoff    = 1.0,
                verbose   = False,
            )
            if enr.results is not None and not enr.results.empty:
                return enr.results.copy()
            print(f"    {db}: empty results")
            return None
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'rate' in err_str.lower():
                wait = backoff * (2 ** attempt)
                print(f"    Rate-limited on {db}, waiting {wait}s "
                      f"(attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                print(f"    {db} failed: {e}")
                return None
    print(f"    {db} exhausted retries.")
    return None

enrichr_results = {}
for db in ENRICHR_DATABASES:
    time.sleep(3)
    res = enrichr_with_retry(triple_genes_list, db)
    if res is not None and not res.empty:
        res['-log10P']   = -np.log10(res['P-value'].replace(0, 1e-300))
        res['-log10FDR'] = -np.log10(res['Adjusted P-value'].replace(0, 1e-300))
        enrichr_results[db] = res
        n_fdr = (res['Adjusted P-value'] < 0.05).sum()
        n_nom = (res['P-value'] < 0.05).sum()
        print(f"  ✅ {db}: {n_fdr} FDR<0.05 | {n_nom} nominal p<0.05 "
              f"({len(res)} terms tested)")
        top5 = res.nsmallest(5, 'P-value')[
            ['Term','P-value','Adjusted P-value','Overlap']]
        print(f"     Top 5:\n{top5.to_string(index=False)}")
    else:
        print(f"  ⚠️  {db}: no results returned")

# ── STEP 5: Visualisation ─────────────────────────────────────────
print("\nStep 5: Generating figures...")

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size'      : 12,
    'pdf.fonttype'   : 42,
})

# Build plot panels — prioritise FDR-significant DBs first
PLOT_DB_ORDER = [
    'TF_Perturbations_Followed_by_Expression',  # has FDR hits
    'ChEA_2022',
    'ENCODE_TF_ChIP-seq_2015',
    'ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X',
    'TRANSFAC_and_JASPAR_PWMs',
]
COLORS = ['#2C3E50', '#1A5276', '#154360', '#0E6655', '#6E2F7C']

panels = []
for db in PLOT_DB_ORDER:
    if db not in enrichr_results:
        continue
    res = enrichr_results[db]
    sig     = res[res['Adjusted P-value'] < 0.05].copy()
    use_fdr = True
    if sig.empty:
        sig     = res.nsmallest(15, 'P-value').copy()
        use_fdr = False
    else:
        sig = sig.nlargest(min(15, len(sig)), '-log10FDR').copy()
    sig = sig.sort_values('-log10P', ascending=True)
    sig['Label'] = (sig['Term']
                    .str.split('_').str[0]
                    .str.split('(').str[0]
                    .str.strip())
    short = (db.replace('TF_Perturbations_Followed_by_Expression', 'TF Perturbations')
               .replace('ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X', 'ENCODE+ChEA')
               .replace('ENCODE_TF_ChIP-seq_2015', 'ENCODE ChIP')
               .replace('TRANSFAC_and_JASPAR_PWMs', 'TRANSFAC+JASPAR')
               .replace('ChEA_2022', 'ChEA 2022'))
    panels.append((short, sig, use_fdr, COLORS[len(panels) % len(COLORS)]))

if not panels:
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    ax.text(0.5, 0.5, 'No Enrichr results available',
            ha='center', va='center', fontsize=13,
            transform=ax.transAxes, color='#666')
    ax.set_axis_off()
else:
    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(8.5 * n, 9), dpi=300)
    if n == 1:
        axes = [axes]

    for ax, (label, df, use_fdr, color) in zip(axes, panels):
        bars = ax.barh(df['Label'], df['-log10P'],
                       color=color, edgecolor='white', height=0.65)

        # Overlap fraction annotations inside bars
        if 'Overlap' in df.columns:
            for bar, (_, row) in zip(bars, df.iterrows()):
                w = bar.get_width()
                if w > 0.4:
                    ax.text(w * 0.97,
                            bar.get_y() + bar.get_height() / 2,
                            str(row['Overlap']),
                            va='center', ha='right',
                            fontsize=7.5, color='white', fontweight='bold')

        # Significance line
        fdr_x = -np.log10(0.05)
        ax.axvline(fdr_x, color='#E74C3C', linestyle='--',
                   lw=1.2, label='p = 0.05')
        ax.legend(fontsize=9, loc='lower right')

        if not use_fdr:
            ax.text(0.97, 0.02,
                    'Top 15 by nominal p-value\n'
                    '(n=100 genes; FDR not reached)',
                    transform=ax.transAxes, fontsize=7.5,
                    ha='right', va='bottom',
                    color='#888', style='italic')

        ax.set_xlabel('-log₁₀(p-value)', fontsize=12, fontweight='bold')
        ax.set_title(
            f'TF Enrichment — {label}\n100 high-confidence CHD8 targets',
            fontsize=11, fontweight='bold', pad=10)
        ax.tick_params(axis='y', labelsize=9)
        sns.despine(ax=ax)

plt.suptitle(
    "Supplementary Figure: TF Co-binding at CHD8-Bound,\n"
    "ATAC-Altered, Transcriptionally Dysregulated Loci (NPC)",
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.subplots_adjust(left=0.22)

FIG_PATH = os.path.join(SAVE_DIR,
    "SuppFig_TF_Motif_Enrichment_100genes_600DPI.png")
PDF_PATH = os.path.join(SAVE_DIR,
    "SuppFig_TF_Motif_Enrichment_100genes_vector.pdf")
plt.savefig(FIG_PATH, dpi=600, bbox_inches='tight')
plt.savefig(PDF_PATH, bbox_inches='tight')
plt.show()
plt.close()
print(f"  Figure saved: {FIG_PATH}")

# ── STEP 6: Save tables ───────────────────────────────────────────
print("\nStep 6: Saving enrichment tables...")
for db, res in enrichr_results.items():
    out_path = os.path.join(
        MOTIF_DIR,
        f"TF_Motif_{db.replace('-','_').replace(' ','_')}.csv"
    )
    res.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}")

# ── SUMMARY ───────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("COMPLETE")
print("=" * 65)
print(f"  Triple-overlap genes analysed : {len(triple_genes):,}")
print(f"  CHD8 peaks used               : {triple_peaks.count():,}")
print(f"  HOMER                         : skipped (Colab limitation)")
for db, res in enrichr_results.items():
    n_fdr = (res['Adjusted P-value'] < 0.05).sum()
    n_nom = (res['P-value'] < 0.05).sum()
    print(f"  {db[:50]:<50} "
          f"{n_fdr} FDR<0.05 | {n_nom} nom p<0.05")

print("\nKey findings for -:")
if 'TF_Perturbations_Followed_by_Expression' in enrichr_results:
    res = enrichr_results['TF_Perturbations_Followed_by_Expression']
    sig = res[res['Adjusted P-value'] < 0.05].nsmallest(5, 'Adjusted P-value')
    if not sig.empty:
        print("  FDR-significant TF perturbations:")
        for _, row in sig.iterrows():
            print(f"    {row['Term']}  "
                  f"(FDR={row['Adjusted P-value']:.4f}, overlap={row['Overlap']})")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import os

SAVE_DIR = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Verified values from pipeline ────────────────────────────
PCT_H3K4_ES  = 100 - 29.8   # 70.2%
PCT_ONLY_ES  = 29.8
PCT_H3K4_NPC = 56.5
PCT_ONLY_NPC = 43.5
N_NPC_TOTAL  = 48735
N_ONLY_NPC   = 21181
N_H3K4_NPC   = 27554
N_ONLY_ES    = 9549
N_ES_TOTAL   = 32017
PCT_DIST     = 96.1
PCT_PROX     = 3.9
N_DIST       = 20352
N_PROX       = 829

# ── Figure setup ──────────────────────────────────────────────
plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size'      : 11,
    'pdf.fonttype'   : 42,
})

fig = plt.figure(figsize=(5.5, 6.5), dpi=600)
gs  = gridspec.GridSpec(
    2, 1,
    height_ratios=[5, 1],
    hspace=0.05,
)
ax      = fig.add_subplot(gs[0])
ax_leg  = fig.add_subplot(gs[1])
ax_leg.axis('off')

# ── Stacked bars ──────────────────────────────────────────────
conditions = ['ES', 'NPC']
pct_h3k4s  = [PCT_H3K4_ES,  PCT_H3K4_NPC]
pct_onlys  = [PCT_ONLY_ES,  PCT_ONLY_NPC]
n_totals   = [N_ES_TOTAL,   N_NPC_TOTAL]
n_onlys    = [N_ONLY_ES,    N_ONLY_NPC]

x = np.arange(len(conditions))
w = 0.52

b1 = ax.bar(x, pct_h3k4s, w,
            color='#1f77b4', edgecolor='black', linewidth=0.8,
            zorder=3)
b2 = ax.bar(x, pct_onlys, w,
            bottom=pct_h3k4s,
            color='#d62728', edgecolor='black', linewidth=0.8,
            zorder=3)

# ── Value labels inside bars ──────────────────────────────────
for i, (h, o, nt, no) in enumerate(
        zip(pct_h3k4s, pct_onlys, n_totals, n_onlys)):
    # H3K4me3 label
    ax.text(i, h / 2,
            f'{h:.1f}%\n(n={nt-no:,})',
            ha='center', va='center',
            fontsize=9.5, fontweight='bold', color='white', zorder=4)
    # CHD8-only label
    ax.text(i, h + o / 2,
            f'{o:.1f}%\n(n={no:,})',
            ha='center', va='center',
            fontsize=9.5, fontweight='bold', color='white', zorder=4)

# ── Annotation arrow for NPC distal fraction ──────────────────
ax.annotate(
    f'{PCT_DIST:.0f}% distal\n{PCT_PROX:.0f}% promoter',
    xy=(1, PCT_H3K4_NPC + PCT_ONLY_NPC / 2),  # centre of red NPC bar
    xytext=(1.55, 78),
    fontsize=8.5, color='#7f0000', va='center',
    arrowprops=dict(arrowstyle='->', color='#d62728', lw=0.9),
    bbox=dict(boxstyle='round,pad=0.35', fc='#fff0f0',
              ec='#d62728', lw=0.8),
    zorder=5,
)

# ── Axes styling ──────────────────────────────────────────────
ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=13, fontweight='bold')
ax.set_ylabel('Percentage of CHD8 peaks (%)',
              fontsize=11, fontweight='bold')
ax.set_ylim(0, 108)
ax.set_xlim(-0.5, 2.2)          # extra right margin for annotation
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, linewidth=0.4, color='#dddddd', zorder=0)
ax.set_axisbelow(True)

ax.set_title(
    'Genomic distribution of CHD8 peaks\nby H3K4me3 co-occupancy state',
    fontsize=12, fontweight='bold',
    pad=10,           # space between title and top of plot
    loc='center',
)

blue_patch = mpatches.Patch(facecolor='#1f77b4', edgecolor='black',
                             linewidth=0.8,
                             label='CHD8 + H3K4me3 (H3K4me3 co-occupied)')
red_patch  = mpatches.Patch(facecolor='#d62728', edgecolor='black',
                             linewidth=0.8,
                             label='CHD8-only (no H3K4me3 signal)')
ax_leg.legend(
    handles=[blue_patch, red_patch],
    loc='center',
    ncol=1,             # one entry per row — no squeezing
    fontsize=10,
    frameon=True,
    framealpha=0.0,
    edgecolor='none',
    handlelength=1.4,
    handleheight=1.0,
    borderpad=0.4,
    labelspacing=0.5,
)

# ── Save ──────────────────────────────────────────────────────
png = os.path.join(SAVE_DIR, 'CHD8_only_peaks_FIXED_600DPI.png')
pdf = os.path.join(SAVE_DIR, 'CHD8_only_peaks_FIXED_vector.pdf')
plt.savefig(png, dpi=600, bbox_inches='tight')
plt.savefig(pdf, bbox_inches='tight')
plt.show()
plt.close()

print(f'✅ Saved: {png}')
print()
print('Figure legend text (for -):')
print()
print('Figure X. Genomic distribution of CHD8 ChIP-seq peaks by H3K4me3 co-occupancy.')
print('Stacked bar chart showing the proportion of CHD8 peaks in ES cells (n = 32,017)')
print('and NPCs (n = 48,735) classified as CHD8+H3K4me3 co-occupied (blue; peaks')
print('overlapping H3K4me3 ChIP-seq signal by ≥1 bp) or CHD8-only lacking H3K4me3')
print('signal (red). Of NPC CHD8-only peaks (n = 21,181; 43.5%), 96.1% localise')
print('outside annotated TSS ±2 kb windows (distal regulatory elements) and 3.9%')
print('overlap promoters. Peak counts (n) are annotated within each bar segment.')
print('Standard chromosomes only (chr1–chr19, chrX, chrY; mm10).')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
import textwrap

# Set quality defaults
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['pdf.fonttype'] = 42

# 1. LOAD DATA (Ensuring proper variable naming)
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
files = {
    "KO": base_path + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    "KD1": base_path + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv",
    "KD2": base_path + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv"
}

def clean_df(path):
    df = pd.read_csv(path, sep='\t')
    df = df[['GENESYMBOL', 'log2FoldChange', 'padj']].dropna()
    df.rename(columns={'GENESYMBOL': 'symbol'}, inplace=True)
    return df

df_ko = clean_df(files["KO"])
df_kd1 = clean_df(files["KD1"])
df_kd2 = clean_df(files["KD2"])

# 2. IDENTIFY DEGS (Using cutoffs consistent)
fdr_cutoff, lfc_cutoff = 0.05, 0.5

def categorize(row):
    if row['padj'] >= fdr_cutoff: return 'NS'
    if row['log2FoldChange'] < -lfc_cutoff: return 'Down'
    if row['log2FoldChange'] > lfc_cutoff: return 'Up'
    return 'NS'

for df in [df_ko, df_kd1, df_kd2]:
    df['category'] = df.apply(categorize, axis=1)

for name, df in zip(["KO", "KD1", "KD2"], [df_ko, df_kd1, df_kd2]):
    print(name)
    print(df['category'].value_counts())
    print()

# 3. PANEL A: VOLCANO PLOTS
fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=600)
colors = {'NS': '#d3d3d3', 'Down': '#2E86AB', 'Up': '#A23B72'} # Magenta and Blue per legend
titles = ['CHD8 KO vs WT', 'CHD8 KD 8.1 vs Scramble', 'CHD8 KD 8.2 vs Scramble']

for i, (df, title) in enumerate(zip([df_ko, df_kd1, df_kd2], titles)):
    ax = axes[i]
    for cat in ['NS', 'Up', 'Down']:
        subset = df[df['category'] == cat]
        ax.scatter(subset['log2FoldChange'], -np.log10(subset['padj']),
                   c=colors[cat], s=12, alpha=0.7, label=f"{cat} (n={len(subset)})", rasterized=True)

    ax.axhline(-np.log10(fdr_cutoff), color='black', linestyle='--', lw=1)
    ax.axvline(-lfc_cutoff, color='black', linestyle='--', lw=1)
    ax.axvline(lfc_cutoff, color='black', linestyle='--', lw=1)
    ax.set_title(title, fontweight='bold', fontsize=16)
    ax.set_xlabel('log$_2$ Fold Change', fontweight='bold', fontsize=14)
    if i == 0: ax.set_ylabel('-log$_{10}$ Adjusted P-value', fontweight='bold', fontsize=14)
    ax.legend(frameon=False, loc='upper right', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Figure1A_Volcano_600DPI.png', dpi=600)
plt.show()

# 4. PANEL B: GO ENRICHMENT (With Text Wrapping & Gene Counts)
print("Running GO Enrichment for KO Downregulated Genes...")
ko_down_genes = df_ko[df_ko['category'] == 'Down']['symbol'].unique().tolist()
enr = gp.enrichr(gene_list=ko_down_genes, gene_sets=['GO_Biological_Process_2023'], organism='mouse')

# Process for plotting
sig_go = enr.results[enr.results['Adjusted P-value'] < 0.05].head(10).copy()
sig_go['-log10FDR'] = -np.log10(sig_go['Adjusted P-value'])

def format_go_label(term, overlap):
    clean_term = term.split(' (GO')[0]
    n_count = overlap.split('/')[0]
    wrapped_term = "\n".join(textwrap.wrap(clean_term, width=35))
    return f"{wrapped_term}\n(n={n_count})"

sig_go['Display_Label'] = sig_go.apply(lambda x: format_go_label(x['Term'], x['Overlap']), axis=1)
sig_go = sig_go.sort_values('-log10FDR', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10), dpi=600)
ax.barh(sig_go['Display_Label'], sig_go['-log10FDR'], color='#2E86AB', edgecolor='white', height=0.6)

ax.set_yticklabels(sig_go['Display_Label'], fontsize=12, fontweight='bold', va='center')
ax.set_xlabel('-log$_{10}$ (FDR)', fontweight='bold', fontsize=14)
ax.set_title('Top Biological Processes (KO Downregulated)', fontweight='bold', fontsize=16, pad=20)
ax.axvline(-np.log10(0.05), color='red', linestyle='--', lw=2)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.subplots_adjust(left=0.4)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gseapy as gp

# 1. LOAD DATA
si_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Scrambled_siRNA_Diff_Chd8_siRNA.tsv"
si_df = pd.read_csv(si_path, sep="\t").dropna(subset=['log2FoldChange', 'padj'])

# 2. GENERATE RANKING
np.random.seed(42)
jitter = np.random.uniform(low=1e-12, high=1e-10, size=len(si_df))
si_df['rank_metric'] = (-np.log10(si_df['padj'] + 1e-300) * np.sign(si_df['log2FoldChange'])) + jitter

# Use GENESYMBOL or SYMBOL based on file structure
gene_col = 'GENESYMBOL' if 'GENESYMBOL' in si_df.columns else 'symbol'
rnk = si_df.sort_values('rank_metric', ascending=False).set_index(gene_col)['rank_metric']

# 3. RUN GSEA
print("🧬 Running GSEA for CHD8 KD...")
pre_res = gp.prerank(
    rnk=rnk,
    gene_sets='GO_Biological_Process_2023',
    organism='Mouse',
    permutation_num=1000,
    seed=42
)

# 4. EXTRACT DATA
res_df = pre_res.res2d.sort_values("FDR q-val")
top_row = res_df.iloc[0]
top_term = top_row['Term']
nes = top_row['NES']
fdr = top_row['FDR q-val']

# Robust gene count extraction
try:
    # Attempt 1: Check for 'Genes' column in the summary table
    gene_list_col = next((c for c in res_df.columns if c.lower() == 'genes'), None)
    if gene_list_col and pd.notna(top_row[gene_list_col]):
        gene_count = len([g for g in str(top_row[gene_list_col]).split(';') if g])
    else:
        # Attempt 2: Use the size of the leading edge from results
        gene_count = pre_res.results[top_term].get('tag_size', 0)
        if gene_count == 0:
            # Attempt 3: Look into the hits/indices
            gene_count = len(pre_res.results[top_term].get('hits', []))
except Exception:
    # Final Fallback: If all else fails, show 0 rather than crashing
    gene_count = "N/A"

print(f"\n✅ Top Enriched Pathway: {top_term}")
print(f"📊 NES: {nes:.3f} | FDR: {fdr:.3e} | Gene Count: {gene_count}")

# 5. PLOT GSEA (Figure 1C)
# Passing the result dictionary as keyword arguments
gp.plot.gseaplot(
    rank_metric=pre_res.ranking,
    term=top_term,
    **pre_res.results[top_term],
    figsize=(10, 8)
)

# Explicitly setting title
plt.title(f"GSEA: {top_term}\n(NES={nes:.2f}, FDR={fdr:.2e}, n={gene_count})",
          fontweight='bold', fontsize=14)

# 6. SAVE OUTPUTS (600 DPI & Supplementary Table)
fig = plt.gcf()

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import os

# 1. HARD-MAPPED PATHS
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"

paths = {
    "KO": os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"),
    "KD81": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv"),
    "KD82": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")
}

# 2. Preparation Function with robust column detection
def prep_df(path, name):
    df = pd.read_csv(path, sep="\t")
    # Identify gene column (MGI_symbol, GENESYMBOL, or Symbol)
    gene_col = next((c for c in df.columns if c.upper() in ["GENESYMBOL", "SYMBOL", "MGI_SYMBOL", "GENE"]), None)
    if gene_col is None:
        print(f"⚠️ Warning: No gene column found in {os.path.basename(path)}. Using index.")
        df['GENESYMBOL'] = df.index
    else:
        df = df.rename(columns={gene_col: 'GENESYMBOL'})

    return (
        df[['GENESYMBOL', 'log2FoldChange']]
        .dropna()
        .rename(columns={'log2FoldChange': f'{name}_LFC'})
    )

# 3. Merge Datasets
merged = prep_df(paths["KO"], "KO")
merged = merged.merge(prep_df(paths["KD81"], "KD81"), on='GENESYMBOL')
merged = merged.merge(prep_df(paths["KD82"], "KD82"), on='GENESYMBOL')
merged = merged.dropna()

# 4. Styling & Plotting
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 12
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

def plot_correlation(ax, x_col, y_col, title, color):
    x_data = merged[x_col]
    y_data = merged[y_col]
    r, p = pearsonr(x_data, y_data)

    # Square scaling for visual comparison
    all_vals = np.concatenate([x_data, y_data])
    base_lim = np.max(np.abs(all_vals)) * 1.1
    ax.set_xlim(-base_lim, base_lim)
    ax.set_ylim(-base_lim, base_lim)

    # Identity line (y=x)
    ax.plot([-base_lim, base_lim], [-base_lim, base_lim],
            color='black', linestyle='--', alpha=0.4, zorder=1)

    # Fixed regplot: zorder moved inside kws dictionaries
    sns.regplot(data=merged, x=x_col, y=y_col, ax=ax,
                scatter_kws={'alpha':0.2, 's':10, 'color':'gray', 'zorder': 2},
                line_kws={'color':color, 'lw':2.5, 'zorder': 3})

    ax.set_title(f"{title}\n$r$ = {r:.2f} (p < 0.001)", fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.6)

# 5. Execute & Save
plot_correlation(ax1, 'KO_LFC', 'KD81_LFC', "CHD8 KO vs Clone 8.1", "#D62728")
plot_correlation(ax2, 'KD81_LFC', 'KD82_LFC', "Clone 8.1 vs Clone 8.2", "#1F77B4")

plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import os

# 1. HARD-MAPPED PATHS
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"

paths = {
    "KO": os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"),
    "KD81": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv"),
    "KD82": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")
}

# 2. Preparation Function with robust column detection
def prep_df(path, name):
    df = pd.read_csv(path, sep="\t")
    # Identify gene column (MGI_symbol, GENESYMBOL, or Symbol)
    gene_col = next((c for c in df.columns if c.upper() in ["GENESYMBOL", "SYMBOL", "MGI_SYMBOL", "GENE"]), None)
    if gene_col is None:
        print(f"⚠️ Warning: No gene column found in {os.path.basename(path)}. Using index.")
        df['GENESYMBOL'] = df.index
    else:
        df = df.rename(columns={gene_col: 'GENESYMBOL'})

    return (
        df[['GENESYMBOL', 'log2FoldChange']]
        .dropna()
        .rename(columns={'log2FoldChange': f'{name}_LFC'})
    )

# 3. Merge Datasets
merged = prep_df(paths["KO"], "KO")
merged = merged.merge(prep_df(paths["KD81"], "KD81"), on='GENESYMBOL')
merged = merged.merge(prep_df(paths["KD82"], "KD82"), on='GENESYMBOL')
merged = merged.dropna()

# 4. Styling & Plotting
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 12
})

# Adjusted to a 1x3 grid and widened the figure to accommodate the 3rd plot comfortably
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 7))

def plot_correlation(ax, x_col, y_col, title, color):
    x_data = merged[x_col]
    y_data = merged[y_col]
    r, p = pearsonr(x_data, y_data)

    # Square scaling for visual comparison
    all_vals = np.concatenate([x_data, y_data])
    base_lim = np.max(np.abs(all_vals)) * 1.1
    ax.set_xlim(-base_lim, base_lim)
    ax.set_ylim(-base_lim, base_lim)

    # Identity line (y=x)
    ax.plot([-base_lim, base_lim], [-base_lim, base_lim],
            color='black', linestyle='--', alpha=0.4, zorder=1)

    # Fixed regplot: zorder moved inside kws dictionaries
    sns.regplot(data=merged, x=x_col, y=y_col, ax=ax,
                scatter_kws={'alpha':0.2, 's':10, 'color':'gray', 'zorder': 2},
                line_kws={'color':color, 'lw':2.5, 'zorder': 3})

    ax.set_title(f"{title}\n$r$ = {r:.2f} (p < 0.001)", fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.6)

# 6. Save Publication-Ready High-Resolution Vector & Raster Images
output_dir = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
os.makedirs(output_dir, exist_ok=True)

pdf_output = os.path.join(output_dir, "Figure3A_GenomeBiology.pdf")
tiff_output = os.path.join(output_dir, "Figure3A_GenomeBiology.tiff")

# Save vector PDF format
plt.savefig(pdf_output, format='pdf', dpi=600, bbox_inches='tight')

# Save lossless 600 DPI LZW compressed TIFF layout
plt.savefig(tiff_output, format='tiff', dpi=600, bbox_inches='tight', pil_kwargs={"compression": "tiff_lzw"})

plt.close()
print(f"✅ - compliant files successfully saved:\n -> {pdf_output}\n -> {tiff_output}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import os

# 1. HARD-MAPPED PATHS
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"

paths = {
    "KO": os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"),
    "KD81": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv"),
    "KD82": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")
}

# 2. Preparation Function with robust column detection
def prep_df(path, name):
    df = pd.read_csv(path, sep="\t")
    gene_col = next((c for c in df.columns if c.upper() in ["GENESYMBOL", "SYMBOL", "MGI_SYMBOL", "GENE"]), None)
    if gene_col is None:
        print(f"⚠️ Warning: No gene column found in {os.path.basename(path)}. Using index.")
        df['GENESYMBOL'] = df.index
    else:
        df = df.rename(columns={gene_col: 'GENESYMBOL'})

    return (
        df[['GENESYMBOL', 'log2FoldChange']]
        .dropna()
        .rename(columns={'log2FoldChange': f'{name}_LFC'})
    )

# 3. Merge Datasets
merged = prep_df(paths["KO"], "KO")
merged = merged.merge(prep_df(paths["KD81"], "KD81"), on='GENESYMBOL')
merged = merged.merge(prep_df(paths["KD82"], "KD82"), on='GENESYMBOL')
merged = merged.dropna()

# 4. Styling
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 12
})

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 7))

def plot_correlation(ax, x_col, y_col, title, color):
    x_data = merged[x_col]
    y_data = merged[y_col]
    r, p = pearsonr(x_data, y_data)

    all_vals = np.concatenate([x_data, y_data])
    base_lim = np.max(np.abs(all_vals)) * 1.1
    ax.set_xlim(-base_lim, base_lim)
    ax.set_ylim(-base_lim, base_lim)

    ax.plot([-base_lim, base_lim], [-base_lim, base_lim],
            color='black', linestyle='--', alpha=0.4, zorder=1)

    sns.regplot(data=merged, x=x_col, y=y_col, ax=ax,
                scatter_kws={'alpha':0.2, 's':10, 'color':'gray', 'zorder': 2},
                line_kws={'color':color, 'lw':2.5, 'zorder': 3})

    ax.set_title(f"{title}\n$r$ = {r:.2f} (p < 0.001)", fontsize=14, fontweight='bold')
    ax.set_xlabel(x_col.replace('_LFC', ' log2FC'))
    ax.set_ylabel(y_col.replace('_LFC', ' log2FC'))
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.6)

# 5. Generate the three panels
plot_correlation(ax1, 'KO_LFC', 'KD81_LFC', 'KO vs Chd8 shRNA 8.1', '#1f77b4')
plot_correlation(ax2, 'KD81_LFC', 'KD82_LFC', 'Chd8 shRNA 8.1 vs 8.2', '#2ca02c')
plot_correlation(ax3, 'KO_LFC', 'KD82_LFC', 'KO vs Chd8 shRNA 8.2', '#d62728')

plt.tight_layout()

# 6. Save Publication-Ready High-Resolution Vector & Raster Images
output_dir = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
os.makedirs(output_dir, exist_ok=True)

pdf_output = os.path.join(output_dir, "Figure3A_GenomeBiology.pdf")
tiff_output = os.path.join(output_dir, "Figure3A_GenomeBiology.tiff")

plt.savefig(pdf_output, format='pdf', dpi=600, bbox_inches='tight')
plt.savefig(tiff_output, format='tiff', dpi=600, bbox_inches='tight', pil_kwargs={"compression": "tiff_lzw"})

plt.close()
print(f"✅ - compliant files successfully saved:\n -> {pdf_output}\n -> {tiff_output}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import os

# 1. HARD-MAPPED PATHS
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"

paths = {
    "KO": os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"),
    "KD81": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv"),
    "KD82": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")
}

# 2. Preparation Function with robust column detection
def prep_df(path, name):
    df = pd.read_csv(path, sep="\t")
    # Identify gene column (MGI_symbol, GENESYMBOL, or Symbol)
    gene_col = next((c for c in df.columns if c.upper() in ["GENESYMBOL", "SYMBOL", "MGI_SYMBOL", "GENE"]), None)
    if gene_col is None:
        print(f"⚠️ Warning: No gene column found in {os.path.basename(path)}. Using index.")
        df['GENESYMBOL'] = df.index
    else:
        df = df.rename(columns={gene_col: 'GENESYMBOL'})

    # Drop duplicate gene symbols to prevent explosive many-to-many merges
    if df['GENESYMBOL'].duplicated().any():
        dup_count = df['GENESYMBOL'].duplicated().sum()
        print(f"[{name}] WARNING: {dup_count} duplicate GENESYMBOL entries found; keeping the first occurrence.")
        df = df.drop_duplicates(subset=['GENESYMBOL'], keep='first')

    return (
        df[['GENESYMBOL', 'log2FoldChange']]
        .dropna()
        .rename(columns={'log2FoldChange': f'{name}_LFC'})
    )

# 3. Merge Datasets
merged = prep_df(paths["KO"], "KO")
merged = merged.merge(prep_df(paths["KD81"], "KD81"), on='GENESYMBOL')
merged = merged.merge(prep_df(paths["KD82"], "KD82"), on='GENESYMBOL')
merged = merged.dropna()

print(f"🧬 Successfully merged datasets! Total intersecting genes to plot: {len(merged)}")

if len(merged) == 0:
    raise ValueError("❌ Error: The merged dataset is completely empty! Check if gene symbol columns match names.")

# 4. Styling & Plotting Configurations (- standard specs)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
})

# 170 mm max full-page width = ~6.7 inches. Fits nicely in a 3-panel single row layout.
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(6.7, 2.5))

def plot_correlation(ax, x_col, y_col, panel_letter, title, color):
    x_data = merged[x_col]
    y_data = merged[y_col]
    r, p = pearsonr(x_data, y_data)

    # Square scaling for clean visual comparison
    all_vals = np.concatenate([x_data, y_data])
    base_lim = np.max(np.abs(all_vals)) * 1.1
    ax.set_xlim(-base_lim, base_lim)
    ax.set_ylim(-base_lim, base_lim)

    # Identity reference line (y=x)
    ax.plot([-base_lim, base_lim], [-base_lim, base_lim],
            color='black', linestyle='--', alpha=0.3, lw=0.5, zorder=1)

    # Fixed regplot with explicit element layers
    sns.regplot(data=merged, x=x_col, y=y_col, ax=ax,
                scatter_kws={'alpha':0.15, 's':1, 'color':'gray', 'zorder': 2},
                line_kws={'color':color, 'lw':1.2, 'zorder': 3})

    # Uppercase bold panel indicators (A, B, C) for journal layout guidelines
    ax.text(-0.22, 1.08, panel_letter, transform=ax.transAxes, fontsize=12, fontweight='bold', va='top', ha='right')

    # Text statistics box
    stats_text = f"$r$ = {r:.2f}\n$p$ < 0.001\n$n$ = {len(merged):,}"
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=7,
            va='top', ha='left', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'))

    ax.set_title(title, fontsize=8, fontweight='bold', pad=8)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.5, lw=0.5)

# =========================================================================
# 5. EXECUTE PLOTTING CALLS
# =========================================================================
plot_correlation(ax1, 'KO_LFC', 'KD81_LFC', 'A', "CHD8 KO vs Clone 8.1", "#D62728")
plot_correlation(ax2, 'KO_LFC', 'KD82_LFC', 'B', "CHD8 KO vs Clone 8.2", "#D62728")
plot_correlation(ax3, 'KD81_LFC', 'KD82_LFC', 'C', "Clone 8.1 vs Clone 8.2", "#1F77B4")

# Set clean, formalized axis titles
ax1.set_xlabel("KO $\log_2$ Fold Change")
ax1.set_ylabel("Clone 8.1 $\log_2$ Fold Change")

ax2.set_xlabel("KO $\log_2$ Fold Change")
ax2.set_ylabel("Clone 8.2 $\log_2$ Fold Change")

ax3.set_xlabel("Clone 8.1 $\log_2$ Fold Change")
ax3.set_ylabel("Clone 8.2 $\log_2$ Fold Change")

plt.tight_layout()

# =========================================================================
# 6. Save Publication-Ready High-Resolution Vector & Raster Images
# =========================================================================
output_dir = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
os.makedirs(output_dir, exist_ok=True)

pdf_output = os.path.join(output_dir, "Figure3A_GenomeBiology.pdf")
tiff_output = os.path.join(output_dir, "Figure3A_GenomeBiology.tiff")

# Save vector PDF format
plt.savefig(pdf_output, format='pdf', dpi=600, bbox_inches='tight')

# Save lossless 600 DPI LZW compressed TIFF layout
plt.savefig(tiff_output, format='tiff', dpi=600, bbox_inches='tight', pil_kwargs={"compression": "tiff_lzw"})

plt.close()
print(f"✅ - compliant files successfully saved with data populated:\n -> {pdf_output}\n -> {tiff_output}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import numpy as np
import os

# 1. HARD-MAPPED PATHS
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"

paths = {
    "KO": os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"),
    "KD81": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv"),
    "KD82": os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")
}

# 2. Preparation Function with robust column detection
def prep_df(path, name):
    df = pd.read_csv(path, sep="\t")
    # Identify gene column (MGI_symbol, GENESYMBOL, or Symbol)
    gene_col = next((c for c in df.columns if c.upper() in ["GENESYMBOL", "SYMBOL", "MGI_SYMBOL", "GENE"]), None)
    if gene_col is None:
        print(f"⚠️ Warning: No gene column found in {os.path.basename(path)}. Using index.")
        df['GENESYMBOL'] = df.index
    else:
        df = df.rename(columns={gene_col: 'GENESYMBOL'})

    # Drop duplicate gene symbols to prevent explosive many-to-many merges
    if df['GENESYMBOL'].duplicated().any():
        dup_count = df['GENESYMBOL'].duplicated().sum()
        print(f"[{name}] WARNING: {dup_count} duplicate GENESYMBOL entries found; keeping the first occurrence.")
        df = df.drop_duplicates(subset=['GENESYMBOL'], keep='first')

    return (
        df[['GENESYMBOL', 'log2FoldChange']]
        .dropna()
        .rename(columns={'log2FoldChange': f'{name}_LFC'})
    )

# 3. Merge Datasets
merged = prep_df(paths["KO"], "KO")
merged = merged.merge(prep_df(paths["KD81"], "KD81"), on='GENESYMBOL')
merged = merged.merge(prep_df(paths["KD82"], "KD82"), on='GENESYMBOL')
merged = merged.dropna()

# 4. - Styling & Typography Configurations
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
})

# 170 mm max full-page width = ~6.7 inches. Using a 6.7 x 2.5 inch layout for perfectly square axes.
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(6.7, 2.5))

def plot_correlation(ax, x_col, y_col, panel_letter, title, color):
    x_data = merged[x_col]
    y_data = merged[y_col]
    r, p = pearsonr(x_data, y_data)

    # Consistent square boundary limits across all comparison axes
    all_vals = np.concatenate([x_data, y_data])
    base_lim = np.max(np.abs(all_vals)) * 1.1
    ax.set_xlim(-base_lim, base_lim)
    ax.set_ylim(-base_lim, base_lim)

    # Reference Identity line (y=x)
    ax.plot([-base_lim, base_lim], [-base_lim, base_lim],
            color='black', linestyle='--', alpha=0.3, lw=0.5, zorder=1)

    # Plot data points and regression trend line
    sns.regplot(data=merged, x=x_col, y=y_col, ax=ax,
                scatter_kws={'alpha':0.15, 's':1, 'color':'gray', 'zorder': 2},
                line_kws={'color':color, 'lw':1.2, 'zorder': 3})

    # Strict Journal Panel Labeling (uppercase bold 12pt)
    ax.text(-0.22, 1.08, panel_letter, transform=ax.transAxes, fontsize=12, fontweight='bold', va='top', ha='right')

    # Statistical Summary Overlay
    stats_text = f"$r$ = {r:.2f}\n$p$ < 0.001\n$n$ = {len(merged):,}"
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=7,
            va='top', ha='left', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='none'))

    ax.set_title(title, fontsize=8, fontweight='bold', pad=8)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.5, lw=0.5)

# 5. Execute Individual Axis Subplots
plot_correlation(ax1, 'KO_LFC', 'KD81_LFC', 'A', "CHD8 KO vs Clone 8.1", "#D62728")
plot_correlation(ax2, 'KO_LFC', 'KD82_LFC', 'B', "CHD8 KO vs Clone 8.2", "#D62728")
plot_correlation(ax3, 'KD81_LFC', 'KD82_LFC', 'C', "Clone 8.1 vs Clone 8.2", "#1F77B4")

# Structured explicit axis titles
ax1.set_xlabel("KO $\log_2$ Fold Change")
ax1.set_ylabel("Clone 8.1 $\log_2$ Fold Change")

ax2.set_xlabel("KO $\log_2$ Fold Change")
ax2.set_ylabel("Clone 8.2 $\log_2$ Fold Change")

ax3.set_xlabel("Clone 8.1 $\log_2$ Fold Change")
ax3.set_ylabel("Clone 8.2 $\log_2$ Fold Change")

plt.tight_layout()

# 6. Save Publication-Ready High-Resolution Vector & Raster Images
output_dir = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
os.makedirs(output_dir, exist_ok=True)

pdf_output = os.path.join(output_dir, "Figure3A_GenomeBiology.pdf")
tiff_output = os.path.join(output_dir, "Figure3A_GenomeBiology.tiff")

# Save vector PDF format
plt.savefig(pdf_output, format='pdf', dpi=600, bbox_inches='tight')

# Save lossless 600 DPI LZW compressed TIFF layout
plt.savefig(tiff_output, format='tiff', dpi=600, bbox_inches='tight', pil_kwargs={"compression": "tiff_lzw"})

plt.close()
print(f"✅ - compliant files successfully saved:\n -> {pdf_output}\n -> {tiff_output}")

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Load and Prep (Using the paths we verified earlier)
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
ko_path = os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")
kd1_path = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv")
kd2_path = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")

def load_and_flag(path):
    df = pd.read_csv(path, sep="\t")
    gene_col = next((c for c in df.columns if c.upper() in ["GENESYMBOL", "SYMBOL", "MGI_SYMBOL"]), "GENESYMBOL")
    df = df.rename(columns={gene_col: 'GENESYMBOL'})
    # Re-create the DOWN flag based on LFC and P-value if not present
    df['DEREG_FLAG'] = np.where((df['log2FoldChange'] < 0) & (df['padj'] < 0.05), 'DOWN', 'NONE')
    return df.set_index('GENESYMBOL')

ko_df = load_and_flag(ko_path)
kd81_df = load_and_flag(kd1_path)
kd82_df = load_and_flag(kd2_path)

# 2. Identify Shared DOWN Genes
shared_genes = list(set(ko_df[ko_df['DEREG_FLAG']=='DOWN'].index) &
                    set(kd81_df[kd81_df['DEREG_FLAG']=='DOWN'].index) &
                    set(kd82_df[kd82_df['DEREG_FLAG']=='DOWN'].index))

# 3. Build Heatmap Matrix (Top 30 by average magnitude)
heatmap_data = pd.DataFrame({
    'Chd8 KO': ko_df.loc[shared_genes, 'log2FoldChange'],
    'KD 8.1': kd81_df.loc[shared_genes, 'log2FoldChange'],
    'KD 8.2': kd82_df.loc[shared_genes, 'log2FoldChange']
}).dropna()

heatmap_data['mean_fc'] = heatmap_data.abs().mean(axis=1)
heatmap_data = heatmap_data.sort_values('mean_fc', ascending=False).head(30).drop(columns=['mean_fc'])

# 4. Generate Clustermap (Adds the Dendrograms reviewers expect)
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"]})

g = sns.clustermap(
    heatmap_data,
    cmap='RdBu_r',
    center=0,
    vmin=-3,
    vmax=0,
    row_cluster=True,   # Clusters genes by similarity
    col_cluster=False,  # Keeps KO -> KD8.1 -> KD8.2 order
    linewidths=0.5,
    linecolor='black',
    figsize=(8, 12),
    cbar_kws={'label': 'log2 Fold Change', 'shrink': 0.8}
)

# Refine labels
g.ax_heatmap.set_title('Top 30 Shared Downregulated Genes\n(Hierarchically Clustered)',
                       fontsize=14, fontweight='bold', pad=20)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=45)
g.ax_heatmap.set_ylabel("Genes", fontweight='bold')

# 5. Save to Drive at 600 DPI
save_path = "/content/drive/MyDrive/Chd8 data/Figure1E_Heatmap_Clustered_Final.png"
g.savefig(save_path, dpi=600, bbox_inches='tight')

print(f"✅ Success! Clustered Heatmap saved to: {save_path}")
plt.show()

In [ ]:
import pandas as pd
import os

print("🧬 Downloading mm10 Gene Annotations directly from UCSC...")

# Direct link to UCSC mm10 refGene table
url = "https://hgdownload.soe.ucsc.edu/goldenPath/mm10/database/refGene.txt.gz"

# Column names for RefGene
cols = ['bin', 'name', 'chrom', 'strand', 'txStart', 'txEnd', 'cdsStart', 'cdsEnd',
        'exonCount', 'exonStarts', 'exonEnds', 'score', 'name2', 'cdsStartStat',
        'cdsEndStat', 'exonFrames']

# Read the table
df = pd.read_csv(url, sep='\t', names=cols, compression='gzip')

# Define TSS: Start if +, End if -
df['TSS'] = df.apply(lambda x: x['txStart'] if x['strand'] == '+' else x['txEnd'], axis=1)

# Create 2kb window (+/- 1000bp)
tss_df = pd.DataFrame({
    'chrom': df['chrom'],
    'start': (df['TSS'] - 1000).clip(lower=0),
    'end': df['TSS'] + 1000,
    'gene': df['name2']
}).drop_duplicates()

# Save as BED
tss_file_path = "mm10_TSS_2kb.bed"
tss_df[['chrom', 'start', 'end', 'gene']].to_csv(tss_file_path, sep='\t', index=False, header=False)

print(f"✅ Created: {tss_file_path} with {len(tss_df)} unique TSS windows.")


In [ ]:
import pandas as pd

print("🧬 Creating mm10 promoter file...")

url = "https://hgdownload.soe.ucsc.edu/goldenPath/mm10/database/refGene.txt.gz"

cols = ['bin', 'name', 'chrom', 'strand', 'txStart', 'txEnd', 'cdsStart', 'cdsEnd',
        'exonCount', 'exonStarts', 'exonEnds', 'score', 'name2',
        'cdsStartStat', 'cdsEndStat', 'exonFrames']

df = pd.read_csv(url, sep='\t', names=cols, compression='gzip')

df = df[df['chrom'].str.contains("^chr[0-9XY]+$", regex=True)]

df['TSS'] = df['txStart']
df.loc[df['strand'] == '-', 'TSS'] = df['txEnd']

tss_df = pd.DataFrame({
    'chrom': df['chrom'],
    'start': (df['TSS'] - 1000).clip(lower=0),
    'end': df['TSS'] + 1000,
    'gene': df['name2']
}).drop_duplicates()

# 🔴 SAVE DIRECTLY TO DRIVE
save_path = "/content/drive/MyDrive/CHD8_Fig1/mm10_TSS_2kb.bed"

tss_df[['chrom', 'start', 'end', 'gene']].to_csv(
    save_path, sep='\t', index=False, header=False
)

print("✅ Saved promoter BED to Drive:")
print(save_path)


In [ ]:
import pybedtools
import pandas as pd
import subprocess

ES_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
NPC_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"

STANDARD_CHROMS = set(
    [f'chr{i}' for i in range(1, 20)] + ['chrX', 'chrY']
)

def is_standard(feature):
    return str(feature.chrom) in STANDARD_CHROMS

es_filt  = pybedtools.BedTool(ES_PATH).filter(is_standard).saveas()
npc_filt = pybedtools.BedTool(NPC_PATH).filter(is_standard).saveas()

# ── TEST 1: column count from first record ───────────────────────
print("TEST 1: BED file column counts")
first_es  = next(iter(es_filt))
first_npc = next(iter(npc_filt))
n_cols_es  = len(first_es.fields)
n_cols_npc = len(first_npc.fields)
print(f"  ES  columns : {n_cols_es}   example: {first_es}")
print(f"  NPC columns : {n_cols_npc}   example: {first_npc}")
print()

# ── TEST 2: load as dataframe using correct column count ─────────
print("TEST 2: Duplicate coordinates in NPC")
col_names = ['chr','start','end'] + [f'f{i}' for i in range(n_cols_npc - 3)]
npc_df = npc_filt.to_dataframe(names=col_names, dtype={'chr': str})
n_total = len(npc_df)
n_dupes = npc_df[['chr','start','end']].duplicated().sum()
print(f"  NPC rows (filtered)         : {n_total:,}")
print(f"  Duplicate chr/start/end     : {n_dupes:,}")
print()

col_names_es = ['chr','start','end'] + [f'f{i}' for i in range(n_cols_es - 3)]
es_df  = es_filt.to_dataframe(names=col_names_es, dtype={'chr': str})
n_es   = len(es_df)
n_dupes_es = es_df[['chr','start','end']].duplicated().sum()
print(f"  ES  rows (filtered)         : {n_es:,}")
print(f"  Duplicate chr/start/end     : {n_dupes_es:,}")
print()

# ── TEST 3: pure Python set arithmetic (no pybedtools) ───────────
print("TEST 3: Manual set intersection (exact coordinate match)")
es_set  = set(zip(es_df['chr'],  es_df['start'],  es_df['end']))
npc_set = set(zip(npc_df['chr'], npc_df['start'], npc_df['end']))

exact_shared   = len(es_set  & npc_set)
exact_es_only  = len(es_set  - npc_set)
exact_npc_only = len(npc_set - es_set)

print(f"  ES  unique coordinate sets  : {len(es_set):,}")
print(f"  NPC unique coordinate sets  : {len(npc_set):,}")
print(f"  Shared (exact match)        : {exact_shared:,}")
print(f"  ES-only                     : {exact_es_only:,}")
print(f"  NPC-only                    : {exact_npc_only:,}")
es_sum  = exact_es_only  + exact_shared
npc_sum = exact_npc_only + exact_shared
print(f"  ES  check: {exact_es_only} + {exact_shared} = {es_sum}  "
      f"(ES total = {len(es_set)})  "
      f"{'OK' if es_sum == len(es_set) else 'BROKEN'}")
print(f"  NPC check: {exact_npc_only} + {exact_shared} = {npc_sum}  "
      f"(NPC total = {len(npc_set)})  "
      f"{'OK' if npc_sum == len(npc_set) else 'BROKEN'}")
print()

# ── TEST 4: pybedtools v=True count directly ─────────────────────
print("TEST 4: pybedtools intersect counts")
es_clean  = es_filt.sort().merge()
npc_clean = npc_filt.sort().merge()

n_es_clean  = es_clean.count()
n_npc_clean = npc_clean.count()
n_shared    = es_clean.intersect(npc_clean, u=True).count()
n_npc_only  = npc_clean.intersect(es_clean, v=True).count()
n_es_only   = es_clean.intersect(npc_clean, v=True).count()

print(f"  es_clean  total : {n_es_clean:,}")
print(f"  npc_clean total : {n_npc_clean:,}")
print(f"  shared (u=True) : {n_shared:,}")
print(f"  npc_only(v=True): {n_npc_only:,}")
print(f"  es_only (v=True): {n_es_only:,}")
print(f"  NPC sum         : {n_npc_only + n_shared:,}  "
      f"(diff = {n_npc_only + n_shared - n_npc_clean:+,})")
print(f"  ES  sum         : {n_es_only + n_shared:,}  "
      f"(diff = {n_es_only + n_shared - n_es_clean:+,})")
print()

# ── TEST 5: inspect first 10 npc_only records ────────────────────
print("TEST 5: First 10 npc_only records from pybedtools v=True")
npc_only_bt = npc_clean.intersect(es_clean, v=True)
for i, feat in enumerate(npc_only_bt):
    print(f"  {feat.chrom}\t{feat.start}\t{feat.end}\t"
          f"n_fields={len(feat.fields)}")
    if i >= 9:
        break
print()

# ── TEST 6: raw bedtools intersect count via shell ────────────────
print("TEST 6: Raw bedtools wc -l check (shell, no Python wrapping)")
# Write temp sorted filtered BED files
npc_tmp = "/content/npc_clean_tmp.bed"
es_tmp  = "/content/es_clean_tmp.bed"
npc_clean.saveas(npc_tmp)
es_clean.saveas(es_tmp)

r1 = subprocess.run(f"wc -l {npc_tmp}", shell=True,
                    capture_output=True, text=True)
r2 = subprocess.run(
    f"bedtools intersect -a {npc_tmp} -b {es_tmp} -v | wc -l",
    shell=True, capture_output=True, text=True)
r3 = subprocess.run(
    f"bedtools intersect -a {npc_tmp} -b {es_tmp} -u | wc -l",
    shell=True, capture_output=True, text=True)

print(f"  Shell wc -l npc_clean       : {r1.stdout.strip()}")
print(f"  Shell bedtools -v (npc_only): {r2.stdout.strip()}")
print(f"  Shell bedtools -u (shared)  : {r3.stdout.strip()}")
npc_v_shell = int(r2.stdout.strip().split()[0])
npc_u_shell = int(r3.stdout.strip().split()[0])
npc_wc      = int(r1.stdout.strip().split()[0])
print(f"  Shell sum                   : {npc_v_shell + npc_u_shell:,}  "
      f"(npc total = {npc_wc:,}; diff = {npc_v_shell + npc_u_shell - npc_wc:+,})")

In [ ]:
import pandas as pd
import os
import pybedtools

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── System File Paths ────────────────────────────────────────────────────────
base_dir       = "/content/drive/MyDrive/CHD8_Fig1"
os.makedirs(base_dir, exist_ok=True)

npc_peaks_path = f"{base_dir}/peaks/CHD8_NPC_merged.bed"

h3k4_rep1_path = "/content/drive/MyDrive/Chd8 data/CHIPSEQ/MACS2_peaks/CHIP/H3K4me3_Undiff_1/H3K4me3_Undiff_1.merged_score_sorted.bed"
h3k4_rep2_path = "/content/drive/MyDrive/Chd8 data/CHIPSEQ/MACS2_peaks/CHIP/H3K4me3_Undiff_2/H3K4me3_Undiff_2.merged_score_sorted.bed"

tss_point_path = f"{base_dir}/mm10_TSS_point.bed"
chrom_sizes    = f"{base_dir}/mm10.chrom.sizes"

# ── STEP 0:
for stale_file in [tss_point_path, chrom_sizes]:
    if os.path.exists(stale_file):
        os.remove(stale_file)
        print(f"🗑️  Removed stale cached file: {stale_file}")

# ── STEP 1: Rebuild point-TSS file from UCSC refGene ─────────────────────────
print("🧬 Rebuilding point-TSS coordinates from UCSC refGene stream...")
url = "https://hgdownload.soe.ucsc.edu/goldenPath/mm10/database/refGene.txt.gz"
cols = ['bin', 'name', 'chrom', 'strand', 'txStart', 'txEnd', 'cdsStart', 'cdsEnd',
        'exonCount', 'exonStarts', 'exonEnds', 'score', 'name2', 'cdsStartStat',
        'cdsEndStat', 'exonFrames']
df = pd.read_csv(url, sep='\t', names=cols, compression='gzip')
df['TSS'] = df.apply(lambda x: x['txStart'] if x['strand'] == '+' else x['txEnd'], axis=1)

tss_point_df = pd.DataFrame({
    'chrom': df['chrom'],
    'start': df['TSS'],
    'end':   df['TSS'] + 1,
    'gene':  df['name2']
}).drop_duplicates()
tss_point_df[['chrom', 'start', 'end', 'gene']].to_csv(
    tss_point_path, sep='\t', index=False, header=False
)
print(f"   ✅ Rebuilt: {tss_point_path}  ({len(tss_point_df):,} TSS entries)")

# Sanity check: show a few raw chrom names as they come from UCSC before normalization
print("   Sample raw chrom names from UCSC refGene:", df['chrom'].unique()[:5])

with open(chrom_sizes, 'w') as f:
    for i in range(1, 20):
        f.write(f"{i}\t250000000\n")
    f.write("X\t170000000\nY\t100000000\n")

# ── STEP 2: Chromosome normalization (handles chr-prefixed AND RefSeq accessions) ──
REFSEQ_MAP = {
    "NC_000067.6": "1",  "NC_000068.7": "2",  "NC_000069.6": "3",  "NC_000070.6": "4",
    "NC_000071.6": "5",  "NC_000072.6": "6",  "NC_000073.6": "7",  "NC_000074.6": "8",
    "NC_000075.6": "9",  "NC_000076.6": "10", "NC_000077.6": "11", "NC_000078.6": "12",
    "NC_000079.6": "13", "NC_000080.6": "14", "NC_000081.6": "15", "NC_000082.6": "16",
    "NC_000083.6": "17", "NC_000084.6": "18", "NC_000085.6": "19",
    "NC_000086.7": "X",  "NC_000087.7": "Y"
}

STANDARD_CHROMS = set([str(i) for i in range(1, 20)] + ['X', 'Y'])

def clean_and_normalize(feature):
    chrom = str(feature.chrom)
    if chrom in REFSEQ_MAP:
        chrom = REFSEQ_MAP[chrom]
    elif chrom.startswith('chr'):          # handles UCSC-style 'chr1', 'chrX', etc.
        chrom = chrom[3:]
    feature.chrom = chrom
    return feature

def is_standard(feature):
    return str(feature.chrom) in STANDARD_CHROMS

# ── STEP 3: Multi-window comparison ──────────────────────────────────────────
WINDOWS_TO_TEST = {
    "2kb":  2000,
    "5kb":  5000,
    "10kb": 10000,
}

def run_fig1B_for_window(window_bp, label, npc_bed, h3k4_bed, tss_point_bed, total_npc):
    print(f"\n{'='*70}")
    print(f"  TSS WINDOW: ±{label} ({window_bp:,} bp upstream/downstream)")
    print(f"{'='*70}")

    tss_bed = tss_point_bed.slop(b=window_bp, g=chrom_sizes).sort().merge()

    promoter_bed = npc_bed.intersect(tss_bed, u=True).saveas()
    promoter     = promoter_bed.count()
    distal       = total_npc - promoter

    pct_promoter = promoter / total_npc * 100
    pct_distal   = distal   / total_npc * 100

    print(f"   Promoter Associated: {promoter:,}  ({pct_promoter:.1f}%)")
    print(f"   Distal Elements:     {distal:,}  ({pct_distal:.1f}%)")

    active = poised = None
    if h3k4_bed is not None:
        active = promoter_bed.intersect(h3k4_bed, u=True).count()
        poised = promoter - active
        pct_active = active / total_npc * 100
        pct_poised = poised / total_npc * 100
        h3k4_overlap_pct = active / promoter * 100 if promoter else 0

        print(f"   Active Promoters (H3K4me3+): {active:,}  ({pct_active:.1f}%)")
        print(f"   Poised Promoters (H3K4me3-): {poised:,}  ({pct_poised:.1f}%)")
        print(f"   Proportion of Promoter Elements with H3K4me3: {h3k4_overlap_pct:.1f}%")

    return {
        "window": label,
        "promoter": promoter, "pct_promoter": pct_promoter,
        "distal": distal, "pct_distal": pct_distal,
        "active": active, "poised": poised,
    }

def run_fig1B_comparison():
    if not os.path.exists(npc_peaks_path):
        print(f"❌ Core processing dependency [NPC peaks] missing at: {npc_peaks_path}")
        return

    print("\n📊 Standardizing and filtering CHD8 peak sets...")
    npc_bed = pybedtools.BedTool(npc_peaks_path).each(clean_and_normalize).filter(is_standard).sort().saveas()
    total_npc = npc_bed.count()
    print(f"   NPC peaks (std chroms): {total_npc:,}")

    tss_point_bed = pybedtools.BedTool(tss_point_path).each(clean_and_normalize).filter(is_standard).sort().saveas()
    total_tss = tss_point_bed.count()
    print(f"   TSS points (std chroms, post-normalization): {total_tss:,}")

    # ── Sanity check: confirm chrom names actually match between the two sets ──
    npc_chroms = sorted(set(f.chrom for f in npc_bed))
    tss_chroms = sorted(set(f.chrom for f in tss_point_bed))
    mismatched = set(npc_chroms) - set(tss_chroms)
    if mismatched:
        print(f"   ⚠️  WARNING: chroms present in CHD8 peaks but absent from TSS set: {mismatched}")
    else:
        print(f"   ✅ Chromosome naming matches between CHD8 peaks and TSS set.")

    h3k4_bed = None
    if os.path.exists(h3k4_rep1_path) and os.path.exists(h3k4_rep2_path):
        print("🕯️  Loading and generating consensus H3K4me3 profiles from biological replicates...")
        h3k4_r1 = pybedtools.BedTool(h3k4_rep1_path).each(clean_and_normalize).filter(is_standard).sort()
        h3k4_r2 = pybedtools.BedTool(h3k4_rep2_path).each(clean_and_normalize).filter(is_standard).sort()
        h3k4_bed = h3k4_r1.intersect(h3k4_r2, u=True).saveas()
    else:
        print("⚠️ Warning: Replicate H3K4me3 tracks not found. Skipping active/poised analysis.")

    results = []
    for label, window_bp in WINDOWS_TO_TEST.items():
        res = run_fig1B_for_window(window_bp, label, npc_bed, h3k4_bed, tss_point_bed, total_npc)
        results.append(res)

    print(f"\n{'='*70}")
    print("   SUMMARY: Comparative distribution vs literature milestones")
    print(f"   (Sugathan 2014 / Cotney 2015: majority promoter-bound;")
    print(f"    Ceballos-Chavez 2015: 48% at low stringency, up to ~78% at high)")
    print(f"{'='*70}")
    for r in results:
        print(f"   ±{r['window']:>4} Window:  promoter = {r['pct_promoter']:.1f}%   distal = {r['pct_distal']:.1f}%")

    return results

# ── Run Analysis ──────────────────────────────────────────────────────────────
results = run_fig1B_comparison()

In [ ]:
import os

# Define the base directory where MACS2 outputs might be
macs2_base_path = "/content/drive/MyDrive/Chd8 data/macs2/"

if os.path.exists(macs2_base_path):
    print("Found MACS2 directory! Listing all files/folders inside:")
    for root, dirs, files in os.walk(macs2_base_path):
        for file in files:
            # Look for raw MACS2 output formats
            if file.endswith(('.narrowPeak', '.broadPeak', '.xls', '.bed')):
                # Print the relative path to see where they are
                print(os.path.relpath(os.path.join(root, file), macs2_base_path))
else:
    print(f"Could not find directory at {macs2_base_path}. Check the spelling or path structure.")

In [ ]:
import pandas as pd

rep1_path = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/CHIP/Chd8_Undiff_1/Chd8_Undiff_1_peaks.narrowPeak"
rep2_path = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/CHIP/Chd8_Undiff_2/Chd8_Undiff_2_peaks.narrowPeak"

def check_peaks(path, name):
    df = pd.read_csv(path, sep="\t", header=None)
    print(f"=== {name} ===")
    print(f"Total raw peaks called: {len(df)}")
    print(f"Peaks with q < 0.05 (-log10q >= 1.3): {len(df[df[8] >= 1.301])}")
    print(f"Peaks with q < 0.01 (-log10q >= 2.0): {len(df[df[8] >= 2.0])}")
    print(f"Peaks with q < 0.001 (-log10q >= 3.0): {len(df[df[8] >= 3.0])}\n")
    return df

rep1 = check_peaks(rep1_path, "Replicate 1 (Undiff)")
rep2 = check_peaks(rep2_path, "Replicate 2 (Undiff)")

In [ ]:
%%bash
# 1. Filter replicates strictly (q < 0.01)
awk '$9 >= 2' "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/CHIP/Chd8_Undiff_1/Chd8_Undiff_1_peaks.narrowPeak" > rep1_strict.bed
awk '$9 >= 2' "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/CHIP/Chd8_Undiff_2/Chd8_Undiff_2_peaks.narrowPeak" > rep2_strict.bed

# 2. Find the overlap (reproducible peaks)
bedtools intersect -a rep1_strict.bed -b rep2_strict.bed -u > CHD8_NPC_strict_consensus.bed

# 3. Check the final clean peak count
wc -l CHD8_NPC_strict_consensus.bed

In [ ]:
!pip install pyranges

In [ ]:
import os

# Scan the main project folder to look for annotation metadata files
search_path = "/content/drive/MyDrive/Chd8 data/"

print("Searching for annotation/TSS files...")
found = False
for root, dirs, files in os.walk(search_path):
    for file in files:
        if any(x in file.lower() for x in ["tss", "refseq", "ensembl", "genes", "annotation"]) and file.endswith(('.bed', '.gtf', '.gff', '.txt')):
            print(os.path.join(root, file))
            found = True

if not found:
    print("Could not find explicit TSS files in '/Chd8 data/'. Let's search the wider Drive:")
    for root, dirs, files in os.walk("/content/drive/MyDrive/"):
        # Limit depth to keep it fast
        if root.count(os.sep) > 5:
            continue
        for file in files:
            if "tss" in file.lower() and file.endswith('.bed'):
                print(os.path.join(root, file))

In [ ]:
import pandas as pd

# Load files exactly as done before
chd8_df = pd.read_csv("CHD8_NPC_strict_consensus.bed", sep="\t", header=None).iloc[:, :3]
chd8_df.columns = ["Chromosome", "Start", "End"]

h3k4_df = pd.read_csv("/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/CHIP/H3K4me3_Undiff_1/H3K4me3_Undiff_1_peaks.narrowPeak", sep="\t", header=None).iloc[:, :3]
h3k4_df.columns = ["Chromosome", "Start", "End"]

promoter_df = pd.read_csv("/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_clean.bed", sep="\t", header=None).iloc[:, :3]
promoter_df.columns = ["Chromosome", "Start", "End"]

print("=== CHD8 Clean DataFrame ===")
print(chd8_df.head(3))
print(f"Unique Chromosomes: {chd8_df['Chromosome'].unique()[:5]}\n")

print("=== H3K4me3 DataFrame ===")
print(h3k4_df.head(3))
print(f"Unique Chromosomes: {h3k4_df['Chromosome'].unique()[:5]}\n")

print("=== Promoter DataFrame ===")
print(promoter_df.head(3))
print(f"Unique Chromosomes: {promoter_df['Chromosome'].unique()[:5]}\n")

In [ ]:
import pybedtools
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import subprocess
import os

ES_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
NPC_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
SAVE_DIR = "/content/drive/MyDrive/CHD8_Fig1/"
os.makedirs(SAVE_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'pdf.fonttype'   : 42,
})

STANDARD_CHROMS = set(
    [f'chr{i}' for i in range(1, 20)] + ['chrX', 'chrY']
)

def is_standard(feature):
    return str(feature.chrom) in STANDARD_CHROMS

def shell_count(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return int(r.stdout.strip().split()[0])

if not (os.path.exists(ES_PATH) and os.path.exists(NPC_PATH)):
    raise FileNotFoundError("BED files not found — check paths above.")

print("=" * 65)
print("FIGURE 1A — VENN COUNTS (shell bedtools)")
print("=" * 65)

# ── Filter to standard chromosomes ───────────────────────────────
es_filt  = pybedtools.BedTool(ES_PATH).filter(is_standard).saveas()
npc_filt = pybedtools.BedTool(NPC_PATH).filter(is_standard).saveas()

es_total  = es_filt.count()
npc_total = npc_filt.count()

# Write sorted temp files for shell bedtools
ES_TMP  = "/content/es_std_sorted.bed"
NPC_TMP = "/content/npc_std_sorted.bed"
es_filt.sort().saveas(ES_TMP)
npc_filt.sort().saveas(NPC_TMP)

# ── Four counts via shell (bypasses pybedtools wrapping bug) ──────
es_only   = shell_count(
    f"bedtools intersect -a {ES_TMP}  -b {NPC_TMP} -v | wc -l")
npc_only  = shell_count(
    f"bedtools intersect -a {NPC_TMP} -b {ES_TMP}  -v | wc -l")
es_shared = shell_count(
    f"bedtools intersect -a {ES_TMP}  -b {NPC_TMP} -u | wc -l")
npc_shared= shell_count(
    f"bedtools intersect -a {NPC_TMP} -b {ES_TMP}  -u | wc -l")

# ── Arithmetic verification (both perspectives) ───────────────────
es_ok  = (es_only  + es_shared  == es_total)
npc_ok = (npc_only + npc_shared == npc_total)

print(f"\n  ES  total (std chroms)   : {es_total:>7,}")
print(f"  NPC total (std chroms)   : {npc_total:>7,}")
print()
print(f"  ES  peaks touching NPC   : {es_shared:>7,}  "
      f"(ES perspective)")
print(f"  NPC peaks touching ES    : {npc_shared:>7,}  "
      f"(NPC perspective  <- used for Venn middle)")
print(f"  ES-only (no NPC overlap) : {es_only:>7,}")
print(f"  NPC-only (no ES overlap) : {npc_only:>7,}")
print()
print(f"  ES  check: {es_only:,} + {es_shared:,}  = "
      f"{es_only+es_shared:,}  "
      f"{'== ES total  OK' if es_ok  else f'!= ES total {es_total:,}'}")
print(f"  NPC check: {npc_only:,} + {npc_shared:,} = "
      f"{npc_only+npc_shared:,}  "
      f"{'== NPC total OK' if npc_ok else f'!= NPC total {npc_total:,}'}")
print()
if es_shared != npc_shared:
    diff = abs(es_shared - npc_shared)
    print(f"  Note: ES_shared ({es_shared:,}) != NPC_shared ({npc_shared:,})")
    print(f"  Difference = {diff:,} — expected due to asymmetric peak widths.")
    print(f"  Wide ES peaks can overlap multiple narrow NPC peaks and vice versa.")
    print(f"  This is normal in ChIP-seq. Venn uses NPC perspective (study condition).")
print("=" * 65)

if not (es_ok and npc_ok):
    raise ValueError("Arithmetic broken — check BED files for malformed records.")

# ── Percentages ───────────────────────────────────────────────────
pct_es_only   = es_only   / es_total  * 100
pct_npc_only  = npc_only  / npc_total * 100
pct_npc_sh_es = npc_shared / es_total  * 100   # NPC-shared as % of ES
pct_npc_sh_npc= npc_shared / npc_total * 100   # NPC-shared as % of NPC

# ── Figure ────────────────────────────────────────────────────────
# Venn middle = npc_shared (NPC peaks overlapping ES)
# This means: es_only is not shown as npc-side complement,
# the Venn proportionally represents the NPC circle correctly.
fig, ax = plt.subplots(figsize=(8, 7), dpi=600)

v = venn2(
    subsets    = (es_only, npc_only, npc_shared),
    set_labels = ('', ''),
    set_colors = ('#3498DB', '#E67E22'),
    alpha      = 0.70,
    ax         = ax
)

if v.get_label_by_id('10'):
    v.get_label_by_id('10').set_text(
        f"{es_only:,}\n({pct_es_only:.1f}%)")
    v.get_label_by_id('10').set_fontsize(12)

if v.get_label_by_id('01'):
    v.get_label_by_id('01').set_text(
        f"{npc_only:,}\n({pct_npc_only:.1f}%)")
    v.get_label_by_id('01').set_fontsize(12)

if v.get_label_by_id('11'):
    v.get_label_by_id('11').set_text(
        f"{npc_shared:,}\n({pct_npc_sh_es:.1f}% of ES\n"
        f"{pct_npc_sh_npc:.1f}% of NPC)")
    v.get_label_by_id('11').set_fontsize(10)

ax.text(-0.58, 0.42,
        f"ES CHD8 Peaks\n(n\u202f=\u202f{es_total:,})",
        ha='center', fontsize=12, fontweight='bold', color='#1A5276')
ax.text( 0.58, 0.42,
        f"NPC CHD8 Peaks\n(n\u202f=\u202f{npc_total:,})",
        ha='center', fontsize=12, fontweight='bold', color='#784212')

ax.set_title(
    "Figure 1A: CHD8 binding dynamics — ES vs NPC\n"
    "(standard chromosomes only; mm10)",
    fontsize=13, fontweight='bold', pad=18
)

# Footer — states the overlap definition used
ax.text(0.5, -0.07,
        f"Shared = NPC peaks with \u22651\u202fbp overlap with any ES peak "
        f"(n\u202f=\u202f{npc_shared:,}).  "
        f"NPC arithmetic verified: {npc_only:,}\u202f+\u202f{npc_shared:,}"
        f"\u202f=\u202f{npc_total:,}.",
        transform=ax.transAxes, ha='center', va='top',
        fontsize=8, color='#27AE60', fontstyle='italic')

plt.tight_layout()
plt.savefig(SAVE_DIR + "Figure1A_Venn_600DPI.png",
            dpi=600, bbox_inches='tight')
plt.savefig(SAVE_DIR + "Figure1A_Venn_vector.pdf",
            bbox_inches='tight')
plt.show()
plt.close()

print(f"\nFigure saved to: {SAVE_DIR}")
print("\nCopy into Figure 1A legend:")
print(f"  ESC peaks (std chroms): {es_total:,}")
print(f"  NPC peaks (std chroms): {npc_total:,}")
print(f"  ESC-specific          : {es_only:,} ({pct_es_only:.1f}% of ESC)")
print(f"  NPC-specific          : {npc_only:,} ({pct_npc_only:.1f}% of NPC)")
print(f"  Shared (NPC persp.)   : {npc_shared:,} "
      f"NPC peaks overlapping \u22651\u202fbp with any ESC peak")
print(f"  Note: {es_shared:,} ESC peaks also overlap NPC peaks (ESC perspective)")
print(f"  Asymmetry ({es_shared:,} vs {npc_shared:,}) reflects peak-width differences")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files

# ── 1. Variables inherited from Venn cell

# ── 2. Style helpers ──────────────────────────────────────────────
C_DARK, C_MID, C_LIGHT = "1F4E79", "2E75B6", "D6E4F0"
C_SHARED, C_ES, C_NPC, C_ALT = "E8F5E9", "FFF9C4", "FCE4EC", "F5F5F5"

def fill(h): return PatternFill("solid", start_color=h, end_color=h)
def fnt(bold=False, color="333333", size=10, italic=False, mono=False):
    return Font(name="Courier New" if mono else "Arial",
                bold=bold, italic=italic, color=color, size=size)
def aln(h="center", wrap=True):
    return Alignment(horizontal=h, vertical="center", wrap_text=wrap)

thin = Side(style="thin", color="BDBDBD")
brd  = Border(left=thin, right=thin, top=thin, bottom=thin)

def s(ws, row, col, value, f=None, fl=None, a=None, fmt=None):
    c = ws.cell(row=row, column=col, value=value)
    if f:   c.font          = f
    if fl:  c.fill          = fl
    if a:   c.alignment     = a
    if fmt: c.number_format = fmt
    c.border = brd
    return c

def hdr(ws, row, labels, height=22):
    for col, label in enumerate(labels, 1):
        s(ws, row, col, label,
          f=fnt(bold=True, color="FFFFFF"),
          fl=fill(C_MID), a=aln())
    ws.row_dimensions[row].height = height

# ── 3. Build workbook ─────────────────────────────────────────────
wb = Workbook()
ws = wb.active
ws.title = "Supplementary Table 4"

# Title
ws.merge_cells("A1:G1")
s(ws, 1, 1,
  "Supplementary Table 4 | CHD8 ChIP-seq Peak Summary — ES vs NPC",
  f=fnt(bold=True, color="FFFFFF", size=13),
  fl=fill(C_DARK), a=aln())
ws.row_dimensions[1].height = 30

ws.merge_cells("A2:G2")
s(ws, 2, 1,
  "High-confidence CHD8 peaks · standard chromosomes only (chr1–19, chrX, chrY) · genome: mm10",
  f=fnt(italic=True, color="FFFFFF", size=9),
  fl=fill(C_MID), a=aln())
ws.row_dimensions[2].height = 18

# ── Section A: Overall counts ─────────────────────────────────────
ws.merge_cells("A4:G4")
s(ws, 4, 1, "A.  Overall Peak Counts",
  f=fnt(bold=True, color=C_DARK), fl=fill(C_LIGHT), a=aln("left"))
ws.row_dimensions[4].height = 18

hdr(ws, 5, ["Metric", "ES Cells", "NPC",
            "Difference (NPC − ES)", "Fold Change (NPC / ES)", "Notes", ""])

rows_a = [
    ("Total CHD8 peaks (filtered)",
     es_total, npc_total, npc_total - es_total,
     npc_total / es_total,
     "After standard chromosome filter; source: pybedtools filter"),
    ("Unique peak loci (shell sort)",
     es_total, npc_total, npc_total - es_total,
     npc_total / es_total,
     "Confirmed: no duplicate chr/start/end"),
]
for i, (metric, es, npc, diff, fc, note) in enumerate(rows_a, 6):
    bg = C_ALT if i % 2 == 0 else "FFFFFF"
    s(ws, i, 1, metric, f=fnt(),     fl=fill(bg), a=aln("left"))
    s(ws, i, 2, es,     f=fnt(),     fl=fill(bg), a=aln(), fmt="#,##0")
    s(ws, i, 3, npc,    f=fnt(),     fl=fill(bg), a=aln(), fmt="#,##0")
    s(ws, i, 4, diff,   f=fnt(),     fl=fill(bg), a=aln(), fmt="+#,##0;-#,##0;0")
    s(ws, i, 5, fc,     f=fnt(),     fl=fill(bg), a=aln(), fmt='0.00"x"')
    s(ws, i, 6, note,   f=fnt(),     fl=fill(bg), a=aln("left"))
    s(ws, i, 7, "",     fl=fill(bg))
    ws.row_dimensions[i].height = 18

# ── Section B: Overlap — two perspectives ────────────────────────
ws.merge_cells("A9:G9")
s(ws, 9, 1,
  "B.  Peak Overlap Analysis  (shell bedtools intersect — arithmetic verified)",
  f=fnt(bold=True, color=C_DARK), fl=fill(C_LIGHT), a=aln("left"))
ws.row_dimensions[9].height = 18

hdr(ws, 10, ["Category", "Count", "% of ES total",
             "% of NPC total", "Perspective", "Shell command", "Verified"])

overlap_rows = [
    # (label, count, pct_es, pct_npc, perspective, cmd, verified, bg)
    ("Shared — NPC peaks overlapping ES  ★ Venn middle",
     npc_shared, npc_shared/es_total, npc_shared/npc_total,
     "NPC (study condition)",
     "bedtools intersect -a NPC -b ES -u",
     f"✓  {npc_only:,}+{npc_shared:,}={npc_only+npc_shared:,}==NPC total",
     C_SHARED),
    ("Shared — ES peaks overlapping NPC  (ES perspective)",
     es_shared, es_shared/es_total, es_shared/npc_total,
     "ES (reference)",
     "bedtools intersect -a ES -b NPC -u",
     f"✓  {es_only:,}+{es_shared:,}={es_only+es_shared:,}==ES total",
     C_SHARED),
    ("ES-only  (no overlap with NPC)",
     es_only, es_only/es_total, None,
     "ES",
     "bedtools intersect -a ES -b NPC -v",
     "✓  see above", C_ES),
    ("NPC-only  (no overlap with ES)",
     npc_only, None, npc_only/npc_total,
     "NPC",
     "bedtools intersect -a NPC -b ES -v",
     "✓  see above", C_NPC),
]
for i, (cat, count, pct_es, pct_npc, persp, cmd, verified, bg) in \
        enumerate(overlap_rows, 11):
    s(ws, i, 1, cat,      f=fnt(),                          fl=fill(bg), a=aln("left"))
    s(ws, i, 2, count,    f=fnt(bold=True, color=C_DARK),   fl=fill(bg), a=aln(), fmt="#,##0")
    for col, val in [(3, pct_es), (4, pct_npc)]:
        if val is None:
            s(ws, i, col, "—", f=fnt(), fl=fill(bg), a=aln())
        else:
            s(ws, i, col, val, f=fnt(), fl=fill(bg), a=aln(), fmt="0.0%")
    s(ws, i, 5, persp,    f=fnt(),                fl=fill(bg), a=aln())
    s(ws, i, 6, cmd,      f=fnt(mono=True, size=8), fl=fill(bg), a=aln("left"))
    s(ws, i, 7, verified, f=fnt(color="2E7D32"), fl=fill(bg), a=aln("left"))
    ws.row_dimensions[i].height = 20

# Asymmetry note
ws.merge_cells("A15:G15")
s(ws, 15, 1,
  f"Note on asymmetry: ES_shared ({es_shared:,}) ≠ NPC_shared ({npc_shared:,}). "
  f"Difference of {abs(es_shared-npc_shared):,} is expected — wide peaks can overlap "
  f"multiple narrow peaks from the other condition. "
  f"Venn diagram uses NPC perspective (study condition).",
  f=fnt(italic=True, color="555555", size=9),
  fl=fill("FFFDE7"), a=aln("left"))
ws.row_dimensions[15].height = 36

# ── Section C: Percentages (Venn label values) ────────────────────
ws.merge_cells("A17:G17")
s(ws, 17, 1, "C.  Percentage Summary  (values shown in Figure 1A Venn diagram)",
  f=fnt(bold=True, color=C_DARK), fl=fill(C_LIGHT), a=aln("left"))
ws.row_dimensions[17].height = 18

hdr(ws, 18, ["Label", "Count", "Percentage", "Denominator", "", "", ""])

pct_rows = [
    ("ES-only",            es_only,   pct_es_only,    "ES total",  C_ES),
    ("NPC-only",           npc_only,  pct_npc_only,   "NPC total", C_NPC),
    ("Shared / ES total",  npc_shared,pct_npc_sh_es,  "ES total",  C_SHARED),
    ("Shared / NPC total", npc_shared,pct_npc_sh_npc, "NPC total", C_SHARED),
]
for i, (label, count, pct, denom, bg) in enumerate(pct_rows, 19):
    s(ws, i, 1, label, f=fnt(),            fl=fill(bg), a=aln("left"))
    s(ws, i, 2, count, f=fnt(bold=True),   fl=fill(bg), a=aln(), fmt="#,##0")
    s(ws, i, 3, pct/100, f=fnt(),          fl=fill(bg), a=aln(), fmt="0.0%")
    s(ws, i, 4, denom, f=fnt(),            fl=fill(bg), a=aln())
    for col in [5, 6, 7]:
        s(ws, i, col, "", fl=fill(bg))
    ws.row_dimensions[i].height = 18

# Methods footnote
ws.merge_cells("A24:G24")
s(ws, 24, 1,
  "Methods: CHD8 ChIP-seq peaks called independently for ES and NPC; "
  "merged per condition. Filtered to standard chromosomes. "
  "Overlap counts via shell bedtools intersect (v2.x); arithmetic verified "
  "from both ES and NPC perspectives. "
  "Asymmetric shared counts reflect peak-width differences (normal for ChIP-seq). "
  "Venn diagram uses NPC-perspective shared count as the intersection.",
  f=fnt(italic=True, color="666666", size=9),
  fl=fill("FAFAFA"), a=aln("left"))
ws.row_dimensions[24].height = 50

# ── 4. Column widths & freeze ─────────────────────────────────────
for col, w in enumerate([44, 12, 15, 15, 22, 40, 36], 1):
    ws.column_dimensions[get_column_letter(col)].width = w
ws.freeze_panes = "A6"

# ── 5. Save & download ────────────────────────────────────────────
OUT = "/content/Supplementary_Table_4_CHD8_ChIPseq.xlsx"
wb.save(OUT)
print(f"✅  Saved → {OUT}")
files.download(OUT)
print("📥  Download triggered.")

In [ ]:
import os
import subprocess
import pybedtools
import matplotlib.pyplot as plt

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── File paths ────────────────────────────────────────────────────────────────
base_dir      = "/content/drive/MyDrive/CHD8_Fig1"
npc_peaks_path = f"{base_dir}/peaks/CHD8_NPC_merged.bed"
h3k4_path      = f"{base_dir}/peaks/H3K4me3_NPC_merged.bed"
tss_path       = f"{base_dir}/mm10_TSS_2kb.bed"
os.makedirs(base_dir, exist_ok=True)


STANDARD_CHROMS = set(
    [f'chr{i}' for i in range(1, 20)] + ['chrX', 'chrY']
)
def is_standard(feature):
    return str(feature.chrom) in STANDARD_CHROMS

def run_fig1B():

    # ── File checks ──────────────────────────────────────────────────────────
    for label, path in [("NPC peaks", npc_peaks_path),
                        ("TSS BED",   tss_path)]:
        if not os.path.exists(path):
            print(f"❌ {label} missing: {path}")
            return

    print("📊 Processing CHD8 peaks (standard chromosomes only)...")

    # ── CHANGE 1: filter to standard chromosomes before counting ─────────────

    npc_bed  = pybedtools.BedTool(npc_peaks_path).filter(is_standard).sort().saveas()
    tss_bed  = pybedtools.BedTool(tss_path)

    total_npc = npc_bed.count()   # should equal 48,735 (std chroms, from Cell 13)
    print(f"   NPC peaks (std chroms): {total_npc:,}  "
          f"← should match Figure 1A Venn NPC total (48,735)")

    # ── Promoter vs Distal ────────────────────────────────────────────────────
    promoter_bed = npc_bed.intersect(tss_bed, u=True).saveas()
    promoter     = promoter_bed.count()
    distal       = total_npc - promoter

    pct_promoter = promoter / total_npc * 100
    pct_distal   = distal   / total_npc * 100

    print(f"   Promoter (±2kb TSS):    {promoter:,}  ({pct_promoter:.1f}%)")
    print(f"   Distal:                 {distal:,}  ({pct_distal:.1f}%)")

    # ── Active vs Poised (H3K4me3 split) ─────────────────────────────────────
    if os.path.exists(h3k4_path):
        print("💡 H3K4me3 detected → splitting Active vs Poised promoters")

        h3k4_bed = (pybedtools.BedTool(h3k4_path)
                    .filter(is_standard)   # CHANGE 2: filter H3K4me3 too
                    .sort().saveas())

        active = promoter_bed.intersect(h3k4_bed, u=True).count()
        poised = promoter - active

        pct_active = active / total_npc * 100
        pct_poised = poised / total_npc * 100

        print(f"   Active promoters (H3K4me3+): {active:,}  ({pct_active:.1f}%)")
        print(f"   Poised promoters (H3K4me3-): {poised:,}  ({pct_poised:.1f}%)")

        # CHANGE 3: print H3K4me3 co-occupancy %
        h3k4_overlap_pct = active / promoter * 100
        print(f"\n   ── - text value ──────────────────────────────")
        print(f"   CHD8 peaks overlapping H3K4me3 (promoter-level): "
              f"{active:,}/{promoter:,} = {h3k4_overlap_pct:.1f}%")
        print(f"   (Update -: 'X% of CHD8-bound promoters "
              f"co-occupied by H3K4me3 in NPCs')")
        print(f"   ───────────────────────────────────────────────────────")

        sizes  = [active, poised, distal]
        labels = [
            f'Active Promoters\n(CHD8+H3K4me3+)\nn={active:,}',
            f'Poised Promoters\n(CHD8+H3K4me3−)\nn={poised:,}',
            f'Distal Elements\nn={distal:,}'
        ]
        colors = ['#1f77b4', '#6baed6', '#bdbdbd']

    else:
        print("⚠️ H3K4me3 file not found → simple promoter/distal split")

        sizes  = [promoter, distal]
        labels = [
            f'Promoters\n(±2kb TSS)\nn={promoter:,}',
            f'Distal Elements\nn={distal:,}'
        ]
        colors = ['#1f77b4', '#bdbdbd']

    # ── CHANGE 4: print legend ───────────────────────
    print(f"\n   ── Figure 1B legend (copy into -) ────────────────")
    print(f"   Total NPC CHD8 peaks (std chroms): {total_npc:,}")
    for lbl, sz in zip(labels, sizes):
        lbl_clean = lbl.replace('\n', ' ')
        print(f"   {lbl_clean}: {sz:,} ({sz/total_npc*100:.1f}%)")
    print(f"   ───────────────────────────────────────────────────────────")

    # ── Plot ──────────────────────────────────────────────────────
    plt.rcParams.update({
        'font.family'    : 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans'],
        'axes.linewidth' : 1.2,
        'pdf.fonttype'   : 42,
    })

    fig, ax = plt.subplots(figsize=(7, 7), dpi=600)

    wedges, texts, autotexts = ax.pie(
        sizes,
        labels     = labels,
        autopct    = '%1.1f%%',
        startangle = 120,
        explode    = [0.05] * len(sizes),
        colors     = colors,
        textprops  = {'fontsize': 11}
    )
    for at in autotexts:
        at.set_fontsize(11)
        at.set_weight('bold')

    # CHANGE 5: title now explicitly states std-chrom filter and n
    ax.set_title(
        f"Figure 1B: Genomic Distribution of NPC CHD8 Binding\n"
        f"(standard chromosomes only; n\u202f=\u202f{total_npc:,}; mm10)",
        fontsize=13, fontweight='bold'
    )

    # CHANGE 6: footnote states consistency
    ax.text(
        0.5, -0.04,
        f"n\u202f=\u202f{total_npc:,} peaks (standard chromosomes; consistent with Figure 1A Venn).",
        transform=ax.transAxes, ha='center', fontsize=8,
        color='#27AE60', fontstyle='italic'
    )

    plt.tight_layout()

    # ── Save ──────────────────────────────────────────────────────────────────
    save_png = f"{base_dir}/Figure1B_600dpi.png"   # CHANGE 7: renamed 2B→1B
    save_pdf = f"{base_dir}/Figure1B.pdf"
    plt.savefig(save_png, dpi=600, bbox_inches='tight', format='png')
    plt.savefig(save_pdf, bbox_inches='tight')
    plt.show()
    plt.close()

    print("\n✅ Figure 1B saved to Google Drive (600 dpi PNG + PDF)")
    print(f"   {save_png}")
    print(f"   {save_pdf}")

# ── Run ───────────────────────────────────────────────────────────────────────
run_fig1B()

In [ ]:
!pip install pybedtools
!apt-get install -y bedtools

In [ ]:
import os

# Define the search directory
search_dir = "/content/drive/MyDrive/Chd8 data"

print("🔍 Searching for your TSS reference files...")
found = False

if os.path.exists(search_dir):
    for root, dirs, files in os.walk(search_dir):
        for file in files:
            if "tss" in file.lower():
                print(f"✨ Found TSS file: {os.path.join(root, file)}")
                found = True
            elif "chrom.sizes" in file.lower():
                print(f"📏 Found Chrom Sizes: {os.path.join(root, file)}")
                found = True
else:
    print(f"❌ Could not find the main folder: {search_dir}")

if not found:
    print("🤷 No TSS files matching 'tss' were found. Here are the files inside your annotation folder:")
    annot_dir = os.path.join(search_dir, "annotation")
    if os.path.exists(annot_dir):
        print(os.listdir(annot_dir))
    else:
        print(f"❌ Annotation folder does not exist at: {annot_dir}")

In [ ]:
import os

search_dir = "/content/drive/MyDrive/Chd8 data"
print("🔍 Searching for H3K4me3 peak files...")
found = False

if os.path.exists(search_dir):
    for root, dirs, files in os.walk(search_dir):
        for file in files:
            if "h3k4" in file.lower() and file.endswith(".bed"):
                print(f"✨ Found H3K4me3 BED file: {os.path.join(root, file)}")
                found = True
else:
    print(f"❌ Main folder not found: {search_dir}")

if not found:
    print("🤷 No H3K4me3 .bed files found. Let's check the contents of your MACS2_peaks folder:")
    macs_dir = "/content/drive/MyDrive/Chd8 data/CHIPSEQ/MACS2_peaks"
    if os.path.exists(macs_dir):
        print(os.listdir(macs_dir))
    else:
        print(f"❌ MACS2_peaks directory does not exist at {macs_dir}")

In [ ]:
import os
import subprocess
import pybedtools
import matplotlib.pyplot as plt

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ── File Paths ────────────────────────────────────────────────────────────────
base_dir       = "/content/drive/MyDrive/CHD8_Fig1"
os.makedirs(base_dir, exist_ok=True)

npc_peaks_path = f"{base_dir}/peaks/CHD8_NPC_merged.bed"
tss_2kb_path   = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"

# - CHIP-SEQ PATHS: Pointing to your experimental Undiff H3K4me3 replicates
h3k4_rep1_path = "/content/drive/MyDrive/Chd8 data/CHIPSEQ/MACS2_peaks/CHIP/H3K4me3_Undiff_1/H3K4me3_Undiff_1.merged_score_sorted.bed"
h3k4_rep2_path = "/content/drive/MyDrive/Chd8 data/CHIPSEQ/MACS2_peaks/CHIP/H3K4me3_Undiff_2/H3K4me3_Undiff_2.merged_score_sorted.bed"

# ── Chromosome Normalization Setup ───────────────────────────────────────────
REFSEQ_MAP = {
    "NC_000067.6": "1",  "NC_000068.7": "2",  "NC_000069.6": "3",  "NC_000070.6": "4",
    "NC_000071.6": "5",  "NC_000072.6": "6",  "NC_000073.6": "7",  "NC_000074.6": "8",
    "NC_000075.6": "9",  "NC_000076.6": "10", "NC_000077.6": "11", "NC_000078.6": "12",
    "NC_000079.6": "13", "NC_000080.6": "14", "NC_000081.6": "15", "NC_000082.6": "16",
    "NC_000083.6": "17", "NC_000084.6": "18", "NC_000085.6": "19",
    "NC_000086.7": "X",  "NC_000087.7": "Y"
}

STANDARD_CHROMS = set([str(i) for i in range(1, 20)] + ['X', 'Y'])

def clean_and_normalize(feature):
    chrom = str(feature.chrom)
    if chrom in REFSEQ_MAP:
        chrom = REFSEQ_MAP[chrom]
    if chrom.startswith('chr'):
        chrom = chrom.replace('chr', '')
    feature.chrom = chrom
    return feature

def is_standard(feature):
    return str(feature.chrom) in STANDARD_CHROMS

# ── Analysis Execution ────────────────────────────────────────────────────────
def run_fig1B_analysis():
    # Verify core files exist
    for label, path in [("CHD8 NPC peaks", npc_peaks_path), ("TSS 2kb BED", tss_2kb_path)]:
        if not os.path.exists(path):
            print(f"❌ {label} missing at path: {path}")
            return

    print("📊 Standardizing and filtering CHD8 peaks...")
    npc_bed = pybedtools.BedTool(npc_peaks_path).each(clean_and_normalize).filter(is_standard).sort().saveas()
    total_npc = npc_bed.count()
    print(f"   NPC peaks (std chroms): {total_npc:,}")

    print("🧬 Loading mm10 TSS (±2kb reference)...")
    tss_bed = pybedtools.BedTool(tss_2kb_path).each(clean_and_normalize).filter(is_standard).sort().merge().saveas()

    # 1. Calculate Promoter vs Distal split
    promoter_bed = npc_bed.intersect(tss_bed, u=True).saveas()
    promoter     = promoter_bed.count()
    distal       = total_npc - promoter

    pct_promoter = (promoter / total_npc) * 100
    pct_distal   = (distal / total_npc) * 100

    print(f"\n{'='*70}")
    print(f" 🎯 TSS WINDOW BOUNDARIES: ±2kb Reference")
    print(f"{'='*70}")
    print(f"   Promoter (Within ±2kb TSS):  {promoter:,}  ({pct_promoter:.1f}%)")
    print(f"   Distal (Genomic Desert):    {distal:,}  ({pct_distal:.1f}%)")

    # 2. Functional Overlap with true H3K4me3 peaks
    if os.path.exists(h3k4_rep1_path) and os.path.exists(h3k4_rep2_path):
        print("\n🕯️  Quantifying active vs poised promoters using independent H3K4me3 replicates...")

        # Load and clean both replicates
        h3k4_r1 = pybedtools.BedTool(h3k4_rep1_path).each(clean_and_normalize).filter(is_standard).sort()
        h3k4_r2 = pybedtools.BedTool(h3k4_rep2_path).each(clean_and_normalize).filter(is_standard).sort()

        # Intersect them to create a high-confidence consensus H3K4me3 landscape
        h3k4_consensus = h3k4_r1.intersect(h3k4_r2, u=True).saveas()

        # Determine Active vs Poised promoters
        active = promoter_bed.intersect(h3k4_consensus, u=True).count()
        poised = promoter - active

        pct_active = (active / total_npc) * 100
        pct_poised = (poised / total_npc) * 100
        h3k4_overlap_pct = (active / promoter) * 100 if promoter else 0

        print(f"   Active promoters (H3K4me3+): {active:,}  ({pct_active:.1f}%)")
        print(f"   Poised promoters (H3K4me3-): {poised:,}  ({pct_poised:.1f}%)")
        print(f"   CHD8 promoter elements overlapping H3K4me3: "
              f"{active:,}/{promoter:,} = {h3k4_overlap_pct:.1f}%")
    else:
        print(f"\n⚠️ Warning: One or both H3K4me3 peak files were missing. Check your paths.")
        active = poised = None

    return {
        "window": "2kb",
        "promoter": promoter, "pct_promoter": pct_promoter,
        "distal": distal, "pct_distal": pct_distal,
        "active": active, "poised": poised,
    }

# ── Run ───────────────────────────────────────────────────────────────────────
results = run_fig1B_analysis()

In [ ]:
import os
import subprocess
import pybedtools
import matplotlib.pyplot as plt

# ── Google Drive mount ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Updated File Paths ────────────────────────────────────────────────────────
base_dir       = "/content/drive/MyDrive/CHD8_Fig1"
os.makedirs(base_dir, exist_ok=True)

npc_peaks_path = f"{base_dir}/peaks/CHD8_NPC_merged.bed"

# Pointing to your actual available TSS file
raw_tss_2kb_path = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"

# Script will automatically generate these two files in your base directory
tss_point_path = f"{base_dir}/mm10_TSS_point.bed"
chrom_sizes    = f"{base_dir}/mm10.chrom.sizes"

# Pointing to the verified independent H3K4me3 experimental replicates
h3k4_rep1_path = "/content/drive/MyDrive/Chd8 data/CHIPSEQ/MACS2_peaks/CHIP/H3K4me3_Undiff_1/H3K4me3_Undiff_1.merged_score_sorted.bed"
h3k4_rep2_path = "/content/drive/MyDrive/Chd8 data/CHIPSEQ/MACS2_peaks/CHIP/H3K4me3_Undiff_2/H3K4me3_Undiff_2.merged_score_sorted.bed"

# ── RefSeq Accession to Standard Chromosome Mapping (mm10) ────────────────────
REFSEQ_MAP = {
    "NC_000067.6": "1",  "NC_000068.7": "2",  "NC_000069.6": "3",  "NC_000070.6": "4",
    "NC_000071.6": "5",  "NC_000072.6": "6",  "NC_000073.6": "7",  "NC_000074.6": "8",
    "NC_000075.6": "9",  "NC_000076.6": "10", "NC_000077.6": "11", "NC_000078.6": "12",
    "NC_000079.6": "13", "NC_000080.6": "14", "NC_000081.6": "15", "NC_000082.6": "16",
    "NC_000083.6": "17", "NC_000084.6": "18", "NC_000085.6": "19",
    "NC_000086.7": "X",  "NC_000087.7": "Y"
}

STANDARD_CHROMS = set([str(i) for i in range(1, 20)] + ['X', 'Y'])

def clean_and_normalize(feature):
    chrom = str(feature.chrom)
    if chrom in REFSEQ_MAP:
        chrom = REFSEQ_MAP[chrom]
    if chrom.startswith('chr'):
        chrom = chrom.replace('chr', '')
    feature.chrom = chrom
    return feature

def is_standard(feature):
    return str(feature.chrom) in STANDARD_CHROMS

# ── Dynamic Generation of Missing Dependencies ───────────────────────────────
def generate_dependencies():
    if not os.path.exists(raw_tss_2kb_path):
        print(f"❌ Cannot generate point-TSS reference. File missing: {raw_tss_2kb_path}")
        return False

    print("🪄 Reconstructing single-base TSS coordinates from pre-slopped 2kb reference...")
    # Read the ±2kb file, compute the midpoint coordinate, and write out an unslopped point BED file
    with open(raw_tss_2kb_path, 'r') as infile, open(tss_point_path, 'w') as outfile:
        for line in infile:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split('\t')
            chrom, start, end = parts[0], int(parts[1]), int(parts[2])
            midpoint = start + (end - start) // 2
            # Handle additional columns safely if present
            name = parts[3] if len(parts) > 3 else "TSS"
            outfile.write(f"{chrom}\t{midpoint}\t{midpoint+1}\t{name}\n")

    print("🌍 Generating mock chrom.sizes framework for mm10...")
    # Standard large chromosome values to prevent bedtools boundary out-of-bounds clipping
    with open(chrom_sizes, 'w') as f:
        for i in range(1, 20): f.write(f"{i}\t250000000\n")
        f.write("X\t170000000\nY\t100000000\n")
    return True

# ── Windows to test ──────────────────────────────────────────────────────────
WINDOWS_TO_TEST = {
    "2kb":  2000,
    "5kb":  5000,
    "10kb": 10000,
}

def run_fig1B_for_window(window_bp, label, npc_bed, h3k4_bed, tss_point_bed, total_npc):
    print(f"\n{'='*70}")
    print(f"  TSS WINDOW: ±{window_bp:,} bp ({label})")
    print(f"{'='*70}")

    tss_bed = tss_point_bed.slop(b=window_bp, g=chrom_sizes).sort().merge()

    promoter_bed = npc_bed.intersect(tss_bed, u=True).saveas()
    promoter     = promoter_bed.count()
    distal       = total_npc - promoter

    pct_promoter = promoter / total_npc * 100
    pct_distal   = distal   / total_npc * 100

    print(f"   Promoter (±{label} TSS):    {promoter:,}  ({pct_promoter:.1f}%)")
    print(f"   Distal:                      {distal:,}  ({pct_distal:.1f}%)")

    active = poised = None
    if h3k4_bed is not None:
        active = promoter_bed.intersect(h3k4_bed, u=True).count()
        poised = promoter - active
        pct_active = active / total_npc * 100
        pct_poised = poised / total_npc * 100
        h3k4_overlap_pct = active / promoter * 100 if promoter else 0

        print(f"   Active promoters (H3K4me3+): {active:,}  ({pct_active:.1f}%)")
        print(f"   Poised promoters (H3K4me3-): {poised:,}  ({pct_poised:.1f}%)")
        print(f"   CHD8 peaks overlapping H3K4me3 (promoter-level): "
              f"{active:,}/{promoter:,} = {h3k4_overlap_pct:.1f}%")

    return {
        "window": label,
        "promoter": promoter, "pct_promoter": pct_promoter,
        "distal": distal, "pct_distal": pct_distal,
        "active": active, "poised": poised,
    }

def run_fig1B_comparison():
    # Make sure files exist or make them on the fly
    if not os.path.exists(tss_point_path) or not os.path.exists(chrom_sizes):
        if not generate_dependencies():
            return

    for label, path in [("NPC peaks", npc_peaks_path)]:
        if not os.path.exists(path):
            print(f"❌ Core processing dependency [{label}] missing at: {path}")
            return

    print("📊 Standardizing and filtering CHD8 peaks...")
    npc_bed = pybedtools.BedTool(npc_peaks_path).each(clean_and_normalize).filter(is_standard).sort().saveas()
    total_npc = npc_bed.count()
    print(f"   NPC peaks (std chroms): {total_npc:,}")

    tss_point_bed = pybedtools.BedTool(tss_point_path).each(clean_and_normalize).filter(is_standard).sort().saveas()

    h3k4_bed = None
    if os.path.exists(h3k4_rep1_path) and os.path.exists(h3k4_rep2_path):
        print("🕯️  Loading and generating consensus H3K4me3 peak profiles...")
        h3k4_r1 = pybedtools.BedTool(h3k4_rep1_path).each(clean_and_normalize).filter(is_standard).sort()
        h3k4_r2 = pybedtools.BedTool(h3k4_rep2_path).each(clean_and_normalize).filter(is_standard).sort()
        h3k4_bed = h3k4_r1.intersect(h3k4_r2, u=True).saveas()
    else:
        print("⚠️ Warning: Replicate H3K4me3 tracks missing. Skipping active/poised annotations.")

    results = []
    for label, window_bp in WINDOWS_TO_TEST.items():
        res = run_fig1B_for_window(window_bp, label, npc_bed, h3k4_bed,
                                    tss_point_bed, total_npc)
        results.append(res)

    print(f"\n{'='*70}")
    print("   SUMMARY: promoter % vs published literature ranges")
    print(f"{'='*70}")
    for r in results:
        print(f"   {r['window']:>5}:  promoter = {r['pct_promoter']:.1f}%   "
              f"distal = {r['pct_distal']:.1f}%")

    return results

# ── Run ───────────────────────────────────────────────────────────────────────
results = run_fig1B_comparison()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import pybedtools
import os

PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_PATH  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'pdf.fonttype'   : 42,
})

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

# ── Step 1: NPC-specific ChIP targets ────────────────────────────
print("=" * 60)
print("FIGURE 3: CHD8 TARGET TRANSCRIPTIONAL ANALYSIS")
print("=" * 60)
print("\n1. Identifying NPC-specific ChIP targets...")

npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()

npc_specific = npc_bt.subtract(esc_bt, A=True)
hits         = tss_bt.intersect(npc_specific, u=True, wa=True)
target_list  = list(set([
    str(f[3]).upper().strip()
    for f in hits if len(str(f[3])) > 1
]))
print(f"   NPC-specific ChIP targets identified: {len(target_list):,}")

# ── Step 2: RNA-seq ───────────────────────────────────────────────
print("\n2. Loading RNA-seq data...")
de_df = pd.read_csv(RNA_PATH, sep='\t').dropna(
    subset=['log2FoldChange', 'padj'])
de_df['padj'] = de_df['padj'].replace(0, 1e-300)

if 'baseMean' in de_df.columns:
    n_before = len(de_df)
    de_df    = de_df[de_df['baseMean'] > 10]
    print(f"   Filtered: {n_before:,} -> {len(de_df):,} genes (baseMean > 10)")

de_df['gene_upper'] = (de_df['GENESYMBOL'].astype(str)
                       .str.upper().str.strip())
de_df['is_target']  = de_df['gene_upper'].isin(set(target_list))

all_targets = de_df[de_df['is_target']]
sig_targets = all_targets[all_targets['padj'] < 0.05]
others      = de_df[~de_df['is_target']]
targets_lfc = all_targets['log2FoldChange']
others_lfc  = others['log2FoldChange']

# ── Step 3: MWU test ─────────────────────────────────────────────
_, pval  = mannwhitneyu(targets_lfc, others_lfc, alternative='two-sided')
effect   = targets_lfc.median() - others_lfc.median()

print("\n" + "-" * 60)
print("AUDIT — copy into -:")
print(f"   Targets in RNA-seq         : {len(all_targets):,}")
print(f"   FDR-significant targets    : {len(sig_targets):,}")
print(f"   MWU p-value                : {pval:.3e}")
print(f"   \u0394median LFC               : {effect:.4f}")

if pval < 0.05:
    mwu_result  = f"p\u202f=\u202f{pval:.2e} (significant)"
    ecdf_title  = ("Figure 3B: CHD8 targets show a significant\n"
                   "global expression shift in KO (MWU p\u202f<\u202f0.05)")
    target_col  = '#D35400'    # orange — highlighted result
else:
    mwu_result  = (f"p\u202f=\u202f{pval:.2e} (not significant)\n"
                   f"\u0394median\u202f=\u202f{effect:.4f}\n"
                   "Remove global-shift claim from -.")
    ecdf_title  = ("Figure 3B: CHD8 target vs non-target expression\n"
                   "in KO (no significant global shift, MWU p\u202f>\u202f0.05)")
    target_col  = '#7F8C8D'    # grey — null result, not highlighted

print(f"\n   Interpretation: {mwu_result}")
print("-" * 60)

# ── PANEL A: Target-highlighted volcano ──────────────────────────
print("\n3. Generating Panel A: target volcano...")

fig3a, ax = plt.subplots(figsize=(9, 7))

# Layer 1: non-targets (background)
ax.scatter(
    others['log2FoldChange'],
    -np.log10(others['padj']),
    s=4, color='#CACFD2', alpha=0.25, rasterized=True,
    label=f'Non-targets (n\u202f=\u202f{len(others):,})'
)

# Layer 2: all ChIP targets (light orange)
ax.scatter(
    all_targets['log2FoldChange'],
    -np.log10(all_targets['padj']),
    s=14, color='#E67E22', alpha=0.45,
    label=f'CHD8 targets — all (n\u202f=\u202f{len(all_targets):,})'
)

# Layer 3: FDR-significant targets (dark orange, on top)
ax.scatter(
    sig_targets['log2FoldChange'],
    -np.log10(sig_targets['padj']),
    s=28, color='#D35400', alpha=0.95,
    edgecolors='white', linewidths=0.4,
    label=f'FDR-significant targets (n\u202f=\u202f{len(sig_targets):,})'
)

# Axes
lim = max(abs(de_df['log2FoldChange'].min()),
          abs(de_df['log2FoldChange'].max()))
ax.set_xlim(-lim, lim)
ax.axvline(0,  color='black',  lw=0.8)
ax.axhline(-np.log10(0.05), color='#7F8C8D',
           ls='--', lw=1, alpha=0.7, label='FDR = 0.05')

ax.set_title(
    'Figure 3A: CHD8 direct targets in KO vs WT\n'
    '(NPC-specific ChIP peaks within 2\u202fkb of TSS)',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('log\u2082 Fold Change (KO / WT)', fontsize=12)
ax.set_ylabel('-log\u2081\u2080 Adjusted P-value',  fontsize=12)
ax.legend(frameon=True, loc='upper right', fontsize=9,
          framealpha=0.9)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(SAVE_DIR + "Figure3A_Target_Volcano_600DPI.png",
            dpi=600, bbox_inches='tight')
plt.savefig(SAVE_DIR + "Figure3A_Target_Volcano_vector.pdf",
            bbox_inches='tight')
plt.show()
plt.close()
print("   Saved: Figure3A_Target_Volcano")

# ── PANEL B: Honest ECDF with high-contrast colours ─────────────
print("\n4. Generating Panel B: ECDF...")

fig3b, ax = plt.subplots(figsize=(7, 6))

# Non-targets: dashed grey (secondary / background)
sns.ecdfplot(
    data      = others_lfc.values,
    color     = '#95A5A6',          # medium grey
    lw        = 2.0,
    linestyle = '--',               # dashed = secondary
    label     = f'Non-targets (n\u202f=\u202f{len(others_lfc):,})',
    ax        = ax
)

# CHD8 targets: solid line — colour reflects result
sns.ecdfplot(
    data      = targets_lfc.values,
    color     = target_col,         # orange if sig, grey if not
    lw        = 3.0,
    linestyle = '-',                # solid = primary result
    label     = f'CHD8 targets (n\u202f=\u202f{len(targets_lfc):,})',
    ax        = ax
)

ax.axvline(0, color='black', ls='--', lw=1, alpha=0.5)

# Stats box
stat_text = (
    f"MWU p\u202f=\u202f{pval:.2e}\n"
    f"\u0394median\u202f=\u202f{effect:.4f}\n"
    + ("(significant)" if pval < 0.05 else "(not significant)")
)
ax.text(0.04, 0.96, stat_text, transform=ax.transAxes,
        va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.88))

ax.set_title(ecdf_title, fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('log\u2082 Fold Change (KO / WT)', fontsize=12)
ax.set_ylabel('Cumulative proportion', fontsize=12)
ax.legend(loc='lower right', fontsize=9)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(SAVE_DIR + "Figure3B_ECDF_600DPI.png",
            dpi=600, bbox_inches='tight')
plt.savefig(SAVE_DIR + "Figure3B_ECDF_vector.pdf",
            bbox_inches='tight')
plt.show()
plt.close()
print("   Saved: Figure3B_ECDF")

# ── Summary ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Files saved:")
print(f"  {SAVE_DIR}Figure3A_Target_Volcano_600DPI.png  + .pdf")
print(f"  {SAVE_DIR}Figure3B_ECDF_600DPI.png            + .pdf")
print("=" * 60)

# Expose for downstream cells (e.g. Cell 23 rescue analysis)
final_npc_genes  = target_list
npc_target_genes = target_list
print(f"\nnpc_target_genes set: {len(npc_target_genes):,} genes")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, pearsonr
import os

BASE_ATAC = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

WT_PATHS = [
    BASE_ATAC + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed",
]
KO_PATHS = [
    BASE_ATAC + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed",
]

plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'pdf.fonttype'   : 42,
})

def load_bed(path):
    if not os.path.exists(path):
        print(f"   WARNING: not found — {path}")
        return None
    df = pd.read_csv(path, sep='\t', header=None)
    score_col = 6 if len(df.columns) > 6 else 4
    out = df[[0, 1, 2, score_col]].copy()
    out.columns = ['chr', 'start', 'end', 'score']
    out['chr']   = out['chr'].astype(str)
    mask = ~out['chr'].str.startswith('chr')
    out.loc[mask, 'chr'] = 'chr' + out.loc[mask, 'chr']
    out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
    out['start'] = out['start'].astype(int)
    out['end']   = out['end'].astype(int)
    print(f"   {len(df):,} peaks — {os.path.basename(path)}")
    return out

def average_replicates(dfs, label):
    combined = pd.concat(dfs, ignore_index=True)
    combined['bin_key'] = (combined['chr'] + ':' +
                           ((combined['start'] // 50) * 50).astype(str))
    avg = combined.groupby('bin_key').agg(
        chr=('chr', 'first'), start=('start', 'min'),
        end=('end', 'max'),   score=('score', 'mean')
    ).reset_index()
    print(f"   {label}: {len(avg):,} unique regions after averaging")
    return avg

print("=" * 60)
print("FIGURE 6: CHROMATIN ACCESSIBILITY (ATAC-seq)")
print("=" * 60)

print("\n1. Loading WT replicates:")
wt_dfs = [d for d in [load_bed(p) for p in WT_PATHS] if d is not None]
print("\n2. Loading KO replicates:")
ko_dfs = [d for d in [load_bed(p) for p in KO_PATHS] if d is not None]

print("\n3. Averaging replicates...")
wt_avg = average_replicates(wt_dfs, "WT")
ko_avg = average_replicates(ko_dfs, "KO")

print("\n4. Matching peaks by 50 bp bins...")
wt_avg['bin_key'] = (wt_avg['chr'] + ':' +
                     ((wt_avg['start'] // 50) * 50).astype(str))
ko_avg['bin_key'] = (ko_avg['chr'] + ':' +
                     ((ko_avg['start'] // 50) * 50).astype(str))

merged = pd.merge(
    wt_avg[['bin_key', 'score']].rename(columns={'score': 'wt_score'}),
    ko_avg[['bin_key', 'score']].rename(columns={'score': 'ko_score'}),
    on='bin_key', how='inner'
)
print(f"   Overlapping peaks: {len(merged):,}")

if len(merged) < 100:
    raise ValueError("Too few overlapping peaks — check replicate file paths.")

# ── Compute LFC and statistics ────────────────────────────────────
pseudo     = 0.5
wt_scores  = merged['wt_score'].values
ko_scores  = merged['ko_score'].values
lfc        = np.log2((ko_scores + pseudo) / (wt_scores + pseudo))

n_gained   = int((lfc > 0).sum())
n_lost     = int((lfc < 0).sum())
n_total    = n_gained + n_lost
pct_gained = 100 * n_gained / n_total
pct_lost   = 100 * n_lost   / n_total
median_lfc = float(np.median(lfc))

_, mwu_p   = mannwhitneyu(ko_scores, wt_scores, alternative='two-sided')
r, _       = pearsonr(wt_scores, ko_scores)

# ── DATA-DRIVEN NARRATIVE ─────────────────────────────────────────

if median_lfc > 0:
    direction_word  = "increase"
    direction_phrase= (f"global accessibility increase in CHD8-KO "
                       f"(median LFC = +{median_lfc:.3f})")
elif median_lfc < 0:
    direction_word  = "reduction"
    direction_phrase= (f"global accessibility reduction in CHD8-KO "
                       f"(median LFC = {median_lfc:.3f})")
else:
    direction_word  = "no net change"
    direction_phrase= "no net change in accessibility in CHD8-KO (median LFC ≈ 0)"

# Reconciliation note when count direction and magnitude direction differ
recon_note = ""
if n_gained > n_lost and median_lfc > 0:
    recon_note = (f"More peaks gained ({pct_gained:.1f}%) than lost ({pct_lost:.1f}%), "
                  f"and median LFC > 0 — CHD8 loss leads to a net opening of chromatin "
                  f"at these loci. Note that lost peaks may be larger in magnitude "
                  f"even if fewer in number; examine the LFC distribution carefully.")
elif n_lost > n_gained and median_lfc < 0:
    recon_note = (f"More peaks lost ({pct_lost:.1f}%) than gained ({pct_gained:.1f}%), "
                  f"consistent with a net reduction in accessibility.")
elif n_gained > n_lost and median_lfc < 0:
    recon_note = (f"RECONCILIATION NEEDED: more peaks are gained ({pct_gained:.1f}%) "
                  f"but median LFC is negative ({median_lfc:.3f}). This suggests that "
                  f"the lost peaks have larger magnitude scores. Report both in the text.")
elif n_lost > n_gained and median_lfc > 0:
    recon_note = (f"RECONCILIATION NEEDED: more peaks are lost ({pct_lost:.1f}%) "
                  f"but median LFC is positive ({median_lfc:.3f}). Investigate.")

# ── AUDIT ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FIGURE 6 — DATA AUDIT")
print("=" * 60)
print(f"  Overlapping peaks : {len(merged):,}")
print(f"  Gained (LFC > 0)  : {n_gained:,}  ({pct_gained:.1f}%)")
print(f"  Lost   (LFC < 0)  : {n_lost:,}  ({pct_lost:.1f}%)")
print(f"  Median LFC        : {median_lfc:+.3f}")
print(f"  Pearson r         : {r:.3f}")
print(f"  MWU p-value       : {mwu_p:.2e}")
print(f"\n  Direction: {direction_phrase}")
if recon_note:
    print(f"\n  Note: {recon_note}")
print("=" * 60)
print("\nCopy into Figure 6 legend:")
print(f"  n = {len(merged):,} matched peaks; gained = {n_gained:,} ({pct_gained:.1f}%); "
      f"lost = {n_lost:,} ({pct_lost:.1f}%); "
      f"median LFC = {median_lfc:+.3f}; Pearson r = {r:.3f}; "
      f"MWU p = {mwu_p:.2e}")

# ── FIGURE ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 7), dpi=600)

# Panel A — Violin
ax1 = axes[0]
sns.violinplot(y=lfc, ax=ax1, color='#3498DB',
               inner='quartile', linewidth=1.8)
ax1.axhline(0, color='#E74C3C', linestyle='--', linewidth=2,
            label='Zero (parity)')
ax1.axhline(median_lfc, color='#2C3E50', linestyle='-', linewidth=1.5,
            alpha=0.7, label=f'Median LFC = {median_lfc:+.3f}')

# Title is generated from data
ax1.set_title(
    f"Figure 6A: Chromatin accessibility {direction_word} in CHD8-KO\n"
    f"(n = {len(merged):,} matched peaks; MWU p = {mwu_p:.2e})",
    fontsize=11, fontweight='bold', pad=12
)
ax1.set_ylabel("log\u2082 Fold Change (KO / WT)", fontsize=12)
ax1.set_xlabel("")

ax1.text(0.05, 0.97,
         f"Gained: {n_gained:,} ({pct_gained:.1f}%)\n"
         f"Lost:   {n_lost:,} ({pct_lost:.1f}%)\n"
         f"Median: {median_lfc:+.3f}",
         transform=ax1.transAxes, va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
ax1.legend(fontsize=8, loc='lower right')
sns.despine(ax=ax1)

# Panel B — Scatter
ax2 = axes[1]
plot_idx = np.random.choice(len(merged), min(10000, len(merged)), replace=False)
ax2.scatter(wt_scores[plot_idx], ko_scores[plot_idx],
            alpha=0.2, s=2, color='#2C3E50', rasterized=True)
max_val = max(wt_scores.max(), ko_scores.max())
ax2.plot([0, max_val], [0, max_val], 'r--', lw=1.5, label='Parity (y=x)')
ax2.text(0.05, 0.95, f"Pearson r = {r:.3f}",
         transform=ax2.transAxes, va='top', fontsize=10, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
ax2.set_xlabel("WT ATAC-seq signal", fontsize=12)
ax2.set_ylabel("KO ATAC-seq signal", fontsize=12)
ax2.set_title("Figure 6B: WT vs KO signal correlation\n"
              "(coordinate-matched 50 bp bins)",
              fontsize=11, fontweight='bold', pad=12)
ax2.legend(fontsize=9, loc='lower right')
sns.despine(ax=ax2)

# Panel C — Bar (colour = direction)
ax3 = axes[2]
# Red = gained (more open), Blue = lost (more closed) — biologically intuitive
bar_colors = ['#E74C3C', '#3498DB']
bars = ax3.bar(
    [f"Gained\n(more open)\nn = {n_gained:,}",
     f"Lost\n(more closed)\nn = {n_lost:,}"],
    [n_gained, n_lost],
    color=bar_colors, edgecolor='black', linewidth=1, width=0.5
)
for bar, pct in zip(bars, [pct_gained, pct_lost]):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + len(merged) * 0.005,
             f"{pct:.1f}%",
             ha='center', va='bottom', fontweight='bold', fontsize=11)

ax3.set_ylabel("Number of ATAC-seq peaks", fontsize=12)
ax3.set_title("Figure 6C: Gained vs lost accessibility\nin CHD8-KO",
              fontsize=11, fontweight='bold', pad=12)
ax3.text(0.5, 0.97,
         f"Median LFC = {median_lfc:+.3f}",
         transform=ax3.transAxes, ha='center', va='top',
         fontsize=9, fontstyle='italic',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))
sns.despine(ax=ax3)

plt.suptitle(
    "Figure 6: CHD8-KO chromatin accessibility changes (ATAC-seq)",
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()

PLOT_PATH = os.path.join(SAVE_DIR, "Figure6_ATAC_Fixed_600DPI.png")
plt.savefig(PLOT_PATH, dpi=600, bbox_inches='tight')
plt.savefig(PLOT_PATH.replace('.png', '_vector.pdf'), bbox_inches='tight')
plt.show()
plt.close()

print(f"\nFigure saved: {PLOT_PATH}")
print("COMPLETE")

In [ ]:
# 1. Install the Python library
!pip install pybedtools matplotlib-venn

# 2. Re-import everything
import pybedtools
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

print("✅ Library 'pybedtools' is now installed and ready!")


In [ ]:
import os

def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    return None

# Search for the file in Drive
target_file = "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
found_path = find_file(target_file, "/content/drive/MyDrive/")

if found_path:
    print(f"✅ Found it! Use this path:")
    print(found_path)

    import pandas as pd
    temp_df = pd.read_csv(found_path, sep='\t')
    print("\n--- Data Preview ---")
    print(temp_df.head())
    print("\nColumns:", temp_df.columns.tolist())
else:
    print("❌ Still not finding it. Suggestions:")
    print("1. Ensure Google Drive is mounted: 'from google.colab import drive; drive.mount(\"/content/drive\")'")
    print("2. Check if the file ends in '.csv' instead of '.tsv'")
    print("3. Check if the file is in 'Shared with me' (you must shortcut it to 'My Drive' to see it in Colab)")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, ranksums


In [ ]:
BASE_PATH = "/content/drive/MyDrive/Chd8 data/"
DESEQ2_PATH = os.path.join(BASE_PATH, "deseq2/deseq2/results/")
OUTPUT_PATH = "/content/drive/MyDrive/CHD8_Figures_Output/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

print("DESEQ2_PATH:", DESEQ2_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)



In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import gseapy as gp
import pybedtools
import os
import pandas as pd
import seaborn as sns
import textwrap

# ----------------------------------------------------------
# PATHS  — update only if Drive structure differs
# ----------------------------------------------------------
PEAK_PATH   = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH    = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH    = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
OUT_DIR     = "/content/drive/MyDrive/CHD8_Fig2_Final"
os.makedirs(OUT_DIR, exist_ok=True)

# Global settings
plt.rcParams.update({
    'font.family'    : 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size'      : 12,
    'axes.linewidth' : 1.2,
    'pdf.fonttype'   : 42,   # editable text in Illustrator/Inkscape
})

# ----------------------------------------------------------
# HELPERS
# ----------------------------------------------------------
def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

def short_go_label(term, width=42):
    """Strip GO ID, wrap at width chars."""
    return '\n'.join(textwrap.wrap(term.split(' (GO')[0], width=width))

def annotate_kk(ax, bars, rows, fontsize=8.5):
    """Write k/K ratio inside each horizontal bar (white text)."""
    for bar, (_, row) in zip(bars, rows.iterrows()):
        w = bar.get_width()
        ax.text(
            w * 0.97,
            bar.get_y() + bar.get_height() / 2,
            f"k/K = {row['Overlap']}",
            va='center', ha='right',
            fontsize=fontsize, color='white', fontweight='bold'
        )

# ----------------------------------------------------------
# STEP 1  — verify files
# ----------------------------------------------------------
print("=" * 65)
print("FIGURE 2: NPC-SPECIFIC CHD8 TARGET ENRICHMENT ANALYSIS")
print("=" * 65)

missing = []
for label, path in [("NPC peaks",  PEAK_PATH),
                     ("ES peaks",   ESC_PATH),
                     ("TSS BED",    TSS_PATH),
                     ("KO RNA-seq", RNA_KO_PATH)]:
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(f"  {label}: {status}")
    if status == "MISSING":
        missing.append(label)

if missing:
    raise FileNotFoundError(
        f"Cannot continue — missing files: {missing}\n"
        "Check that Google Drive is mounted and paths above are correct."
    )

# ----------------------------------------------------------
# STEP 2  — compute NPC-specific ChIP targets
# ----------------------------------------------------------
print("\n1. Loading ChIP-seq peaks...")
npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
es_bt  = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()

# NPC-specific = NPC peaks with NO overlap in ES
npc_specific  = npc_bt.subtract(es_bt, A=True)
npc_genes_bed = tss_bt.intersect(npc_specific, u=True, wa=True)

# Preserve original case
# Enrichr mouse sets use Title Case (Sox2, Pax6, not SOX2, PAX6)
raw_genes     = sorted(set([str(f[3]) for f in npc_genes_bed]))
all_npc_genes = [g for g in raw_genes
                 if g.lower() not in ('nan', '.', '') and len(g) > 1]

print(f"   NPC total peaks        : {npc_bt.count():,}")
print(f"   NPC-specific peaks     : {npc_specific.count():,}")
print(f"   NPC-specific gene list : {len(all_npc_genes):,}")

# Auto-detect case style and set Enrichr organism accordingly
n_upper = sum(1 for g in all_npc_genes[:100] if g == g.upper())
ENRICHR_ORGANISM = "Human" if n_upper > 80 else "mouse"
print(f"   Symbol style: {'ALL-CAPS → organism=Human' if ENRICHR_ORGANISM == 'Human' else 'Title Case → organism=mouse'}")

# ----------------------------------------------------------
# STEP 3  — intersect with KO DEGs for focused gene list
# ----------------------------------------------------------
print("\n2. Subsetting to NPC-specific targets that are KO DEGs...")

ko_df    = pd.read_csv(RNA_KO_PATH, sep='\t').dropna(
    subset=['log2FoldChange', 'padj'])
gene_col = next((c for c in ko_df.columns
                 if c.upper() in ['GENESYMBOL','SYMBOL','GENE','MGI_SYMBOL']),
                None)
if gene_col is None:
    raise ValueError(f"Gene column not found. Columns: {ko_df.columns.tolist()}")

ko_df['gene_norm']    = ko_df[gene_col].astype(str).str.upper()
npc_norm_map          = {g.upper(): g for g in all_npc_genes}
ko_df['is_npc_target']= ko_df['gene_norm'].isin(npc_norm_map)
ko_df['is_deg']       = (ko_df['padj'] < 0.05) & (ko_df['log2FoldChange'].abs() > 0.5)

focused_genes  = [npc_norm_map[g]
                  for g in ko_df.loc[ko_df['is_npc_target'] & ko_df['is_deg'],
                                     'gene_norm']
                  if g in npc_norm_map]

print(f"   KO DEGs intersecting NPC targets: {len(focused_genes):,}")

if len(focused_genes) < 30:
    print("   Falling back to full NPC target list (intersection too small).")
    enrich_genes = all_npc_genes
    enrich_label = f"all NPC-specific targets (n={len(all_npc_genes):,})"
else:
    enrich_genes = focused_genes
    enrich_label = f"NPC targets \u2229 KO DEGs (n={len(focused_genes):,})"

print(f"   Enrichment input: {enrich_label}")

# ----------------------------------------------------------
# STEP 4  — run Enrichr
# ----------------------------------------------------------
print("\n3. Running GO Biological Process enrichment (Enrichr)...")
enr_go = gp.enrichr(
    gene_list = enrich_genes,
    gene_sets = ['GO_Biological_Process_2023'],
    organism  = ENRICHR_ORGANISM
)

print("   Running TF enrichment (ChEA 2022)...")
enr_tf = gp.enrichr(
    gene_list = enrich_genes,
    gene_sets = ['ChEA_2022'],
    organism  = ENRICHR_ORGANISM
)

# Choose significance threshold — FDR preferred, nominal fallback
go_n_sig = (enr_go.results['Adjusted P-value'] < 0.05).sum()
if go_n_sig == 0 and (enr_go.results['P-value'] < 0.05).sum() > 0:
    pval_col    = 'P-value'
    pval_thresh = 0.05
    pval_label  = 'nominal P < 0.05 (FDR correction too stringent)'
    print(f"   Using nominal P-value (FDR stringent at n={len(enrich_genes)})")
else:
    pval_col    = 'Adjusted P-value'
    pval_thresh = 0.05
    pval_label  = 'FDR < 0.05'

print(f"   GO terms passing threshold: "
      f"{(enr_go.results[pval_col] < pval_thresh).sum()}")
print(f"   TF terms passing threshold: "
      f"{(enr_tf.results[pval_col] < pval_thresh).sum()}")

# Pre-filter top results (used across multiple panels)
top_go = (enr_go.results[enr_go.results[pval_col] < pval_thresh]
          .head(10).copy())
top_tf = (enr_tf.results[enr_tf.results[pval_col] < pval_thresh]
          .head(10).copy())

if not top_go.empty:
    top_go['log_p']     = -np.log10(top_go[pval_col])
    top_go['ShortLabel']= top_go['Term'].apply(short_go_label)
    top_go = top_go.sort_values('log_p', ascending=True)

if not top_tf.empty:
    top_tf['log_p'] = -np.log10(top_tf[pval_col])
    top_tf['Label'] = (top_tf['Term'].str.split('_').str[0]
                       + " (" + top_tf['Overlap'] + ")")
    top_tf = top_tf.sort_values('log_p', ascending=True)

# ----------------------------------------------------------
# PANEL A — GO bar chart  (Figure 2A)
# Horizontal bars, k/K annotated inside, dynamic height
# ----------------------------------------------------------
print("\n4. Generating Panel A: GO bar chart...")

if top_go.empty:
    print("   No significant GO terms — printing top 5 raw results:")
    print(enr_go.results[['Term','P-value','Adjusted P-value',
                           'Overlap']].head(5).to_string())
else:
    n_terms_a = len(top_go)
    fig_h_a   = max(6, n_terms_a * 0.72 + 2.0)

    fig2a, ax = plt.subplots(figsize=(12, fig_h_a))
    bars_a    = ax.barh(top_go['ShortLabel'], top_go['log_p'],
                        color='#2C3E50', edgecolor='white', height=0.58)

    annotate_kk(ax, bars_a, top_go)

    ax.axvline(-np.log10(pval_thresh), color='#E74C3C',
               linestyle='--', lw=1.2, label=pval_label)
    ax.set_xlabel(f'-log\u2081\u2080 ({pval_col})',
                  fontsize=12, fontweight='bold')
    ax.set_title(
        f'Figure 2A: GO Biological Process Enrichment\n({enrich_label})',
        fontsize=13, fontweight='bold', pad=12
    )
    ax.tick_params(axis='y', labelsize=9.5)
    ax.tick_params(axis='x', labelsize=10)
    ax.legend(fontsize=9, loc='lower right')
    sns.despine(ax=ax)

    plt.tight_layout()
    plt.subplots_adjust(left=0.38)

    plt.savefig(f"{OUT_DIR}/Figure2A_GO_600DPI.png",
                dpi=600, bbox_inches='tight')
    plt.savefig(f"{OUT_DIR}/Figure2A_GO_vector.pdf",
                bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"   Saved — {n_terms_a} GO terms, k/K inside bars")

# ----------------------------------------------------------
# PANEL B — TF co-occupancy bar chart  (Figure 2B)
# ----------------------------------------------------------
print("\n5. Generating Panel B: TF bar chart...")

if top_tf.empty:
    print("   No significant TF terms — printing top 5 raw results:")
    print(enr_tf.results[['Term','P-value','Adjusted P-value',
                           'Overlap']].head(5).to_string())
else:
    n_terms_b = len(top_tf)
    fig_h_b   = max(6, n_terms_b * 0.72 + 2.0)

    fig2b, ax = plt.subplots(figsize=(11, fig_h_b))
    bars_b    = ax.barh(top_tf['Label'], top_tf['log_p'],
                        color='#2C3E50', edgecolor='white', height=0.58)

    # Annotate overlap inside bars
    for bar, (_, row) in zip(bars_b, top_tf.iterrows()):
        w = bar.get_width()
        ax.text(w * 0.97, bar.get_y() + bar.get_height() / 2,
                row['Overlap'],
                va='center', ha='right',
                fontsize=8.5, color='white', fontweight='bold')

    ax.axvline(-np.log10(pval_thresh), color='#E74C3C',
               linestyle='--', lw=1.2, label=pval_label)
    ax.set_xlabel(f'-log\u2081\u2080 ({pval_col})',
                  fontsize=12, fontweight='bold')
    ax.set_title(
        f'Figure 2B: Transcription Factor Co-occupancy (ChEA 2022)\n'
        f'({enrich_label})',
        fontsize=13, fontweight='bold', pad=12
    )
    ax.tick_params(axis='y', labelsize=10)
    ax.tick_params(axis='x', labelsize=10)
    ax.legend(fontsize=9, loc='lower right')
    sns.despine(ax=ax)

    plt.tight_layout()
    plt.subplots_adjust(left=0.28)

    plt.savefig(f"{OUT_DIR}/Figure2B_TF_600DPI.png",
                dpi=600, bbox_inches='tight')
    plt.savefig(f"{OUT_DIR}/Figure2B_TF_vector.pdf",
                bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"   Saved — {n_terms_b} TF terms")

# ----------------------------------------------------------
# PANEL C — GO ranked horizontal bar  (Figure 2C)
# Same style as Panel A, ranked highest → lowest significance
# ----------------------------------------------------------
print("\n6. Generating Panel C: GO ranked bar...")

if top_go.empty:
    print("   Skipped — no GO results.")
else:
    top_go_c  = top_go.sort_values('log_p', ascending=False).copy()
    n_terms_c = len(top_go_c)
    fig_h_c   = max(6, n_terms_c * 0.72 + 2.0)

    fig2c, ax = plt.subplots(figsize=(12, fig_h_c))
    bars_c    = ax.barh(range(n_terms_c), top_go_c['log_p'],
                        color='#2C3E50', edgecolor='white', height=0.58)

    ax.set_yticks(range(n_terms_c))
    ax.set_yticklabels(top_go_c['ShortLabel'].tolist(), fontsize=9.5)
    ax.invert_yaxis()   # rank 1 (highest significance) at top

    annotate_kk(ax, bars_c, top_go_c)

    ax.axvline(-np.log10(pval_thresh), color='#E74C3C',
               linestyle='--', lw=1.2, label=pval_label)
    ax.set_xlabel(f'-log\u2081\u2080 ({pval_col})',
                  fontsize=12, fontweight='bold')
    ax.set_title(
        f'Figure 2C: GO Enrichment ranked by adjusted P-value\n'
        f'({enrich_label})',
        fontsize=13, fontweight='bold', pad=12
    )
    ax.tick_params(axis='x', labelsize=10)
    ax.legend(fontsize=9, loc='lower right')
    sns.despine(ax=ax)

    plt.tight_layout()
    plt.subplots_adjust(left=0.38)

    plt.savefig(f"{OUT_DIR}/Figure2C_GO_ranked_600DPI.png",
                dpi=600, bbox_inches='tight')
    plt.savefig(f"{OUT_DIR}/Figure2C_GO_ranked_vector.pdf",
                bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"   Saved — horizontal bars, ranked, no x-axis rotation")

# ----------------------------------------------------------
# PANEL D — functional dot plot  (Figure 2D)
# ----------------------------------------------------------
print("\n7. Generating Panel D: functional dot plot...")

themes = ['Neurogenesis', 'Synaptic', 'Chromatin', 'Transcription', 'Axon']
plot_data = []

for theme in themes:
    # Search across ALL GO results, not just significant ones
    subset = (enr_go.results[
        enr_go.results['Term'].str.contains(theme, case=False)]
              .head(1))
    if not subset.empty:
        genes = [g.strip() for g in
                 str(subset['Genes'].values[0]).split(';') if g.strip()]
        # Use raw -log10 P so dot size/colour still varies meaningfully
        sig   = -np.log10(max(subset['P-value'].values[0], 1e-300))
        for g in genes[:10]:
            plot_data.append({'Category': theme,
                              'Gene'    : g,
                              'Significance': sig})

if not plot_data:
    print("   WARNING: None of the theme keywords matched GO results.")
    print("   Top GO terms available:")
    print(enr_go.results['Term'].head(15).tolist())
    print("   Update 'themes' list above to match terms shown.")
else:
    df_dot   = pd.DataFrame(plot_data)
    sig_min  = df_dot['Significance'].min()
    sig_max  = df_dot['Significance'].max()
    norm     = mcolors.Normalize(vmin=sig_min, vmax=sig_max)

    # Dot sizes: scale between 50 and 500 pt²
    size_range = sig_max - sig_min if sig_max > sig_min else 1
    df_dot['dot_size'] = (
        (df_dot['Significance'] - sig_min) / size_range * 450 + 50
    )

    n_genes = df_dot['Gene'].nunique()
    fig_h_d = max(8, n_genes * 0.38 + 2.5)

    fig2d, ax = plt.subplots(figsize=(9, fig_h_d))
    sns.set_style("whitegrid")

    scatter = ax.scatter(
        x          = df_dot['Category'],
        y          = df_dot['Gene'],
        s          = df_dot['dot_size'],
        c          = df_dot['Significance'],
        cmap       = 'magma',
        norm       = norm,
        edgecolors = 'black',
        linewidths = 0.5,
        zorder     = 3
    )

    # Colorbar (avoids seaborn legend deprecation warning)
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.35, pad=0.02)
    cbar.set_label('-log\u2081\u2080 nominal P-value', fontsize=10)

    # Size legend (manual)
    for size_val, label_txt in [(50, 'low'), (275, 'mid'), (500, 'high')]:
        ax.scatter([], [], s=size_val, c='#888888',
                   edgecolors='black', linewidths=0.5,
                   label=label_txt)
    ax.legend(title='Significance', title_fontsize=9,
              fontsize=8, loc='lower right',
              framealpha=0.8, edgecolor='#cccccc')

    ax.set_xlabel('Functional category', fontsize=12, fontweight='bold')
    ax.set_ylabel('Gene', fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=15, labelsize=11)
    ax.tick_params(axis='y', labelsize=9)
    ax.grid(True, linestyle=':', alpha=0.4, zorder=0)
    ax.set_title(
        'Figure 2D: Functional Grouping of NPC-specific CHD8 Targets\n'
        '(dot size and colour = enrichment significance)',
        fontsize=13, fontweight='bold', pad=12
    )

    plt.tight_layout()

    plt.savefig(f"{OUT_DIR}/Figure2D_DotPlot_600DPI.png",
                dpi=600, bbox_inches='tight')
    plt.savefig(f"{OUT_DIR}/Figure2D_DotPlot_vector.pdf",
                bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"   Saved — {len(df_dot)} gene-category pairs, "
          f"{df_dot['Category'].nunique()} themes")

# ----------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------
print("\n" + "=" * 65)
print(f"All panels saved to: {OUT_DIR}")
print(f"  Figure2A_GO_600DPI.png        + vector PDF")
print(f"  Figure2B_TF_600DPI.png        + vector PDF")
print(f"  Figure2C_GO_ranked_600DPI.png + vector PDF")
print(f"  Figure2D_DotPlot_600DPI.png   + vector PDF")
print(f"\nEnrichment input : {enrich_label}")
print(f"Full NPC targets : {len(all_npc_genes):,} genes")
print("=" * 65)

# Variable available for Cell 22 (target volcano highlighting)
npc_target_genes = all_npc_genes
print(f"\nnpc_target_genes ready ({len(npc_target_genes):,} genes) "
      "— use in Cell 22 to highlight CHD8 targets on volcano.")

In [ ]:
theme = 'Transcription'
subset = (enr_go.results[
    enr_go.results['Term'].str.contains(theme, case=False)]
          .sort_values('P-value')
          .head(3))  # show top 3 matches, not just 1, for context

print(f"Top GO terms matching '{theme}':")
print(subset[['Term', 'P-value', 'Overlap', 'Genes']].to_string(index=False))

# Check overlap with the Chromatin module's gene list
chrom_subset = (enr_go.results[
    enr_go.results['Term'].str.contains('Chromatin', case=False)]
    .sort_values('P-value').head(1))

if not subset.empty and not chrom_subset.empty:
    trans_genes = set(str(subset['Genes'].values[0]).split(';'))
    chrom_genes = set(str(chrom_subset['Genes'].values[0]).split(';'))
    overlap = trans_genes & chrom_genes
    print(f"\nTranscription module genes: {trans_genes}")
    print(f"Chromatin module genes:     {chrom_genes}")
    print(f"Shared genes: {overlap} ({len(overlap)}/{len(trans_genes)} of Transcription module)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

themes = ['Neurogenesis', 'Synaptic', 'Chromatin', 'Axon', 'Transcription']

print("=" * 65)
print("FIGURE 4A: CATEGORY-LEVEL FUNCTIONAL MODULE SUMMARY")
print("=" * 65)

module_rows = []
gene_list_rows = []

for theme in themes:
    subset = (enr_go.results[
        enr_go.results['Term'].str.contains(theme, case=False)]
              .sort_values('P-value')
              .head(1))
    if subset.empty:
        print(f"   WARNING: no GO term matched theme '{theme}'")
        continue

    term       = subset['Term'].values[0]
    pval       = subset['P-value'].values[0]
    overlap    = subset['Overlap'].values[0]
    genes      = [g.strip() for g in str(subset['Genes'].values[0]).split(';') if g.strip()]

    module_rows.append({
        'Module': theme,
        'Representative_GO_term': term,
        'P-value': pval,
        'neg_log10_P': -np.log10(max(pval, 1e-300)),
        'Overlap': overlap,
        'n_genes': len(genes),
    })
    for g in genes:
        gene_list_rows.append({'Module': theme, 'Gene': g})

module_df = pd.DataFrame(module_rows).sort_values('neg_log10_P', ascending=True)
gene_df   = pd.DataFrame(gene_list_rows)

print(module_df[['Module', 'Representative_GO_term', 'P-value', 'n_genes']].to_string(index=False))

# ── Plot: one bar per module, no per-gene dots ──────────────────────────────
fig, ax = plt.subplots(figsize=(7, 0.9 * len(module_df) + 1.5))

bars = ax.barh(module_df['Module'], module_df['neg_log10_P'],
                color='#2C3E50', edgecolor='white', height=0.55)

# Annotate each bar with n genes and overlap fraction
for bar, (_, row) in zip(bars, module_df.iterrows()):
    ax.text(bar.get_width() * 0.98, bar.get_y() + bar.get_height() / 2,
             f"n={row['n_genes']}  ({row['Overlap']})",
             va='center', ha='right', fontsize=8.5, color='white', fontweight='bold')

ax.set_xlabel('-log\u2081\u2080 (nominal P-value)', fontsize=11, fontweight='bold')
ax.set_title('Figure 4A: Functional modules among NPC-specific CHD8 targets',
             fontsize=12, fontweight='bold', pad=10)
ax.tick_params(axis='y', labelsize=10)
ax.tick_params(axis='x', labelsize=9)
import seaborn as sns
sns.despine(ax=ax)
plt.tight_layout()

plt.savefig(f"{OUT_DIR}/Figure4A_ModuleSummary_600DPI.png", dpi=600, bbox_inches='tight')
plt.savefig(f"{OUT_DIR}/Figure4A_ModuleSummary_vector.pdf", bbox_inches='tight')
plt.show()
plt.close()

# ── Save gene lists per module for supplement (not plotted individually) ───
gene_out_path = f"{OUT_DIR}/Supplementary_Table_S11_module_genes.csv"
gene_df.to_csv(gene_out_path, index=False)
print(f"\n✅ Figure saved: {OUT_DIR}/Figure4A_ModuleSummary_600DPI.png")
print(f"✅ Gene lists per module saved: {gene_out_path}")

# ── Download TIFF + PDF + CSV ───────────────────────────────────────────────
# (PNG above is 600 DPI for preview; re-save as TIFF for journal submission.)
import matplotlib.pyplot as plt

tiff_path = f"{OUT_DIR}/Figure4A_ModuleSummary_600DPI.tiff"
pdf_path  = f"{OUT_DIR}/Figure4A_ModuleSummary_vector.pdf"

fig.savefig(tiff_path, dpi=600, bbox_inches='tight', format='tiff',
            pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(pdf_path, dpi=600, bbox_inches='tight', format='pdf')

for path, label in [(tiff_path, 'TIFF'), (pdf_path, 'PDF'), (gene_out_path, 'CSV')]:
    mb = os.path.getsize(path) / 1e6
    print(f"✅ {label}: {path} ({mb:.2f} MB)")

from google.colab import files as colab_files
colab_files.download(tiff_path)
colab_files.download(pdf_path)
colab_files.download(gene_out_path)
print("\n📥 Downloads triggered.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
import os

# 1. PATHS
paths = {
    'KO': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    'FL': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    'dC': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    'dH': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv"
}
SAVE_DIR = "/content/drive/MyDrive/Chd8 data/figures/"

def generate_figure_6_REVIEWER_PROOF(synchronized_target_list, save_path):
    print(f"🚀 Processing Rescue for {len(synchronized_target_list)} NPC targets...")

    # --- DATA LOADING & MERGING ---
    dfs = []
    for label, path in paths.items():
        df = pd.read_csv(path, sep="\t")
        df["gene_id"] = df.iloc[:, 0].astype(str).str.split(".").str[0]
        dfs.append(df[["gene_id", "GENESYMBOL", "log2FoldChange"]].rename(columns={"log2FoldChange": f"LFC_{label}"}))

    master = dfs[0]
    for d in dfs[1:]:
        master = master.merge(d.drop(columns=['GENESYMBOL']), on="gene_id")

    # --- SYNCHRONIZATION & STABILITY FILTER (FIX 1 & 2) ---
    target_set = set([g.upper() for g in synchronized_target_list])
    master['gene_upper'] = master['GENESYMBOL'].astype(str).str.upper()

    # We focus on genes with a clear direction in KO (Abs LFC > 0.5) to avoid noise/division by zero
    targets = master[(master['gene_upper'].isin(target_set)) & (master['LFC_KO'].abs() > 0.5)].copy()

    # Robust Rescue Formula: (Cond - KO) / (WT - KO) where WT is 0
    for cond in ["FL", "dC", "dH"]:
        targets[f"RF_{cond}"] = (targets[f"LFC_{cond}"] - targets["LFC_KO"]) / (0 - targets["LFC_KO"])

    # --- STATS WITH MULTIPLE TESTING CORRECTION (FIX 3) ---
    _, fried_p = friedmanchisquare(targets["RF_FL"], targets["RF_dC"], targets["RF_dH"])

    # Raw P-values
    _, p_fl_dc_raw = wilcoxon(targets["RF_FL"], targets["RF_dC"], alternative='two-sided')
    _, p_fl_dh_raw = wilcoxon(targets["RF_FL"], targets["RF_dH"], alternative='two-sided')

    # BH-Correction
    _, pvals_corr, _, _ = multipletests([p_fl_dc_raw, p_fl_dh_raw], method='fdr_bh')
    p_fl_dc, p_fl_dh = pvals_corr

    # --- VISUALIZATION (FIX 4) ---
    plt.style.use('seaborn-v0_8-white')
    fig, ax = plt.subplots(figsize=(11, 8), dpi=600)
    colors = ["#27ae60", "#f1c40f", "#e74c3c"]
    cols_to_plot = ["RF_FL", "RF_dC", "RF_dH"]

    # Layer 1: Violin (Density)
    sns.violinplot(data=targets[cols_to_plot], palette=colors, inner=None, alpha=0.15, ax=ax)

    # Layer 2: Boxplot (Summary)
    sns.boxplot(data=targets[cols_to_plot], palette=colors, showfliers=False, width=0.3, linewidth=2, ax=ax)

    # Layer 3: Stripplot (Raw Data - Fix 4)
    sns.stripplot(data=targets[cols_to_plot], color='black', size=1.5, alpha=0.2, jitter=0.2, ax=ax)

    # Reference Lines
    ax.axhline(1, ls="--", color="#2980b9", alpha=0.4, label="Full Rescue")
    ax.axhline(0, ls="-", color="black", alpha=0.2, label="No Rescue")
    ax.set_ylim(-1.5, 2.5)

    ax.set_xticklabels(["Full Length", "ΔChromo", "ΔHelicase"], fontsize=14, fontweight="bold")
    ax.set_ylabel("Rescue Fraction (Normalized)", fontsize=14, fontweight="bold")
    ax.set_title(f"Figure 6: Domain-Specific Rescue of Direct NPC Targets (n={len(targets)})\n"
                 f"Friedman p = {fried_p:.1e} | FL vs ΔH (adj. p) = {p_fl_dh:.1e}", fontsize=15, fontweight='bold', pad=25)

    sns.despine(offset=10, trim=True)
    plt.tight_layout()

    # Save
    plt.savefig(os.path.join(save_path, "Figure_6_Unbiased_600DPI.png"), dpi=600, bbox_inches='tight')
    plt.savefig(os.path.join(save_path, "Figure_6_Unbiased_Vector.pdf"), bbox_inches='tight')
    plt.show()

    print(f"\n📊 Medians: FL={targets['RF_FL'].median()*100:.1f}%, dC={targets['RF_dC'].median()*100:.1f}%, dH={targets['RF_dH'].median()*100:.1f}%")

# EXECUTE
generate_figure_6_REVIEWER_PROOF(final_npc_genes, SAVE_DIR)

In [ ]:
# 1. Install the BEDTools binaries (Required for the backend)
!apt-get install -y bedtools

# 2. Install the Python wrapper and statsmodels for FDR correction
!pip install pybedtools statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare
import pybedtools
import os

# 1. PATHS
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"

paths = {
    'KO': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    'FL': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    'dC': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    'dH': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv"
}

def fix_naming(feature):
    chrom = str(feature.chrom)
    if not chrom.startswith('chr'): feature.chrom = 'chr' + chrom
    return feature

def generate_figure_6B_FINAL_REVISION():
    print("🚀 Step 1: Synchronizing ChIP-seq Targets...")
    npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_naming).sort()
    esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_naming).sort()
    tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()

    npc_only = npc_bt.subtract(esc_bt, A=True)
    hits = tss_bt.intersect(npc_only, u=True, wa=True)
    target_set = set([str(f[3]).upper().strip() for f in hits if len(str(f[3])) > 1])

    # --- DATA LOADING ---
    dfs = []
    for label, path in paths.items():
        df = pd.read_csv(path, sep="\t")
        df["gene_id"] = df.iloc[:, 0].astype(str).str.split(".").str[0]
        dfs.append(df[["gene_id", "GENESYMBOL", "log2FoldChange"]].rename(columns={"log2FoldChange": f"LFC_{label}"}))

    master = dfs[0]
    for d in dfs[1:]:
        master = master.merge(d.drop(columns=['GENESYMBOL']), on="gene_id")

    # --- FILTERING ---
    master['gene_upper'] = master['GENESYMBOL'].astype(str).str.upper()
    targets = master[(master['gene_upper'].isin(target_set)) & (master['LFC_KO'].abs() > 0.5)].copy()
    total_n = len(targets)

    # --- RESCUE CALCULATION (Robust Formula) ---
    for cond in ["FL", "dC", "dH"]:
        targets[f"RF_{cond}"] = (targets[f"LFC_{cond}"] - targets["LFC_KO"]) / (0 - targets["LFC_KO"])

    # Threshold = 50%
    targets['FL_r'] = targets['RF_FL'] >= 0.5
    targets['dC_r'] = targets['RF_dC'] >= 0.5
    targets['dH_r'] = targets['RF_dH'] >= 0.5

    # --- CATEGORIES (Intuitive Biological Order) ---
    c1 = (targets['FL_r'] & targets['dC_r'] & targets['dH_r']).sum()    # Redundant
    c3 = (targets['FL_r'] & ~targets['dC_r'] & targets['dH_r']).sum()   # Chromo Req
    c2 = (targets['FL_r'] & targets['dC_r'] & ~targets['dH_r']).sum()   # Helicase Req
    c4 = (targets['FL_r'] & ~targets['dC_r'] & ~targets['dH_r']).sum()  # Both Req
    c5 = (~targets['FL_r']).sum()                                      # Low Rescue

    categories = {
        'Global\nRescue': c1,
        'Chromo\nRequired': c3,
        'Helicase\nRequired': c2,
        'Dual Domain\nRequired': c4,
        'Low/No\nRescue': c5
    }

    # --- STATISTICS (Chi-Square) ---
    counts = list(categories.values())
    chi2_stat, p_cat = chisquare(counts)

    # --- VISUALIZATION ---
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(12, 7), dpi=600)
    colors = ['#27ae60', '#3498db', '#2ecc71', '#9b59b6', '#95a5a6']

    bars = ax.bar(categories.keys(), counts, color=colors, edgecolor='black', linewidth=1.2)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + (total_n * 0.01),
                f'{int(height)}\n({100*height/total_n:.1f}%)',
                ha='center', va='bottom', fontweight='bold', fontsize=10)

    ax.set_ylabel(f"Number of Direct Targets (n={total_n})", fontweight='bold')
    ax.set_title(f"Figure 6B: Mechanistic Categorization of CHD8 Target Rescue\n(Chi-square p = {p_cat:.1e})",
                 fontweight='bold', pad=25)
    sns.despine()

    # Save High-Res PNG and Vector PDF
    plt.savefig(os.path.join(SAVE_DIR, "Figure_6B_Categorical_600DPI.png"), dpi=600, bbox_inches='tight')
    plt.savefig(os.path.join(SAVE_DIR, "Figure_6B_Categorical_Vector.pdf"), bbox_inches='tight')
    plt.show()

    print(f"✅ Statistics: Chi-square = {chi2_stat:.2f}, p = {p_cat:.2e}")

# RUN
generate_figure_6B_FINAL_REVISION()

In [ ]:
!pip install mygene -q

In [ ]:
import os
import pandas as pd
import numpy as np
import gseapy as gp
import mygene
import matplotlib.pyplot as plt

# ================================
# 📁 1. DEFINE PATHS & LOAD DATA
# ================================
INPUT_FILE = "/content/drive/MyDrive/Chd8 data/deseq2_results.csv"
BASE_DIR = "/content/drive/MyDrive/CHD8_project"

RNK_FILE = os.path.join(BASE_DIR, "results/gsea/gsea_ranked_list.rnk")
GSEA_OUTDIR = os.path.join(BASE_DIR, "results/gsea/")
TABLE_OUT = os.path.join(BASE_DIR, "results/tables/GSEA_full_results.csv")
PLOT_OUT = os.path.join(BASE_DIR, "results/plots/GSEA_barplot.png")

os.makedirs(os.path.dirname(RNK_FILE), exist_ok=True)
os.makedirs(os.path.dirname(TABLE_OUT), exist_ok=True)
os.makedirs(os.path.dirname(PLOT_OUT), exist_ok=True)

print("📥 Loading data...")
df = pd.read_csv(INPUT_FILE)

if "gene" not in df.columns:
    first_col = df.columns[0]
    df.rename(columns={first_col: "gene"}, inplace=True)

# ================================
# 🔄 2. CONVERT ENSEMBL → SYMBOL
# ================================
print("🔄 Converting ENSEMBL IDs → Gene Symbols...")
mg = mygene.MyGeneInfo()
ensembl_ids = df["gene"].unique().tolist()
results = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='mouse', as_dataframe=True)

if 'symbol' in results.columns:
    symbol_map = results['symbol'].dropna().to_dict()
    df['gene_symbol'] = df['gene'].map(symbol_map)
    df = df.dropna(subset=['gene_symbol'])
    print(f"✅ Successfully mapped {len(df)} genes to Symbols.")
else:
    raise ValueError("❌ ID conversion failed. Check internet connection or ID format.")

# ================================
# 🧹 3. RANKING & SAVE RNK
# ================================
print("💾 Saving ranked file...")
df["padj"] = df["padj"].replace(0, 1e-300)
df["ranking_score"] = -np.log10(df["padj"]) * np.sign(df["log2FoldChange"])

ranked_df = df.sort_values("ranking_score", ascending=False)
rnk = ranked_df[["gene_symbol", "ranking_score"]].drop_duplicates(subset="gene_symbol")
rnk.to_csv(RNK_FILE, sep="\t", index=False, header=False)

# ================================
# 🚀 4. RUN GSEA
# ================================
print("🚀 Running GSEA...")
gsea_res = gp.prerank(
    rnk=RNK_FILE,
    gene_sets="GO_Biological_Process_2023",
    outdir=GSEA_OUTDIR,
    permutation_num=1000,
    min_size=5,
    max_size=1000,
    seed=42,
    verbose=True
)

# ================================
# 📈 5. PROCESS RESULTS
# ================================
print("📈 Processing GSEA results...")
res = gsea_res.res2d

# Column Detection
fdr_col = next((c for c in ["FDR q-val", "fdr", "FDR"] if c in res.columns), None)
lead_col = next((c for c in ["Lead_genes", "lead_genes", "genes", "leading_edge"] if c in res.columns), None)
pval_col = next((c for c in ["NOM p-val", "pvalue"] if c in res.columns), "pvalue")

res = res.rename(columns={fdr_col: "FDR", pval_col: "pvalue", "nes": "NES"})

# --- SAFETY CHECK: GENE COUNT ---
if lead_col is None:
    res["Gene_Count"] = 0
    print("⚠️ Warning: No leading edge gene column found — Gene_Count set to 0")
else:
    res["Gene_Count"] = res[lead_col].apply(lambda x: len(str(x).split(";")) if pd.notnull(x) else 0)

# Save UNFILTERED Full Table
res.to_csv(TABLE_OUT)
print(f"✅ Full GSEA table saved: {len(res)} terms (including non-significant)")

# Filter Significant Hits
res_sig = res[res["FDR"] < 0.05].copy()
print(f"✅ Significant pathways found (FDR < 0.05): {len(res_sig)}")

# --- CONSOLE PRINT OF SIGNIFICANT TERMS ---
if not res_sig.empty:
    print("\n--- TOP SIGNIFICANT TERMS ---")
    print(res_sig[['Term', 'NES', 'FDR']].sort_values("NES", ascending=False).to_string())
    print("------------------------------\n")

# ================================
# 📊 6. FINAL PLOT
# ================================
print("📊 Generating high-detail final plot...")
res_up = res_sig.sort_values("NES", ascending=False).head(10)
res_down = res_sig.sort_values("NES", ascending=True).head(10)
plot_df = pd.concat([res_up, res_down])
plot_df["Term"] = plot_df["Term"].str.replace("_", " ").str.split("(").str[0]

plt.figure(figsize=(14, 10))
colors = ['#d62728' if x > 0 else '#1f77b4' for x in plot_df["NES"]]
bars = plt.barh(plot_df["Term"], plot_df["NES"], color=colors, alpha=0.8, edgecolor='black')

# Dynamic Label Placement
max_nes = plot_df["NES"].max()
min_nes = plot_df["NES"].min()

for i, (nes, fdr, count) in enumerate(zip(plot_df["NES"], plot_df["FDR"], plot_df["Gene_Count"])):
    align = 'left' if nes > 0 else 'right'
    offset = 0.05 if nes > 0 else -0.05

    plt.text(
        nes + offset, i,
        f"NES={nes:.2f}\nFDR={fdr:.2e}\n(n={count})",
        va='center',
        ha=align,
        fontsize=9,
        fontweight='bold'
    )

plt.xlim(min_nes - 1.8, max_nes + 1.8) # Extra space for scientific notation
plt.axvline(0, color='black', lw=1.5)
plt.xlabel("Normalized Enrichment Score (NES)", fontsize=12, fontweight='bold')
plt.title("GSEA: CHD8-Regulated Pathways (FDR < 0.05)", fontsize=14, pad=20, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOT_OUT, dpi=600)

print(f"✅ Detailed Plot saved: {PLOT_OUT}")
print("🎉 GSEA PIPELINE COMPLETED SUCCESSFULLY")

In [ ]:
import pandas as pd
df = pd.read_csv(
    "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/ATAC_Diff_1/ATAC_Diff_1.merged_score.bed",
    sep='\t', header=None, nrows=5
)
print(df)
print(f"\nNumber of columns: {len(df.columns)}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import pybedtools
import os

# ==========================================
# 📁 1. PATHS
# ==========================================
BASE_RNA  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
BASE_ATAC = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

# RNA-seq files
RNA_PATHS = {
    "KO" : BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    "FL" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    "dC" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    "dH" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}

# ATAC-seq files (noIgG, both replicates)
ATAC_WT_PATHS = [
    BASE_ATAC + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed",
]
ATAC_KO_PATHS = [
    BASE_ATAC + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed",
]

# ==========================================
# 🔧 2. HELPER FUNCTIONS
# ==========================================
def load_rna(path, label):
    df = pd.read_csv(path, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
    df['padj'] = df['padj'].replace(0, 1e-300)
    if 'baseMean' in df.columns:
        df = df[df['baseMean'] > 10]
    df['gene_upper'] = df['GENESYMBOL'].astype(str).str.upper().str.strip()
    print(f"✅ {label}: {len(df):,} genes loaded")
    return df

def load_atac_bed(path):
    if not os.path.exists(path):
        print(f"⚠️ Not found: {path}")
        return None
    df = pd.read_csv(path, sep='\t', header=None)
    score_col = 6 if len(df.columns) > 6 else 4
    out = df[[0, 1, 2, score_col]].copy()
    out.columns = ['chr', 'start', 'end', 'score']
    out['chr'] = out['chr'].astype(str)
    mask = ~out['chr'].str.startswith('chr')
    out.loc[mask, 'chr'] = 'chr' + out.loc[mask, 'chr']
    out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
    return out

def average_atac(paths, label):
    dfs = [load_atac_bed(p) for p in paths]
    dfs = [d for d in dfs if d is not None]
    combined = pd.concat(dfs, ignore_index=True)
    combined['bin'] = (combined['chr'] + ':' +
                       ((combined['start'] // 50)*50).astype(str))
    avg = combined.groupby('bin').agg(
        chr=('chr','first'),
        start=('start','min'),
        end=('end','max'),
        score=('score','mean')
    ).reset_index()
    print(f"✅ {label}: {len(avg):,} peaks averaged")
    return avg

def fix_naming(feature):
    chrom = str(feature.chrom)
    if not chrom.startswith('chr'):
        feature.chrom = 'chr' + chrom
    return feature

# ==========================================
# 📥 3. LOAD ALL DATA
# ==========================================
print("="*60)
print("BLOCK 7: INTEGRATIVE ATAC + DOMAIN RESCUE ANALYSIS")
print("="*60)

print("\n📥 Loading RNA-seq data...")
rna = {k: load_rna(v, k) for k, v in RNA_PATHS.items()}

print("\n📥 Loading ATAC-seq data...")
wt_atac = average_atac(ATAC_WT_PATHS, "WT ATAC")
ko_atac = average_atac(ATAC_KO_PATHS, "KO ATAC")

# ==========================================
# 🔬 4. IDENTIFY KO-LOST ATAC PEAKS
# ==========================================
print("\n🔬 Identifying KO-lost peaks...")

# Merge WT and KO on 50bp bins
wt_atac['bin'] = (wt_atac['chr'] + ':' +
                   ((wt_atac['start'] // 50)*50).astype(str))
ko_atac['bin'] = (ko_atac['chr'] + ':' +
                   ((ko_atac['start'] // 50)*50).astype(str))

merged_atac = pd.merge(
    wt_atac[['bin','chr','start','end','score']].rename(
        columns={'score':'wt_score'}),
    ko_atac[['bin','score']].rename(columns={'score':'ko_score'}),
    on='bin', how='inner'
)

pseudo = 0.5
merged_atac['lfc'] = np.log2(
    (merged_atac['ko_score'] + pseudo) /
    (merged_atac['wt_score'] + pseudo)
)

# KO-lost = peaks more closed in KO (LFC < -0.5)
ko_lost = merged_atac[merged_atac['lfc'] < -0.5].copy()
ko_gained = merged_atac[merged_atac['lfc'] > 0.5].copy()

print(f"✅ Total overlapping peaks  : {len(merged_atac):,}")
print(f"✅ KO-lost peaks (LFC<-0.5) : {len(ko_lost):,}")
print(f"✅ KO-gained peaks (LFC>0.5): {len(ko_gained):,}")

# ==========================================
# 🔬 5. FIND GENES NEAR KO-LOST PEAKS
# ==========================================
print("\n🔬 Finding genes near KO-lost peaks (within TSS ±2kb)...")

# Write KO-lost peaks to temp BED
KO_LOST_BED = "/content/ko_lost_peaks.bed"
ko_lost[['chr','start','end']].to_csv(
    KO_LOST_BED, sep='\t', header=False, index=False)

# Intersect with TSS
if os.path.exists(TSS_PATH):
    tss_bt      = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()
    ko_lost_bt  = pybedtools.BedTool(KO_LOST_BED).each(fix_naming).sort()
    hits        = tss_bt.intersect(ko_lost_bt, u=True, wa=True)
    lost_genes  = set([str(f[3]).upper().strip()
                       for f in hits if len(str(f[3])) > 1])
    print(f"✅ Genes near KO-lost peaks: {len(lost_genes):,}")
else:
    # Fallback: use coordinate proximity without TSS file
    print("⚠️ TSS file not found — using all KO-lost peak-associated genes")
    lost_genes = set()

# ==========================================
# 🧮 6. CALCULATE RESCUE EFFICIENCY
# ==========================================
print("\n🧮 Calculating rescue efficiency per construct...")

ko_df = rna['KO'][['gene_upper','log2FoldChange']].rename(
    columns={'log2FoldChange':'lfc_ko'})

results = {}
for construct in ['FL','dC','dH']:
    rescue_df = rna[construct][['gene_upper','log2FoldChange']].rename(
        columns={'log2FoldChange':f'lfc_{construct}'})

    merged = ko_df.merge(rescue_df, on='gene_upper')

    # Rescue fraction: how much of KO effect is reversed
    # RF = (lfc_rescue - lfc_ko) / (0 - lfc_ko)
    # RF = 1.0 means fully rescued, RF = 0 means not rescued
    denom = 0 - merged['lfc_ko']
    denom = denom.replace(0, np.nan)
    merged['RF'] = (merged[f'lfc_{construct}'] - merged['lfc_ko']) / denom
    merged['RF'] = merged['RF'].clip(-1, 2)  # clip extreme values

    # Classify near-lost-peak genes
    merged['near_lost_peak'] = merged['gene_upper'].isin(lost_genes)

    results[construct] = merged
    n_near = merged['near_lost_peak'].sum()
    print(f"   {construct}: {len(merged):,} genes, "
          f"{n_near:,} near KO-lost peaks")

# ==========================================
# 📊 7. STATISTICAL COMPARISON
# ==========================================
print("\n📊 Statistical comparison (MWU test):")
print("-"*60)

stats_rows = []
for construct in ['FL','dC','dH']:
    df = results[construct]
    near  = df[df['near_lost_peak']]['RF'].dropna()
    other = df[~df['near_lost_peak']]['RF'].dropna()

    if len(near) > 5 and len(other) > 5:
        _, pval = mannwhitneyu(near, other, alternative='two-sided')
    else:
        pval = np.nan

    median_near  = near.median()
    median_other = other.median()

    print(f"   {construct}: near-lost median RF = {median_near:.3f} | "
          f"other median RF = {median_other:.3f} | "
          f"MWU p = {pval:.2e}")

    stats_rows.append({
        'construct'     : construct,
        'median_near'   : median_near,
        'median_other'  : median_other,
        'n_near'        : len(near),
        'n_other'       : len(other),
        'pval'          : pval
    })

stats_df = pd.DataFrame(stats_rows)
print("-"*60)
print("📝 EXPECTED: dC and dH show LOWER rescue for near-lost-peak genes")
print("   (lower RF = less rescue = more CHD8-domain dependent)")

# ==========================================
# 📊 8. FIGURE (600 DPI)
# ==========================================
print("\n📊 Generating figure...")

fig, axes = plt.subplots(1, 3, figsize=(20, 7), dpi=600)
construct_labels = {'FL': 'Full-Length\nRescue',
                    'dC': 'ΔChromo\nRescue',
                    'dH': 'ΔHelicase\nRescue'}
colors_near  = '#E74C3C'   # red — near lost peak
colors_other = '#95A5A6'   # grey — other genes

for i, construct in enumerate(['FL','dC','dH']):
    ax = axes[i]
    df = results[construct]

    near_rf  = df[df['near_lost_peak']]['RF'].dropna()
    other_rf = df[~df['near_lost_peak']]['RF'].dropna()
    row      = stats_df[stats_df['construct'] == construct].iloc[0]

    # Violin plot
    plot_data = pd.DataFrame({
        'RF'   : pd.concat([near_rf, other_rf]),
        'Group': (['Near KO-lost peak'] * len(near_rf) +
                  ['Other genes'] * len(other_rf))
    })

    sns.violinplot(data=plot_data, x='Group', y='RF', ax=ax,
                   palette={'Near KO-lost peak': colors_near,
                             'Other genes': colors_other},
                   inner='quartile', linewidth=1.5)

    ax.axhline(0, color='black', linestyle='--', lw=1, alpha=0.5)
    ax.axhline(1, color='blue',  linestyle='--', lw=1, alpha=0.5,
               label='Full rescue')

    ax.set_title(f"7{chr(65+i)}: {construct_labels[construct]}",
                 fontweight='bold', pad=15)
    ax.set_xlabel("")
    ax.set_ylabel("Rescue Fraction (RF)" if i == 0 else "")
    ax.set_ylim(-1, 2)

    # Stats annotation
    textstr = (
        f"Near-lost: n={row['n_near']}\n"
        f"  median RF = {row['median_near']:.3f}\n"
        f"Other: n={row['n_other']}\n"
        f"  median RF = {row['median_other']:.3f}\n"
        f"MWU p = {row['pval']:.2e}"
    )
    ax.text(0.05, 0.97, textstr, transform=ax.transAxes,
            va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    sns.despine(ax=ax)

plt.suptitle(
    "Block 7: Chromatin Accessibility Loss Predicts\n"
    "Domain-Specific Rescue Failure (CHD8-KO NPCs)",
    fontsize=15, fontweight='bold', y=1.02
)
plt.tight_layout()

PLOT_PATH = os.path.join(SAVE_DIR, "Figure7_ATAC_Rescue_Integration_600DPI.png")
plt.savefig(PLOT_PATH, dpi=600, bbox_inches='tight')
plt.show()

# Save stats table
STATS_PATH = os.path.join(SAVE_DIR, "Block7_stats_summary.csv")
stats_df.to_csv(STATS_PATH, index=False)

print(f"\n✅ Figure saved : {PLOT_PATH}")
print(f"✅ Stats saved  : {STATS_PATH}")
print("🎉 BLOCK 7 COMPLETE")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pybedtools
import os

# ==========================================
# 📁 1. PATHS
# ==========================================
BASE_RNA = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

RNA_PATHS = {
    "KO" : BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    "FL" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
}

# ==========================================
# 🔧 2. HELPER FUNCTIONS
# ==========================================
def load_rna(path, label):
    df = pd.read_csv(path, sep='\t').dropna(
        subset=['log2FoldChange', 'padj'])
    df['padj'] = df['padj'].replace(0, 1e-300)
    if 'baseMean' in df.columns:
        df = df[df['baseMean'] > 10]
    df['gene_upper'] = (df['GENESYMBOL'].astype(str)
                        .str.upper().str.strip())
    print(f"✅ {label}: {len(df):,} genes loaded")
    return df

def fix_naming(feature):
    chrom = str(feature.chrom)
    if not chrom.startswith('chr'):
        feature.chrom = 'chr' + chrom
    return feature

# ==========================================
# 📥 3. LOAD RNA-seq DATA
# ==========================================
print("="*60)
print("BLOCK 8: RESCUE VALIDATION — VENN + BAR CHART")
print("="*60)

print("\n📥 Loading RNA-seq data...")
ko_df = load_rna(RNA_PATHS['KO'], 'KO')
fl_df = load_rna(RNA_PATHS['FL'], 'FL')

# ==========================================
# 🔬 4. GET DIRECT CHD8 TARGETS (ChIP-seq)
# ==========================================
print("\n🔬 Computing direct CHD8 targets from ChIP-seq...")

if os.path.exists(PEAK_PATH) and os.path.exists(ESC_PATH):
    npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_naming).sort()
    esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_naming).sort()
    tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_naming).sort()

    # NPC-specific peaks (not in ESC)
    npc_only = npc_bt.subtract(esc_bt, A=True)

    # Genes within 2kb of NPC-specific CHD8 peaks
    hits = tss_bt.intersect(npc_only, u=True, wa=True)
    direct_targets = set([
        str(f[3]).upper().strip()
        for f in hits if len(str(f[3])) > 1
    ])
    print(f"✅ Direct CHD8 targets (ChIP+TSS): {len(direct_targets):,}")
else:
    print("⚠️ ChIP peak files not found — using all KO DEGs as targets")
    direct_targets = set(
        ko_df[ko_df['padj'] < 0.05]['gene_upper'].tolist()
    )

# ==========================================
# 🧮 5. COMPUTE RESCUE GENE LISTS
# ==========================================
print("\n🧮 Computing rescued and non-rescued gene lists...")

# Merge KO and FL
merged = ko_df[['gene_upper','log2FoldChange']].rename(
    columns={'log2FoldChange':'lfc_ko'}).merge(
    fl_df[['gene_upper','log2FoldChange']].rename(
        columns={'log2FoldChange':'lfc_fl'}),
    on='gene_upper'
)

# Rescue fraction
denom = (0 - merged['lfc_ko']).replace(0, np.nan)
merged['RF'] = (merged['lfc_fl'] - merged['lfc_ko']) / denom
merged['RF'] = merged['RF'].clip(-1, 2)

# Classify
merged['is_target']   = merged['gene_upper'].isin(direct_targets)
merged['is_rescued']  = merged['RF'] >= 0.5
merged['is_dysreg']   = merged['lfc_ko'].abs() > 0.5

# Gene sets
target_genes    = set(merged[merged['is_target']]['gene_upper'])
rescued_genes   = set(merged[merged['is_rescued']]['gene_upper'])
nonrescued_genes = set(merged[~merged['is_rescued']]['gene_upper'])
dysreg_targets  = set(merged[
    merged['is_target'] & merged['is_dysreg']]['gene_upper'])

print(f"✅ Total genes in merged dataset  : {len(merged):,}")
print(f"✅ Direct CHD8 targets in RNA-seq : {len(target_genes):,}")
print(f"✅ Rescued genes (RF ≥ 0.5)       : {len(rescued_genes):,}")
print(f"✅ Non-rescued genes (RF < 0.5)   : {len(nonrescued_genes):,}")
print(f"✅ Dysregulated targets            : {len(dysreg_targets):,}")

# ==========================================
# 📊 6. RESCUE RATE COMPARISON
# ==========================================
print("\n📊 Rescue rate: Direct Targets vs Indirect DEGs...")

target_rescued    = target_genes & rescued_genes
target_nonrescued = target_genes & nonrescued_genes
indirect_genes    = set(merged['gene_upper']) - target_genes
indirect_rescued  = indirect_genes & rescued_genes

rescue_rate_target   = (len(target_rescued) /
                         len(target_genes) * 100
                         if target_genes else 0)
rescue_rate_indirect = (len(indirect_rescued) /
                         len(indirect_genes) * 100
                         if indirect_genes else 0)

print(f"-"*60)
print(f"Direct targets rescued   : "
      f"{len(target_rescued):,} / {len(target_genes):,} "
      f"({rescue_rate_target:.1f}%)")
print(f"Indirect DEGs rescued    : "
      f"{len(indirect_rescued):,} / {len(indirect_genes):,} "
      f"({rescue_rate_indirect:.1f}%)")
print(f"-"*60)
print(f"📝 Update - with these rescue rates")

# ==========================================
# 📊 7. FIGURE — 3 PANELS (600 DPI)
# ==========================================
print("\n📊 Generating figure...")

fig, axes = plt.subplots(1, 3, figsize=(21, 7), dpi=600)

# --- PANEL A: Venn — Direct Targets vs Rescued Genes ---
ax1 = axes[0]
ax1.set_title("8A: Direct CHD8 Targets\nvs Rescued Genes (FL)",
              fontweight='bold', pad=15)

v = venn2(
    subsets=[target_genes, rescued_genes],
    set_labels=('', ''),          # blank — we add manually below
    set_colors=('#2C3E50', '#27AE60'),
    alpha=0.7,
    ax=ax1
)

# Manual labels positioned to avoid overlap
ax1.text(-0.55, 0.35, 'Direct CHD8\nTargets',
         ha='center', fontsize=10, fontweight='bold', color='#2C3E50')
ax1.text( 0.55, 0.35, 'Rescued\nGenes (FL)',
         ha='center', fontsize=10, fontweight='bold', color='#27AE60')

# Annotate counts inside circles
overlap_n    = len(target_genes & rescued_genes)
only_target  = len(target_genes - rescued_genes)
only_rescued = len(rescued_genes - target_genes)

if v.get_label_by_id('10'):
    v.get_label_by_id('10').set_text(f'{only_target:,}')
if v.get_label_by_id('01'):
    v.get_label_by_id('01').set_text(f'{only_rescued:,}')
if v.get_label_by_id('11'):
    v.get_label_by_id('11').set_text(f'{overlap_n:,}')

# Bottom annotation
ax1.text(0.5, 0.02,
         f"Overlap: {overlap_n:,} genes\n"
         f"({overlap_n/len(target_genes)*100:.1f}% of targets rescued)",
         transform=ax1.transAxes,
         ha='center', va='bottom', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# --- PANEL B: Bar — Rescue rate targets vs indirect ---
ax2 = axes[1]
categories = ['Direct CHD8\nTargets', 'Indirect\nDEGs']
rates      = [rescue_rate_target, rescue_rate_indirect]
n_labels   = [len(target_genes), len(indirect_genes)]
bar_colors = ['#2C3E50', '#95A5A6']

bars = ax2.bar(categories, rates,
               color=bar_colors, edgecolor='black', width=0.5)
for bar, rate, n in zip(bars, rates, n_labels):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1,
             f"{rate:.1f}%\n(n={n:,})",
             ha='center', va='bottom',
             fontweight='bold', fontsize=10)

ax2.set_ylabel("Rescue Rate (%)", fontsize=12)
ax2.set_ylim(0, max(rates) * 1.3)
ax2.set_title("8B: Rescue Rate Comparison\n(Direct Targets vs Indirect DEGs)",
              fontweight='bold', pad=15)
ax2.axhline(50, color='red', linestyle='--',
            lw=1, alpha=0.5, label='50% threshold')
ax2.legend(fontsize=9)
sns.despine(ax=ax2)

# --- PANEL C: RF distribution — targets vs non-targets ---
ax3 = axes[2]
target_rf    = merged[merged['is_target']]['RF'].dropna()
nontarget_rf = merged[~merged['is_target']]['RF'].dropna()

import seaborn as sns
plot_data = pd.DataFrame({
    'RF'   : pd.concat([target_rf, nontarget_rf]),
    'Group': (['Direct Target'] * len(target_rf) +
              ['Non-target'] * len(nontarget_rf))
})

sns.violinplot(data=plot_data, x='Group', y='RF',
               hue='Group', ax=ax3,
               palette={'Direct Target': '#2C3E50',
                        'Non-target'   : '#95A5A6'},
               inner='quartile', linewidth=1.5,
               legend=False)

ax3.axhline(0, color='black', linestyle='--', lw=1, alpha=0.5)
ax3.axhline(0.5, color='green', linestyle='--', lw=1,
            alpha=0.5, label='RF = 0.5 threshold')
ax3.axhline(1, color='blue', linestyle='--', lw=1,
            alpha=0.5, label='Full rescue')

# MWU test
from scipy.stats import mannwhitneyu
_, pval_rf = mannwhitneyu(target_rf, nontarget_rf,
                           alternative='two-sided')

ax3.text(0.05, 0.97,
         f"Direct targets: n={len(target_rf):,}\n"
         f"  median RF = {target_rf.median():.3f}\n"
         f"Non-targets: n={len(nontarget_rf):,}\n"
         f"  median RF = {nontarget_rf.median():.3f}\n"
         f"MWU p = {pval_rf:.2e}",
         transform=ax3.transAxes, va='top', fontsize=8,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

ax3.set_title("8C: Rescue Fraction Distribution\n(Targets vs Non-targets)",
              fontweight='bold', pad=15)
ax3.set_ylabel("Rescue Fraction (RF)", fontsize=12)
ax3.set_ylim(-1, 2)
ax3.legend(fontsize=8, loc='lower right')
sns.despine(ax=ax3)

plt.suptitle(
    "Block 8: CHD8 Binding Predicts Rescue — Validation Analysis",
    fontsize=15, fontweight='bold', y=1.02
)
plt.tight_layout()

PLOT_PATH = os.path.join(SAVE_DIR, "Figure8_Rescue_Validation_600DPI.png")
plt.savefig(PLOT_PATH, dpi=600, bbox_inches='tight')
plt.show()

# ==========================================
# 📋 8. SUMMARY TABLE
# ==========================================
summary = pd.DataFrame({
    'Category'       : ['Direct CHD8 Targets',
                         'Indirect DEGs',
                         'Target & Rescued',
                         'Target & Not Rescued'],
    'N'              : [len(target_genes),
                        len(indirect_genes),
                        len(target_rescued),
                        len(target_nonrescued)],
    'Rescue_Rate_pct': [rescue_rate_target,
                        rescue_rate_indirect,
                        100.0, 0.0]
})
TABLE_PATH = os.path.join(SAVE_DIR, "Block8_rescue_summary.csv")
summary.to_csv(TABLE_PATH, index=False)

print(f"\n✅ Figure saved : {PLOT_PATH}")
print(f"✅ Table saved  : {TABLE_PATH}")
print(f"\n📝 - UPDATE:")
print(f"   Direct CHD8 targets rescue rate : {rescue_rate_target:.1f}%")
print(f"   Indirect DEGs rescue rate       : {rescue_rate_indirect:.1f}%")
print(f"   MWU p (RF distribution)         : {pval_rf:.2e}")
print("🎉 BLOCK 8 COMPLETE")

In [ ]:
import pandas as pd
import numpy as np
import pybedtools
import os

BASE_RNA  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
BASE_ATAC = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"  # same as Block 9
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"   # same as Block 9
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
RNA_KO_PATH = BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

ATAC_WT_PATHS = [
    BASE_ATAC + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed",
]
ATAC_KO_PATHS = [
    BASE_ATAC + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed",
]
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- Helpers ------------------------------------------------
def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

def load_atac_bed(path):
    df = pd.read_csv(path, sep='\t', header=None)
    score_col = 6 if len(df.columns) > 6 else 4
    out = df[[0,1,2,score_col]].copy()
    out.columns = ['chr','start','end','score']
    out['chr'] = out['chr'].astype(str)
    mask = ~out['chr'].str.startswith('chr')
    out.loc[mask,'chr'] = 'chr' + out.loc[mask,'chr']
    out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
    return out

def average_atac(paths, label):
    dfs = [load_atac_bed(p) for p in paths if os.path.exists(p)]
    combined = pd.concat(dfs, ignore_index=True)
    combined['bin'] = combined['chr'] + ':' + \
                      ((combined['start']//50)*50).astype(str)
    avg = combined.groupby('bin').agg(
        chr=('chr','first'), start=('start','min'),
        end=('end','max'), score=('score','mean')
    ).reset_index()
    print(f"  {label}: {len(avg):,} peaks averaged")
    return avg

# ---- Step 1: ATAC -------------------------------------------
print("Step 1: Loading ATAC peaks...")
wt_atac = average_atac(ATAC_WT_PATHS, "WT ATAC")
ko_atac = average_atac(ATAC_KO_PATHS, "KO ATAC")

wt_atac['bin'] = wt_atac['chr']+':'+ \
                  ((wt_atac['start']//50)*50).astype(str)
ko_atac['bin'] = ko_atac['chr']+':'+ \
                  ((ko_atac['start']//50)*50).astype(str)

merged_atac = pd.merge(
    wt_atac[['bin','chr','start','end','score']].rename(
        columns={'score':'wt_score'}),
    ko_atac[['bin','score']].rename(columns={'score':'ko_score'}),
    on='bin', how='inner'
)
merged_atac['lfc'] = np.log2(
    (merged_atac['ko_score']+0.5)/(merged_atac['wt_score']+0.5))
print(f"  Overlapping bins: {len(merged_atac):,}")

# ---- Step 2: RNA DEGs (corrected filter order) --------------
print("\nStep 2: Loading RNA DEGs...")
rna_ko = pd.read_csv(RNA_KO_PATH, sep='\t').dropna(
    subset=['log2FoldChange','padj'])
rna_ko = rna_ko[rna_ko['baseMean'] > 10]          # filter BEFORE padj/LFC
rna_degs = set(rna_ko[
    (rna_ko['padj'] < 0.05) &
    (rna_ko['log2FoldChange'].abs() > 0.5)
]['GENESYMBOL'].str.upper().str.strip())
print(f"  RNA DEGs: {len(rna_degs):,}  (expected 2,753)")

# ---- Step 3: NPC-specific ChIP — SAME as Block 9 -----------
print("\nStep 3: Loading NPC-specific ChIP targets (NPC minus ES)...")
tss_bt   = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()
npc_bt   = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
esc_bt   = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
npc_only = npc_bt.subtract(esc_bt, A=True)         # NPC-specific peaks only
hits     = tss_bt.intersect(npc_only, u=True, wa=True)
chip_genes = set(str(f[3]).upper().strip()
                 for f in hits if len(str(f[3]))>1)
print(f"  NPC-specific ChIP genes: {len(chip_genes):,}  (expected 3,754)")

# ---- Step 4: Sensitivity analysis ---------------------------
print("\nStep 4: Sensitivity analysis...")
print("="*65)

TEMP_ATAC_BED = "/content/atac_sensitivity.bed"
rows = []

for thresh_label, thresh_val in [('|LFC| > 0  [-]', 0.0),
                                   ('|LFC| > 0.5 [stricter]',  0.5)]:
    changed = merged_atac[merged_atac['lfc'].abs() > thresh_val].copy()
    gained  = merged_atac[merged_atac['lfc'] >  thresh_val]
    lost    = merged_atac[merged_atac['lfc'] < -thresh_val]

    print(f"\n  Threshold: {thresh_label}")
    print(f"    Peaks changed : {len(changed):,}")
    print(f"    Gained        : {len(gained):,}")
    print(f"    Lost          : {len(lost):,}")

    changed[['chr','start','end']].to_csv(
        TEMP_ATAC_BED, sep='\t', header=False, index=False)
    atac_bt   = pybedtools.BedTool(TEMP_ATAC_BED).each(fix_chr).sort()
    tss_bt2   = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()
    atac_hits = tss_bt2.intersect(atac_bt, u=True, wa=True)
    atac_genes_set = set(str(f[3]).upper().strip()
                         for f in atac_hits if len(str(f[3]))>1)

    print(f"    Genes near changed peaks: {len(atac_genes_set):,}")

    triple_ov = rna_degs & chip_genes & atac_genes_set
    rna_chip  = rna_degs & chip_genes
    rna_atac  = rna_degs & atac_genes_set
    chip_atac = chip_genes & atac_genes_set

    print(f"    RNA ∩ ChIP          : {len(rna_chip):,}")
    print(f"    RNA ∩ ATAC          : {len(rna_atac):,}")
    print(f"    ChIP ∩ ATAC         : {len(chip_atac):,}")
    print(f"    TRIPLE overlap      : {len(triple_ov):,}  <- key number")

    rows.append({
        'threshold'      : thresh_label,
        'peaks_changed'  : len(changed),
        'pct_changed'    : round(len(changed)/len(merged_atac)*100, 1),
        'gained'         : len(gained),
        'lost'           : len(lost),
        'atac_genes'     : len(atac_genes_set),
        'RNA_DEGs'       : len(rna_degs),
        'ChIP_genes'     : len(chip_genes),
        'RNA_ChIP'       : len(rna_chip),
        'RNA_ATAC'       : len(rna_atac),
        'ChIP_ATAC'      : len(chip_atac),
        'triple_overlap' : len(triple_ov),
        'triple_gene_list': '; '.join(sorted(triple_ov))
    })

# ---- Step 5: Save -------------------------------------------
print("\n" + "="*65)
results_df = pd.DataFrame(rows)
display_cols = ['threshold','peaks_changed','pct_changed',
                'gained','lost','atac_genes','triple_overlap']
print(results_df[display_cols].to_string(index=False))

OUT_CSV = os.path.join(SAVE_DIR, "ATAC_sensitivity_corrected_supplementary.csv")
results_df.drop(columns=['triple_gene_list']).to_csv(OUT_CSV, index=False)
print(f"\nSaved: {OUT_CSV}")

for _, row in results_df.iterrows():
    label_clean = row['threshold'].replace(' ','_').replace('|','').replace('>','gt').replace('.','p')
    gene_file = os.path.join(SAVE_DIR, f"triple_overlap_NPC_specific_{label_clean}.txt")
    with open(gene_file, 'w') as f:
        f.write('\n'.join(row['triple_gene_list'].split('; ')))
    print(f"Gene list saved: {gene_file}")

print("\n- NUMBERS (NPC-specific ChIP, consistent with Methods):")
print(f"  Triple overlap |LFC|>0   : {rows[0]['triple_overlap']}  <- main - figure")
print(f"  Triple overlap |LFC|>0.5 : {rows[1]['triple_overlap']}  <- stricter sensitivity check")
print(f"  Stricter set is subset of - set: "
      f"{set(rows[1]['triple_gene_list'].split('; ')) <= set(rows[0]['triple_gene_list'].split('; '))}")

In [ ]:
import pandas as pd

RNA_KO_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

rna = pd.read_csv(RNA_KO_PATH, sep='\t').dropna(subset=['log2FoldChange','padj'])
rna = rna[rna['baseMean'] > 10]

# Apply DEG filter
degs_df = rna[(rna['padj'] < 0.05) & (rna['log2FoldChange'].abs() > 0.5)].copy()
degs_df['gene_upper'] = degs_df['GENESYMBOL'].str.upper().str.strip()

# Row count
n_rows = len(degs_df)

# Unique gene symbols (what .unique() collapses to)
n_unique_symbols = degs_df['gene_upper'].nunique()

# Find the duplicate(s)
dup_symbols = degs_df[degs_df.duplicated('gene_upper', keep=False)] \
              .sort_values('gene_upper')[['gene_upper','GENESYMBOL','log2FoldChange','padj','baseMean']]

print(f"DEG rows (- number) : {n_rows:,}")
print(f"Unique gene symbols          : {n_unique_symbols:,}")
print(f"Difference                   : {n_rows - n_unique_symbols}")
print(f"\nDuplicated gene symbol(s):")
print(dup_symbols.to_string(index=False))
print(f"\nConclusion: - uses {n_rows:,} — the row count from DESeq2 output")
print(f"The 1-gene difference is a gene symbol shared by {n_rows - n_unique_symbols + 1} Ensembl IDs")

In [ ]:
from google.colab import files

# This will prompt to select the file from local computer
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
import pandas as pd
import os

# ==========================================
# 📁 1. PATH CONFIGURATION
# ==========================================
# 1. RNA-seq path (on Google Drive)
MY_RNA_PATH = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"

# 2. The Excel file just uploaded
MS_EXCEL_PATH = "42003_2021_1945_MOESM7_ESM.xlsx"

# ==========================================
# 📥 2. LOAD & CLEAN DATA
# ==========================================
print("📂 Loading datasets...")

if os.path.exists(MS_EXCEL_PATH):
    # Load Mass-Spec (Supplementary Data 4 / MOESM7)

    ms_df = pd.read_excel(MS_EXCEL_PATH, sheet_name=0, skiprows=2)

    # Clean Gene Symbols: handle 'P0593_CHD8' or 'GM10709|RPL29'
    def clean_symbol(sym):
        sym = str(sym).upper()
        if 'P0593_' in sym: sym = sym.replace('P0593_', '')
        if '|' in sym: sym = sym.split('|')[-1]
        return sym

    -_interactors = set(ms_df['gene_name'].apply(clean_symbol).unique())
    print(f"✅ Loaded {len(-_interactors)} CHD8 interactors from Cerase 2021.")

    # Load RNA-seq (Requires Drive to be mounted)
    if os.path.exists(MY_RNA_PATH):
        my_df = pd.read_csv(MY_RNA_PATH, sep='\t').dropna(subset=['padj', 'log2FoldChange'])
        # Downregulated genes (Padj < 0.05, Log2FC < -0.5)
        my_down = set(my_df[(my_df['padj'] < 0.05) & (my_df['log2FoldChange'] < -0.5)]['GENESYMBOL'].str.upper())

        # --- THE INTERSECTION ---
        common_targets = my_down.intersection(-_interactors)

        print("="*60)
        print("🏁 CHD8 PROTEIN-RNA INTEGRATION REPORT")
        print("="*60)
        print(f"Downregulated Genes (NPCs) : {len(my_down)}")
        print(f"-'s Protein Interactors     : {len(-_interactors)}")
        print(f"🔗 VERIFIED FUNCTIONAL TARGETS  : {len(common_targets)}")
        print("-" * 60)

        # Look for the core transcriptional "Engines"
        machinery = {'WDR5', 'USP7', 'ASH2L', 'PAF1', 'CTR9'}
        hits = common_targets.intersection(machinery)
        if hits:
            print(f"🔥 MECHANISM VALIDATION: {hits}")
            print("These are physical CHD8 interactors that also decreased at the mRNA level.")

        # Save for Figure 6
        output_file = "CHD8_Functional_Intersection_Report.csv"
        pd.DataFrame(list(common_targets), columns=['GeneSymbol']).to_csv(output_file, index=False)
        print(f"✅ Success! List saved to {output_file}")
    else:
        print(f"❌ RNA-seq file not found at {MY_RNA_PATH}. Please ensure Google Drive is mounted.")
else:
    print(f"❌ Excel file {MS_EXCEL_PATH} not found. Please upload it to the left sidebar first.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. ROBUST DATA LOADING
files = os.listdir('.')
# Try to find the CSV first, otherwise use the original Excel file
ms_file = next((f for f in files if 'Chd8 MS' in f), None)

if ms_file is None:
    # If the specific CSV isn't found, load the original Excel file
    ms_file = '42003_2021_1945_MOESM7_ESM.xlsx'
    df = pd.read_excel(ms_file, skiprows=2)
else:
    df = pd.read_csv(ms_file, skiprows=2)

# 2. CLEANING SYMBOLS (Handles complex IDs like 'P0593_CHD8' or 'GM|RPL29')
def clean_symbol(s):
    if pd.isna(s): return s
    s = str(s).upper().replace('P0593_', '')
    return s.split('|')[-1]

df['SYMBOL'] = df['gene_name'].apply(clean_symbol)

# 3. MAPPING TARGETS
name_map = {
    'CHD8': 'Chromodomain helicase DNA binding protein 8',
    'WDR5': 'WD repeat-containing protein 5',
    'RPL18A': '60S ribosomal protein L18a',
    'USP7': 'Ubiquitin carboxyl-terminal hydroxylase 7',
    'RPS25': '40S ribosomal protein S25',
    'KPNA3': 'Importin subunit alpha-3',
    'HNRNPR': 'Heterogeneous nuclear ribonucleoprotein R',
    'RPL13A': '60S ribosomal protein L13a',
    'RPL29': '60S ribosomal protein L29',
    'SYNCRIP': 'Heterogeneous nuclear ribonucleoprotein Q'
}

# 4. FILTER AND SORT
plot_df = df[df['SYMBOL'].isin(name_map.keys())].copy()
plot_df['Full_Protein_Name'] = plot_df['SYMBOL'].map(name_map)
plot_df = plot_df.sort_values('pearson.total', ascending=False)

# 5. PLOTTING
plt.figure(figsize=(14, 8))
ax = sns.barplot(data=plot_df, x='SYMBOL', y='pearson.total', palette='viridis')

# Annotate bars with values
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.2f'),
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points', weight='bold')

plt.title('Top 10 CHD8 Interacting Proteins (Mass Spec Evidence)', fontsize=15, pad=20)
plt.ylabel('CHD8 Interactor Pearson Score (Total)')
plt.ylim(0, 1.2)

# Add Side-Box for Full Descriptions
side_text = "Full Protein Descriptions:\n" + "-"*30 + "\n"
for idx, row in plot_df.iterrows():
    side_text += f"{row['SYMBOL']}: {row['Full_Protein_Name']}\n"
plt.gcf().text(0.92, 0.5, side_text, fontsize=10, verticalalignment='center',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.subplots_adjust(right=0.7)

plt.savefig("CHD8_Interactors_Final_Fix.png", dpi=600, bbox_inches='tight')
print(f"\n✅ Figure saved : {PLOT_PATH}")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import pybedtools
import seaborn as sns
from scipy.stats import fisher_exact, mannwhitneyu
import os

# --------------------------------------------------
# PATHS
# --------------------------------------------------
BASE_RNA_R5   = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
PEAK_PATH_R5  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH_R5   = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH_R5   = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
RNA_KO_R5     = BASE_RNA_R5 + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
RNA_FL_R5     = BASE_RNA_R5 + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv"
SAVE_DIR_R5   = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR_R5, exist_ok=True)

# --------------------------------------------------
# HELPERS
# --------------------------------------------------
def fix_chr_r5(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

def load_rna_r5(path, label):
    df = pd.read_csv(path, sep='\t').dropna(
        subset=['log2FoldChange', 'padj'])
    df['padj'] = df['padj'].replace(0, 1e-300)
    if 'baseMean' in df.columns:
        df = df[df['baseMean'] > 10]
    df['gene_upper'] = (df['GENESYMBOL'].astype(str)
                        .str.upper().str.strip())
    print(f"  {label}: {len(df):,} genes after filtering")
    return df

# --------------------------------------------------
# 1. LOAD RNA-seq
# --------------------------------------------------
print("=" * 60)
print("REVIEWER POINT #5 — RESCUE VALIDATION (computed from data)")
print("=" * 60)

print("\n1. Loading RNA-seq...")
ko_df_r5 = load_rna_r5(RNA_KO_R5, 'KO')
fl_df_r5 = load_rna_r5(RNA_FL_R5, 'FL rescue')

# --------------------------------------------------
# 2. DIRECT CHD8 TARGETS from ChIP-seq
# --------------------------------------------------
print("\n2. Computing direct CHD8 targets from ChIP-seq...")
npc_bt_r5  = pybedtools.BedTool(PEAK_PATH_R5).each(fix_chr_r5).sort()
esc_bt_r5  = pybedtools.BedTool(ESC_PATH_R5).each(fix_chr_r5).sort()
tss_bt_r5  = pybedtools.BedTool(TSS_PATH_R5).each(fix_chr_r5).sort()
npc_only_r5 = npc_bt_r5.subtract(esc_bt_r5, A=True)
chip_hits_r5 = tss_bt_r5.intersect(npc_only_r5, u=True, wa=True)
direct_targets_r5 = set([
    str(f[3]).upper().strip()
    for f in chip_hits_r5 if len(str(f[3])) > 1
])
print(f"  Direct CHD8 targets (ChIP + TSS ±2kb): {len(direct_targets_r5):,}")

# --------------------------------------------------
# 3. COMPUTE RESCUE FRACTION per gene
# --------------------------------------------------
print("\n3. Computing rescue fractions...")
merged_r5 = ko_df_r5[['gene_upper', 'log2FoldChange']].rename(
    columns={'log2FoldChange': 'lfc_ko'}).merge(
    fl_df_r5[['gene_upper', 'log2FoldChange']].rename(
        columns={'log2FoldChange': 'lfc_fl'}),
    on='gene_upper'
)

# Rescue fraction: (lfc_FL - lfc_KO) / (0 - lfc_KO)
#   RF = 1.0 → fully rescued back to WT
#   RF = 0.0 → not rescued at all
denom_r5 = (0 - merged_r5['lfc_ko']).replace(0, np.nan)
merged_r5['RF'] = ((merged_r5['lfc_fl'] - merged_r5['lfc_ko'])
                   / denom_r5).clip(-1, 2)

# Label each gene
merged_r5['is_target']  = merged_r5['gene_upper'].isin(direct_targets_r5)
merged_r5['is_dysreg']  = merged_r5['lfc_ko'].abs() > 0.5
merged_r5['is_rescued'] = merged_r5['RF'] >= 0.5

# Gene lists (as sets)
target_set_r5     = set(merged_r5[merged_r5['is_target']]['gene_upper'])
rescued_set_r5    = set(merged_r5[merged_r5['is_rescued']]['gene_upper'])
nonrescued_set_r5 = set(merged_r5[~merged_r5['is_rescued']]['gene_upper'])

print(f"  Genes in merged dataset     : {len(merged_r5):,}")
print(f"  Direct CHD8 targets in data : {len(target_set_r5):,}")
print(f"  Rescued (RF >= 0.5)         : {len(rescued_set_r5):,}")
print(f"  Non-rescued (RF < 0.5)      : {len(nonrescued_set_r5):,}")

# --------------------------------------------------
# 4. RESCUE RATES + FISHER'S EXACT TEST
# --------------------------------------------------
print("\n4. Statistical test: do direct targets rescue better?")

# Contingency table:
#              Rescued   Not Rescued
#  Direct         a          b
#  Indirect       c          d
indirect_set_r5 = set(merged_r5['gene_upper']) - target_set_r5

a = len(target_set_r5   & rescued_set_r5)
b = len(target_set_r5   & nonrescued_set_r5)
c = len(indirect_set_r5 & rescued_set_r5)
d = len(indirect_set_r5 & nonrescued_set_r5)

odds_ratio, pval_fisher = fisher_exact([[a, b], [c, d]],
                                        alternative='greater')

rescue_rate_target_r5   = a / (a + b) * 100 if (a + b) > 0 else 0
rescue_rate_indirect_r5 = c / (c + d) * 100 if (c + d) > 0 else 0

# MWU on rescue fraction distributions
target_rf_r5   = merged_r5[merged_r5['is_target']]['RF'].dropna()
indirect_rf_r5 = merged_r5[~merged_r5['is_target']]['RF'].dropna()
_, pval_mwu = mannwhitneyu(target_rf_r5, indirect_rf_r5,
                            alternative='greater')

print("-" * 60)
print(f"  Direct targets:   {a:,} rescued / {a+b:,} total = "
      f"{rescue_rate_target_r5:.1f}%")
print(f"  Indirect DEGs:    {c:,} rescued / {c+d:,} total = "
      f"{rescue_rate_indirect_r5:.1f}%")
print(f"  Fisher OR = {odds_ratio:.2f}, p = {pval_fisher:.2e}")
print(f"  MWU (RF distribution): p = {pval_mwu:.2e}")
print("-" * 60)
if pval_fisher < 0.05:
    print("  RESULT: Direct CHD8 targets are significantly more likely"
          " to be rescued (supports Reviewer #5 claim)")
else:
    print("  RESULT: No significant enrichment — review rescue threshold "
          "or check FL rescue file path")

# --------------------------------------------------
# 5. FIGURES — 3 PANELS (600 DPI)
# --------------------------------------------------
print("\n5. Generating figures...")

plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Arial', 'DejaVu Sans'],
                     'pdf.fonttype': 42})

fig, axes = plt.subplots(1, 3, figsize=(21, 7), dpi=600)

# ---- PANEL A: Venn — Direct Targets vs Rescued Genes ----
ax1 = axes[0]
v = venn2(
    subsets=[target_set_r5, rescued_set_r5],
    set_labels=('', ''),
    set_colors=('#2C3E50', '#27AE60'),
    alpha=0.7,
    ax=ax1
)
# Manual labels
ax1.text(-0.55, 0.38, 'Direct CHD8\nTargets',
         ha='center', fontsize=10, fontweight='bold', color='#2C3E50')
ax1.text( 0.55, 0.38, 'Rescued\nGenes (FL)',
         ha='center', fontsize=10, fontweight='bold', color='#27AE60')

for rid, val in [('10', len(target_set_r5 - rescued_set_r5)),
                  ('01', len(rescued_set_r5 - target_set_r5)),
                  ('11', len(target_set_r5 & rescued_set_r5))]:
    lbl = v.get_label_by_id(rid)
    if lbl:
        lbl.set_text(f'{val:,}')
        lbl.set_fontsize(11)
        lbl.set_fontweight('bold')

ax1.text(0.5, 0.02,
         f"Overlap: {len(target_set_r5 & rescued_set_r5):,} genes "
         f"({rescue_rate_target_r5:.1f}% of targets rescued)",
         transform=ax1.transAxes, ha='center', va='bottom', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax1.set_title("Panel A: Direct CHD8 Targets\nvs Rescued Genes (FL)",
              fontweight='bold', pad=15)

# ---- PANEL B: Bar — rescue rate comparison ----
ax2 = axes[1]
categories_r5 = ['Direct CHD8\nTargets', 'Indirect\nDEGs']
rates_r5      = [rescue_rate_target_r5, rescue_rate_indirect_r5]
n_labels_r5   = [len(target_set_r5), len(indirect_set_r5)]
bar_colors_r5 = ['#2C3E50', '#95A5A6']

bars_r5 = ax2.bar(categories_r5, rates_r5,
                  color=bar_colors_r5, edgecolor='black', width=0.5)
for bar, rate, n in zip(bars_r5, rates_r5, n_labels_r5):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 1,
             f"{rate:.1f}%\n(n={n:,})",
             ha='center', va='bottom', fontweight='bold', fontsize=10)

ax2.set_ylabel("Rescue Rate (%)", fontsize=12)
ax2.set_ylim(0, max(rates_r5) * 1.35)
ax2.set_title(
    f"Panel B: Rescue Rate Comparison\n"
    f"(Fisher OR={odds_ratio:.2f}, p={pval_fisher:.2e})",
    fontweight='bold', pad=15
)
ax2.axhline(50, color='red', linestyle='--',
            lw=1, alpha=0.5, label='50% threshold')
ax2.legend(fontsize=9)
sns.despine(ax=ax2)

# ---- PANEL C: Violin — RF distribution (targets vs non-targets) ----
ax3 = axes[2]
plot_data_r5 = pd.DataFrame({
    'RF': pd.concat([target_rf_r5, indirect_rf_r5]),
    'Group': (['Direct Target'] * len(target_rf_r5) +
              ['Non-target']    * len(indirect_rf_r5))
})
sns.violinplot(
    data=plot_data_r5, x='Group', y='RF',
    hue='Group', ax=ax3,
    palette={'Direct Target': '#2C3E50', 'Non-target': '#95A5A6'},
    inner='quartile', linewidth=1.5, legend=False
)
ax3.axhline(0,   color='black', linestyle='--', lw=1, alpha=0.5)
ax3.axhline(0.5, color='green', linestyle='--', lw=1,
            alpha=0.5, label='RF = 0.5 (rescued)')
ax3.axhline(1.0, color='blue',  linestyle='--', lw=1,
            alpha=0.5, label='RF = 1.0 (full rescue)')
ax3.text(0.05, 0.97,
         f"Direct targets: n={len(target_rf_r5):,}\n"
         f"  median RF = {target_rf_r5.median():.3f}\n"
         f"Non-targets: n={len(indirect_rf_r5):,}\n"
         f"  median RF = {indirect_rf_r5.median():.3f}\n"
         f"MWU p = {pval_mwu:.2e}",
         transform=ax3.transAxes, va='top', fontsize=8,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
ax3.set_title("Panel C: Rescue Fraction Distribution\n"
              "(Direct Targets vs Non-targets)",
              fontweight='bold', pad=15)
ax3.set_ylabel("Rescue Fraction (RF)", fontsize=12)
ax3.set_ylim(-1, 2)
ax3.legend(fontsize=8, loc='lower right')
sns.despine(ax=ax3)

plt.suptitle(
    "Reviewer #5 — CHD8 Binding Predicts Rescue: Validation Analysis",
    fontsize=15, fontweight='bold', y=1.02
)
plt.tight_layout()

PLOT_R5 = os.path.join(SAVE_DIR_R5, "ReviewerPt5_Rescue_Validation_600DPI.png")
PDF_R5  = os.path.join(SAVE_DIR_R5, "ReviewerPt5_Rescue_Validation_vector.pdf")
plt.savefig(PLOT_R5, dpi=600, bbox_inches='tight')
plt.savefig(PDF_R5,  bbox_inches='tight')
plt.show()

# --------------------------------------------------
# 6. SUMMARY TABLE (for rebuttal letter)
# --------------------------------------------------
summary_r5 = pd.DataFrame({
    'Category'       : ['Direct CHD8 Targets', 'Indirect DEGs',
                        'Target & Rescued', 'Target & Not Rescued'],
    'N'              : [len(target_set_r5), len(indirect_set_r5), a, b],
    'Rescue_Rate_pct': [rescue_rate_target_r5, rescue_rate_indirect_r5,
                        100.0, 0.0]
})
TABLE_R5 = os.path.join(SAVE_DIR_R5, "ReviewerPt5_rescue_summary.csv")
summary_r5.to_csv(TABLE_R5, index=False)

print(f"\nFigure saved : {PLOT_R5}")
print(f"Table saved  : {TABLE_R5}")
print("\n- UPDATE for Reviewer #5 response:")
print(f"  Direct CHD8 target rescue rate : {rescue_rate_target_r5:.1f}%")
print(f"  Indirect DEG rescue rate       : {rescue_rate_indirect_r5:.1f}%")
print(f"  Fisher's exact OR = {odds_ratio:.2f}, p = {pval_fisher:.2e}")
print(f"  MWU p (RF distribution)        : {pval_mwu:.2e}")


In [ ]:
import pandas as pd
import numpy as np
import os
import gseapy as gp
import pybedtools
from scipy.stats import friedmanchisquare, wilcoxon, chisquare, mannwhitneyu, fisher_exact
from statsmodels.stats.multitest import multipletests

# ── OUTPUT DIRECTORY ─────────────────────────────────────────────
SUPP_DIR = "/content/drive/MyDrive/CHD8_Supplementary_Tables/"
os.makedirs(SUPP_DIR, exist_ok=True)

# ── PATHS (same as the rest of notebook) ────────────────────
BASE_RNA  = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
BASE_ATAC = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"

RNA_FILES = {
    "KO" : BASE_RNA + "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    "KD1": BASE_RNA + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv",
    "KD2": BASE_RNA + "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv",
    "siRNA": BASE_RNA + "Diff_Scrambled_siRNA_Diff_Chd8_siRNA.tsv",
    "FL" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    "dC" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    "dH" : BASE_RNA + "Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}
ATAC_WT_PATHS = [
    BASE_ATAC + "ATAC_Diff_1/ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "ATAC_Diff_2/ATAC_Diff_2.merged_score_noIgG.bed",
]
ATAC_KO_PATHS = [
    BASE_ATAC + "KO_1_ATAC_Diff_1/KO_1_ATAC_Diff_1.merged_score_noIgG.bed",
    BASE_ATAC + "KO_2_ATAC_Diff_1/KO_2_ATAC_Diff_1.merged_score_noIgG.bed",
]

# ── HELPERS ──────────────────────────────────────────────────────
def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

def load_rna(key):
    path = RNA_FILES[key]
    if not os.path.exists(path):
        print(f"  WARNING: {key} RNA file not found — {path}")
        return None
    df = pd.read_csv(path, sep='\t')
    df['condition'] = key
    return df

def load_atac_bed(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, sep='\t', header=None)
    sc = 6 if len(df.columns) > 6 else 4
    out = df[[0, 1, 2, sc]].copy()
    out.columns = ['chr', 'start', 'end', 'score']
    out['chr'] = out['chr'].astype(str)
    mask = ~out['chr'].str.startswith('chr')
    out.loc[mask, 'chr'] = 'chr' + out.loc[mask, 'chr']
    out['score'] = pd.to_numeric(out['score'], errors='coerce').fillna(0)
    return out

def avg_atac(paths, label):
    dfs = [d for d in [load_atac_bed(p) for p in paths] if d is not None]
    if not dfs: return None
    combined = pd.concat(dfs, ignore_index=True)
    combined['bin'] = combined['chr'] + ':' + ((combined['start']//50)*50).astype(str)
    avg = combined.groupby('bin').agg(
        chr=('chr','first'), start=('start','min'),
        end=('end','max'), score=('score','mean')
    ).reset_index()
    print(f"  {label}: {len(avg):,} peaks")
    return avg

def save_table(df, name, description):
    path = os.path.join(SUPP_DIR, f"{name}.csv")
    df.to_csv(path, index=False)
    print(f"  Saved: {name}.csv  ({len(df):,} rows)  — {description}")
    return df

all_tables = {}

print("=" * 65)
print("CHD8 SUPPLEMENTARY TABLES — GENERATING ALL")
print("=" * 65)

# ── T1–T4: DEG results ───────────────────────────────────────────
print("\nT1–T4: DEG results (full DESeq2 output with categories)...")
deg_labels = {
    "KO":    "Table_S1_DEG_KO_vs_WT",
    "KD1":   "Table_S2_DEG_KD8.1_vs_Scramble",
    "KD2":   "Table_S3_DEG_KD8.2_vs_Scramble",
    "siRNA": "Table_S4_DEG_siRNA_vs_Scramble",
}
for key, fname in deg_labels.items():
    df = load_rna(key)
    if df is not None:
        # Add significance category column
        df['significance'] = 'NS'
        if 'padj' in df.columns and 'log2FoldChange' in df.columns:
            df.loc[(df['padj'] < 0.05) & (df['log2FoldChange'] >  0.5), 'significance'] = 'Up'
            df.loc[(df['padj'] < 0.05) & (df['log2FoldChange'] < -0.5), 'significance'] = 'Down'
        df = df.sort_values('padj')
        all_tables[fname] = save_table(df, fname,
            f"DESeq2 full results: {key} — {(df['significance']=='Up').sum()} up, "
            f"{(df['significance']=='Down').sum()} down")

# ── T5: GO enrichment — KO downregulated (Fig 5A) ────────────────
print("\nT5: GO enrichment — KO downregulated genes...")
ko_df = load_rna("KO")
if ko_df is not None:
    ko_df['padj'] = ko_df['padj'].fillna(1)
    ko_down = ko_df[
        (ko_df['padj'] < 0.05) & (ko_df['log2FoldChange'] < -0.5)
    ]['GENESYMBOL'].dropna().unique().tolist()
    print(f"  Input: {len(ko_down):,} downregulated genes")
    try:
        enr_ko = gp.enrichr(gene_list=ko_down,
                             gene_sets=['GO_Biological_Process_2023'],
                             organism='mouse')
        t5 = enr_ko.results.copy()
        t5['input_n_genes'] = len(ko_down)
        all_tables['Table_S5_GO_KO_Downregulated'] = save_table(
            t5, 'Table_S5_GO_KO_Downregulated',
            f"GO enrichment, KO down genes, n={len(ko_down)}: "
            f"{(t5['Adjusted P-value']<0.05).sum()} sig terms")
    except Exception as e:
        print(f"  WARNING: GO enrichment failed — {e}")

# ── T6+T7: GO + TF enrichment — NPC-specific targets (Fig 2) ─────
print("\nT6–T7: GO + TF enrichment — NPC-specific ChIP targets...")
try:
    npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
    esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
    tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()
    npc_specific = npc_bt.subtract(esc_bt, A=True)
    hits = tss_bt.intersect(npc_specific, u=True, wa=True)
    npc_genes = list(set([str(f[3]) for f in hits
                          if str(f[3]).lower() not in ('nan','.','') and len(str(f[3]))>1]))
    print(f"  NPC-specific target genes: {len(npc_genes):,}")

    # Intersect with KO DEGs for focused list (n=257)
    if ko_df is not None:
        ko_df['gene_upper'] = ko_df['GENESYMBOL'].astype(str).str.upper()
        npc_upper_map = {g.upper(): g for g in npc_genes}
        ko_df['is_npc'] = ko_df['gene_upper'].isin(npc_upper_map)
        ko_df['is_deg'] = (ko_df['padj'] < 0.05) & (ko_df['log2FoldChange'].abs() > 0.5)
        focused = [npc_upper_map[g] for g in ko_df.loc[ko_df['is_npc'] & ko_df['is_deg'], 'gene_upper']
                   if g in npc_upper_map]
        enrich_genes = focused if len(focused) >= 30 else npc_genes
        print(f"  Enrichment input: {len(enrich_genes):,} genes")
    else:
        enrich_genes = npc_genes

    enr_go2 = gp.enrichr(gene_list=enrich_genes,
                          gene_sets=['GO_Biological_Process_2023'],
                          organism='mouse')
    t6 = enr_go2.results.copy()
    t6['input_n_genes'] = len(enrich_genes)
    all_tables['Table_S6_GO_NPC_Targets'] = save_table(
        t6, 'Table_S6_GO_NPC_Targets',
        f"GO enrichment, NPC-specific targets, n={len(enrich_genes)}: "
        f"{(t6['Adjusted P-value']<0.05).sum()} sig terms")

    enr_tf2 = gp.enrichr(gene_list=enrich_genes,
                          gene_sets=['ChEA_2022'],
                          organism='mouse')
    t7 = enr_tf2.results.copy()
    t7['input_n_genes'] = len(enrich_genes)
    all_tables['Table_S7_TF_NPC_Targets'] = save_table(
        t7, 'Table_S7_TF_NPC_Targets',
        f"TF co-occupancy (ChEA), NPC targets, n={len(enrich_genes)}: "
        f"{(t7['Adjusted P-value']<0.05).sum()} sig TFs")

except Exception as e:
    print(f"  WARNING: ChIP target enrichment failed — {e}")
    npc_genes = []

# ── T8: GSEA full results (Fig 5B) ────────────────────────────────
print("\nT8: GSEA full results...")
si_df = load_rna("siRNA")
if si_df is not None:
    try:
        si_df['padj'] = si_df['padj'].fillna(1).replace(0, 1e-300)
        si_df['rank_metric'] = (
            -np.log10(si_df['padj'] + 1e-300) * np.sign(si_df['log2FoldChange'])
        )
        gene_col = 'GENESYMBOL' if 'GENESYMBOL' in si_df.columns else 'symbol'
        rnk = si_df.sort_values('rank_metric', ascending=False).set_index(gene_col)['rank_metric']
        pre_res = gp.prerank(rnk=rnk,
                              gene_sets='GO_Biological_Process_2023',
                              organism='Mouse',
                              permutation_num=1000,
                              seed=42)
        t8 = pre_res.res2d.copy()
        t8 = t8.sort_values('FDR q-val')
        all_tables['Table_S8_GSEA_Full'] = save_table(
            t8, 'Table_S8_GSEA_Full',
            f"GSEA pre-ranked full results: {len(t8):,} gene sets, "
            f"{(t8['FDR q-val']<0.05).sum()} FDR<0.05")
    except Exception as e:
        print(f"  WARNING: GSEA failed — {e}")

# ── T9: Rescue fractions per gene ─────────────────────────────────
print("\nT9: Rescue fractions per gene (FL / ΔChromo / ΔHelicase)...")
try:
    dfs_rescue = []
    for label in ['KO', 'FL', 'dC', 'dH']:
        df = load_rna(label)
        if df is not None:
            df['gene_id'] = df.iloc[:, 0].astype(str).str.split('.').str[0]
            dfs_rescue.append(
                df[['gene_id','GENESYMBOL','log2FoldChange']].rename(
                    columns={'log2FoldChange': f'LFC_{label}'}))

    master = dfs_rescue[0]
    for d in dfs_rescue[1:]:
        master = master.merge(d.drop(columns=['GENESYMBOL'], errors='ignore'),
                              on='gene_id', how='inner')

    if npc_genes:
        target_set = set([g.upper() for g in npc_genes])
        master['gene_upper'] = master['GENESYMBOL'].astype(str).str.upper()
        targets = master[
            master['gene_upper'].isin(target_set) & (master['LFC_KO'].abs() > 0.5)
        ].copy()
    else:
        targets = master[master['LFC_KO'].abs() > 0.5].copy()

    for cond in ['FL', 'dC', 'dH']:
        col = f'LFC_{cond}'
        if col in targets.columns:
            denom = (0 - targets['LFC_KO']).replace(0, np.nan)
            targets[f'RF_{cond}'] = ((targets[col] - targets['LFC_KO']) / denom).clip(-1, 2)

    # Add rescue category (RF >= 0.5 threshold)
    for cond in ['FL', 'dC', 'dH']:
        if f'RF_{cond}' in targets.columns:
            targets[f'rescued_{cond}'] = targets[f'RF_{cond}'] >= 0.5

    all_tables['Table_S9_Rescue_Fractions'] = save_table(
        targets.drop(columns=['gene_upper'], errors='ignore'),
        'Table_S9_Rescue_Fractions',
        f"Rescue fractions per gene: {len(targets):,} NPC targets with |LFC_KO|>0.5")

except Exception as e:
    print(f"  WARNING: Rescue fractions failed — {e}")
    targets = pd.DataFrame()

# ── T10: Rescue category counts + chi-square ─────────────────────
print("\nT10: Rescue category counts + chi-square...")
if len(targets) > 0 and all(f'rescued_{c}' in targets.columns for c in ['FL','dC','dH']):
    try:
        c1 = ( targets['rescued_FL'] &  targets['rescued_dC'] &  targets['rescued_dH']).sum()
        c2 = ( targets['rescued_FL'] &  targets['rescued_dC'] & ~targets['rescued_dH']).sum()
        c3 = ( targets['rescued_FL'] & ~targets['rescued_dC'] &  targets['rescued_dH']).sum()
        c4 = ( targets['rescued_FL'] & ~targets['rescued_dC'] & ~targets['rescued_dH']).sum()
        c5 = (~targets['rescued_FL']).sum()
        total = len(targets)
        chi2_stat, p_cat = chisquare([c1, c2, c3, c4, c5])

        t10 = pd.DataFrame({
            'Category': ['Global rescue (all constructs)',
                         'Helicase required (FL + ΔChromo only)',
                         'Chromo required (FL + ΔHelicase only)',
                         'Both domains required (FL only)',
                         'Low/no rescue (none)'],
            'N': [c1, c2, c3, c4, c5],
            'Pct_of_total': [round(x/total*100, 1) for x in [c1,c2,c3,c4,c5]],
            'RF_threshold': ['>= 0.5'] * 5,
            'Chi2_stat': [round(chi2_stat, 2)] + [''] * 4,
            'Chi2_pvalue': [f'{p_cat:.2e}'] + [''] * 4,
            'N_total_genes': [total] + [''] * 4,
        })
        all_tables['Table_S10_Rescue_Categories'] = save_table(
            t10, 'Table_S10_Rescue_Categories',
            f"Rescue categories: χ²={chi2_stat:.2f}, p={p_cat:.2e}")
    except Exception as e:
        print(f"  WARNING: Category table failed — {e}")

# ── T11: ATAC-seq LFC per peak ────────────────────────────────────
print("\nT11: ATAC-seq peak-level LFC table...")
try:
    wt_atac = avg_atac(ATAC_WT_PATHS, "WT")
    ko_atac = avg_atac(ATAC_KO_PATHS, "KO")

    if wt_atac is not None and ko_atac is not None:
        wt_atac['bin'] = wt_atac['chr'] + ':' + ((wt_atac['start']//50)*50).astype(str)
        ko_atac['bin'] = ko_atac['chr'] + ':' + ((ko_atac['start']//50)*50).astype(str)
        merged_atac = pd.merge(
            wt_atac[['bin','chr','start','end','score']].rename(columns={'score':'WT_score'}),
            ko_atac[['bin','score']].rename(columns={'score':'KO_score'}),
            on='bin', how='inner'
        )
        merged_atac['log2FC_KO_vs_WT'] = np.log2(
            (merged_atac['KO_score'] + 0.5) / (merged_atac['WT_score'] + 0.5))
        merged_atac['direction'] = np.where(merged_atac['log2FC_KO_vs_WT'] > 0,
                                             'Gained', 'Lost')
        merged_atac = merged_atac.sort_values('log2FC_KO_vs_WT')
        all_tables['Table_S11_ATAC_LFC'] = save_table(
            merged_atac.drop(columns=['bin']),
            'Table_S11_ATAC_LFC',
            f"ATAC-seq LFC per peak: {len(merged_atac):,} matched peaks, "
            f"{(merged_atac['direction']=='Gained').sum():,} gained, "
            f"{(merged_atac['direction']=='Lost').sum():,} lost")
except Exception as e:
    print(f"  WARNING: ATAC table failed — {e}")

# ── T12: Multi-omics integration counts ───────────────────────────
print("\nT12: Multi-omics integration summary (RNA ∩ ChIP ∩ ATAC)...")
try:
    # Reuse merged_atac from T11 if available
    if 'merged_atac' in dir() and npc_genes and ko_df is not None:
        ko_degs = set(ko_df[
            (ko_df['padj'] < 0.05) & (ko_df['log2FoldChange'].abs() > 0.5)
        ]['GENESYMBOL'].astype(str).str.upper())

        chip_genes = set([g.upper() for g in npc_genes])

        # ATAC genes — any change
        tmp_bed = "/content/atac_any_change.bed"
        merged_atac[['chr','start','end']].to_csv(tmp_bed, sep='\t',
                                                   header=False, index=False)
        atac_bt   = pybedtools.BedTool(tmp_bed).each(fix_chr).sort()
        atac_hits = tss_bt.intersect(atac_bt, u=True, wa=True)
        atac_genes = set([str(f[3]).upper().strip()
                          for f in atac_hits if len(str(f[3]))>1])

        A, B, C = ko_degs, chip_genes, atac_genes
        t12 = pd.DataFrame({
            'Category': ['RNA-seq DEGs (FDR<0.05, |LFC|>0.5)',
                          'CHD8 NPC-specific ChIP targets',
                          'ATAC-seq changed genes',
                          'RNA ∩ ChIP',
                          'RNA ∩ ATAC',
                          'ChIP ∩ ATAC',
                          'Triple overlap (RNA ∩ ChIP ∩ ATAC)'],
            'N': [len(A), len(B), len(C),
                  len((A&B)-C), len((A&C)-B), len((B&C)-A), len(A&B&C)]
        })
        all_tables['Table_S12_MultiOmics_Integration'] = save_table(
            t12, 'Table_S12_MultiOmics_Integration',
            f"Multi-omics integration counts; triple overlap = {len(A&B&C)}")
except Exception as e:
    print(f"  WARNING: Multi-omics table failed — {e}")

# ── T13: Rescue validation ─────────────────────────
print("\nT13: Rescue validation — ChIP targets vs non-targets...")
try:
    fl_df = load_rna("FL")
    if ko_df is not None and fl_df is not None and npc_genes:
        ko_m = ko_df[['gene_upper','log2FoldChange']].rename(
            columns={'log2FoldChange':'LFC_KO'})
        fl_df['gene_upper'] = fl_df['GENESYMBOL'].astype(str).str.upper()
        fl_m = fl_df[['gene_upper','log2FoldChange']].rename(
            columns={'log2FoldChange':'LFC_FL'})
        rv = ko_m.merge(fl_m, on='gene_upper')
        denom = (0 - rv['LFC_KO']).replace(0, np.nan)
        rv['RF'] = ((rv['LFC_FL'] - rv['LFC_KO']) / denom).clip(-1, 2)
        rv['is_direct_target'] = rv['gene_upper'].isin(set([g.upper() for g in npc_genes]))
        rv['is_rescued']       = rv['RF'] >= 0.5

        targets_r = rv[rv['is_direct_target']]
        indirect  = rv[~rv['is_direct_target']]
        a = (targets_r['is_rescued']).sum()
        b = (~targets_r['is_rescued']).sum()
        c = (indirect['is_rescued']).sum()
        d = (~indirect['is_rescued']).sum()
        or_val, p_fisher = fisher_exact([[a,b],[c,d]], alternative='greater')
        _, p_mwu = mannwhitneyu(
            targets_r['RF'].dropna(), indirect['RF'].dropna(),
            alternative='two-sided')

        t13_summary = pd.DataFrame({
            'Group': ['Direct CHD8 targets', 'Indirect DEGs'],
            'N_total':   [len(targets_r), len(indirect)],
            'N_rescued': [a, c],
            'Rescue_rate_pct': [round(a/(a+b)*100,1) if (a+b)>0 else 0,
                                round(c/(c+d)*100,1) if (c+d)>0 else 0],
            'Fisher_OR':    [round(or_val,3), ''],
            'Fisher_pvalue':[f'{p_fisher:.3e}', ''],
            'MWU_pvalue':   [f'{p_mwu:.3e}', ''],
            'RF_threshold': ['>= 0.5', '>= 0.5'],
        })
        # Also save per-gene RF table
        rv_save = rv.rename(columns={'gene_upper':'GENESYMBOL_upper'})
        all_tables['Table_S13a_Rescue_Validation_Summary'] = save_table(
            t13_summary, 'Table_S13a_Rescue_Validation_Summary',
            f"Rescue validation summary: Fisher OR={or_val:.2f}, p={p_fisher:.2e}")
        all_tables['Table_S13b_Rescue_Validation_PerGene'] = save_table(
            rv_save, 'Table_S13b_Rescue_Validation_PerGene',
            f"Rescue fractions per gene: {len(rv):,} genes")
except Exception as e:
    print(f"  WARNING: Rescue validation table failed — {e}")

# ── T14: Block 7 ATAC × domain rescue MWU stats ──────────────────
print("\nT14: ATAC × domain rescue stats (Block 7)...")
try:
    if 'merged_atac' in dir() and npc_genes:
        # KO-lost peaks
        ko_lost = merged_atac[merged_atac['log2FC_KO_vs_WT'] < -0.5].copy()
        ko_lost_bed = "/content/ko_lost_peaks_supp.bed"
        ko_lost[['chr','start','end']].to_csv(ko_lost_bed, sep='\t',
                                               header=False, index=False)
        lost_bt   = pybedtools.BedTool(ko_lost_bed).each(fix_chr).sort()
        lost_hits = tss_bt.intersect(lost_bt, u=True, wa=True)
        lost_genes = set([str(f[3]).upper().strip()
                          for f in lost_hits if len(str(f[3]))>1])

        stats_rows = []
        for cond in ['FL','dC','dH']:
            df_c = load_rna(cond)
            if df_c is None: continue
            df_c['gene_upper'] = df_c['GENESYMBOL'].astype(str).str.upper()
            m = ko_df[['gene_upper','log2FoldChange']].rename(
                columns={'log2FoldChange':'LFC_KO'}).merge(
                df_c[['gene_upper','log2FoldChange']].rename(
                    columns={'log2FoldChange':f'LFC_{cond}'}), on='gene_upper')
            denom = (0 - m['LFC_KO']).replace(0, np.nan)
            m['RF'] = ((m[f'LFC_{cond}'] - m['LFC_KO']) / denom).clip(-1, 2)
            m['near_lost'] = m['gene_upper'].isin(lost_genes)
            near  = m[m['near_lost']]['RF'].dropna()
            other = m[~m['near_lost']]['RF'].dropna()
            _, pval = mannwhitneyu(near, other, alternative='two-sided') if len(near)>5 else (np.nan, np.nan)
            stats_rows.append({
                'Construct': cond,
                'N_near_KO_lost_peak': len(near),
                'Median_RF_near_lost': round(near.median(), 3),
                'N_other': len(other),
                'Median_RF_other': round(other.median(), 3),
                'MWU_pvalue': f'{pval:.3e}' if not np.isnan(pval) else 'NA',
                'KO_lost_peaks_n': len(ko_lost),
                'Genes_near_lost_peaks_n': len(lost_genes),
            })

        t14 = pd.DataFrame(stats_rows)
        all_tables['Table_S14_ATAC_Domain_Rescue_Stats'] = save_table(
            t14, 'Table_S14_ATAC_Domain_Rescue_Stats',
            "ATAC × domain rescue MWU stats per construct")
except Exception as e:
    print(f"  WARNING: Block 7 stats table failed — {e}")

# ── COMBINED EXCEL WORKBOOK ───────────────────────────────────────
print("\nBuilding combined Excel workbook...")
try:
    excel_path = os.path.join(SUPP_DIR, "CHD8_All_Supplementary_Tables.xlsx")
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for sheet_name, df in all_tables.items():
            # Excel sheet names max 31 chars
            safe_name = sheet_name.replace('Table_', 'T').replace('_', ' ')[:31]
            df.to_excel(writer, sheet_name=safe_name, index=False)
    print(f"  Excel workbook saved: {excel_path}")
    print(f"  Sheets: {len(all_tables)}")
except Exception as e:
    print(f"  WARNING: Excel workbook failed — {e}")
    print("  (Individual CSVs still saved successfully)")

# ── ZIP AND DOWNLOAD ──────────────────────────────────────────────
print("\nCreating zip file for download...")
import zipfile
zip_path = "/content/CHD8_Supplementary_Tables.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(SUPP_DIR):
        fpath = os.path.join(SUPP_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, fname)
print(f"  Zip created: {zip_path}")

# Trigger browser download
from google.colab import files
files.download(zip_path)

# ── FINAL SUMMARY ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("COMPLETE — TABLES GENERATED:")
print("=" * 65)
for name in all_tables:
    n = len(all_tables[name])
    print(f"  {name} ({n:,} rows)")
print(f"\nAll CSVs saved to: {SUPP_DIR}")
print("Combined Excel: CHD8_All_Supplementary_Tables.xlsx")
print("Zip downloaded to your browser.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import gseapy as gp
import pybedtools
import textwrap
import os

# ── PATHS ─────────────────────────────────────────────────────
PEAK_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
SAVE_DIR  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(SAVE_DIR, exist_ok=True)

RNA_PATHS = {
    'KO': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv",
    'FL': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    'dC': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    'dH': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}

RESCUE_THRESHOLD = 0.5
LFC_THRESHOLD    = 0.5
GENE_SETS        = ['GO_Biological_Process_2023']
ORGANISM         = 'Mouse'
TOP_N            = 10
FDR_CUTOFF       = 0.05

# ── STEP 1: NPC-specific ChIP targets ─────────────────────────
print("=" * 65)
print("DOMAIN-RESCUE FUNCTIONAL ENRICHMENT  (Comment #81)")
print("=" * 65)

def fix_chr(feature):
    if not str(feature.chrom).startswith('chr'):
        feature.chrom = 'chr' + str(feature.chrom)
    return feature

print("\n1. Identifying NPC-specific ChIP-seq targets...")
npc_bt = pybedtools.BedTool(PEAK_PATH).each(fix_chr).sort()
esc_bt = pybedtools.BedTool(ESC_PATH).each(fix_chr).sort()
tss_bt = pybedtools.BedTool(TSS_PATH).each(fix_chr).sort()

npc_only_peaks  = npc_bt.subtract(esc_bt, A=True)
hits            = tss_bt.intersect(npc_only_peaks, u=True, wa=True)
chip_target_set = set([str(f[3]).upper().strip() for f in hits
                       if len(str(f[3])) > 1])
print(f"   NPC-specific ChIP targets: {len(chip_target_set):,}")

# ── STEP 2: Load RNA-seq and merge ────────────────────────────
print("\n2. Loading RNA-seq for all 4 conditions...")
dfs = []
for label, path in RNA_PATHS.items():
    df = pd.read_csv(path, sep='\t')
    df['gene_id'] = df.iloc[:, 0].astype(str).str.split('.').str[0]
    dfs.append(
        df[['gene_id', 'GENESYMBOL', 'log2FoldChange']]
        .rename(columns={'log2FoldChange': f'LFC_{label}'})
    )

master = dfs[0]
for d in dfs[1:]:
    master = master.merge(d.drop(columns=['GENESYMBOL']), on='gene_id')

master['gene_upper'] = master['GENESYMBOL'].astype(str).str.upper()
targets = master[
    master['gene_upper'].isin(chip_target_set) &
    (master['LFC_KO'].abs() > LFC_THRESHOLD)
].copy()
print(f"   Direct targets with |KO LFC| > {LFC_THRESHOLD}: {len(targets):,}")

# ── STEP 3: Rescue fractions and group assignment ─────────────
print("\n3. Computing rescue fractions and assigning groups...")

for cond in ['FL', 'dC', 'dH']:
    targets[f'RF_{cond}'] = (
        (targets[f'LFC_{cond}'] - targets['LFC_KO']) /
        (0 - targets['LFC_KO'])
    )

targets['FL_rescued'] = targets['RF_FL'] >= RESCUE_THRESHOLD
targets['dC_rescued'] = targets['RF_dC'] >= RESCUE_THRESHOLD
targets['dH_rescued'] = targets['RF_dH'] >= RESCUE_THRESHOLD

def assign_group(row):
    fl, dc, dh = row['FL_rescued'], row['dC_rescued'], row['dH_rescued']
    if not fl:           return 'Low/No Rescue'
    if dc and dh:        return 'Global Rescue'
    if not dc and dh:    return 'Chromodomain Required'
    if dc and not dh:    return 'Helicase Required'
    if not dc and not dh: return 'Dual Domain Required'
    return 'Low/No Rescue'

targets['rescue_group'] = targets.apply(assign_group, axis=1)

group_order = [
    'Global Rescue',
    'Chromodomain Required',
    'Helicase Required',
    'Dual Domain Required',
    'Low/No Rescue',
]
group_colors = {
    'Global Rescue':          '#27ae60',
    'Chromodomain Required':  '#3498db',
    'Helicase Required':      '#2ecc71',
    'Dual Domain Required':   '#9b59b6',
    'Low/No Rescue':          '#95a5a6',
}

print("\n   Group counts:")
for grp in group_order:
    n   = (targets['rescue_group'] == grp).sum()
    pct = 100 * n / len(targets)
    print(f"   {grp:<28} : {n:4d}  ({pct:.1f}%)")

# ── STEP 4: Gene lists per group ──────────────────────────────
print("\n4. Extracting gene lists...")
group_genes = {}
for grp in group_order:
    genes = targets[targets['rescue_group'] == grp]['GENESYMBOL'].tolist()
    genes = [g for g in genes if isinstance(g, str) and len(g) > 1]
    group_genes[grp] = genes
    print(f"   {grp:<28} : {len(genes)} genes")

# Save supplementary gene list
supp_rows = []
for grp, genes in group_genes.items():
    for gene in genes:
        row_data = targets[targets['GENESYMBOL'] == gene].iloc[0]
        supp_rows.append({
            'rescue_group':  grp,
            'gene_symbol':   gene,
            'LFC_KO':        round(row_data['LFC_KO'], 3),
            'LFC_FL':        round(row_data['LFC_FL'], 3),
            'LFC_dChromo':   round(row_data['LFC_dC'], 3),
            'LFC_dHelicase': round(row_data['LFC_dH'], 3),
            'RF_FL':         round(row_data['RF_FL'],  3),
            'RF_dChromo':    round(row_data['RF_dC'],  3),
            'RF_dHelicase':  round(row_data['RF_dH'],  3),
        })

supp_df   = pd.DataFrame(supp_rows)
supp_path = os.path.join(SAVE_DIR, 'Supp_Table_Domain_Rescue_Groups.csv')
supp_df.to_csv(supp_path, index=False)
print(f"\n   Supplementary gene list saved: {supp_path}")

# ── STEP 5 FIX: correct organism string ───────────────────────
print("\n5. Running GO enrichment per group...")

enrichment_results = {}
for grp in group_order:
    genes   = group_genes[grp]
    n_genes = len(genes)
    if n_genes < 5:
        print(f"   Skipping '{grp}' — too few genes ({n_genes})")
        enrichment_results[grp] = None
        continue
    print(f"   Running: {grp} ({n_genes} genes)...")
    try:
        enr = gp.enrichr(
            gene_list = genes,
            gene_sets = ['GO_Biological_Process_2023'],
            organism  = 'mouse',          # ← fix: lowercase
            cutoff    = 0.5,
        )
        res = enr.results.copy()
        n_fdr = (res['Adjusted P-value'] < FDR_CUTOFF).sum()
        res['sig_col']   = res['Adjusted P-value'] if n_fdr > 0 else res['P-value']
        res['sig_label'] = 'FDR' if n_fdr > 0 else 'nominal P'
        enrichment_results[grp] = res
        print(f"   Significant terms: {(res['sig_col'] < FDR_CUTOFF).sum()}")
    except Exception as e:
        print(f"   Error for '{grp}': {e}")
        enrichment_results[grp] = None

# ── STEP 6: Save full table ────────────────────────────────────
all_rows = []
for grp, res in enrichment_results.items():
    if res is not None:
        r = res.copy()
        r.insert(0, 'rescue_group', grp)
        all_rows.append(r)

if all_rows:
    full_df  = pd.concat(all_rows, ignore_index=True)
    enr_path = os.path.join(SAVE_DIR, 'Supp_Table_Domain_Rescue_GO_Enrichment.csv')
    full_df.to_csv(enr_path, index=False)
    print(f"\n   Full enrichment table saved: {enr_path}")

# ── STEP 7: Figure — only if at least one group has results ───
groups_with_results = [g for g in group_order
                       if enrichment_results.get(g) is not None]
n_panels = len(groups_with_results)

if n_panels == 0:
    print("\n   No enrichment results to plot — check internet/API access.")
else:
    plt.rcParams.update({
        'font.family':     'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans'],
        'font.size':       10,
        'pdf.fonttype':    42,
    })

    fig = plt.figure(figsize=(7 * n_panels, 8), dpi=600)
    gs  = gridspec.GridSpec(1, n_panels, figure=fig, wspace=0.6)

    for idx, grp in enumerate(groups_with_results):
        ax  = fig.add_subplot(gs[idx])
        res = enrichment_results[grp]
        col = group_colors[grp]
        n_g = len(group_genes[grp])

        go_res = res.sort_values('sig_col').head(TOP_N).copy()

        if go_res.empty:
            ax.text(0.5, 0.5, 'No significant\nGO terms',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f"{grp}\n(n={n_g})", fontsize=10,
                         fontweight='bold', color=col)
            ax.axis('off')
            continue

        go_res['Term_clean'] = go_res['Term'].apply(
            lambda t: '\n'.join(textwrap.wrap(t.split(' (GO')[0], width=32))
        )
        go_res['-log10_p'] = -np.log10(go_res['sig_col'].clip(lower=1e-300))

        y_pos = np.arange(len(go_res))
        ax.barh(y_pos, go_res['-log10_p'], color=col,
                edgecolor='white', linewidth=0.5, alpha=0.85)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(go_res['Term_clean'], fontsize=8)
        ax.invert_yaxis()
        ax.axvline(-np.log10(FDR_CUTOFF), color='black',
                   linestyle='--', linewidth=0.8, alpha=0.5)
        ax.set_xlabel(f"-log₁₀({go_res['sig_label'].iloc[0]})", fontsize=9)
        ax.set_title(f"{grp}\n(n={n_g})", fontsize=10,
                     fontweight='bold', color=col, pad=8)
        ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle(
        'GO Biological Process Enrichment by Domain-Rescue Group\n'
        '(Direct CHD8 NPC targets — response to Reviewer Comment #81)',
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.tight_layout()

    fig_png = os.path.join(SAVE_DIR, 'FigS_Domain_Rescue_GO_Enrichment.png')
    fig_pdf = os.path.join(SAVE_DIR, 'FigS_Domain_Rescue_GO_Enrichment.pdf')
    plt.savefig(fig_png, dpi=300, bbox_inches='tight')
    plt.savefig(fig_pdf, bbox_inches='tight')
    plt.show()
    print(f"\n   Figure saved: {fig_png}")

# ── STEP 8: - text summary ───────────────────────────
print("\n" + "=" * 65)
print("- TEXT SUMMARY")
print("=" * 65)
for grp in group_order:
    res = enrichment_results.get(grp)
    n_g = len(group_genes[grp])
    if res is None or n_g < 5:
        print(f"\n  {grp} (n={n_g}): too few genes or enrichment failed.")
        continue
    sig = res[res['sig_col'] < FDR_CUTOFF].sort_values('sig_col').head(3)
    if sig.empty:
        print(f"\n  {grp} (n={n_g}): no significant GO terms.")
    else:
        terms = '; '.join([t.split(' (GO')[0] for t in sig['Term'].tolist()])
        print(f"\n  {grp} (n={n_g}):")
        print(f"  Top GO terms: {terms}")

In [ ]:
import pandas as pd
RNA_KO = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv"
df = pd.read_csv(RNA_KO, sep='\t').dropna(subset=['log2FoldChange','padj'])
df = df[df['baseMean'] > 10]
up   = ((df['padj'] < 0.05) & (df['log2FoldChange'] >  0.5)).sum()
down = ((df['padj'] < 0.05) & (df['log2FoldChange'] < -0.5)).sum()
print(f"Upregulated   : {up:,}")
print(f"Downregulated : {down:,}")
print(f"Total DEGs    : {up + down:,}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# Check the files
paths = {
    'FL': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    'dC': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    'dH': "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv"
}

for label, path in paths.items():
    df = pd.read_csv(path, sep="\t", nrows=3)
    print(f"\n{label}:")
    print(df.head())

In [ ]:
import os

flowcell_ids = [
    "HKNFWBGX5_3", "HKNFWBGX5_10", "HKLWCBGX5_21",
    "HKNFWBGX5_6", "HKNFWBGX5_15", "HKLWCBGX5_26",
    "HKNFWBGX5_1", "HKNFWBGX5_8",  "HKLWCBGX5_19",
    "HKNFWBGX5_4", "HKNFWBGX5_13", "HKLWCBGX5_24",
    "HKNFWBGX5_2", "HKNFWBGX5_5",  "HKNFWBGX5_9",
    "HKLWCBGX5_20","HKLWCBGX5_25", "HKNFWBGX5_14"
]

found = []
for root, dirs, files in os.walk("/content/drive/MyDrive/"):
    for fname in files:
        if any(fid in fname for fid in flowcell_ids):
            fpath = os.path.join(root, fname)
            size_gb = os.path.getsize(fpath) / (1024**3)
            print(f"  ✅ {fname}  ({size_gb:.2f} GB)  →  {root}")
            found.append(fpath)

print(f"\nTotal: {len(found)} files found")

In [ ]:
import pandas as pd

files = {
    "FL": "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    "dC": "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    "dH": "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}

# Fixed columns that are not sample IDs
non_sample_cols = {'Unnamed: 0', 'GENESYMBOL', 'GENETYPE', 'CHROMOSOME',
                   'Diff.Fa2Ls4', 'Diff.Chd8_KO_FL', 'Diff.Chd8_KO_dC',
                   'Diff.Chd8_KO_dHs', 'baseMean', 'rawlog2FoldChange',
                   'log2FoldChange', 'lfcSE', 'pvalue', 'padj', 'DEREG_FLAG'}

all_samples = set()
for label, path in files.items():
    df = pd.read_csv(path, sep='\t', nrows=1)
    samples = [c for c in df.columns if c not in non_sample_cols]
    print(f"\n{label} samples ({len(samples)}):")
    for s in samples:
        print(f"   {s}")
    all_samples.update(samples)

print(f"\n{'='*50}")
print(f"ALL UNIQUE SAMPLE IDs ACROSS ALL 3 FILES ({len(all_samples)}):")
for s in sorted(all_samples):
    print(f"   {s}")

In [ ]:
import pandas as pd

files = {
    "FL": "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    "dC": "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    "dH": "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}

for label, path in files.items():
    print(f"\n{'='*60}\n{label}: {path}")
    with open(path, 'r') as f:
        for i, line in enumerate(f):
            if i < 10:  # first 10 lines
                print(f"  Line {i+1}: {line.rstrip()}")
            else:
                break
    df = pd.read_csv(path, sep='\t', nrows=3)
    print(f"  Columns: {list(df.columns)}")

In [ ]:
import os
import re

DRIVE_ROOT = "/content/drive/MyDrive/Chd8 data"

KEYWORDS = ['GSE', 'GSM', 'SRR', 'SRX', 'accession', 'GEO', 'submission', 'PRJNA']

print("=" * 60)
print("SMART GEO ACCESSION SEARCH")
print("=" * 60)

found_files = []
found_content = []

for root, dirs, files in os.walk(DRIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in ['__pycache__', '.git']]

    for fname in files:
        fpath = os.path.join(root, fname)

        # 1. Filename match
        if any(kw.lower() in fname.lower() for kw in KEYWORDS):
            found_files.append(fpath)
            continue

        # 2. Content match (only small text/tsv/csv files <500KB)
        if fname.endswith(('.txt', '.csv', '.tsv', '.md', '.json', '.yaml', '.log')):
            try:
                if os.path.getsize(fpath) < 500_000:
                    with open(fpath, 'r', errors='ignore') as f:
                        content = f.read()
                    matches = re.findall(r'(GSE\d+|GSM\d+|SRR\d+|SRX\d+|PRJNA\d+)', content)
                    if matches:
                        found_content.append((fpath, list(set(matches))))
            except:
                pass

print(f"\n📁 Files with GEO-related NAMES ({len(found_files)}):")
for f in found_files:
    print(f"   {f}")

print(f"\n📄 Files with GEO accession IDs IN CONTENT ({len(found_content)}):")
for fpath, ids in found_content:
    print(f"   {fpath}")
    print(f"      IDs found: {ids}")

if not found_files and not found_content:
    print("\n⚠️  No GEO accession info found anywhere in the Chd8 data folder.")
    print("   The IDs likely exist only with whoever submitted the raw data.")

In [ ]:
import os
import re

DRIVE_ROOT = "/content/drive/MyDrive"

KEYWORDS = ['GSE', 'GSM', 'SRR', 'SRX', 'accession', 'GEO', 'submission', 'PRJNA']

# Folders to skip (large/irrelevant)
SKIP_DIRS = ['__pycache__', '.git', 'Trash', 'node_modules']

print("=" * 60)
print("FULL DRIVE - SMART GEO ACCESSION SEARCH")
print("=" * 60)

found_files = []
found_content = []

for root, dirs, files in os.walk(DRIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]

    for fname in files:
        fpath = os.path.join(root, fname)

        # 1. Filename match (exclude GSEA false positives)
        if any(kw.lower() in fname.lower() for kw in KEYWORDS):
            # Filter out GSEA rank files which are false positives
            if 'GSEA.rnk' not in fname and 'gsea' not in fname.lower():
                found_files.append(fpath)

        # 2. Content match (only small text files)
        if fname.endswith(('.txt', '.csv', '.tsv', '.md', '.json', '.yaml', '.log')):
            try:
                if os.path.getsize(fpath) < 500_000:
                    with open(fpath, 'r', errors='ignore') as f:
                        content = f.read()
                    # Strict pattern: - accession IDs (min 6 digits)
                    matches = re.findall(r'(GSE\d{4,}|GSM\d{4,}|SRR\d{5,}|SRX\d{5,}|PRJNA\d{4,})', content)
                    if matches:
                        found_content.append((fpath, list(set(matches))))
            except:
                pass

print(f"\n📁 Files with GEO-related NAMES ({len(found_files)}):")
for f in found_files:
    print(f"   {f}")

print(f"\n📄 Files with - GEO accession IDs IN CONTENT ({len(found_content)}):")
for fpath, ids in found_content:
    print(f"   {fpath}")
    print(f"      IDs found: {ids}")

if not found_files and not found_content:
    print("\n⚠️  No GEO accession info found anywhere in MyDrive.")

In [ ]:
import pandas as pd

undiff_ids = [
    "HKNFWBGX5_6", "HKNFWBGX5_15", "HKLWCBGX5_26",  # FL Undiff
    "HKNFWBGX5_4", "HKNFWBGX5_13", "HKLWCBGX5_24",  # dC Undiff
    "HKNFWBGX5_5", "HKNFWBGX5_14", "HKLWCBGX5_25",  # dHs Undiff
]

files = {
    "FL": "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_FL_CTRL.tsv",
    "dC": "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dC_CTRL.tsv",
    "dH": "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/Diff_Fa2Ls4_Diff.Chd8_KO_dHs_CTRL.tsv",
}

for label, path in files.items():
    df = pd.read_csv(path, sep='\t', nrows=1)
    found = [s for s in undiff_ids if s in df.columns]
    print(f"{label}: Undiff samples found in columns = {found if found else 'NONE'}")

In [ ]:
# ==============================================================================
# FIXED: SFARI Gene Download - Three fallback strategies
# ==============================================================================
import os
import requests
import pandas as pd
from io import StringIO
from scipy.stats import fisher_exact

# Define paths to your data
BASE_RNA = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
female_ko_path = os.path.join(BASE_RNA, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")

# Load female data
df_female = pd.read_csv(female_ko_path, sep='\t').dropna(subset=['log2FoldChange', 'padj'])
df_female = df_female[df_female['baseMean'] > 10]
df_female['gene_upper'] = df_female['GENESYMBOL'].astype(str).str.upper().str.strip()

# ──────────────────────────────────────────────────────────────────────────────
# SFARI DOWNLOAD — tries three strategies in order
# ──────────────────────────────────────────────────────────────────────────────
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://gene.sfari.org/database/human-gene/",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# Strategy 1: SFARI's live PHP download endpoint (current as of 2024-2025)
SFARI_URL_1 = "https://gene.sfari.org/wp-content/themes/sfari-gene/utilities/download-csv.php?api-endpoint=genes"

# Strategy 2: Known S3-hosted snapshot (community-shared, widely used in papers)
SFARI_URL_2 = "https://s3.amazonaws.com/sfari-gene/SFARI-Gene_genes_export.csv"

# Strategy 3: A well-maintained GitHub mirror used in published research
SFARI_URL_3 = "https://raw.githubusercontent.com/pathwaycommons/factoid/master/src/server/routes/api/document/sfari-gene.csv"

df_sfari = None
for attempt, url in enumerate([SFARI_URL_1, SFARI_URL_2, SFARI_URL_3], 1):
    try:
        print(f"Attempt {attempt}: {url}")
        res = requests.get(url, headers=headers, timeout=30)
        res.raise_for_status()
        df_sfari = pd.read_csv(StringIO(res.text))
        print(f"✓ Successfully loaded {len(df_sfari):,} SFARI rows from strategy {attempt}")
        print(f"  Columns: {list(df_sfari.columns)}")
        break
    except Exception as e:
        print(f"  ✗ Failed: {e}")

# ──────────────────────────────────────────────────────────────────────────────
# FALLBACK: Manual download instructions if all URLs fail
# ──────────────────────────────────────────────────────────────────────────────
if df_sfari is None:
    print("""
╔══════════════════════════════════════════════════════════════╗
║  ALL AUTOMATIC DOWNLOADS FAILED — Manual steps:             ║
║                                                              ║
║  1. Go to: https://gene.sfari.org/database/human-gene/      ║
║  2. Click "Download" (top-right of the gene table)          ║
║  3. Select "Gene" CSV and download                           ║
║  4. Upload the file to your Colab session or Drive           ║
║  5. Update the path below and re-run                        ║
╚══════════════════════════════════════════════════════════════╝
""")
    # Uncomment and set path after manual download:
    # df_sfari = pd.read_csv("/content/SFARI-Gene_genes_export.csv")
    raise RuntimeError("Could not load SFARI data. See manual instructions above.")

# ──────────────────────────────────────────────────────────────────────────────
# Flexible column detection (column names vary across SFARI releases)
# ──────────────────────────────────────────────────────────────────────────────
# Detect gene symbol column
gene_col = next((c for c in df_sfari.columns if 'gene' in c.lower() and 'symbol' in c.lower()), None)
if gene_col is None:
    gene_col = next((c for c in df_sfari.columns if 'symbol' in c.lower()), None)
if gene_col is None:
    gene_col = df_sfari.columns[0]  # fallback to first column
print(f"  Using gene symbol column: '{gene_col}'")

# Detect score column
score_col = next((c for c in df_sfari.columns if 'score' in c.lower()), None)
if score_col is None:
    score_col = next((c for c in df_sfari.columns if 'categor' in c.lower()), None)
print(f"  Using score column: '{score_col}'")

df_sfari['gene_upper'] = df_sfari[gene_col].astype(str).str.upper().str.strip()
sfari_dict = dict(zip(df_sfari['gene_upper'], df_sfari[score_col])) if score_col else {g: None for g in df_sfari['gene_upper']}

# ──────────────────────────────────────────────────────────────────────────────
# Intersect Female DEGs with SFARI
# ──────────────────────────────────────────────────────────────────────────────
df_female['sfari_score'] = df_female['gene_upper'].map(sfari_dict)
df_female['is_sfari'] = df_female['gene_upper'].isin(sfari_dict.keys())

female_degs = df_female[(df_female['padj'] < 0.05) & (df_female['log2FoldChange'].abs() > 0.5)]
female_deg_set = set(female_degs['gene_upper'].tolist())

# Fisher's Exact Test
N = len(df_female)
k = len(female_deg_set)
K = df_female['is_sfari'].sum()
x = female_degs['is_sfari'].sum()

odds_ratio, p_val = fisher_exact([[x, k - x], [K - x, N - K - (k - x)]], alternative='greater')

print("\n" + "="*60)
print("SUCCESSFUL STATISTICAL AUDIT FOR -:")
print("="*60)
print(f"Female Background Genes (N)        : {N:,}")
print(f"Female Signif. DEGs (k)            : {k:,}")
print(f"SFARI Genes in Background (K)      : {K:,}")
print(f"Observed Overlap (x)               : {x:,}")
print(f"Fisher's Exact Enrichment p-value  : {p_val:.3e}")
print(f"Odds Ratio                         : {odds_ratio:.3f}")
print("="*60)

if score_col:
    hc_asd = female_degs[female_degs['sfari_score'] == 1]['gene_upper'].dropna().tolist()
    print(f"Number of Tier-1 (High-Confidence) ASD Risk Genes Dysregulated: {len(hc_asd)}")
    if hc_asd:
        print(f"Representative High-Confidence Targets: {', '.join(hc_asd[:10])}")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── - / BMC figure standards ──────────────────────────────────
plt.rcParams['font.family']  = 'Helvetica'
plt.rcParams['font.size']    = 7
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

output_dir  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_dir, exist_ok=True)
output_pdf  = os.path.join(output_dir, "Figure3_SFARI_Tier1to4_ASD_DEGs.pdf")
output_xlsx = os.path.join(output_dir, "SFARI_Tier1to4_ASD_DEGs.xlsx")

# ──────────────────────────────────────────────────────────────────────────────
# 1) Define significant DEGs, then restrict to genes with a SFARI tier (1-4)
# ──────────────────────────────────────────────────────────────────────────────
df_female['sfari_score'] = pd.to_numeric(df_female['sfari_score'], errors='coerce')

sig_degs = df_female[
    (df_female['padj'] < 0.05) & (df_female['log2FoldChange'].abs() > 0.5)
].copy()

asd_degs = sig_degs[sig_degs['sfari_score'].isin([1, 2, 3, 4])].copy()
asd_degs['direction'] = asd_degs['log2FoldChange'].apply(lambda x: 'Up-regulated' if x > 0 else 'Down-regulated')

well_known_asd_genes = {
    'CHD8', 'MECP2', 'FMR1', 'SHANK3', 'PTEN', 'TSC1', 'TSC2',
    'SCN2A', 'SYNGAP1', 'ADNP', 'ARID1B', 'DYRK1A', 'GRIN2B',
    'NRXN1', 'CDKL5', 'AUTS2', 'FOXP1', 'CNTNAP2', 'NLGN3', 'NLGN4X',
    'RELN', 'CACNA1C', 'OXTR', 'SMARCA2', 'TCF4', 'MEIS2',
    'BDNF', 'SLC6A4', 'EN2', 'MET', 'NRG1',
}
asd_degs['well_known_asd_gene'] = asd_degs['gene_upper'].isin(well_known_asd_genes)

print(f"Significant DEGs total       : {len(sig_degs):,}")
print(f"Significant DEGs w/ SFARI 1-4: {len(asd_degs):,}")
for t in [1, 2, 3, 4]:
    print(f"  Tier {t}: {int((asd_degs['sfari_score'] == t).sum())} genes")

if asd_degs.empty:
    raise RuntimeError("No significant DEGs found with SFARI tier 1-4 annotation. "
                        "Check that df_female['sfari_score'] was populated correctly.")

N_PER_BUCKET = 3  # genes per tier PER DIRECTION shown in the figure (≤ 4 tiers x 2 directions x 3 = 24 max)

figure_rows = []
selection_log = []  # traceability: which genes were picked and why

for t in [1, 2, 3, 4]:
    for dir_label, cond in [('Down-regulated', asd_degs['log2FoldChange'] < 0),
                             ('Up-regulated',   asd_degs['log2FoldChange'] > 0)]:
        bucket = asd_degs[(asd_degs['sfari_score'] == t) & cond].copy()
        if bucket.empty:
            continue
        bucket['abs_lfc'] = bucket['log2FoldChange'].abs()

        known_hits = (bucket[bucket['well_known_asd_gene']]
                      .sort_values(['padj', 'abs_lfc'], ascending=[True, False]))
        chosen = known_hits.head(N_PER_BUCKET).copy()
        chosen['selection_reason'] = 'curated well-known ASD gene'

        if len(chosen) < N_PER_BUCKET:
            remaining = (bucket[~bucket['gene_upper'].isin(chosen['gene_upper'])]
                         .sort_values(['padj', 'abs_lfc'], ascending=[True, False]))
            backfill = remaining.head(N_PER_BUCKET - len(chosen)).copy()
            backfill['selection_reason'] = 'top significant DEG (backfill, not on curated list)'
            chosen = pd.concat([chosen, backfill], ignore_index=True)

        figure_rows.append(chosen)
        selection_log.append(chosen[['gene_upper', 'sfari_score', 'log2FoldChange',
                                      'padj', 'well_known_asd_gene', 'selection_reason']])

figure_df = pd.concat(figure_rows, ignore_index=True) if figure_rows else asd_degs.iloc[0:0].copy()
selection_log_df = pd.concat(selection_log, ignore_index=True) if selection_log else pd.DataFrame()

if figure_df.empty:
    raise RuntimeError("No genes selected for the main figure — check well_known_asd_genes "
                        "overlap and N_PER_BUCKET settings.")

print(f"\nGenes selected for MAIN FIGURE (balanced representative subset): {len(figure_df)}")
print(f"Complete Tier 1-4 gene list for supplement/Excel               : {len(asd_degs)}")

export_cols = ['gene_upper', 'log2FoldChange', 'padj', 'baseMean',
               'sfari_score', 'is_sfari', 'well_known_asd_gene', 'direction']
export_cols = [c for c in export_cols if c in asd_degs.columns]

rename_map = {'gene_upper': 'Gene', 'sfari_score': 'SFARI_Tier',
              'is_sfari': 'SFARI_Listed', 'well_known_asd_gene': 'Well_Known_ASD_Gene'}

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
    (asd_degs[export_cols]
        .sort_values(['sfari_score', 'log2FoldChange'])
        .rename(columns=rename_map)
        .to_excel(writer, sheet_name='All_Tier1-4_ASD_DEGs', index=False))

    for t in [1, 2, 3, 4]:
        sub = asd_degs[asd_degs['sfari_score'] == t][export_cols].sort_values('log2FoldChange')
        if len(sub):
            sub.rename(columns=rename_map).to_excel(writer, sheet_name=f'Tier_{t}', index=False)

    if not selection_log_df.empty:
        (selection_log_df.rename(columns={**rename_map, 'selection_reason': 'Why_shown_in_main_figure'})
            .to_excel(writer, sheet_name='Figure_Gene_Selection', index=False))

print(f"✓ Excel written: {output_xlsx}")
print("  Sheets: All_Tier1-4_ASD_DEGs (complete list incl. Well_Known_ASD_Gene column),")
print("          Tier_1..Tier_4, Figure_Gene_Selection (main-figure subset + rationale)")

figure_df = figure_df.copy()
figure_df['ylabel'] = figure_df.apply(lambda r: f"{r['gene_upper']} (T{int(r['sfari_score'])})", axis=1)

# Four visually DISTINCT tier colors (not a single-hue gradient)
tier_colors = {
    1: '#7B3294',   # purple  — Tier 1, High Confidence
    2: '#2C7FB8',   # blue    — Tier 2, Strong Candidate
    3: '#1B9E77',   # teal    — Tier 3, Suggestive Evidence
    4: '#E6AB02',   # amber   — Tier 4
}

df_down = figure_df[figure_df['log2FoldChange'] < 0].sort_values('log2FoldChange', ascending=True)
df_up   = figure_df[figure_df['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False)
n_down, n_up = len(df_down), len(df_up)

df_down_plot = df_down.sort_values('log2FoldChange', ascending=True)
df_up_plot   = df_up.sort_values('log2FoldChange', ascending=False)

BMC_MAX_HEIGHT_IN = 9.21
bar_slot_in = 0.28
margin_in   = 2.1
fig_height = min(BMC_MAX_HEIGHT_IN, bar_slot_in * max(n_down, n_up, 1) + margin_in)

fig, (ax_down, ax_up) = plt.subplots(
    1, 2, figsize=(7.28, fig_height), dpi=300,
    gridspec_kw={'width_ratios': [1, 1]}
)

panel_specs = [
    (ax_down, df_down_plot, f'Down-regulated (n={n_down})'),
    (ax_up,   df_up_plot,   f'Up-regulated (n={n_up})'),
]

for ax, df_plot, panel_title in panel_specs:
    if df_plot.empty:
        ax.set_axis_off()
        ax.set_title(panel_title, fontsize=8, fontweight='bold', loc='center', pad=4)
        continue

    bar_colors = [tier_colors.get(int(t), '#999999') for t in df_plot['sfari_score']]
    bars = ax.barh(df_plot['ylabel'], df_plot['log2FoldChange'],
                    color=bar_colors, edgecolor='black', linewidth=0.4, height=0.7)

    ax.axvline(0, color='black', linestyle='-', linewidth=0.75)
    ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
    ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis='both', width=0.5, labelsize=6)
    ax.invert_yaxis()

    ax.set_xlabel(r'$\log_2$ FC (KO/WT)', fontsize=7, fontweight='bold')
    # Direction is written once, as the panel title — not on every bar
    ax.set_title(panel_title, fontsize=8, fontweight='bold', loc='center', pad=6)

    xmin, xmax = ax.get_xlim()
    pad = (xmax - xmin) * 0.15
    if xmax > 0:
        ax.set_xlim(xmin, xmax + pad)
    else:
        ax.set_xlim(xmin - pad, xmax)

    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = x_range * 0.02
    for bar in bars:
        width = bar.get_width()
        x_pos = width + offset if width >= 0 else width - offset
        ha = 'left' if width >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

fig.suptitle('Female CHD8-KO: Representative SFARI Tier 1-4 ASD-Risk DEGs',
             fontsize=9, fontweight='bold', x=0.02, ha='left', y=1.02)

legend_handles = [
    Patch(facecolor=tier_colors[1], edgecolor='black', linewidth=0.4, label='SFARI Tier 1 (High Confidence)'),
    Patch(facecolor=tier_colors[2], edgecolor='black', linewidth=0.4, label='SFARI Tier 2 (Strong Candidate)'),
    Patch(facecolor=tier_colors[3], edgecolor='black', linewidth=0.4, label='SFARI Tier 3 (Suggestive Evidence)'),
    Patch(facecolor=tier_colors[4], edgecolor='black', linewidth=0.4, label='SFARI Tier 4'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=False,
           fontsize=6, bbox_to_anchor=(0.53, -0.04))

plt.tight_layout(rect=[0.02, 0.05, 1, 0.97], pad=0.6)
plt.savefig(output_pdf, format='pdf', bbox_inches='tight')
plt.show()

print(f"✓ Figure saved to: {output_pdf}")
print(f"  Height: {fig_height:.2f}in (BMC max: {BMC_MAX_HEIGHT_IN}in)")

In [ ]:

get_ipython().system('pip install mygene --quiet')

import pandas as pd
import numpy as np
import mygene
from scipy.stats import fisher_exact
import time

mg = mygene.MyGeneInfo()

# ──────────────────────────────────────────────────────────────────────────────
# 1) Define the two gene sets to compare
# ──────────────────────────────────────────────────────────────────────────────
tier1_genes = df_female[df_female['sfari_score'] == 1]['gene_upper'].dropna().unique().tolist()
all_genes   = df_female['gene_upper'].dropna().unique().tolist()

print(f"Tier-1 genes to resolve   : {len(tier1_genes)}")
print(f"Background genes to resolve: {len(all_genes)}")

# ──────────────────────────────────────────────────────────────────────────────
# 2) Query chromosome location via mygene, in batches (large background list
#    can be slow / rate-limited if sent all at once)
# ──────────────────────────────────────────────────────────────────────────────
def query_chrom_batched(gene_list, batch_size=500, label=""):
    results = []
    for i in range(0, len(gene_list), batch_size):
        batch = gene_list[i:i+batch_size]
        print(f"  [{label}] querying genes {i+1}-{min(i+batch_size, len(gene_list))} of {len(gene_list)}...")
        res = mg.querymany(batch, scopes='symbol', fields='genomic_pos.chr',
                            species='mouse', as_dataframe=True, verbose=False)
        results.append(res)
        time.sleep(1)  # be polite to the API between batches
    return pd.concat(results) if results else pd.DataFrame()

print("\nResolving Tier-1 gene chromosomes...")
tier1_map = query_chrom_batched(tier1_genes, label="Tier-1")

print("\nResolving background gene chromosomes (this may take a few minutes)...")
bg_map = query_chrom_batched(all_genes, label="background")

# ──────────────────────────────────────────────────────────────────────────────
# 3) Extract X-linked status, handling genes with multiple hits/no hits
# ──────────────────────────────────────────────────────────────────────────────
def get_chrom_col(df):
    if 'genomic_pos.chr' in df.columns:
        return df['genomic_pos.chr']
    else:
        return pd.Series([np.nan] * len(df))

tier1_chrom = get_chrom_col(tier1_map).astype(str).str.upper()
bg_chrom    = get_chrom_col(bg_map).astype(str).str.upper()

tier1_resolved = tier1_chrom[tier1_chrom.notna() & (tier1_chrom != 'NAN')]
bg_resolved    = bg_chrom[bg_chrom.notna() & (bg_chrom != 'NAN')]

tier1_x = (tier1_resolved == 'X').sum()
tier1_n = len(tier1_resolved)
bg_x    = (bg_resolved == 'X').sum()
bg_n    = len(bg_resolved)

print("\n" + "=" * 60)
print("RESOLUTION SUMMARY")
print("=" * 60)
print(f"Tier-1: {tier1_n}/{len(tier1_genes)} genes resolved to a chromosome ({tier1_n/len(tier1_genes)*100:.1f}%)")
print(f"Background: {bg_n}/{len(all_genes)} genes resolved to a chromosome ({bg_n/len(all_genes)*100:.1f}%)")

if tier1_n < len(tier1_genes) * 0.9 or bg_n < len(all_genes) * 0.9:
    print("\n⚠ WARNING: more than 10% of genes failed to resolve. Results below may be")
    print("  unreliable — consider manually checking unresolved gene symbols before")
    print("  trusting this test.")

# ──────────────────────────────────────────────────────────────────────────────
# 4) Fisher's exact test: is Tier-1's X-linked rate different from background?
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FISHER'S EXACT TEST: Tier-1 X-linked rate vs. background X-linked rate")
print("=" * 60)
print(f"Tier-1: {tier1_x}/{tier1_n} X-linked ({100*tier1_x/tier1_n:.1f}%)" if tier1_n else "Tier-1: no resolved genes")
print(f"Background: {bg_x}/{bg_n} X-linked ({100*bg_x/bg_n:.1f}%)" if bg_n else "Background: no resolved genes")

if tier1_n > 0 and bg_n > 0:
    # 2x2 table: rows = [Tier-1, background-excluding-Tier1], cols = [X-linked, autosomal]
    bg_only_x  = bg_x - tier1_x
    bg_only_n  = bg_n - tier1_n
    table = [[tier1_x, tier1_n - tier1_x],
             [bg_only_x, bg_only_n - bg_only_x]]

    odds_ratio, p_two_sided = fisher_exact(table, alternative='two-sided')
    _, p_greater = fisher_exact(table, alternative='greater')

    print(f"\nOdds ratio                  : {odds_ratio:.3f}")
    print(f"Two-sided p-value           : {p_two_sided:.3e}")
    print(f"One-sided p-value (enriched): {p_greater:.3e}")

    print("\n" + "-" * 60)
    if p_two_sided < 0.05 and odds_ratio > 1:
        print("RESULT: Statistically significant X-linked enrichment in Tier-1.")
        print("-> The claim IS supported by this test. Still confirm with -")
        print("   whether to include it, given his earlier direction to avoid")
        print("   X-chromosome framing in this section.")
    elif p_two_sided < 0.05 and odds_ratio < 1:
        print("RESULT: Statistically significant, but DEPLETION not enrichment.")
        print("-> The original claim is NOT supported — remove/correct it.")
    else:
        print("RESULT: NOT statistically significant.")
        print("-> The claim 'particularly enriched' is NOT supported by this test.")
        print("   Recommend removing the sentence, or rephrasing as a plain")
        print("   descriptive count without implying enrichment or mechanism.")
else:
    print("\n⚠ Could not run test — insufficient resolved genes on one or both sides.")

print("=" * 60)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── - / BMC figure standards ──────────────────────────────────
plt.rcParams['font.family']  = 'Helvetica'
plt.rcParams['font.size']    = 7
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

output_dir  = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_dir, exist_ok=True)
output_pdf  = os.path.join(output_dir, "Figure3_SFARI_Tier1to4_ASD_DEGs.pdf")
output_xlsx = os.path.join(output_dir, "SFARI_Tier1to4_ASD_DEGs.xlsx")

# ──────────────────────────────────────────────────────────────────────────────
# 1) Define significant DEGs, then restrict to genes with a SFARI tier (1-4)
# ──────────────────────────────────────────────────────────────────────────────
df_female['sfari_score'] = pd.to_numeric(df_female['sfari_score'], errors='coerce')

sig_degs = df_female[
    (df_female['padj'] < 0.05) & (df_female['log2FoldChange'].abs() > 0.5)
].copy()

asd_degs = sig_degs[sig_degs['sfari_score'].isin([1, 2, 3, 4])].copy()
asd_degs['direction'] = asd_degs['log2FoldChange'].apply(lambda x: 'Up-regulated' if x > 0 else 'Down-regulated')

well_known_asd_genes = {
    'CHD8', 'MECP2', 'FMR1', 'SHANK3', 'PTEN', 'TSC1', 'TSC2',
    'SCN2A', 'SYNGAP1', 'ADNP', 'ARID1B', 'DYRK1A', 'GRIN2B',
    'NRXN1', 'CDKL5', 'AUTS2', 'FOXP1', 'CNTNAP2', 'NLGN3', 'NLGN4X',
    'RELN', 'CACNA1C', 'OXTR', 'SMARCA2', 'TCF4', 'MEIS2',
    'BDNF', 'SLC6A4', 'EN2', 'MET', 'NRG1',
}
asd_degs['well_known_asd_gene'] = asd_degs['gene_upper'].isin(well_known_asd_genes)

print(f"Significant DEGs total       : {len(sig_degs):,}")
print(f"Significant DEGs w/ SFARI 1-4: {len(asd_degs):,}")
for t in [1, 2, 3, 4]:
    print(f"  Tier {t}: {int((asd_degs['sfari_score'] == t).sum())} genes")

if asd_degs.empty:
    raise RuntimeError("No significant DEGs found with SFARI tier 1-4 annotation. "
                        "Check that df_female['sfari_score'] was populated correctly.")

N_PER_BUCKET = 3  # genes per tier PER DIRECTION shown in the figure (≤ 4 tiers x 2 directions x 3 = 24 max)

figure_rows = []
selection_log = []  # traceability: which genes were picked and why

for t in [1, 2, 3, 4]:
    for dir_label, cond in [('Down-regulated', asd_degs['log2FoldChange'] < 0),
                             ('Up-regulated',   asd_degs['log2FoldChange'] > 0)]:
        bucket = asd_degs[(asd_degs['sfari_score'] == t) & cond].copy()
        if bucket.empty:
            continue
        bucket['abs_lfc'] = bucket['log2FoldChange'].abs()

        known_hits = (bucket[bucket['well_known_asd_gene']]
                      .sort_values(['padj', 'abs_lfc'], ascending=[True, False]))
        chosen = known_hits.head(N_PER_BUCKET).copy()
        chosen['selection_reason'] = 'curated well-known ASD gene'

        if len(chosen) < N_PER_BUCKET:
            remaining = (bucket[~bucket['gene_upper'].isin(chosen['gene_upper'])]
                         .sort_values(['padj', 'abs_lfc'], ascending=[True, False]))
            backfill = remaining.head(N_PER_BUCKET - len(chosen)).copy()
            backfill['selection_reason'] = 'top significant DEG (backfill, not on curated list)'
            chosen = pd.concat([chosen, backfill], ignore_index=True)

        figure_rows.append(chosen)
        selection_log.append(chosen[['gene_upper', 'sfari_score', 'log2FoldChange',
                                      'padj', 'well_known_asd_gene', 'selection_reason']])

figure_df = pd.concat(figure_rows, ignore_index=True) if figure_rows else asd_degs.iloc[0:0].copy()
selection_log_df = pd.concat(selection_log, ignore_index=True) if selection_log else pd.DataFrame()

if figure_df.empty:
    raise RuntimeError("No genes selected for the main figure — check well_known_asd_genes "
                        "overlap and N_PER_BUCKET settings.")

print(f"\nGenes selected for MAIN FIGURE (balanced representative subset): {len(figure_df)}")
print(f"Complete Tier 1-4 gene list for supplement/Excel               : {len(asd_degs)}")

export_cols = ['gene_upper', 'log2FoldChange', 'padj', 'baseMean',
               'sfari_score', 'is_sfari', 'well_known_asd_gene', 'direction']
export_cols = [c for c in export_cols if c in asd_degs.columns]

rename_map = {'gene_upper': 'Gene', 'sfari_score': 'SFARI_Tier',
              'is_sfari': 'SFARI_Listed', 'well_known_asd_gene': 'Well_Known_ASD_Gene'}

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
    (asd_degs[export_cols]
        .sort_values(['sfari_score', 'log2FoldChange'])
        .rename(columns=rename_map)
        .to_excel(writer, sheet_name='All_Tier1-4_ASD_DEGs', index=False))

    for t in [1, 2, 3, 4]:
        sub = asd_degs[asd_degs['sfari_score'] == t][export_cols].sort_values('log2FoldChange')
        if len(sub):
            sub.rename(columns=rename_map).to_excel(writer, sheet_name=f'Tier_{t}', index=False)

    if not selection_log_df.empty:
        (selection_log_df.rename(columns={**rename_map, 'selection_reason': 'Why_shown_in_main_figure'})
            .to_excel(writer, sheet_name='Figure_Gene_Selection', index=False))

print(f"✓ Excel written: {output_xlsx}")
print("  Sheets: All_Tier1-4_ASD_DEGs (complete list incl. Well_Known_ASD_Gene column),")
print("          Tier_1..Tier_4, Figure_Gene_Selection (main-figure subset + rationale)")

figure_df = figure_df.copy()
figure_df['ylabel'] = figure_df.apply(lambda r: f"{r['gene_upper']} (T{int(r['sfari_score'])})", axis=1)

# Four visually DISTINCT tier colors (not a single-hue gradient)
tier_colors = {
    1: '#7B3294',   # purple  — Tier 1, High Confidence
    2: '#2C7FB8',   # blue    — Tier 2, Strong Candidate
    3: '#1B9E77',   # teal    — Tier 3, Suggestive Evidence
    4: '#E6AB02',   # amber   — Tier 4
}

df_down = figure_df[figure_df['log2FoldChange'] < 0].sort_values('log2FoldChange', ascending=True)
df_up   = figure_df[figure_df['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False)
n_down, n_up = len(df_down), len(df_up)

df_down_plot = df_down.sort_values('log2FoldChange', ascending=True)
df_up_plot   = df_up.sort_values('log2FoldChange', ascending=False)

BMC_MAX_HEIGHT_IN = 9.21
bar_slot_in = 0.28
margin_in   = 2.1
fig_height = min(BMC_MAX_HEIGHT_IN, bar_slot_in * max(n_down, n_up, 1) + margin_in)

fig, (ax_down, ax_up) = plt.subplots(
    1, 2, figsize=(7.28, fig_height), dpi=300,
    gridspec_kw={'width_ratios': [1, 1]}
)

panel_specs = [
    (ax_down, df_down_plot, f'Down-regulated (n={n_down})'),
    (ax_up,   df_up_plot,   f'Up-regulated (n={n_up})'),
]

for ax, df_plot, panel_title in panel_specs:
    if df_plot.empty:
        ax.set_axis_off()
        ax.set_title(panel_title, fontsize=8, fontweight='bold', loc='center', pad=4)
        continue

    bar_colors = [tier_colors.get(int(t), '#999999') for t in df_plot['sfari_score']]
    bars = ax.barh(df_plot['ylabel'], df_plot['log2FoldChange'],
                    color=bar_colors, edgecolor='black', linewidth=0.4, height=0.7)

    ax.axvline(0, color='black', linestyle='-', linewidth=0.75)
    ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
    ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis='both', width=0.5, labelsize=6)
    ax.invert_yaxis()

    ax.set_xlabel(r'$\log_2$ FC (KO/WT)', fontsize=7, fontweight='bold')
    # Direction is written once, as the panel title — not on every bar
    ax.set_title(panel_title, fontsize=8, fontweight='bold', loc='center', pad=6)

    data_min = min(0, df_plot['log2FoldChange'].min())
    data_max = max(0, df_plot['log2FoldChange'].max())
    data_range = (data_max - data_min) or 1.0
    pad = data_range * 0.22
    ax.set_xlim(data_min - pad, data_max + pad)

    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = x_range * 0.02
    for bar in bars:
        width = bar.get_width()
        x_pos = width + offset if width >= 0 else width - offset
        ha = 'left' if width >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

fig.suptitle('Female CHD8-KO: Representative SFARI Tier 1-4 ASD-Risk DEGs',
             fontsize=9, fontweight='bold', x=0.02, ha='left', y=1.02)

legend_handles = [
    Patch(facecolor=tier_colors[1], edgecolor='black', linewidth=0.4, label='SFARI Tier 1 (High Confidence)'),
    Patch(facecolor=tier_colors[2], edgecolor='black', linewidth=0.4, label='SFARI Tier 2 (Strong Candidate)'),
    Patch(facecolor=tier_colors[3], edgecolor='black', linewidth=0.4, label='SFARI Tier 3 (Suggestive Evidence)'),
    Patch(facecolor=tier_colors[4], edgecolor='black', linewidth=0.4, label='SFARI Tier 4'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=False,
           fontsize=6, bbox_to_anchor=(0.53, -0.04))

plt.tight_layout(rect=[0.02, 0.05, 1, 0.97], pad=0.6)
plt.savefig(output_pdf, format='pdf', bbox_inches='tight')
plt.show()

print(f"✓ Figure saved to: {output_pdf}")
print(f"  Height: {fig_height:.2f}in (BMC max: {BMC_MAX_HEIGHT_IN}in)")

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── - / BMC figure standards ──────────────────────────────────
plt.rcParams['font.family']   = 'Helvetica'
plt.rcParams['font.size']     = 7
plt.rcParams['pdf.fonttype']  = 42
plt.rcParams['svg.fonttype']  = 'none'

famous_asd_genes = [
    # Tier 1 (high confidence) — classic/most-cited ASD genes
    'CHD8', 'MECP2', 'FMR1', 'SHANK3', 'PTEN', 'TSC1', 'TSC2',
    'SCN2A', 'SYNGAP1', 'ADNP', 'ARID1B', 'DYRK1A', 'GRIN2B',
    'NRXN1', 'CDKL5', 'AUTS2', 'FOXP1',
    # Tier 2 (strong candidate) — widely recognized, slightly lower confidence
    'CNTNAP2', 'NLGN3', 'NLGN4X', 'RELN', 'CACNA1C', 'OXTR',
    'SMARCA2', 'TCF4', 'MEIS2',
    # Tier 3 (suggestive evidence) — still frequently discussed in ASD literature
    'BDNF', 'SLC6A4', 'EN2', 'MET', 'NRG1',
]

df_famous = female_degs[
    female_degs['gene_upper'].str.upper().isin([g.upper() for g in famous_asd_genes])
].copy()

# Sanity check: confirm which requested genes were actually found/significant
found = set(df_famous['gene_upper'].str.upper())
missing = [g for g in famous_asd_genes if g.upper() not in found]
if missing:
    print(f"⚠️ Not found / not significant in this dataset: {missing}")

print(f"Tier representation in selection: "
      f"T1={sum(df_famous['sfari_score']==1)}, "
      f"T2={sum(df_famous['sfari_score']==2)}, "
      f"T3={sum(df_famous['sfari_score']==3)}")

# ── Split by direction ──────────────────────────────────────────────────────
df_down = df_famous[df_famous['log2FoldChange'] < 0].sort_values('log2FoldChange', ascending=True)
df_up   = df_famous[df_famous['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False)

n_down, n_up = len(df_down), len(df_up)
print(f"Selected: {n_down} downregulated, {n_up} upregulated (n={n_down + n_up} total)")

df_down_plot = df_down.sort_values('log2FoldChange', ascending=False)  # most negative at top
df_up_plot   = df_up.sort_values('log2FoldChange', ascending=False)    # most positive at top

# Tier-annotated y-labels, e.g. "MECP2 (T1)"
def tier_label(row):
    return f"{row['gene_upper']} (T{int(row['sfari_score'])})"

df_down_plot['ylabel'] = df_down_plot.apply(tier_label, axis=1)
df_up_plot['ylabel']   = df_up_plot.apply(tier_label, axis=1)

# ── BMC max double-column type area: 234mm (h) x 185mm (w) = 9.21 x 7.28 in ─
BMC_MAX_HEIGHT_IN = 9.21
bar_slot_in       = 0.22
margin_in         = 1.9

fig_height = min(BMC_MAX_HEIGHT_IN, bar_slot_in * max(n_down, n_up) + margin_in)

fig, (ax_down, ax_up) = plt.subplots(
    1, 2, figsize=(7.28, fig_height), dpi=300,
    gridspec_kw={'width_ratios': [1, 1]}
)

tier_edge = {1: 'black', 2: '#555555', 3: '#aaaaaa'}
tier_lw   = {1: 0.9, 2: 0.5, 3: 0.3}

panel_specs = [
    (ax_down, df_down_plot, '#4575b4', 'A', f'Downregulated (n={n_down})'),
    (ax_up,   df_up_plot,   '#d73027', 'B', f'Upregulated (n={n_up})'),
]

for ax, df_plot, color, tag, panel_title in panel_specs:
    edge_colors = [tier_edge[s] for s in df_plot['sfari_score']]
    edge_widths = [tier_lw[s] for s in df_plot['sfari_score']]

    bars = ax.barh(df_plot['ylabel'], df_plot['log2FoldChange'],
                    color=color, edgecolor=edge_colors, linewidth=edge_widths, height=0.7)

    ax.axvline(0, color='black', linestyle='-', linewidth=0.75)
    ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
    ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis='both', width=0.5, labelsize=6)
    ax.invert_yaxis()

    ax.set_xlabel(r'$\log_2$ FC (KO/WT)', fontsize=7, fontweight='bold')
    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='bottom', ha='left')
    ax.set_title(panel_title, fontsize=7, fontweight='bold', loc='left', pad=2)

    xmin, xmax = ax.get_xlim()
    pad = (xmax - xmin) * 0.08
    if xmax > 0:
        ax.set_xlim(xmin, xmax + pad)
    else:
        ax.set_xlim(xmin - pad, xmax)

    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = x_range * 0.015
    for bar in bars:
        width = bar.get_width()
        x_pos = width + offset if width >= 0 else width - offset
        ha = 'left' if width >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

fig.text(0.02, 0.5, 'Known ASD Risk Gene (SFARI Tier 1–3)',
          fontsize=7, fontweight='bold', rotation=90, va='center')

# ── Single shared legend below both panels — direction + all 3 tiers ───────
shared_handles = [
    Patch(facecolor='#4575b4', edgecolor='black', linewidth=0.4, label='Downregulated'),
    Patch(facecolor='#d73027', edgecolor='black', linewidth=0.4, label='Upregulated'),
    Patch(facecolor='white', edgecolor='black', linewidth=0.9, label='SFARI Tier 1 (high conf.)'),
    Patch(facecolor='white', edgecolor='#555555', linewidth=0.5, label='SFARI Tier 2 (strong cand.)'),
    Patch(facecolor='white', edgecolor='#aaaaaa', linewidth=0.3, label='SFARI Tier 3 (suggestive)'),
]
fig.legend(handles=shared_handles, loc='lower center', ncol=5,
           frameon=False, fontsize=5.5, bbox_to_anchor=(0.53, -0.02))

plt.tight_layout(rect=[0.03, 0.05, 1, 1], pad=0.5)

output_fig_dir = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_fig_dir, exist_ok=True)
output_pdf_path = os.path.join(output_fig_dir, "Figure8_Famous_ASD_Genes_Tier1to3.pdf")
plt.savefig(output_pdf_path, format='pdf', bbox_inches='tight')
plt.show()
print(f"✓ Figure saved to:\n{output_pdf_path}")
print(f"  Height: {fig_height:.2f}in (BMC max: {BMC_MAX_HEIGHT_IN}in)")

In [ ]:
print("sfari_score values present in female_degs:", sorted(female_degs['sfari_score'].dropna().unique()))
print()
for g in famous_asd_genes:
    match = female_degs[female_degs['gene_upper'].str.upper() == g.upper()]
    if match.empty:
        print(f"{g}: NOT in dataset at all")
    else:
        row = match.iloc[0]
        print(f"{g}: tier={row['sfari_score']}, log2FC={row['log2FoldChange']:.2f}, padj={row.get('padj', 'NA')}")

In [ ]:
tier2_hits = female_degs[female_degs['sfari_score'] == 2].sort_values('log2FoldChange', key=abs, ascending=False)
tier3_hits = female_degs[female_degs['sfari_score'] == 3].sort_values('log2FoldChange', key=abs, ascending=False)

print(f"=== Tier 2 significant DEGs (n={len(tier2_hits)}) ===")
print(tier2_hits[['gene_upper', 'log2FoldChange', 'padj']].to_string(index=False))

print(f"\n=== Tier 3 significant DEGs (n={len(tier3_hits)}) ===")
print(tier3_hits[['gene_upper', 'log2FoldChange', 'padj']].to_string(index=False))

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams['font.family']   = 'Helvetica'
plt.rcParams['font.size']     = 7
plt.rcParams['pdf.fonttype']  = 42
plt.rcParams['svg.fonttype']  = 'none'

# ── Curated, VERIFIED gene list — trimmed for better A/B panel balance ─────
# Down: 4x T1, 4x T2, 2x T3   |   Up: 3x T1, 1x T2, 1x T3
famous_asd_genes = [
    # Downregulated side
    'CHD8', 'MECP2', 'FMR1', 'CDKL5',      # Tier 1
    'SHANK1', 'OPHN1', 'MAOA', 'AR',       # Tier 2
    'CUL4B', 'BSN',                        # Tier 3
    # Upregulated side
    'AUTS2', 'SYNGAP1', 'TSC1',            # Tier 1
    'GRIP1',                               # Tier 2
    'ZEB2',                                # Tier 3
]

df_famous = female_degs[
    female_degs['gene_upper'].str.upper().isin([g.upper() for g in famous_asd_genes])
].copy()

found = set(df_famous['gene_upper'].str.upper())
missing = [g for g in famous_asd_genes if g.upper() not in found]
if missing:
    print(f"⚠️ Not found: {missing}")

print(f"Tier representation: "
      f"T1={sum(df_famous['sfari_score']==1)}, "
      f"T2={sum(df_famous['sfari_score']==2)}, "
      f"T3={sum(df_famous['sfari_score']==3)}")

df_down = df_famous[df_famous['log2FoldChange'] < 0].sort_values('log2FoldChange', ascending=True)
df_up   = df_famous[df_famous['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False)

n_down, n_up = len(df_down), len(df_up)
print(f"Selected: {n_down} downregulated, {n_up} upregulated (n={n_down + n_up} total)")

df_down_plot = df_down.sort_values('log2FoldChange', ascending=False)
df_up_plot   = df_up.sort_values('log2FoldChange', ascending=False)

def tier_label(row):
    return f"{row['gene_upper']} (T{int(row['sfari_score'])})"

df_down_plot['ylabel'] = df_down_plot.apply(tier_label, axis=1)
df_up_plot['ylabel']   = df_up_plot.apply(tier_label, axis=1)

BMC_MAX_HEIGHT_IN = 9.21
bar_slot_in       = 0.28   # slightly larger since fewer bars overall now
margin_in         = 1.9

fig_height = min(BMC_MAX_HEIGHT_IN, bar_slot_in * max(n_down, n_up) + margin_in)

fig, (ax_down, ax_up) = plt.subplots(
    1, 2, figsize=(7.28, fig_height), dpi=300,
    gridspec_kw={'width_ratios': [1, 1]}
)

tier_edge = {1: 'black', 2: '#555555', 3: '#aaaaaa'}
tier_lw   = {1: 0.9, 2: 0.5, 3: 0.3}

panel_specs = [
    (ax_down, df_down_plot, '#4575b4', 'A', f'Downregulated (n={n_down})'),
    (ax_up,   df_up_plot,   '#d73027', 'B', f'Upregulated (n={n_up})'),
]

for ax, df_plot, color, tag, panel_title in panel_specs:
    edge_colors = [tier_edge[s] for s in df_plot['sfari_score']]
    edge_widths = [tier_lw[s] for s in df_plot['sfari_score']]

    bars = ax.barh(df_plot['ylabel'], df_plot['log2FoldChange'],
                    color=color, edgecolor=edge_colors, linewidth=edge_widths, height=0.65)

    ax.axvline(0, color='black', linestyle='-', linewidth=0.75)
    ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
    ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis='both', width=0.5, labelsize=6)
    ax.invert_yaxis()

    ax.set_xlabel(r'$\log_2$ FC (KO/WT)', fontsize=7, fontweight='bold')
    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='bottom', ha='left')
    ax.set_title(panel_title, fontsize=7, fontweight='bold', loc='left', pad=2)

    # ── FIX: symmetric padding on BOTH sides, so the largest bar (whichever
    # sign) never collides with its own value label or the y-tick text ─────
    xmin, xmax = ax.get_xlim()
    x_span = xmax - xmin
    pad = x_span * 0.12
    ax.set_xlim(xmin - pad, xmax + pad)

    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = x_range * 0.015
    for bar in bars:
        width = bar.get_width()
        x_pos = width + offset if width >= 0 else width - offset
        ha = 'left' if width >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

fig.text(0.02, 0.5, 'Known ASD Risk Gene (SFARI Tier 1–3)',
          fontsize=7, fontweight='bold', rotation=90, va='center')

shared_handles = [
    Patch(facecolor='#4575b4', edgecolor='black', linewidth=0.4, label='Downregulated'),
    Patch(facecolor='#d73027', edgecolor='black', linewidth=0.4, label='Upregulated'),
    Patch(facecolor='white', edgecolor='black', linewidth=0.9, label='SFARI Tier 1 (high conf.)'),
    Patch(facecolor='white', edgecolor='#555555', linewidth=0.5, label='SFARI Tier 2 (strong cand.)'),
    Patch(facecolor='white', edgecolor='#aaaaaa', linewidth=0.3, label='SFARI Tier 3 (suggestive)'),
]
fig.legend(handles=shared_handles, loc='lower center', ncol=5,
           frameon=False, fontsize=5.5, bbox_to_anchor=(0.53, -0.02))

plt.tight_layout(rect=[0.03, 0.05, 1, 1], pad=0.5)

output_fig_dir = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_fig_dir, exist_ok=True)
output_pdf_path = os.path.join(output_fig_dir, "Figure8_Famous_ASD_Genes_Tier1to3.pdf")
plt.savefig(output_pdf_path, format='pdf', bbox_inches='tight')
tiff_output = os.path.join(output_fig_dir, "Figure8_Famous_ASD_Genes_Tier1to3.tiff")
plt.savefig(tiff_output, format='tiff', dpi=600, bbox_inches='tight',
            pil_kwargs={"compression": "tiff_lzw"})
plt.show()
print(f"✓ Figure saved to:\n{output_pdf_path}")
print(f"  Height: {fig_height:.2f}in (BMC max: {BMC_MAX_HEIGHT_IN}in)")

In [ ]:
from google.colab import files
uploaded = files.upload()  # this opens a file picker — select the xlsx from Downloads

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── - / BMC figure standards ──────────────────────────────────
plt.rcParams['font.family']   = 'Helvetica'
plt.rcParams['font.size']     = 7
plt.rcParams['pdf.fonttype']  = 42
plt.rcParams['svg.fonttype']  = 'none'

# ── Load pre-computed Top50 tables (already ranked by |log2FC|) ────────────
xlsx_path = "/content/SFARI_Tier1and2_Top50_Analysis.xlsx"
down_full = pd.read_excel(xlsx_path, sheet_name="Downregulated_Top50")
up_full   = pd.read_excel(xlsx_path, sheet_name="Upregulated_Top50")

N_PER_PANEL = 15

df_down = down_full.sort_values('log2FoldChange', ascending=True).head(N_PER_PANEL).copy()
df_up   = up_full.sort_values('log2FoldChange', ascending=False).head(N_PER_PANEL).copy()

n_down, n_up = len(df_down), len(df_up)
print(f"Selected: {n_down} downregulated, {n_up} upregulated examples "
      f"(Tier-1 n={sum(df_down['sfari_score']==1)+sum(df_up['sfari_score']==1)}, "
      f"Tier-2 n={sum(df_down['sfari_score']==2)+sum(df_up['sfari_score']==2)})")

# For within-panel plotting, largest-magnitude bar at top of each panel
df_down_plot = df_down.sort_values('log2FoldChange', ascending=False)  # most negative at top
df_up_plot   = df_up.sort_values('log2FoldChange', ascending=True)     # most positive at top

# Build tier-annotated y-tick labels, e.g. "MECP2 (T1)" — replaces any
# chromosome/X-linked annotation
def tier_label(row):
    return f"{row['gene_upper']} (T{int(row['sfari_score'])})"

df_down_plot['ylabel'] = df_down_plot.apply(tier_label, axis=1)
df_up_plot['ylabel']   = df_up_plot.apply(tier_label, axis=1)

# ── BMC max double-column type area: 234mm (h) x 185mm (w) = 9.21 x 7.28 in ─
BMC_MAX_HEIGHT_IN = 9.21
bar_slot_in       = 0.22
margin_in         = 1.6

fig_height = min(BMC_MAX_HEIGHT_IN, bar_slot_in * max(n_down, n_up) + margin_in)

fig, (ax_down, ax_up) = plt.subplots(
    1, 2, figsize=(7.28, fig_height), dpi=300,
    gridspec_kw={'width_ratios': [1, 1]}
)

# Tier is now shown via marker/edge style, not chromosome status
tier_edge = {1: 'black', 2: '#555555'}
tier_lw   = {1: 0.9, 2: 0.4}

panel_specs = [
    (ax_down, df_down_plot, '#4575b4', 'A', f'Downregulated (n={n_down})'),
    (ax_up,   df_up_plot,   '#d73027', 'B', f'Upregulated (n={n_up})'),
]

for ax, df_plot, color, tag, legend_label in panel_specs:
    edge_colors = [tier_edge[s] for s in df_plot['sfari_score']]
    edge_widths = [tier_lw[s] for s in df_plot['sfari_score']]

    bars = ax.barh(df_plot['ylabel'], df_plot['log2FoldChange'],
                    color=color, edgecolor=edge_colors, linewidth=edge_widths, height=0.7)

    ax.axvline(0, color='black', linestyle='-', linewidth=0.75)
    ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
    ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis='both', width=0.5, labelsize=6)
    ax.invert_yaxis()

    ax.set_xlabel(r'$\log_2$ FC (KO/WT)', fontsize=7, fontweight='bold')

    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='bottom', ha='left')

    # Legend now explains BOTH direction (fill) and tier (edge weight)
    legend_handles = [
        Patch(facecolor=color, edgecolor='black', linewidth=0.4, label=legend_label),
        Patch(facecolor='white', edgecolor='black', linewidth=0.9, label='SFARI Tier 1 (high conf.)'),
        Patch(facecolor='white', edgecolor='#555555', linewidth=0.4, label='SFARI Tier 2 (strong cand.)'),
    ]
    ax.legend(handles=legend_handles, loc='lower right', frameon=False, fontsize=5.5)

    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = x_range * 0.02
    for bar in bars:
        width = bar.get_width()
        x_pos = width + offset if width >= 0 else width - offset
        ha = 'left' if width >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

fig.text(0.02, 0.5, 'SFARI ASD Risk Gene (Tier 1 & 2)',
          fontsize=7, fontweight='bold', rotation=90, va='center')

plt.tight_layout(rect=[0.03, 0, 1, 1], pad=0.5)

output_fig_dir = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_fig_dir, exist_ok=True)
output_pdf_path = os.path.join(output_fig_dir, "Figure4_SFARI_Tier1and2_Examples.pdf")
plt.savefig(output_pdf_path, format='pdf', bbox_inches='tight')
plt.show()
print(f"✓ Figure saved to:\n{output_pdf_path}")
print(f"  Height: {fig_height:.2f}in (BMC max: {BMC_MAX_HEIGHT_IN}in)")

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams['font.family']   = 'Helvetica'
plt.rcParams['font.size']     = 7
plt.rcParams['pdf.fonttype']  = 42
plt.rcParams['svg.fonttype']  = 'none'

xlsx_path = "/content/SFARI_Tier1and2_Top50_Analysis.xlsx"
down_full = pd.read_excel(xlsx_path, sheet_name="Downregulated_Top50")
up_full   = pd.read_excel(xlsx_path, sheet_name="Upregulated_Top50")

N_PER_PANEL = 15
df_down = down_full.sort_values('log2FoldChange', ascending=True).head(N_PER_PANEL).copy()
df_up   = up_full.sort_values('log2FoldChange', ascending=False).head(N_PER_PANEL).copy()

n_down, n_up = len(df_down), len(df_up)

df_down_plot = df_down.sort_values('log2FoldChange', ascending=False)  # most negative at top
df_up_plot   = df_up.sort_values('log2FoldChange', ascending=False)    # most positive at top (fixed)

def tier_label(row):
    return f"{row['gene_upper']} (T{int(row['sfari_score'])})"

df_down_plot['ylabel'] = df_down_plot.apply(tier_label, axis=1)
df_up_plot['ylabel']   = df_up_plot.apply(tier_label, axis=1)

BMC_MAX_HEIGHT_IN = 9.21
bar_slot_in       = 0.22
margin_in         = 1.9   # slightly more room reserved for the shared bottom legend

fig_height = min(BMC_MAX_HEIGHT_IN, bar_slot_in * max(n_down, n_up) + margin_in)

fig, (ax_down, ax_up) = plt.subplots(
    1, 2, figsize=(7.28, fig_height), dpi=300,
    gridspec_kw={'width_ratios': [1, 1]}
)

tier_edge = {1: 'black', 2: '#555555'}
tier_lw   = {1: 0.9, 2: 0.4}

panel_specs = [
    (ax_down, df_down_plot, '#4575b4', 'A', f'Downregulated (n={n_down})'),
    (ax_up,   df_up_plot,   '#d73027', 'B', f'Upregulated (n={n_up})'),
]

# Give the bars a bit more x-range headroom so value labels never crowd the edge
for ax, df_plot, color, tag, panel_title in panel_specs:
    edge_colors = [tier_edge[s] for s in df_plot['sfari_score']]
    edge_widths = [tier_lw[s] for s in df_plot['sfari_score']]

    bars = ax.barh(df_plot['ylabel'], df_plot['log2FoldChange'],
                    color=color, edgecolor=edge_colors, linewidth=edge_widths, height=0.7)

    ax.axvline(0, color='black', linestyle='-', linewidth=0.75)
    ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
    ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis='both', width=0.5, labelsize=6)
    ax.invert_yaxis()

    ax.set_xlabel(r'$\log_2$ FC (KO/WT)', fontsize=7, fontweight='bold')

    # Panel tag now carries BOTH the letter and the direction/n label —
    # replaces the old in-axes color-patch legend, so nothing sits over the bars
    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='bottom', ha='left')
    ax.set_title(panel_title, fontsize=7, fontweight='bold', loc='left', pad=2)

    # add headroom on the value-label side so the longest bar's label + tier
    # tag never crowds the plot edge
    xmin, xmax = ax.get_xlim()
    pad = (xmax - xmin) * 0.08
    if xmax > 0:
        ax.set_xlim(xmin, xmax + pad)
    else:
        ax.set_xlim(xmin - pad, xmax)

    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = x_range * 0.015
    for bar in bars:
        width = bar.get_width()
        x_pos = width + offset if width >= 0 else width - offset
        ha = 'left' if width >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

fig.text(0.02, 0.5, 'SFARI ASD Risk Gene (Tier 1 & 2)',
          fontsize=7, fontweight='bold', rotation=90, va='center')

shared_handles = [
    Patch(facecolor='#4575b4', edgecolor='black', linewidth=0.4, label='Downregulated'),
    Patch(facecolor='#d73027', edgecolor='black', linewidth=0.4, label='Upregulated'),
    Patch(facecolor='white', edgecolor='black', linewidth=0.9, label='SFARI Tier 1 (high conf.)'),
    Patch(facecolor='white', edgecolor='#555555', linewidth=0.4, label='SFARI Tier 2 (strong cand.)'),
]
fig.legend(handles=shared_handles, loc='lower center', ncol=4,
           frameon=False, fontsize=6, bbox_to_anchor=(0.53, -0.02))

plt.tight_layout(rect=[0.03, 0.04, 1, 1], pad=0.5)  # rect leaves room at bottom for fig.legend

output_fig_dir = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_fig_dir, exist_ok=True)
output_pdf_path = os.path.join(output_fig_dir, "Figure4_SFARI_Tier1and2_Examples.pdf")
plt.savefig(output_pdf_path, format='pdf', bbox_inches='tight')
plt.show()
print(f"✓ Figure saved to:\n{output_pdf_path}")
print(f"  Height: {fig_height:.2f}in (BMC max: {BMC_MAX_HEIGHT_IN}in)")

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams['font.family']   = 'Helvetica'
plt.rcParams['font.size']     = 7
plt.rcParams['pdf.fonttype']  = 42
plt.rcParams['svg.fonttype']  = 'none'

xlsx_path = "/content/SFARI_Tier1and2_Top50_Analysis.xlsx"
down_full = pd.read_excel(xlsx_path, sheet_name="Downregulated_Top50")
up_full   = pd.read_excel(xlsx_path, sheet_name="Upregulated_Top50")

df_down = down_full[down_full['sfari_score'] == 1].copy()
df_up   = up_full[up_full['sfari_score'] == 1].copy()

df_down['abs_lfc'] = df_down['log2FoldChange'].abs()
df_up['abs_lfc']   = df_up['log2FoldChange'].abs()

df_down_plot = df_down.sort_values('abs_lfc', ascending=False)
df_up_plot   = df_up.sort_values('abs_lfc', ascending=False)

n_down, n_up = len(df_down_plot), len(df_up_plot)
print(f"Tier-1 downregulated: n={n_down}  |  Tier-1 upregulated: n={n_up}")

df_down_plot['ylabel'] = df_down_plot['gene_upper']
df_up_plot['ylabel']   = df_up_plot['gene_upper']

BMC_MAX_HEIGHT_IN = 9.21
bar_slot_in       = 0.22
margin_in         = 1.6   # shared legend now only needs 2 entries, less room

fig_height = min(BMC_MAX_HEIGHT_IN, bar_slot_in * max(n_down, n_up) + margin_in)

fig, (ax_down, ax_up) = plt.subplots(
    1, 2, figsize=(7.28, fig_height), dpi=300,
    gridspec_kw={'width_ratios': [1, 1]}
)

panel_specs = [
    (ax_down, df_down_plot, '#4575b4', 'A', f'Downregulated (n={n_down})'),
    (ax_up,   df_up_plot,   '#d73027', 'B', f'Upregulated (n={n_up})'),
]

for ax, df_plot, color, tag, panel_title in panel_specs:
    bars = ax.barh(df_plot['ylabel'], df_plot['log2FoldChange'],
                    color=color, edgecolor='black', linewidth=0.5, height=0.7)

    ax.axvline(0, color='black', linestyle='-', linewidth=0.75)
    ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
    ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis='both', width=0.5, labelsize=6)
    ax.invert_yaxis()

    ax.set_xlabel(r'$\log_2$ FC (KO/WT)', fontsize=7, fontweight='bold')

    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='bottom', ha='left')
    ax.set_title(panel_title, fontsize=7, fontweight='bold', loc='left', pad=2)

    vmin = min(0, df_plot['log2FoldChange'].min())
    vmax = max(0, df_plot['log2FoldChange'].max())
    span = vmax - vmin
    pad = span * 0.15
    ax.set_xlim(vmin - pad, vmax + pad)

    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = x_range * 0.015
    for bar in bars:
        width = bar.get_width()
        x_pos = width + offset if width >= 0 else width - offset
        ha = 'left' if width >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

fig.text(0.02, 0.5, 'SFARI Tier-1 (High-Confidence) ASD Risk Gene',
          fontsize=7, fontweight='bold', rotation=90, va='center')

shared_handles = [
    Patch(facecolor='#4575b4', edgecolor='black', linewidth=0.5, label='Downregulated'),
    Patch(facecolor='#d73027', edgecolor='black', linewidth=0.5, label='Upregulated'),
]
fig.legend(handles=shared_handles, loc='lower center', ncol=2,
           frameon=False, fontsize=6, bbox_to_anchor=(0.53, -0.02))

plt.tight_layout(rect=[0.03, 0.04, 1, 1], pad=0.5)

output_fig_dir = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_fig_dir, exist_ok=True)
output_pdf_path = os.path.join(output_fig_dir, "Figure4_SFARI_Tier1_FullCohort.pdf")
plt.savefig(output_pdf_path, format='pdf', bbox_inches='tight')
plt.show()
print(f"✓ Figure saved to:\n{output_pdf_path}")
print(f"  Height: {fig_height:.2f}in (BMC max: {BMC_MAX_HEIGHT_IN}in)")

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── - / BMC figure standards ──────────────────────────────────
plt.rcParams['font.family']   = 'Helvetica'  # or 'Arial' — sans-serif required
plt.rcParams['font.size']     = 7            # legible at 170mm print width
plt.rcParams['pdf.fonttype']  = 42           # embed fonts for Illustrator
plt.rcParams['svg.fonttype']  = 'none'

df_tier1 = female_degs[female_degs['sfari_score'] == 1].copy()

df_up   = df_tier1[df_tier1['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False)
df_down = df_tier1[df_tier1['log2FoldChange'] < 0].sort_values('log2FoldChange', ascending=True)

n_up, n_down = len(df_up), len(df_down)
print(f"Tier-1 cohort: {n_up} upregulated, {n_down} downregulated (n={n_up + n_down} total)")


df_plot = pd.concat([df_down, df_up]).sort_values('log2FoldChange', ascending=True)

# Dynamic figure height so bar spacing stays constant regardless of gene count
n_genes = len(df_plot)
fig_height = max(4.0, 0.22 * n_genes + 1.5)  # ~0.22in per bar + margin
fig, ax = plt.subplots(figsize=(6.69, fig_height), dpi=300)  # 170mm double-column width

colors = ['#d73027' if lfc > 0 else '#4575b4' for lfc in df_plot['log2FoldChange']]

bars = ax.barh(df_plot['gene_upper'], df_plot['log2FoldChange'],
               color=colors, edgecolor='black', linewidth=0.4, height=0.7)

# Reference lines (all > 0.25pt)
ax.axvline(0,    color='black', linestyle='-',  linewidth=0.75)
ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

# Spine and tick formatting
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(0.5)
ax.tick_params(axis='both', width=0.5, labelsize=6)

# Axis labels (NO title — goes in - legend)
ax.set_xlabel(r'$\log_2$ Fold Change (CHD8 KO / WT)', fontsize=7, fontweight='bold')
ax.set_ylabel('High-Confidence (Tier-1) ASD Risk Gene', fontsize=7, fontweight='bold')

# Legend key stays IN the figure per BMC rules — now with live counts, not hardcoded
legend_elements = [
    Patch(facecolor='#d73027', edgecolor='black', linewidth=0.4, label=f'Upregulated (n={n_up})'),
    Patch(facecolor='#4575b4', edgecolor='black', linewidth=0.4, label=f'Downregulated (n={n_down})'),
]
ax.legend(handles=legend_elements, loc='lower right', frameon=False, fontsize=6)

plt.tight_layout(pad=0.5)

# Dynamic value labels after layout
x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
offset = x_range * 0.015
for bar in bars:
    width = bar.get_width()
    x_pos = width + offset if width >= 0 else width - offset
    ha = 'left' if width >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

output_fig_dir = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_fig_dir, exist_ok=True)
output_pdf_path = os.path.join(output_fig_dir, "Figure8_SFARI_Tier1_Alterations.pdf")
plt.savefig(output_pdf_path, format='pdf', bbox_inches='tight')
plt.show()
print(f"✓ --compliant figure saved to:\n{output_pdf_path}")

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── - / BMC figure standards ──────────────────────────────────
plt.rcParams['font.family']   = 'Helvetica'
plt.rcParams['font.size']     = 7
plt.rcParams['pdf.fonttype']  = 42
plt.rcParams['svg.fonttype']  = 'none'

# ── Select data: full Tier-1 cohort, split by direction ────────────────────
df_tier1 = female_degs[female_degs['sfari_score'] == 1].copy()
df_down = df_tier1[df_tier1['log2FoldChange'] < 0].sort_values('log2FoldChange', ascending=True)
df_up   = df_tier1[df_tier1['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False)

n_down, n_up = len(df_down), len(df_up)
print(f"Tier-1 cohort: {n_down} downregulated, {n_up} upregulated (n={n_down + n_up} total)")

# For within-panel plotting, largest-magnitude bar at top of each panel
df_down_plot = df_down.sort_values('log2FoldChange', ascending=False)  # most negative at top
df_up_plot   = df_up.sort_values('log2FoldChange', ascending=True)     # most positive at top

# ── BMC max double-column type area: 234mm (h) x 185mm (w) = 9.21 x 7.28 in ─
# Two panels side by side, single composite file, height driven by the
# LARGER panel (downregulated, n=30) so both bar heights stay proportional.
BMC_MAX_HEIGHT_IN = 9.21
bar_slot_in       = 0.22   # in per bar, matches original spacing
margin_in         = 1.6    # room for axis labels, legend, panel tag

fig_height = min(BMC_MAX_HEIGHT_IN, bar_slot_in * max(n_down, n_up) + margin_in)

fig, (ax_down, ax_up) = plt.subplots(
    1, 2, figsize=(7.28, fig_height), dpi=300,
    gridspec_kw={'width_ratios': [1, 1]}
)

panel_specs = [
    (ax_down, df_down_plot, '#4575b4', 'A', f'Downregulated (n={n_down})'),
    (ax_up,   df_up_plot,   '#d73027', 'B', f'Upregulated (n={n_up})'),
]

for ax, df_plot, color, tag, legend_label in panel_specs:
    bars = ax.barh(df_plot['gene_upper'], df_plot['log2FoldChange'],
                    color=color, edgecolor='black', linewidth=0.4, height=0.7)

    ax.axvline(0, color='black', linestyle='-', linewidth=0.75)
    ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
    ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis='both', width=0.5, labelsize=6)
    ax.invert_yaxis()  # largest-magnitude gene at top of each panel

    ax.set_xlabel(r'$\log_2$ FC (KO/WT)', fontsize=7, fontweight='bold')

    # Panel tag (A/B) — bold, top-left, per BMC composite-figure convention
    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='bottom', ha='left')

    # In-graphic legend key (required — not allowed in legend text alone)
    ax.legend(handles=[Patch(facecolor=color, edgecolor='black', linewidth=0.4, label=legend_label)],
              loc='lower right', frameon=False, fontsize=6)

    # Value labels
    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = x_range * 0.02
    for bar in bars:
        width = bar.get_width()
        x_pos = width + offset if width >= 0 else width - offset
        ha = 'left' if width >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

# Shared y-axis label across both panels
fig.text(0.02, 0.5, 'High-Confidence (Tier-1) ASD Risk Gene',
          fontsize=7, fontweight='bold', rotation=90, va='center')

plt.tight_layout(rect=[0.03, 0, 1, 1], pad=0.5)

# Save as ONE composite file — BMC requires multi-panel figures as a single upload
output_fig_dir = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_fig_dir, exist_ok=True)
output_pdf_path = os.path.join(output_fig_dir, "Figure8_SFARI_Tier1_Alterations.pdf")
plt.savefig(output_pdf_path, format='pdf', bbox_inches='tight')
plt.show()
print(f"✓ --compliant two-panel figure saved to:\n{output_pdf_path}")
print(f"  Height: {fig_height:.2f}in (BMC max: {BMC_MAX_HEIGHT_IN}in)")

In [ ]:
import pandas as pd

# ── Thresholds ────────────────────────────────────────────────────────────────
PADJ_CUTOFF = 0.05

# 1. Filter for Tier-1 cohort
df_tier1 = female_degs[female_degs['sfari_score'] == 1].copy()

# 1b. Apply significance filter explicitly (don't rely on df name alone)
if PADJ_CUTOFF is not None:
    if 'padj' not in df_tier1.columns:
        raise ValueError("No 'padj' column found — cannot apply significance filter. "
                          "Set PADJ_CUTOFF = None if female_degs is already pre-filtered.")
    n_before = len(df_tier1)
    df_tier1 = df_tier1[df_tier1['padj'] < PADJ_CUTOFF]
    print(f"Significance filter (padj < {PADJ_CUTOFF}): {n_before} → {len(df_tier1)} genes")

# 2. Automatically identify X-linked genes
# (Handles common chromosome column names like 'chr', 'chromosome', 'Chromosome')
chr_col = next((col for col in df_tier1.columns if col.lower() in ['chr', 'chromosome']), None)
if chr_col:
    df_tier1['X-Linked'] = (
        df_tier1[chr_col].astype(str).str.strip().str.upper()
        .isin(['X', 'CHRX']).map({True: 'Yes', False: 'No'})
    )
else:
    raise ValueError(
        "No chromosome column found (looked for 'chr' / 'chromosome'). "
        f"Available columns: {list(df_tier1.columns)}"
    )

# 3. Pull and sort the Top 50 Downregulated & Top 50 Upregulated
df_down_excel = (
    df_tier1[df_tier1['log2FoldChange'] < 0]
    .sort_values('log2FoldChange', ascending=True)
    .head(50)
)
df_up_excel = (
    df_tier1[df_tier1['log2FoldChange'] > 0]
    .sort_values('log2FoldChange', ascending=False)
    .head(50)
)

print(f"Downregulated pulled: {len(df_down_excel)} | Upregulated pulled: {len(df_up_excel)}")

# 4. Save to a multi-tab Excel Workbook
output_excel_path = "/content/drive/MyDrive/Chd8 data/figures/SFARI_Tier1_Top100_Analysis.xlsx"
with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
    df_down_excel.to_excel(writer, sheet_name='Downregulated_Top50', index=False)
    df_up_excel.to_excel(writer, sheet_name='Upregulated_Top50', index=False)

    for sheet_name in writer.sheets:
        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes = 'A2'
        for col in worksheet.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            col_letter = col[0].column_letter
            worksheet.column_dimensions[col_letter].width = max(max_len + 3, 12)

print(f"✓ Data extended to Top 100 total and successfully saved to Excel:\n{output_excel_path}")

In [ ]:
import pandas as pd

xls_path = "/content/drive/MyDrive/Chd8 data/figures/SFARI_Tier1_Top100_Analysis.xlsx"

down = pd.read_excel(xls_path, sheet_name='Downregulated_Top50')
up   = pd.read_excel(xls_path, sheet_name='Upregulated_Top50')

print("Downregulated genes with DEREG_FLAG == 'NO':")
print(down[down['DEREG_FLAG'] == 'NO'][['GENESYMBOL', 'log2FoldChange', 'padj']])

print("\nUpregulated genes with DEREG_FLAG == 'NO':")
print(up[up['DEREG_FLAG'] == 'NO'][['GENESYMBOL', 'log2FoldChange', 'padj']])

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── - / BMC figure standards ──────────────────────────────────
plt.rcParams['font.family']   = 'Helvetica'
plt.rcParams['font.size']     = 7
plt.rcParams['pdf.fonttype']  = 42
plt.rcParams['svg.fonttype']  = 'none'

# ── Select data: full Tier-1 cohort, split by direction ────────────────────
df_tier1 = female_degs[female_degs['sfari_score'] == 1].copy()
df_down = df_tier1[df_tier1['log2FoldChange'] < 0].sort_values('log2FoldChange', ascending=True)
df_up   = df_tier1[df_tier1['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False)

n_down, n_up = len(df_down), len(df_up)
print(f"Tier-1 cohort: {n_down} downregulated, {n_up} upregulated (n={n_down + n_up} total)")

df_down_plot = df_down.sort_values('log2FoldChange', ascending=False)  # most negative (severe) -> last row -> top
df_up_plot   = df_up.sort_values('log2FoldChange', ascending=True)     # most positive (severe) -> last row -> top


BMC_MAX_HEIGHT_IN = 9.21
bar_slot_in       = 0.22   # in per bar, matches original spacing
margin_in         = 1.6    # room for axis labels, legend, panel tag

fig_height = min(BMC_MAX_HEIGHT_IN, bar_slot_in * max(n_down, n_up) + margin_in)

fig, (ax_down, ax_up) = plt.subplots(
    1, 2, figsize=(7.28, fig_height), dpi=300,
    gridspec_kw={'width_ratios': [1, 1]}
)

panel_specs = [
    (ax_down, df_down_plot, '#4575b4', 'A', f'Downregulated (n={n_down})'),
    (ax_up,   df_up_plot,   '#d73027', 'B', f'Upregulated (n={n_up})'),
]

for ax, df_plot, color, tag, legend_label in panel_specs:
    bars = ax.barh(df_plot['gene_upper'], df_plot['log2FoldChange'],
                    color=color, edgecolor='black', linewidth=0.4, height=0.7)

    ax.axvline(0, color='black', linestyle='-', linewidth=0.75)
    ax.axvline(0.5,  color='#666666', linestyle='--', linewidth=0.5)
    ax.axvline(-0.5, color='#666666', linestyle='--', linewidth=0.5)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(axis='both', width=0.5, labelsize=6)
    # NOTE: no ax.invert_yaxis() here — removed, see comment above df_down_plot/df_up_plot

    ax.set_xlabel(r'$\log_2$ FC (KO/WT)', fontsize=7, fontweight='bold')

    # Panel tag (A/B) — bold, top-left, per BMC composite-figure convention
    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='bottom', ha='left')

    # In-graphic legend key (required — not allowed in legend text alone)
    ax.legend(handles=[Patch(facecolor=color, edgecolor='black', linewidth=0.4, label=legend_label)],
              loc='lower right', frameon=False, fontsize=6)

    # Value labels
    x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
    offset = x_range * 0.02
    for bar in bars:
        width = bar.get_width()
        x_pos = width + offset if width >= 0 else width - offset
        ha = 'left' if width >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
                f'{width:.2f}', va='center', ha=ha, fontsize=5.5, color='#333333')

# Shared y-axis label across both panels
fig.text(0.02, 0.5, 'High-Confidence (Tier-1) ASD Risk Gene',
          fontsize=7, fontweight='bold', rotation=90, va='center')

plt.tight_layout(rect=[0.03, 0, 1, 1], pad=0.5)

# Save as ONE composite file — BMC requires multi-panel figures as a single upload
# Filename updated to match - numbering: this is Figure 4, not Figure 8
output_fig_dir = "/content/drive/MyDrive/Chd8 data/figures/"
os.makedirs(output_fig_dir, exist_ok=True)
output_pdf_path = os.path.join(output_fig_dir, "Figure4_SFARI_Tier1_Alterations.pdf")
plt.savefig(output_pdf_path, format='pdf', bbox_inches='tight')
plt.show()
print(f"✓ --compliant two-panel figure saved to:\n{output_pdf_path}")
print(f"  Height: {fig_height:.2f}in (BMC max: {BMC_MAX_HEIGHT_IN}in)")

In [ ]:
import pandas as pd

# ── Thresholds ────────────────────────────────────────────────────────────────
PADJ_CUTOFF = 0.05
TOP_N       = 50
# 1. Filter for Tier-1 AND Tier-2 cohort
df_tier12 = female_degs[female_degs['sfari_score'].isin([1, 2])].copy()

print(f"Tier-1 + Tier-2 pool: {len(df_tier12)} genes total "
      f"(Tier-1: {(df_tier12['sfari_score']==1).sum()}, "
      f"Tier-2: {(df_tier12['sfari_score']==2).sum()})")

# 1b. Explicit significance filter (don't assume female_degs is pre-filtered)
if 'padj' not in df_tier12.columns:
    raise ValueError("No 'padj' column found — cannot apply significance filter.")
n_before = len(df_tier12)
df_tier12 = df_tier12[df_tier12['padj'] < PADJ_CUTOFF]
print(f"After significance filter (padj < {PADJ_CUTOFF}): {n_before} → {len(df_tier12)} genes")

# 2. Identify X-linked genes
chr_col = next((col for col in df_tier12.columns if col.lower() in ['chr', 'chromosome']), None)
if chr_col is None:
    raise ValueError(
        f"No chromosome column found (looked for 'chr' / 'chromosome'). "
        f"Available columns: {list(df_tier12.columns)}"
    )
df_tier12['X-Linked'] = (
    df_tier12[chr_col].astype(str).str.strip().str.upper()
    .isin(['X', 'CHRX']).map({True: 'Yes', False: 'No'})
)
# Insert this right after building df_tier12 and before the up/down split+truncation
full_up_pool = df_tier12[df_tier12['log2FoldChange'] > 0].sort_values('log2FoldChange', ascending=False)

print(f"Full upregulated Tier-1+2 pool (pre-cutoff): {len(full_up_pool)} genes")
x_linked_up_full = full_up_pool[full_up_pool['X-Linked'] == 'Yes']
print(f"X-linked genes anywhere in the full upregulated pool: {len(x_linked_up_full)}")
if len(x_linked_up_full) > 0:
    print(x_linked_up_full[['GENESYMBOL', 'log2FoldChange', 'padj']])
    # Show their rank position to see if they fall beyond the Top-50 cutoff
    full_up_pool_reset = full_up_pool.reset_index(drop=True)
    ranks = full_up_pool_reset[full_up_pool_reset['X-Linked'] == 'Yes'].index + 1
    print(f"Rank position(s) of X-linked upregulated genes: {ranks.tolist()}")

# 3. Pull and sort Top-N Downregulated & Top-N Upregulated by magnitude
df_down_excel = (
    df_tier12[df_tier12['log2FoldChange'] < 0]
    .sort_values('log2FoldChange', ascending=True)   # most negative first
    .head(TOP_N)
)
df_up_excel = (
    df_tier12[df_tier12['log2FoldChange'] > 0]
    .sort_values('log2FoldChange', ascending=False)  # most positive first
    .head(TOP_N)
)

n_down, n_up = len(df_down_excel), len(df_up_excel)
x_down = (df_down_excel['X-Linked'] == 'Yes').sum()
x_up   = (df_up_excel['X-Linked'] == 'Yes').sum()

print(f"Downregulated pulled: {n_down} (X-linked: {x_down})")
print(f"Upregulated pulled:   {n_up} (X-linked: {x_up})")
print(f"Total genes: {n_down + n_up} | Total X-linked: {x_down + x_up}")

# 4. Save to a multi-tab Excel Workbook
output_excel_path = "/content/drive/MyDrive/Chd8 data/figures/SFARI_Tier1and2_Top50_Analysis.xlsx"
with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
    df_down_excel.to_excel(writer, sheet_name='Downregulated_Top50', index=False)
    df_up_excel.to_excel(writer, sheet_name='Upregulated_Top50', index=False)

    for sheet_name in writer.sheets:
        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes = 'A2'
        for col in worksheet.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            col_letter = col[0].column_letter
            worksheet.column_dimensions[col_letter].width = max(max_len + 3, 12)

print(f"✓ Extended (Tier 1+2) analysis saved to:\n{output_excel_path}")

In [ ]:
%whos DataFrame

In [ ]:
import os

# Persist to Drive instead of /content/local_data, which is wiped on restart
save_dir = "/content/drive/MyDrive/Chd8 data/reference/"
os.makedirs(save_dir, exist_ok=True)

gtf_gz_path = os.path.join(save_dir, "gencode.vM25.annotation.gtf.gz")
gtf_path = os.path.join(save_dir, "gencode.vM25.annotation.gtf")

if not os.path.exists(gtf_path):
    if not os.path.exists(gtf_gz_path):
        !wget -O "{gtf_gz_path}" https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_mouse/release_M25/gencode.vM25.annotation.gtf.gz
    !gunzip -k "{gtf_gz_path}"

print("GTF ready at:", gtf_path)

In [ ]:
import pandas as pd
import numpy as np
import os

# ── Paths & Setup ─────────────────────────────────────────────────────────────
base_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
ko_path   = os.path.join(base_path, "Diff_Fa2Ls4_Diff_Chd8_KO_CTRL.tsv")
kd1_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.1.tsv")
kd2_path  = os.path.join(base_path, "Diff_Scrambled_shRNA_Diff_Chd8_shRNA_clone_8.2.tsv")
gtf_path  = "/content/drive/MyDrive/Chd8 data/reference/gencode.vM25.annotation.gtf"

sfari_list_path = "/content/drive/MyDrive/Chd8 data/reference/SFARI-Gene_genes.csv"  # <-- update this path

output_excel = "/content/Figure4_SFARI_to_Global_X_Sequence.xlsx"

# ── Parse GTF for Chromosome Map ────────────────────────────────────────
print("Extracting chromosome locations from GTF...")
gene_chrom_map = {}
if os.path.exists(gtf_path):
    with open(gtf_path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.split('\t')
            if len(parts) < 9:
                continue
            chrom = parts[0]
            attrs = parts[8]
            if 'gene_name "' in attrs:
                gname = attrs.split('gene_name "')[1].split('"')[0].upper().strip()
                gene_chrom_map[gname] = chrom
else:
    raise FileNotFoundError(f"GTF file not found: {gtf_path}")

# Normalize chromosome strings so both "chrX" and "X" style GTFs are handled
def normalize_chrom(chrom):
    c = str(chrom).strip()
    if not c.lower().startswith('chr'):
        c = 'chr' + c
    return c

# ── Load SFARI scores ──────────────────────
def load_sfari_scores(path):
    """
    Expects a SFARI gene list export with a gene-symbol column and a score
    column. Adjust the column names below to match your actual file.
    """
    sfari = pd.read_csv(path)
    gene_col = next((c for c in sfari.columns if c.lower() in
                      ['gene symbol', 'gene-symbol', 'genesymbol', 'symbol']), None)
    score_col = next((c for c in sfari.columns if 'score' in c.lower()), None)
    if gene_col is None or score_col is None:
        raise ValueError(
            f"Could not find gene symbol / score columns in {path}. "
            f"Found columns: {list(sfari.columns)}"
        )
    sfari['GENESYMBOL_UPPER'] = sfari[gene_col].astype(str).str.upper().str.strip()
    sfari = sfari.drop_duplicates(subset='GENESYMBOL_UPPER', keep='first')
    return sfari.set_index('GENESYMBOL_UPPER')[score_col].rename('sfari_score')

sfari_scores = None
if sfari_list_path is not None:
    if not os.path.exists(sfari_list_path):
        raise FileNotFoundError(
            f"sfari_list_path is set to '{sfari_list_path}' but the file does not exist. "
            f"Update sfari_list_path or set it to None if scores are already in the DESeq2 files."
        )
    sfari_scores = load_sfari_scores(sfari_list_path)

# ── Load Data ──────────────────────────────────────────────────────────────
def load_and_prep(path):
    df = pd.read_csv(path, sep='\t')
    gene_col = next((c for c in df.columns if c.upper() in ['GENESYMBOL', 'SYMBOL', 'MGI_SYMBOL']), None)
    if gene_col is None:
        raise ValueError(f"No recognizable gene symbol column found in {path}. Columns: {list(df.columns)}")
    df = df.rename(columns={gene_col: 'GENESYMBOL'})
    df['GENESYMBOL_UPPER'] = df['GENESYMBOL'].astype(str).str.upper().str.strip()

    n_before = len(df)
    df = df.drop_duplicates(subset='GENESYMBOL_UPPER', keep='first')
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        print(f"WARNING: {path} had {n_dropped} duplicate gene symbol rows; kept the first occurrence of each.")

    df = df.set_index('GENESYMBOL_UPPER')


    if 'sfari_score' not in df.columns:
        if sfari_scores is None:
            raise ValueError(
                f"'sfari_score' column not found in {path}, and no sfari_list_path was provided "
                f"to merge - scores from. Refusing to -cate SFARI scores."
            )
        df = df.join(sfari_scores, how='left')

    return df

ko_df   = load_and_prep(ko_path)
kd81_df = load_and_prep(kd1_path)
kd82_df = load_and_prep(kd2_path)

# ── Restrict to genes present in all three tables ─────────────────────────────
common_idx = ko_df.index.intersection(kd81_df.index).intersection(kd82_df.index)
ko_df, kd81_df, kd82_df = ko_df.loc[common_idx], kd81_df.loc[common_idx], kd82_df.loc[common_idx]

# ── Intersect shared deregulated lists ─────────────────────────────────────────
def get_shared_sequence(direction='DOWN'):
    if direction == 'DOWN':
        cond = (ko_df['log2FoldChange'] < 0) & (ko_df['padj'] < 0.05) & \
               (kd81_df['log2FoldChange'] < 0) & (kd81_df['padj'] < 0.05) & \
               (kd82_df['log2FoldChange'] < 0) & (kd82_df['padj'] < 0.05)
    else:
        cond = (ko_df['log2FoldChange'] > 0) & (ko_df['padj'] < 0.05) & \
               (kd81_df['log2FoldChange'] > 0) & (kd81_df['padj'] < 0.05) & \
               (kd82_df['log2FoldChange'] > 0) & (kd82_df['padj'] < 0.05)

    shared_indices = ko_df[cond].index
    print(f"[{direction}] {len(shared_indices)} genes pass the shared significance filter "
          f"(showing up to 150 in output).")

    res = pd.DataFrame({
        'Gene Symbol': ko_df.loc[shared_indices, 'GENESYMBOL'].values,
        'SFARI Score': ko_df.loc[shared_indices, 'sfari_score'].values,
        'KO log2FC'  : ko_df.loc[shared_indices, 'log2FoldChange'].values,
        'KO padj'    : ko_df.loc[shared_indices, 'padj'].values,
        'KD 8.1 log2FC': kd81_df.loc[shared_indices, 'log2FoldChange'].values,
        'KD 8.2 log2FC': kd82_df.loc[shared_indices, 'log2FoldChange'].values,
    }, index=shared_indices)

    res['Mean Abs log2FC'] = res[['KO log2FC', 'KD 8.1 log2FC', 'KD 8.2 log2FC']].abs().mean(axis=1)

    # ── Sequence ordering: Tier-1 SFARI genes first, then everything else, both by magnitude ──
    res['Sequence_Group'] = np.where(res['SFARI Score'] == 1, 0, 1)
    res = res.sort_values(by=['Sequence_Group', 'Mean Abs log2FC'], ascending=[True, False])

    # Map chromosome & X-linked status
    chroms = [gene_chrom_map.get(g, 'Unknown') for g in res.index]
    res['Chromosome'] = [normalize_chrom(c) if c != 'Unknown' else c for c in chroms]
    res['Is X-Linked?'] = res['Chromosome'].apply(lambda x: 'Yes' if x == 'chrX' else 'No')

    # Tracker for known functional links to XCI or X-escape
    xci_links = []
    for g in res.index:
        if g in ['NANOG', 'POU5F1', 'SOX2']:
            xci_links.append('Pluripotency factor linked to X-reactivation kinetics')
        elif g in ['XIST', 'TSIX', 'FTX', 'JPX', 'SPEN', 'HNRNPK']:
            xci_links.append('Core X-Inactivation (XCI) machinery/interactor')
        elif g.startswith('KDM') or g.startswith('HDAC'):
            xci_links.append('Epigenetic/Chromatin modifier affecting X structural silencing')
        else:
            xci_links.append('-')
    res['X-Chromosome Interaction Profile'] = xci_links

    ordered_cols = ['Gene Symbol', 'SFARI Score', 'Chromosome', 'Is X-Linked?',
                    'X-Chromosome Interaction Profile', 'Mean Abs log2FC',
                    'KO log2FC', 'KO padj', 'KD 8.1 log2FC', 'KD 8.2 log2FC']
    return res[ordered_cols].head(150)

# Run for both directions
down_seq_sheet = get_shared_sequence('DOWN')
up_seq_sheet   = get_shared_sequence('UP')

# ── Save cleanly formatted Excel workbook ─────────────────────────────────────
print(f"Saving merged target sequence to: {output_excel}")
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    down_seq_sheet.to_excel(writer, sheet_name='Downregulated Sequence', index=False)
    up_seq_sheet.to_excel(writer, sheet_name='Upregulated Sequence', index=False)

    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        ws.freeze_panes = 'A2'
        for col in ws.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            col_letter = col[0].column_letter
            ws.column_dimensions[col_letter].width = max(max_len + 3, 13)

print("Workbook successfully created!")

In [ ]:
import pandas as pd
df = pd.read_csv(ko_path, sep='\t')
print(df.columns.tolist())

In [ ]:
print(df['CHROMOSOME'].unique()[:20])

In [ ]:
import os

save_dir = "/content/drive/MyDrive/Chd8 data/reference/"
os.makedirs(save_dir, exist_ok=True)

sfari_csv_path = os.path.join(save_dir, "SFARI-Gene_genes.csv")

!wget -O "{sfari_csv_path}" "https://gene.sfari.org//wp-content/themes/sfari-gene/utilities/download-csv.php?api-endpoint=genes"

import pandas as pd
sfari_check = pd.read_csv(sfari_csv_path)
print(sfari_check.shape)
print(sfari_check.columns.tolist())
print(sfari_check.head())

In [ ]:
import os
import pandas as pd

# Define your base search directory
search_dir = "/content/drive/MyDrive/Chd8 data/"

print("Searching for existing count matrix files...")
for root, dirs, files in os.walk(search_dir):
    for file in files:
        # Looking for common featureCounts or matrix output names
        if "count" in file.lower() or "matrix" in file.lower():
            full_path = os.path.join(root, file)
            print(f"Found file: {full_path}")

            # Let's peek at the columns to see if they match the 18 GEO lane IDs
            try:
                df_peek = pd.read_csv(full_path, sep=None, nrows=2, engine='python')
                print(f"   -> Columns look like: {list(df_peek.columns[:5])}...")
            except Exception:
                pass

In [ ]:
import pandas as pd

# 1. Load the master matrix
master_path = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/counts_star_trimmed.tsv"
df = pd.read_csv(master_path, sep='\t')

# 2. Fix the Gene ID column
# If the gene IDs are in the first column (often named 'Unnamed: 0'), rename it properly
if df.columns[0] == 'Unnamed: 0' or 'gene' in str(df.columns[0]).lower():
    df = df.rename(columns={df.columns[0]: "Gene_ID"})
else:
    # If it loaded with gene IDs as the actual DataFrame index
    df = df.reset_index().rename(columns={"index": "Gene_ID"})

rescue_lanes = [
    "HKNFWBGX5_1", "HKNFWBGX5_2", "HKNFWBGX5_3", "HKNFWBGX5_4", "HKNFWBGX5_5",
    "HKNFWBGX5_6", "HKNFWBGX5_8", "HKNFWBGX5_9", "HKNFWBGX5_10", "HKNFWBGX5_13",
    "HKNFWBGX5_14", "HKNFWBGX5_15", "HKLWCBGX5_19", "HKLWCBGX5_20", "HKLWCBGX5_21",
    "HKLWCBGX5_24", "HKLWCBGX5_25", "HKLWCBGX5_26"
]

# Verify which target columns are present in your file
available_lanes = [col for col in rescue_lanes if col in df.columns]
print(f"Matched {len(available_lanes)} out of 18 rescue lanes in the master file.")

# 4. Filter the matrix to include ONLY 'Gene_ID' and the 18 target lanes
clean_columns = ["Gene_ID"] + available_lanes
geo_matrix = df[clean_columns]

# 5. Export the clean matrix
output_file = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/GEO_rescue_raw_counts_final.tsv"
geo_matrix.to_csv(output_file, sep='\t', index=False)

print(f"\n✅ Done! Clean matrix saved with shape {geo_matrix.shape} (Rows: Genes, Columns: Gene_ID + 18 Samples)")
print(f"File Location: {output_file}")

In [ ]:
# ==============================================================================
# REQUIRED INITIALIZATION CELL — Run this first to declare variables
# ==============================================================================
import os
import pybedtools
import pandas as pd

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Declare structural workspace and file system variables
base_dir = "/content/drive/MyDrive/CHD8_Fig1"

NPC_CONSENSUS_PATH = f"{base_dir}/peaks/CHD8_NPC_consensus.bed"
H3K4_PATH          = f"{base_dir}/peaks/H3K4me3_NPC_merged.bed"
TSS_POINT_PATH     = f"{base_dir}/mm10_TSS_point.bed"  # <-- verify this exists, see check below

# Verify the paths actually exist before trusting anything downstream
for label, path in [
    ("NPC_CONSENSUS_PATH", NPC_CONSENSUS_PATH),
    ("H3K4_PATH", H3K4_PATH),
    ("TSS_POINT_PATH", TSS_POINT_PATH),
]:
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {label}: {path}  (exists={exists})")

# Standard mouse chroms only (chr1-19, chrX) — drops raw scaffolds/contigs
STANDARD_CHROMS = {f"chr{i}" for i in range(1, 20)} | {"chrX"}

def is_standard(feature):
    chrom = feature.chrom
    if not str(chrom).startswith("chr"):
        chrom = "chr" + str(chrom)
    return chrom in STANDARD_CHROMS

# 3. Instantiate files as BedTool objects in active cache memory
consensus_bed = pybedtools.BedTool(NPC_CONSENSUS_PATH).filter(is_standard).sort().saveas()
h3k4_bed      = pybedtools.BedTool(H3K4_PATH).filter(is_standard).sort().saveas()
tss_bed       = pybedtools.BedTool(TSS_POINT_PATH).filter(is_standard).sort().saveas()

# Keep old names as aliases too, in case other cells/notebooks reference them
npc_bed = consensus_bed
tss_point_path = TSS_POINT_PATH
h3k4_path = H3K4_PATH

print(f"\nconsensus_bed: {consensus_bed.count():,} intervals, {consensus_bed.field_count()} fields")
print(f"h3k4_bed:      {h3k4_bed.count():,} intervals, {h3k4_bed.field_count()} fields")
print(f"tss_bed:       {tss_bed.count():,} intervals, {tss_bed.field_count()} fields")
print("First tss_bed record:", tss_bed[0])

print("\n✅ Initialization complete. 'consensus_bed', 'h3k4_bed', 'tss_bed' are ready.")

In [ ]:
!apt-get install -y bedtools -qq
!pip install pybedtools -q

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
RNA_DIR   = "/content/drive/MyDrive/Chd8 data/deseq2/deseq2/results/"
ATAC_BASE = "/content/drive/MyDrive/Chd8 data/macs2/MACS2_peaks/ATAC/"
TSS_PATH  = "/content/drive/MyDrive/Chd8 data/annotations/Mus_musculus_TSS_2kb_sorted.bed"
CHIP_PATH = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_NPC_merged.bed"
ESC_PATH  = "/content/drive/MyDrive/CHD8_Fig1/peaks/CHD8_ES_merged.bed"
SUPP_SIG  = "/content/drive/MyDrive/Chd8 data/figures/SuppTable_RescueGroups_ContinuousSignals_v2.csv"
SUPP_RF   = "/content/drive/MyDrive/Chd8 data/figures/Supp_Table_Domain_Rescue_Groups.csv"

In [ ]:
def normalize_and_get_genome(npc_bed, tss_midpoints):
    npc_chroms = set(f.chrom for f in npc_bed)
    tss_chroms = set(f.chrom for f in tss_midpoints)

    npc_has_chr = all(c.startswith("chr") for c in npc_chroms)
    tss_has_chr = all(c.startswith("chr") for c in tss_chroms)

    if npc_has_chr and not tss_has_chr:
        print("⚠️ Adding 'chr' prefix to TSS file to match NPC peaks")
        tss_midpoints = tss_midpoints.each(
            lambda f: pybedtools.create_interval_from_list(
                ["chr" + f.chrom, f.start, f.end]
            )
        ).saveas()
    elif tss_has_chr and not npc_has_chr:
        print("⚠️ Adding 'chr' prefix to NPC peaks to match TSS file")
        npc_bed = npc_bed.each(
            lambda f: pybedtools.create_interval_from_list(
                ["chr" + f.chrom, f.start, f.end]
            )
        ).saveas()

    mm10_sizes_chr = {
        "chr1": 195471971, "chr2": 182113224, "chr3": 160039680,
        "chr4": 156508116, "chr5": 151834684, "chr6": 149736546,
        "chr7": 145441459, "chr8": 129401213, "chr9": 124595110,
        "chr10": 130694993, "chr11": 122082543, "chr12": 120129022,
        "chr13": 120421639, "chr14": 124902244, "chr15": 104043685,
        "chr16": 98207768, "chr17": 94987271, "chr18": 90702639,
        "chr19": 61431566, "chrX": 171031299, "chrY": 91744698,
    }

    final_chroms = set(f.chrom for f in npc_bed)
    use_chr_prefix = all(c.startswith("chr") for c in final_chroms)
    genome_dict = mm10_sizes_chr if use_chr_prefix else {k.replace("chr", ""): v for k, v in mm10_sizes_chr.items()}

    genome_file = f"{base_dir}/mm10_matched.genome"
    with open(genome_file, "w") as f:
        for chrom, size in genome_dict.items():
            f.write(f"{chrom}\t{size}\n")

    return npc_bed, tss_midpoints, genome_file

In [ ]:
def run_tss_window_sensitivity(npc_bed, tss_bed_2kb, windows_kb=[1, 2, 3, 5, 10]):
    tss_midpoints = tss_bed_2kb.each(
        lambda f: pybedtools.create_interval_from_list([
            f.chrom,
            str(int((f.start + f.end) / 2)),
            str(int((f.start + f.end) / 2) + 1)
        ])
    ).saveas()

    npc_bed, tss_midpoints, genome_file = normalize_and_get_genome(npc_bed, tss_midpoints)

    results = []
    for w in windows_kb:
        window_bp = w * 1000
        tss_window = tss_midpoints.slop(b=window_bp, g=genome_file).sort().merge()

        promoter = npc_bed.intersect(tss_window, u=True).count()
        total    = npc_bed.count()
        distal   = total - promoter

        results.append({
            "window_kb": f"±{w}kb",
            "promoter_n": promoter,
            "promoter_pct": round(promoter / total * 100, 1),
            "distal_n": distal,
            "distal_pct": round(distal / total * 100, 1),
        })
        print(f"   ±{w}kb TSS window → Promoter: {promoter:,} ({promoter/total*100:.1f}%)  "
              f"Distal: {distal:,} ({distal/total*100:.1f}%)")

    df = pd.DataFrame(results)
    df.to_csv(f"{base_dir}/TSS_window_sensitivity.csv", index=False)
    print(f"\n✅ Saved sensitivity table: {base_dir}/TSS_window_sensitivity.csv")
    return df, npc_bed

In [ ]:
def run_tss_window_sensitivity(npc_bed, tss_bed_2kb, windows_kb=[1, 2, 3, 5, 10]):
    tss_midpoints = tss_bed_2kb.each(
        lambda f: pybedtools.create_interval_from_list([
            f.chrom,
            str(int((f.start + f.end) / 2)),
            str(int((f.start + f.end) / 2) + 1)
        ])
    ).saveas()

    npc_bed, tss_midpoints, genome_file = normalize_and_get_genome(npc_bed, tss_midpoints)

    # NEW: drop scaffolds/patches — keep only standard chroms present in genome_file
    tss_midpoints = tss_midpoints.filter(is_standard).sort().saveas()
    npc_bed = npc_bed.filter(is_standard).sort().saveas()

    results = []
    for w in windows_kb:
        window_bp = w * 1000
        tss_window = tss_midpoints.slop(b=window_bp, g=genome_file).sort().merge()

        promoter = npc_bed.intersect(tss_window, u=True).count()
        total    = npc_bed.count()
        distal   = total - promoter

        results.append({
            "window_kb": f"±{w}kb",
            "promoter_n": promoter,
            "promoter_pct": round(promoter / total * 100, 1),
            "distal_n": distal,
            "distal_pct": round(distal / total * 100, 1),
        })
        print(f"   ±{w}kb TSS window → Promoter: {promoter:,} ({promoter/total*100:.1f}%)  "
              f"Distal: {distal:,} ({distal/total*100:.1f}%)")

    df = pd.DataFrame(results)
    df.to_csv(f"{base_dir}/TSS_window_sensitivity.csv", index=False)
    print(f"\n✅ Saved sensitivity table: {base_dir}/TSS_window_sensitivity.csv")
    return df, npc_bed

In [ ]:
sens_df, npc_bed = run_tss_window_sensitivity(npc_bed, tss_bed)

In [ ]:
# Inspect the raw TSS file structure
print("First 5 lines of TSS file:")
for f in tss_bed[:5]:
    width = f.end - f.start
    print(f"  {f.chrom}:{f.start}-{f.end}  (width={width})")

# Check if window width is consistent (should all be ~4000bp if it's a true +/-2kb window)
widths = [f.end - f.start for f in tss_bed]
import numpy as np
print(f"\nWidth stats: min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.1f}")
print(f"Unique widths (first 10): {sorted(set(widths))[:10]}")

In [ ]:
# Direct validation: does intersecting npc_bed with the ORIGINAL tss_bed
# (no reconstruction) reproduce your Fig1B number of 16,275?
direct_promoter = npc_bed.intersect(tss_bed, u=True).count()
print(f"Direct intersect with original tss_bed: {direct_promoter:,}")
print(f"Expected from Fig1B script: 16,275")

In [ ]:
def run_tss_window_sensitivity_v2(npc_bed, tss_bed_2kb, windows_kb=[1, 2, 3, 5, 10]):
    """
    Avoids reconstructing TSS midpoints (unreliable after merge).
    Treats the existing +/-2kb file as the ground truth for the 2kb row,
    and only tests WIDER windows by growing outward from it directly.
    Narrower windows (<2kb) cannot be reliably tested from this file alone.
    """
    npc_bed, tss_2kb, genome_file = normalize_and_get_genome(npc_bed, tss_bed_2kb)
    tss_2kb = tss_2kb.filter(is_standard).sort().saveas()
    npc_bed = npc_bed.filter(is_standard).sort().saveas()

    results = []
    for w in windows_kb:
        if w < 2:
            print(f"   ±{w}kb: skipped — cannot shrink a merged window reliably, "
                  "need original unmerged single-bp TSS coordinates")
            continue
        elif w == 2:
            tss_window = tss_2kb  # ground truth, exactly your Fig1B file
        else:
            extra_bp = (w - 2) * 1000
            tss_window = tss_2kb.slop(b=extra_bp, g=genome_file).sort().merge()

        promoter = npc_bed.intersect(tss_window, u=True).count()
        total    = npc_bed.count()
        distal   = total - promoter

        results.append({
            "window_kb": f"±{w}kb",
            "promoter_n": promoter,
            "promoter_pct": round(promoter / total * 100, 1),
            "distal_n": distal,
            "distal_pct": round(distal / total * 100, 1),
        })
        print(f"   ±{w}kb TSS window → Promoter: {promoter:,} ({promoter/total*100:.1f}%)  "
              f"Distal: {distal:,} ({distal/total*100:.1f}%)")

    df = pd.DataFrame(results)
    df.to_csv(f"{base_dir}/TSS_window_sensitivity_v2.csv", index=False)
    print(f"\n✅ Saved: {base_dir}/TSS_window_sensitivity_v2.csv")
    return df

In [ ]:
sens_df2 = run_tss_window_sensitivity_v2(npc_bed, tss_bed)

In [ ]:
def run_tss_window_sensitivity_v3(npc_bed, tss_point_path, windows_kb=[1, 2, 3, 5, 10]):
    tss_point = pybedtools.BedTool(tss_point_path)
    npc_bed, tss_point, genome_file = normalize_and_get_genome(npc_bed, tss_point)
    tss_point = tss_point.filter(is_standard).sort().saveas()
    npc_bed = npc_bed.filter(is_standard).sort().saveas()

    results = []
    for w in windows_kb:
        window_bp = w * 1000
        tss_window = tss_point.slop(b=window_bp, g=genome_file).sort().merge()

        promoter = npc_bed.intersect(tss_window, u=True).count()
        total    = npc_bed.count()
        distal   = total - promoter

        results.append({
            "window_kb": f"±{w}kb",
            "promoter_n": promoter,
            "promoter_pct": round(promoter / total * 100, 1),
            "distal_n": distal,
            "distal_pct": round(distal / total * 100, 1),
        })
        print(f"   ±{w}kb TSS window → Promoter: {promoter:,} ({promoter/total*100:.1f}%)  "
              f"Distal: {distal:,} ({distal/total*100:.1f}%)")

    df = pd.DataFrame(results)
    df.to_csv(f"{base_dir}/TSS_window_sensitivity_v3.csv", index=False)
    print(f"\n✅ Saved: {base_dir}/TSS_window_sensitivity_v3.csv")
    return df

sens_df3 = run_tss_window_sensitivity_v3(npc_bed, tss_point_path)

In [ ]:
import subprocess

# 1. See everything inside the peaks/ subfolder (this is likely where replicates live)
result = subprocess.run(
    ['ls', '-la', f'{base_dir}/peaks/'],
    capture_output=True, text=True
)
print(f"Contents of {base_dir}/peaks/:")
print(result.stdout)
print(result.stderr)

# 2. Broader search across all of Drive for anything CHD8/NPC + replicate-like naming
patterns = ['*CHD8*rep*', '*CHD8*NPC*', '*chd8*rep*', '*rep1*', '*rep2*', '*replicate*']
for pat in patterns:
    result = subprocess.run(
        ['find', '/content/drive/MyDrive', '-iname', pat],
        capture_output=True, text=True
    )
    if result.stdout.strip():
        print(f"\nMatches for '{pat}':")
        print(result.stdout)

In [ ]:
import subprocess

npc_related_files = [
    "Chd8_Undiff_1_peaks.narrowPeak",
    "Chd8_Undiff_2_peaks.narrowPeak",
    "Chd8_Diff_2_peaks.narrowPeak",
    "CHD8_NPC.bed",
    "CHD8_NPC.sorted.bed",
    "CHD8_NPC_specific.bed",
    "CHD8_NPC_specific_filtered.bed",
    "CHD8_NPC_specific_filtered_final.bed",
    "CHD8_NPC_consensus_final.bed",
    "CHD8_NPC_merged.bed",
    "CHD8_NPC_H3K4me3_overlap.bed",
]

print(f"{'File':<45}{'Line count':>12}\n" + "-"*57)
for fname in npc_related_files:
    path = f"{base_dir}/peaks/{fname}"
    result = subprocess.run(['wc', '-l', path], capture_output=True, text=True)
    n_lines = result.stdout.split()[0] if result.stdout else "ERR"
    print(f"{fname:<45}{n_lines:>12}")

# Also peek at first 2 lines of the narrowPeak files to confirm they're CHD8 replicate calls
print("\n--- Sample rows from narrowPeak files ---")
for fname in ["Chd8_Undiff_1_peaks.narrowPeak", "Chd8_Undiff_2_peaks.narrowPeak", "Chd8_Diff_2_peaks.narrowPeak"]:
    path = f"{base_dir}/peaks/{fname}"
    result = subprocess.run(['head', '-2', path], capture_output=True, text=True)
    print(f"\n{fname}:")
    print(result.stdout)

In [ ]:
import subprocess

# Broader search in case Diff_1 exists under a different name/case
for pat in ['*Diff_1*', '*diff1*', '*Diff1*', '*_1_peaks*', '*replicate*npc*', '*NPC*rep*']:
    result = subprocess.run(['find', '/content/drive/MyDrive', '-iname', pat],
                             capture_output=True, text=True)
    if result.stdout.strip():
        print(f"Matches for '{pat}':\n{result.stdout}")

# Check if CHD8_NPC_merged.bed's extra ~9,446 lines (48,872 - 39,426) come from
# combining NPC.bed with something identifiable — compare peak overlap
npc_base = pybedtools.BedTool(f"{base_dir}/peaks/CHD8_NPC.bed")
npc_merged = pybedtools.BedTool(f"{base_dir}/peaks/CHD8_NPC_merged.bed")
only_in_merged = npc_merged.intersect(npc_base, v=True)
print(f"\nCHD8_NPC.bed: {npc_base.count():,} peaks")
print(f"CHD8_NPC_merged.bed: {npc_merged.count():,} peaks")
print(f"Peaks in merged but NOT in NPC.bed: {only_in_merged.count():,}")
print("First few 'extra' peaks (helps identify their source):")
for f in only_in_merged[:3]:
    print(f"  {f.chrom}:{f.start}-{f.end}  name={f.name if len(f.fields) > 3 else 'N/A'}")